# noisy_cost_comparison — 物理誤り率を与えたときの、Kobayashi 方式（D4 code switching）と Litinski 方式（15-to-1 蒸留）の資源比較

**問い**：物理誤り率 $p$ と要求される出力誤り $\varepsilon_{\rm target}$ が与えられたとき、論理 $T$ 魔法状態 1 個を作るのに、どちらの方式が少ない時空間体積（物理量子ビット × 時間ステップ）で済むか。

**このノートがすること**（重い計算はしない。数分で終わる）
1. **Kobayashi 方式**：`D4_noisy_simulation.ipynb` の回路レベル・モンテカルロの結果 $\varepsilon_K(d,p)$（非 Clifford 回路を Takada らの方法でシミュレーション）を読み込み、$d$ と $p$ への依存のモデルを当てはめる（不確かさはブートストラップ）。資源は同じ回路から数える。
2. **Litinski 方式**：Litinski（2019）の公開コードの誤り予算（本研究で再実装し、公開値 8 件を再現：`validation.ipynb` N6）を、*A Game of Surface Codes*（GoSC）Sec. 3 の 11-tile 15-to-1 と 176-tile 225-to-1、および Litinski（2019）の最適化された蒸留器に適用する。パッチの論理誤り率 $p_L(d,p)$ は同じ雑音パラメータの表面符号シミュレーションで較正する。
3. 同じ単位（物理量子ビット × 時間ステップ、全ゲート 1 時間ステップ）で比較する。

**比較の非対称性**：Kobayashi 側は切り替えを含む回路全体のモンテカルロ、Litinski 側は「較正した表面符号メモリの誤り率」を入れた解析的な誤り予算モデル（格子手術や蒸留器全体を回路レベルでシミュレーションしたものではない）。どちらにも計上していない費用がある（Kobayashi：FSC → 通常パッチの変換、Litinski：状態注入、両方：古典処理の待ち時間）。比較は「各方式の符号の中で 1 個の $T$ 状態を作る費用」であり、共通の出力パッチへ受け渡す費用までは含まない。

**雑音のシナリオ**：A＝アイドル雑音あり（$p_{\rm idle}=p$）、B＝アイドル雑音なし（Takada らと同じ）。ゲート・初期化・測定の誤りはどちらも $p$。

**結果の位置づけ**：Kobayashi 方式の数値は**本研究で構成した回路・測定スケジュール・復号器**（[KZH] が引用する JIT decoder [Dav] は参照できなかった）での値であり、Kobayashi 方式一般の性能ではない。手法・仮定の詳細は `docs/methods.md`。

In [ ]:
# ==== 環境設定（Google Colab / ローカル共通）====
# このノートブックは単独で実行できる（共通コードは下に埋め込まれており、他のファイルを読み込まない）。
# 出力（results/, figures/）はこのランタイムの作業ディレクトリに保存される。Colab ではランタイム終了で
# 消えるので、残したい場合は Google Drive を使う：
#   from google.colab import drive; drive.mount('/content/drive')
#   os.makedirs('/content/drive/MyDrive/D4_vs_MSD', exist_ok=True); os.chdir('/content/drive/MyDrive/D4_vs_MSD')
import os, sys, subprocess, json
IN_COLAB = 'google.colab' in sys.modules
RESULTS, FIGURES = 'results', 'figures'
os.makedirs(RESULTS, exist_ok=True); os.makedirs(FIGURES, exist_ok=True)
SEED = 20261024
print('cwd =', os.getcwd(), '| Colab =', IN_COLAB)

In [ ]:
# ==== 外部ライブラリ（Colab に無いものだけをインストール）====
import importlib.util
for _mod, _pkg in [('stim', 'stim'), ('pymatching', 'pymatching'), ('mpmath', 'mpmath'), ('scipy', 'scipy'),
                   ('sympy', 'sympy'), ('pandas', 'pandas'), ('matplotlib', 'matplotlib')]:
    if importlib.util.find_spec(_mod) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _pkg], check=True)
import numpy as np, pandas as pd, scipy, stim, pymatching, mpmath
print('numpy', np.__version__, '| scipy', scipy.__version__, '| stim', stim.__version__,
      '| pymatching', pymatching.__version__, '| mpmath', mpmath.__version__)

In [ ]:
# ==== 図のスタイル（明るい背景・細い線・識別は色＋線種＋マーカー）====
import matplotlib
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.facecolor': '#fcfcfb', 'axes.facecolor': '#fcfcfb', 'savefig.facecolor': '#fcfcfb',
                     'axes.edgecolor': '#52514e', 'axes.labelcolor': '#0b0b0b', 'text.color': '#0b0b0b',
                     'xtick.color': '#52514e', 'ytick.color': '#52514e', 'axes.grid': True,
                     'grid.color': '#e4e3df', 'grid.linewidth': 0.8, 'lines.linewidth': 2,
                     'lines.markersize': 7, 'axes.spines.top': False, 'axes.spines.right': False,
                     'font.size': 10, 'legend.frameon': False})
C_KOB, C_LIT, C_AUX = '#2a78d6', '#eb6834', '#1baf7a'     # categorical slots 1,2,3 (validated palette)
C_R, C_G, C_B = '#e34948', '#008300', '#2a78d6'            # colours of the r, g, b qubit species

## 共通ライブラリ（埋め込み・関数定義のみ）
Google Colab ではノートブックごとに別のランタイムで実行されるため、他のファイルを読み込まずに済むよう、共通コードをこのノートブックに埋め込んでいる（各ノートの共通コードは同一のソースから自動生成されており、内容は一致する）。
**このセクションは上から順に実行するだけでよく、編集する必要はない**（Colab では見出しの左の三角で折りたためる）。

### 共通ライブラリ（F2 線形代数・ゲート／ガジェット・スケジューラ・Clifford+T 分解・計数ルール R1–R3）

In [ ]:
"""common_lib: F2 linear algebra, gate/gadget model, scheduler, decompositions, cost model rules."""
import itertools, math
import numpy as np
from collections import Counter, defaultdict

In [ ]:
# ---------------------------------------------------------------- F2 algebra
def f2_rref(M):
    """Row-reduce a uint8 matrix over F2. Returns (R, pivot_cols)."""
    M = (np.array(M, dtype=np.uint8) & 1).copy()
    rows, cols = M.shape
    piv = []
    r = 0
    for c in range(cols):
        if r >= rows:
            break
        nz = np.nonzero(M[r:, c])[0]
        if len(nz) == 0:
            continue
        p = r + nz[0]
        if p != r:
            M[[r, p]] = M[[p, r]]
        others = np.nonzero(M[:, c])[0]
        others = others[others != r]
        if len(others):
            M[others] ^= M[r]
        piv.append(c)
        r += 1
    return M[:r], piv


def f2_rank(M):
    if len(M) == 0:
        return 0
    return len(f2_rref(M)[1])


def f2_nullspace(M, n=None):
    """Basis (rows) of {x : M x = 0} over F2."""
    M = np.array(M, dtype=np.uint8)
    if M.size == 0:
        return np.eye(n, dtype=np.uint8)
    n = M.shape[1]
    Rm, piv = f2_rref(M)
    free = [c for c in range(n) if c not in set(piv)]
    basis = []
    for f in free:
        v = np.zeros(n, dtype=np.uint8)
        v[f] = 1
        for i, p in enumerate(piv):
            if Rm[i, f]:
                v[p] = 1
        basis.append(v)
    return np.array(basis, dtype=np.uint8).reshape(len(basis), n)


def f2_in_rowspace(Rref, piv, v):
    """Is v in the row space of the RREF matrix Rref (pivots piv)?"""
    v = np.array(v, dtype=np.uint8) & 1
    w = v.copy()
    for i, p in enumerate(piv):
        if w[p]:
            w ^= Rref[i]
    return not w.any()


def f2_solve(A, b):
    """Solve x A = b (x row vector, A rows = generators). Returns x or None."""
    A = np.array(A, dtype=np.uint8)
    m, n = A.shape
    aug = np.concatenate([A, np.eye(m, dtype=np.uint8)], axis=1)
    Rm, piv = f2_rref(aug)
    w = np.concatenate([np.array(b, dtype=np.uint8) & 1, np.zeros(m, dtype=np.uint8)])
    for i, p in enumerate(piv):
        if p >= n:
            break
        if w[p]:
            w ^= Rm[i]
    if w[:n].any():
        return None
    return w[n:]




import itertools
from collections import Counter, defaultdict

ARITY = {'INIT_X': 1, 'INIT_Z': 1, 'MEAS_X': 1, 'MEAS_Z': 1, 'H': 1, 'X': 1, 'Z': 1, 'S': 1, 'SDG': 1,
         'T': 1, 'TDG': 1, 'CNOT': 2, 'CZ': 2, 'CS': 2, 'CSDG': 2, 'CCZ': 3}
NONCLIFFORD = {'T', 'TDG', 'CS', 'CSDG', 'CCZ'}

In [ ]:
# ------------------------------------------------------------------ minimum-weight logical (MILP)
def min_weight_odd(Hc, lvec, cols):
    """min |x| over x in F2^cols with Hc x = 0 and lvec . x = 1 (exact MILP, HiGHS)."""
    from scipy.optimize import milp, LinearConstraint, Bounds
    H = np.array(Hc, dtype=float)[:, cols]
    H = H[np.any(H, axis=1)]
    l = np.array(lvec, dtype=float)[cols]
    m, n = H.shape
    # variables: x (n, binary), y (m, integer), z (1, integer)
    nv = n + m + 1
    c = np.concatenate([np.ones(n), np.zeros(m + 1)])
    A1 = np.concatenate([H, -2 * np.eye(m), np.zeros((m, 1))], axis=1)
    A2 = np.concatenate([l[None, :], np.zeros((1, m)), -2 * np.ones((1, 1))], axis=1)
    cons = [LinearConstraint(A1, 0, 0), LinearConstraint(A2, 1, 1)]
    integrality = np.ones(nv)
    ub = np.concatenate([np.ones(n), np.full(m + 1, np.inf)])
    res = milp(c, constraints=cons, integrality=integrality, bounds=Bounds(np.zeros(nv), ub))
    assert res.status == 0, res.message
    x = np.round(res.x[:n]).astype(np.uint8)
    out = np.zeros(len(lvec), dtype=np.uint8)
    out[cols] = x
    return int(x.sum()), out

In [ ]:
# ------------------------------------------------------------------ gadgets
def pauli_check_gadget(anc, xq=(), zq=()):
    """Measure X(xq) Z(zq) with ancilla anc (standard CNOT circuit)."""
    g = [('INIT_X', anc)]
    g += [('CNOT', anc, q) for q in sorted(xq)]           # ancilla controls -> X on data
    g += [('CZ', anc, q) for q in sorted(zq)]             # controlled Z on data
    g += [('MEAS_X', anc)]
    return g


def z_check_gadget(anc, zq):
    g = [('INIT_Z', anc)] + [('CNOT', q, anc) for q in sorted(zq)] + [('MEAS_Z', anc)]
    return g


def clifford_x_gadget(anc, parts):
    """Measure a product of X-type Clifford operators S = S_1 S_2 ... with
    S_k = X(s_k) prod CZ(pairs_k): ancilla |+>, controlled-S (= CNOTs from anc and
    CCZ(anc,i,j) for each CZ pair; rightmost factor first), measure X.
    parts: list of (xqubits, czpairs) in operator order S_1 S_2 ..."""
    g = [('INIT_X', anc)]
    for xq, pairs in reversed(parts):
        g += [('CCZ', anc, i, j) for (i, j) in pairs]      # CZ part acts first within S_k
        g += [('CNOT', anc, q) for q in sorted(xq)]
    g += [('MEAS_X', anc)]
    return g

In [ ]:
# ------------------------------------------------------------------ scheduling
def asap_layers(gates):
    """Greedy ASAP layering of a gate list respecting qubit order (no reordering of gates that
    share a qubit).  Returns list of layers (lists of gates)."""
    ready = defaultdict(int)
    layers = []
    for gt in gates:
        qs = gt[1:]
        t = max(ready[q] for q in qs)
        while len(layers) <= t:
            layers.append([])
        layers[t].append(gt)
        for q in qs:
            ready[q] = t + 1
    return layers


def gadget_support(gad):
    s = set()
    for gt in gad:
        s |= set(gt[1:])
    return s


def disjoint_groups(gadgets):
    """Greedy colouring: partition gadgets so that gadgets in a group touch disjoint qubits.
    Measurements within a group are then exactly simultaneous and the sequential-group
    schedule is valid for commuting (on the relevant subspace) generators."""
    groups = []          # list of (used_qubits, [gadgets])
    order = sorted(range(len(gadgets)), key=lambda i: -len(gadget_support(gadgets[i])))
    for i in order:
        sup = gadget_support(gadgets[i])
        for used, members in groups:
            if not (used & sup):
                used |= sup
                members.append(gadgets[i])
                break
        else:
            groups.append((set(sup), [gadgets[i]]))
    return [m for _, m in groups]


def layer_duration(layer, dur):
    return max(dur[g[0]] for g in layer)


def circuit_time(layers, dur):
    return sum(layer_duration(l, dur) for l in layers)


def round_schedule(gadgets, dur):
    """Upper-bound schedule of one round: disjoint groups executed sequentially, each group
    ASAP-layered.  Returns dict with depth (layers), time (weighted by dur), groups."""
    groups = disjoint_groups(gadgets)
    depth = 0
    time = 0
    for grp in groups:
        gl = asap_layers([gt for gad in grp for gt in gad])
        depth += len(gl)
        time += circuit_time(gl, dur)
    return {'groups': len(groups), 'depth_ub': depth, 'time_ub': time}


def round_lower_bound(gadgets, dur):
    """Lower bound on the duration of one round: every qubit executes its gates sequentially,
    so time >= max_q sum of durations of gates acting on q."""
    busy = defaultdict(float)
    cnt = defaultdict(int)
    for gad in gadgets:
        for gt in gad:
            for q in gt[1:]:
                busy[q] += dur[gt[0]]
                cnt[q] += 1
    return {'depth_lb': max(cnt.values()), 'time_lb': max(busy.values())}


def gate_counts(gates):
    c = Counter(g[0] for g in gates)
    return dict(c)

In [ ]:
# ------------------------------------------------------------------ decompositions
# Explicit Clifford+T decompositions (verified numerically in the validation notebook).
def ccz_to_clifford_t(a, b, c):
    """CCZ with 7 T/T^dag and 6 CNOT (standard Toffoli decomposition, Nielsen & Chuang Fig. 4.9,
    with the target Hadamards removed)."""
    return [('CNOT', b, c), ('TDG', c), ('CNOT', a, c), ('T', c), ('CNOT', b, c), ('TDG', c),
            ('CNOT', a, c), ('T', b), ('T', c), ('CNOT', a, b), ('T', a), ('TDG', b), ('CNOT', a, b)]


def cs_to_clifford_t(a, b):
    """CS = diag(1,1,1,i) = T_a T_b CNOT(a,b) T^dag_b CNOT(a,b)."""
    return [('T', a), ('T', b), ('CNOT', a, b), ('TDG', b), ('CNOT', a, b)]


def csdg_to_clifford_t(a, b):
    return [('TDG', a), ('TDG', b), ('CNOT', a, b), ('T', b), ('CNOT', a, b)]


def decompose(gates):
    out = []
    for g in gates:
        if g[0] == 'CCZ':
            out += ccz_to_clifford_t(*g[1:])
        elif g[0] == 'CS':
            out += cs_to_clifford_t(*g[1:])
        elif g[0] == 'CSDG':
            out += csdg_to_clifford_t(*g[1:])
        else:
            out.append(g)
    return out


DUR_UNIT = {k: 1 for k in ARITY}            # every primitive = 1 time unit (ASSUMPTION, parametrised)

In [ ]:
# ------------------------------------------------- commuting-class scheduler
DIAG_GATES = {'CZ', 'CCZ', 'CS', 'CSDG', 'T', 'TDG', 'S', 'SDG', 'Z'}


def gate_roles(gt):
    """Per-qubit role of an entangling/1q unitary: 'Z' (diagonal in Z basis), 'X' (X-type)."""
    name = gt[0]
    if name in DIAG_GATES:
        return {q: 'Z' for q in gt[1:]}
    if name == 'CNOT':
        return {gt[1]: 'Z', gt[2]: 'X'}
    if name == 'X':
        return {gt[1]: 'X'}
    return {q: '?' for q in gt[1:]}      # H, INIT, MEAS: never shared between gadgets


def gadget_roles(gad):
    roles = {}
    for gt in gad:
        for q, r in gate_roles(gt).items():
            if roles.get(q, r) != r:
                roles[q] = '?'
            else:
                roles[q] = r
    return roles


def commuting_classes(gadgets, n_orders=6, seed=0):
    """Best (fewest classes) of several greedy colourings (different gadget orders)."""
    import random
    best = None
    orders = [list(range(len(gadgets)))]
    orders.append(sorted(orders[0], key=lambda i: tuple(sorted(set(gadget_roles(gadgets[i]).values())))))
    rnd = random.Random(seed)
    for _ in range(n_orders):
        o = orders[0][:]
        rnd.shuffle(o)
        orders.append(o)
    for o in orders:
        cl = _commuting_classes_greedy([gadgets[i] for i in o])
        if best is None or len(cl) < len(best):
            best = cl
    return best


def _commuting_classes_greedy(gadgets):
    """Greedy colouring of the gadget conflict graph.  Two gadgets are compatible if on every
    shared qubit both act with the same role ('Z' or 'X'); then all their gates commute and the
    joint circuit is equivalent to measuring them one after another (sufficient condition)."""
    roles = [gadget_roles(g) for g in gadgets]
    classes = []     # list of (role_map, [idx])
    for i, rm in enumerate(roles):
        placed = False
        for cmap, members in classes:
            ok = all(cmap.get(q, r) == r and r != '?' for q, r in rm.items() if q in cmap)
            if ok:
                for q, r in rm.items():
                    cmap[q] = r
                members.append(i)
                placed = True
                break
        if not placed:
            classes.append((dict(rm), [i]))
    return [[gadgets[i] for i in m] for _, m in classes]


def class_layers(gadgets):
    """Layering of one commuting class: init layer, greedy layering of the (mutually commuting)
    entangling gates, measurement layer."""
    inits = [g for gad in gadgets for g in gad if g[0].startswith('INIT')]
    meas = [g for gad in gadgets for g in gad if g[0].startswith('MEAS')]
    body = [g for gad in gadgets for g in gad if not (g[0].startswith('INIT') or g[0].startswith('MEAS'))]
    # greedy: most-constrained gates first
    body = sorted(body, key=lambda g: -len(g))
    layers = []
    busy = []
    for g in body:
        qs = set(g[1:])
        for li, b in enumerate(busy):
            if not (b & qs) and layers[li][0][0] == g[0]:     # keep one gate type per layer
                b |= qs
                layers[li].append(g)
                break
        else:
            busy.append(set(qs))
            layers.append([g])
    return ([inits] if inits else []) + layers + ([meas] if meas else [])


def round_schedule_classes(gadgets, dur):
    classes = commuting_classes(gadgets)
    depth = 0
    time = 0
    for c in classes:
        ls = class_layers(c)
        depth += len(ls)
        time += circuit_time(ls, dur)
    return {'classes': len(classes), 'depth_ub': depth, 'time_ub': time}


import math
from collections import Counter

SOURCE_LIT = 'literature'
SOURCE_COUNT = 'counted (explicit circuit)'
SOURCE_ASSUME = 'assumption (parameter)'
SOURCE_UNRES = 'unresolved (excluded / illustrative)'

DEFAULT_PARAMS = dict(
    kappa0=1.0,      # FSC |+_L> preparation rounds = ceil(kappa0*d)   [ASSUMPTION; Kobayashi: not specified]
    kappa_in=1.0,    # switch-in rounds            = ceil(kappa_in*d)  [ASSUMPTION; Kobayashi: O(d)]
    kappa_out=1.0,   # FSC EC rounds after return  = ceil(kappa_out*d) [ASSUMPTION; Kobayashi: O(d)]
    tau_rounds=1.0,  # Litinski: 1 time step = tau_rounds*d code cycles [LITERATURE: d cycles]
    t_inj_FT=6,      # Litinski App. A (Li's injection): 3 + 3 code cycles, d-independent [LITERATURE]
    t_inj_NL=1,      # noiseless-minimal injection: 1 round [ASSUMPTION]
)


def rounds_for(model, kappa, d):
    if model == 'NL_min':
        return 1
    return int(math.ceil(kappa * d))

In [ ]:
# ----------------------------------------------------------------- summaries
def summarize(rows):
    tot = Counter()
    for r in rows:
        for k, v in r['gates'].items():
            tot[k] += v
    peak = max(r['qubits'] for r in rows)
    rounds = sum(r['rounds'] for r in rows)
    tl = sum(r['time_lb'] for r in rows if r['time_lb'] is not None)
    tu = sum(r['time_ub'] for r in rows if r['time_ub'] is not None)
    v_round = sum(r['qubits'] * r['rounds'] for r in rows)
    v_tl = sum(r['qubits'] * r['time_lb'] for r in rows if r['time_lb'] is not None)
    v_tu = sum(r['qubits'] * r['time_ub'] for r in rows if r['time_ub'] is not None)
    nonclif = {k: v for k, v in tot.items() if k in NONCLIFFORD}
    tcount = 7 * tot.get('CCZ', 0) + 3 * (tot.get('CS', 0) + tot.get('CSDG', 0)) + tot.get('T', 0) + tot.get('TDG', 0)
    return dict(peak_qubits=peak, rounds=rounds, time_lb=tl, time_ub=tu, V_qubit_round=v_round,
                V_qubit_time_lb=v_tl, V_qubit_time_ub=v_tu, V_peak_x_rounds_UB=peak * rounds,
                gates=dict(tot), nonclifford=nonclif, T_count_if_decomposed=tcount)


def uncounted_costs(protocol, comparison):
    """Costs that are known to exist but are not included in the totals (listed, never summed as 0)."""
    out = [('classical decoding / control latency', 'both comparisons, both protocols')]
    if protocol == 'Kobayashi_D4':
        out.append(('JIT-decoder classical processing', 'not modelled'))
        if comparison == 'B':
            out.append(('FSC -> standard rotated surface-code patch conversion', 'UNRESOLVED: no protocol constructed'))
    else:
        out.append(('twist-based Z_m (x) Y_0 seam checks between |m> and |0>', 'O(d) measurement qubits, not constructed'))
    return out

In [ ]:
# ------------------------------------------------------------------ table helpers
def jsonify_columns(df, cols=('gates', 'nonclifford')):
    """Store dict-valued columns as JSON strings (same format as the saved CSV files)."""
    import json
    for col in cols:
        if col in df.columns:
            df[col] = df[col].apply(lambda g: g if isinstance(g, str) else json.dumps({k: float(v) for k, v in g.items()}))
    return df


def identify_polynomial(values, npts=4):
    """Interpolate a polynomial in d through the first npts points (degree <= npts-1) and report
    whether it reproduces ALL given points exactly.  values: dict d -> integer.  Returns (expr, ok)."""
    import sympy
    x = sympy.Symbol('d')
    ds = sorted(values)
    if len(ds) <= npts:
        return None, False
    p = sympy.interpolate([(dd, int(round(values[dd]))) for dd in ds[:npts]], x)
    ok = all(sympy.nsimplify(p.subs(x, dd) - int(round(values[dd]))) == 0 for dd in ds)
    return sympy.expand(p), ok

### D4 ライブラリ（境界付き D4 符号・transversal $U$・code switching・疎状態シミュレータ・資源モデル）

In [ ]:
"""d4_lib: D4 Clifford stabilizer code (Kobayashi-Zhu-Hsin) on the bounded square lattice, transversal U, code switching, resource model."""
import itertools, cmath
import numpy as np
from collections import Counter

R, G, B = 0, 1, 2
CNAME = "rgb"

In [ ]:
# ------------------------------------------------- phase-shift operators
class PSOp:
    """Operator  O |x> = (-1)^{f(x)} |x + s>  on n qubits (computational basis x in F2^n).

    f(x) = const + sum_{i in lin} x_i + sum_{(i,j) in quad} x_i x_j   (mod 2), i<j.
    X-type Clifford stabilizers of the D4 code, Pauli X/Z strings and products
    thereof all have this form.  Composition rule (A o B)|x> = A B |x>:
        s = s_A + s_B,   f(x) = f_B(x) + f_A(x + s_B).
    """
    __slots__ = ("s", "lin", "quad", "const")

    def __init__(self, s=frozenset(), lin=frozenset(), quad=frozenset(), const=0):
        self.s = frozenset(s)
        self.lin = frozenset(lin)
        self.quad = frozenset(tuple(sorted(p)) for p in quad)
        self.const = const & 1

    def shifted_phase(self, t):
        """Return (lin, quad, const) of x -> f(x + t)."""
        lin = set(self.lin)
        const = self.const
        for i in self.lin:
            if i in t:
                const ^= 1
        for (i, j) in self.quad:
            ti, tj = i in t, j in t
            if ti and tj:
                const ^= 1
            if tj:
                lin ^= {i}
            if ti:
                lin ^= {j}
        return frozenset(lin), self.quad, const

    def __matmul__(self, other):
        lA, qA, cA = self.shifted_phase(other.s)
        return PSOp(self.s ^ other.s, lA ^ other.lin, qA ^ other.quad, cA ^ other.const)

    def phase_at(self, x):
        """f(x) for x given as a set/frozenset of 1-positions."""
        v = self.const
        for i in self.lin:
            if i in x:
                v ^= 1
        for (i, j) in self.quad:
            if i in x and j in x:
                v ^= 1
        return v

    def apply(self, x):
        """O|x> = sign |x'> ; returns (x', sign_bit)."""
        return frozenset(x) ^ self.s, self.phase_at(x)

    def support(self):
        sup = set(self.s) | set(self.lin)
        for p in self.quad:
            sup |= set(p)
        return sup


def commutator_affine(A, B):
    """A B = (-1)^{g(x)} B A on |x>.  Returns g as (lin, const) -- quadratic terms cancel."""
    AB = A @ B
    BA = B @ A
    assert AB.s == BA.s
    q = AB.quad ^ BA.quad
    assert not q, "commutator has quadratic part (should not happen)"
    return AB.lin ^ BA.lin, AB.const ^ BA.const

In [ ]:
# ------------------------------------------------------------- the lattice
class D4Lattice:
    def __init__(self, L):
        assert L >= 1
        self.L = L
        self.edges = []          # (kind, x, y), kind 'h' or 'v'
        self.eidx = {}
        for y in range(L + 1):
            for x in range(L):
                self._add(('h', x, y))
        for x in range(L + 1):
            for y in range(L):
                self._add(('v', x, y))
        self.E = len(self.edges)
        self.n = 3 * self.E
        self.vertices = [(x, y) for y in range(L + 1) for x in range(L + 1)]
        self.plaquettes = [(x, y) for y in range(L) for x in range(L)]

    def _add(self, e):
        self.eidx[e] = len(self.edges)
        self.edges.append(e)

    def q(self, e, c):
        if not isinstance(e, int):
            e = self.eidx[e]
        return 3 * e + c

    # --- geometry --------------------------------------------------------
    def endpoints(self, e):
        k, x, y = self.edges[e] if isinstance(e, int) else e
        return ((x, y), (x + 1, y)) if k == 'h' else ((x, y), (x, y + 1))

    def incident(self, v):
        x, y = v
        out = []
        for e in [('h', x, y), ('h', x - 1, y), ('v', x, y), ('v', x, y - 1)]:
            if e in self.eidx:
                out.append(self.eidx[e])
        return out

    def plaq_edges(self, p):
        """(e01, e13, e02, e23) of plaquette p=(x,y)."""
        x, y = p
        ei = self.eidx
        return ei[('h', x, y)], ei[('v', x + 1, y)], ei[('v', x, y)], ei[('h', x, y + 1)]

    def plaq_boundary(self, p):
        return list(self.plaq_edges(p))

    def has_plaq(self, x, y):
        return 0 <= x < self.L and 0 <= y < self.L

    def boundary_of_edge(self, e):
        k, x, y = self.edges[e]
        L = self.L
        if k == 'v' and x == 0:
            return 'Lr'
        if k == 'h' and y == L:
            return 'Lb'
        if k == 'h' and y == 0:
            return 'Lrb'
        if k == 'v' and x == L:
            return 'Lrb'
        return None

    def boundary_edges(self, name):
        return [e for e in range(self.E) if self.boundary_of_edge(e) == name]

    # --- Z-type checks (rows of H_Z) --------------------------------------
    def z_checks(self, colours=(R, G, B)):
        """List of (label, set_of_qubits) Z-type stabilizer generators.
        Plaquettes for every colour + single-edge boundary conditions (Fig. 4 / eq. (7))."""
        rows = []
        for p in self.plaquettes:
            for c in colours:
                rows.append((('plaq', p, CNAME[c]), {self.q(e, c) for e in self.plaq_boundary(p)}))
        for e in range(self.E):
            bd = self.boundary_of_edge(e)
            if bd == 'Lr' and R in colours:
                rows.append((('bdZ', e, 'r'), {self.q(e, R)}))
            elif bd == 'Lb' and B in colours:
                rows.append((('bdZ', e, 'b'), {self.q(e, B)}))
            elif bd == 'Lrb':
                if R in colours and B in colours:
                    rows.append((('bdZ', e, 'rb'), {self.q(e, R), self.q(e, B)}))
                if G in colours:
                    rows.append((('bdZ', e, 'g'), {self.q(e, G)}))
        return rows

    # --- X-type Clifford stabilizers ----------------------------------------
    def cz_pairs(self, v, c):
        """CZ pairs (qubit, qubit) of the dressing of S^c_{X;v} (SM eq. for S_X, cup products
        evaluated with the hypercube formula; plaquettes outside the lattice are dropped)."""
        x, y = v
        pairs = []
        if c == R:      # int v u e' u e'' : v = vertex 0 of p(x,y);   CZ^{g,b}
            if self.has_plaq(x, y):
                e01, e13, e02, e23 = self.plaq_edges((x, y))
                pairs += [(self.q(e01, G), self.q(e13, B)), (self.q(e02, G), self.q(e23, B))]
        elif c == G:    # int e'' u v u e' : v = vertex 1 of p(x-1,y), vertex 2 of p(x,y-1); CZ^{r,b}
            if self.has_plaq(x - 1, y):
                e01, e13, e02, e23 = self.plaq_edges((x - 1, y))
                pairs.append((self.q(e01, R), self.q(e13, B)))
            if self.has_plaq(x, y - 1):
                e01, e13, e02, e23 = self.plaq_edges((x, y - 1))
                pairs.append((self.q(e02, R), self.q(e23, B)))
        else:           # int e' u e'' u v : v = vertex 3 of p(x-1,y-1); CZ^{r,g}
            if self.has_plaq(x - 1, y - 1):
                e01, e13, e02, e23 = self.plaq_edges((x - 1, y - 1))
                pairs += [(self.q(e01, R), self.q(e13, G)), (self.q(e02, R), self.q(e23, G))]
        return pairs

    def SX(self, v, c, dressed=True):
        s = {self.q(e, c) for e in self.incident(v)}
        quad = self.cz_pairs(v, c) if dressed else []
        # CZ applied first, then X (S = (prod X)(prod CZ)); X and CZ act on disjoint colours.
        return PSOp(s=s, quad=quad)

    def vertex_x_types(self, v, colours=(R, G, B)):
        """Allowed colour combinations at vertex v: the F2 span of colour sets whose
        X-part commutes with the boundary Z conditions.  Returns a list of colour tuples
        (a basis), derived automatically from the boundary Z checks."""
        bdrows = [rows for (lab, rows) in self.z_checks(colours) if lab[0] == 'bdZ']
        inc = self.incident(v)
        cand = []
        for k in range(1, len(colours) + 1):
            for combo in itertools.combinations(colours, k):
                s = {self.q(e, c) for e in inc for c in combo}
                if all(len(s & rw) % 2 == 0 for rw in bdrows):
                    cand.append(combo)
        # reduce to an F2 basis over colour space, preferring single colours
        basis, vecs = [], []
        for combo in sorted(cand, key=len):
            vec = np.zeros(3, dtype=np.uint8)
            vec[list(combo)] = 1
            if f2_rank(np.array(vecs + [vec])) > len(vecs):
                vecs.append(vec)
                basis.append(combo)
        return basis

    def x_generators(self, colours=(R, G, B), dressed=True):
        """List of (label, PSOp).  At a vertex where only a product of colours is allowed
        (L_rb), the generator is the ordered product S^r S^b (Fig. 4 shows S^r S^b there)."""
        gens = []
        for v in self.vertices:
            for combo in self.vertex_x_types(v, colours):
                op = PSOp()
                for c in combo:
                    op = op @ self.SX(v, c, dressed)
                gens.append((('X', v, ''.join(CNAME[c] for c in combo)), op))
        return gens

    def vec(self, qubits):
        v = np.zeros(self.n, dtype=np.uint8)
        v[list(qubits)] = 1
        return v

In [ ]:
# ---------------------------------------------------------- code data
class CodeData:
    """Linear-algebra data of a (D4 or folded-surface) code on a lattice."""

    def __init__(self, lat, colours=(R, G, B), dressed=True):
        self.lat = lat
        self.colours = tuple(colours)
        self.zlabels, zr = zip(*lat.z_checks(colours))
        self.HZ = np.array([lat.vec(r) for r in zr], dtype=np.uint8)
        self.xgens = lat.x_generators(colours, dressed)
        self.S = np.array([lat.vec(op.s) for _, op in self.xgens], dtype=np.uint8)
        self.cols = sorted({lat.q(e, c) for e in range(lat.E) for c in colours})
        self.HZr, self.HZpiv = f2_rref(self.HZ)
        self.Sr, self.Spiv = f2_rref(self.S)
        # logical representatives
        CZ = f2_nullspace(self.HZ[:, self.cols])
        self.t_log = None
        for row in CZ:
            t = np.zeros(lat.n, dtype=np.uint8)
            t[self.cols] = row
            if not f2_in_rowspace(self.Sr, self.Spiv, t):
                self.t_log = t
                break
        KS = f2_nullspace(self.S[:, self.cols])
        self.z_log = None
        for row in KS:
            u = np.zeros(lat.n, dtype=np.uint8)
            u[self.cols] = row
            if not f2_in_rowspace(self.HZr, self.HZpiv, u):
                self.z_log = u
                break
        self.k = len(self.cols) - f2_rank(self.HZ[:, self.cols]) - f2_rank(self.S[:, self.cols])


def distances(cd):
    """(d_Z, d_X_shift) : min weight of a logical Z-type Pauli, and min weight of an
    X-shift lying in C_Z but outside the stabilizer shift span."""
    dz, uz = min_weight_odd(cd.S, cd.t_log, cd.cols)
    dx, tx = min_weight_odd(cd.HZ, cd.z_log, cd.cols)
    return dz, dx, uz, tx

In [ ]:
# ------------------------------------------- transversal automorphism U = W V
class DiagPhase:
    """Diagonal unitary |x> -> omega^{w(x)} |x>, omega = exp(i pi/4), w integer mod 8.
    w(x) = sum_k c_k * prod_{i in mono_k} x_i  (monomials of degree <= 2)."""

    def __init__(self, terms=()):
        self.terms = list(terms)   # (coef mod 8, tuple_of_qubits)

    def value(self, x):
        v = 0
        for c, mono in self.terms:
            if all(i in x for i in mono):
                v += c
        return v % 8

    def support(self):
        s = set()
        for _, m in self.terms:
            s |= set(m)
        return s

    def terms_touching(self, qs):
        return DiagPhase([(c, m) for c, m in self.terms if set(m) & qs])


def W_operator(lat, cs_signs=(+1, -1), boundary_T=-1):
    """W of eq. (8):  prod_p CS^{r,b}_{e01,e13} (CS^{r,b}_{e02,e23})^dag  x  prod_{e in bdry_rb} (T^r_e)^dag.
    CS = diag(1,1,1,i) -> omega^{2 x y};  T^dag -> omega^{-x}."""
    terms = []
    for p in lat.plaquettes:
        e01, e13, e02, e23 = lat.plaq_edges(p)
        terms.append(((2 * cs_signs[0]) % 8, (lat.q(e01, R), lat.q(e13, B))))
        terms.append(((2 * cs_signs[1]) % 8, (lat.q(e02, R), lat.q(e23, B))))
    if boundary_T:
        for e in lat.boundary_edges('Lrb'):
            terms.append((boundary_T % 8, (lat.q(e, R),)))
    return DiagPhase(terms)


def V_map(lat, x):
    """V = prod_e CNOT^{(r,g)}_e CNOT^{(b,g)}_e :  x_g <- x_g + x_r + x_b  (set representation)."""
    x = set(x)
    out = set(x)
    for e in range(lat.E):
        qr, qg, qb = lat.q(e, R), lat.q(e, G), lat.q(e, B)
        if (qr in x) ^ (qb in x):
            out ^= {qg}
    return frozenset(out)


def V_support_map(lat, qs):
    """Qubits whose values determine (V x) restricted to qs."""
    out = set()
    for q in qs:
        e, c = divmod(q, 3)
        out.add(q)
        if c == G:
            out |= {3 * e + R, 3 * e + B}
    return out


def CZ_basis(cd):
    if not hasattr(cd, '_CZbasis'):
        nb = f2_nullspace(cd.HZ[:, cd.cols])
        full = np.zeros((nb.shape[0], cd.lat.n), dtype=np.uint8)
        full[:, cd.cols] = nb
        cd._CZbasis = full
    return cd._CZbasis


def project_CZ(cd, Q, functionals=()):
    """Exact image of C_Z under x -> (x|_Q, phi_1(x), ..., phi_k(x)) for F2-linear functionals
    phi_j (callables on a set of qubits, evaluated on basis vectors).  Returns a list of
    configurations (frozenset of Q-qubits) or, if functionals are given, (config, bits)."""
    Q = sorted(Q)
    Bs = CZ_basis(cd)
    P = Bs[:, Q]
    if functionals:
        extra = np.array([[phi(frozenset(np.nonzero(b)[0])) for phi in functionals] for b in Bs],
                         dtype=np.uint8).reshape(len(Bs), len(functionals))
        P = np.concatenate([P, extra], axis=1)
    Pr, piv = f2_rref(P)
    k = Pr.shape[0]
    assert k <= 24, f"local projection too large ({k})"
    out = []
    nq = len(Q)
    for bits in itertools.product((0, 1), repeat=k):
        v = np.zeros(P.shape[1], dtype=np.uint8)
        for b, row in zip(bits, Pr):
            if b:
                v ^= row
        cfg = frozenset(Q[i] for i in np.nonzero(v[:nq])[0])
        out.append((cfg, tuple(int(z) for z in v[nq:])) if functionals else cfg)
    return out


def check_conjugation(cd, Wop, gen_op, target_op):
    """Exhaustively check  U S U^dag == S'  on C_Z, where U = W V.
    Returns (ok, n_local_configs)."""
    lat = cd.lat
    As = V_map(lat, gen_op.s)
    if As != target_op.s:
        return False, 0
    # qubits on which the phase difference can depend
    Wt = Wop.terms_touching(set(As))
    Q = set(Wt.support()) | V_support_map(lat, gen_op.support()) | target_op.support() | set(As)
    configs = project_CZ(cd, Q)
    for x in configs:
        # omega^{ w(x+As) - w(x) + 4 f(A x) - 4 f'(x) }   (only W-terms touching As contribute)
        d = Wt.value(x ^ As) - Wt.value(x) + 4 * gen_op.phase_at(V_map(lat, x)) - 4 * target_op.phase_at(x)
        if d % 8:
            return False, len(configs)
    return True, len(configs)


def stabilizer_path_phase(cd, x, target):
    """Given x and target = x + (sum of generator shifts), apply generators to |x> and
    return the accumulated sign bit, i.e. psi(target) = (-1)^bit psi(x) for code states."""
    shifts = cd.S
    delta = cd.lat.vec(set(x) ^ set(target))
    coeff = f2_solve(shifts, delta)
    if coeff is None:
        return None
    bit = 0
    cur = frozenset(x)
    for i in np.nonzero(coeff)[0]:
        op = cd.xgens[i][1]
        cur, b = op.apply(cur)
        bit ^= b
    assert cur == frozenset(target)
    return bit


def logical_phases(cd, Wop):
    """Eigenphases (in units of pi/4) of U on |0_L>, |1_L>, assuming U preserves the code space
    (checked separately).  Uses x0 = 0 and x1 = t_log."""
    lat = cd.lat
    res = []
    for x in [frozenset(), frozenset(np.nonzero(cd.t_log)[0])]:
        Ax = V_map(lat, x)
        bit = stabilizer_path_phase(cd, x, Ax)
        assert bit is not None, "V does not map the coset to itself"
        res.append((Wop.value(Ax) + 4 * bit) % 8)
    return res

In [ ]:
# ------------------------------------------- per-configuration protocol phase
def lrb_vertices(lat):
    vs = set()
    for e in lat.boundary_edges('Lrb'):
        vs |= set(lat.endpoints(e))
    return vs


def cprime(lat, a):
    """0-cochain c' with  delta c' = a_r + a_b  and c' = 0 on L_rb vertices (BFS).
    a: set of qubits (configuration of r,b qubits).  Returns dict vertex -> bit.
    Raises if a_r + a_b is not exact relative to L_rb."""
    from collections import deque
    val = {v: 0 for v in lrb_vertices(lat)}
    dq = deque(val)
    while dq:
        u = dq.popleft()
        for e in lat.incident(u):
            p, q = lat.endpoints(e)
            w = q if p == u else p
            bit = val[u] ^ (lat.q(e, R) in a) ^ (lat.q(e, B) in a)
            if w in val:
                if val[w] != bit:
                    raise ValueError("a_r + a_b not exact")
            else:
                val[w] = bit
                dq.append(w)
    return val


def g_phase_poly(lat, v):
    """CZ^{r,b} pairs of S^g_{X;v}  (f^g_v)."""
    return lat.cz_pairs(v, G)


def protocol_phase(lat, Wop, a, gverts):
    """Phase exponent (mod 8, units of pi/4) acquired by FSC configuration a through
    switch-in (S^g measurement) -> U -> Z^g measurement -> S^g correction -> Pauli frame,
    i.e.  F(a) = w(a) + 4 * sum_v c'_v(a) f^g_v(a)."""
    cp = cprime(lat, a)
    phi = 0
    for v in gverts:
        if cp[v]:
            for (i, j) in g_phase_poly(lat, v):
                if i in a and j in a:
                    phi ^= 1
    return (Wop.value(a) + 4 * phi) % 8


def check_protocol_invariance(lat, fsc, Wop, gverts, gen_op):
    """Exhaustive local proof that F(a + t) = F(a) (mod 8) for all a in C_FSC, t = shift of gen_op."""
    t = frozenset(gen_op.s)
    # c'(t) (linear in a)
    ct = cprime(lat, t)
    N2 = {v for v in gverts if ct[v]}
    N1 = {v for v in gverts if set(q for pr in g_phase_poly(lat, v) for q in pr) & t}
    Wt = Wop.terms_touching(set(t))
    Q = set(Wt.support()) | set(t)
    for v in N1 | N2:
        Q |= {q for pr in g_phase_poly(lat, v) for q in pr}
    # paths from a reference vertex w0 to each w in N1 (BFS on the lattice graph)
    w0 = min(N1 | N2 | {lat.endpoints(next(iter(t)) // 3)[0]})
    paths = {}
    from collections import deque
    prev = {w0: None}
    dq = deque([w0])
    while dq:
        u = dq.popleft()
        for e in lat.incident(u):
            p, q = lat.endpoints(e)
            w = q if p == u else p
            if w not in prev:
                prev[w] = (u, e)
                dq.append(w)
    for w in N1:
        es = []
        cur = w
        while prev[cur] is not None:
            u, e = prev[cur]
            es.append(e)
            cur = u
        paths[w] = es
        Q |= {lat.q(e, c) for e in es for c in (R, B)}
    beta_fn = lambda b: cprime(lat, b)[w0]          # c'_{w0}(a) is F2-linear in a
    configs = project_CZ(fsc, Q, functionals=[beta_fn])
    for a, (beta,) in configs:
        if True:
            cpa = {}
            for w in N1:
                bit = beta
                for e in paths[w]:
                    bit ^= (lat.q(e, R) in a) ^ (lat.q(e, B) in a)
                cpa[w] = bit
            at = a ^ t
            d = Wt.value(at) - Wt.value(a)
            phi = 0
            for w in N1:
                if cpa[w]:
                    for (i, j) in g_phase_poly(lat, w):
                        phi ^= ((i in at) and (j in at)) ^ ((i in a) and (j in a))
            for w in N2:
                for (i, j) in g_phase_poly(lat, w):
                    phi ^= ((i in at) and (j in at))
            if (d + 4 * phi) % 8:
                return False, len(configs)
    return True, len(configs)


OMEGA = cmath.exp(1j * cmath.pi / 4)


class SparseState:
    def __init__(self, amps):
        self.a = {k: v for k, v in amps.items() if abs(v) > 1e-14}

    def norm2(self):
        return sum(abs(v) ** 2 for v in self.a.values())

    def normalize(self):
        n = self.norm2() ** 0.5
        self.a = {k: v / n for k, v in self.a.items()}
        return self

    def apply_psop(self, op):
        out = {}
        for x, v in self.a.items():
            y, b = op.apply(x)
            out[y] = out.get(y, 0) + (-v if b else v)
        return SparseState(out)

    def apply_diag(self, D):
        return SparseState({x: v * OMEGA ** D.value(x) for x, v in self.a.items()})

    def apply_perm(self, f):
        out = {}
        for x, v in self.a.items():
            y = f(x)
            out[y] = out.get(y, 0) + v
        return SparseState(out)

    def project(self, op, sign):
        """(1 + sign*O)/2 |psi>  (unnormalised)."""
        Opsi = self.apply_psop(op).a
        out = {}
        for k in set(self.a) | set(Opsi):
            out[k] = 0.5 * (self.a.get(k, 0) + sign * Opsi.get(k, 0))
        return SparseState(out)

    def expect(self, op):
        Opsi = self.apply_psop(op).a
        return sum(np.conj(self.a.get(k, 0)) * v for k, v in Opsi.items())

    def measure_op(self, op, rng, forced=None):
        """Projective measurement of a Hermitian involution O; returns (outcome bit, state)."""
        p0 = self.project(op, +1)
        n0 = p0.norm2()
        if forced is None:
            bit = 0 if rng.random() < n0 else 1
        else:
            bit = forced
        st = p0 if bit == 0 else self.project(op, -1)
        prob = n0 if bit == 0 else 1 - n0
        return bit, st.normalize() if prob > 1e-12 else st, prob

    def measure_z(self, q, rng, forced=None):
        p1 = sum(abs(v) ** 2 for x, v in self.a.items() if q in x)
        bit = (1 if rng.random() < p1 else 0) if forced is None else forced
        st = SparseState({x: v for x, v in self.a.items() if (q in x) == bool(bit)})
        prob = p1 if bit else 1 - p1
        return bit, st.normalize() if prob > 1e-12 else st, prob

    def overlap(self, other):
        return sum(np.conj(v) * other.a.get(k, 0) for k, v in self.a.items())


def coset_state(cd, rep, phases=True):
    """Code state supported on rep + span(shifts):  psi(rep)=1, psi(x+s_v)=(-1)^{f_v(x)} psi(x)."""
    amps = {frozenset(rep): 1.0}
    # breadth-first application of generators
    frontier = [frozenset(rep)]
    while frontier:
        new = []
        for x in frontier:
            for _, op in cd.xgens:
                y, b = op.apply(x)
                if y not in amps:
                    amps[y] = amps[x] * (-1 if b else 1)
                    new.append(y)
                else:
                    if phases:
                        assert abs(amps[y] - amps[x] * (-1 if b else 1)) < 1e-9, "inconsistent code state"
        frontier = new
    return SparseState(amps).normalize()




def frozen_qubits(lat):
    """Qubits fixed to |0> by single-qubit boundary Z stabilizers (L_r: r, L_b: b, L_rb: g)."""
    fz = set()
    for lab, rows in lat.z_checks():
        if lab[0] == 'bdZ' and len(rows) == 1:
            fz |= rows
    return fz


class D4Protocol:
    def __init__(self, L, prune_frozen=True):
        self.L = L
        self.lat = lat = D4Lattice(L)
        self.frozen = frozen_qubits(lat) if prune_frozen else set()
        self.prune = prune_frozen
        self.d4 = CodeData(lat)
        self.fsc = CodeData(lat, (R, B), dressed=False)
        # ancilla allocation: one ancilla per stabilizer generator label (D4 superset)
        self.anc = {}
        nxt = lat.n
        labels = [lab for lab, _ in self.d4.xgens] + list(self.d4.zlabels)
        labels += [lab for lab, _ in self.fsc.xgens] + list(self.fsc.zlabels)
        for lab in labels:
            key = self._anc_key(lab)
            if key not in self.anc and not self._is_frozen_check(lab):
                self.anc[key] = nxt
                nxt += 1
        self.gverts = [lab[1] for lab, op in self.d4.xgens if lab[2] == 'g']

    @staticmethod
    def _anc_key(lab):
        return lab

    def _is_frozen_check(self, lab):
        return lab[0] == 'bdZ' and lab[2] in ('r', 'b', 'g') and self.prune

    def data_qubits(self, colours):
        return {self.lat.q(e, c) for e in range(self.lat.E) for c in colours} - self.frozen

    # ---------------------------------------------------------- gadgets
    def _keep_pair(self, i, j):
        return i not in self.frozen and j not in self.frozen

    def x_gadget(self, lab, dressed):
        _, v, cols = lab
        parts = []
        for ch in cols:
            c = CNAME.index(ch)
            xq = {self.lat.q(e, c) for e in self.lat.incident(v)} - self.frozen
            pairs = [p for p in self.lat.cz_pairs(v, c) if self._keep_pair(*p)] if dressed else []
            parts.append((xq, pairs))
        return clifford_x_gadget(self.anc[lab], parts)

    def z_gadget(self, lab, rows):
        return z_check_gadget(self.anc[lab], set(rows) - self.frozen)

    def round_gadgets(self, code, which='all'):
        """Gadgets of one stabilizer round.  code: 'D4' or 'FSC'.
        which: 'all' | 'X' | 'Z' | 'g' (D4 S^g only)."""
        cd = self.d4 if code == 'D4' else self.fsc
        dressed = code == 'D4'
        gads = []
        if which in ('all', 'X', 'g'):
            for lab, op in cd.xgens:
                if which == 'g' and lab[2] != 'g':
                    continue
                gads.append(self.x_gadget(lab, dressed))
        if which in ('all', 'Z'):
            for lab, (l2, rows) in zip(cd.zlabels, self.lat.z_checks(cd.colours)):
                if self._is_frozen_check(lab):
                    continue
                gads.append(self.z_gadget(lab, rows))
        return gads

    # ---------------------------------------------------------- transversal U
    def U_gates(self):
        lat, fz = self.lat, self.frozen
        gates = []
        for e in range(lat.E):
            qr, qg, qb = lat.q(e, R), lat.q(e, G), lat.q(e, B)
            for ctrl in (qr, qb):
                if ctrl in fz:
                    continue
                if qg in fz:
                    continue   # L_rb: CNOT(r,g) CNOT(b,g) = identity on Z^r Z^b = +1 (g removed)
                gates.append(('CNOT', ctrl, qg))
        for p in lat.plaquettes:
            e01, e13, e02, e23 = lat.plaq_edges(p)
            for (a, b, name) in [(lat.q(e01, R), lat.q(e13, B), 'CS'), (lat.q(e02, R), lat.q(e23, B), 'CSDG')]:
                if a in fz or b in fz:
                    continue
                gates.append((name, a, b))
        for e in lat.boundary_edges('Lrb'):
            gates.append(('TDG', lat.q(e, R)))
        return gates

    # ---------------------------------------------------------- switch-out
    def switch_out_gates(self, worst_case=True):
        lat = self.lat
        gq = sorted(self.data_qubits((G,)))
        gates = [('MEAS_Z', q) for q in gq]
        cz = []
        for v in self.gverts:
            for p in lat.cz_pairs(v, G):
                if self._keep_pair(*p):
                    cz.append(('CZ',) + tuple(p))
        return gates, cz      # cz: worst-case correction (all vertices in kappa)

    def qubit_sets(self):
        fsc_data = self.data_qubits((R, B))
        d4_data = self.data_qubits((R, G, B))
        fsc_anc = {self.anc[l] for l, _ in self.fsc.xgens} | {self.anc[l] for l in self.fsc.zlabels
                                                              if not self._is_frozen_check(l)}
        d4_anc = {self.anc[l] for l, _ in self.d4.xgens} | {self.anc[l] for l in self.d4.zlabels
                                                            if not self._is_frozen_check(l)}
        return dict(fsc_data=fsc_data, d4_data=d4_data, fsc_anc=fsc_anc, d4_anc=d4_anc)

In [ ]:
# ----------------------------------------------------------------- Kobayashi (D4)
class D4CostData:
    """All d-dependent counts of the D4 protocol, computed once per L from explicit circuits."""

    def __init__(self, L, dur=None, decomposed=False):
        dur = dur or DUR_UNIT
        P = D4Protocol(L)
        self.L = L
        qs = P.qubit_sets()
        self.n = {k: len(v) for k, v in qs.items()}
        self.n_total = len(qs['d4_data'] | qs['d4_anc'])
        dec = decompose if decomposed else (lambda g: g)
        gad = {
            'D4': [dec(g) for g in P.round_gadgets('D4')],
            'D4g': [dec(g) for g in P.round_gadgets('D4', 'g')],
            'FSC': P.round_gadgets('FSC'),
        }
        self.round = {}
        for k, g in gad.items():
            ub = round_schedule_classes(g, dur)
            lb = round_lower_bound(g, dur)
            self.round[k] = dict(time_lb=lb['time_lb'], time_ub=ub['time_ub'], depth_lb=lb['depth_lb'],
                                 depth_ub=ub['depth_ub'], classes=ub['classes'],
                                 gates=Counter(gate_counts([x for gg in g for x in gg])),
                                 n_anc=len(g))
        self.n_anc_g = len(gad['D4g'])
        U = dec(P.U_gates())
        Ul = class_layers([U])              # all U gates mutually commute (validated)
        self.U = dict(gates=Counter(gate_counts(U)), time=circuit_time(Ul, dur), depth=len(Ul))
        meas, cz = P.switch_out_gates()
        czl = class_layers([cz]) if cz else []
        ml = [meas]
        self.out = dict(gates_worst=Counter(gate_counts(meas + cz)),
                        gates_expected=Counter({**gate_counts(meas), 'CZ': len(cz) / 2}),
                        time=circuit_time(ml + czl, dur), depth=len(ml) + len(czl))
        self.x_corr = dict(time=dur['X'], gates_worst=Counter({'X': self.n['fsc_data']}))
        self.init = Counter({'INIT_X': self.n['fsc_data'], 'INIT_Z': len(qs['d4_data']) - self.n['fsc_data']})


def d4_stages(cd, model, params=DEFAULT_PARAMS, comparison='A', data_only=False):
    d = cd.L
    n = cd.n
    fsc_q = n['fsc_data'] + (0 if data_only else n['fsc_anc'])
    d4_q = n['d4_data'] + (0 if data_only else n['d4_anc'])
    rows = []

    def add(stage, qubits, rounds, rkey, extra_time, gates, source, note=''):
        rt = cd.round[rkey] if rkey else dict(time_lb=0, time_ub=0)
        rows.append(dict(protocol='Kobayashi_D4', comparison=comparison, model=model, d=d, stage=stage,
                         qubits=qubits, rounds=rounds,
                         time_lb=rounds * rt['time_lb'] + extra_time,
                         time_ub=rounds * rt['time_ub'] + extra_time,
                         gates=dict(gates), source=source, note=note))

    r0 = rounds_for(model, params['kappa0'], d)
    g0 = Counter(cd.init)
    for k, v in cd.round['FSC']['gates'].items():
        g0[k] += v * r0
    g0['X'] += cd.x_corr['gates_worst']['X']
    add('S0 FSC |+_L> preparation', fsc_q, r0, 'FSC', cd.x_corr['time'], g0,
        SOURCE_ASSUME if model == 'FT_ref' else SOURCE_COUNT,
        'kappa0*d rounds (not specified in Kobayashi); X-correction worst case counted')
    if model == 'NL_min':
        r1, key = 1, 'D4g'
        q1 = n['d4_data'] + (0 if data_only else cd.n_anc_g)
        note1 = 'noiseless minimum: one round of S^g_X measurements only'
    else:
        r1, key = rounds_for(model, params['kappa_in'], d), 'D4'
        q1 = d4_q
        note1 = 'O(d) rounds of all D4 stabilizers (Kobayashi step 2); kappa_in*d assumed'
    g1 = Counter()
    for k, v in cd.round[key]['gates'].items():
        g1[k] += v * r1
    add('S1 switch-in (gauging) to D4', q1, r1, key, 0, g1,
        SOURCE_ASSUME if model == 'FT_ref' else SOURCE_COUNT, note1)
    add('S2 transversal U = W V (logical T^dag)', n['d4_data'], 0, None, cd.U['time'], cd.U['gates'],
        SOURCE_COUNT, 'constant depth; gates from eq. (8)')
    add('S3 switch-out: Z^g meas + S^g correction', n['d4_data'], 0, None, cd.out['time'],
        cd.out['gates_worst'], SOURCE_COUNT, 'CZ correction counted worst case (expected = half)')
    r4 = 0 if model == 'NL_min' else rounds_for(model, params['kappa_out'], d)
    g4 = Counter()
    for k, v in cd.round['FSC']['gates'].items():
        g4[k] += v * r4
    add('S4 FSC error-correction rounds', fsc_q, r4, 'FSC', 0, g4,
        SOURCE_ASSUME if model == 'FT_ref' else SOURCE_COUNT,
        'O(d) rounds (Kobayashi step 4); kappa_out*d assumed; 0 in the noiseless minimum')
    add('S5 logical X frame: T^dag|+> -> T|+>', 0, 0, None, 0, Counter(), SOURCE_COUNT,
        'Pauli frame update, no physical cost')
    # NOTE (comparison B): the FSC -> standard rotated-patch conversion is UNRESOLVED.  It is deliberately
    # NOT added as a zero-cost stage; it is reported separately by uncounted_costs().
    return rows



def d4_conversion_illustrative(cd, model, params=DEFAULT_PARAMS, data_only=False):
    """ILLUSTRATIVE ONLY: conversion modelled as one lattice-surgery time step (tau) during which the
    FSC plus 2 tiles (target patch + ancilla strip) are active.  Not taken from literature."""
    d = cd.L
    r = rounds_for(model, params['tau_rounds'], d)
    q = cd.n['fsc_data'] + (0 if data_only else cd.n['fsc_anc']) + 2 * (d * d if data_only else 2 * d * d)
    return dict(protocol='Kobayashi_D4', comparison='B_illustrative', model=model, d=d,
                stage='S6 conversion (ILLUSTRATIVE: 1 tau lattice surgery)', qubits=q, rounds=r,
                time_lb=None, time_ub=None, gates={}, source=SOURCE_UNRES,
                note='illustrative assumption, not a validated protocol')

In [ ]:
# ---------------------------------------------------------- consistency checks of a code
def code_consistency(cd):
    """Exact F2 checks of a (possibly non-commuting) Clifford stabilizer code on its Z-subspace C_Z:
      shift_in_CZ : every X-type generator preserves C_Z (its X part commutes with all Z checks)
      noncommuting_pairs : generator pairs that do NOT commute on C_Z
      bad_squares : generators with S^2 != 1 on C_Z
      k : number of logical qubits = dim C_Z - rank(shifts)   (generators have independent shifts)"""
    lat = cd.lat
    HZ, S = cd.HZ, cd.S
    res = {}
    res['shift_in_CZ'] = not ((HZ.astype(int) @ S.T.astype(int)) % 2).any()
    supp = [op.support() for _, op in cd.xgens]
    bad = []
    for i in range(len(cd.xgens)):
        for j in range(i + 1, len(cd.xgens)):
            if not (supp[i] & supp[j]):
                continue
            lin, const = commutator_affine(cd.xgens[i][1], cd.xgens[j][1])
            if const or not f2_in_rowspace(cd.HZr, cd.HZpiv, lat.vec(lin)):
                bad.append((cd.xgens[i][0], cd.xgens[j][0]))
    res['noncommuting_pairs'] = bad
    sq = []
    for lab, op in cd.xgens:
        s2 = op @ op
        if s2.s or s2.quad or s2.const or not f2_in_rowspace(cd.HZr, cd.HZpiv, lat.vec(s2.lin)):
            sq.append(lab)
    res['bad_squares'] = sq
    res['n_qubits'] = len(cd.cols)
    res['dim_CZ'] = len(cd.cols) - f2_rank(HZ[:, cd.cols])
    res['rank_shifts'] = f2_rank(S[:, cd.cols])
    res['n_xgens'] = len(cd.xgens)
    res['k'] = res['dim_CZ'] - res['rank_shifts']
    res['shifts_independent'] = res['rank_shifts'] == res['n_xgens']
    return res


def U_target(lat, lab):
    """Image of generator `lab` under U predicted by SM eq. (permutingstabilizers):
    S^r -> S^r S^g, S^g -> S^g, S^b -> S^g S^b, and (boundary L_rb) S^r S^b -> S^r S^b."""
    _, v, cols = lab
    if cols == 'rb':
        return lat.SX(v, R) @ lat.SX(v, B)
    op = PSOp()
    for ch in cols:
        c = CNAME.index(ch)
        for cc in ([c] if c == G else [c, G]):
            op = op @ lat.SX(v, cc)
    return op

In [ ]:
# ---------------------------------------------------------- Knill-Laflamme distance lower bound
def _milp_binary(c_obj, A_eq_mod2, rhs_mod2, extra_ub=None, fixed_zero=()):
    """min c.x over binary x subject to A x = rhs (mod 2) (rows of A_eq_mod2), and optional
    linear constraints extra_ub = [(row, ub), ...] (row.x <= ub).
    Returns (value, x) if an optimum was found, (None, None) only if the problem is proven
    infeasible (scipy status 2).  Any other status (time/iteration limit, unbounded, numerical
    failure) raises RuntimeError, so that a solver failure is never mistaken for "no solution"."""
    from scipy.optimize import milp, LinearConstraint, Bounds
    A = np.array(A_eq_mod2, dtype=float)
    m, n = A.shape
    nv = n + m
    c = np.concatenate([np.asarray(c_obj, float), np.zeros(m)])
    cons = [LinearConstraint(np.concatenate([A, -2 * np.eye(m)], axis=1), rhs_mod2, rhs_mod2)]
    for row, ub in (extra_ub or []):
        cons.append(LinearConstraint(np.concatenate([np.asarray(row, float), np.zeros(m)])[None, :], -np.inf, ub))
    ub_ = np.concatenate([np.ones(n), np.full(m, np.inf)])
    for i in fixed_zero:
        ub_[i] = 0
    res = milp(c, constraints=cons, integrality=np.ones(nv), bounds=Bounds(np.zeros(nv), ub_))
    if res.status == 2:
        return None, None
    if res.status != 0:
        raise RuntimeError(f'MILP solver did not finish (status {res.status}: {res.message})')
    return round(res.fun), np.round(res.x[:n]).astype(np.uint8)


def low_weight_stabilizer_shifts(cd, wmax, frozen=()):
    """Exact enumeration (MILP with no-good cuts) of all t in span(S) with 1 <= |t| <= wmax,
    where span(S) = C_Z  intersect  z_log^perp  (k = 1).  Frozen qubits are excluded
    (an X on a frozen qubit violates its single-qubit Z check, so t would leave C_Z).
    Returns (found, stop) where stop records why the enumeration ended:
    'infeasible' (no further t with 1 <= |t| <= wmax) or 'min weight > wmax'."""
    cols = [c for c in cd.cols if c not in set(frozen)]
    H = np.vstack([cd.HZ[:, cols], cd.z_log[cols][None, :]])
    H = H[np.any(H, axis=1)]
    rhs = np.zeros(len(H))
    n = len(cols)
    cuts = [(np.ones(n), wmax), (-np.ones(n), -1)]
    found = []
    while True:
        val, x = _milp_binary(np.ones(n), H, rhs, extra_ub=cuts)
        if val is None:
            stop = 'infeasible'
            break
        if val > wmax:
            stop = 'min weight > wmax'
            break
        sup = np.nonzero(x)[0]
        t = np.zeros(cd.lat.n, dtype=np.uint8)
        t[[cols[i] for i in sup]] = 1
        found.append(t)
        row = -np.ones(n)
        row[sup] = 1
        cuts.append((row, len(sup) - 1))                  # exclude exactly this vector
    return found, stop


def kl_distance_lower_bound(cd, wmax, frozen=()):
    r"""Rigorous lower-bound argument for the Knill-Laflamme distance of the (non-Pauli) code cd.

    For a Pauli E = X^t Z^u (up to phase):
      * t not in C_Z           -> E maps the code space to an orthogonal space: detectable;
      * t in the logical coset -> |t| >= X-shift distance (checked separately);
      * t in span(S)           -> X^t = S_c F_c with S_c a stabilizer and F_c = (-1)^{f_c} diagonal,
        so P E P = P F_c Z^u P.  Expanding (-1)^{f_c} in Z-type Paulis Z^v with v supported on
        Q_f = supp(f_c), a violation of KL requires some v + u to be a logical Z operator l.
        Hence |E| >= |t| + |u \ t| >= |t| + min_l |l \ (Q_f u t)|  =: LB_t.
    Returns (min over enumerated t of LB_t, ts, details, stop) where stop is the termination
    reason of the enumeration (see low_weight_stabilizer_shifts).  t = 0 is covered by d_Z."""
    lat = cd.lat
    ts, stop = low_weight_stabilizer_shifts(cd, wmax, frozen)
    cols = [c for c in cd.cols if c not in set(frozen)]
    Srows = cd.S[:, cols]
    Srows = Srows[np.any(Srows, axis=1)]
    best, details = None, []
    for t in ts:
        coef = f2_solve(cd.S, t)
        op = PSOp()
        for i in np.nonzero(coef)[0]:
            op = op @ cd.xgens[i][1]
        assert op.s == frozenset(np.nonzero(t)[0])
        Qf = {q for pr in op.quad for q in pr} | set(op.lin) | set(np.nonzero(t)[0])
        cost = np.array([0.0 if c in Qf else 1.0 for c in cols])
        A = np.vstack([Srows, cd.t_log[cols][None, :]])
        rhs = np.concatenate([np.zeros(len(Srows)), [1.0]])
        val, _ = _milp_binary(cost, A, rhs)
        assert val is not None                            # a logical Z always exists
        lb = int(t.sum()) + val
        details.append(dict(weight_t=int(t.sum()), Qf_size=len(Qf), LB=lb))
        best = lb if best is None else min(best, lb)
    return best, ts, details, stop

In [ ]:
# ---------------------------------------------------------- full resource tables (used by main + comparison)
def d4_resource_tables(D_LIST, dur=None, params=None, verbose=True):
    """Per-stage and summary tables of the Kobayashi protocol for every d in D_LIST, gate set
    ('primitive' | 'clifford_t'), counting variant (data_only), time model and comparison A/B.
    Returns (stage_df, sum_df, blocks)."""
    import pandas as pd, time
    dur = dur or DUR_UNIT
    params = params or DEFAULT_PARAMS
    records, summaries, blocks = [], [], {}
    for d in D_LIST:
        t0 = time.time()
        for gs in ['primitive', 'clifford_t']:
            cd = D4CostData(d, dur, decomposed=(gs == 'clifford_t'))
            for data_only in [False, True]:
                for model in ['FT_ref', 'NL_min']:
                    for comp in ['A', 'B']:
                        st = d4_stages(cd, model, params, comp, data_only=data_only)
                        for r in st:
                            r.update(gateset=gs, data_only=data_only)
                        records += st
                        s = summarize(st)
                        s.update(protocol='Kobayashi_D4', d=d, model=model, comparison=comp, gateset=gs,
                                 data_only=data_only)
                        summaries.append(s)
            blocks[f'{d}|{gs}'] = dict(d=d, gateset=gs, n=cd.n, n_total=cd.n_total, n_anc_g=cd.n_anc_g,
                                       round={k: {kk: (dict(vv) if isinstance(vv, Counter) else vv) for kk, vv in v.items()}
                                              for k, v in cd.round.items()},
                                       U_time=cd.U['time'], U_gates=dict(cd.U['gates']), out_time=cd.out['time'],
                                       out_gates=dict(cd.out['gates_worst']), xcorr_time=cd.x_corr['time'])
        if verbose:
            print(f'Kobayashi d={d}: {time.time() - t0:.1f}s')
    return jsonify_columns(pd.DataFrame(records)), jsonify_columns(pd.DataFrame(summaries)), blocks

### 雑音付き D4 code switching ライブラリ（回路・雑音モデル・Takada らの Pauli proxy 法による非 Clifford シミュレーション・復号器）

In [ ]:
"""noisy_d4_lib: circuit-level noisy simulation of the Kobayashi-Zhu-Hsin D4 code switching protocol.

Non-Clifford simulation method: Clifford-stabilizer simulation with Pauli proxies
(Y. Takada, S. D. Bartlett, D. J. Williamson, arXiv:2609.16929, Algorithm 2 (Sec. VII D) and Sec. VIII).
For every Monte Carlo shot the noisy non-Clifford circuit is mapped to a Clifford circuit (the
"Pauli-proxy circuit") with the same measurement statistics:
  * every diagonal third-level gate D (CCZ of the D4 gadgets, CS / CS^dag / T^dag of U) is removed;
  * the X component e of the sampled Pauli errors and of applied X corrections, propagated forward
    through the Clifford gates of the true circuit, is tracked ("continuing errors"); where e meets
    D the diagonal Clifford gate C_e = D^dag X^e D X^e (a product of S, Z, CZ) is inserted
    (= the controlled Pauli dressing Lambda(D_U(E)) of Takada Fig. 14 / Eq. (165));
  * Z components commute with every diagonal gate and need no propagation.
All Clifford gates, sampled Pauli errors and inserted dressings are executed by stim's TableauSimulator.

Requires (as globals, embedded before this code): numpy as np, stim, common_lib, d4_lib.
"""
import itertools
import zlib
import numpy as np

In [ ]:
# ------------------------------------------------------------------ gate algebra
DIAG_THETA = {                     # phase polynomials (units of pi/4) of the diagonal 3rd-level gates
    'CCZ': lambda x: 4 * x[0] * x[1] * x[2],
    'NCCZ': lambda x: 4 * x[0] * x[1] * x[2],      # virtual noiseless CCZ (deferred switch-out correction)
    'CS': lambda x: 2 * x[0] * x[1],
    'CSDG': lambda x: 6 * x[0] * x[1],
    'T': lambda x: x[0],
    'TDG': lambda x: 7 * x[0],
}
NONCLIFF = set(DIAG_THETA)
_DRESS = {}


def diag_clifford_gates(delta, k):
    """delta: dict bits -> phase exponent mod 8 (even).  Returns gates ('S'|'Z'|'SDG', i), ('CZ', i, j)
    (local indices) equal to diag(omega^delta) up to a global phase; asserts the result is exact."""
    c0 = delta[(0,) * k]
    a, gates = [], []
    for i in range(k):
        e = tuple(int(j == i) for j in range(k))
        ai = (delta[e] - c0) % 8
        assert ai % 2 == 0, 'dressing is not Clifford'
        a.append(ai // 2)
        if ai // 2:
            gates.append(({1: 'S', 2: 'Z', 3: 'SDG'}[ai // 2], i))
    b = {}
    for i, j in itertools.combinations(range(k), 2):
        e = tuple(int(m in (i, j)) for m in range(k))
        v = (delta[e] - c0 - 2 * a[i] - 2 * a[j]) % 8
        assert v in (0, 4), 'dressing has a non-Clifford quadratic term'
        b[(i, j)] = v // 4
        if v:
            gates.append(('CZ', i, j))
    for x in itertools.product((0, 1), repeat=k):
        pred = c0 + sum(2 * a[i] * x[i] for i in range(k)) + sum(4 * b[ij] * x[ij[0]] * x[ij[1]] for ij in b)
        assert (pred - delta[x]) % 8 == 0, 'dressing has a cubic term'
    return gates


def dressing(name, e):
    """C_e = D^dag X^e D X^e for diagonal gate `name` and X pattern e (tuple of bits), local indices."""
    key = (name, e)
    if key not in _DRESS:
        th = DIAG_THETA[name]
        k = len(e)
        delta = {x: (th(tuple(xi ^ ei for xi, ei in zip(x, e))) - th(x)) % 8
                 for x in itertools.product((0, 1), repeat=k)}
        _DRESS[key] = diag_clifford_gates(delta, k)
    return _DRESS[key]

In [ ]:
# ------------------------------------------------------------------ noise model
class NoiseModel:
    """Standard circuit-level Pauli noise (Takada et al. App. A 2) plus optional idling noise:
    after every k-qubit unitary a k-qubit depolarizing channel (probability p; p_ccz for CCZ, p_nc for
    the gates of U), preparation followed by a flip (p_reset), measurement preceded by a flip (p_meas),
    every idle qubit of a time step depolarized with p_idle."""

    def __init__(self, p, p_meas=None, p_reset=None, p_idle=None, p_ccz=None, p_nc=None):
        self.p = p
        self.p_meas = p if p_meas is None else p_meas
        self.p_reset = p if p_reset is None else p_reset
        self.p_idle = p if p_idle is None else p_idle
        self.p_ccz = p if p_ccz is None else p_ccz
        self.p_nc = p if p_nc is None else p_nc

    def locations(self, ops, idle):
        """Noise locations of one time step: list of (when, kind, qubits, prob), when in {'pre','post'}."""
        loc = []
        for op in ops:
            nm, qs = op[0], op[1:]
            if nm in ('NCCZ', 'MZ0'):
                continue
            if nm == 'FLIPX':
                loc.append(('pre', 'X', qs, self.p_meas))
            elif nm == 'MZ':
                loc.append(('pre', 'X', qs, self.p_meas))
            elif nm == 'MX':
                loc.append(('pre', 'Z', qs, self.p_meas))
            elif nm == 'RZ':
                loc.append(('post', 'X', qs, self.p_reset))
            elif nm == 'RX':
                loc.append(('post', 'Z', qs, self.p_reset))
            elif nm == 'CCZ':
                loc.append(('post', 'D3', qs, self.p_ccz))
            elif nm in ('CS', 'CSDG', 'T', 'TDG'):
                loc.append(('post', 'D%d' % len(qs), qs, self.p_nc))
            else:
                loc.append(('post', 'D%d' % len(qs), qs, self.p))
        for q in idle:
            loc.append(('post', 'D1', (q,), self.p_idle))
        return [l for l in loc if l[3] > 0]


PAULI_CHOICES = {k: [''.join(p) for p in itertools.product('IXYZ', repeat=k) if set(p) != {'I'}]
                 for k in (1, 2, 3)}

In [ ]:
# ------------------------------------------------------------------ parallel FSC schedule
_FSC_SCHED = {}


def _fsc_checks(P):
    """[(type 'X'|'Z', ancilla, label, [data qubits])] of one FSC round (frozen qubits removed)."""
    lat, cd = P.lat, P.fsc
    out = []
    for lab, op in cd.xgens:
        _, v, cols = lab
        qs = []
        for ch in cols:
            cc = CNAME.index(ch)
            qs += sorted({lat.q(e, cc) for e in lat.incident(v)} - P.frozen)
        out.append(('X', P.anc[lab], lab, qs))
    for lab, (l2, rows) in zip(cd.zlabels, lat.z_checks(cd.colours)):
        if P._is_frozen_check(lab):
            continue
        out.append(('Z', P.anc[lab], lab, sorted(set(rows) - P.frozen)))
    return out


def _fsc_anneal(P, checks, T, seed, iters):
    """Assign a time slot 0..T-1 to every (check, data qubit) CNOT such that (i) no ancilla and no data qubit
    is used twice in a slot and (ii) for every X check A and Z check B the number of shared qubits on
    which B's CNOT precedes A's is even (so all checks can be measured at the same time).  Start: the
    direction-based order of the bulk (X stars E,N,S,W = 0,1,2,3; Z plaquettes E,S,N,W = 0,1,2,3, one of
    the 16 compatible bulk orders); simulated annealing fixes the boundary (weight-6 X^r X^b and
    weight-2 Z^r Z^b checks of L_rb)."""
    lat = P.lat
    XO, ZO = {'E': 0, 'W': 3, 'N': 1, 'S': 2}, {'S': 1, 'N': 2, 'W': 3, 'E': 0}
    rng = np.random.default_rng(seed)
    var, slot = [], []
    for ci, (typ, anc, lab, qs) in enumerate(checks):
        for q in qs:
            k, x, y = lat.edges[q // 3]
            if typ == 'X':
                d = ('E' if x == lab[1][0] else 'W') if k == 'h' else ('N' if y == lab[1][1] else 'S')
                s0 = XO[d]
            elif lab[0] == 'plaq':
                d = ('S' if y == lab[1][1] else 'N') if k == 'h' else ('W' if x == lab[1][0] else 'E')
                s0 = ZO[d]
            else:
                s0 = 0
            var.append((ci, q)); slot.append(s0)
    slot = np.array(slot)
    by_check, by_qubit = {}, {}
    for i, (ci, q) in enumerate(var):
        by_check.setdefault(ci, []).append(i); by_qubit.setdefault(q, []).append(i)
    pairs = {}
    for q, idx in by_qubit.items():
        xs = [i for i in idx if checks[var[i][0]][0] == 'X']
        zs = [i for i in idx if checks[var[i][0]][0] == 'Z']
        for a_ in xs:
            for b_ in zs:
                pairs.setdefault((var[a_][0], var[b_][0]), []).append((a_, b_))
    pairs = list(pairs.values())
    pv = {}
    for k, pl in enumerate(pairs):
        for a_, b_ in pl:
            pv.setdefault(a_, []).append(k); pv.setdefault(b_, []).append(k)
    bad = lambda k: sum(slot[b_] < slot[a_] for a_, b_ in pairs[k]) % 2
    conf = lambda i: (sum(slot[j] == slot[i] for j in by_check[var[i][0]] if j != i)
                      + sum(slot[j] == slot[i] for j in by_qubit[var[i][1]] if j != i))
    local = lambda i: conf(i) + sum(bad(k) for k in pv.get(i, []))
    total = lambda: sum(conf(i) for i in range(len(var))) // 2 + sum(bad(k) for k in range(len(pairs)))
    cost, temp = total(), 1.0
    for it in range(iters):
        if cost == 0:
            break
        i = int(rng.integers(len(var)))
        if local(i) == 0 and rng.random() < 0.9:
            continue
        old, before = slot[i], local(i)
        slot[i] = rng.integers(T)
        dlt = local(i) - before
        if dlt <= 0 or rng.random() < np.exp(-dlt / temp):
            cost += dlt
        else:
            slot[i] = old
        if it % 20000 == 0:
            cost, temp = total(), max(0.05, temp * 0.85)
    return var, slot, total()


def fsc_memory_circuit(P, sched, basis, rounds, p):
    """stim memory experiment of the FSC with the parallel schedule (validation): data in |+> (X) or |0>
    (Z), `rounds` rounds, final data measurement; observable = logical X (t_log) or a logical Z.  Noise:
    DEPOLARIZE2(p) after every CNOT, flips p on ancilla reset / before measurement / final data."""
    checks, T, layers = sched['checks'], sched['T'], sched['layers']
    data = sorted({q for typ, anc, lab, qs in checks for q in qs})
    idx = {q: i for i, q in enumerate(data)}
    t_log = [int(q) for q in np.nonzero(P.fsc.t_log)[0]]
    Hx = np.array([[1 if q in qs else 0 for q in data] for typ, anc, lab, qs in checks if typ == 'X'], dtype=np.uint8)
    tv = np.array([1 if q in t_log else 0 for q in data], dtype=np.uint8)
    zl = next([data[i] for i in np.nonzero(v)[0]] for v in f2_nullspace(Hx) if int(v.astype(int) @ tv.astype(int)) % 2)
    xa = [anc for typ, anc, lab, qs in checks if typ == 'X']
    za = [anc for typ, anc, lab, qs in checks if typ == 'Z']
    nX, nZ = len(xa), len(za)
    cir = stim.Circuit()
    cir.append('RX' if basis == 'X' else 'R', data)
    for r in range(rounds):
        cir.append('RX', xa); cir.append('R', za)
        if p:
            cir.append('Z_ERROR', xa, p); cir.append('X_ERROR', za, p)
        cir.append('TICK')
        for t in range(T):
            tg = []
            for typ, anc, q in layers[t]:
                tg += [anc, q] if typ == 'X' else [q, anc]
            if tg:
                cir.append('CX', tg)
                if p:
                    cir.append('DEPOLARIZE2', tg, p)
            cir.append('TICK')
        if p:
            cir.append('Z_ERROR', xa, p); cir.append('X_ERROR', za, p)
        cir.append('MX', xa); cir.append('M', za)
        nm = nX + nZ
        for k in range(nX):
            if r:
                cir.append('DETECTOR', [stim.target_rec(-nm + k), stim.target_rec(-2 * nm + k)])
            elif basis == 'X':
                cir.append('DETECTOR', [stim.target_rec(-nm + k)])
        for k in range(nZ):
            if r:
                cir.append('DETECTOR', [stim.target_rec(-nZ + k), stim.target_rec(-nZ - nm + k)])
            elif basis == 'Z':
                cir.append('DETECTOR', [stim.target_rec(-nZ + k)])
    if p:
        cir.append('Z_ERROR' if basis == 'X' else 'X_ERROR', data, p)
    cir.append('MX' if basis == 'X' else 'M', data)
    nd = len(data)
    off = 0 if basis == 'X' else nX
    for k, (anc, qs) in enumerate([(anc, qs) for typ, anc, lab, qs in checks if typ == basis]):
        cir.append('DETECTOR', [stim.target_rec(-nd + idx[q]) for q in qs] + [stim.target_rec(-nd - (nX + nZ) + off + k)])
    cir.append('OBSERVABLE_INCLUDE', [stim.target_rec(-nd + idx[q]) for q in (t_log if basis == 'X' else zl)], 0)
    return cir


def fsc_parallel_schedule(P, T=6, max_seeds=30, iters=3000000):
    """Measure all FSC checks at the same time in T CNOT layers (round = T + 2 time steps; T = 6 is the
    minimum because the X^r X^b checks of L_rb have weight 6; the bulk alone would need 4).  The slot
    assignment is searched (deterministic seeds) and ACCEPTED ONLY IF the stim memory experiments in both
    bases are deterministic without noise and have circuit-level distance >= L (no harmful hook errors).
    Cached per L.  Returns dict(T, checks, layers[t] = [(type, ancilla, data qubit)], seed, distance)."""
    L = P.L
    if L in _FSC_SCHED:
        return _FSC_SCHED[L]
    checks = _fsc_checks(P)
    for seed in range(max_seeds):
        var, slot, cost = _fsc_anneal(P, checks, T, seed, iters)
        if cost:
            continue
        layers = [[] for _ in range(T)]
        for i, (ci, q) in enumerate(var):
            layers[int(slot[i])].append((checks[ci][0], checks[ci][1], q))
        sch = dict(T=T, checks=checks, layers=layers, seed=seed)
        ok, dist = True, {}
        for basis in 'XZ':
            s0 = fsc_memory_circuit(P, sch, basis, 3, 0.0).compile_detector_sampler(seed=1).sample(20, append_observables=True)
            c1 = fsc_memory_circuit(P, sch, basis, max(L, 2), 1e-3)
            dist[basis] = len(c1.detector_error_model(decompose_errors=True).shortest_graphlike_error())
            ok &= (not s0.any()) and dist[basis] >= L
        if ok:
            sch['distance'] = dist
            _FSC_SCHED[L] = sch
            return sch
    raise RuntimeError(f'no valid parallel FSC schedule found for L={L}')

In [ ]:
# ------------------------------------------------------------------ the noisy protocol circuit
class D4SwitchingCircuit:
    """Time-step level description of the noisy KZH protocol on the bounded square lattice.

    Blocks (in order):
      ('S0', r)  r = 1..R0   FSC |+_L> preparation: data |+>, rounds of all FSC checks
      ('S1', r)  r = 1..Rin  switch-in: g |0>, rounds of all D4 stabilizers ([KZH] step 2)
      ('U',)                 transversal U = W V ([KZH] eq. (8), step 3)
      ('SO',)                switch-out: Z^g readout ([KZH] step 4; deferred-correction form)
      ('S4', r)  r = 1..Rout FSC rounds after switching back ([KZH] step 4)
      ('FIN',)               one noiseless FSC round (fictitious error correction, Takada Sec. VI D)
    Qubits are re-indexed compactly: data (non-frozen r,g,b) first, then ancillas.
    sequential=True measures the gadgets of a commuting class one after another (validation only)."""

    def __init__(self, L, R0, Rin, Rout, sequential=False, n_pre_u=2, fsc_schedule='classes'):
        self.n_pre_u = n_pre_u
        P = D4Protocol(L)
        lat = P.lat
        self.P, self.lat, self.L = P, lat, L
        self.R0, self.Rin, self.Rout = R0, Rin, Rout
        data = sorted(P.data_qubits((R, G, B)))
        gadgets = {'D4': P.round_gadgets('D4'), 'FSC': P.round_gadgets('FSC')}
        ancs = sorted({g[0][1] for k in gadgets for g in gadgets[k]})
        c = {q: i for i, q in enumerate(data + ancs)}
        self.c = c
        self.nq, self.nd = len(data) + len(ancs), len(data)
        self.data_rb = sorted(c[q] for q in P.data_qubits((R, B)))
        self.data_g = sorted(c[q] for q in P.data_qubits((G,)))
        self.data_all = sorted(c[q] for q in data)
        conv = {'INIT_X': 'RX', 'INIT_Z': 'RZ', 'MEAS_X': 'MX', 'MEAS_Z': 'MZ', 'CNOT': 'CX'}
        cv = lambda g: (conv.get(g[0], g[0]),) + tuple(c[q] for q in g[1:])
        # label of every ancilla, per code
        self.lab_of = {}
        for code, cd in (('D4', P.d4), ('FSC', P.fsc)):
            labs = [l for l, _ in cd.xgens] + [l for l in cd.zlabels if not P._is_frozen_check(l)]
            for l in labs:
                self.lab_of[(code, c[P.anc[l]])] = l

        def round_moments(code, which='all'):
            gl = gadgets[code]
            if which != 'all':
                gl = [g for g in gl if g[0][0] == ('INIT_Z' if which == 'Z' else 'INIT_X')]
            classes = commuting_classes(gl)
            if sequential:
                classes = [[g] for cls in classes for g in cls]
            moms = []
            for cls in classes:
                for layer in class_layers(cls):
                    moms.append([cv(g) for g in layer])
            return moms
        # D4 rounds: Pauli Z checks first, then (after the real-time flux correction) the X-type
        # Clifford stabilizers, so that fluxes detected by the Z checks are removed before they meet CCZs
        if fsc_schedule == 'parallel' and not sequential:
            # all FSC checks at once: ancilla init, 6 CNOT layers (verified order, fsc_parallel_schedule), measure
            fs = fsc_parallel_schedule(P)
            moms = [[('RX' if typ == 'X' else 'RZ', c[anc]) for typ, anc, lab, qs in fs['checks']]]
            for t in range(fs['T']):
                moms.append([('CX', c[anc], c[q]) if typ == 'X' else ('CX', c[q], c[anc])
                             for (typ, anc, q) in fs['layers'][t]])
            moms.append([('MX' if typ == 'X' else 'MZ', c[anc]) for typ, anc, lab, qs in fs['checks']])
            fsc_moms = moms
        else:
            # default: X class then Z class (14 steps).  The verified 8-step parallel schedule gives a larger
            # output error with OUR flux decoder (its phenomenological error model does not represent the
            # hook errors that the interleaving splits across rounds), so it is used only as a sensitivity
            fsc_moms = round_moments('FSC')
        self.round_moms = {'FSC': fsc_moms, 'D4z': round_moments('D4', 'Z'), 'D4x': round_moments('D4', 'X')}
        self.anc = {code: sorted({c[g[0][1]] for g in gadgets[code]}) for code in gadgets}
        self.U_moms = [[cv(g) for g in layer] for layer in class_layers([P.U_gates()])]
        # ---- switch-out data: S^g generators, Gamma (g configuration -> kappa), CZ^{r,b} pairs
        self.gens_g = [(lab, op) for lab, op in P.d4.xgens if lab[2] == 'g']
        gpos = {q: i for i, q in enumerate(self.data_g)}
        Sg = np.zeros((len(self.gens_g), len(self.data_g)), dtype=np.uint8)
        for i, (lab, op) in enumerate(self.gens_g):
            for q in op.s:
                if q in c and c[q] in gpos:
                    Sg[i, gpos[c[q]]] = 1
        self.Sg = Sg
        self.Gamma = left_inverse_T(Sg)
        self.gpairs = [[(c[i], c[j]) for (i, j) in op.quad if i in c and j in c] for lab, op in self.gens_g]
        # physical switch-out correction CZ(Gamma(a_dec)) ([KZH] step 4): the CZ pairs of the selected S^g
        # (pairs occurring an even number of times cancel) are applied in a FIXED schedule: a greedy edge
        # colouring of the union of all pairs; colour l = time step l.  The number of steps does not depend
        # on the outcome (worst case, used for the time count); empty steps are still idle steps.
        allpairs = sorted({tuple(sorted(pr)) for prs in self.gpairs for pr in prs})
        colour, used = {}, {}
        for pr in allpairs:
            k = 0
            while any(k in used.get(q, ()) for q in pr):
                k += 1
            colour[pr] = k
            for q in pr:
                used.setdefault(q, set()).add(k)
        self.so_colour = colour
        self.so_steps = 1 + (max(colour.values()) + 1 if colour else 0)    # g readout + CZ layers
        self.network = [('NCCZ', self.data_g[e], i, j)
                        for v in range(len(self.gens_g)) for e in np.nonzero(self.Gamma[v])[0]
                        for (i, j) in self.gpairs[v]]
        # Pauli-frame strings Z^r Z^b for the S^g eigenvalues (protocol step 5 of the noiseless check V7)
        self.frame_strings = []
        for lab, op in self.gens_g:
            x0, y0 = lab[1]
            qs = [lat.q(lat.eidx[('v', x0, y)], cc) for y in range(y0) for cc in (R, B)]
            self.frame_strings.append(sorted(c[q] for q in qs if q in c))
        # ---- Z checks of the FSC (rb data, frozen checks removed) -> used for the S1 X frame
        self.zchecks = {}
        for code, cd in (('D4', P.d4), ('FSC', P.fsc)):
            rows = []
            for lab, (l2, qs) in zip(cd.zlabels, lat.z_checks(cd.colours)):
                if P._is_frozen_check(lab):
                    continue
                rows.append((lab, sorted(c[q] for q in qs if q in c)))
            self.zchecks[code] = rows
        self.xgens = {}
        for code, cd in (('D4', P.d4), ('FSC', P.fsc)):
            self.xgens[code] = [(lab, sorted(c[q] for q in op.s if q in c),
                                 [(c[i], c[j]) for (i, j) in op.quad if i in c and j in c]) for lab, op in cd.xgens]
        self._logical()
        self.blocks = self._blocks()

    # ------------------------------------------------------------ logical Clifford stabilizer
    def _logical(self):
        """Ideal output = D_F |+_L>_FSC with F(a) = w(a) + 4 sum_v c'_v(a) f^g_v(a) (protocol phase,
        validated in V7/V7b/V8).  Its logical Clifford stabilizer is H = D_F X^t D_F^dag = X^t D_Delta,
        Delta(x) = F(x+t) - F(x) = c0 + sum 2 a_i x_i + sum 4 b_ij x_i x_j (mod 8)."""
        P, lat, c = self.P, self.lat, self.c
        F = {}

        def add(mono, coef):
            m = frozenset(mono)
            if any(q not in c for q in m):
                return                                   # frozen qubits are |0>
            m = frozenset(c[q] for q in m)
            F[m] = (F.get(m, 0) + coef) % 8
        for coef, mono in W_operator(lat).terms:
            add(mono, coef)
        # c'_v(a): path sums along a BFS tree from the L_rb vertices (linear in a)
        from collections import deque
        root = lrb_vertices(lat)
        path = {v: [] for v in root}
        dq = deque(root)
        while dq:
            u = dq.popleft()
            for e in lat.incident(u):
                p, q = lat.endpoints(e)
                w = q if p == u else p
                if w not in path:
                    path[w] = path[u] + [e]
                    dq.append(w)
        for v in P.gverts:
            lin = [lat.q(e, cc) for e in path[v] for cc in (R, B)]
            for (i, j) in lat.cz_pairs(v, G):
                for k in lin:
                    add({k, i, j}, 4)
        self.Fpoly = {m: v for m, v in F.items() if v}
        t = sorted(c[int(q)] for q in np.nonzero(P.fsc.t_log)[0])
        self.t_log = t
        tset = set(t)
        D = {}
        for m, coef in self.Fpoly.items():           # F(x+t): substitute x_i -> 1 - x_i for i in t
            fixed = [q for q in m if q not in tset]
            flip = [q for q in m if q in tset]
            for sub in itertools.product((0, 1), repeat=len(flip)):
                mono = frozenset(fixed + [q for q, s in zip(flip, sub) if s])
                sign = (-1) ** sum(sub)
                D[mono] = (D.get(mono, 0) + coef * sign) % 8
        for m, coef in self.Fpoly.items():
            D[m] = (D.get(m, 0) - coef) % 8
        D = {m: v for m, v in D.items() if v}
        self.Delta_c0 = D.pop(frozenset(), 0)
        self.Delta_a, self.Delta_b = {}, {}
        for m, v in D.items():
            if len(m) == 1:
                assert v % 2 == 0, 'logical stabilizer not Clifford (odd linear term)'
                self.Delta_a[next(iter(m))] = v // 2 % 4
            elif len(m) == 2:
                assert v % 4 == 0, 'logical stabilizer not Clifford (quadratic term)'
                self.Delta_b[tuple(sorted(m))] = v // 4 % 2
            else:
                raise AssertionError('logical stabilizer not Clifford (cubic term)')
        self.Delta_a = {q: a for q, a in self.Delta_a.items() if a}
        self.Delta_b = {ij: b for ij, b in self.Delta_b.items() if b}

    def logical_proxy(self, fx, sign=1):
        """stim.PauliString of the Pauli proxy of the dressed logical stabilizer X^e H X^e
        (e = X frame fx on the data): proxy(X^t D_Delta R_e) = X^t * prod_{a_i=2} Z_i * R_e, where
        R_e = D_Delta^dag X^e D_Delta X^e = phase * Z-string (Takada Secs. VI D, VII B)."""
        n = self.nq
        ps = stim.PauliString(n)
        for q in self.t_log:
            ps[q] = 'X'
        zs = stim.PauliString(n)
        for q, a in self.Delta_a.items():
            if a == 2:
                zs[q] = 'Z'
        R = stim.PauliString(n)
        ph = 1
        for q, a in self.Delta_a.items():
            if fx[q] and a % 2 == 1:
                ph *= 1j ** a
                R *= _single(n, q, 'Z')
            elif fx[q] and a == 2:
                ph *= -1
        for (i, j), b in self.Delta_b.items():
            if fx[i] and fx[j]:
                ph *= -1
                R *= _single(n, i, 'Z') * _single(n, j, 'Z')
            elif fx[i]:
                R *= _single(n, j, 'Z')
            elif fx[j]:
                R *= _single(n, i, 'Z')
        out = ps * zs * R
        return out * complex(ph * sign)

    # ------------------------------------------------------------ blocks
    def _blocks(self):
        B_ = []
        rb, g, alld = self.data_rb, self.data_g, self.data_all
        for r in range(1, self.R0 + 1):
            pre = [[('RX', q) for q in rb]] if r == 1 else []
            B_.append(dict(name=('S0', r), code='FSC', moms=pre + self.round_moms['FSC'], data=rb))
        for r in range(1, self.Rin + 1 + self.n_pre_u):
            pre = [[('RZ', q) for q in g]] if r == 1 else []
            B_.append(dict(name=('S1', r, 'z'), code='D4', moms=pre + self.round_moms['D4z'], data=alld))
            if r <= self.Rin:            # rounds Rin+1.. have only the Z checks (flux checks before U)
                B_.append(dict(name=('S1', r, 'x'), code='D4', moms=self.round_moms['D4x'], data=alld))
        B_.append(dict(name=('U',), code=None, moms=self.U_moms, data=alld))
        B_.append(dict(name=('SO',), code=None,
                       moms=[[('FLIPX', q) for q in g], list(self.network), [('MZ0', q) for q in g]], data=alld))
        for r in range(1, self.Rout + 1):
            B_.append(dict(name=('S4', r), code='FSC', moms=self.round_moms['FSC'], data=rb))
        B_.append(dict(name=('FIN',), code='FSC', moms=self.round_moms['FSC'], data=rb, noiseless=True))
        return B_


def _single(n, q, p):
    s = stim.PauliString(n)
    s[q] = p
    return s


def left_inverse_T(S):
    """Gamma (m x n) over F2 with Gamma @ S^T = I_m, for S (m x n) of full row rank."""
    m, n = S.shape
    A = (S.T % 2).astype(np.uint8)
    M = np.concatenate([A, np.eye(n, dtype=np.uint8)], axis=1)
    r = 0
    for col in range(m):
        nz = np.nonzero(M[r:, col])[0]
        assert len(nz), 'S^T is not of full column rank'
        p = r + nz[0]
        M[[r, p]] = M[[p, r]]
        for rr in range(n):
            if rr != r and M[rr, col]:
                M[rr] ^= M[r]
        r += 1
    Gm = M[:m, m:]
    assert not ((Gm.astype(int) @ A.astype(int)) % 2 - np.eye(m, dtype=int)).any()
    return Gm

In [ ]:
# ------------------------------------------------------------------ exact reference (validation only)
class TrueSim:
    """Exact sparse state-vector simulation of the TRUE (non-Clifford) circuit, used to validate the
    Pauli-proxy simulation.  Basis states are uint64 bit masks: data qubits 0..nd-1, ancillas get a
    slot nd+k while they are alive (reset ... measurement)."""
    OMEGA = np.exp(1j * np.pi / 4)

    def __init__(self, nd, keys, amps):
        self.nd = nd
        self.keys = np.asarray(keys, dtype=np.uint64)
        self.amps = np.asarray(amps, dtype=complex)
        self.slot = {}
        self.free = []
        self.nslots = 0

    def bit(self, q):
        if q < self.nd:
            return q
        if q not in self.slot:
            raise KeyError(q)
        return self.slot[q]

    def _alloc(self, q):
        if q < self.nd:
            return q
        if q in self.slot:
            return self.slot[q]
        b = self.free.pop() if self.free else self.nd + self.nslots
        if b == self.nd + self.nslots:
            self.nslots += 1
        self.slot[q] = b
        assert b < 64
        return b

    def _bits(self, b):
        return ((self.keys >> np.uint64(b)) & np.uint64(1)).astype(np.int64)

    def _merge(self):
        u, inv = np.unique(self.keys, return_inverse=True)
        re = np.bincount(inv, weights=self.amps.real, minlength=len(u))
        im = np.bincount(inv, weights=self.amps.imag, minlength=len(u))
        a = re + 1j * im
        keep = np.abs(a) > 1e-12
        self.keys, self.amps = u[keep], a[keep]

    def X(self, q):
        self.keys = self.keys ^ np.uint64(1 << self.bit(q))

    def Z(self, q):
        self.amps = self.amps * (1 - 2 * self._bits(self.bit(q)))

    def Y(self, q):
        self.Z(q)
        self.X(q)
        self.amps = self.amps * 1j

    def H(self, q):
        b = self.bit(q)
        m = np.uint64(1 << b)
        v = self._bits(b)
        k0 = self.keys & ~m
        a = self.amps / np.sqrt(2)
        self.keys = np.concatenate([k0, k0 | m])
        self.amps = np.concatenate([a, a * (1 - 2 * v)])
        self._merge()

    def diag(self, qs, theta):
        bits = [self._bits(self.bit(q)) for q in qs]
        ph = np.array([theta(t) for t in zip(*bits)], dtype=float) if len(self.keys) else np.zeros(0)
        self.amps = self.amps * self.OMEGA ** ph

    def CX(self, c, t):
        vc = self._bits(self.bit(c)).astype(np.uint64)
        self.keys = self.keys ^ (vc << np.uint64(self.bit(t)))

    def measure_z(self, q, forced):
        b = self.bit(q)
        v = self._bits(b)
        p1 = float(np.sum(np.abs(self.amps[v == 1]) ** 2))
        prob = p1 if forced else 1 - p1
        keep = v == forced
        self.keys, self.amps = self.keys[keep], self.amps[keep]
        if prob > 1e-12:
            self.amps = self.amps / np.sqrt(prob)
        return prob

    def release(self, q, outcome):
        if q >= self.nd:
            if outcome:
                self.X(q)
            self.free.append(self.slot.pop(q))

    def reset(self, q, basis, rng):
        b = self._alloc(q)
        if q < self.nd:                      # data reset: measure and flip to |0>
            v = self._bits(b)
            p1 = float(np.sum(np.abs(self.amps[v == 1]) ** 2))
            o = int(rng.random() < p1)
            self.measure_z(q, o)
            if o:
                self.X(q)
        if basis == 'X':
            self.H(q)

    def run_moment(self, mom, errs, outcomes, rng):
        """Execute a time step with the given sampled errors; measurements forced to `outcomes`
        (list in the order of the moment's measurements).  Returns list of probabilities."""
        for when, ps, qs in errs:
            if when == 'pre':
                for pc, q in zip(ps, qs):
                    if pc != 'I':
                        getattr(self, pc)(q)
        probs = []
        k = 0
        for op in mom:
            nm, qs = op[0], op[1:]
            if nm in DIAG_THETA:
                self.diag(qs, DIAG_THETA[nm])
            elif nm == 'CX':
                self.CX(*qs)
            elif nm == 'CZ':
                self.diag(qs, lambda x: 4 * x[0] * x[1])
            elif nm in ('S', 'SDG'):
                self.diag(qs, (lambda x: 2 * x[0]) if nm == 'S' else (lambda x: 6 * x[0]))
            elif nm in ('X', 'Z', 'Y'):
                getattr(self, nm)(qs[0])
            elif nm in ('RX', 'RZ'):
                self.reset(qs[0], nm[1], rng)
            elif nm in ('MX', 'MZ', 'MZ0'):
                q = qs[0]
                if nm == 'MX':
                    self.H(q)
                o = outcomes[k]
                k += 1
                probs.append(self.measure_z(q, o))
                if q >= self.nd:
                    self.release(q, o)
            elif nm == 'FLIPX':
                pass
            else:
                raise ValueError(nm)
        for when, ps, qs in errs:
            if when == 'post':
                for pc, q in zip(ps, qs):
                    if pc != 'I':
                        getattr(self, pc)(q)
        return probs

    def expect_logical(self, circ):
        """<Psi| X^t D_Delta |Psi> on the data qubits (all ancillas released)."""
        assert not self.slot
        x = self.keys
        ph = np.full(len(x), circ.Delta_c0, dtype=float)
        bits = {}
        get = lambda q: bits.setdefault(q, ((x >> np.uint64(q)) & np.uint64(1)).astype(np.int64))
        for q, a in circ.Delta_a.items():
            ph += 2 * a * get(q)
        for (i, j), b in circ.Delta_b.items():
            ph += 4 * b * get(i) * get(j)
        Da = self.amps * self.OMEGA ** ph
        tm = np.uint64(sum(1 << q for q in circ.t_log))
        k2 = x ^ tm
        order = np.argsort(x)
        xs = x[order]
        pos = np.searchsorted(xs, k2)
        pos[pos >= len(xs)] = 0
        hit = xs[pos] == k2
        amp_t = np.zeros(len(x), dtype=complex)
        amp_t[hit] = self.amps[order][pos[hit]]
        return complex(np.sum(np.conj(amp_t) * Da))

In [ ]:
# ------------------------------------------------------------------ compiled protocol (fast Monte Carlo)
_KIND = {'X': 0, 'Z': 1, 'D1': 2, 'D2': 3, 'D3': 4}
_STIM1 = {'H': 'H', 'S': 'S', 'SDG': 'S_DAG', 'X': 'X', 'Y': 'Y', 'Z': 'Z'}     # single-qubit Cliffords -> stim


class CompiledMoment:
    __slots__ = ('ops', 'base', 'meas', 'cx_c', 'cx_t', 'resets', 'clears', 'nonc', 'loc_slice')


class CompiledProtocol:
    """Pre-compiled time steps of a D4SwitchingCircuit + flattened noise-location table."""

    def __init__(self, circ, noise):
        self.circ, self.noise = circ, noise
        locs = []
        self.blocks = []
        live = set()
        for blk in circ.blocks:
            moms = []
            for mom in blk['moms']:
                cm = CompiledMoment()
                cm.ops = mom
                lines, meas, cxc, cxt, rs, cl, nc = [], [], [], [], [], [], []
                for op in mom:
                    nm, qs = op[0], op[1:]
                    if nm in NONCLIFF:
                        nc.append((nm, qs))
                    elif nm == 'FLIPX':
                        pass
                    elif nm == 'CX':
                        lines.append('CX %d %d' % qs)
                        cxc.append(qs[0]); cxt.append(qs[1])
                    elif nm == 'CZ':
                        lines.append('CZ %d %d' % qs)
                    elif nm in ('RX', 'RZ'):
                        lines.append(('RX %d' if nm == 'RX' else 'R %d') % qs[0])
                        rs.append(qs[0])
                    elif nm in ('MX', 'MZ', 'MZ0'):
                        lines.append(('MX %d' if nm == 'MX' else 'M %d') % qs[0])
                        meas.append((qs[0], 'X' if nm == 'MX' else 'Z'))
                        if nm == 'MZ0':
                            cl.append(qs[0])
                    else:
                        lines.append('%s %d' % (_STIM1[nm], qs[0]))
                cm.base = stim.Circuit('\n'.join(lines)) if lines else None
                cm.meas = meas
                cm.cx_c, cm.cx_t = np.array(cxc, dtype=np.int64), np.array(cxt, dtype=np.int64)
                cm.resets, cm.clears = np.array(rs, dtype=np.int64), np.array(cl, dtype=np.int64)
                cm.nonc = nc
                touched = {q for op in mom for q in op[1:]}
                idle = [q for q in blk['data'] if q not in touched] + [q for q in live if q not in touched]
                if all(op[0] in ('NCCZ', 'FLIPX') for op in mom):
                    idle = []          # virtual moment of the deferred switch-out (no physical time step)
                ml = [] if blk.get('noiseless') else noise.locations(mom, idle)
                s0 = len(locs)
                locs += ml
                cm.loc_slice = (s0, len(locs))
                for op in mom:
                    if op[0] in ('RX', 'RZ') and op[1] >= circ.nd:
                        live.add(op[1])
                    if op[0] in ('MX', 'MZ'):
                        live.discard(op[1])
                moms.append(cm)
            self.blocks.append(dict(name=blk['name'], code=blk['code'], moms=moms))
        self.loc_when = np.array([0 if l[0] == 'pre' else 1 for l in locs], dtype=np.int8)
        self.loc_kind = np.array([_KIND[l[1]] for l in locs], dtype=np.int8)
        self.loc_p = np.array([l[3] for l in locs])
        self.loc_q = [l[2] for l in locs]
        self.n_loc = len(locs)


class FastShot:
    """One Monte Carlo shot of the Pauli-proxy circuit (same semantics as ProxyRun, compiled)."""
    _G1 = {'S': 's', 'SDG': 's_dag', 'Z': 'z'}

    def __init__(self, comp, rng, forced=None):
        self.comp, self.rng, self.circ = comp, rng, comp.circ
        self.sim = stim.TableauSimulator(seed=int(rng.integers(2 ** 62)))
        self.sim.set_num_qubits(comp.circ.nq)
        self.fx = np.zeros(comp.circ.nq, dtype=np.uint8)
        if forced is not None:                  # {location index: pauli string}
            self.errs = {i: (comp.loc_when[i], ps, comp.loc_q[i]) for i, ps in forced.items()}
            self.n_err = len(forced)
            self.n_dress = 0
            return
        hit = np.nonzero(rng.random(comp.n_loc) < comp.loc_p)[0]
        self.errs = {}
        for i in hit:
            k = comp.loc_kind[i]
            qs = comp.loc_q[i]
            if k == 0:
                ps = 'X'
            elif k == 1:
                ps = 'Z'
            else:
                ch = PAULI_CHOICES[len(qs)]
                ps = ch[rng.integers(len(ch))]
            self.errs[i] = (comp.loc_when[i], ps, qs)
        self.n_err = len(hit)
        self.n_dress = 0

    def _pauli(self, ps, qs):
        sim, fx = self.sim, self.fx
        for pc, q in zip(ps, qs):
            if pc == 'X':
                sim.x(q); fx[q] ^= 1
            elif pc == 'Y':
                sim.y(q); fx[q] ^= 1
            elif pc == 'Z':
                sim.z(q)

    def run_block(self, bi):
        """Run block bi; returns list of (qubit, basis, outcome) of its measurements."""
        blk = self.comp.blocks[bi]
        sim, fx = self.sim, self.fx
        nmeas = 0
        meas = []
        for cm in blk['moms']:
            a, b = cm.loc_slice
            errs = [self.errs[i] for i in range(a, b) if i in self.errs] if b > a and self.errs else []
            for w, ps, qs in errs:
                if w == 0:
                    self._pauli(ps, qs)
            for nm, qs in cm.nonc:
                e = tuple(int(fx[q]) for q in qs)
                if any(e):
                    for gt in dressing(nm, e):
                        if gt[0] == 'CZ':
                            sim.cz(qs[gt[1]], qs[gt[2]])
                        else:
                            getattr(sim, self._G1[gt[0]])(qs[gt[1]])
                        self.n_dress += 1
            if cm.base is not None:
                sim.do_circuit(cm.base)
            if len(cm.cx_c):
                fx[cm.cx_t] ^= fx[cm.cx_c]
            if len(cm.resets):
                fx[cm.resets] = 0
            if len(cm.clears):
                fx[cm.clears] = 0
            for w, ps, qs in errs:
                if w == 1:
                    self._pauli(ps, qs)
            nmeas += len(cm.meas)
            meas += cm.meas
        if not nmeas:
            return []
        rec = sim.current_measurement_record()[-nmeas:]
        return [(q, bb, int(o)) for (q, bb), o in zip(meas, rec)]

    def apply_x(self, qubits):
        for q in qubits:
            self.sim.x(q)
            self.fx[q] ^= 1

    def init_frame(self, zrows, qubits):
        """X frame at the start of the first non-Clifford block (S1).  S0 is a Clifford circuit, so stim's
        state is the true state E|ideal>, |ideal> = FSC|+_L> (all checks +1); the X part of E is any
        solution x of H_Z x = s (s = current Z-check signs): solutions differ by X stabilizers / logical X
        of |ideal>, which act trivially on it, and Algorithm 2 is exact for any such representation."""
        s = []
        for lab, qs in zrows:
            ps = stim.PauliString(self.circ.nq)
            for q in qs:
                ps[q] = 'Z'
            v = self.sim.peek_observable_expectation(ps)
            assert v in (1, -1), 'Z check not deterministic at the end of S0'
            s.append(0 if v == 1 else 1)
        H = np.zeros((len(zrows), len(qubits)), dtype=np.uint8)
        pos = {q: i for i, q in enumerate(qubits)}
        for r, (lab, qs) in enumerate(zrows):
            for q in qs:
                H[r, pos[q]] = 1
        x = f2_solve(H.T, np.array(s, dtype=np.uint8))      # x H^T = s  <=>  H x = s
        assert x is not None
        self.fx[:] = 0
        for i in np.nonzero(x)[0]:
            self.fx[qubits[i]] = 1
        return np.array(s)

    _G = {'CX': 'cx', 'CZ': 'cz', 'S': 's', 'SDG': 's_dag', 'Z': 'z', 'X': 'x', 'Y': 'y'}

    def run_block_lockstep(self, bi, ts, rng):
        """Validation: same as run_block, but op by op, with the exact simulation `ts` (TrueSim) of the
        TRUE non-Clifford circuit run in lockstep with the same errors.  Before every measurement stim's
        peek value (+1/-1 deterministic, 0 random) is compared with the exact probability of the outcome.
        Returns list of (qubit, basis, outcome, peek, exact probability)."""
        blk = self.comp.blocks[bi]
        sim, fx = self.sim, self.fx
        out = []
        for cm in blk['moms']:
            a, b = cm.loc_slice
            errs = [self.errs[i] for i in range(a, b) if i in self.errs]
            terrs = [('pre' if w == 0 else 'post', ps, qs) for w, ps, qs in errs]
            for w, ps, qs in errs:
                if w == 0:
                    self._pauli(ps, qs)
            for nm, qs in cm.nonc:
                e = tuple(int(fx[q]) for q in qs)
                if any(e):
                    for gt in dressing(nm, e):
                        getattr(sim, self._G[gt[0]])(*[qs[i] for i in gt[1:]])
                        self.n_dress += 1
            outs, peeks = [], []
            for op in cm.ops:
                nm, qs = op[0], op[1:]
                if nm in NONCLIFF or nm == 'FLIPX':
                    continue
                if nm in ('RX', 'RZ'):
                    (sim.reset_x if nm == 'RX' else sim.reset_z)(qs[0])
                elif nm in ('MX', 'MZ', 'MZ0'):
                    q = qs[0]
                    if nm == 'MX':
                        peeks.append(sim.peek_x(q)); sim.h(q); o = sim.measure(q); sim.h(q)
                    else:
                        peeks.append(sim.peek_z(q)); o = sim.measure(q)
                    outs.append(int(o))
                else:
                    getattr(sim, self._G[nm])(*qs)
            if len(cm.cx_c):
                fx[cm.cx_t] ^= fx[cm.cx_c]
            if len(cm.resets):
                fx[cm.resets] = 0
            if len(cm.clears):
                fx[cm.clears] = 0
            for w, ps, qs in errs:
                if w == 1:
                    self._pauli(ps, qs)
            probs = ts.run_moment(cm.ops, terrs, outs, rng)
            out += [(q, bb, o, pk, pr) for (q, bb), o, pk, pr in zip(cm.meas, outs, peeks, probs)]
        return out

In [ ]:
# ------------------------------------------------------------------ decoders
def _wt(p):
    p = min(max(p, 1e-12), 0.5 - 1e-9)
    return float(np.log((1 - p) / p))


_XFRAC = {0: 1.0, 1: 0.0, 2: 2 / 3, 3: 8 / 15, 4: 32 / 63}       # prob. that a location puts X/Y on a qubit


class FluxDecoder:
    """Real-time ("just-in-time") decoding of the Pauli Z checks (non-Abelian fluxes = X errors).

    [KZH] step 2 pairs the fluxes "in real time ... whenever the observation time of the syndromes is
    comparable to the separation of the syndromes" with the decoder of [Dav] (not accessible here).
    Implementation used in this study (own construction, documented as such): after every round t the
    space-time matching graph of all rounds <= t (with an open time boundary after round t) is decoded
    by minimum-weight perfect matching (PyMatching); the space-like edges whose round is <= t - delay
    are committed and the corresponding X corrections are applied physically at once (feed-forward).
    Before the first non-Clifford block (end of S0) and before U (end of S1) everything is committed.
    The g readout of the switch-out is decoded together with the g fluxes of S1 and gives the
    corrected configuration used for kappa."""

    def __init__(self, comp, delay=0, gauge_w=0.05):
        circ = comp.circ
        self.comp, self.circ, self.delay, self.gauge_w = comp, circ, delay, gauge_w
        zr = {code: circ.zchecks[code] for code in ('D4', 'FSC')}
        g_rows = [(lab, qs) for lab, qs in zr['D4'] if lab[0] == 'plaq' and lab[2] == 'g']
        # flux rounds
        self.fr = []
        for bi, blk in enumerate(comp.blocks):
            nm = blk['name']
            if nm[0] in ('S0', 'S4', 'FIN') or (nm[0] == 'S1' and nm[2] == 'z'):
                self.fr.append(dict(block=bi, name=nm, rows=zr[blk['code']]))
            elif nm[0] == 'SO':
                self.fr.append(dict(block=bi, name=nm, rows=g_rows))
        off = 0
        for k, f in enumerate(self.fr):
            f['off'] = off
            f['idx'] = {lab: off + i for i, (lab, qs) in enumerate(f['rows'])}
            off += len(f['rows'])
        self.n_nodes = off
        # relation for the SO detectors: prod_{q in g-plaq} Z^g Z^r Z^b (pre-V) = sum of D4 rows
        lat, c = circ.lat, circ.c
        inv = {v: k for k, v in c.items()}
        d4rows = zr['D4']
        H = np.zeros((len(d4rows), circ.nq), dtype=np.uint8)
        for i, (lab, qs) in enumerate(d4rows):
            H[i, qs] = 1
        self.so_rel = {}
        for lab, qs in g_rows:
            v = np.zeros(circ.nq, dtype=np.uint8)
            for q in qs:
                e = inv[q] // 3
                for cc in (R, G, B):
                    raw = lat.q(e, cc)
                    if raw in c:
                        v[c[raw]] ^= 1
            sol = f2_solve(H, v)
            assert sol is not None, 'SO relation not in the D4 Z-check span'
            self.so_rel[lab] = [d4rows[i][0] for i in np.nonzero(sol)[0]]
        # rows containing each qubit, per code
        self.rows_of = {code: {} for code in ('D4', 'FSC')}
        for code in ('D4', 'FSC'):
            for lab, qs in zr[code]:
                for q in qs:
                    self.rows_of[code].setdefault(q, []).append(lab)
        self._error_rates()
        self._build()

    def _error_rates(self):
        """Phenomenological edge probabilities estimated from the circuit-level noise locations."""
        comp = self.comp
        self.p_data = []           # per flux round: dict qubit -> prob of X/Y since the previous flux round
        self.p_meas = []           # per flux round: dict label -> prob that the check outcome is flipped
        prev_block = -1
        for f in self.fr:
            blk = comp.blocks[f['block']]
            pd, pa = {}, {}
            for bj in range(prev_block + 1, f['block'] + 1):
                for cm in comp.blocks[bj]['moms']:
                    a, b = cm.loc_slice
                    for i in range(a, b):
                        fr_ = _XFRAC[comp.loc_kind[i]] * comp.loc_p[i]
                        for q in comp.loc_q[i]:
                            if q < self.circ.nd:
                                pd[q] = pd.get(q, 0) + fr_
                            elif bj == f['block']:
                                pa[q] = pa.get(q, 0) + fr_
            prev_block = f['block']
            pm = {}
            if blk['code']:
                for lab, qs in f['rows']:
                    a = self.circ.P.anc[lab]
                    pm[lab] = pa.get(self.circ.c[a], 0)
            self.p_data.append(pd)
            self.p_meas.append(pm)

    def _build(self):
        """Matching graphs M_t for t = 0..len(fr)-1 (rounds <= t, open boundary after t)."""
        self.mt = []
        nfr = len(self.fr)
        for t in range(nfr):
            m = pymatching.Matching()
            look = {}

            def add(u, v, w, info):
                key = (u, v) if v == -1 else (min(u, v), max(u, v))
                if key in look and look[key][0] <= w:
                    return
                look[key] = (w, info)
            for k in range(t + 1):
                f = self.fr[k]
                code = self.comp.blocks[f['block']]['code'] or 'D4'
                rows = f['rows']
                rowset = {lab for lab, _ in rows}
                # space edges: X on a data qubit before round k
                qrows = {}
                for lab, qs in rows:
                    for q in qs:
                        qrows.setdefault(q, []).append(f['idx'][lab])
                for q, nodes in qrows.items():
                    if k == 0:
                        w = self.gauge_w           # random first-round outcomes of S0 (gauge choice)
                    else:
                        w = _wt(self.p_data[k].get(q, 1e-6))
                    if len(nodes) == 2:
                        add(nodes[0], nodes[1], w, ('s', q, k))
                    elif len(nodes) == 1:
                        add(nodes[0], -1, w, ('s', q, k))
                # time edges to the previous occurrence of the same label
                if k > 0:
                    for lab, qs in rows:
                        prev = self._prev(k, lab)
                        if prev is not None:
                            kp, node_p = prev
                            w = _wt(self.p_meas[kp].get(lab, 1e-6))
                            add(node_p, f['idx'][lab], w, ('t', lab, kp))
            # switch-out round: a measurement error of a check of the last D4 Z round enters the reference
            # of the g-readout detectors (hyper-edge with its next detector); represent it by an edge between
            # the two nodes known at this time and fix the stored reference when it is committed ('m')
            if self.fr[t]['name'][0] == 'SO':
                f = self.fr[t]
                kp = t - 1
                for lab, qs in f['rows']:
                    for l2 in self.so_rel[lab]:
                        if l2 != lab and l2 in self.fr[kp]['idx']:
                            add(self.fr[kp]['idx'][l2], f['idx'][lab], _wt(self.p_meas[kp].get(l2, 1e-6)),
                                ('m', l2, kp))
            # open time boundary: nodes whose label is measured again after t
            for k in range(t + 1):
                f = self.fr[k]
                for lab, qs in f['rows']:
                    if self._next(k, lab) is not None and self._next(k, lab) > t and self._latest(k, lab, t):
                        add(f['idx'][lab], -1, _wt(self.p_meas[k].get(lab, 1e-6) + 1e-9), ('f', lab, k))
            for (u, v), (w, info) in look.items():
                if v == -1:
                    m.add_boundary_edge(u, weight=w)
                else:
                    m.add_edge(u, v, weight=w)
            nn = self.fr[t]['off'] + len(self.fr[t]['rows'])
            self.mt.append((m, {k_: v[1] for k_, v in look.items()}, nn))

    def _prev(self, k, lab):
        for kp in range(k - 1, -1, -1):
            if lab in self.fr[kp]['idx']:
                return kp, self.fr[kp]['idx'][lab]
        return None

    def _next(self, k, lab):
        for kn in range(k + 1, len(self.fr)):
            if lab in self.fr[kn]['idx']:
                return kn
        return None

    def _latest(self, k, lab, t):
        """True if round k holds the latest occurrence of lab among rounds <= t."""
        for kk in range(k + 1, t + 1):
            if lab in self.fr[kk]['idx']:
                return False
        return True

    # ---------------------------------------------------------------- per shot
    def start(self):
        self.last = {}
        self.active = set()
        self.k = -1

    def push_round(self, k, outcomes_by_label=None, readout=None):
        """Add the detection events of flux round k.  outcomes_by_label: dict label -> bit (Z checks);
        readout: dict g-qubit -> bit (switch-out)."""
        f = self.fr[k]
        self.k = k
        if f['name'][0] == 'SO':
            par = {}
            for lab, qs in f['rows']:
                v = 0
                for q in qs:
                    v ^= readout[q]
                ref = 0
                for l2 in self.so_rel[lab]:
                    ref ^= self.last.get(l2, 0)
                par[lab] = v
                if v ^ ref:
                    self.active ^= {f['idx'][lab]}
            self.readout = dict(readout)
            return
        for lab, qs in f['rows']:
            m = outcomes_by_label[lab]
            if m ^ self.last.get(lab, 0):
                self.active ^= {f['idx'][lab]}
            self.last[lab] = m

    def decode(self, flush=False, min_age=None):
        """Decode with all rounds <= current; returns list of committed space corrections (qubit, k).
        Commits space-like edges with round k <= t - delay (flush: k <= t - min_age, default all)."""
        t = self.k
        m, look, nn = self.mt[t]
        if not self.active:
            return []
        syn = np.zeros(m.num_detectors, dtype=np.uint8)
        for u in self.active:
            syn[u] = 1
        edges = m.decode_to_edges_array(syn)
        out = []
        for u, v in edges:
            u, v = int(u), int(v)
            key = (u, -1) if v == -1 else ((v, -1) if u == -1 else (min(u, v), max(u, v)))
            info = look[key]
            if info[0] == 'm' and flush:
                self.active ^= {key[0], key[1]}
                self.last[info[1]] = self.last.get(info[1], 0) ^ 1     # it was a measurement error
                continue
            if info[0] != 's':
                continue
            q, k = info[1], info[2]
            lim = (t - (min_age or 0)) if flush else (t - self.delay)
            if k > lim:
                continue
            out.append((q, k))
            if key[1] == -1:
                self.active ^= {key[0]}
            else:
                self.active ^= {key[0], key[1]}
        return out

    def register_correction(self, q, code):
        """A physical X correction on data qubit q: flip the stored reference of every Z check of the
        current code containing q (as if it had been applied before that check was measured)."""
        for lab in self.rows_of[code].get(q, []):
            if lab in self.last:
                self.last[lab] ^= 1


class ChargeDecoder:
    """Decoding of the X-type checks (Abelian charges = Z errors) after the protocol: minimum-weight
    matching on the detector error model of the Pauli-proxy circuit (stim + PyMatching).  The DEM
    describes the Pauli errors of the circuit; the extra randomness produced by non-Abelian dressings
    is not in the model (it is present in the simulated data).  Output: predicted flip of the logical
    observable (proxy of the logical Clifford stabilizer, including the Pauli-frame strings)."""

    def __init__(self, comp, dem_noise=None, aligned=True):
        circ = comp.circ
        self.comp, self.circ = comp, circ
        nz = dem_noise or comp.noise
        # measurement bookkeeping
        meas_lab = []          # per absolute measurement: (block_name, kind, label or qubit)
        blk_meas = []
        for bi, blk in enumerate(comp.blocks):
            ids = []
            for cm in blk['moms']:
                for q, b in cm.meas:
                    if blk['code']:
                        lab = circ.lab_of[(blk['code'], q)]
                        meas_lab.append((blk['name'], b, lab))
                    else:
                        meas_lab.append((blk['name'], 'R', q))
                    ids.append(len(meas_lab) - 1)
            blk_meas.append(ids)
        self.n_meas = len(meas_lab)
        self.blk_meas = blk_meas
        idx = {}
        for i, (bn, b, lab) in enumerate(meas_lab):
            idx[(bn, lab)] = i
        self.meas_idx = idx
        sup = {code: {lab: set(xs) for lab, xs, quad in circ.xgens[code]} for code in ('D4', 'FSC')}
        names = [blk['name'] for blk in comp.blocks]
        S0 = [n for n in names if n[0] == 'S0']
        S1 = [n for n in names if n[0] == 'S1' and n[2] == 'x']
        S4 = [n for n in names if n[0] in ('S4', 'FIN')]
        dets = []
        fscl = [lab for lab, _, _ in circ.xgens['FSC']]
        d4l = [lab for lab, _, _ in circ.xgens['D4']]
        for j, n in enumerate(S0):
            for lab in fscl:
                dets.append([idx[(n, lab)]] + ([idx[(S0[j - 1], lab)]] if j else []))

        def span_solve(target, basis_labels, code):
            M = np.zeros((len(basis_labels), circ.nq), dtype=np.uint8)
            for i, l in enumerate(basis_labels):
                M[i, list(sup[code][l])] = 1
            v = np.zeros(circ.nq, dtype=np.uint8)
            v[list(target)] = 1
            sol = f2_solve(M, v)
            return None if sol is None else [basis_labels[i] for i in np.nonzero(sol)[0]]
        for j, n in enumerate(S1):
            for lab in d4l:
                if j == 0:
                    if not S0:
                        continue
                    combo = span_solve(sup['D4'][lab], fscl, 'FSC')
                    if combo is None:
                        continue                       # S^g: random first outcome (gauging)
                    dets.append([idx[(n, lab)]] + [idx[(S0[-1], l)] for l in combo])
                else:
                    dets.append([idx[(n, lab)], idx[(S1[j - 1], lab)]])
        lat, c = circ.lat, circ.c
        inv = {v: k for k, v in c.items()}
        for j, n in enumerate(S4):
            first = {}
            for lab in fscl:
                if j == 0:
                    tgt = set(sup['FSC'][lab])
                    for q in list(sup['FSC'][lab]):
                        gq = lat.q(inv[q] // 3, G)
                        if gq in c:
                            tgt ^= {c[gq]}
                    combo = span_solve(tgt, d4l, 'D4')
                    assert combo is not None, 'S4 X check not determined by the last D4 round'
                    first[lab] = [idx[(n, lab)]] + [idx[(S1[-1], l)] for l in combo]
                else:
                    dets.append([idx[(n, lab)], idx[(S4[j - 1], lab)]])
            if j == 0:
                # detector basis across the switch-out: aligned=True uses, at a vertex with separate r and
                # b checks, D_r and D_r + D_b (S^g of the last D4 round cancels in D_r + D_b), so that a g
                # charge created during the last D4 round is a graph-like error, but an r charge then has
                # 4 symptoms; aligned=False (D_r, D_b) has the opposite property.  D4Experiment decodes
                # with both and keeps the lower-weight solution (no single basis makes all graph-like)
                for lab, d in first.items():
                    v, col = lab[1], lab[2]
                    if aligned and col == 'b' and ('X', v, 'r') in first:
                        dets.append(sorted(set(d) ^ set(first[('X', v, 'r')])))
                    else:
                        dets.append(d)
        self.dets = dets
        # logical observable: proxy logical (measured by MPP at the end of the DEM circuit) + frame terms
        tset = set(circ.t_log)
        self.cross = [len(set(fs) & tset) % 2 for fs in circ.frame_strings]
        self.obs_meas = [idx[(S1[-1], lab)] for (lab, op), cr in zip(circ.gens_g, self.cross) if cr]
        self._dem_circuit(nz)
        # sparse detector evaluation
        self.det_ptr = np.cumsum([0] + [len(d) for d in dets])
        self.det_idx = np.concatenate([np.array(d, dtype=np.int64) for d in dets])

    def _dem_circuit(self, nz):
        circ, comp = self.circ, self.comp
        lines = []
        for bi, blk in enumerate(comp.blocks):
            noiseless = blk['name'][0] == 'FIN'
            for cm in blk['moms']:
                pre, post = [], []
                touched = {q for op in cm.ops for q in op[1:]}
                a, b = cm.loc_slice
                for i in range(a, b):
                    k, p, qs = comp.loc_kind[i], comp.loc_p[i], comp.loc_q[i]
                    if k == 0:
                        ln = 'X_ERROR(%.12g) %d' % (p, qs[0])
                    elif k == 1:
                        ln = 'Z_ERROR(%.12g) %d' % (p, qs[0])
                    elif k == 2:
                        ln = 'DEPOLARIZE1(%.12g) %d' % (p, qs[0])
                    elif k == 3:
                        ln = 'DEPOLARIZE2(%.12g) %d %d' % (p, qs[0], qs[1])
                    else:
                        ln = 'DEPOLARIZE1(%.12g) %d %d %d' % (p * 48 / 63, *qs)
                    (pre if comp.loc_when[i] == 0 else post).append(ln)
                lines += pre
                if cm.base is not None:
                    lines.append(str(cm.base))
                lines += post
            lines.append('TICK')
        n = circ.nq
        lp = ['X%d' % q for q in circ.t_log if circ.Delta_a.get(q) != 2]
        lp += ['Y%d' % q for q in circ.t_log if circ.Delta_a.get(q) == 2]
        lp += ['Z%d' % q for q, a in circ.Delta_a.items() if a == 2 and q not in set(circ.t_log)]
        lines.append('MPP ' + '*'.join(lp))
        tot = self.n_meas + 1
        for d in self.dets:
            lines.append('DETECTOR ' + ' '.join('rec[%d]' % (i - tot) for i in d))
        lines.append('OBSERVABLE_INCLUDE(0) rec[-1] ' + ' '.join('rec[%d]' % (i - tot) for i in self.obs_meas))
        self.stim_circuit = stim.Circuit('\n'.join(lines))
        dem = self.stim_circuit.detector_error_model(decompose_errors=True, ignore_decomposition_failures=True,
                                                     approximate_disjoint_errors=True)
        self.matching = pymatching.Matching.from_detector_error_model(dem)
        # graph-like components of the DEM (for the flux-informed re-weighting)
        comp = {}
        for ins in dem.flattened():
            if ins.type != 'error':
                continue
            p = ins.args_copy()[0]
            cur_d, cur_o = [], 0
            parts = []
            for t in ins.targets_copy():
                if t.is_separator():
                    parts.append((tuple(sorted(cur_d)), cur_o)); cur_d, cur_o = [], 0
                elif t.is_relative_detector_id():
                    cur_d.append(t.val)
                elif t.is_logical_observable_id():
                    cur_o ^= 1
            parts.append((tuple(sorted(cur_d)), cur_o))
            for key in parts:
                if not key[0] or len(key[0]) > 2:
                    continue
                q0 = comp.get(key, 0.0)
                comp[key] = q0 * (1 - p) + p * (1 - q0)
        keys = list(comp)
        self.comp_keys = keys
        self.comp_index = {k[0]: i for i, k in enumerate(keys)}
        from scipy.sparse import csc_matrix
        rows, cols = [], []
        for j, (dd, o) in enumerate(keys):
            for dtt in dd:
                rows.append(dtt); cols.append(j)
        nd = self.stim_circuit.num_detectors
        self.H = csc_matrix((np.ones(len(rows), dtype=np.uint8), (rows, cols)), shape=(nd, len(keys)))
        self.O = csc_matrix((np.ones(sum(o for _, o in keys), dtype=np.uint8),
                             ([0] * sum(o for _, o in keys), [j for j, (dd, o) in enumerate(keys) if o])),
                            shape=(1, len(keys)))
        self.w0 = np.array([_wt(comp[k]) for k in keys])
        # detectors containing each measurement
        self.dets_of_meas = {}
        for j, d_ in enumerate(self.dets):
            for i in d_:
                self.dets_of_meas.setdefault(i, []).append(j)

    def detectors(self, outcomes):
        o = np.asarray(outcomes, dtype=np.uint8)[self.det_idx]
        return np.bitwise_xor.reduceat(o, self.det_ptr[:-1]) if len(o) else np.zeros(0, np.uint8)

    def flux_erasures(self, flux_events, w_erase=0.1):
        """Flux-informed re-weighting (our construction, following [KZH]'s description of cleaning up the
        Abelian charges after the fluxes have been paired): a flux on qubit q present during the X-type
        round ('S1', r, 'x') randomizes the outcomes of the D4 stabilizers whose CZ dressing contains q and
        leaves Z errors on the CZ partners with probability 1/2.  The corresponding time-like components
        and the space-like components of the partner Z errors get the weight w_erase."""
        circ = self.circ
        if not hasattr(self, '_cz_of'):
            self._cz_of = {}
            for lab, xs, quad in circ.xgens['D4']:
                for (i, j) in quad:
                    self._cz_of.setdefault(i, []).append((lab, j))
                    self._cz_of.setdefault(j, []).append((lab, i))
            self._stars = {}
            for lab, xs, quad in circ.xgens['D4']:
                for q in xs:
                    self._stars.setdefault(q, []).append(lab)
        er = set()
        for q, r in flux_events:
            blk = ('S1', r, 'x')
            nxt = ('S1', r + 1, 'x')
            for lab, partner in self._cz_of.get(q, []):
                m = self.meas_idx.get((blk, lab))
                if m is None:
                    continue
                dd = tuple(sorted(self.dets_of_meas.get(m, [])))
                if 1 <= len(dd) <= 2 and dd in self.comp_index:
                    er.add(self.comp_index[dd])
                # space-like component: Z on the partner flips the two stars containing it
                st = self._stars.get(partner, [])
                for rr in (blk, nxt):
                    ds = []
                    for l2 in st:
                        m2 = self.meas_idx.get((rr, l2))
                        if m2 is not None:
                            ds += [j for j in self.dets_of_meas.get(m2, []) if self.dets[j][0] == m2]
                    dd = tuple(sorted(ds))
                    if 1 <= len(dd) <= 2 and dd in self.comp_index:
                        er.add(self.comp_index[dd])
        return er

    def predict(self, outcomes, erasures=None, w_erase=0.1):
        """Returns (predicted flip of the logical observable, weight of the matching)."""
        det = self.detectors(outcomes)
        if not erasures:
            pred, w = self.matching.decode(det, return_weight=True)
            return int(pred[0]), float(w)
        w = self.w0.copy()
        w[list(erasures)] = w_erase
        m = pymatching.Matching.from_check_matrix(self.H, weights=w, faults_matrix=self.O)
        pred, wt = m.decode(det, return_weight=True)
        return int(pred[0]), float(wt)


class D4Experiment:
    """Noisy KZH protocol + decoders.  run(shots) returns per-shot logical infidelity samples
    (0, 1/2 or 1; Takada Sec. VI D with the infidelity (1 - <H>)/2 of the output) and diagnostics."""

    def __init__(self, L, R0, Rin, Rout, noise, delay=0, pre_u_age=0, n_pre_u=2, flux_informed=True,
                 quiet_window=False, fsc_schedule='classes'):
        self.pre_u_age = pre_u_age
        self.flux_informed = flux_informed
        self.circ = D4SwitchingCircuit(L, R0, Rin, Rout, n_pre_u=n_pre_u, fsc_schedule=fsc_schedule)
        self.comp = CompiledProtocol(self.circ, noise)
        self.noise = noise
        self.flux = FluxDecoder(self.comp, delay=delay)
        self.charges = [ChargeDecoder(self.comp, aligned=a) for a in (True, False)]
        # diagnostic (not a physical model): no noise in the switching window = the last D4 round, the Z
        # rounds before U, U, the switch-out and the first FSC round, where the single faults that our
        # decoders fail to correct are located (validation N5).  Gives an optimistic bound.  The decoders keep
        # the priors of the full noise model (they are built before the window is silenced).
        self.quiet_window = quiet_window
        win = {('S1', Rin, 'z'), ('S1', Rin, 'x'), ('U',), ('SO',), ('S4', 1)}
        win |= {('S1', r, 'z') for r in range(Rin + 1, Rin + 1 + n_pre_u)}
        self.window = win
        if quiet_window:
            for blk in self.comp.blocks:
                if blk['name'] in win:
                    a, b = blk['moms'][0].loc_slice[0], blk['moms'][-1].loc_slice[1]
                    self.comp.loc_p[a:b] = 0.0
        self.gset = set(self.circ.data_g)

    def shot(self, rng, decode=True, forced=None, ts=None, init=None):
        """One Monte Carlo shot.  forced: {location index or dynamic fault id: Pauli string} = single-fault
        injection (all other noise off); dynamic fault ids are ('SOCZ', i, j) (after the physical switch-out
        CZ on i, j, if it is applied in this shot) and ('SOIDLE', layer, q).  ts: TrueSim run in lockstep
        (validation only; then the returned stats contain the lockstep record)."""
        circ, comp, fl = self.circ, self.comp, self.flux
        fstatic = {k: v for k, v in forced.items() if not isinstance(k, tuple)} if forced is not None else None
        fdyn = {k: v for k, v in forced.items() if isinstance(k, tuple)} if forced is not None else None
        sh = FastShot(comp, rng, fstatic)
        if init is not None:
            init(sh)               # validation: prepared initial state (see lockstep_full)
        lock = []
        fl.start()
        outs = []
        kfr = 0
        m_g_last = None
        flux_events = []
        stats = dict(n_err=sh.n_err, n_corr=0)
        names = [b['name'] for b in comp.blocks]
        last_S0 = max([i for i, n in enumerate(names) if n[0] == 'S0'], default=-1)
        last_S1 = max(i for i, n in enumerate(names) if n[0] == 'S1')          # Z-only round before U
        last_S1x = max(i for i, n in enumerate(names) if n[0] == 'S1' and n[2] == 'x')
        for bi, blk in enumerate(comp.blocks):
            nm = blk['name']
            if nm == ('S1', 1, 'z'):
                sh.init_frame(circ.zchecks['FSC'], circ.data_rb)
            if init is not None and nm[0] == 'S0':
                # validation start (lockstep_full): S0 replaced by the prepared state; its (noiseless,
                # ideal) outcomes are all 0
                res = [(q, bb, 0) for cm in comp.blocks[bi]['moms'] for (q, bb) in cm.meas]
            elif ts is None:
                res = sh.run_block(bi)
            else:
                r5 = sh.run_block_lockstep(bi, ts, rng)
                lock += r5
                res = [(q, bb, o) for q, bb, o, pk, pr in r5]
            outs += [o for (_, _, o) in res]
            if bi == last_S1x:
                gl = {circ.lab_of[('D4', q)]: o for (q, b, o) in res if b == 'X'}
                m_g_last = [gl[lab] for lab, _ in circ.gens_g]
            if nm[0] in ('S0', 'S4', 'FIN') or (nm[0] == 'S1' and nm[2] == 'z'):
                code = blk['code']
                zl = {circ.lab_of[(code, q)]: o for (q, b, o) in res if b == 'Z'}
                fl.push_round(kfr, zl)
                kfr += 1
                if decode:
                    flush = bi in (last_S0, last_S1) or nm[0] == 'FIN'
                    # before U: corrections supported only by the newest round stay undecided (decoded
                    # after the switch-out with the S4 / g-readout data)
                    cc = fl.decode(flush=flush, min_age=self.pre_u_age if bi == last_S1 else 0)
                    corr = [q for q, k in cc]
                    if nm[0] == 'S1':
                        # the flux was present during the X-type round preceding this Z round
                        flux_events += [(q, nm[1] - 1) for q in corr if nm[1] >= 2]
                    sh.apply_x(corr)
                    if ts is not None:
                        for q in corr:
                            ts.X(q)
                    for q in corr:
                        fl.register_correction(q, code)
                    stats['n_corr'] += len(corr)
            elif nm[0] == 'SO':
                readout = {q: o for (q, b, o) in res}
                fl.push_round(kfr, readout=readout)
                kfr += 1
                a_dec = dict(readout)
                phys = []
                if decode:
                    for q, k in fl.decode(flush=True):
                        if q in self.gset:
                            a_dec[q] ^= 1
                        else:
                            phys.append(q)
                    for q in phys:
                        fl.register_correction(q, 'FSC')
                    stats['n_corr'] += len(phys)
                stats['so_cz'] = self._switch_out(sh, readout, a_dec, phys, rng, fdyn,
                                                  noisy=forced is None and not self.quiet_window, ts=ts)
        # Pauli frame of the S^g eigenvalues (last S1 round), then the logical Clifford stabilizer
        for v, m in enumerate(m_g_last):
            if m:
                for q in circ.frame_strings[v]:
                    sh.sim.z(q)
                    if ts is not None:
                        ts.Z(q)
        val = sh.sim.peek_observable_expectation(circ.logical_proxy(sh.fx))
        if ts is not None:
            stats['lockstep'] = lock
            stats['H_true'] = ts.expect_logical(circ)
            stats['H_proxy'] = val
        if decode:
            sols = [ch.predict(outs, ch.flux_erasures(flux_events) if self.flux_informed else None)
                    for ch in self.charges]
            flip = min(sols, key=lambda s: s[1])[0]
        else:
            flip = 0
        val = val * (-1) ** flip
        stats['n_dress'] = sh.n_dress
        return (1 - val) / 2, stats

    def _switch_out(self, sh, readout, a_dec, phys, rng, fdyn=None, noisy=True, ts=None):
        """Physical switch-out after the Z^g readout ([KZH] step 4), in the order of the real protocol:
        (1) remove the virtual correction CZ(Gamma(a_raw)) of the deferred-measurement network (noiseless
            Clifford bookkeeping of the simulation, no physical operation);
        (2) the X corrections decided by the flux decoder (applied as a noiseless Pauli layer, like all
            feed-forward X corrections of this simulation);
        (3) the physical CZ(Gamma(a_dec)) in the fixed schedule circ.so_colour: in every layer each CZ is
            followed by DEPOLARIZE2(p) and every other rb data qubit by DEPOLARIZE1(p_idle) (exact Pauli
            propagation through the later layers, because they are applied in order).
        fdyn: forced dynamic faults (single-fault injection).  Returns the number of physical CZs."""
        circ, sim = self.circ, sh.sim
        ar = np.array([readout[q] for q in circ.data_g], dtype=np.uint8)
        ad = np.array([a_dec[q] for q in circ.data_g], dtype=np.uint8)
        kr = (circ.Gamma.astype(int) @ ar) % 2
        kd = (circ.Gamma.astype(int) @ ad) % 2

        def pairs_of(k):
            cnt = {}
            for v in np.nonzero(k)[0]:
                for pr in circ.gpairs[v]:
                    pr = tuple(sorted(pr))
                    cnt[pr] = cnt.get(pr, 0) ^ 1
            return sorted(pr for pr, c in cnt.items() if c)

        def pauli(ps, qs):
            sh._pauli(ps, qs)
            if ts is not None:
                for pc, q in zip(ps, qs):
                    if pc != 'I':
                        getattr(ts, pc)(q)

        def cz(i, j):
            sim.cz(i, j)
            if ts is not None:
                ts.diag((i, j), lambda x: 4 * x[0] * x[1])
        for (i, j) in pairs_of(kr):                       # (1)
            cz(i, j)
        sh.apply_x(phys)                                  # (2)
        if ts is not None:
            for q in phys:
                ts.X(q)
        gates = pairs_of(kd)                              # (3)
        nl = circ.so_steps - 1
        p, pi = self.noise.p, self.noise.p_idle
        for layer in range(nl):
            here = [pr for pr in gates if circ.so_colour[pr] == layer]
            busy = {q for pr in here for q in pr}
            for (i, j) in here:
                cz(i, j)
            for (i, j) in here:
                if noisy and p > 0 and rng.random() < p:
                    pauli(PAULI_CHOICES[2][rng.integers(15)], (i, j))
                if fdyn and ('SOCZ', i, j) in fdyn:
                    pauli(fdyn[('SOCZ', i, j)], (i, j))
            for q in circ.data_rb:
                if q in busy:
                    continue
                if noisy and pi > 0 and rng.random() < pi:
                    pauli('XYZ'[rng.integers(3)], (q,))
                if fdyn and ('SOIDLE', layer, q) in fdyn:
                    pauli(fdyn[('SOIDLE', layer, q)], (q,))
        return len(gates)

    def run(self, shots, rng, decode=True):
        inf = np.zeros(shots)
        st = []
        for s in range(shots):
            inf[s], d = self.shot(rng, decode)
            st.append(d)
        return inf, st

In [ ]:
# ------------------------------------------------------------------ applicability of Takada et al.
def equal_up_to_phase(A, B, tol=1e-12):
    """True iff A = e^{i phi} B for one global phase (exact operator comparison, NOT elementwise |.|)."""
    A, B = np.asarray(A, complex), np.asarray(B, complex)
    if A.shape != B.shape:
        return False
    k = np.unravel_index(np.argmax(np.abs(B)), B.shape)
    if abs(B[k]) < tol:
        return bool(np.allclose(A, 0, atol=tol))
    ph = A[k] / B[k]
    return bool(abs(abs(ph) - 1) < 1e-9 and np.allclose(A, ph * B, atol=1e-9))


def takada_conditions(L):
    """Algebraic checks of the sufficient conditions of Takada et al. (Definition 2 and Sec. VIII) for
    the D4 code switching protocol on the L x L lattice.  Returns dict name -> (passed, detail)."""
    P = D4Protocol(L)
    lat, d4, fsc = P.lat, P.d4, P.fsc
    res = {}
    # (a) admissible factorisation: every X-type Clifford stabilizer = X factors x CZ factors with disjoint
    #     supports (transversal, so every effectively measured operator stays Hermitian, Eq. (131))
    bad = 0
    for lab, op in d4.xgens:
        qs = list(op.s) + [q for pr in op.quad for q in pr]
        bad += len(qs) != len(set(qs))
    res['factors transversal (X and CZ factors on disjoint qubits)'] = (bad == 0, '%d generators' % len(d4.xgens))
    # (b) Pauli dressings: X/Y errors on the Clifford factors (CZ; S, CZ in the logical stabilizer; S,
    #     CZ dressings) give weight-1 Z dressings; all dressings commute (Eq. (91))
    I2 = np.eye(2); X = np.array([[0, 1], [1, 0]]); Y = np.array([[0, -1j], [1j, 0]]); Zm = np.diag([1, -1])
    CZm = np.diag([1, 1, 1, -1]); Sm = np.diag([1, 1j])
    ok = True
    for E in (X, Y):
        for k in (0, 1):                                                   # error on either qubit of the CZ
            Ek = np.kron(E, I2) if k == 0 else np.kron(I2, E)
            D = Ek @ CZm @ Ek.conj().T @ CZm.conj().T                          # group commutator [E, CZ]
            ok &= equal_up_to_phase(D, np.kron(I2, Zm) if k == 0 else np.kron(Zm, I2))
        D1 = E @ Sm @ E.conj().T @ Sm.conj().T                                 # [E, S]
        ok &= equal_up_to_phase(D1, Zm)
    for E in (Zm,):                                                        # Z errors commute: no dressing
        ok &= equal_up_to_phase(np.kron(E, I2) @ CZm @ np.kron(E, I2).conj().T @ CZm.conj().T, np.eye(4))
    res['dressings of CZ / S by X, Y are single-qubit Z on the partner / same qubit (exact up to a global '
        'phase); Z errors give no dressing'] = (bool(ok), '')
    # (c) effective commutation: group commutators of measured generators are Z-type Paulis in the
    #     persistent Z-stabilizer span (D4 Z checks), Eq. (58) / Lemma 1
    HZr, HZp = f2_rref(d4.HZ)
    nbad, npair = 0, 0
    for (l1, a), (l2, b) in itertools.combinations(d4.xgens, 2):
        lin, const = commutator_affine(a, b)
        npair += 1
        if lin and not f2_in_rowspace(HZr, HZp, lat.vec(lin)):
            nbad += 1
    res['D4 generators: commutators in the Z-stabilizer span'] = (nbad == 0, '%d pairs' % npair)
    # switch-in: S^g measured on FSC|+_L> (x) |0>_g: commutators with the FSC generators and the logical X
    # must lie in the span of the FSC Z checks and the single-qubit Z^g (current Z stabilizers)
    # The state before switch-in, FSC|+_L> (x) |0>_g, is stabilized by the FSC Z checks, the single-qubit
    # Z^g, and by the D4-dressed X-type operators S^r_v, S^b_v (their CZ^{g,.} dressing acts trivially
    # on |0>_g) and a dressed logical X = X^t D (D diagonal Clifford, also trivial on |0>_g).  Takada
    # Sec. VIII B: the generators may be chosen such that the new measurements (S^g) effectively commute
    # or anticommute with them.  Check this choice:
    zin = [rows for lab, rows in lat.z_checks((R, B))] + [{lat.q(e, G)} for e in range(lat.E)]
    Hin = np.array([lat.vec(r) for r in zin], dtype=np.uint8)
    Hir, Hip = f2_rref(Hin)
    gens_in = [op for lab, op in d4.xgens if lab[2] != 'g']
    nbad = 0
    for lab, op in d4.xgens:
        for g in gens_in:
            lin, const = commutator_affine(op, g)
            if lin and not f2_in_rowspace(Hir, Hip, lat.vec(lin)):
                nbad += 1
    # dressed logical X: find D = prod CZ^{c_k} over the CZ pairs of the D4 generators with
    # [X^t D, S] in the D4 Z-stabilizer span for every X-type generator S (linear system over F2)
    t = set(int(q) for q in np.nonzero(fsc.t_log)[0])
    pairs = sorted({pr for _, op in d4.xgens for pr in op.quad})
    rows_eq = []                          # for each generator: commutator linear part as function of c
    HZr2, HZp2 = f2_rref(d4.HZ)
    # basis of the complement of the Z span to express "lin in span" as linear equations
    Npre = f2_nullspace(d4.HZ)            # vectors orthogonal to the span: lin in span <=> N.lin = 0
    A_cols, b_vec = [], None
    base_list = []
    for lab, op in d4.xgens:
        lin0, _ = commutator_affine(PSOp(s=t), op)
        base_list.append(lat.vec(lin0) if lin0 else np.zeros(lat.n, dtype=np.uint8))
    for pr in pairs:
        col = []
        for lab, op in d4.xgens:
            lin, _ = commutator_affine(PSOp(quad=[pr]), op)
            col.append(lat.vec(lin) if lin else np.zeros(lat.n, dtype=np.uint8))
        A_cols.append(col)
    # equations: N @ (base_v + sum_k c_k col_k,v) = 0 for all v
    eqA = np.array([[int((Npre.astype(int) @ A_cols[k][v].astype(int) % 2)[i]) for k in range(len(pairs))]
                    for v in range(len(d4.xgens)) for i in range(Npre.shape[0])], dtype=np.uint8)
    eqb = np.array([int((Npre.astype(int) @ base_list[v].astype(int) % 2)[i])
                    for v in range(len(d4.xgens)) for i in range(Npre.shape[0])], dtype=np.uint8)
    sol = f2_solve(eqA.T, eqb) if eqA.size else np.zeros(0, dtype=np.uint8)
    logical_ok = sol is not None
    if logical_ok:
        D = [pr for pr, cbit in zip(pairs, sol) if cbit]
        Xbar = PSOp(s=t, quad=D)
        # dressing trivial on |0>_g: every CZ of D touches a g qubit
        triv = all(any(q % 3 == G for q in pr) for pr in D)
        for lab, op in d4.xgens:
            lin, const = commutator_affine(Xbar, op)
            if lin and not f2_in_rowspace(Hir, Hip, lat.vec(lin)):
                nbad += 1
        logical_ok = triv
    res['switch-in: S^g vs dressed generators of FSC|+_L>|0>_g (current Z span), dressed logical X exists'] = (
        nbad == 0 and logical_ok, 'logical dressing with %s CZ pairs' % (len(D) if logical_ok else '-'))
    # (d) primary-support exactness (Definition 1): X supports of the X-type generators (and the logical
    #     X) are linearly independent, so no product of them is a non-Pauli operator without X part
    Ssup = np.vstack([d4.S, d4.t_log[None, :]])
    rk = f2_rank(Ssup[:, d4.cols])
    res['primary-support exactness (independent X supports)'] = (rk == Ssup.shape[0], 'rank %d / %d' % (rk, Ssup.shape[0]))
    # (e) the transversal U and the deferred switch-out network consist of CNOTs and diagonal third-level
    #     gates only (Sec. VIII A, Eq. (178)): Pauli / diagonal-Clifford errors stay of that form
    names = {g[0] for g in P.U_gates()}
    res['U = CNOT + diagonal 3rd-level gates (Sec. VIII A)'] = (names <= {'CNOT', 'CS', 'CSDG', 'TDG'}, str(sorted(names)))
    # (f) logical Clifford stabilizer of the output: Clifford (no cubic / odd terms) and effectively
    #     commuting with every FSC X check (dressing = Z string in the FSC Z span with sign +-1)
    circ = D4SwitchingCircuit(L, 1, 1, 1)
    Hf = np.array([[1 if q in qs else 0 for q in range(circ.nq)] for lab, qs in circ.zchecks['FSC']], dtype=np.uint8)
    Hfr, Hfp = f2_rref(Hf)
    nbad = 0
    for lab, xs, quad in circ.xgens['FSC']:
        fx = np.zeros(circ.nq, dtype=np.uint8)
        fx[xs] = 1
        R_ = circ.logical_proxy(fx) * circ.logical_proxy(np.zeros(circ.nq, dtype=np.uint8))
        zpart = np.array([1 if R_[q] in (2, 3) else 0 for q in range(circ.nq)], dtype=np.uint8)
        xpart = np.array([1 if R_[q] in (1, 2) else 0 for q in range(circ.nq)], dtype=np.uint8)
        if xpart.any() or (zpart.any() and not f2_in_rowspace(Hfr, Hfp, zpart)) or R_.sign.imag != 0:
            nbad += 1
    res['output logical stabilizer: Clifford, effectively commutes with FSC X checks'] = (nbad == 0, '')
    return res


def lockstep_validation(L, Rin, Rout, noise, rng, sequential=False, corr_prob=0.05):
    """Run the Pauli-proxy shot (FastShot) and the exact simulation of the TRUE circuit (TrueSim) in
    lockstep from the start of S1 (initial state E|+_L>_FSC|0>_g with a random Pauli E on the FSC data),
    with the same sampled circuit-level errors and random feed-forward X corrections.  Returns dict with
    the number of measurements, the number of mismatches (deterministic vs random, probability of the
    outcome), and the final <H> from both simulations."""
    circ = D4SwitchingCircuit(L, 0, Rin, Rout, sequential=sequential)
    comp = CompiledProtocol(circ, noise)
    P, c, fsc = circ.P, circ.c, circ.P.fsc
    zero = coset_state(fsc, set())
    one = coset_state(fsc, set(int(q) for q in np.nonzero(fsc.t_log)[0]))
    amps = {}
    for st in (zero, one):
        for k, v in st.a.items():
            amps[k] = amps.get(k, 0) + v / np.sqrt(2)
    keys = [sum(1 << c[q] for q in k) for k in amps]
    ts = TrueSim(circ.nd, keys, list(amps.values()))
    sh = FastShot(comp, rng)
    n = circ.nq
    stabs = []
    for lab, xs, quad in circ.xgens['FSC']:
        ps = stim.PauliString(n)
        for q in xs:
            ps[q] = 'X'
        stabs.append(ps)
    for lab, qs in circ.zchecks['FSC']:
        ps = stim.PauliString(n)
        for q in qs:
            ps[q] = 'Z'
        stabs.append(ps)
    ps = stim.PauliString(n)
    for q in circ.t_log:
        ps[q] = 'X'
    stabs.append(ps)
    for q in range(n):
        if q not in circ.data_rb:
            ps = stim.PauliString(n)
            ps[q] = 'Z'
            stabs.append(ps)
    sh.sim.set_state_from_stabilizers(stabs, allow_redundant=True)
    for q in rng.choice(circ.data_rb, 2, replace=False):
        pc = 'XYZ'[rng.integers(3)]
        getattr(ts, pc)(int(q)); getattr(sh.sim, pc.lower())(int(q))
    sh.init_frame(circ.zchecks['FSC'], circ.data_rb)
    nmeas = nbad = nrand = 0
    m_g = None
    for bi, blk in enumerate(comp.blocks):
        res = sh.run_block_lockstep(bi, ts, rng)
        for q, bb, o, pk, pr in res:
            nmeas += 1
            if pk == 0:
                nrand += 1
                nbad += abs(pr - 0.5) > 1e-9
            else:
                nbad += (o != (0 if pk == 1 else 1)) or abs(pr - 1) > 1e-9
        nm = blk['name']
        if nm[0] == 'S1' and nm[2] == 'x':
            gl = {circ.lab_of[('D4', q)]: o for q, bb, o, pk, pr in res if bb == 'X'}
            m_g = [gl[lab] for lab, _ in circ.gens_g]
        if corr_prob and nm[0] in ('S1', 'S4') and rng.random() < corr_prob * len(blk['moms']):
            q = int(rng.choice(circ.data_rb if nm[0] == 'S4' else circ.data_all))
            sh.apply_x([q]); ts.X(q)
    for v, m in enumerate(m_g):
        if m:
            for q in circ.frame_strings[v]:
                sh.sim.z(q); ts.Z(q)
    true = ts.expect_logical(circ)
    proxy = sh.sim.peek_observable_expectation(circ.logical_proxy(sh.fx))
    return dict(n_meas=nmeas, n_random=nrand, n_mismatch=int(nbad), H_true=true, H_proxy=proxy,
                n_dress=sh.n_dress, agree=bool(nbad == 0 and abs(true - proxy) < 1e-9))


def lockstep_full(L, Rin, Rout, noise, rng, ex=None):
    """Validation of the adaptive protocol AS USED IN PRODUCTION (D4Experiment.shot, from S1 on): real-time
    flux decoding with X corrections, the switch-out with the decoded readout and the physical CZ layers
    with their noise, the Pauli frame and the charge decoder, run in lockstep on the Pauli-proxy circuit
    (stim) and on the exact sparse state vector of the TRUE non-Clifford circuit (TrueSim), with the same
    sampled errors and outcomes.  S0 (a Clifford circuit, simulated exactly by stim alone) is replaced by
    its ideal output E|+_L>_FSC|0>_g with a random 2-qubit Pauli E on the FSC data (noiseless S0 outcomes
    for the decoders; the exact state vector of S0's many parallel ancillas does not fit in memory).  Checks every measurement (proxy
    deterministic <-> exact probability 1, proxy random <-> exact 1/2) and the final <H> before the
    charge-decoder correction."""
    ex = ex or D4Experiment(L, 1, Rin, Rout, noise)
    circ = ex.circ
    P, c, fsc = circ.P, circ.c, circ.P.fsc
    zero = coset_state(fsc, set())
    one = coset_state(fsc, set(int(q) for q in np.nonzero(fsc.t_log)[0]))
    amps = {}
    for st_ in (zero, one):
        for k, v in st_.a.items():
            amps[k] = amps.get(k, 0) + v / np.sqrt(2)
    ts = TrueSim(circ.nd, [sum(1 << c[q] for q in k) for k in amps], list(amps.values()))
    n = circ.nq
    errs = [('XYZ'[rng.integers(3)], int(q)) for q in rng.choice(circ.data_rb, 2, replace=False)]

    def init(sh):
        stabs = []
        for lab, xs, quad in circ.xgens['FSC']:
            ps = stim.PauliString(n)
            for q in xs:
                ps[q] = 'X'
            stabs.append(ps)
        for lab, qs in circ.zchecks['FSC']:
            ps = stim.PauliString(n)
            for q in qs:
                ps[q] = 'Z'
            stabs.append(ps)
        ps = stim.PauliString(n)
        for q in circ.t_log:
            ps[q] = 'X'
        stabs.append(ps)
        for q in range(n):
            if q not in circ.data_rb:
                ps = stim.PauliString(n)
                ps[q] = 'Z'
                stabs.append(ps)
        sh.sim.set_state_from_stabilizers(stabs, allow_redundant=True)
        for pc, q in errs:
            getattr(ts, pc)(q)
            getattr(sh.sim, pc.lower())(q)
    inf, st = ex.shot(rng, ts=ts, init=init)
    nbad = nrand = 0
    for q, bb, o, pk, pr in st['lockstep']:
        if pk == 0:
            nrand += 1
            nbad += abs(pr - 0.5) > 1e-9
        else:
            nbad += (o != (0 if pk == 1 else 1)) or abs(pr - 1) > 1e-9
    ht, hp = complex(st['H_true']), float(st['H_proxy'])
    return dict(n_meas=len(st['lockstep']), n_random=nrand, n_mismatch=int(nbad), H_true=ht.real, H_proxy=hp,
                n_err=st['n_err'], n_corr=st['n_corr'], so_cz=st['so_cz'], infidelity=float(inf),
                agree=bool(nbad == 0 and abs(ht - hp) < 1e-9))


def single_fault_scan(ex, n_sample=None, seeds=2, rng_seed=1, keep_failures=True, only_window=False):
    """Single-fault injection over ALL noise locations of the protocol as simulated: every static
    location of the compiled circuit (gates incl. CCZ / CS / T^dag, resets, measurements, idling) with
    every Pauli of its channel, plus the dynamic switch-out faults (each Pauli after each possible physical
    CZ, idle faults of the CZ layers).  All other noise is off.  Every fault is run with `seeds` different
    measurement-randomness seeds (the outcomes of random measurements, hence the branches, differ); a
    fault 'fails' if any seed gives a nonzero output infidelity.  Not covered: other branches than the
    sampled ones.  only_window: restrict to the faults in the switching window (ex.window).
    Returns dict with the counts, the probability-weighted first-order coefficient
        c1 = sum_f (P(f) / p) * mean_seeds(infidelity)      (eps ~ c1 p for small p, this decoder),
    its split into the switching window (ex.window) and the rest, and (optionally) the failing faults."""
    comp, circ, nz = ex.comp, ex.circ, ex.noise
    p = nz.p
    names = [b['name'] for b in comp.blocks]
    blk_of = np.zeros(comp.n_loc, dtype=int)
    for bi, b in enumerate(comp.blocks):
        for cm in b['moms']:
            a_, b_ = cm.loc_slice
            blk_of[a_:b_] = bi
    faults = []                                   # (fault id, pauli, weight P(f)/p, block name)
    for i in range(comp.n_loc):
        k, qs, pl = comp.loc_kind[i], comp.loc_q[i], comp.loc_p[i]
        if pl == 0:
            continue
        chs = ['X'] if k == 0 else ['Z'] if k == 1 else PAULI_CHOICES[len(qs)]
        for ps in chs:
            faults.append((int(i), ps, pl / p / len(chs), names[blk_of[i]]))
    for pr in circ.so_colour:
        for ps in PAULI_CHOICES[2]:
            faults.append((('SOCZ',) + pr, ps, 1 / 15, ('SO',)))
    if nz.p_idle > 0:
        for layer in range(circ.so_steps - 1):
            for q in circ.data_rb:
                for ps in 'XYZ':
                    faults.append((('SOIDLE', layer, q), ps, nz.p_idle / p / 3, ('SO',)))
    if only_window:                               # exhaustive over the switching window only
        faults = [f for f in faults if f[3] in ex.window]
    r_ = np.random.default_rng(rng_seed)
    n_all = len(faults)
    if n_sample and n_sample < n_all:
        idx = r_.choice(n_all, n_sample, replace=False)
        faults = [faults[j] for j in idx]
        scale = n_all / n_sample                  # unbiased estimate of the full sum
    else:
        scale = 1.0
    c1 = c1_win = 0.0
    nfail = nfail_win = 0
    by_block, fails = {}, []
    for fid, ps, w, bn in faults:
        infs = [ex.shot(np.random.default_rng((rng_seed, s_, zlib.crc32(str(fid).encode()))), forced={fid: ps})[0]
                for s_ in range(seeds)]
        m = float(np.mean(infs))
        c1 += w * m * scale
        inwin = bn in ex.window
        if inwin:
            c1_win += w * m * scale
        if max(infs) > 0:
            nfail += 1
            nfail_win += inwin
            by_block[str(bn)] = by_block.get(str(bn), 0) + 1
            if keep_failures:
                fails.append(dict(fault=str(fid), pauli=ps, block=str(bn), in_window=inwin, infidelities=infs))
    return dict(n_faults_total=n_all, n_tested=len(faults), seeds=seeds, n_fail=nfail, n_fail_window=nfail_win,
                n_fail_outside=nfail - nfail_win, by_block=by_block, c1=c1, c1_window=c1_win,
                c1_outside=c1 - c1_win, failures=fails)


def block_steps(circ, blk):
    """Physical time steps of a block: the switch-out is the g readout plus the fixed CZ-layer schedule
    (its simulation moments FLIPX / NCCZ / MZ0 are a deferred-measurement representation, not time)."""
    return circ.so_steps if blk['name'][0] == 'SO' else len(blk['moms'])

### 雑音付き 15-to-1 ライブラリ（Litinski の誤り予算・密度行列計算・GoSC 配置・表面符号の p_L 較正）

In [ ]:
"""msd_noisy_lib: output error and cost of 15-to-1 magic-state distillation under circuit-level noise.

Error model (D. Litinski, "Magic State Distillation: Not as Costly as You Think", Quantum 3, 205 (2019),
and its published resource-estimation code github.com/litinski/magicstates, Apache-2.0; re-implemented
here, the error budgets are transcribed from that code):
  * the distillation circuit acts on 5 logical qubits (surface-code patches) and consists of Z-type
    pi/8 Pauli-product rotations; a faulty rotation is followed by a P_{pi/2}, P_{-pi/4} or P_{pi/4}
    error with probabilities p1, p2, p3 (faulty magic state: p/3 each; lattice-surgery measurement
    errors and errors in the ancilla region add to p1, p2);
  * patches suffer X / Z storage errors with probability 0.5 * p_L(d) per code cycle each;
  * p_L(p, d) = 0.1 (100 p)^((d+1)/2) per patch and code cycle (GoSC Eq. (10)), or a calibration of the
    same form obtained by stim simulations of the rotated surface code under the SAME circuit-level
    noise model as the D4 simulation (`calibrate_surface_code`);
  * the output error is the infidelity of the post-selected output state (exact density-matrix
    calculation, 128-bit arithmetic), the success probability is the post-selection probability.
Protocols:
  * `litinski2019_one_level(p, dx, dz, dm)` and `litinski2019_two_level(...)`: Litinski's optimised
    factories (transcribed; validated against the outputs stored in his notebook);
  * `gosc_15to1(p, d)` and `gosc_225to1(p, d1, d2)`: the A Game of Surface Codes Sec. 3 layouts
    (Fig. 17c: 11 tiles x 11 tau; Fig. 19: 176 tiles x 15 tau) evaluated with the same error model.
    The per-rotation error budget of these layouts is OUR adaptation of Litinski (2019) to the GoSC
    geometry (single distance d, two ancilla tiles per multi-patch measurement) - see `gosc_steps`.
"""
import numpy as np
import itertools
import mpmath
from mpmath import mp

mp.prec = 128


def plog_gosc(p, d):
    """GoSC Eq. (10) / Litinski (2019): logical error per patch per code cycle."""
    return mpmath.mpf('0.1') * (100 * mpmath.mpf(p)) ** (mpmath.mpf(d + 1) / 2)

In [ ]:
# ------------------------------------------------------------------ 5-qubit density-matrix engine
class DM5:
    """Density matrix of n (=5) qubits.  backend='ld': numpy extended precision (np.clongdouble, ~19 digits,
    fast; used for the searches); backend='mp': numpy object array of mpmath.mpc (128 bits; used to check
    the transcription against Litinski's published outputs down to 1e-25).  Qubit 1 = most significant bit
    (Litinski's kron ordering)."""

    def __init__(self, n=5, singles=None, backend='ld'):
        self.n, self.backend = n, backend
        N = 2 ** n
        idx = np.arange(N)
        self.bits = [(idx >> (n - 1 - q)) & 1 for q in range(n)]
        if singles is None:
            h = 1 / mpmath.sqrt(2) if backend == 'mp' else np.longdouble(1) / np.sqrt(np.longdouble(2))
            singles = [[h, h]] * n
        if backend == 'mp':
            psi = np.array([mpmath.mpc(1)] * N, dtype=object)
            for q, s in enumerate(singles):
                psi = psi * np.array([mpmath.mpc(s[b]) for b in self.bits[q]], dtype=object)
        else:
            psi = np.ones(N, dtype=np.clongdouble)
            for q, s in enumerate(singles):
                psi = psi * np.array([s[b] for b in self.bits[q]], dtype=np.clongdouble)
        self.rho = np.outer(psi, np.conj(psi))

    def _f(self, x):
        return mpmath.mpf(x) if self.backend == 'mp' else np.longdouble(float(x)) if not isinstance(x, np.longdouble) else x

    def _eig(self, axis):
        s = np.ones(2 ** self.n, dtype=int)
        for q, a in enumerate(axis):
            if a:
                s = s * (1 - 2 * self.bits[q])
        return s

    def rot(self, axis, p1=0, p2=0, p3=0):
        """Z-type Pauli-product rotation exp(i pi/8 P) followed by the faulty-rotation channel of
        Litinski's `apply_rot`: with prob p1 / p2 / p3 the angle is 5pi/8 / -pi/8 / 3pi/8."""
        s = self._eig(axis)
        diff = s[:, None] - s[None, :]            # in {-2, 0, 2}
        if self.backend == 'mp':
            p1, p2, p3 = mpmath.mpf(p1), mpmath.mpf(p2), mpmath.mpf(p3)
            w = [(1 - p1 - p2 - p3, mp.pi / 8), (p1, 5 * mp.pi / 8), (p2, -mp.pi / 8), (p3, 3 * mp.pi / 8)]
            m2 = sum(wk * mpmath.expj(2 * ph) for wk, ph in w)
            M = np.where(diff == 2, m2, np.where(diff == -2, mpmath.conj(m2), mpmath.mpf(1)))
        else:
            p1, p2, p3 = (np.longdouble(float(x)) for x in (p1, p2, p3))
            pi = np.longdouble('3.14159265358979323846264338327950288')
            w = [(1 - p1 - p2 - p3, pi / 8), (p1, 5 * pi / 8), (p2, -pi / 8), (p3, 3 * pi / 8)]
            m2 = sum(wk * (np.cos(2 * ph) + 1j * np.sin(2 * ph)) for wk, ph in w)
            M = np.where(diff == 2, m2, np.where(diff == -2, np.conj(m2), 1)).astype(np.clongdouble)
        self.rho = self.rho * M

    def z_err(self, q, p):
        if p == 0:
            return
        p = mpmath.mpf(p) if self.backend == 'mp' else np.longdouble(float(p))
        sg = (1 - 2 * self.bits[q])
        self.rho = self.rho * np.where(sg[:, None] * sg[None, :] == 1, 1, 1 - 2 * p)

    def x_err(self, q, p):
        if p == 0:
            return
        p = mpmath.mpf(p) if self.backend == 'mp' else np.longdouble(float(p))
        f = np.arange(2 ** self.n) ^ (1 << (self.n - 1 - q))
        self.rho = (1 - p) * self.rho + p * self.rho[np.ix_(f, f)]

    def store(self, px, pz):
        for q in range(self.n):
            self.x_err(q, px[q])
            self.z_err(q, pz[q])

    def postselect_x(self, measured):
        """Project the qubits in `measured` onto |+> (X = +1).  Returns (success probability, normalised
        2x2 density matrix of the remaining qubit)."""
        keep = [q for q in range(self.n) if q not in measured]
        assert len(keep) == 1
        k = keep[0]
        out = np.zeros((2, 2), dtype=self.rho.dtype)
        for a in range(2):
            for b in range(2):
                ia = self.bits[k] == a
                ib = self.bits[k] == b
                out[a, b] = self.rho[np.ix_(ia, ib)].sum() / 2 ** len(measured)
        ps = (out[0, 0] + out[1, 1]).real
        return ps, out / ps


def run_steps(steps, n=5, out_qubit=0, init=None, backend='ld'):
    """Evaluate a protocol given as a list of steps
       ('rot', axis (0/1 tuple), p1, p2, p3)   faulty Z-type pi/8 rotation
       ('store', px (list), pz (list))        X / Z Pauli channels
    on |+>^n; the qubits != out_qubit are measured in X and post-selected on +1.  Returns
    (success probability, output infidelity).  The ideal output is taken from the same circuit without
    errors; the infidelity is evaluated as the population <psi_perp|rho|psi_perp> of the orthogonal
    state (no cancellation in 1 - F)."""
    def go(noisy):
        dm = DM5(n, init, backend)
        for st in steps:
            if st[0] == 'rot':
                dm.rot(st[1], *(st[2:5] if noisy else (0, 0, 0)))
            elif st[0] == 'store' and noisy:
                dm.store(st[1], st[2])
        return dm.postselect_x([q for q in range(n) if q != out_qubit])
    ps0, r0 = go(False)
    # ideal output |psi> from the noiseless (pure) reduced state: rho0 = |psi><psi|
    if backend == 'mp':
        a = mpmath.sqrt(r0[0, 0].real)
        psi = [a, r0[1, 0] / a] if a != 0 else [mpmath.mpc(0), mpmath.mpc(1)]
        perp = [-mpmath.conj(psi[1]), mpmath.conj(psi[0])]
    else:
        a = np.sqrt(r0[0, 0].real)
        psi = [a, r0[1, 0] / a]
        perp = [-np.conj(psi[1]), np.conj(psi[0])]
    ps, r = go(True)
    eps = sum(np.conj(perp[i]) * r[i, j] * perp[j] for i in range(2) for j in range(2)).real
    return ps, eps


def infidelity(rho1, ideal1):
    """1 - <ideal|rho|ideal> for a single-qubit rho (2x2) and an ideal ket."""
    f = sum(np.conj(ideal1[a]) * rho1[a, b] * ideal1[b] for a in range(2) for b in range(2))
    return 1 - f.real

In [ ]:
# ------------------------------------------------------------------ Litinski (2019), transcribed
_Z = 1


def _ax(*qs):
    return tuple(int(q in qs) for q in range(5))


def litinski2019_one_level_steps(p, dx, dz, dm, plog=plog_gosc):
    """Error budget of Litinski's (15-to-1)_{dx,dz,dm} (onelevel15to1.py), qubit 1 = output (index 0)."""
    p = mpmath.mpf(p)
    px, pz, pm = plog(p, dx), plog(p, dz), plog(p, dm)
    t = p / 3
    S = []
    single = lambda q: ('rot', _ax(q), t + 0.5 * (dm / dz) * pz * dm, t + 0.5 * dz * pm, t)
    multi = lambda qs, w: ('rot', _ax(*qs), t + 0.5 * pm * dm, t + 0.5 * pm * dm + 0.5 * w * dx / dm * pm, t)
    sx = lambda c0, c: ('store', [c0] + [c] * 4, [0] * 5)
    S += [single(1), single(2), single(3), multi((1, 2, 3), 3 * dz)]
    S += [('store', [0] + [0.5 * (dz / dx) * px * dm] * 3 + [0], [0] + [0.5 * (dx / dz) * pz * dm] * 3 + [0])]
    S += [multi((0, 1, 2), dx + 2 * dz), multi((0, 1, 3), dx + 3 * dz)]
    S += [('store', [0] * 5, [0.5 * ((dx + 2 * dz) + (dx + 3 * dz)) / dx * px * dm, 0, 0, 0, 0])]
    S += [('store', [0.5 * px * dm] + [0.5 * (dz / dx) * px * dm] * 3 + [0],
           [0.5 * px * dm] + [0.5 * (dx / dz) * pz * dm] * 3 + [0])]
    S += [multi((0, 2, 3), dx + 3 * dz), multi((0, 3, 4), dx + 4 * dz), single(4)]
    S += [('store', [0] * 5, [0.5 * ((dx + 3 * dz) + (dx + 4 * dz)) / dx * px * dm, 0, 0, 0, 0])]
    full = lambda a: ('store', [a * px] + [0.5 * (dz / dx) * px * dm] * 4, [a * px] + [0.5 * (dx / dz) * pz * dm] * 4)
    S += [full(0.5 * dm)]
    S += [multi((0, 1, 4), dx + 4 * dz), multi((0, 2, 4), dx + 4 * dz)]
    S += [('store', [0] * 5, [0.5 * ((dx + 4 * dz) + (dx + 4 * dz)) / dx * px * dm, 0, 0, 0, 0])]
    S += [full(0.5 * dm)]
    S += [multi((0, 1, 2, 3, 4), dx + 4 * dz), multi((2, 3, 4), 3 * dz)]
    S += [('store', [0] * 5, [0.5 * (dx + 4 * dz) / dx * px * dm, 0, 0, 0, 0])]
    S += [full(0.5 * (dm + 2 * dx))]
    S += [multi((1, 3, 4), 4 * dz), multi((1, 2, 4), 4 * dz)]
    S += [('store', [0] + [0.5 * (dz / dx) * px * dm] * 4, [0] + [0.5 * (dx / dz) * pz * dm] * 4)]
    return S


def litinski2019_one_level(p, dx, dz, dm, plog=plog_gosc, backend='ld'):
    ps, eps = run_steps(litinski2019_one_level_steps(p, dx, dz, dm, plog), out_qubit=0, backend=backend)
    qubits = 2 * ((dx + 4 * dz) * 3 * dx + 2 * dm)
    cycles = 6 * dm / ps
    return dict(protocol='Litinski2019 (15-to-1)_{%d,%d,%d}' % (dx, dz, dm), eps=float(eps), p_success=float(ps),
                qubits=int(qubits), cycles=float(cycles), volume=float(qubits * cycles), levels=1)


def litinski2019_two_level_steps(p, dx, dz, dm, dx2, dz2, dm2, nl1, plog=plog_gosc, backend='ld'):
    p = mpmath.mpf(p)
    px2, pz2, pm2 = plog(p, dx2), plog(p, dz2), plog(p, dm2)
    ps1, pl1 = run_steps(litinski2019_one_level_steps(p, dx, dz, dm, plog), out_qubit=0, backend=backend)
    ps1, pl1 = mpmath.mpf(float(ps1)) if backend != 'mp' else ps1, mpmath.mpf(float(pl1)) if backend != 'mp' else pl1
    pfail = 1 - ps1
    l1time = max(6 * dm / (mpmath.mpf(nl1) / 2) / (1 - pfail), dm2)
    lmove = 10 * dm2 + mpmath.mpf(nl1) / 4 * (dx + 4 * dz)
    R = lambda qs, w: ('rot', _ax(*qs), pl1 + 0.5 * lmove * pm2,
                       0.5 * lmove * pm2 + 0.5 * w * dx2 / dm2 * pm2, 0)
    st = lambda mask, c0=0: ('store', [c0] + [0.5 * (dz2 / dx2) * px2 * l1time if m else 0 for m in mask],
                             [c0] + [0.5 * (dx2 / dz2) * pz2 * l1time if m else 0 for m in mask])
    zs = lambda v: ('store', [0] * 5, [v, 0, 0, 0, 0])
    a = 0.5 * px2 * l1time
    S = [R((1,), dx2 + dz2 + dm2), R((2,), 3 * dz2 + dm2), st([1, 1, 0, 0]),
         R((3,), dx2 + 3 * dz2 + dm2), R((4,), dz2 + dm2), st([1, 1, 1, 1]),
         R((0, 1, 2), dx2 + 2 * dz2 + dm2), R((1, 2, 3), 4 * dz2 + dm2),
         zs(0.5 * (dx2 + 2 * dz2 + dm2) * dm2 / dx2 * px2), st([1, 1, 1, 1], a),
         R((0, 2, 3), dx2 + 3 * dz2 + dm2), R((0, 1, 3), dx2 + 4 * dz2 + dm2),
         zs(0.5 * ((dx2 + 3 * dz2 + dm2) + (dx2 + 4 * dz2 + dm2)) * dm2 / dx2 * px2), st([1, 1, 1, 1], a),
         R((0, 1, 4), dx2 + 4 * dz2 + dm2), R((0, 3, 4), dx2 + 4 * dz2 + dm2),
         zs(0.5 * 2 * (dx2 + 4 * dz2 + dm2) * dm2 / dx2 * px2), st([1, 1, 1, 1], a),
         R((0, 2, 4), dx2 + 4 * dz2 + dm2), R((0, 1, 2, 3, 4), dx2 + 4 * dz2 + dm2),
         zs(0.5 * 2 * (dx2 + 4 * dz2 + dm2) * dm2 / dx2 * px2), st([1, 1, 1, 1], a),
         R((1, 3, 4), dx2 + 4 * dz2 + dm2), R((2, 3, 4), 3 * dz2 + dm2),
         zs(0.5 * (dx2 + 4 * dz2 + dm2) * dm2 / dx2 * px2), st([1, 1, 1, 1], 0.5 * px2 * (dm2 + 2 * dx2)),
         R((1, 2, 4), 4 * dz2 + dm2), st([1, 1, 0, 1])]
    # note: Litinski's code passes the X/Z storage of qubit 1 only where stated (copied as above)
    return S, ps1, pl1, l1time


def litinski2019_two_level(p, dx, dz, dm, dx2, dz2, dm2, nl1, plog=plog_gosc, backend='ld'):
    S, ps1, pl1, l1time = litinski2019_two_level_steps(p, dx, dz, dm, dx2, dz2, dm2, nl1, plog, backend)
    ps2, eps = run_steps(S, out_qubit=0, backend=backend)
    qubits = 2 * int((dx2 + 4 * dz2) * 3 * dx2 + nl1 * ((dx + 4 * dz) * (3 * dx + dm2 / 2) + 2 * dm)
                     + 20 * dm2 * dm2 + 2 * dx2 * dm2)
    cycles = 7.5 * l1time / ps2
    return dict(protocol='Litinski2019 (15-to-1)x(15-to-1)_{%d,%d,%d;%d,%d,%d;%d}' % (dx, dz, dm, dx2, dz2, dm2, nl1),
                eps=float(eps), p_success=float(ps2), qubits=int(qubits), cycles=float(cycles),
                volume=float(qubits * cycles), levels=2, eps_level1=float(pl1))

In [ ]:
# ------------------------------------------------------------------ surface-code p_L calibration (stim)
def noisy_surface_code(d, rounds, basis, p, p_idle=None):
    """Rotated surface-code memory (stim's generated circuit) with the SAME circuit-level noise model and
    time-step convention as the D4 simulation (noisy_d4_lib.NoiseModel / D4SwitchingCircuit rounds):
    X-type ancillas are prepared / measured with RX / MX (no H layers); stim's combined MR is split into a
    measurement step and a reset step, so that one round = reset, 4 CNOT layers, measurement = 6 time
    steps; 2-qubit gates are followed by DEPOLARIZE2(p), resets by a flip, measurements are preceded by a
    flip, and the idle qubits of every time step are depolarized with p_idle."""
    p_idle = p if p_idle is None else p_idle
    base = stim.Circuit.generated('surface_code:rotated_memory_' + basis, distance=d, rounds=rounds).flattened()
    allq, hq = set(), set()
    for ins in base:
        if ins.name in ('R', 'CX', 'M', 'MR', 'H'):
            allq |= {t.value for t in ins.targets_copy()}
        if ins.name == 'H':
            hq |= {t.value for t in ins.targets_copy()}
    out = stim.Circuit()

    def emit(pre, body, post, touched):
        for e in pre:
            out.append(*e)
        for b in body:
            out.append(*b) if isinstance(b, tuple) else out.append(b)
        for e in post:
            out.append(*e)
        idle = sorted(allq - touched)
        if idle and p_idle > 0:
            out.append('DEPOLARIZE1', idle, p_idle)
        out.append('TICK')

    def flush(layer):
        touched, pre, body, post, resets = set(), [], [], [], []
        for ins in layer:
            nm = ins.name
            ts = [t.value for t in ins.targets_copy() if t.is_qubit_target]
            if nm == 'H':
                continue
            if nm in ('R', 'RX'):
                for q in ts:
                    x = q in hq or nm == 'RX'
                    body.append(('RX' if x else 'R', [q]))
                    post.append(('Z_ERROR' if x else 'X_ERROR', [q], p))
            elif nm in ('MR', 'M', 'MX'):
                # one target per instruction keeps stim's measurement-record order
                for q in ts:
                    x = q in hq or nm == 'MX'
                    pre.append(('Z_ERROR' if x else 'X_ERROR', [q], p))
                    body.append(('MX' if x else 'M', [q]))
                    if nm == 'MR':
                        resets.append(q)
            elif nm == 'CX':
                body.append(('CX', ts))
                post.append(('DEPOLARIZE2', ts, p))
            else:
                body.append(ins)
                continue
            touched |= set(ts)
        if not body:
            return                                   # H-only layer: no time step
        emit(pre, body, post, touched)
        if resets:                                   # the reset half of MR: its own time step
            emit([], [('RX' if q in hq else 'R', [q]) for q in resets],
                 [('Z_ERROR' if q in hq else 'X_ERROR', [q], p) for q in resets], set(resets))
    layer = []
    for ins in base:
        if ins.name == 'TICK':
            flush(layer)
            layer = []
        elif ins.name in ('DETECTOR', 'OBSERVABLE_INCLUDE', 'QUBIT_COORDS', 'SHIFT_COORDS'):
            if layer:
                flush(layer)
                layer = []
            out.append(ins)
        else:
            layer.append(ins)
    if layer:
        flush(layer)
    return out


def surface_code_logical_rate(d, p, shots, rounds=None, seed=0):
    """Logical error per round (X + Z memories summed) of the rotated surface code (MWPM)."""
    rounds = rounds or 3 * d
    tot = 0.0
    out = {}
    for basis in ('x', 'z'):
        c = noisy_surface_code(d, rounds, basis, p)
        m = pymatching.Matching.from_detector_error_model(c.detector_error_model(decompose_errors=True))
        det, obs = c.compile_detector_sampler(seed=seed).sample(shots, separate_observables=True)
        pred = m.decode_batch(det)
        PL = float(np.mean(pred[:, 0] != obs[:, 0]))
        eps = (1 - (1 - 2 * PL) ** (1 / rounds)) / 2 if PL < 0.5 else 0.5
        out[basis] = (PL, eps)
        tot += eps
    return tot, out


def fit_plog(rows):
    """Fit p_L = A (p/p_th)^((d+1)/2) to rows (d, p, p_L) by least squares in log p_L."""
    ds = np.array([r[0] for r in rows], float)
    ps = np.array([r[1] for r in rows], float)
    pl = np.array([r[2] for r in rows], float)
    k = (ds + 1) / 2
    # log pl = log A + k log p - k log pth  -> linear in (log A, log pth)
    A_ = np.stack([np.ones_like(k), -k], axis=1)
    y = np.log(pl) - k * np.log(ps)
    (lA, lpth), *_ = np.linalg.lstsq(A_, y, rcond=None)
    return float(np.exp(lA)), float(np.exp(lpth))


def make_plog(A, pth):
    return lambda p, d: mpmath.mpf(A) * (mpmath.mpf(p) / mpmath.mpf(pth)) ** (mpmath.mpf(d + 1) / 2)

In [ ]:
# ------------------------------------------------------------------ GoSC Sec. 3 layouts with the same model
# the 11 rotations of GoSC Fig. 15 (= msd_lib.fig15_rotations(): columns 5..15 of the 15-to-1 matrix,
# qubit indices 0..4 = qubits 1..5; validated in MSD_15to1_validation M1)
GOSC_ROTATIONS = [(2, 3, 4), (1, 3, 4), (1, 2, 4), (1, 2, 3), (0, 3, 4), (0, 2, 4), (0, 2, 3),
                  (0, 1, 4), (0, 1, 3), (0, 1, 2), (0, 1, 2, 3, 4)]


def gosc_15to1_steps(p, d, plog=plog_gosc, p_in=None, anc_tiles=2, in_channel='pauli'):
    """GoSC Fig. 15 / 17c: qubits 1-4 start in |m> (faulty, error p_in), qubit 5 in |+>; the 11 Z-type
    pi/8 rotations of Fig. 15 (msd_lib.fig15_rotations) are auto-corrected pi/8 rotations of 1 tau = d
    code cycles each; qubits 1-4 are measured in X, qubit 5 is the output.
    Error budget per rotation (our adaptation of Litinski 2019 to the single-distance GoSC layout):
      p1 (P_{pi/2})  = e1 + 0.5 p_L d                        (time-like error of the P x Z_m measurement)
      p2 (P_{-pi/4}) = e2 + 0.5 p_L d + 0.5 anc_tiles p_L d   (space-like errors in the ancilla tiles)
      p3 (P_{pi/4})  = e3
    with (e1, e2, e3) = (p_in/3,)*3 for a raw state with a random Pauli error (in_channel='pauli', GoSC
    Sec. 4 assumption) or (p_in, 0, 0) for a distilled input (in_channel='z', Litinski two-level code);
    every data tile suffers X and Z storage errors 0.5 p_L d per tau."""
    p = mpmath.mpf(p)
    p_in = p if p_in is None else mpmath.mpf(p_in)
    pl = plog(p, d)
    e = (p_in / 3,) * 3 if in_channel == 'pauli' else (p_in, 0, 0)
    S = [('rot', _ax(q), *e) for q in range(4)]               # the four |m> states (0 tau)
    st = ('store', [0.5 * pl * d] * 5, [0.5 * pl * d] * 5)
    for qs in GOSC_ROTATIONS:
        S.append(('rot', _ax(*qs), e[0] + 0.5 * pl * d, e[1] + 0.5 * pl * d + 0.5 * anc_tiles * pl * d, e[2]))
        S.append(st)
    return S


def gosc_15to1(p, d, plog=plog_gosc, backend='ld', **kw):
    """Output error and cost of the GoSC 11-tile 15-to-1 block (Fig. 17c): 11 tiles x 11 tau."""
    ps, eps = run_steps(gosc_15to1_steps(p, d, plog, **kw), out_qubit=4, backend=backend)
    qubits = 11 * 2 * d * d                       # 11 tiles, d^2 data + d^2 measurement qubits each
    cycles = 11 * d / ps
    return dict(protocol='GoSC 15-to-1 (11 tiles x 11 tau), d=%d' % d, eps=float(eps), p_success=float(ps),
                qubits=int(qubits), cycles=float(cycles), volume=float(qubits * cycles), levels=1, d=d)


def gosc_225to1(p, d1, d2, plog=plog_gosc, move_tiles=5, backend='ld', level1=None):
    """GoSC Fig. 19 (176 tiles, 15 tau): 11 level-1 15-to-1 blocks (distance d1) feed a level-2 15-to-1
    whose 15 rotations (incl. the 4 single-qubit ones) consume distilled states.  Level-1 output error
    enters as a Z-type (P_{pi/2}) error (Litinski two-level code); moving a level-1 state through the
    blue ancilla of Fig. 19 adds 0.5 * move_tiles * p_L(d2) * d2 (our assumption, move_tiles = 5)."""
    ps1, e1 = level1 if level1 is not None else run_steps(gosc_15to1_steps(p, d1, plog), out_qubit=4, backend=backend)
    ps1, e1 = mpmath.mpf(float(ps1)), mpmath.mpf(float(e1))
    pl2 = plog(mpmath.mpf(p), d2)
    mv = 0.5 * move_tiles * pl2 * d2
    S = []
    st = ('store', [0.5 * pl2 * d2] * 5, [0.5 * pl2 * d2] * 5)
    for q in range(4):
        S.append(('rot', _ax(q), e1 + mv + 0.5 * pl2 * d2, 0.5 * pl2 * d2 + pl2 * d2, 0))
        S.append(st)
    for qs in GOSC_ROTATIONS:
        S.append(('rot', _ax(*qs), e1 + mv + 0.5 * pl2 * d2, 0.5 * pl2 * d2 + pl2 * d2, 0))
        S.append(st)
    ps2, eps = run_steps(S, out_qubit=4, backend=backend)
    qubits = 2 * (121 * d1 * d1 + 55 * d2 * d2)   # 11 level-1 blocks of 11 tiles + 55 level-2 tiles
    # GoSC Sec. 3.5: each of the 15 level-2 steps needs one level-1 output; when the level-1 block fails
    # (probability 1 - ps1) that 225-to-1 step is skipped, so 15 steps take 15 / ps1 steps on average;
    # 1 step = d2 cycles; the whole protocol is repeated 1 / ps2 times on average
    cycles = (15 / ps1) * d2 / ps2
    return dict(protocol='GoSC 225-to-1 (176 tiles x 15 tau), d1=%d, d2=%d' % (d1, d2), eps=float(eps),
                p_success=float(ps2), qubits=int(qubits), cycles=float(cycles), volume=float(qubits * cycles),
                levels=2, d=d2, d1=d1, eps_level1=float(e1))

In [ ]:
print('library loaded')

## §0. パラメータと入力データ（データの来歴）
* `DATA_MODE`：解析するデータを**明示的に**選ぶ。自動の切り替え（fallback）はしない。
  * `'reference'`（既定）：このノートに埋め込まれた本番データ（リポジトリの `results/` と同一。下の SHA-256 で照合できる）。リポジトリの比較表・図を再現する。
  * `'production'`：`D4_noisy_simulation.ipynb` を `QUICK = False` で実行して得た `results/d4_noisy_mc.csv`, `results/surface_code_plog.csv`。無ければ停止する。
  * `'quick'`：`QUICK = True` の動作確認の結果 `results/*_quick.csv`。無ければ停止する。点数が少ないのでモデルを当てはめられず、§II-2 で停止する（集計の確認まで）。
* Colab ではノートごとに別のランタイムなので、`production` / `quick` を使う場合はシミュレーションのノートで作られた `results/` の 2 つの CSV を、このノートのランタイムの `results/` にアップロードする（または両方のノートで同じ Google Drive のフォルダを作業ディレクトリにする）。
* 入力の SHA-256（先頭 16 桁）、実行 ID（`run_id`）、ショット数を表示し、比較表（`cost_comparison.csv`）の各行にも記録する。
* `P_EVAL`：比較する物理誤り率。`TARGETS`：要求出力誤り。`N_BOOT`：ブートストラップの回数。
* `STEPS_PER_CYCLE`：表面符号 1 ラウンド（code cycle）の時間ステップ数（較正に使った回路と同じ 6：`validation.ipynb` N7）。
* `PLOG_SOURCE`：`'calibrated'`（較正、既定）または `'gosc'`（GoSC 式 (10)：$0.1(100p)^{(d+1)/2}$）。

In [ ]:
DATA_MODE = 'reference'
P_EVAL = [1e-4, 2e-4, 3e-4, 5e-4, 1e-3]
TARGETS = [1e-4, 1e-6, 1e-8, 1e-10]
N_BOOT = 200
SCENARIOS = {'A': 'アイドル雑音あり (p_idle = p)', 'B': 'アイドル雑音なし (p_idle = 0)'}
STEPS_PER_CYCLE = 6
PLOG_SOURCE = 'calibrated'
import io, hashlib, platform
EMBED_MC = """run_id,point_id,chunk_id,seed,scenario,window,d,p,p_idle,kappa0,kappa_in,kappa_out,R0,Rin,Rout,shots,sum_infidelity,n_fail,n_half,mean_errors,mean_so_cz,seconds
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.001|pi0.001|k1,1,1",0,2026163183072,A,noisy,3,0.001,0.001,1.0,1.0,1.0,3,3,3,400,63.0,65,4,11.835,3.95,1.5993270874023438
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.001|pi0.001|k1,1,1",1,2026163183073,A,noisy,3,0.001,0.001,1.0,1.0,1.0,3,3,3,400,74.5,77,5,11.9025,4.035,2.057814836502075
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.001|pi0.001|k1,1,1",2,2026163183074,A,noisy,3,0.001,0.001,1.0,1.0,1.0,3,3,3,400,70.0,72,4,12.1175,3.88,1.5028963088989258
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0005|pi0.0005|k1,1,1",0,2026169983276,A,noisy,3,0.0005,0.0005,1.0,1.0,1.0,3,3,3,400,19.5,20,1,5.9175,3.93,1.5993854999542236
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0005|pi0.0005|k1,1,1",1,2026169983277,A,noisy,3,0.0005,0.0005,1.0,1.0,1.0,3,3,3,400,17.0,18,2,5.8025,4.025,1.3189659118652344
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.001|pi0.001|k1,1,1",0,2026164883123,A,noisy,5,0.001,0.001,1.0,1.0,1.0,5,5,5,400,59.5,62,5,50.2925,16.225,8.211983442306519
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0005|pi0.0005|k1,1,1",2,2026169983278,A,noisy,3,0.0005,0.0005,1.0,1.0,1.0,3,3,3,400,24.5,26,3,5.7475,3.94,1.4740033149719238
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.001|pi0.001|k1,1,1",0,2026166583174,A,noisy,7,0.001,0.001,1.0,1.0,1.0,7,7,7,100,18.5,20,3,133.07,35.6,7.061913013458252
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0005|pi0.0005|k1,1,1",3,2026169983279,A,noisy,3,0.0005,0.0005,1.0,1.0,1.0,3,3,3,400,19.5,21,3,5.805,4.135,1.7266404628753662
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0005|pi0.0005|k1,1,1",4,2026169983280,A,noisy,3,0.0005,0.0005,1.0,1.0,1.0,3,3,3,400,10.5,12,3,6.06,4.135,1.9384870529174805
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.001|pi0.001|k1,1,1",0,2026168283225,A,noisy,9,0.001,0.001,1.0,1.0,1.0,9,9,9,40,10.0,10,0,280.175,64.05,7.238384246826172
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0005|pi0.0005|k1,1,1",5,2026169983281,A,noisy,3,0.0005,0.0005,1.0,1.0,1.0,3,3,3,400,24.5,25,1,6.05,3.915,2.0982611179351807
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.001|pi0.001|k1,1,1",1,2026164883124,A,noisy,5,0.001,0.001,1.0,1.0,1.0,5,5,5,400,76.0,79,6,50.4575,15.9,7.159563779830933
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.001|pi0.001|k1,1,1",1,2026166583175,A,noisy,7,0.001,0.001,1.0,1.0,1.0,7,7,7,100,25.0,26,2,134.31,35.22,6.089454174041748
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0005|pi0.0005|k1,1,1",6,2026169983282,A,noisy,3,0.0005,0.0005,1.0,1.0,1.0,3,3,3,400,12.5,13,1,5.9125,4.1,1.846588373184204
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0005|pi0.0005|k1,1,1",7,2026169983283,A,noisy,3,0.0005,0.0005,1.0,1.0,1.0,3,3,3,400,23.0,24,2,5.8725,4.06,1.5690500736236572
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.001|pi0.001|k1,1,1",1,2026168283226,A,noisy,9,0.001,0.001,1.0,1.0,1.0,9,9,9,40,10.5,11,1,271.35,63.1,5.796316862106323
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0005|pi0.0005|k1,1,1",8,2026169983284,A,noisy,3,0.0005,0.0005,1.0,1.0,1.0,3,3,3,400,18.0,19,2,5.815,4.06,1.5137417316436768
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0005|pi0.0005|k1,1,1",9,2026169983285,A,noisy,3,0.0005,0.0005,1.0,1.0,1.0,3,3,3,400,24.5,25,1,6.0775,4.06,1.2160377502441406
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.001|pi0.001|k1,1,1",2,2026166583176,A,noisy,7,0.001,0.001,1.0,1.0,1.0,7,7,7,100,25.5,26,1,132.68,36.16,6.5817108154296875
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.001|pi0.001|k1,1,1",2,2026164883125,A,noisy,5,0.001,0.001,1.0,1.0,1.0,5,5,5,400,54.0,57,6,50.6625,15.985,8.17550253868103
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.001|pi0.001|k1,1,1",2,2026168283227,A,noisy,9,0.001,0.001,1.0,1.0,1.0,9,9,9,40,9.5,10,1,273.225,65.75,5.0929529666900635
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.001|pi0.001|k1,1,1",3,2026166583177,A,noisy,7,0.001,0.001,1.0,1.0,1.0,7,7,7,100,19.5,20,1,132.87,34.6,6.342412710189819
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0005|pi0.0005|k1,1,1",0,2026171683327,A,noisy,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,16.5,17,1,25.3675,15.745,7.40043830871582
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.001|pi0.001|k1,1,1",3,2026168283228,A,noisy,9,0.001,0.001,1.0,1.0,1.0,9,9,9,40,11.0,12,2,280.675,65.7,6.215469121932983
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.001|pi0.001|k1,1,1",3,2026164883126,A,noisy,5,0.001,0.001,1.0,1.0,1.0,5,5,5,400,71.5,76,9,50.835,15.85,7.19082498550415
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.001|pi0.001|k1,1,1",4,2026166583178,A,noisy,7,0.001,0.001,1.0,1.0,1.0,7,7,7,100,14.5,16,3,132.84,35.44,6.322511196136475
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.001|pi0.001|k1,1,1",4,2026168283229,A,noisy,9,0.001,0.001,1.0,1.0,1.0,9,9,9,40,10.0,10,0,273.1,64.9,5.774221658706665
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0005|pi0.0005|k1,1,1",1,2026171683328,A,noisy,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,8.0,9,2,25.12,16.315,7.949523687362671
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",0,2026173383378,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.5,1,1,66.24,36.12,5.441285610198975
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.001|pi0.001|k1,1,1",5,2026166583179,A,noisy,7,0.001,0.001,1.0,1.0,1.0,7,7,7,100,17.0,17,0,132.83,36.22,6.210883855819702
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.001|pi0.001|k1,1,1",5,2026168283230,A,noisy,9,0.001,0.001,1.0,1.0,1.0,9,9,9,40,4.0,6,4,276.075,63.4,5.589233875274658
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",1,2026173383379,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.5,2,1,65.09,35.34,5.331078290939331
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0005|pi0.0005|k1,1,1",2,2026171683329,A,noisy,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,8.0,9,2,25.31,16.01,7.048199892044067
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.001|pi0.001|k1,1,1",6,2026166583180,A,noisy,7,0.001,0.001,1.0,1.0,1.0,7,7,7,100,21.5,26,9,134.23,36.52,5.99691915512085
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.001|pi0.001|k1,1,1",6,2026168283231,A,noisy,9,0.001,0.001,1.0,1.0,1.0,9,9,9,40,13.0,13,0,276.0,63.85,6.146549224853516
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",2,2026173383380,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,2.0,2,0,65.33,36.58,5.310939311981201
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0005|pi0.0005|k1,1,1",3,2026171683330,A,noisy,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,6.0,7,2,24.825,16.045,6.99460768699646
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.001|pi0.001|k1,1,1",7,2026166583181,A,noisy,7,0.001,0.001,1.0,1.0,1.0,7,7,7,100,20.5,22,3,131.89,35.38,5.64560341835022
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.001|pi0.001|k1,1,1",7,2026168283232,A,noisy,9,0.001,0.001,1.0,1.0,1.0,9,9,9,40,9.0,9,0,270.875,61.0,6.036188125610352
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",3,2026173383381,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,2.0,2,0,66.28,36.68,5.153737306594849
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0005|pi0.0005|k1,1,1",4,2026171683331,A,noisy,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,12.5,13,1,25.2975,15.81,6.918516635894775
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.001|pi0.001|k1,1,1",8,2026166583182,A,noisy,7,0.001,0.001,1.0,1.0,1.0,7,7,7,100,16.5,17,1,133.14,36.96,6.3298680782318115
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",4,2026173383382,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.0,1,0,65.31,35.98,4.831371784210205
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.001|pi0.001|k1,1,1",8,2026168283233,A,noisy,9,0.001,0.001,1.0,1.0,1.0,9,9,9,40,12.5,14,3,279.475,62.7,6.473812580108643
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",5,2026173383383,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,4.5,5,1,66.52,36.86,4.743048667907715
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.001|pi0.001|k1,1,1",9,2026166583183,A,noisy,7,0.001,0.001,1.0,1.0,1.0,7,7,7,100,17.0,18,2,130.15,35.5,6.437046766281128
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0005|pi0.0005|k1,1,1",5,2026171683332,A,noisy,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,5.5,6,1,25.0675,15.715,7.428736209869385
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.001|pi0.001|k1,1,1",9,2026168283234,A,noisy,9,0.001,0.001,1.0,1.0,1.0,9,9,9,40,6.5,7,1,273.325,64.9,6.424477577209473
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",6,2026173383384,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.5,2,1,65.63,36.08,5.1174156665802
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0005|pi0.0005|k1,1,1",6,2026171683333,A,noisy,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,11.0,11,0,25.595,16.12,6.861898899078369
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.001|pi0.001|k1,1,1",10,2026168283235,A,noisy,9,0.001,0.001,1.0,1.0,1.0,9,9,9,40,9.5,10,1,277.3,66.6,5.600525379180908
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",7,2026173383385,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,3.5,4,1,67.13,35.86,5.056771278381348
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.001|pi0.001|k1,1,1",11,2026168283236,A,noisy,9,0.001,0.001,1.0,1.0,1.0,9,9,9,40,6.0,6,0,278.25,64.25,5.511841058731079
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0005|pi0.0005|k1,1,1",7,2026171683334,A,noisy,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,5.0,5,0,25.0675,15.83,6.559880256652832
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",0,2026175083429,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,136.45,63.75,6.654486894607544
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",8,2026173383386,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,65.5,35.46,5.640232801437378
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",1,2026175083430,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,140.875,64.1,5.240606784820557
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.001|pi0.001|k1,1,1",12,2026168283237,A,noisy,9,0.001,0.001,1.0,1.0,1.0,9,9,9,40,7.5,8,1,279.725,63.3,6.5049848556518555
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",9,2026173383387,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,68.21,36.2,5.129402160644531
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0005|pi0.0005|k1,1,1",8,2026171683335,A,noisy,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,6.0,7,2,25.1775,15.815,6.977843284606934
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",2,2026175083431,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,137.55,63.6,5.2918760776519775
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.001|pi0.001|k1,1,1",13,2026168283238,A,noisy,9,0.001,0.001,1.0,1.0,1.0,9,9,9,40,8.0,8,0,277.725,64.1,5.898513078689575
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",10,2026173383388,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,3.0,3,0,66.69,35.52,5.863378286361694
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0005|pi0.0005|k1,1,1",9,2026171683336,A,noisy,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,10.0,11,2,24.9725,15.93,6.371595144271851
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",3,2026175083432,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,138.95,64.25,5.205681324005127
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.001|pi0.001|k1,1,1",14,2026168283239,A,noisy,9,0.001,0.001,1.0,1.0,1.0,9,9,9,40,9.0,9,0,276.575,63.3,6.591931343078613
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",11,2026173383389,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,65.35,35.88,6.556151866912842
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0005|pi0.0005|k1,1,1",10,2026171683337,A,noisy,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,8.0,8,0,25.11,15.935,6.407170295715332
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",4,2026175083433,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,141.375,62.2,6.019532203674316
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",12,2026173383390,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,3.0,3,0,67.1,35.92,5.642038106918335
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.001|pi0.001|k1,1,1",15,2026168283240,A,noisy,9,0.001,0.001,1.0,1.0,1.0,9,9,9,40,10.0,10,0,275.1,63.15,6.108478307723999
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0005|pi0.0005|k1,1,1",11,2026171683338,A,noisy,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,10.0,10,0,25.3975,16.25,6.17050576210022
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",5,2026175083434,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,2.5,3,1,140.125,64.6,5.974650621414185
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",13,2026173383391,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,2.5,3,1,67.67,35.28,4.7490739822387695
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.001|pi0.001|k1,1,1",16,2026168283241,A,noisy,9,0.001,0.001,1.0,1.0,1.0,9,9,9,40,11.0,12,2,278.4,65.1,6.0372254848480225
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0005|pi0.0005|k1,1,1",12,2026171683339,A,noisy,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,10.0,10,0,25.08,15.72,6.979674339294434
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",6,2026175083435,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,137.475,64.4,5.70063853263855
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",14,2026173383392,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,3.0,3,0,67.03,35.8,5.871912240982056
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.001|pi0.001|k1,1,1",17,2026168283242,A,noisy,9,0.001,0.001,1.0,1.0,1.0,9,9,9,40,7.0,7,0,280.075,61.35,6.040611743927002
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",7,2026175083436,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,134.5,63.05,5.007215738296509
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",15,2026173383393,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.5,1,1,66.01,35.16,4.478693723678589
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0005|pi0.0005|k1,1,1",13,2026171683340,A,noisy,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,13.0,13,0,24.815,15.94,6.648428678512573
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.001|pi0.001|k1,1,1",18,2026168283243,A,noisy,9,0.001,0.001,1.0,1.0,1.0,9,9,9,40,10.5,11,1,275.0,65.1,6.3414857387542725
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",8,2026175083437,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,138.75,63.65,5.840289831161499
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",16,2026173383394,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,2.5,3,1,67.55,36.28,5.914303302764893
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0005|pi0.0005|k1,1,1",14,2026171683341,A,noisy,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,8.0,9,2,25.08,16.37,6.721953868865967
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.001|pi0.001|k1,1,1",19,2026168283244,A,noisy,9,0.001,0.001,1.0,1.0,1.0,9,9,9,40,12.0,12,0,273.15,63.45,6.5905232429504395
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",17,2026173383395,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,2.5,3,1,65.87,35.76,4.763309001922607
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",9,2026175083438,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,138.35,65.3,5.7309653759002686
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0005|pi0.0005|k1,1,1",15,2026171683342,A,noisy,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,3.5,4,1,25.305,15.975,8.532187700271606
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",18,2026173383396,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,67.4,36.68,6.123244285583496
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.001|pi0.001|k1,1,1",20,2026168283245,A,noisy,9,0.001,0.001,1.0,1.0,1.0,9,9,9,40,9.0,9,0,282.875,62.75,7.303220272064209
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",10,2026175083439,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,140.35,64.45,6.868194580078125
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0003|pi0.0003|k1,1,1",0,2026176783480,A,noisy,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,3.0,3,0,3.555,3.91,1.6061415672302246
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0003|pi0.0003|k1,1,1",1,2026176783481,A,noisy,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,13.5,14,1,3.5825,3.88,2.380178213119507
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",19,2026173383397,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,2.0,2,0,67.02,35.52,6.51298451423645
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0005|pi0.0005|k1,1,1",16,2026171683343,A,noisy,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,5.5,6,1,25.1225,16.01,7.966556072235107
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0003|pi0.0003|k1,1,1",2,2026176783482,A,noisy,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,6.0,6,0,3.5375,3.925,2.372882604598999
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",11,2026175083440,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,140.35,63.45,7.820518732070923
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0003|pi0.0003|k1,1,1",3,2026176783483,A,noisy,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,8.5,9,1,3.5,3.96,1.8989648818969727
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0003|pi0.0003|k1,1,1",4,2026176783484,A,noisy,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,8.0,8,0,3.5,3.955,2.68450927734375
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0003|pi0.0003|k1,1,1",5,2026176783485,A,noisy,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,11.0,11,0,3.6575,3.88,2.3696200847625732
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",20,2026173383398,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.5,2,1,68.22,35.86,8.619519233703613
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0003|pi0.0003|k1,1,1",6,2026176783486,A,noisy,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,13.5,14,1,3.65,4.02,2.479948043823242
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",12,2026175083441,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,138.475,64.1,8.005561828613281
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0005|pi0.0005|k1,1,1",17,2026171683344,A,noisy,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,12.0,12,0,24.7075,16.14,10.091200590133667
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0003|pi0.0003|k1,1,1",7,2026176783487,A,noisy,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,10.0,10,0,3.63,4.08,2.281708002090454
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0003|pi0.0003|k1,1,1",8,2026176783488,A,noisy,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,5.0,5,0,3.585,4.09,2.4166533946990967
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",21,2026173383399,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.5,2,1,65.95,36.02,8.361802577972412
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0003|pi0.0003|k1,1,1",9,2026176783489,A,noisy,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,7.0,7,0,3.59,4.04,2.1497762203216553
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",13,2026175083442,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,137.65,64.2,8.433956623077393
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0003|pi0.0003|k1,1,1",10,2026176783490,A,noisy,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,9.0,9,0,3.5,3.945,2.451770782470703
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0005|pi0.0005|k1,1,1",18,2026171683345,A,noisy,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,12.0,13,2,25.245,15.875,10.668009519577026
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0003|pi0.0003|k1,1,1",11,2026176783491,A,noisy,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,9.0,9,0,3.5825,3.965,2.2509021759033203
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0003|pi0.0003|k1,1,1",12,2026176783492,A,noisy,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,9.0,9,0,3.635,3.905,2.416207790374756
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",22,2026173383400,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,3.0,3,0,68.17,35.54,7.829989194869995
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0003|pi0.0003|k1,1,1",13,2026176783493,A,noisy,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,7.0,7,0,3.6425,3.995,2.3970541954040527
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",14,2026175083443,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,2.0,2,0,136.125,64.65,8.589159727096558
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0003|pi0.0003|k1,1,1",14,2026176783494,A,noisy,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,8.0,8,0,3.5225,4.22,2.240751028060913
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0003|pi0.0003|k1,1,1",15,2026176783495,A,noisy,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,7.0,7,0,3.535,3.995,2.2883899211883545
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0005|pi0.0005|k1,1,1",19,2026171683346,A,noisy,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,11.0,12,2,25.365,15.935,10.560908794403076
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",23,2026173383401,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.0,1,0,65.51,35.04,7.701725482940674
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0003|pi0.0003|k1,1,1",16,2026176783496,A,noisy,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,5.5,6,1,3.6175,3.96,2.27054500579834
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",15,2026175083444,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,136.65,64.5,7.965218782424927
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0003|pi0.0003|k1,1,1",17,2026176783497,A,noisy,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,10.5,11,1,3.6,4.13,2.239072322845459
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0003|pi0.0003|k1,1,1",18,2026176783498,A,noisy,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,6.5,7,1,3.5325,4.0,2.176645278930664
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0003|pi0.0003|k1,1,1",19,2026176783499,A,noisy,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,7.5,8,1,3.4925,3.92,2.199390172958374
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",24,2026173383402,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,4.5,5,1,66.68,36.34,7.963213682174683
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0005|pi0.0005|k1,1,1",20,2026171683347,A,noisy,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,10.5,11,1,25.7675,16.305,10.478081941604614
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0003|pi0.0003|k1,1,1",20,2026176783500,A,noisy,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,7.0,7,0,3.45,3.835,2.2485969066619873
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",16,2026175083445,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,140.45,63.35,7.706145763397217
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0003|pi0.0003|k1,1,1",21,2026176783501,A,noisy,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,1.0,1,0,3.5175,3.955,2.263380289077759
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0003|pi0.0003|k1,1,1",22,2026176783502,A,noisy,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,3.5,4,1,3.66,3.975,1.531635046005249
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",25,2026173383403,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.0,1,0,65.93,34.96,7.772620916366577
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0003|pi0.0003|k1,1,1",23,2026176783503,A,noisy,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,5.0,5,0,3.495,4.075,2.251601219177246
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0003|pi0.0003|k1,1,1",24,2026176783504,A,noisy,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,8.0,8,0,3.6,3.99,2.2366926670074463
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",17,2026175083446,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,139.25,61.2,8.849502801895142
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0003|pi0.0003|k1,1,1",25,2026176783505,A,noisy,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,3.5,4,1,3.6725,3.955,2.3036887645721436
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",0,2026178483531,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,2.5,3,1,15.12,15.795,10.695608377456665
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0003|pi0.0003|k1,1,1",26,2026176783506,A,noisy,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,5.5,6,1,3.5775,3.9,2.6286733150482178
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",26,2026173383404,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,2.0,2,0,66.77,35.94,7.790203332901001
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",18,2026175083447,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.5,1,1,136.325,63.3,7.4362406730651855
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",1,2026178483532,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.5,2,1,15.3025,16.055,10.095163106918335
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",27,2026173383405,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.0,1,0,65.37,36.32,8.307907819747925
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",19,2026175083448,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,2.0,2,0,134.15,64.5,7.738283395767212
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",0,2026180183582,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.5,1,1,39.3,36.52,7.534251689910889
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",28,2026173383406,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,67.24,35.84,7.917621612548828
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",2,2026178483533,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,2.0,2,0,15.4525,16.03,9.572991609573364
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",20,2026175083449,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,139.075,65.4,8.168633699417114
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",1,2026180183583,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.41,35.68,7.865453720092773
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",29,2026173383407,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,2.0,2,0,65.62,35.48,7.870383977890015
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",2,2026180183584,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.72,35.36,7.154900789260864
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",21,2026175083450,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,136.025,65.05,8.191836833953857
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",3,2026178483534,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.0,1,0,15.075,16.125,10.393146991729736
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",30,2026173383408,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.0,1,0,68.29,36.12,8.01288628578186
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",3,2026180183585,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.86,35.4,7.4570677280426025
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",22,2026175083451,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,138.65,63.55,8.73271131515503
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",4,2026178483535,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,15.035,16.29,9.226537466049194
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",31,2026173383409,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,2.0,2,0,66.99,35.96,8.253233909606934
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",4,2026180183586,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,2.5,3,1,39.46,35.64,7.800858736038208
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",23,2026175083452,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,137.5,66.3,7.626655340194702
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",32,2026173383410,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,66.95,36.3,5.541006803512573
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",5,2026178483536,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,15.245,16.055,8.988674879074097
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",5,2026180183587,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.97,35.86,5.156819105148315
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",24,2026175083453,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,2.5,3,1,140.75,63.0,5.1375086307525635
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",33,2026173383411,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,2.0,2,0,66.74,36.28,4.8212690353393555
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",6,2026180183588,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,39.98,35.52,5.141221284866333
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",6,2026178483537,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,3.5,4,1,15.215,15.945,6.496012449264526
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",25,2026175083454,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,139.875,63.2,4.529304504394531
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",34,2026173383412,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.5,1,1,67.02,36.32,7.214850425720215
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",7,2026180183589,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,38.75,35.72,7.210231304168701
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",7,2026178483538,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,2.5,3,1,15.395,15.69,8.825499296188354
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",26,2026175083455,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,136.55,61.7,8.199450492858887
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",35,2026173383413,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,4.0,4,0,66.64,35.86,7.690057277679443
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",8,2026180183590,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.73,36.56,7.024973392486572
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",27,2026175083456,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,137.975,64.0,8.756442785263062
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",8,2026178483539,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.5,2,1,15.1175,16.045,10.082839488983154
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",36,2026173383414,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.0,1,0,66.8,35.72,7.91709041595459
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",9,2026180183591,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.99,37.46,7.786877393722534
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",28,2026175083457,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,139.475,63.65,8.051371335983276
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",10,2026180183592,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,2.0,2,0,40.85,35.66,6.824056625366211
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",37,2026173383415,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.5,2,1,65.65,35.9,7.742191791534424
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",9,2026178483540,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.0,1,0,15.185,15.67,10.496634721755981
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",29,2026175083458,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,2.0,2,0,140.65,63.65,8.116864681243896
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",11,2026180183593,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,38.84,35.78,6.6954851150512695
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",38,2026173383416,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,3.0,3,0,64.37,35.82,7.733865022659302
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",10,2026178483541,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,4.0,4,0,15.29,15.91,9.524892568588257
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",12,2026180183594,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.63,35.9,7.325904846191406
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",30,2026175083459,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,2.0,2,0,137.275,62.15,8.598584413528442
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",39,2026173383417,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,4.0,4,0,66.65,35.48,8.090754270553589
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",13,2026180183595,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.5,1,1,40.66,36.24,7.290642738342285
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",11,2026178483542,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,3.0,3,0,15.125,15.72,10.684124946594238
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",31,2026175083460,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,138.875,62.0,8.590910911560059
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",40,2026173383418,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,3.0,4,2,65.44,36.36,8.200129985809326
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",14,2026180183596,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,38.72,35.06,7.975282669067383
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",12,2026178483543,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,4.0,5,2,15.235,15.83,10.390008211135864
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",32,2026175083461,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,2.0,2,0,138.775,62.15,9.213609218597412
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",41,2026173383419,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,2.5,3,1,66.06,34.96,8.016772270202637
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",15,2026180183597,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,39.36,34.74,7.5338194370269775
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",42,2026173383420,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,66.53,35.76,7.71874213218689
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",33,2026175083462,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,139.025,63.3,8.252586841583252
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",13,2026178483544,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.5,2,1,15.135,16.075,10.269022464752197
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",16,2026180183598,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.95,35.36,6.352908611297607
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",43,2026173383421,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,3.0,3,0,67.08,35.92,7.9039435386657715
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",34,2026175083463,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,138.65,63.25,8.36369252204895
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",17,2026180183599,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.5,1,1,40.39,36.48,7.315327882766724
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",14,2026178483545,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,2.0,2,0,15.1025,16.145,10.607717514038086
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",44,2026173383422,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.5,2,1,67.14,36.28,8.071385383605957
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",35,2026175083464,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,139.425,65.45,8.368907690048218
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",18,2026180183600,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,39.62,36.46,6.807900905609131
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",15,2026178483546,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,2.0,2,0,15.45,16.2,10.824414730072021
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",45,2026173383423,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,2.0,2,0,66.89,35.96,8.122828245162964
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",19,2026180183601,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,39.67,36.54,7.038560390472412
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",36,2026175083465,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.5,1,1,134.225,63.0,7.962398529052734
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",20,2026180183602,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.03,36.06,7.33088231086731
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",46,2026173383424,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,66.55,36.32,8.051303386688232
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",16,2026178483547,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,14.9725,15.93,10.185774803161621
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",37,2026175083466,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,135.775,65.05,8.746052265167236
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",21,2026180183603,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,38.43,35.96,8.04850959777832
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",47,2026173383425,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,65.93,36.52,7.901119709014893
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",38,2026175083467,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,138.85,63.05,9.028774738311768
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",17,2026178483548,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.5,2,1,15.46,16.045,10.619033813476562
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",22,2026180183604,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,38.85,36.18,7.1833884716033936
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",48,2026173383426,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.0,1,0,64.7,36.24,8.00986361503601
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",39,2026175083468,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,136.625,63.4,8.3920578956604
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",18,2026178483549,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,3.0,3,0,15.215,15.565,10.412122011184692
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",49,2026173383427,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.5,2,1,65.31,35.38,6.743541717529297
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",23,2026180183605,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.15,36.18,8.177383422851562
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",40,2026175083469,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,2.0,2,0,137.325,62.3,6.5621116161346436
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",50,2026173383428,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,67.29,36.08,4.566798210144043
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",24,2026180183606,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.08,36.28,5.438906908035278
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",19,2026178483550,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,3.0,3,0,15.4625,16.075,7.171216249465942
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",41,2026175083470,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,136.4,62.6,5.64605188369751
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",51,2026173383429,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.0,1,0,65.13,36.28,5.598504543304443
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",25,2026180183607,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.73,36.46,4.912973880767822
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",20,2026178483551,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,2.5,3,1,15.0225,15.86,6.919499158859253
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",42,2026175083471,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,2.0,2,0,133.6,65.5,5.8504109382629395
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",26,2026180183608,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.26,36.58,5.182452917098999
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",52,2026173383430,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.0,1,0,65.62,35.78,5.597063302993774
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",27,2026180183609,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.05,36.3,4.3616156578063965
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",43,2026175083472,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,138.225,67.05,5.916983127593994
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",21,2026178483552,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,4.0,4,0,15.3275,15.905,6.8341333866119385
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",53,2026173383431,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,2.0,2,0,65.62,35.86,6.432465553283691
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",28,2026180183610,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.39,36.8,4.615811586380005
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",44,2026175083473,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,139.3,65.15,4.52243447303772
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",22,2026178483553,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,4.5,5,1,14.8925,16.19,6.101979732513428
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",54,2026173383432,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,2.0,2,0,66.9,35.16,6.252604722976685
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",29,2026180183611,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.02,36.08,4.125249624252319
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",45,2026175083474,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,134.225,63.65,5.745496034622192
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",30,2026180183612,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.94,35.4,4.088706731796265
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",23,2026178483554,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,3.0,3,0,14.8925,15.76,6.528548240661621
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",55,2026173383433,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,6.5,7,1,66.6,35.78,5.97795033454895
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",46,2026175083475,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,138.5,64.35,5.052825450897217
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",31,2026180183613,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.87,36.3,5.200575351715088
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",56,2026173383434,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,65.7,35.44,5.184296369552612
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",24,2026178483555,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,2.0,2,0,15.145,16.13,6.822000980377197
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",47,2026175083476,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,137.575,64.3,5.716989994049072
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",32,2026180183614,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,40.13,36.68,4.422437429428101
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",57,2026173383435,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.0,1,0,66.54,35.5,4.582324266433716
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",48,2026175083477,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,2.0,2,0,136.3,64.2,5.26490044593811
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",25,2026178483556,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,2.0,2,0,14.8175,15.965,6.972574234008789
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",33,2026180183615,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.31,36.02,4.791346073150635
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",58,2026173383436,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,2.0,2,0,67.35,36.34,5.34616231918335
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",34,2026180183616,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.6,35.84,5.164601802825928
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",49,2026175083478,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.5,1,1,138.775,64.15,6.055113792419434
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",26,2026178483557,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,2.0,2,0,14.84,15.965,6.020049810409546
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",59,2026173383437,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,65.58,35.92,4.983769655227661
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",35,2026180183617,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.33,36.46,4.624802827835083
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",50,2026175083479,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,136.55,62.9,5.6728291511535645
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",60,2026173383438,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,2.0,2,0,67.91,34.68,4.2575459480285645
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",27,2026178483558,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,15.3275,15.455,7.1326093673706055
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",36,2026180183618,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.1,36.42,5.968341588973999
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",61,2026173383439,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,65.35,36.42,4.988492727279663
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",51,2026175083480,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,137.125,64.65,5.767919540405273
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",28,2026178483559,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.5,2,1,15.3025,15.89,6.493571043014526
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",37,2026180183619,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,2.0,2,0,40.78,36.56,4.7146759033203125
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",62,2026173383440,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,3.0,3,0,66.66,35.98,5.460747003555298
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",52,2026175083481,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,137.975,63.7,6.1478517055511475
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",38,2026180183620,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.5,1,1,39.74,36.12,4.5437397956848145
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",29,2026178483560,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,4.0,4,0,15.1925,15.91,6.947497367858887
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",63,2026173383441,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.5,2,1,66.49,36.18,5.1798317432403564
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",53,2026175083482,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,134.9,65.0,5.5982184410095215
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",39,2026180183621,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.2,36.28,5.695009231567383
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",64,2026173383442,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,3.0,3,0,66.63,36.44,5.451192378997803
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",30,2026178483561,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,15.2225,15.995,6.730580568313599
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",54,2026175083483,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,2.5,3,1,139.475,65.55,4.728021860122681
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",40,2026180183622,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.96,35.44,4.885676622390747
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",55,2026175083484,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,136.575,63.65,4.7256834506988525
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",65,2026173383443,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,2.5,3,1,67.74,36.06,5.6895222663879395
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",31,2026178483562,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,2.5,3,1,15.2075,15.96,6.937859296798706
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",41,2026180183623,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,40.0,36.1,5.19174599647522
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",66,2026173383444,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,2.0,2,0,66.29,35.86,4.715672016143799
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",56,2026175083485,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,138.3,65.1,5.714437246322632
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",42,2026180183624,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.94,36.62,5.1439056396484375
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",32,2026178483563,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,3.0,3,0,15.2525,16.265,6.8307647705078125
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",67,2026173383445,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,2.0,2,0,66.6,35.4,5.279275178909302
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",57,2026175083486,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.5,2,1,136.775,63.45,5.887336015701294
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",43,2026180183625,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,41.75,35.92,4.417768955230713
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",33,2026178483564,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,3.5,4,1,15.07,15.985,5.754980087280273
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",68,2026173383446,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.0,1,0,67.04,35.88,5.812643051147461
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",44,2026180183626,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.23,35.86,4.350738763809204
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",58,2026175083487,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,137.475,63.4,5.881534576416016
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",34,2026178483565,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.0,1,0,15.18,16.21,6.094212770462036
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",69,2026173383447,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,2.0,2,0,65.24,35.6,5.3381593227386475
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",45,2026180183627,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.85,36.56,5.202219009399414
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",59,2026175083488,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,137.875,64.45,5.61385440826416
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",70,2026173383448,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.5,2,1,65.47,36.14,4.864925861358643
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",46,2026180183628,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.06,35.3,4.75336480140686
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",60,2026175083489,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,137.275,63.35,4.7592527866363525
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",35,2026178483566,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.0,1,0,15.225,16.18,7.099633455276489
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",71,2026173383449,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.0,1,0,64.88,35.68,4.695812940597534
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",47,2026180183629,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,39.5,36.42,4.904387712478638
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",61,2026175083490,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,136.525,63.3,5.835804224014282
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",36,2026178483567,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,4.0,4,0,15.39,15.74,6.463404417037964
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",48,2026180183630,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,38.93,35.94,3.8843817710876465
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",72,2026173383450,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.0,1,0,67.03,36.2,6.124234676361084
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",62,2026175083491,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,140.3,63.0,6.295300722122192
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",49,2026180183631,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,40.31,37.06,4.735268592834473
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",37,2026178483568,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,14.9975,16.075,7.21717381477356
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",73,2026173383451,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.0,1,0,66.4,35.9,5.071934938430786
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",50,2026180183632,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.47,36.4,4.853192090988159
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",63,2026175083492,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,136.525,64.8,5.529266119003296
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",74,2026173383452,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,66.32,37.0,5.616392135620117
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",38,2026178483569,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,3.0,3,0,15.2825,15.67,6.861806154251099
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",51,2026180183633,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.87,34.64,4.401766538619995
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",64,2026175083493,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,2.0,2,0,138.0,62.05,5.571795463562012
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",75,2026173383453,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,2.5,3,1,67.09,35.66,5.686567068099976
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",39,2026178483570,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.0,1,0,14.8625,16.505,6.067059755325317
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",52,2026180183634,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.53,36.12,5.227229118347168
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",65,2026175083494,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,138.3,63.3,5.604185581207275
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",76,2026173383454,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,2.0,3,2,66.71,36.28,5.962792158126831
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",53,2026180183635,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,41.09,36.02,4.944231748580933
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",40,2026178483571,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.5,2,1,15.1475,16.105,6.33705735206604
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",66,2026175083495,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,2.0,2,0,138.2,66.1,5.134177207946777
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",54,2026180183636,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,39.89,34.88,4.028098821640015
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",77,2026173383455,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,66.39,35.66,5.509028911590576
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",55,2026180183637,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,39.67,36.0,4.090490102767944
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",67,2026175083496,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,140.1,65.3,5.941087961196899
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",41,2026178483572,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,15.0075,16.175,7.671936750411987
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",78,2026173383456,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,65.14,36.9,4.80409049987793
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",56,2026180183638,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.33,35.5,4.584468841552734
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",68,2026175083497,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,2.0,2,0,136.3,64.55,5.6273674964904785
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",79,2026173383457,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,2.0,2,0,66.17,35.66,5.045222759246826
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",42,2026178483573,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,4.0,4,0,15.005,16.28,6.8741559982299805
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",57,2026180183639,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.99,36.02,4.979504346847534
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",69,2026175083498,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,140.925,64.5,5.667988300323486
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",80,2026173383458,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,4.5,5,1,65.99,36.16,5.677990198135376
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",58,2026180183640,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.62,35.9,4.263143539428711
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",43,2026178483574,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,2.0,2,0,15.6125,16.065,6.409752130508423
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",70,2026175083499,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,139.475,64.4,6.30945348739624
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",81,2026173383459,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.0,1,0,65.3,35.9,5.735301733016968
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",59,2026180183641,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,39.86,35.94,5.075005292892456
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",44,2026178483575,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,4.5,5,1,14.82,15.55,7.390578746795654
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",82,2026173383460,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.5,3,3,65.24,36.14,4.99352240562439
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",71,2026175083500,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,2.0,2,0,136.8,64.3,6.008246898651123
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",60,2026180183642,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,40.71,35.54,5.356457471847534
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",45,2026178483576,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.5,2,1,15.4175,16.195,6.577374696731567
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",83,2026173383461,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,3.5,4,1,66.38,35.82,4.836672306060791
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",61,2026180183643,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,3.0,3,0,39.84,35.3,4.094623804092407
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",72,2026175083501,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,137.575,64.0,6.776436805725098
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",84,2026173383462,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,3.5,4,1,66.27,36.68,5.452178716659546
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",62,2026180183644,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.24,36.3,5.486329078674316
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",46,2026178483577,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,2.0,2,0,15.24,15.525,6.530868768692017
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",73,2026175083502,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,138.725,63.05,5.895959854125977
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",63,2026180183645,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.05,36.08,4.907872200012207
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",85,2026173383463,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,66.03,35.84,5.305283308029175
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",47,2026178483578,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,2.0,2,0,15.015,16.17,7.131768226623535
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",74,2026175083503,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,137.7,66.45,5.393018007278442
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",64,2026180183646,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,40.01,36.02,5.316498279571533
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",86,2026173383464,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,3.5,4,1,67.32,35.62,5.6424806118011475
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",75,2026175083504,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.5,1,1,142.6,65.35,5.635173320770264
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",48,2026178483579,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,2.0,2,0,15.0325,16.395,6.374073028564453
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",65,2026180183647,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.02,36.18,4.441939353942871
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",87,2026173383465,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.0,1,0,67.05,36.26,5.5446577072143555
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",76,2026175083505,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,139.7,61.9,5.635395526885986
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",66,2026180183648,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.0,36.44,4.880783557891846
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",49,2026178483580,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.5,2,1,14.77,16.025,6.574329853057861
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",88,2026173383466,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,2.0,2,0,65.7,35.6,5.805118083953857
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",67,2026180183649,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,40.73,35.44,4.3961546421051025
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",77,2026175083506,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,139.25,63.9,6.1081602573394775
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",50,2026178483581,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.5,2,1,15.295,16.075,6.533459186553955
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",89,2026173383467,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,2.0,2,0,66.85,35.74,5.702904462814331
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",68,2026180183650,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.12,36.36,5.250910043716431
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",78,2026175083507,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,139.2,65.6,5.091390371322632
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",90,2026173383468,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.0,1,0,65.35,36.28,5.0279152393341064
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",69,2026180183651,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.5,35.98,4.5613203048706055
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",51,2026178483582,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,2.0,2,0,15.47,16.15,7.200099229812622
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",79,2026175083508,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,138.75,62.15,5.227787733078003
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",91,2026173383469,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,3.0,3,0,65.04,35.72,5.106202840805054
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",70,2026180183652,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.11,35.66,4.609943389892578
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",52,2026178483583,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,2.5,3,1,15.07,16.19,5.961489200592041
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",80,2026175083509,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,136.775,62.25,6.464274168014526
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",71,2026180183653,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,39.62,36.38,4.775909662246704
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",92,2026173383470,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,66.64,36.3,5.894289970397949
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",53,2026178483584,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.0,1,0,14.855,16.145,6.40495491027832
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",81,2026175083510,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,136.85,64.4,5.404188632965088
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",72,2026180183654,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,2.0,2,0,40.94,35.66,5.004958152770996
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",93,2026173383471,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,2.0,2,0,65.67,36.44,5.041778326034546
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",73,2026180183655,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.98,35.52,4.4685187339782715
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",54,2026178483585,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,6.0,7,2,15.1975,16.145,6.413637161254883
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",82,2026175083511,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,2,2,137.675,64.25,5.976957082748413
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",94,2026173383472,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,3.0,3,0,67.17,36.0,5.735209226608276
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",74,2026180183656,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.12,35.62,4.608817100524902
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",95,2026173383473,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,3.0,3,0,67.13,35.7,4.583767890930176
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",83,2026175083512,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,138.325,66.3,5.834866285324097
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",55,2026178483586,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,2.0,2,0,15.0275,15.745,7.917114973068237
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",75,2026180183657,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.25,36.62,4.543576002120972
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",96,2026173383474,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,2.0,2,0,66.05,36.62,5.1877429485321045
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",84,2026175083513,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,137.85,65.35,5.799665689468384
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",76,2026180183658,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.5,2,1,40.44,36.34,4.667422294616699
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",56,2026178483587,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,6.5,7,1,15.0325,15.94,6.717864274978638
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",97,2026173383475,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.0,1,0,67.58,35.64,5.006431341171265
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",77,2026180183659,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.47,34.86,4.477702856063843
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",85,2026175083514,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,138.975,63.9,6.290144681930542
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",57,2026178483588,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,2.0,2,0,15.6325,16.545,6.757210731506348
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",98,2026173383476,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,2.5,3,1,67.68,36.3,5.6919708251953125
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",78,2026180183660,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.5,1,1,39.47,35.06,4.508905649185181
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",86,2026175083515,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,135.7,64.1,5.908071279525757
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",99,2026173383477,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,2.0,2,0,67.42,36.34,4.864486217498779
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",79,2026180183661,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,39.31,36.34,5.517163276672363
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",58,2026178483589,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.0,1,0,15.4925,16.57,6.4852049350738525
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",87,2026175083516,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,139.925,63.65,5.339833736419678
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",100,2026173383478,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,66.5,35.08,5.852089881896973
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",80,2026180183662,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,40.84,35.4,5.354454517364502
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",59,2026178483590,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,2.5,3,1,15.18,16.11,6.023272514343262
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",88,2026175083517,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,139.0,63.3,4.860352993011475
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",101,2026173383479,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,2.0,2,0,65.23,36.8,5.630921363830566
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",81,2026180183663,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,2,2,40.27,34.88,5.432390213012695
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",89,2026175083518,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,138.8,64.65,5.911094427108765
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",60,2026178483591,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.5,2,1,14.9975,16.61,6.671715021133423
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",82,2026180183664,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.8,35.5,4.705536842346191
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",102,2026173383480,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.5,2,1,67.91,36.88,5.335033416748047
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",90,2026175083519,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,2.0,2,0,140.7,63.85,5.665547132492065
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",61,2026178483592,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,15.1025,16.27,6.497474908828735
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",83,2026180183665,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.31,36.02,5.468026638031006
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",103,2026173383481,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,67.05,37.16,5.33803915977478
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",91,2026175083520,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,140.4,64.5,5.425693511962891
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",84,2026180183666,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.33,34.98,5.083965063095093
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",104,2026173383482,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,3.5,4,1,67.14,35.78,4.980454444885254
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",62,2026178483593,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,3.0,3,0,15.195,15.805,6.762469530105591
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",92,2026175083521,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,139.575,61.3,6.463782072067261
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",85,2026180183667,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,38.73,35.98,4.605864763259888
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",105,2026173383483,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,67.07,35.82,5.308197975158691
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",63,2026178483594,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,2.0,2,0,15.415,16.1,6.803461790084839
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",86,2026180183668,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,40.62,35.88,4.378536939620972
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",93,2026175083522,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,134.4,64.5,5.7091240882873535
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0005|pi0.0005|k1,1,1",106,2026173383484,A,noisy,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.0,1,0,67.09,36.46,4.956261873245239
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",87,2026180183669,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.17,35.1,3.9783594608306885
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",64,2026178483595,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,3.0,4,2,14.815,15.85,7.337753057479858
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",94,2026175083523,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,139.15,62.45,5.80077862739563
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",88,2026180183670,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.41,35.92,4.507551908493042
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",95,2026175083524,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,134.3,64.7,5.4714133739471436
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",89,2026180183671,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.5,2,1,40.76,35.66,3.864201068878174
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",65,2026178483596,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.0,1,0,14.86,16.005,6.961472988128662
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",0,2026181883633,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,80.3,64.6,6.439388036727905
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",96,2026175083525,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,139.625,63.2,4.807026147842407
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",90,2026180183672,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,40.65,35.86,4.631331205368042
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",66,2026178483597,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,2.0,2,0,14.79,16.05,6.864456415176392
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",1,2026181883634,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.925,61.9,4.900158405303955
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",91,2026180183673,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,3.0,3,0,40.65,35.66,4.089603900909424
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",97,2026175083526,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,137.375,66.2,5.348618030548096
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",2,2026181883635,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,80.3,64.3,5.2192628383636475
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",92,2026180183674,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.85,34.72,4.710278034210205
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",67,2026178483598,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,15.295,16.115,7.030996084213257
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",98,2026175083527,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,137.525,65.35,5.70672345161438
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",3,2026181883636,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.75,62.15,4.633999824523926
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",93,2026180183675,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.95,36.12,5.218432426452637
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",68,2026178483599,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,3.5,4,1,15.0525,15.94,7.14518404006958
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",4,2026181883637,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.35,62.9,4.959452390670776
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",99,2026175083528,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,136.85,62.75,6.308449029922485
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",94,2026180183676,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,40.24,36.14,5.027599096298218
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",100,2026175083529,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,140.325,63.55,5.129321813583374
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",69,2026178483600,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,3.5,4,1,14.9,15.88,6.24256443977356
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",5,2026181883638,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.525,63.7,5.737354755401611
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",95,2026180183677,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.95,36.36,6.177226543426514
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",6,2026181883639,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.1,64.5,4.256973743438721
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",101,2026175083530,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,2.0,2,0,138.725,61.25,5.245427370071411
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",70,2026178483601,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,2.0,2,0,15.0975,16.08,6.70301365852356
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",96,2026180183678,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,2.0,2,0,40.04,36.04,4.389000415802002
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",7,2026181883640,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,84.7,66.4,5.330447435379028
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",102,2026175083531,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,140.2,63.4,5.350336790084839
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",97,2026180183679,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,40.2,35.38,5.001558065414429
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",71,2026178483602,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,3.0,3,0,14.8425,15.98,6.296044111251831
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",8,2026181883641,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,81.925,62.8,4.775622844696045
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",98,2026180183680,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.76,36.12,4.109494686126709
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",103,2026175083532,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,137.6,61.75,6.240614175796509
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",9,2026181883642,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.25,63.55,4.0490453243255615
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",72,2026178483603,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,4.0,4,0,15.2025,16.2,6.838795900344849
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",99,2026180183681,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.02,34.78,5.190775632858276
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",104,2026175083533,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,2.0,2,0,142.15,65.0,4.678457975387573
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",10,2026181883643,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.5,1,1,82.65,65.95,5.121794939041138
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",73,2026178483604,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,5.0,5,0,15.01,15.805,6.066697835922241
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",105,2026175083534,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,137.825,64.75,5.550603628158569
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",100,2026180183682,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,39.56,36.56,6.159163236618042
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",11,2026181883644,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,1.0,1,0,81.625,62.8,4.043464183807373
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",101,2026180183683,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.5,1,1,39.94,36.5,4.214838981628418
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",12,2026181883645,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,1.0,1,0,81.475,63.6,4.693230390548706
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",74,2026178483605,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,3.0,4,2,15.3075,15.975,7.07391357421875
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",106,2026175083535,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,139.7,65.55,6.611950874328613
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",102,2026180183684,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,40.66,35.94,5.1442084312438965
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",13,2026181883646,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,81.3,62.75,4.584615707397461
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",107,2026175083536,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,134.925,64.65,6.152278661727905
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",75,2026178483606,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,2.0,2,0,15.185,16.585,7.1046905517578125
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",14,2026181883647,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,84.25,64.95,4.639387845993042
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",103,2026180183685,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,39.47,34.92,5.062373638153076
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",76,2026178483607,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.0,1,0,15.19,15.95,6.066637754440308
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",15,2026181883648,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.5,1,1,81.775,65.1,4.414953708648682
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",104,2026180183686,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.5,1,1,39.49,36.96,4.401432752609253
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",108,2026175083537,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,137.575,63.8,6.635478734970093
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",105,2026180183687,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.53,35.74,4.1398162841796875
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",16,2026181883649,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,84.025,63.6,4.871745586395264
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",109,2026175083538,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,138.925,63.6,5.681901454925537
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",77,2026178483608,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.5,2,1,14.975,16.035,6.915912389755249
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",17,2026181883650,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.8,62.65,4.149526357650757
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",106,2026180183688,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.49,36.64,5.216187953948975
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",110,2026175083539,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,2.0,2,0,138.575,64.25,5.463135242462158
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",18,2026181883651,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.25,64.05,4.347812175750732
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",78,2026178483609,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,3.5,4,1,14.7825,16.12,7.102633476257324
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",107,2026180183689,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.31,35.66,6.211535930633545
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",111,2026175083540,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,136.65,64.45,6.32785177230835
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",19,2026181883652,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,85.25,65.05,5.015796184539795
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",79,2026178483610,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,2.0,2,0,14.9625,15.825,6.213160991668701
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",108,2026180183690,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.5,1,1,41.23,35.92,5.281655550003052
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",112,2026175083541,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,137.6,65.75,6.193662166595459
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",20,2026181883653,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.125,64.05,5.427556753158569
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",109,2026180183691,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,39.27,35.52,5.14142370223999
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",80,2026178483611,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,3.0,4,2,14.795,16.06,6.34612774848938
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",21,2026181883654,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.975,62.5,4.699342966079712
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",113,2026175083542,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,139.75,64.3,6.6430344581604
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",110,2026180183692,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.15,36.5,4.528587341308594
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",22,2026181883655,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,81.825,61.55,4.615880012512207
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0003|pi0.0003|k1,1,1",81,2026178483612,A,noisy,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,4.5,5,1,15.29,16.14,6.7909417152404785
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",0,2026183583684,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,9.0,9,0,2.3475,4.115,1.680004596710205
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",114,2026175083543,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,135.575,63.15,5.4974730014801025
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",1,2026183583685,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,3.0,3,0,2.4475,4.015,1.3998651504516602
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",111,2026180183693,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.38,36.36,6.096661806106567
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",2,2026183583686,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,3.0,3,0,2.24,4.1,1.1345107555389404
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",23,2026181883656,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,86.525,65.65,5.126720666885376
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",3,2026183583687,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,7.0,7,0,2.3875,3.995,1.1579933166503906
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",4,2026183583688,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,4.0,4,0,2.465,4.105,1.1486940383911133
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",112,2026180183694,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.09,36.48,4.401136159896851
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",5,2026183583689,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,4.0,4,0,2.3825,4.065,1.458275318145752
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",6,2026183583690,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,3.5,4,1,2.13,4.13,1.0958940982818604
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",115,2026175083544,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,137.95,64.45,6.767735958099365
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",24,2026181883657,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.75,64.15,5.540801286697388
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",7,2026183583691,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,2.0,2,0,2.4025,4.0,1.5694692134857178
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",8,2026183583692,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,1.0,1,0,2.32,3.92,1.1475648880004883
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",113,2026180183695,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.5,2,1,39.78,36.0,4.935144424438477
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",9,2026183583693,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,2.0,2,0,2.325,4.015,1.9977643489837646
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",116,2026175083545,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,135.55,64.15,5.376107692718506
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",10,2026183583694,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,2.0,2,0,2.475,3.89,1.4345903396606445
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",11,2026183583695,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,1.0,1,0,2.4325,4.1,1.1953814029693604
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",25,2026181883658,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.4,63.0,6.132565498352051
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",114,2026180183696,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.5,2,1,39.49,36.42,4.232922315597534
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",12,2026183583696,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,2.0,2,0,2.395,3.89,1.2916181087493896
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",13,2026183583697,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,2.0,2,0,2.3675,3.82,1.1966676712036133
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",117,2026175083546,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,140.175,63.85,5.044809341430664
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",14,2026183583698,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,4.0,4,0,2.2625,4.045,1.497666597366333
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",26,2026181883659,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,84.075,63.8,4.527341365814209
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",15,2026183583699,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,1.0,1,0,2.3625,4.15,1.0693061351776123
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",16,2026183583700,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,1.5,2,1,2.28,3.975,1.4036970138549805
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",115,2026180183697,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.13,36.92,6.313565254211426
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",17,2026183583701,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,3.0,3,0,2.4775,3.93,1.1665732860565186
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",118,2026175083547,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,139.95,64.85,5.569797515869141
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",27,2026181883660,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,81.8,62.55,4.339820146560669
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",18,2026183583702,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,4.0,4,0,2.395,4.02,1.4802052974700928
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",19,2026183583703,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,5.0,5,0,2.495,3.955,1.3886499404907227
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",20,2026183583704,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,2.0,2,0,2.38,4.015,1.260749101638794
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",116,2026180183698,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.5,1,1,40.67,36.04,5.935471296310425
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",21,2026183583705,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,7.0,7,0,2.24,3.855,1.5286107063293457
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",28,2026181883661,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.425,64.95,5.027038812637329
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",22,2026183583706,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,0.0,0,0,2.36,3.89,1.2836685180664062
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",119,2026175083548,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,139.1,63.45,6.046159267425537
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",23,2026183583707,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,3.0,3,0,2.4025,3.985,1.72402024269104
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",117,2026180183699,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,39.69,36.96,4.346996307373047
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",24,2026183583708,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,5.0,5,0,2.2225,3.975,1.425734519958496
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",29,2026181883662,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.075,63.95,4.247463941574097
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",25,2026183583709,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,5.0,5,0,2.3125,3.925,1.441572904586792
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",26,2026183583710,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,3.0,3,0,2.39,3.875,1.4703986644744873
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",120,2026175083549,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.5,1,1,138.975,65.05,5.965477705001831
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",118,2026180183700,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,40.95,36.12,4.565410137176514
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",27,2026183583711,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,6.0,6,0,2.4925,3.89,1.6113414764404297
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",30,2026181883663,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,79.9,63.45,4.7673420906066895
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",28,2026183583712,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,4.0,4,0,2.43,4.005,1.3425109386444092
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",29,2026183583713,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,3.5,4,1,2.35,4.13,1.6464366912841797
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",119,2026180183701,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.53,36.1,5.088407516479492
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",121,2026175083550,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,2.0,2,0,142.9,64.55,5.976214170455933
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",30,2026183583714,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,1.0,1,0,2.4475,3.975,1.5489509105682373
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",31,2026181883664,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,2.0,2,0,85.175,63.7,4.759791374206543
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",31,2026183583715,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,3.0,3,0,2.41,3.91,1.364661455154419
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",32,2026183583716,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,3.0,3,0,2.435,3.975,1.0788977146148682
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",33,2026183583717,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,2.0,2,0,2.475,3.905,1.712507724761963
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",120,2026180183702,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.35,36.72,4.589830160140991
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",32,2026181883665,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,79.9,64.45,4.219191312789917
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",34,2026183583718,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,2.0,2,0,2.25,4.025,1.6440081596374512
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",122,2026175083551,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,140.25,64.0,6.333124160766602
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",35,2026183583719,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,3.5,4,1,2.35,3.99,1.300342082977295
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",121,2026180183703,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.5,1,1,39.89,35.84,4.493659496307373
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",36,2026183583720,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,0.5,1,1,2.435,4.025,1.8877253532409668
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",33,2026181883666,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.3,63.05,4.730729579925537
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",37,2026183583721,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,2.0,2,0,2.38,4.035,1.4505424499511719
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",38,2026183583722,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,6.0,6,0,2.3625,3.98,1.3953237533569336
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",123,2026175083552,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,137.2,64.6,6.56607985496521
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",122,2026180183704,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.62,35.98,4.087779760360718
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",39,2026183583723,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,3.0,3,0,2.2125,4.005,1.3632869720458984
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",34,2026181883667,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,84.225,65.0,4.55513072013855
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",40,2026183583724,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,2.0,2,0,2.37,4.16,1.4776771068572998
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",41,2026183583725,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,3.5,4,1,2.3175,3.985,1.6826300621032715
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",124,2026175083553,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,136.925,63.9,4.492559909820557
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",42,2026183583726,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,2.0,2,0,2.4775,4.125,1.6229526996612549
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",123,2026180183705,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.5,1,1,39.88,35.16,5.557708978652954
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",35,2026181883668,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,1.0,1,0,82.575,62.85,4.759463548660278
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",43,2026183583727,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,4.0,4,0,2.2775,3.965,1.9548425674438477
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",44,2026183583728,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,6.0,6,0,2.47,4.065,1.065070390701294
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",125,2026175083554,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,140.725,65.5,4.516960382461548
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",45,2026183583729,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,1.0,1,0,2.3725,4.05,1.0287377834320068
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",46,2026183583730,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,2.0,2,0,2.2425,3.78,0.9767787456512451
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",124,2026180183706,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.5,1,1,40.16,36.5,5.350644588470459
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",36,2026181883669,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,81.35,65.5,4.848870754241943
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",47,2026183583731,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,3.0,3,0,2.3275,4.07,1.071739673614502
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",48,2026183583732,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,1.0,1,0,2.34,3.995,1.3531150817871094
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",49,2026183583733,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,7.0,7,0,2.4025,3.985,1.3499565124511719
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",126,2026175083555,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,2.0,2,0,135.575,61.35,5.762887001037598
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",37,2026181883670,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,1.0,1,0,82.025,64.85,3.8808701038360596
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",50,2026183583734,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,6.0,6,0,2.4325,3.99,1.2196013927459717
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",51,2026183583735,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,2.5,3,1,2.42,3.92,1.2405786514282227
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",125,2026180183707,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.58,36.54,5.521076202392578
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",52,2026183583736,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,1.0,1,0,2.4675,3.965,1.1809818744659424
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",53,2026183583737,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,1.0,1,0,2.3875,3.875,1.2425081729888916
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",38,2026181883671,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,1.0,1,0,82.075,64.65,4.473219394683838
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",54,2026183583738,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,3.0,3,0,2.49,4.245,1.347184181213379
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",127,2026175083556,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,139.75,65.75,6.0275444984436035
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",55,2026183583739,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,6.0,6,0,2.2825,4.16,1.4765217304229736
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",126,2026180183708,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.13,37.42,5.973202705383301
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",56,2026183583740,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,6.5,7,1,2.3625,3.915,1.416391134262085
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",39,2026181883672,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.2,64.2,4.764034986495972
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",57,2026183583741,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,1.5,2,1,2.34,3.815,1.4601802825927734
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",58,2026183583742,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,1.0,1,0,2.385,4.03,1.2791063785552979
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",128,2026175083557,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,2.0,2,0,134.725,62.85,5.1744983196258545
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",59,2026183583743,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,3.0,3,0,2.31,3.965,1.8232698440551758
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",127,2026180183709,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,39.46,36.0,5.931141138076782
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",40,2026181883673,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.775,65.3,4.404319763183594
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",60,2026183583744,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,1.0,1,0,2.5025,3.925,1.194155216217041
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",61,2026183583745,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,1.0,1,0,2.2775,4.015,1.366837978363037
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",62,2026183583746,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,1.0,1,0,2.235,4.115,1.3521935939788818
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",129,2026175083558,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,137.65,65.0,5.644318342208862
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0002|pi0.0002|k1,1,1",63,2026183583747,A,noisy,3,0.0002,0.0002,1.0,1.0,1.0,3,3,3,400,4.5,5,1,2.2625,4.085,1.110579013824463
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",128,2026180183710,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,39.99,36.18,4.538499593734741
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",41,2026181883674,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,81.875,62.25,5.39504599571228
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",129,2026180183711,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.03,36.26,4.40264630317688
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",130,2026175083559,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,136.975,62.3,5.858292579650879
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",42,2026181883675,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.95,64.5,4.797826528549194
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",0,2026185283735,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,10.1025,16.2,5.976269483566284
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",131,2026175083560,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,139.725,63.85,4.964989185333252
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",130,2026180183712,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.44,36.36,5.077960968017578
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",43,2026181883676,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.45,61.0,4.714402914047241
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",1,2026185283736,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,2.0,2,0,10.0425,16.22,5.632958650588989
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",131,2026180183713,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.96,35.82,3.984546661376953
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",44,2026181883677,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.5,1,1,83.0,66.45,4.984223365783691
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",132,2026175083561,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,135.75,68.1,6.4030749797821045
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",132,2026180183714,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.63,35.76,5.3433544635772705
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",2,2026185283737,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.9075,16.21,6.53838586807251
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",45,2026181883678,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,1.0,1,0,83.425,64.2,4.452922105789185
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",133,2026175083562,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,141.3,65.5,4.8863537311553955
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",133,2026180183715,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.5,1,1,39.34,35.22,4.513993740081787
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",3,2026185283738,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,9.98,16.245,5.88441014289856
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",46,2026181883679,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.55,64.9,5.177772045135498
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",134,2026175083563,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,136.2,62.4,5.863052845001221
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",134,2026180183716,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.9,35.58,4.42768406867981
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",47,2026181883680,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,81.2,63.05,5.030040264129639
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",4,2026185283739,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.0975,15.94,5.86386251449585
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",135,2026175083564,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,136.675,65.05,5.593541860580444
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",135,2026180183717,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.54,36.26,5.310696363449097
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",48,2026181883681,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,85.1,62.4,5.193885803222656
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",5,2026185283740,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,2.0,2,0,10.175,16.26,6.156046390533447
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",136,2026175083565,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,137.85,63.7,5.917289733886719
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",136,2026180183718,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,38.67,37.04,5.113191843032837
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",49,2026181883682,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,1.0,1,0,83.15,64.2,3.896044969558716
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",6,2026185283741,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,3.0,3,0,10.1225,16.15,6.695003509521484
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",137,2026175083566,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,139.1,63.55,5.4593346118927
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",137,2026180183719,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,40.21,36.18,5.6464409828186035
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",50,2026181883683,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.775,62.15,5.067005634307861
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",51,2026181883684,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,87.0,64.75,4.597133159637451
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",138,2026180183720,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.1,36.76,5.386847972869873
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",138,2026175083567,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,137.475,63.5,5.733904600143433
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",7,2026185283742,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,9.9975,16.24,6.1389594078063965
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",52,2026181883685,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.725,63.25,4.477098226547241
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",139,2026175083568,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.5,1,1,133.975,63.55,4.740947961807251
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",139,2026180183721,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,38.78,35.94,6.2043681144714355
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",8,2026185283743,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,10.175,15.895,7.126058578491211
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",53,2026181883686,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.0,62.15,3.4595274925231934
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",140,2026175083569,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,133.725,63.65,5.657108545303345
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",54,2026181883687,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,84.45,65.15,3.506671905517578
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",140,2026180183722,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.62,36.34,5.70318078994751
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",9,2026185283744,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.0025,16.025,7.064846515655518
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",55,2026181883688,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,1.0,1,0,83.75,62.55,4.0768139362335205
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",141,2026175083570,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,140.2,63.8,5.344528675079346
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",141,2026180183723,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.7,35.3,5.062384128570557
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",56,2026181883689,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,81.725,62.65,4.466449022293091
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",10,2026185283745,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.885,15.995,5.539990186691284
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",142,2026175083571,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,136.8,63.5,4.477015256881714
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",142,2026180183724,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.5,35.9,6.0203282833099365
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",11,2026185283746,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.1575,16.475,4.742584705352783
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",57,2026181883690,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.325,62.75,5.299101829528809
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",143,2026175083572,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.5,2,1,138.775,63.9,5.120434999465942
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",143,2026180183725,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,39.76,35.66,4.422967433929443
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",58,2026181883691,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.55,63.7,5.496978759765625
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",144,2026175083573,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,136.475,62.85,5.903396129608154
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",12,2026185283747,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,9.9575,16.115,6.706052303314209
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",144,2026180183726,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.95,35.86,4.572906732559204
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",59,2026181883692,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.725,64.15,4.846886157989502
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",145,2026175083574,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,3.5,4,1,138.075,63.95,5.021121025085449
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",145,2026180183727,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,38.77,36.38,4.787032127380371
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",13,2026185283748,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.0275,15.935,7.276895046234131
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",60,2026181883693,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,84.325,65.3,3.908149480819702
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",146,2026175083575,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,137.075,63.05,5.225555896759033
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",146,2026180183728,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,38.59,37.38,5.4219584465026855
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",61,2026181883694,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.625,64.3,5.42728066444397
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",14,2026185283749,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.895,16.23,6.177462100982666
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",147,2026180183729,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.13,36.28,5.101110458374023
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",147,2026175083576,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,137.2,63.15,6.077069044113159
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",62,2026181883695,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.4,62.4,4.361687421798706
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",15,2026185283750,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.5,1,1,10.36,16.085,7.358930349349976
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",148,2026175083577,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,141.25,64.5,4.917444944381714
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",148,2026180183730,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.18,36.1,5.392759084701538
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",63,2026181883696,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,84.325,64.0,5.7745044231414795
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",149,2026180183731,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,40.78,36.44,5.239285469055176
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",149,2026175083578,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.5,1,1,138.275,65.55,5.359851360321045
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",16,2026185283751,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,10.0375,16.175,6.5044920444488525
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",64,2026181883697,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,81.125,64.85,4.478715658187866
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",150,2026180183732,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,40.36,34.82,4.55714225769043
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",150,2026175083579,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,137.575,63.25,6.015309572219849
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",17,2026185283752,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.97,15.78,5.629756927490234
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",65,2026181883698,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.85,65.05,5.128922462463379
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",151,2026180183733,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.64,35.62,4.332479000091553
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",66,2026181883699,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,81.05,62.8,4.140413284301758
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",151,2026175083580,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,136.875,63.9,5.405520677566528
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",18,2026185283753,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,10.3425,15.765,6.822334051132202
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",152,2026180183734,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.33,35.38,5.493649482727051
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",67,2026181883700,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,81.725,64.6,4.0053863525390625
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",152,2026175083581,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,137.825,62.1,6.2687647342681885
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",68,2026181883701,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.675,64.35,4.003441095352173
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",19,2026185283754,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.5,1,1,10.3925,15.765,5.564115524291992
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",153,2026180183735,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,40.22,36.36,5.229784727096558
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",153,2026175083582,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,138.05,64.55,5.99381160736084
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",69,2026181883702,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.8,65.05,5.085119247436523
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",154,2026180183736,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.54,36.38,4.9910008907318115
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",20,2026185283755,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,2.0,2,0,10.095,15.885,5.959303617477417
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",155,2026180183737,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.23,36.58,5.139873266220093
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",21,2026185283756,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.5,1,1,10.29,15.895,5.226434230804443
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",154,2026175083583,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,137.95,63.95,6.5690436363220215
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",70,2026181883703,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.025,66.05,6.706620693206787
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",156,2026180183738,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.33,36.52,4.934604644775391
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",71,2026181883704,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,86.25,66.05,4.313753843307495
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",22,2026185283757,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.83,15.795,6.1280128955841064
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",155,2026175083584,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,138.75,65.1,6.192381143569946
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",72,2026181883705,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.075,66.4,4.082483768463135
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",157,2026180183739,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,2.0,2,0,39.04,35.68,5.368858098983765
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",156,2026175083585,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,140.525,66.4,5.47970724105835
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",23,2026185283758,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,10.1925,15.955,6.9810004234313965
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",73,2026181883706,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,81.425,63.25,4.453895092010498
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",158,2026180183740,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.21,36.1,4.734555006027222
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",157,2026175083586,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.5,2,1,138.825,63.25,5.53995680809021
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",74,2026181883707,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.825,65.55,4.825838088989258
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",24,2026185283759,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.115,15.875,5.599121809005737
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",159,2026180183741,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,2.0,2,0,40.05,35.22,4.648267507553101
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",158,2026175083587,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,135.8,64.65,4.096152305603027
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",25,2026185283760,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.5,1,1,10.085,15.95,5.055962085723877
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",75,2026181883708,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,2.0,2,0,83.425,62.6,6.398873805999756
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",160,2026180183742,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.62,36.44,5.507647275924683
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",159,2026175083588,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,2.0,2,0,138.95,63.6,5.6731178760528564
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",161,2026180183743,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.16,36.2,3.5703213214874268
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",26,2026185283761,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,10.3325,15.99,5.320484638214111
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",76,2026181883709,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,81.875,65.05,5.041285753250122
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",162,2026180183744,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,38.92,35.6,4.298542737960815
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",160,2026175083589,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,140.375,64.3,6.214169025421143
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",77,2026181883710,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,84.45,63.55,5.178731203079224
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",27,2026185283762,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,9.71,16.145,5.687934637069702
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",163,2026180183745,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,39.53,35.4,4.510307788848877
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",161,2026175083590,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,2.0,2,0,142.125,63.9,5.496169567108154
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",78,2026181883711,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,84.275,64.25,5.135354995727539
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",28,2026185283763,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.5,1,1,10.1025,16.175,6.113364219665527
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",164,2026180183746,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,38.35,35.64,3.968085289001465
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",162,2026175083591,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,139.9,65.0,4.847449541091919
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",79,2026181883712,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,85.525,65.6,4.5992701053619385
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",165,2026180183747,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,40.35,36.14,4.932832717895508
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",29,2026185283764,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.995,16.335,6.790263891220093
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",163,2026175083592,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,138.125,63.45,5.190622329711914
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",80,2026181883713,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,81.575,63.5,5.3115456104278564
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",166,2026180183748,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,2.0,2,0,38.18,36.24,5.054492473602295
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",30,2026185283765,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,9.945,15.925,6.239068508148193
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",164,2026175083593,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,136.35,64.4,5.617785692214966
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",81,2026181883714,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,2.0,2,0,83.0,64.3,5.1591644287109375
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",167,2026180183749,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.02,35.9,6.0064661502838135
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",82,2026181883715,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,1.0,1,0,81.1,62.85,4.235722780227661
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",165,2026175083594,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,135.775,64.25,4.946828365325928
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",31,2026185283766,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,3.0,3,0,10.145,16.065,7.473282098770142
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",168,2026180183750,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.78,36.14,5.5928990840911865
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",83,2026181883716,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.825,63.45,3.5952484607696533
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",166,2026175083595,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,3.5,4,1,138.125,64.1,6.676070213317871
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",169,2026180183751,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,38.73,35.62,4.338647127151489
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",84,2026181883717,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,81.35,62.05,4.8907716274261475
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",32,2026185283767,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.5,1,1,9.885,16.04,6.344454765319824
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",170,2026180183752,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.09,35.22,3.9324114322662354
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",167,2026175083596,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,136.2,63.05,5.447707414627075
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",33,2026185283768,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.8275,16.025,5.383397579193115
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",85,2026181883718,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,84.025,64.7,5.744280099868774
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",171,2026180183753,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.69,36.6,4.508511304855347
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",168,2026175083597,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,3.0,3,0,140.35,61.5,6.096356391906738
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",34,2026185283769,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.1575,15.66,5.620478391647339
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",86,2026181883719,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,1.0,1,0,83.125,63.45,6.176900386810303
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",172,2026180183754,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.5,1,1,38.57,35.86,4.427931308746338
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",169,2026175083598,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,2.0,3,2,140.0,61.0,4.681039094924927
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",87,2026181883720,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,84.475,63.55,4.587730407714844
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",173,2026180183755,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.5,1,1,41.66,36.68,4.9552836418151855
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",35,2026185283770,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,9.965,15.935,6.511265277862549
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",170,2026175083599,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,135.4,64.7,5.77526068687439
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",88,2026181883721,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,1.0,1,0,81.2,62.85,5.1314780712127686
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",174,2026180183756,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.67,36.14,4.761195659637451
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",36,2026185283771,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,10.195,16.03,6.696488857269287
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",89,2026181883722,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,84.175,66.55,4.721282482147217
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",171,2026175083600,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,141.725,64.7,6.224257946014404
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",175,2026180183757,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,39.85,36.44,4.879611492156982
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",176,2026180183758,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.5,1,1,39.59,36.58,4.210514545440674
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",172,2026175083601,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,2.0,2,0,144.475,64.15,4.630103826522827
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",37,2026185283772,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.5,1,1,10.1175,16.145,7.290086507797241
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",90,2026181883723,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.4,65.5,5.3708250522613525
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",173,2026175083602,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,138.875,63.3,4.8925395011901855
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",38,2026185283773,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.5,1,1,9.735,16.19,5.0501298904418945
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",177,2026180183759,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.78,37.2,6.122099876403809
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",91,2026181883724,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.825,63.0,6.091778993606567
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",39,2026185283774,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.2825,15.895,5.220513820648193
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",174,2026175083603,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,139.125,64.7,5.719082593917847
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",92,2026181883725,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,85.8,64.45,4.268906831741333
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",178,2026180183760,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.08,34.94,5.840544939041138
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",93,2026181883726,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,85.025,63.65,5.029395341873169
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",179,2026180183761,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,39.85,36.42,4.166198015213013
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",40,2026185283775,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.1225,15.91,5.94559383392334
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",175,2026175083604,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,3.0,3,0,140.625,61.5,6.233231544494629
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",180,2026180183762,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,40.41,35.82,4.672870397567749
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",94,2026181883727,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,84.825,62.3,5.6057798862457275
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",41,2026185283776,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.0075,15.885,5.718093633651733
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",176,2026175083605,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,139.85,65.0,5.415120840072632
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",181,2026180183763,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.16,37.76,4.959107160568237
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",95,2026181883728,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.025,67.3,5.792301177978516
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",177,2026175083606,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,141.7,65.1,5.002162218093872
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",42,2026185283777,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.5,2,1,10.0925,15.72,7.095295667648315
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",182,2026180183764,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,39.36,35.54,4.938956022262573
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",96,2026181883729,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,84.125,63.25,4.1049909591674805
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",178,2026175083607,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,137.375,65.5,6.356434106826782
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",43,2026185283778,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.995,16.195,5.709166765213013
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",97,2026181883730,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.275,63.6,4.539316177368164
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",183,2026180183765,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.5,1,1,39.07,35.64,6.109923362731934
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",179,2026175083608,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,138.5,63.8,4.702471494674683
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",44,2026185283779,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.5,1,1,10.1325,15.84,5.47821307182312
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",98,2026181883731,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.775,64.35,5.382149696350098
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",184,2026180183766,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,2.0,2,0,39.41,36.14,5.132597208023071
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",180,2026175083609,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,136.275,63.6,5.13581109046936
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",45,2026185283780,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.9325,15.9,5.666937351226807
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",185,2026180183767,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,39.36,36.16,4.484784126281738
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",99,2026181883732,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,81.5,64.75,5.77206015586853
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",181,2026175083610,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,4.0,4,0,137.25,62.8,5.604012489318848
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",186,2026180183768,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,40.7,35.56,4.0165793895721436
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",100,2026181883733,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,81.275,65.85,4.7799201011657715
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",46,2026185283781,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,10.2525,15.955,6.105160474777222
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",182,2026175083611,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,139.15,63.9,5.135498523712158
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",187,2026180183769,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,39.71,36.72,4.919432163238525
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",101,2026181883734,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.7,66.15,4.6051390171051025
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",188,2026180183770,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,39.97,35.02,4.422682046890259
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",183,2026175083612,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,138.65,64.35,5.319473028182983
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",47,2026185283782,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,10.1575,16.2,7.6291892528533936
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",102,2026181883735,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,80.575,63.5,4.009263515472412
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",189,2026180183771,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.76,37.26,5.230600833892822
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",103,2026181883736,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,84.15,64.1,4.70109486579895
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",184,2026175083613,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,136.5,63.95,5.562847137451172
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",48,2026185283783,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.5,2,1,10.09,15.98,5.61445164680481
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",185,2026175083614,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,2.0,2,0,138.575,63.85,4.81293797492981
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",190,2026180183772,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,40.75,35.76,5.555056571960449
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",104,2026181883737,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,1.0,1,0,83.3,64.75,5.397423267364502
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",49,2026185283784,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.9625,15.98,6.397590160369873
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",105,2026181883738,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.025,61.75,3.7890331745147705
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",191,2026180183773,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.93,36.16,4.638891935348511
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",186,2026175083615,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,136.425,61.7,6.0039472579956055
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",50,2026185283785,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.9725,15.955,5.777320384979248
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",106,2026181883739,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,81.3,64.85,5.552134990692139
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",192,2026180183774,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,41.08,35.56,5.109675645828247
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",187,2026175083616,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,138.2,63.1,5.872526407241821
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",51,2026185283786,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.91,15.9,5.548523664474487
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",193,2026180183775,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.45,35.18,4.1879658699035645
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",107,2026181883740,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,84.25,65.45,4.931966543197632
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",188,2026175083617,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,2.0,2,0,142.575,66.35,5.202485084533691
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",194,2026180183776,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,39.76,36.46,4.451906204223633
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",108,2026181883741,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.975,64.9,4.240437984466553
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",52,2026185283787,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.1825,15.89,6.825220346450806
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",109,2026181883742,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,80.95,64.8,3.340371608734131
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",195,2026180183777,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.13,36.48,4.383254289627075
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",189,2026175083618,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,136.575,65.65,7.0464465618133545
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",110,2026181883743,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.075,63.45,3.280081272125244
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",53,2026185283788,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,2.0,2,0,10.1325,15.845,6.726094722747803
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",196,2026180183778,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.14,35.28,4.0224103927612305
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",190,2026175083619,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.5,2,1,138.425,64.2,6.151406764984131
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",197,2026180183779,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.12,36.68,4.142399311065674
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",111,2026181883744,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.8,62.95,5.6453516483306885
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",54,2026185283789,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,10.0175,15.865,6.26860237121582
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",191,2026175083620,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,139.025,66.35,5.468846082687378
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",198,2026180183780,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.5,1,1,39.06,36.92,4.859631776809692
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",112,2026181883745,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.55,62.7,5.265563488006592
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",55,2026185283790,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.7475,16.21,6.849045991897583
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",199,2026180183781,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,41.07,36.4,4.582602262496948
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",113,2026181883746,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.6,66.05,4.29403018951416
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",192,2026175083621,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,2.0,2,0,135.025,64.05,6.174445390701294
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",56,2026185283791,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.5,2,1,10.0,15.495,5.896164894104004
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",200,2026180183782,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,40.25,35.72,5.506181716918945
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",114,2026181883747,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.6,65.45,5.5917298793792725
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",193,2026175083622,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,136.475,63.5,5.470630884170532
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",115,2026181883748,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,80.625,62.15,4.432425260543823
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",57,2026185283792,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.175,16.005,5.603097200393677
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",201,2026180183783,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.84,35.52,5.293118238449097
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",116,2026181883749,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,84.15,65.1,3.519623279571533
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",194,2026175083623,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,138.225,64.4,7.485238313674927
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",202,2026180183784,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.5,1,1,39.13,36.54,4.413292169570923
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",58,2026185283793,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.15,15.82,5.081758499145508
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",117,2026181883750,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.525,65.0,4.777837514877319
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",203,2026180183785,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.87,35.34,4.327500104904175
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",195,2026175083624,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,2.0,2,0,138.425,64.85,5.4253833293914795
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",59,2026185283794,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.155,16.275,6.325851202011108
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",118,2026181883751,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.45,64.2,4.491022109985352
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",204,2026180183786,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.5,1,1,40.22,36.62,5.395793199539185
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",196,2026175083625,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,137.625,62.95,5.73999810218811
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",60,2026185283795,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.945,16.055,5.852182626724243
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",119,2026181883752,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.275,62.3,5.093490362167358
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",205,2026180183787,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.99,36.12,5.2002246379852295
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",197,2026175083626,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,140.6,65.85,4.835248947143555
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",120,2026181883753,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.025,65.9,5.082257032394409
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",61,2026185283796,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,2.5,3,1,10.3575,15.76,6.598757743835449
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",206,2026180183788,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.3,35.84,4.564610242843628
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",198,2026175083627,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,139.775,64.25,5.225745916366577
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",121,2026181883754,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.2,65.35,5.134638547897339
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",207,2026180183789,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.5,1,1,40.44,36.5,4.930056571960449
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",62,2026185283797,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.5,2,1,10.0525,16.125,5.521733999252319
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0005|pi0.0005|k1,1,1",199,2026175083628,A,noisy,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,134.925,64.3,6.116028308868408
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",122,2026181883755,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.2,63.7,5.05560827255249
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",208,2026180183790,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.16,36.0,3.8399901390075684
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",63,2026185283798,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.875,15.83,5.536908388137817
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",209,2026180183791,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.49,36.32,4.2904863357543945
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",123,2026181883756,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,81.6,63.5,5.610337734222412
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",0,2026186983786,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.79,36.4,5.23774528503418
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",64,2026185283799,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.5,2,1,10.445,16.105,5.776021242141724
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",210,2026180183792,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.09,36.38,4.898943901062012
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",124,2026181883757,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,84.025,66.35,4.250008583068848
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",1,2026186983787,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.81,36.46,4.294041156768799
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",65,2026185283800,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.1025,15.63,5.879488706588745
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",211,2026180183793,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.3,36.74,5.159624814987183
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",2,2026186983788,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.47,36.02,4.3485167026519775
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",125,2026181883758,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.8,63.4,5.276422739028931
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",212,2026180183794,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.41,34.88,4.380356788635254
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",126,2026181883759,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.6,62.95,4.6251606941223145
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",66,2026185283801,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.095,15.7,6.574138641357422
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",3,2026186983789,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.33,36.22,5.4949963092803955
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",213,2026180183795,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,39.61,36.82,4.292969703674316
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",127,2026181883760,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,79.175,65.6,4.174633741378784
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",4,2026186983790,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.87,36.02,5.166106224060059
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",67,2026185283802,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.09,15.765,6.772727727890015
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",214,2026180183796,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.71,36.38,4.366669654846191
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",128,2026181883761,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,84.35,63.05,4.613085031509399
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",5,2026186983791,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.53,36.16,4.767362356185913
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",215,2026180183797,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.5,2,1,40.28,36.28,4.681114912033081
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",129,2026181883762,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.35,64.55,4.497187614440918
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",68,2026185283803,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.5,2,1,9.9075,16.075,6.936369180679321
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",6,2026186983792,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.9,35.92,4.662321329116821
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",216,2026180183798,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.07,35.96,5.283563613891602
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",130,2026181883763,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.65,63.7,4.922643423080444
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",7,2026186983793,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.69,36.9,4.279438257217407
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",69,2026185283804,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.5,1,1,10.3225,15.965,7.506762981414795
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",131,2026181883764,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.6,64.1,3.871263265609741
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",217,2026180183799,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.66,35.68,5.353695869445801
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",8,2026186983794,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.6,35.88,4.328822135925293
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",132,2026181883765,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.7,64.75,4.545163154602051
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",218,2026180183800,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.46,36.32,4.734195232391357
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",70,2026185283805,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,10.0325,15.905,6.0865325927734375
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",9,2026186983795,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.0,36.0,4.509737491607666
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",219,2026180183801,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.02,35.12,4.393805742263794
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",133,2026181883766,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.475,63.65,5.93769097328186
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",10,2026186983796,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.23,35.84,4.240248918533325
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",71,2026185283806,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.05,16.2,6.87559700012207
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",220,2026180183802,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,41.07,35.32,4.804591417312622
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",11,2026186983797,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.71,36.36,4.6968090534210205
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",134,2026181883767,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,84.025,64.6,5.161117315292358
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",12,2026186983798,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.49,36.02,4.226195335388184
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",221,2026180183803,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.35,36.06,4.793449401855469
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",72,2026185283807,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,10.03,15.98,7.193806171417236
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",135,2026181883768,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,85.15,63.05,5.182598114013672
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",13,2026186983799,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.67,35.42,4.734040021896362
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",222,2026180183804,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,41.08,36.66,5.105442523956299
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",136,2026181883769,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,84.775,66.0,5.180773973464966
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",73,2026185283808,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.06,15.6,6.379474401473999
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",14,2026186983800,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.76,36.2,4.4737513065338135
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",223,2026180183805,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.07,36.9,5.0508036613464355
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",137,2026181883770,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.25,64.65,4.484386444091797
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",74,2026185283809,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,10.29,16.05,7.331435441970825
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",15,2026186983801,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,1.0,1,0,26.79,35.94,4.920199871063232
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",224,2026180183806,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.46,35.8,4.550453424453735
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",138,2026181883771,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,79.175,62.8,4.624802112579346
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",16,2026186983802,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.5,1,1,26.94,36.5,4.673474550247192
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",75,2026185283810,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.9525,15.93,4.841270446777344
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",225,2026180183807,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.88,35.62,5.0685365200042725
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",139,2026181883772,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,86.55,65.5,5.689346075057983
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",17,2026186983803,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.17,35.44,4.51566481590271
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",226,2026180183808,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.08,36.86,5.190387487411499
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",76,2026185283811,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.2675,16.035,6.554430961608887
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",140,2026181883773,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.375,63.7,5.157373666763306
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",18,2026186983804,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.6,35.1,4.052777528762817
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",227,2026180183809,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.66,35.8,5.068638324737549
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",77,2026185283812,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,2.0,3,2,10.145,16.25,5.373535633087158
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",141,2026181883774,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,81.3,62.55,5.038823127746582
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",19,2026186983805,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.11,34.86,5.214450836181641
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",228,2026180183810,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.06,36.98,4.772720575332642
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",142,2026181883775,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.3,66.0,4.477820158004761
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",20,2026186983806,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.24,36.84,4.4990057945251465
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",78,2026185283813,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,2.0,2,0,10.2725,16.01,6.948013067245483
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",229,2026180183811,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,39.09,34.98,5.7346110343933105
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",21,2026186983807,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.7,35.94,3.7752392292022705
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",143,2026181883776,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,1.0,1,0,84.05,65.05,5.0437843799591064
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",79,2026185283814,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.1125,16.175,5.777849197387695
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",144,2026181883777,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,81.65,64.25,4.587830543518066
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",22,2026186983808,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.57,36.86,4.962402105331421
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",230,2026180183812,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.26,37.04,5.830363035202026
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",80,2026185283815,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.2375,16.065,5.9281439781188965
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",23,2026186983809,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.64,36.78,4.338000059127808
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",145,2026181883778,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.2,65.15,4.819983720779419
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",231,2026180183813,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,40.25,36.16,4.9899797439575195
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",146,2026181883779,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,81.55,64.1,3.8336102962493896
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",24,2026186983810,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.24,35.78,5.149516820907593
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",232,2026180183814,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,39.93,36.64,4.968015909194946
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",81,2026185283816,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,2.0,2,0,10.175,16.12,7.1833176612854
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",25,2026186983811,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,25.89,36.22,3.50435209274292
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",147,2026181883780,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,84.05,65.85,5.409433603286743
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",233,2026180183815,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,40.4,36.04,3.9433248043060303
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",82,2026185283817,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,3.0,3,0,9.895,15.785,6.297051191329956
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",148,2026181883781,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,81.325,63.15,3.8939921855926514
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",26,2026186983812,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,25.72,36.24,5.455550193786621
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0003|pi0.0003|k1,1,1",234,2026180183816,A,noisy,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,38.34,35.98,4.938938617706299
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",149,2026181883782,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,80.4,63.5,4.396557331085205
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",27,2026186983813,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.02,35.52,4.387381553649902
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",83,2026185283818,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,10.08,16.11,6.0225136280059814
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",150,2026181883783,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,81.85,63.55,4.6412224769592285
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",28,2026186983814,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.06,35.08,4.304075241088867
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",84,2026185283819,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,10.01,16.16,6.597375869750977
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",151,2026181883784,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,81.675,65.35,3.993192672729492
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",29,2026186983815,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,25.83,35.86,4.0215253829956055
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",0,2026188683837,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.85,64.1,5.840991020202637
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",152,2026181883785,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.0,64.05,4.43517541885376
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",30,2026186983816,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.58,36.92,5.640515089035034
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",85,2026185283820,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,2.0,2,0,9.6975,15.705,7.4941301345825195
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",1,2026188683838,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.725,66.7,5.149625301361084
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",153,2026181883786,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.675,65.15,5.1362690925598145
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",31,2026186983817,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.23,35.72,4.58202600479126
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",2,2026188683839,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,1.0,1,0,56.125,64.35,4.718433380126953
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",86,2026185283821,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,2.0,2,0,9.9625,16.225,6.956548690795898
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",154,2026181883787,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,1.0,1,0,84.1,63.65,5.0781848430633545
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",32,2026186983818,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.02,35.68,4.4297935962677
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",3,2026188683840,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,53.75,62.65,5.098930597305298
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",155,2026181883788,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.225,61.45,4.6765642166137695
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",87,2026185283822,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,2.0,2,0,10.215,16.02,6.187226295471191
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",33,2026186983819,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.55,36.24,5.205243110656738
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",4,2026188683841,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,52.2,63.5,4.5699028968811035
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",156,2026181883789,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.0,62.8,5.236498594284058
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",34,2026186983820,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.52,35.68,5.339791774749756
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",5,2026188683842,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.1,62.85,3.9950497150421143
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",88,2026185283823,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.88,16.11,6.202368259429932
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",157,2026181883790,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.65,63.85,4.27398157119751
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",35,2026186983821,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.97,35.58,4.205392599105835
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",6,2026188683843,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,56.8,63.3,4.829823732376099
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",89,2026185283824,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,2.0,2,0,10.0975,16.285,7.0657806396484375
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",158,2026181883791,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,81.875,64.0,4.830713272094727
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",36,2026186983822,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.43,35.9,4.369099855422974
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",7,2026188683844,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,53.625,63.85,5.143062591552734
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",159,2026181883792,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,81.5,64.9,5.158700466156006
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",37,2026186983823,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.26,36.76,4.908802032470703
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",90,2026185283825,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,3.0,3,0,10.31,15.82,6.224738597869873
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",8,2026188683845,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.475,64.6,5.02644944190979
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",160,2026181883793,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,85.775,64.85,4.261616230010986
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",38,2026186983824,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.58,36.12,5.40641450881958
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",9,2026188683846,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.15,63.65,4.87201189994812
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",91,2026185283826,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,9.96,16.105,6.345468282699585
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",161,2026181883794,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,86.3,66.75,4.608651161193848
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",39,2026186983825,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.01,35.36,4.438206672668457
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",10,2026188683847,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.825,61.35,5.019442081451416
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",92,2026185283827,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,2.0,2,0,10.1975,15.925,6.205318212509155
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",162,2026181883795,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.45,64.15,5.185590744018555
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",40,2026186983826,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.27,35.6,5.1122050285339355
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",11,2026188683848,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.975,63.0,4.178284645080566
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",163,2026181883796,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.5,1,1,82.2,64.25,5.009986877441406
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",93,2026185283828,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.0225,16.27,6.142300844192505
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",41,2026186983827,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.62,35.68,4.9287331104278564
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",12,2026188683849,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,1.0,1,0,54.875,65.1,4.665113687515259
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",164,2026181883797,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.4,64.65,4.625289678573608
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",13,2026188683850,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,56.325,63.35,4.713654518127441
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",42,2026186983828,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.58,36.04,5.4907920360565186
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",94,2026185283829,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.255,16.175,6.882821321487427
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",165,2026181883798,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,81.675,63.9,4.359620094299316
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",14,2026188683851,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.6,64.0,4.065525054931641
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",43,2026186983829,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.34,35.44,5.690856456756592
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",166,2026181883799,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,84.775,65.65,4.1098761558532715
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",95,2026185283830,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,2.0,2,0,10.13,15.86,6.677279233932495
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",15,2026188683852,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.7,62.9,5.558949947357178
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",44,2026186983830,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,1.0,1,0,26.4,34.96,4.970334768295288
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",167,2026181883800,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,85.65,61.7,4.24064040184021
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",96,2026185283831,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.12,16.01,6.368216037750244
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",16,2026188683853,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.825,64.6,4.891148328781128
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",45,2026186983831,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.08,36.76,4.204200029373169
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",168,2026181883801,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.8,64.15,5.195029258728027
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",46,2026186983832,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.88,35.2,4.790536880493164
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",97,2026185283832,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.1175,16.175,6.479047060012817
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",17,2026188683854,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,56.625,65.85,5.929100036621094
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",169,2026181883802,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.1,64.95,4.560499429702759
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",47,2026186983833,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,1.0,1,0,26.52,37.02,4.591771364212036
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",170,2026181883803,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,81.9,63.5,4.522876977920532
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",18,2026188683855,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.05,64.1,5.1738739013671875
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",98,2026185283833,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.96,16.115,6.700850009918213
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",48,2026186983834,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.5,1,1,26.68,34.74,4.483277082443237
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",171,2026181883804,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.2,66.05,4.7564473152160645
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",19,2026188683856,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.875,64.05,4.78585410118103
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",99,2026185283834,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.1975,16.085,6.5482025146484375
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",49,2026186983835,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.39,35.98,5.2754576206207275
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",172,2026181883805,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.25,65.1,4.3553712368011475
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",20,2026188683857,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,57.325,63.4,4.6024956703186035
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",173,2026181883806,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,81.75,60.7,5.0267274379730225
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",50,2026186983836,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.68,35.4,5.176072359085083
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",100,2026185283835,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,10.2,15.975,6.441522121429443
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",21,2026188683858,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,53.75,66.25,5.345123052597046
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",51,2026186983837,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,1.0,1,0,27.22,35.76,4.823989629745483
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",174,2026181883807,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.8,63.2,5.321948528289795
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",22,2026188683859,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,56.4,62.65,4.976382493972778
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",101,2026185283836,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,2.5,3,1,9.965,15.965,6.376079797744751
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",175,2026181883808,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.325,63.95,4.063628673553467
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",52,2026186983838,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,1.0,1,0,26.9,36.16,5.301552772521973
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",23,2026188683860,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,53.95,63.55,5.634363651275635
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",102,2026185283837,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,10.4,15.835,6.1282196044921875
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",53,2026186983839,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.22,34.54,4.16657018661499
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",176,2026181883809,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.025,63.3,5.173670053482056
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",24,2026188683861,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.6,64.8,4.397257566452026
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",177,2026181883810,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.325,65.45,3.181828022003174
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",54,2026186983840,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.82,35.8,3.4281513690948486
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",103,2026185283838,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.5,1,1,10.38,16.285,4.980974912643433
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",25,2026188683862,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,53.725,63.5,4.018547773361206
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",178,2026181883811,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.5,1,1,81.525,65.65,3.1477015018463135
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",55,2026186983841,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.22,36.02,3.3843159675598145
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",104,2026185283839,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.9875,16.1,4.559609413146973
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",26,2026188683863,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,56.0,63.5,3.9462132453918457
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",179,2026181883812,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,85.375,62.15,3.191687822341919
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",56,2026186983842,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,25.97,36.38,3.7360031604766846
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",105,2026185283840,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.285,16.28,4.59239935874939
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",27,2026188683864,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,56.425,63.7,3.718111276626587
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",180,2026181883813,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.15,64.55,3.7882156372070312
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",57,2026186983843,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,25.97,34.72,3.6634140014648438
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",28,2026188683865,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.4,65.4,3.9023969173431396
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",181,2026181883814,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.75,64.85,3.7288334369659424
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",106,2026185283841,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.5,2,1,10.1325,16.23,4.7715229988098145
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",58,2026186983844,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.65,36.2,3.4902656078338623
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",182,2026181883815,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,84.725,62.0,3.424513816833496
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",29,2026188683866,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.05,64.0,3.816591739654541
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",59,2026186983845,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.04,35.14,3.591493844985962
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",107,2026185283842,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.255,15.865,4.533828496932983
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",183,2026181883816,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.3,64.55,4.972139120101929
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",60,2026186983846,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.28,36.08,5.474196434020996
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",30,2026188683867,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,53.5,67.05,6.176398277282715
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",108,2026185283843,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,10.005,15.965,6.3750035762786865
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",184,2026181883817,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,1.0,1,0,82.975,62.25,4.773474454879761
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",61,2026186983847,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.62,34.94,5.302468299865723
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",31,2026188683868,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.325,62.75,5.298880338668823
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",185,2026181883818,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,81.525,65.05,4.699392080307007
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",109,2026185283844,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,10.0375,15.96,7.315118074417114
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",62,2026186983848,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.22,34.9,5.643082141876221
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",32,2026188683869,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.65,64.8,5.557696104049683
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",186,2026181883819,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.375,65.0,5.118019104003906
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",110,2026185283845,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,9.9825,15.79,6.362906455993652
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",63,2026186983849,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,1.0,1,0,26.49,35.92,4.7489914894104
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",33,2026188683870,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.875,62.95,4.750157594680786
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",187,2026181883820,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,81.875,65.3,4.5523154735565186
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",34,2026188683871,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,53.45,63.8,4.779389142990112
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",64,2026186983850,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.64,35.3,5.248120546340942
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",111,2026185283846,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.19,16.105,7.875330924987793
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",188,2026181883821,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,81.225,65.7,5.197821855545044
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",65,2026186983851,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.83,35.9,4.414645671844482
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",35,2026188683872,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.325,60.85,4.674163579940796
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",189,2026181883822,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.475,65.1,4.617670059204102
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",112,2026185283847,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,2.0,3,2,10.315,15.475,6.502208709716797
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",66,2026186983852,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,1.0,1,0,26.99,36.76,5.182848215103149
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",36,2026188683873,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.75,63.35,5.595644950866699
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",190,2026181883823,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,81.0,62.55,5.563603401184082
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",67,2026186983853,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.36,35.14,4.948601007461548
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",113,2026185283848,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,9.9925,16.025,5.991543292999268
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",37,2026188683874,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.85,64.55,4.827667236328125
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",191,2026181883824,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.125,63.3,5.20661187171936
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",68,2026186983854,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.23,34.68,4.685370683670044
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",38,2026188683875,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.325,63.05,5.216524600982666
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",114,2026185283849,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.0125,16.26,6.257181167602539
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",192,2026181883825,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.425,66.25,4.472618103027344
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",69,2026186983855,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.42,36.16,3.9670040607452393
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",193,2026181883826,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.4,63.7,4.379798412322998
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",39,2026188683876,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.8,63.45,6.741379499435425
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",115,2026185283850,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,9.8875,16.075,6.7254109382629395
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",70,2026186983856,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.82,36.34,5.02424430847168
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",194,2026181883827,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,1.0,1,0,79.625,63.15,4.668278932571411
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",40,2026188683877,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,57.325,64.05,5.853706359863281
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",71,2026186983857,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,1.0,1,0,25.89,36.5,5.425447702407837
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",116,2026185283851,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,2.0,3,2,10.435,15.95,6.203023433685303
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",195,2026181883828,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,84.2,62.2,5.090868949890137
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",41,2026188683878,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,52.525,64.1,5.613592624664307
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",72,2026186983858,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.94,36.66,5.519070386886597
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",117,2026185283852,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.6275,16.14,6.754968643188477
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",196,2026181883829,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,81.85,63.85,4.731677532196045
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",42,2026188683879,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,57.225,63.3,5.705010652542114
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",73,2026186983859,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.23,36.86,5.643640518188477
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",197,2026181883830,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.5,1,1,86.175,62.35,4.906874179840088
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",118,2026185283853,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,3.0,3,0,9.8625,15.935,7.228874444961548
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",43,2026188683880,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.5,65.15,5.2775068283081055
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",74,2026186983860,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.14,36.42,5.515223741531372
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",198,2026181883831,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,1.0,1,0,81.25,64.9,4.872617244720459
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",44,2026188683881,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.475,65.05,4.374820947647095
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",119,2026185283854,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.2025,16.305,6.525050640106201
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",75,2026186983861,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,1.0,1,0,25.98,35.22,5.104520797729492
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",199,2026181883832,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,1.0,1,0,81.875,65.15,5.520384073257446
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",45,2026188683882,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,56.175,64.2,5.2774646282196045
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",76,2026186983862,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.41,35.44,4.747161149978638
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",200,2026181883833,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,86.025,63.15,4.407512664794922
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",120,2026185283855,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,2,2,10.1775,15.995,6.721415996551514
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",46,2026188683883,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.75,62.5,5.620282411575317
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",201,2026181883834,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,1.0,1,0,82.775,63.0,4.306583404541016
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",77,2026186983863,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.4,36.8,5.70517897605896
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",121,2026185283856,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,10.0825,16.075,6.943605184555054
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",47,2026188683884,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.2,62.75,4.614221811294556
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",202,2026181883835,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,81.8,65.75,5.209103584289551
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",78,2026186983864,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.92,36.42,7.261159420013428
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",48,2026188683885,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.3,63.8,5.138772964477539
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",203,2026181883836,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.5,1,1,82.4,63.15,5.102558374404907
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",122,2026185283857,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.5,2,1,10.2825,15.91,8.357940912246704
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",79,2026186983865,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.83,35.5,5.364340782165527
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",49,2026188683886,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,53.85,63.3,4.577074289321899
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",204,2026181883837,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,81.325,62.9,5.674598693847656
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",123,2026185283858,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,2.0,2,0,10.2375,16.0,6.019518613815308
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",80,2026186983866,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.51,36.58,4.764026641845703
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",50,2026188683887,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,56.05,64.15,4.751029014587402
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",205,2026181883838,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.55,64.0,4.760613918304443
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",81,2026186983867,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.18,35.46,4.923125982284546
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",51,2026188683888,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.3,65.35,5.638699293136597
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",124,2026185283859,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,2.0,2,0,9.965,16.195,7.0607264041900635
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",206,2026181883839,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.075,64.85,4.601316452026367
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",82,2026186983868,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.44,35.94,5.648185729980469
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",52,2026188683889,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,57.3,64.85,5.163418531417847
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",125,2026185283860,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,9.96,15.855,5.9156293869018555
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",207,2026181883840,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.5,63.7,5.285299301147461
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",83,2026186983869,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.5,1,1,26.76,34.88,5.135685205459595
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",208,2026181883841,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.225,64.8,4.548864126205444
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",53,2026188683890,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,1.0,1,0,55.2,65.65,6.278810501098633
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",126,2026185283861,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.945,16.005,6.7303807735443115
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",209,2026181883842,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.3,64.7,3.6664187908172607
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",84,2026186983870,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.91,36.34,5.247407674789429
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",54,2026188683891,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.425,64.0,4.63852071762085
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",127,2026185283862,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,2.0,2,0,10.1,16.18,6.358371734619141
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",210,2026181883843,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.475,63.85,4.905738353729248
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",85,2026186983871,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.38,36.16,5.170975923538208
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",55,2026188683892,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.325,64.4,5.660948991775513
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",211,2026181883844,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,81.75,64.15,4.677962303161621
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",128,2026185283863,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,10.025,16.145,6.159141302108765
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",86,2026186983872,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,1.0,1,0,26.35,36.08,5.578697681427002
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",56,2026188683893,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.5,63.35,4.962804555892944
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",212,2026181883845,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,81.25,62.45,5.235181570053101
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",129,2026185283864,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.96,16.23,6.128177642822266
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",57,2026188683894,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.35,62.65,4.835669755935669
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",87,2026186983873,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.15,36.68,6.049410581588745
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",213,2026181883846,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,1.0,1,0,85.175,62.95,5.436744213104248
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",88,2026186983874,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.03,36.44,5.1864423751831055
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",58,2026188683895,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.975,64.8,5.491440773010254
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",130,2026185283865,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.9125,15.915,6.311890125274658
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",214,2026181883847,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.225,65.85,4.364008903503418
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",59,2026188683896,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.4,63.85,4.692948818206787
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",89,2026186983875,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.23,36.12,4.9117271900177
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",215,2026181883848,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,84.375,63.0,5.671869277954102
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",131,2026185283866,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,3.0,3,0,10.0725,16.215,7.771513938903809
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",90,2026186983876,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.01,36.3,4.474873781204224
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",60,2026188683897,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.525,63.95,5.228625059127808
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",216,2026181883849,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.35,64.85,4.976405143737793
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",91,2026186983877,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.79,36.1,4.9609293937683105
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",132,2026185283867,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.9775,16.005,6.67023229598999
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",61,2026188683898,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,53.225,64.15,5.6634461879730225
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",217,2026181883850,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,84.325,64.3,4.1489577293396
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",92,2026186983878,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.12,37.14,4.873773813247681
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",218,2026181883851,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,84.625,63.6,4.770702362060547
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",62,2026188683899,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.65,65.3,6.540838956832886
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",133,2026185283868,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,3.0,3,0,10.0575,16.255,7.471205234527588
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",93,2026186983879,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.8,35.28,4.285366773605347
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",219,2026181883852,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,85.65,62.2,4.186654567718506
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",63,2026188683900,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.775,65.0,6.017514705657959
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",134,2026185283869,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,10.1925,15.96,6.513622522354126
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",94,2026186983880,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.97,36.74,5.801966667175293
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",220,2026181883853,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,81.825,63.9,4.937658309936523
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",64,2026188683901,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.325,67.1,4.1099653244018555
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",95,2026186983881,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.1,36.18,4.956535339355469
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",135,2026185283870,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.2825,15.98,6.972816467285156
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",221,2026181883854,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.225,64.45,6.3050830364227295
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",65,2026188683902,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,53.775,65.05,5.280519008636475
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",96,2026186983882,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.32,37.14,4.9994893074035645
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",222,2026181883855,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.35,63.6,4.535771608352661
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",136,2026185283871,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,10.15,15.955,6.190693140029907
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",66,2026188683903,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.375,64.0,5.720898151397705
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",97,2026186983883,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,1.0,1,0,26.15,36.22,5.1009886264801025
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",223,2026181883856,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.2,64.65,4.313320159912109
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",67,2026188683904,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.3,63.1,5.045880079269409
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",137,2026185283872,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,3.0,3,0,10.2075,16.18,6.9071128368377686
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",224,2026181883857,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,81.525,63.9,4.048380374908447
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",98,2026186983884,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.5,1,1,26.87,36.28,5.812494516372681
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",68,2026188683905,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,1.0,1,0,55.075,63.3,5.300104856491089
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",225,2026181883858,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,81.0,63.6,4.482348442077637
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",99,2026186983885,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,1.0,1,0,26.31,35.44,4.767199277877808
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",138,2026185283873,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.905,15.815,7.130754470825195
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",226,2026181883859,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,80.7,63.25,4.49430513381958
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",69,2026188683906,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.5,66.1,5.678104877471924
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",100,2026186983886,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.58,36.68,4.978971242904663
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",139,2026185283874,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,2.0,2,0,9.905,16.175,6.710425615310669
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",70,2026188683907,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.025,63.25,4.1927406787872314
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",101,2026186983887,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.89,36.06,5.2119128704071045
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",227,2026181883860,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,1.0,1,0,83.1,64.75,6.1521947383880615
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",102,2026186983888,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.76,36.38,3.9107508659362793
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",140,2026185283875,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.3325,15.905,6.079238176345825
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",71,2026188683908,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.85,63.7,5.9399635791778564
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",228,2026181883861,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,80.825,64.2,5.557373285293579
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",103,2026186983889,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.24,36.48,4.907058238983154
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",72,2026188683909,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.875,63.85,5.778836488723755
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",141,2026185283876,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,10.2775,15.745,6.4637415409088135
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",229,2026181883862,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,81.85,63.7,5.06207537651062
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",104,2026186983890,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.62,36.02,4.468602657318115
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",230,2026181883863,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,1.0,1,0,84.0,64.15,4.368606805801392
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",73,2026188683910,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.15,64.45,6.518624782562256
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",142,2026185283877,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.0425,16.155,6.370449781417847
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",105,2026186983891,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.88,36.1,5.935234546661377
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",231,2026181883864,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,1.0,1,0,82.25,62.8,4.481473445892334
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",74,2026188683911,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,56.7,63.2,5.03078031539917
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",143,2026185283878,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,10.055,16.085,6.363684892654419
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",106,2026186983892,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,1.0,1,0,26.81,36.98,5.405671119689941
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",232,2026181883865,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,79.95,64.15,5.195532560348511
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",75,2026188683912,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.825,62.9,4.824792146682739
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",107,2026186983893,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.5,1,1,26.57,35.48,4.962941884994507
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",144,2026185283879,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.8975,16.035,6.301941633224487
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",233,2026181883866,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,84.825,64.7,5.083762168884277
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",76,2026188683913,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.25,64.3,6.797320604324341
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",234,2026181883867,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.275,61.95,4.537441968917847
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",145,2026185283880,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.21,16.005,5.3089189529418945
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",108,2026186983894,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.85,35.58,5.697443962097168
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",77,2026188683914,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,53.375,62.25,4.94267201423645
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",235,2026181883868,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,81.825,66.6,5.044170379638672
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",109,2026186983895,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.62,35.48,4.523772954940796
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",146,2026185283881,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,3.0,3,0,10.3275,16.2,7.243408441543579
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",78,2026188683915,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.25,63.55,5.144283056259155
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",236,2026181883869,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,81.475,64.4,4.919111251831055
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",110,2026186983896,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.29,36.14,4.847444534301758
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",147,2026185283882,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,9.895,15.925,6.416110277175903
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",237,2026181883870,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,84.15,64.45,4.32159948348999
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",79,2026188683916,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.9,62.05,5.687412738800049
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",111,2026186983897,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.5,1,1,26.12,36.1,6.057007074356079
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",238,2026181883871,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,81.525,64.85,4.430179834365845
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",80,2026188683917,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.075,62.85,5.1531805992126465
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",148,2026185283883,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,9.6875,16.25,5.90397047996521
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",112,2026186983898,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,25.68,36.56,5.897250652313232
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",239,2026181883872,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.1,63.1,5.305887460708618
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",149,2026185283884,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.0,16.025,5.218255043029785
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",81,2026188683918,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.05,63.6,5.818849802017212
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",113,2026186983899,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,1.0,1,0,26.62,37.02,4.846529006958008
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",240,2026181883873,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,84.3,62.7,4.801564931869507
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",82,2026188683919,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.4,63.7,4.657526969909668
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",150,2026185283885,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.5,2,1,10.3725,15.98,6.604308128356934
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",114,2026186983900,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.4,36.54,6.548781394958496
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",241,2026181883874,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.1,63.2,5.271009206771851
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",83,2026188683920,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,52.825,61.6,5.369017124176025
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",151,2026185283886,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.9975,16.195,5.885520696640015
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",115,2026186983901,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.0,36.48,4.789791584014893
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",242,2026181883875,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,1.5,2,1,83.9,66.3,5.404728889465332
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",84,2026188683921,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,1.0,1,0,53.8,62.7,5.575776815414429
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",116,2026186983902,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,25.53,36.48,5.334423780441284
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",152,2026185283887,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,9.9375,16.21,6.341843605041504
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",243,2026181883876,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.325,62.1,5.47416090965271
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",85,2026188683922,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.275,64.0,5.3091819286346436
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",244,2026181883877,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,84.15,64.3,3.9510014057159424
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",117,2026186983903,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,25.91,36.5,5.411147356033325
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",153,2026185283888,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,10.12,16.34,5.787951707839966
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",86,2026188683923,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,56.075,63.9,5.993866920471191
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",245,2026181883878,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.425,62.75,5.1162331104278564
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",118,2026186983904,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.09,35.96,5.048654556274414
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",154,2026185283889,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,9.92,16.14,6.817013502120972
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",119,2026186983905,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.82,35.84,4.157327651977539
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",246,2026181883879,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,84.7,63.95,4.72146463394165
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",87,2026188683924,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,56.025,63.4,6.444118499755859
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",155,2026185283890,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.5,1,1,10.065,16.255,5.552257776260376
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",120,2026186983906,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.54,36.42,4.799455642700195
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",247,2026181883880,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.975,64.85,4.958629131317139
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",88,2026188683925,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,57.1,64.35,5.852753400802612
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",156,2026185283891,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,10.2,15.905,6.079258680343628
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",121,2026186983907,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.5,1,1,26.83,35.66,5.269885063171387
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",248,2026181883881,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,1.0,1,0,85.75,66.2,4.978948354721069
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",89,2026188683926,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,1.0,1,0,56.55,62.8,5.359020948410034
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",249,2026181883882,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,82.1,65.7,3.8448309898376465
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",122,2026186983908,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.73,36.22,4.945730447769165
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",157,2026185283892,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.9975,15.865,7.023561716079712
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",90,2026188683927,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.3,63.8,5.428412675857544
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",250,2026181883883,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,1.0,1,0,82.5,64.9,4.411106824874878
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",123,2026186983909,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.22,35.62,5.709528923034668
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",91,2026188683928,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,56.25,65.5,5.301611423492432
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",158,2026185283893,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.9325,16.12,6.136981725692749
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0003|pi0.0003|k1,1,1",251,2026181883884,A,noisy,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,83.0,60.4,5.3618574142456055
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",124,2026186983910,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.76,36.04,4.59478759765625
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",0,2026190383888,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.125,4.1,1.5835118293762207
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",1,2026190383889,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,4.0,4,0,1.125,3.945,1.0158255100250244
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",92,2026188683929,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.8,63.15,4.512682199478149
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",2,2026190383890,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,2.5,3,1,1.1725,4.11,1.2909817695617676
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",3,2026190383891,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.2125,4.0,1.227114200592041
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",125,2026186983911,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,1.0,1,0,28.3,36.96,4.538732290267944
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",159,2026185283894,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,2.0,2,0,9.98,15.665,6.783443212509155
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",4,2026190383892,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.1725,3.955,1.1854126453399658
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",93,2026188683930,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,57.9,63.3,4.867891788482666
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",5,2026190383893,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.1225,4.035,1.6168928146362305
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",6,2026190383894,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0375,3.965,1.6726398468017578
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",126,2026186983912,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,1.0,1,0,26.39,36.68,5.010632514953613
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",7,2026190383895,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.15,4.0,1.2519919872283936
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",8,2026190383896,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.1725,3.89,1.3968110084533691
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",160,2026185283895,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.32,16.17,7.582509517669678
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",9,2026190383897,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.27,3.91,1.5649664402008057
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",94,2026188683931,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,56.375,65.05,6.823799133300781
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",10,2026190383898,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.125,3.74,1.2473576068878174
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",127,2026186983913,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.27,36.56,5.247918128967285
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",11,2026190383899,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.2475,3.955,0.9680180549621582
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",12,2026190383900,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.2275,4.065,1.5667107105255127
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",13,2026190383901,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.1875,4.03,1.4082436561584473
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",161,2026185283896,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.0725,15.91,6.435380220413208
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",95,2026188683932,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,57.95,63.35,5.912025213241577
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",128,2026186983914,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.31,35.6,4.783278226852417
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",14,2026190383902,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.1475,3.93,1.833036184310913
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",15,2026190383903,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.095,4.175,1.1271512508392334
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",16,2026190383904,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.1825,4.095,1.2276065349578857
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",17,2026190383905,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.08,4.02,1.836911916732788
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",96,2026188683933,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.0,64.9,5.081230878829956
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",129,2026186983915,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.62,36.46,5.287809610366821
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",18,2026190383906,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.21,3.77,1.2059154510498047
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",162,2026185283897,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.0625,16.01,7.299635171890259
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",19,2026190383907,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.185,4.0,1.2849390506744385
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",20,2026190383908,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,3.5,4,1,1.295,3.98,1.3264834880828857
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",21,2026190383909,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.1925,3.995,1.067143440246582
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",97,2026188683934,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,56.5,64.45,5.111436367034912
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",130,2026186983916,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,1.0,1,0,25.82,35.92,5.528886318206787
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",22,2026190383910,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.24,3.9,1.2176003456115723
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",23,2026190383911,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.17,3.98,1.3314898014068604
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",24,2026190383912,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.1925,4.155,1.292853832244873
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",163,2026185283898,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,2.0,2,0,10.0775,16.015,7.15135383605957
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",25,2026190383913,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.1875,4.005,1.8612148761749268
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",98,2026188683935,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.3,63.9,5.169182538986206
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",131,2026186983917,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.42,36.6,4.891782760620117
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",26,2026190383914,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.1275,3.86,1.4938106536865234
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",27,2026190383915,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.16,4.1,1.6250410079956055
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",28,2026190383916,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.12,3.92,1.3463082313537598
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",164,2026185283899,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.5,1,1,9.7225,15.965,6.931712865829468
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",99,2026188683936,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.65,64.6,5.666492223739624
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",29,2026190383917,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.22,3.99,1.3598616123199463
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",132,2026186983918,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.62,35.68,6.094835519790649
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",30,2026190383918,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.1775,4.025,1.3269166946411133
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",31,2026190383919,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.175,4.05,1.02158784866333
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",32,2026190383920,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.16,3.79,1.1029713153839111
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",33,2026190383921,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.145,3.91,1.1992883682250977
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",100,2026188683937,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.8,65.95,5.978965759277344
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",34,2026190383922,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.185,4.175,1.1960039138793945
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",133,2026186983919,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,1.0,1,0,26.66,35.26,5.874723196029663
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",165,2026185283900,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.175,15.865,7.719282388687134
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",35,2026190383923,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.5,2,1,1.2625,3.92,1.3857994079589844
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",36,2026190383924,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.1575,4.125,1.3323006629943848
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",37,2026190383925,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.19,4.145,1.34920334815979
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",101,2026188683938,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,56.625,63.05,4.850383758544922
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",38,2026190383926,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.1375,4.05,0.977576732635498
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",134,2026186983920,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.15,35.56,5.404220342636108
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",39,2026190383927,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.225,3.975,1.3348667621612549
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",40,2026190383928,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,3.0,3,0,1.2875,3.995,0.9991791248321533
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",166,2026185283901,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.9325,16.27,6.87612771987915
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",41,2026190383929,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.2125,4.005,1.2582178115844727
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",42,2026190383930,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.17,3.975,1.5287978649139404
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",102,2026188683939,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.275,62.15,5.686041355133057
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",135,2026186983921,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.04,36.36,4.914179086685181
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",43,2026190383931,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.17,4.12,1.0760095119476318
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",44,2026190383932,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.1675,3.93,1.1209120750427246
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",45,2026190383933,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.2175,4.16,1.1046044826507568
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",167,2026185283902,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.68,15.625,6.010240793228149
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",46,2026190383934,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.2525,3.945,1.091834306716919
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",47,2026190383935,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.2,4.06,1.2386155128479004
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",136,2026186983922,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.39,35.14,5.326578855514526
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",103,2026188683940,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.55,63.2,5.969435214996338
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",48,2026190383936,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,3.0,3,0,1.215,3.825,1.1922571659088135
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",49,2026190383937,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.16,4.08,1.0971786975860596
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",50,2026190383938,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.185,3.99,0.8752326965332031
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",51,2026190383939,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.1975,4.065,1.1588349342346191
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",168,2026185283903,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,3.0,3,0,10.2,16.295,6.349538326263428
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",52,2026190383940,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.1475,3.995,1.1651005744934082
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",137,2026186983923,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.08,36.14,5.440199613571167
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",53,2026190383941,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.13,4.075,0.9441080093383789
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",104,2026188683941,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.8,61.15,6.868381023406982
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",54,2026190383942,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.1375,4.025,1.2139434814453125
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",55,2026190383943,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.105,3.81,1.147670030593872
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",56,2026190383944,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.2825,4.26,1.5964009761810303
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",138,2026186983924,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.66,35.76,5.066014528274536
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",169,2026185283904,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,9.94,15.805,7.104767560958862
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",57,2026190383945,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.205,4.105,1.657752275466919
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",105,2026188683942,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.375,64.15,5.314675569534302
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",58,2026190383946,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.1475,4.03,1.1891303062438965
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",59,2026190383947,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.1475,3.975,1.0190696716308594
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",60,2026190383948,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.1375,4.17,1.0577380657196045
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",139,2026186983925,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.04,36.02,5.072826862335205
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",61,2026190383949,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.2575,3.885,1.2283923625946045
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",62,2026190383950,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.2475,3.905,1.080857515335083
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",106,2026188683943,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.775,65.95,5.400669097900391
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",170,2026185283905,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.5,2,1,10.1825,16.155,6.958937406539917
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",63,2026190383951,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.1775,4.155,1.2229595184326172
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",64,2026190383952,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.175,3.895,1.4302058219909668
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",65,2026190383953,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,4.0,4,0,1.105,4.02,0.9869725704193115
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",140,2026186983926,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.06,37.44,6.152089595794678
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",66,2026190383954,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.2425,4.095,1.1259305477142334
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",67,2026190383955,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.225,3.965,0.9790487289428711
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",107,2026188683944,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,57.85,64.75,5.455209255218506
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",171,2026185283906,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,9.995,15.87,5.635777711868286
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",68,2026190383956,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.15,4.16,1.514972448348999
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",69,2026190383957,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,3.0,3,0,1.1075,4.185,1.3433411121368408
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",70,2026190383958,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.115,3.845,0.9468820095062256
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",141,2026186983927,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.89,36.68,5.657350540161133
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",108,2026188683945,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.55,63.35,4.629311800003052
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",71,2026190383959,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.18,4.025,1.6711680889129639
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",172,2026185283907,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.86,15.91,5.368900299072266
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",72,2026190383960,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.1875,4.065,1.5236999988555908
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",73,2026190383961,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.165,4.075,0.9628221988677979
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",142,2026186983928,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.22,36.24,4.4121716022491455
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",74,2026190383962,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.1425,4.095,0.9357211589813232
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",75,2026190383963,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,2.5,3,1,1.235,3.985,0.973325252532959
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",109,2026188683946,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.625,64.1,5.600177526473999
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",76,2026190383964,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,3.0,3,0,1.195,4.07,1.8218753337860107
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",173,2026185283908,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,10.13,16.18,6.590958833694458
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",77,2026190383965,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,3.0,3,0,1.16,4.035,1.3444788455963135
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",143,2026186983929,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.15,37.08,4.82450008392334
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",78,2026190383966,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.5,2,1,1.1625,3.995,1.239004135131836
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",79,2026190383967,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,5.0,5,0,1.2175,3.98,1.2511157989501953
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",80,2026190383968,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.075,3.96,1.2157597541809082
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",110,2026188683947,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.025,64.25,6.790483236312866
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",81,2026190383969,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.1575,4.135,0.9042434692382812
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",144,2026186983930,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,1.0,1,0,26.59,35.38,4.533332347869873
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",174,2026185283909,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.12,15.585,6.103929758071899
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",82,2026190383970,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.2825,4.04,1.4622364044189453
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",83,2026190383971,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.0975,4.18,1.2607243061065674
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",84,2026190383972,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.5,1,1,1.2075,3.885,1.0983169078826904
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",85,2026190383973,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.1725,4.005,1.0643935203552246
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",111,2026188683948,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,56.5,64.05,5.856152296066284
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",145,2026186983931,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.28,35.86,5.14227294921875
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",86,2026190383974,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.095,3.885,1.5772123336791992
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",87,2026190383975,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.1675,3.8,1.0535082817077637
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",175,2026185283910,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,10.005,15.78,6.496405601501465
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",88,2026190383976,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.2325,4.015,1.202458143234253
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",89,2026190383977,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.13,4.075,1.2356128692626953
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",112,2026188683949,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,56.85,64.8,5.431924819946289
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",90,2026190383978,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.1175,3.81,1.0433530807495117
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",146,2026186983932,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.63,35.6,5.53910756111145
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",91,2026190383979,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.21,4.07,1.4036781787872314
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",176,2026185283911,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,10.24,16.155,5.687427759170532
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",92,2026190383980,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.1925,3.92,1.5297067165374756
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",93,2026190383981,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.2475,4.055,1.1707079410552979
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",147,2026186983933,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.38,35.7,4.597913503646851
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",113,2026188683950,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,56.8,64.5,5.278829097747803
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",94,2026190383982,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.2175,3.97,1.2288122177124023
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",95,2026190383983,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.22,4.02,1.732830286026001
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",96,2026190383984,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.075,3.84,1.323744773864746
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",97,2026190383985,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.2075,3.91,1.2569811344146729
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",148,2026186983934,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.83,35.64,4.85921835899353
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",177,2026185283912,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,2.0,3,2,10.0675,16.115,7.473369598388672
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",114,2026188683951,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,56.65,64.05,5.080973148345947
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",98,2026190383986,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.155,3.99,1.3926661014556885
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",99,2026190383987,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.2625,4.075,1.3788630962371826
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",100,2026190383988,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.2125,4.08,1.186589241027832
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",101,2026190383989,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.1525,4.095,1.3032774925231934
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",178,2026185283913,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,10.3475,16.15,5.135789632797241
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",149,2026186983935,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.9,35.8,5.391855955123901
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",115,2026188683952,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.575,62.7,5.330643892288208
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",102,2026190383990,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.1775,3.94,1.9128520488739014
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",103,2026190383991,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.19,4.08,1.730731725692749
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",150,2026186983936,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.24,35.78,4.34894323348999
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",104,2026190383992,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.2,4.02,1.471508264541626
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",116,2026188683953,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.0,63.3,4.854341506958008
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",105,2026190383993,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.18,4.11,1.0770933628082275
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",179,2026185283914,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,10.1,15.895,6.931204795837402
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",106,2026190383994,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.2125,4.045,1.7114036083221436
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",107,2026190383995,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.175,3.98,0.863943338394165
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",151,2026186983937,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.18,35.98,4.843956232070923
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",108,2026190383996,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.315,4.04,1.4947044849395752
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",117,2026188683954,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.775,65.85,5.842973947525024
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",109,2026190383997,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.285,3.985,1.3314030170440674
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",110,2026190383998,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.1625,4.01,0.9705078601837158
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",180,2026185283915,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.5,2,1,9.975,15.99,6.093873977661133
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",152,2026186983938,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.15,37.56,4.015902519226074
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",111,2026190383999,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.2125,3.965,1.2373952865600586
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",112,2026190384000,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,2.5,3,1,1.305,4.31,1.0158467292785645
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",113,2026190384001,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.205,4.025,1.0921730995178223
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",114,2026190384002,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,3.0,3,0,1.2075,3.865,1.4004528522491455
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",118,2026188683955,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,53.875,64.25,5.900814056396484
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",115,2026190384003,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.21,3.98,1.1729791164398193
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",153,2026186983939,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.25,35.12,5.731389760971069
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",181,2026185283916,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.2475,15.845,6.744197845458984
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",116,2026190384004,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.1775,3.86,1.6739907264709473
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",117,2026190384005,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.165,3.94,1.2270705699920654
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",119,2026188683956,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.775,65.85,4.868882417678833
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",118,2026190384006,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,4.0,4,0,1.18,4.18,1.3309786319732666
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",119,2026190384007,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.1875,4.005,0.8579299449920654
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",154,2026186983940,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.68,35.86,4.970659255981445
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",120,2026190384008,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.0875,4.04,1.1540729999542236
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",182,2026185283917,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.5,1,1,9.9225,16.135,5.555265188217163
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",121,2026190384009,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.24,4.31,1.4768733978271484
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",122,2026190384010,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.1275,3.865,1.426490068435669
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",120,2026188683957,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.5,1,1,53.15,65.6,5.754876136779785
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",123,2026190384011,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,4.0,4,0,1.1925,4.1,1.1669645309448242
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",155,2026186983941,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.61,36.3,5.258217811584473
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",124,2026190384012,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.5,1,1,1.2525,3.94,0.9476993083953857
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",125,2026190384013,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.1975,3.995,1.1899313926696777
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",183,2026185283918,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,2.0,2,0,10.24,16.105,6.5128772258758545
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",126,2026190384014,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.2375,3.855,1.2268495559692383
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",121,2026188683958,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,53.8,63.85,5.062314033508301
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",127,2026190384015,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.16,4.12,1.1873645782470703
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",128,2026190384016,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.135,3.995,1.1203320026397705
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",156,2026186983942,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.84,35.64,5.630007028579712
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",129,2026190384017,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.225,4.2,1.1974186897277832
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",130,2026190384018,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.195,3.935,1.2545397281646729
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",184,2026185283919,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.125,16.1,6.075132608413696
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",131,2026190384019,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.26,4.035,1.1262767314910889
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",122,2026188683959,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,57.125,65.2,5.796380996704102
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",132,2026190384020,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.185,4.03,1.1431610584259033
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",157,2026186983943,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.4,36.84,5.297258138656616
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",133,2026190384021,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.16,3.93,1.3288483619689941
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",134,2026190384022,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.16,4.09,1.1497538089752197
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",135,2026190384023,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.185,3.905,1.5934035778045654
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",123,2026188683960,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,56.45,65.1,4.841353893280029
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",185,2026185283920,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,3.0,3,0,9.8675,16.155,5.906904220581055
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",158,2026186983944,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.5,1,1,26.82,35.52,4.380146503448486
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",136,2026190384024,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.18,4.08,1.8870315551757812
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",137,2026190384025,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.195,3.92,1.2692832946777344
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",138,2026190384026,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.105,4.125,1.2111523151397705
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",124,2026188683961,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.725,64.45,4.3895204067230225
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",139,2026190384027,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.205,4.085,0.9262306690216064
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",159,2026186983945,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,25.49,37.12,5.115887641906738
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",140,2026190384028,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.2975,3.9,1.3639981746673584
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",186,2026185283921,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,10.215,16.16,6.2807936668396
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",141,2026190384029,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.24,4.02,1.0187525749206543
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",142,2026190384030,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.175,4.125,1.5698308944702148
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",125,2026188683962,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,53.975,61.9,5.7005345821380615
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",143,2026190384031,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.1825,4.02,1.79203200340271
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",160,2026186983946,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.77,35.5,5.201369285583496
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",187,2026185283922,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.0,1,0,10.1675,15.81,5.147944450378418
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",144,2026190384032,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.1975,4.01,1.3554461002349854
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",145,2026190384033,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.23,4.085,1.1096606254577637
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",146,2026190384034,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.13,3.955,1.0692625045776367
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",161,2026186983947,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.52,36.04,4.075317621231079
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",147,2026190384035,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,3.0,3,0,1.2525,3.955,1.4279189109802246
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",126,2026188683963,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,53.0,63.6,5.965396881103516
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",148,2026190384036,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.1675,3.935,1.3155794143676758
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",149,2026190384037,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.19,4.175,1.071894645690918
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",188,2026185283923,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,2.0,2,0,9.8275,15.9,6.8698718547821045
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",162,2026186983948,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.74,37.22,4.444118022918701
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",150,2026190384038,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.2525,3.915,1.804260492324829
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",127,2026188683964,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,56.75,62.35,5.037409543991089
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",151,2026190384039,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.185,3.99,2.043062925338745
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",152,2026190384040,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.2075,3.83,1.5045828819274902
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",153,2026190384041,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.175,4.0,1.1237201690673828
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0002|pi0.0002|k1,1,1",189,2026185283924,A,noisy,5,0.0002,0.0002,1.0,1.0,1.0,5,5,5,400,1.5,2,1,9.785,16.09,6.198523759841919
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",163,2026186983949,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,1.0,1,0,26.67,36.04,4.962990760803223
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",128,2026188683965,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.825,63.9,4.470405578613281
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",154,2026190384042,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.38,4.065,1.1202971935272217
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",155,2026190384043,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.1675,4.1,1.6643435955047607
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",156,2026190384044,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,2.5,3,1,1.1425,3.825,0.8564565181732178
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",157,2026190384045,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.0975,4.02,0.8721966743469238
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",164,2026186983950,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.5,1,1,25.88,36.44,4.4144368171691895
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",158,2026190384046,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.25,3.82,1.113067626953125
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",129,2026188683966,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.925,65.35,5.38785719871521
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",0,2026192083939,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.96,16.085,5.71845006942749
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",159,2026190384047,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.2725,3.965,1.6144521236419678
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",160,2026190384048,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.5,2,1,1.155,3.95,1.1335680484771729
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",161,2026190384049,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,3.0,3,0,1.26,3.81,1.5163447856903076
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",165,2026186983951,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.43,36.48,5.505898714065552
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",130,2026188683967,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.225,64.2,4.596333742141724
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",162,2026190384050,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.1825,4.08,1.8576304912567139
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",1,2026192083940,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.135,15.84,5.463925838470459
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",163,2026190384051,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.1975,3.945,1.726576805114746
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",166,2026186983952,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.94,36.9,4.795181035995483
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",131,2026188683968,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.7,64.2,4.1967737674713135
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",164,2026190384052,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.2,4.1,1.82804274559021
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",165,2026190384053,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.2625,4.02,1.3418858051300049
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",2,2026192083941,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,1.0,1,0,5.0075,15.765,4.744151592254639
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",166,2026190384054,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.2,4.18,1.7516520023345947
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",132,2026188683969,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,56.775,62.6,4.334792613983154
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",167,2026190384055,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.125,4.07,1.4478769302368164
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",167,2026186983953,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.39,36.16,5.25418496131897
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",168,2026190384056,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.2375,3.935,1.9373972415924072
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",3,2026192083942,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.5,1,1,4.8325,15.875,4.958362102508545
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",169,2026190384057,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,1.5,2,1,1.245,3.88,1.4822664260864258
production-20261029T053925Z-29f79c,"A|noisy|d3|p0.0001|pi0.0001|k1,1,1",170,2026190384058,A,noisy,3,0.0001,0.0001,1.0,1.0,1.0,3,3,3,400,4.0,4,0,1.1425,3.94,0.9984626770019531
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",133,2026188683970,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,53.825,62.85,4.883211374282837
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",168,2026186983954,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.79,36.46,4.8245673179626465
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",4,2026192083943,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.2275,15.915,5.488866329193115
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",134,2026188683971,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,1.0,1,0,53.325,64.25,5.1867146492004395
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",169,2026186983955,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.9,36.68,4.83160662651062
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",0,2026193783990,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.84,36.06,4.433849334716797
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",5,2026192083944,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.9325,16.0,5.727274417877197
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",135,2026188683972,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.425,65.25,4.946819067001343
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",170,2026186983956,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,1.0,1,0,27.58,36.38,5.506967306137085
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",1,2026193783991,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.64,35.48,4.910578966140747
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",6,2026192083945,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.1425,15.845,5.599041700363159
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",171,2026186983957,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.17,36.34,4.81968355178833
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",136,2026188683973,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.975,62.5,6.042086839675903
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",2,2026193783992,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.75,36.08,4.371836423873901
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",7,2026192083946,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.9925,16.025,5.71047043800354
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",3,2026193783993,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.24,36.4,3.838860511779785
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",137,2026188683974,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.85,62.25,4.492690086364746
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",172,2026186983958,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.25,35.56,5.9909374713897705
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",4,2026193783994,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.27,36.22,4.446050643920898
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",8,2026192083947,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.96,16.09,5.52262544631958
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",138,2026188683975,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,56.675,64.5,5.632449626922607
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",173,2026186983959,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.22,36.66,4.644383192062378
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",5,2026193783995,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.23,35.76,5.341634750366211
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",174,2026186983960,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.89,36.46,4.527843952178955
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",9,2026192083948,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.985,16.12,5.8609535694122314
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",139,2026188683976,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.95,64.65,5.473510980606079
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",6,2026193783996,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,12.79,34.9,4.674428939819336
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",175,2026186983961,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.15,36.34,4.611431121826172
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",140,2026188683977,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.5,1,1,53.65,63.75,5.203681230545044
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",10,2026192083949,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.105,16.185,6.126597881317139
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",7,2026193783997,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.18,35.38,5.065685272216797
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",176,2026186983962,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.33,36.12,5.676795959472656
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",141,2026188683978,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,56.125,63.4,5.37894082069397
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",11,2026192083950,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.185,15.8,5.266963481903076
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",8,2026193783998,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.2,36.24,3.7599642276763916
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",177,2026186983963,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.61,35.64,5.656602382659912
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",142,2026188683979,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.575,63.6,6.029266357421875
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",9,2026193783999,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,12.99,35.46,4.526123285293579
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",12,2026192083951,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.335,16.175,5.624240398406982
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",178,2026186983964,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.5,1,1,27.25,36.58,5.62537956237793
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",10,2026193784000,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.16,36.82,4.163457870483398
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",143,2026188683980,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.9,65.1,4.778735160827637
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",13,2026192083952,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.04,16.21,6.357877731323242
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",179,2026186983965,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.5,36.68,4.704146862030029
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",11,2026193784001,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.25,35.32,4.932081460952759
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",144,2026188683981,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.075,63.55,5.627442359924316
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",14,2026192083953,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,1.0,1,0,4.9375,15.635,5.8555214405059814
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",12,2026193784002,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.41,36.1,4.501039981842041
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",180,2026186983966,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,1.0,1,0,26.84,34.72,4.860064744949341
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",145,2026188683982,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.675,63.15,4.800529718399048
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",181,2026186983967,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.95,35.1,4.535374402999878
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",15,2026192083954,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.9975,16.035,6.01920223236084
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",13,2026193784003,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.62,35.86,5.105533599853516
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",146,2026188683983,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.95,65.4,4.481677770614624
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",182,2026186983968,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.43,36.42,5.029603481292725
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",14,2026193784004,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,1.0,1,0,13.58,36.4,4.70393705368042
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",16,2026192083955,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.3475,15.945,5.732816219329834
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",147,2026188683984,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,58.375,64.6,5.6111390590667725
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",15,2026193784005,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,12.96,35.94,4.788589954376221
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",183,2026186983969,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,25.91,36.06,5.604217767715454
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",17,2026192083956,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.99,16.285,6.222678899765015
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",148,2026188683985,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,57.475,63.95,5.462170839309692
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",16,2026193784006,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.01,36.7,4.576574325561523
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",184,2026186983970,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.3,35.44,5.229172468185425
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",18,2026192083957,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.04,15.705,5.495026111602783
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",149,2026188683986,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,52.475,64.65,6.397757053375244
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",17,2026193784007,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.78,37.1,4.904821395874023
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",185,2026186983971,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.66,37.16,5.00494122505188
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",19,2026192083958,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.815,16.035,5.123169660568237
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",18,2026193784008,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,12.97,35.04,5.57632040977478
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",150,2026188683987,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.325,64.55,6.271573543548584
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",186,2026186983972,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.41,36.22,4.706713438034058
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",20,2026192083959,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.185,16.145,4.80107855796814
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",19,2026193784009,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.22,36.48,4.290854215621948
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",187,2026186983973,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.6,35.88,5.867082118988037
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",151,2026188683988,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.45,62.7,6.650348663330078
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",21,2026192083960,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.9675,16.085,4.41037130355835
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",20,2026193784010,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.69,36.22,4.235321998596191
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",188,2026186983974,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.27,36.16,5.078959703445435
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",152,2026188683989,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.325,66.6,5.176646709442139
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",21,2026193784011,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.67,36.18,3.7133169174194336
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",22,2026192083961,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.115,16.065,6.48141622543335
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",189,2026186983975,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.65,36.3,4.31451940536499
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",153,2026188683990,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,53.525,63.7,5.04461669921875
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",22,2026193784012,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.59,36.18,5.3796422481536865
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",23,2026192083962,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.045,15.725,5.783815860748291
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",190,2026186983976,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.37,34.94,4.878334045410156
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",154,2026188683991,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.75,62.65,5.0316548347473145
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",23,2026193784013,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,12.91,35.98,5.030251502990723
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",24,2026192083963,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.325,16.17,5.302042245864868
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",191,2026186983977,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.18,36.5,5.927456617355347
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",155,2026188683992,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,56.675,64.25,4.7666707038879395
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",24,2026193784014,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,12.68,35.58,5.150451421737671
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",25,2026192083964,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.5,1,1,4.9175,15.84,5.063188791275024
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",192,2026186983978,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.5,1,1,25.84,35.58,4.642388582229614
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",25,2026193784015,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,1.0,1,0,13.53,35.26,3.9865143299102783
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",156,2026188683993,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.9,63.05,5.833949327468872
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",26,2026192083965,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.1825,15.785,5.018433570861816
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",193,2026186983979,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.72,36.1,4.680609703063965
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",26,2026193784016,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.46,36.06,5.340155124664307
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",157,2026188683994,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,56.0,65.5,5.086089849472046
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",27,2026192083966,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.1025,16.02,5.606216669082642
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",194,2026186983980,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.42,36.1,4.758132219314575
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",27,2026193784017,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,1.0,1,0,13.48,35.06,5.486597776412964
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",158,2026188683995,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,56.45,64.4,5.026014566421509
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",28,2026192083967,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.11,16.205,4.924506664276123
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",195,2026186983981,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,1.0,1,0,27.07,36.14,5.152304410934448
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",28,2026193784018,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,12.82,35.46,5.415876388549805
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",159,2026188683996,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.1,62.1,6.246577024459839
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",29,2026192083968,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.0,15.85,5.0551371574401855
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",196,2026186983982,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.11,34.82,4.587024450302124
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",160,2026188683997,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.125,63.8,4.172242164611816
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",29,2026193784019,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.7,35.9,5.1914825439453125
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",30,2026192083969,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.075,16.045,5.479465484619141
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",197,2026186983983,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.91,36.64,6.145042896270752
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",30,2026193784020,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.12,35.6,5.003675699234009
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",161,2026188683998,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.5,1,1,54.825,65.3,5.424848318099976
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",31,2026192083970,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.985,15.72,4.864694118499756
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",198,2026186983984,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.5,1,1,26.76,35.32,5.034426212310791
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",162,2026188683999,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.3,63.15,4.556703090667725
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",31,2026193784021,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.1,37.08,5.505915641784668
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",32,2026192083971,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.085,16.275,5.3617424964904785
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",199,2026186983985,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.51,36.56,4.693188428878784
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",163,2026188684000,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,1.0,1,0,55.6,65.0,5.2353668212890625
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",32,2026193784022,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.51,36.06,5.117217302322388
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",33,2026192083972,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.1825,16.315,5.32389497756958
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",200,2026186983986,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.03,35.98,5.474027872085571
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",33,2026193784023,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.71,36.42,5.110716342926025
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",164,2026188684001,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.725,64.35,5.96000075340271
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",201,2026186983987,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.27,34.08,4.773852109909058
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",34,2026192083973,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.5,1,1,4.95,16.15,5.704310417175293
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",34,2026193784024,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.05,35.72,5.132810115814209
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",165,2026188684002,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,1.0,1,0,56.475,63.25,5.83394193649292
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",35,2026192083974,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,1.0,1,0,4.875,15.84,4.948913335800171
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",202,2026186983988,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,25.88,35.82,5.651486158370972
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",35,2026193784025,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.46,35.66,5.354283571243286
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",166,2026188684003,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.525,67.05,5.565965414047241
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",36,2026192083975,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.135,15.895,5.658063888549805
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",203,2026186983989,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.63,36.16,5.737736940383911
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",36,2026193784026,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.28,35.12,4.842746734619141
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",167,2026188684004,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,56.275,64.2,5.114092826843262
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",37,2026193784027,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,12.78,37.74,3.7829878330230713
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",204,2026186983990,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.42,37.1,5.421403169631958
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",37,2026192083976,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.8975,15.8,6.082733392715454
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",38,2026193784028,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.05,36.18,4.2526535987854
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",168,2026188684005,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.1,63.6,6.5059356689453125
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",205,2026186983991,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.9,36.92,5.7095420360565186
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",38,2026192083977,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.98,15.7,5.416726112365723
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",39,2026193784029,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.97,33.7,5.129448890686035
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",169,2026188684006,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.65,63.1,5.082757472991943
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",39,2026192083978,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.5,1,1,5.0675,15.91,5.32422399520874
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",206,2026186983992,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.55,36.46,5.668090581893921
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",40,2026193784030,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.35,35.68,4.206631660461426
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",170,2026188684007,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.425,65.25,5.914342641830444
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",207,2026186983993,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.15,34.68,4.56527853012085
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",40,2026192083979,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.065,15.9,6.49961256980896
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",41,2026193784031,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,12.59,35.6,4.419163703918457
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",171,2026188684008,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,53.825,64.0,5.41379976272583
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",208,2026186983994,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.79,35.52,5.556853771209717
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",42,2026193784032,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.2,35.4,5.546343088150024
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",41,2026192083980,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,1.0,1,0,5.21,16.0,6.633434772491455
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",172,2026188684009,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.375,62.4,4.443953514099121
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",209,2026186983995,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.21,36.02,5.666340589523315
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",43,2026193784033,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.29,36.16,5.0046398639678955
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",173,2026188684010,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.875,63.95,4.865919828414917
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",42,2026192083981,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.99,15.82,5.876795291900635
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",210,2026186983996,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.95,36.52,5.569941282272339
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",44,2026193784034,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.31,35.9,5.088494062423706
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",43,2026192083982,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.985,15.69,5.139983654022217
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",174,2026188684011,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,56.8,64.35,6.292080402374268
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",211,2026186983997,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.05,36.02,5.043527126312256
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",45,2026193784035,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,12.54,36.04,5.453916788101196
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",44,2026192083983,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.15,15.91,5.0961754322052
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",175,2026188684012,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,56.95,64.65,5.162598371505737
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",212,2026186983998,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.2,36.98,5.051105976104736
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",46,2026193784036,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.13,36.08,4.482323884963989
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",176,2026188684013,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,1.0,1,0,54.05,62.65,5.173074007034302
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",45,2026192083984,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.9725,16.055,6.478355884552002
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",213,2026186983999,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.36,36.84,4.357432126998901
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",47,2026193784037,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.06,36.44,4.878926515579224
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",177,2026188684014,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.0,64.9,4.7709503173828125
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",214,2026186984000,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.66,36.74,4.713555812835693
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",48,2026193784038,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,12.97,36.58,5.526438236236572
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",46,2026192083985,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,1.0,1,0,5.035,16.125,7.204013347625732
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",178,2026188684015,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.725,63.45,4.508735418319702
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",215,2026186984001,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.62,36.02,4.422088623046875
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",47,2026192083986,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.045,15.975,5.192262411117554
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",49,2026193784039,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.52,37.3,5.775453329086304
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",179,2026188684016,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.425,65.05,4.858585834503174
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",216,2026186984002,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.97,35.84,4.83517599105835
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",48,2026192083987,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.2325,15.99,4.243765115737915
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",50,2026193784040,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.34,36.1,5.303657531738281
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",217,2026186984003,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,25.87,36.66,4.828869819641113
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",180,2026188684017,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.475,63.8,5.475512504577637
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",49,2026192083988,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,1.0,1,0,5.04,15.61,5.196286916732788
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",51,2026193784041,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.61,35.88,5.25140643119812
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",181,2026188684018,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,56.45,62.35,5.104971885681152
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",218,2026186984004,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,25.83,35.22,5.333739519119263
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",50,2026192083989,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.2075,15.92,4.579156160354614
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",52,2026193784042,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.79,36.54,4.9830827713012695
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",219,2026186984005,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.37,35.76,4.9516918659210205
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",182,2026188684019,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.25,61.95,5.486698865890503
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",51,2026192083990,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,1.0,1,0,5.0275,15.965,6.03555154800415
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",53,2026193784043,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.33,36.12,4.164694309234619
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",220,2026186984006,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,25.62,36.52,5.178861141204834
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",183,2026188684020,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,56.35,61.75,5.279821395874023
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",54,2026193784044,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.12,34.98,4.687041759490967
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",52,2026192083991,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.0325,15.705,5.972442626953125
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",221,2026186984007,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.24,36.08,4.75853157043457
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",184,2026188684021,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,53.75,63.6,5.54709005355835
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",55,2026193784045,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.08,35.76,4.8718602657318115
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",53,2026192083992,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.065,16.16,5.03930926322937
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",222,2026186984008,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.67,36.24,5.551515102386475
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",56,2026193784046,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.24,37.24,4.256914854049683
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",185,2026188684022,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,56.0,62.9,5.551579475402832
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",54,2026192083993,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.14,15.865,5.568026304244995
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",223,2026186984009,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,1.0,1,0,26.31,35.56,5.993843078613281
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",186,2026188684023,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,56.45,63.5,4.810693979263306
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",57,2026193784047,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.32,35.56,5.294038772583008
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",55,2026192083994,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.155,16.185,5.830347299575806
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",224,2026186984010,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.61,36.8,4.385268926620483
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",58,2026193784048,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,12.84,35.02,4.652348041534424
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",187,2026188684024,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.5,65.7,5.826818227767944
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",225,2026186984011,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.35,35.26,4.856744766235352
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",56,2026192083995,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.15,15.925,5.911863803863525
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",59,2026193784049,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.38,35.94,4.658106088638306
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",188,2026188684025,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.375,63.55,5.522263765335083
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",226,2026186984012,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,1.0,1,0,26.2,36.8,5.568357229232788
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",57,2026192083996,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.0925,15.815,5.352177858352661
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",60,2026193784050,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.09,35.68,5.425370693206787
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",189,2026188684026,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,53.5,63.05,6.017996788024902
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",227,2026186984013,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.5,36.48,5.022669792175293
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",61,2026193784051,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.56,36.68,4.548644781112671
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",58,2026192083997,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,1.0,1,0,5.22,16.055,6.30056095123291
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",190,2026188684027,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,56.225,65.9,5.307189464569092
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",62,2026193784052,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,12.88,35.58,4.145013809204102
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",228,2026186984014,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.96,36.2,5.561668634414673
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",59,2026192083998,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.0975,15.91,6.628707647323608
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",191,2026188684028,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.625,62.3,5.288099050521851
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",63,2026193784053,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,12.52,35.48,4.620980262756348
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",229,2026186984015,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,25.58,35.3,4.214815616607666
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",192,2026188684029,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,53.2,64.85,4.758569717407227
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",64,2026193784054,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.18,36.54,5.454540252685547
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",60,2026192083999,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.1475,15.805,6.053745269775391
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",230,2026186984016,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.09,35.68,5.342730283737183
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",193,2026188684030,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,53.275,63.85,5.334810733795166
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",65,2026193784055,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.12,36.22,4.552762508392334
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",61,2026192084000,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.9875,16.015,5.827031850814819
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",231,2026186984017,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,1.0,1,0,26.4,35.76,5.4339940547943115
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",194,2026188684031,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.375,62.95,4.464138031005859
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",66,2026193784056,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.19,34.92,5.006486892700195
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",232,2026186984018,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,1.0,1,0,27.02,35.74,4.621004819869995
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",62,2026192084001,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.0325,15.61,5.606319427490234
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",195,2026188684032,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,56.875,63.9,5.264805316925049
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",67,2026193784057,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.02,36.02,4.5179853439331055
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",233,2026186984019,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.01,36.46,4.86330509185791
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",63,2026192084002,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.02,16.34,5.4720635414123535
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",196,2026188684033,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.825,65.15,4.671399116516113
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",68,2026193784058,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.66,36.12,5.361748456954956
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",234,2026186984020,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.2,36.42,4.902389764785767
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",197,2026188684034,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,53.45,63.05,4.259982585906982
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",64,2026192084003,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.105,15.865,5.991464614868164
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",69,2026193784059,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,1.0,1,0,12.93,36.08,5.055225610733032
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",235,2026186984021,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,25.81,35.98,5.619009256362915
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",198,2026188684035,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,57.25,64.0,5.085480451583862
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",65,2026192084004,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.12,15.8,5.644487142562866
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",70,2026193784060,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.11,35.62,4.786368370056152
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",236,2026186984022,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.78,36.3,4.798859357833862
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",199,2026188684036,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,56.175,64.35,5.2602338790893555
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",71,2026193784061,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,1.0,1,0,13.37,36.92,4.145165920257568
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",66,2026192084005,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,1.0,1,0,4.945,15.91,6.213957071304321
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",237,2026186984023,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.8,36.16,5.087170362472534
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",200,2026188684037,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,56.7,63.15,4.709978818893433
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",72,2026193784062,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,12.6,35.28,5.067627191543579
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",67,2026192084006,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.5,1,1,4.9175,16.09,5.786723852157593
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",238,2026186984024,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,1.0,1,0,26.24,34.7,4.815042495727539
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",201,2026188684038,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.6,66.05,5.044779062271118
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",73,2026193784063,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,12.59,36.04,4.8765716552734375
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",239,2026186984025,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.19,36.16,4.148118495941162
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",68,2026192084007,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.055,15.8,5.177998065948486
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",74,2026193784064,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.38,35.94,4.175202131271362
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",202,2026188684039,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.625,64.95,6.256411790847778
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",69,2026192084008,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.0925,16.025,4.820506811141968
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0002|pi0.0002|k1,1,1",240,2026186984026,A,noisy,7,0.0002,0.0002,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.56,35.84,6.269914627075195
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",75,2026193784065,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.07,36.5,4.8168044090271
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",203,2026188684040,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,56.25,66.75,4.608938217163086
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",70,2026192084009,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.95,16.085,5.608618974685669
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",76,2026193784066,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.29,36.24,4.2918336391448975
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",204,2026188684041,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.65,62.9,5.6991119384765625
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",77,2026193784067,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.35,35.84,4.2018141746521
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",71,2026192084010,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.045,15.91,5.042670011520386
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",0,2026195484041,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.95,64.9,5.558650016784668
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",205,2026188684042,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,56.45,63.85,5.147239685058594
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",78,2026193784068,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.4,34.22,4.12835168838501
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",1,2026195484042,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,29.0,64.55,3.8240387439727783
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",72,2026192084011,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,1.0,1,0,4.9,16.16,6.810531377792358
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",79,2026193784069,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.73,35.52,3.8678672313690186
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",206,2026188684043,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.375,64.1,6.044558048248291
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",2,2026195484043,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.75,64.65,4.22043776512146
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",73,2026192084012,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,1.0,1,0,5.11,15.78,5.676697492599487
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",80,2026193784070,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.29,35.8,5.209622383117676
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",207,2026188684044,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,1.0,1,0,58.65,65.05,5.392639875411987
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",3,2026195484044,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.1,64.15,4.373784303665161
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",81,2026193784071,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.51,35.56,4.744166135787964
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",74,2026192084013,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.0375,15.86,5.555450201034546
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",4,2026195484045,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.725,63.7,4.09373664855957
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",208,2026188684045,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,53.675,67.25,5.340275049209595
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",82,2026193784072,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.88,35.78,4.752021789550781
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",5,2026195484046,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.825,64.05,4.530243635177612
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",75,2026192084014,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.185,15.625,5.927075147628784
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",209,2026188684046,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.05,65.8,5.540649175643921
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",83,2026193784073,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.31,36.32,4.247438192367554
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",6,2026195484047,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.55,63.75,5.155555248260498
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",76,2026192084015,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.935,15.825,5.319188356399536
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",210,2026188684047,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,53.825,63.6,6.015873193740845
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",84,2026193784074,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.32,36.78,5.529271602630615
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",7,2026195484048,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.45,65.9,4.154698371887207
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",77,2026192084016,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.8525,16.04,5.483324289321899
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",85,2026193784075,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.8,35.78,3.7932894229888916
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",211,2026188684048,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,51.125,63.1,5.576600551605225
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",8,2026195484049,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.75,63.6,5.349011421203613
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",78,2026192084017,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,1.0,1,0,5.065,15.735,5.671157360076904
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",86,2026193784076,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.41,35.12,5.158084154129028
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",212,2026188684049,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.925,64.15,5.118666172027588
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",9,2026195484050,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.025,64.15,5.052340507507324
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",87,2026193784077,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.08,36.72,4.629908800125122
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",79,2026192084018,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.925,15.86,5.642895936965942
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",213,2026188684050,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.275,63.3,5.3558502197265625
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",10,2026195484051,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.8,65.0,5.133627653121948
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",88,2026193784078,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,12.76,36.18,5.15885066986084
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",80,2026192084019,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.1225,16.26,6.939875364303589
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",214,2026188684051,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,1.0,1,0,54.15,65.0,5.260200023651123
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",11,2026195484052,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.35,65.95,4.817552804946899
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",89,2026193784079,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,12.73,34.76,4.174394845962524
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",215,2026188684052,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.6,66.5,5.312592029571533
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",81,2026192084020,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.975,15.885,5.548368453979492
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",90,2026193784080,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,12.7,35.1,4.251205682754517
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",12,2026195484053,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.825,64.0,6.0549163818359375
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",216,2026188684053,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.3,65.65,4.33155369758606
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",91,2026193784081,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.02,36.42,4.966015100479126
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",82,2026192084021,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.0875,15.82,5.939391136169434
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",13,2026195484054,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.0,63.55,5.184548854827881
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",217,2026188684054,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.675,62.6,4.840334892272949
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",83,2026192084022,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.045,15.87,4.72946834564209
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",92,2026193784082,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.2,35.94,5.774656295776367
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",14,2026195484055,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.1,63.8,5.536526679992676
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",218,2026188684055,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.4,63.85,4.875193357467651
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",84,2026192084023,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,1.0,1,0,4.895,15.91,5.313754081726074
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",15,2026195484056,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.575,62.8,4.957777500152588
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",93,2026193784083,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,1.0,1,0,13.92,35.6,5.9772937297821045
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",219,2026188684056,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,56.575,65.9,4.506153106689453
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",16,2026195484057,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.0,63.4,4.470607042312622
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",85,2026192084024,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.98,16.17,5.494245290756226
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",94,2026193784084,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,1.0,1,0,13.39,36.14,5.826667547225952
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",220,2026188684057,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.925,63.9,5.308961629867554
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",17,2026195484058,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.5,1,1,26.625,62.25,5.021059036254883
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",86,2026192084025,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,1.0,1,0,4.9125,16.025,5.438018798828125
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",95,2026193784085,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.87,35.62,5.161357164382935
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",221,2026188684058,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.925,64.9,6.066205978393555
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",18,2026195484059,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.2,64.85,4.929027795791626
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",87,2026192084026,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.0,16.015,5.097761631011963
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",96,2026193784086,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.22,36.3,5.119338512420654
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",222,2026188684059,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,55.675,62.7,5.907238245010376
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",19,2026195484060,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.3,62.1,5.038572788238525
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",88,2026192084027,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.065,15.82,4.688495635986328
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",97,2026193784087,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.43,36.3,5.401484251022339
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",223,2026188684060,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,57.175,62.4,4.785930395126343
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",89,2026192084028,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.09,16.455,4.983857154846191
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",20,2026195484061,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.275,63.9,5.592329263687134
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",98,2026193784088,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.75,37.14,4.395688056945801
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",224,2026188684061,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,53.95,65.75,6.064196825027466
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",21,2026195484062,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.3,65.8,5.039863109588623
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",90,2026192084029,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.9975,16.13,5.2209792137146
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",99,2026193784089,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.5,35.76,5.6478025913238525
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",22,2026195484063,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.625,63.1,4.184101104736328
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",225,2026188684062,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,52.75,63.3,5.357374668121338
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",91,2026192084030,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,1.0,1,0,5.1525,15.775,5.6150383949279785
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",100,2026193784090,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.64,36.48,3.936943531036377
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",23,2026195484064,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,25.95,62.05,3.951993465423584
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",226,2026188684063,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,0.0,0,0,54.95,63.1,5.04076075553894
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",101,2026193784091,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.47,36.88,5.600439071655273
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",92,2026192084031,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,1.0,1,0,5.035,15.955,6.307000160217285
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",24,2026195484065,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.7,64.8,4.449725866317749
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0002|pi0.0002|k1,1,1",227,2026188684064,A,noisy,9,0.0002,0.0002,1.0,1.0,1.0,9,9,9,40,1.0,1,0,56.125,65.15,5.650309324264526
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",102,2026193784092,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,12.96,37.26,4.828031778335571
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.001|pi0|k1,1,1",0,2026197184092,B,noisy,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,5.5,6,1,2.7475,4.035,1.1734397411346436
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",93,2026192084032,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.1075,16.415,5.725979804992676
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",25,2026195484066,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.525,63.05,5.615630388259888
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.001|pi0|k1,1,1",1,2026197184093,B,noisy,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,7.5,8,1,3.0,4.005,1.0737254619598389
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.001|pi0|k1,1,1",2,2026197184094,B,noisy,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,3.5,4,1,2.62,4.13,0.9728631973266602
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.001|pi0|k1,1,1",3,2026197184095,B,noisy,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,2.5,3,1,2.8325,4.015,1.4636528491973877
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",103,2026193784093,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.38,36.34,4.569659948348999
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.001|pi0|k1,1,1",4,2026197184096,B,noisy,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,7.0,8,2,2.765,4.13,1.5617351531982422
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",94,2026192084033,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.055,16.3,4.987981081008911
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.001|pi0|k1,1,1",5,2026197184097,B,noisy,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,6.0,7,2,2.8225,3.905,1.3645896911621094
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",26,2026195484067,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.9,63.85,5.861526012420654
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.001|pi0|k1,1,1",6,2026197184098,B,noisy,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,5.0,5,0,2.765,4.115,1.4116146564483643
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",104,2026193784094,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.83,36.02,4.240000486373901
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.001|pi0|k1,1,1",7,2026197184099,B,noisy,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,5.0,5,0,2.6825,4.075,1.5665953159332275
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.001|pi0|k1,1,1",8,2026197184100,B,noisy,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,6.0,7,2,2.7225,4.065,1.2074987888336182
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",95,2026192084034,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,1.0,1,0,5.2825,15.99,5.481159687042236
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",27,2026195484068,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,29.275,63.25,4.7340850830078125
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.001|pi0|k1,1,1",9,2026197184101,B,noisy,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,10.5,12,3,2.72,4.055,1.5322139263153076
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",105,2026193784095,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,12.95,36.04,4.382319450378418
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.001|pi0|k1,1,1",10,2026197184102,B,noisy,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,4.5,5,1,2.7725,4.21,1.1670300960540771
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.001|pi0|k1,1,1",11,2026197184103,B,noisy,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,3.5,5,3,2.785,3.88,1.650825023651123
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",28,2026195484069,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.825,66.05,4.502368688583374
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.001|pi0|k1,1,1",12,2026197184104,B,noisy,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,6.0,6,0,2.885,4.005,1.1917576789855957
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",96,2026192084035,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.08,16.035,5.719781875610352
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",106,2026193784096,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.41,36.52,4.697225332260132
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.001|pi0|k1,1,1",13,2026197184105,B,noisy,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,6.0,7,2,2.9425,3.87,1.3769469261169434
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.001|pi0|k1,1,1",14,2026197184106,B,noisy,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,5.0,6,2,2.5925,4.005,1.718702793121338
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.001|pi0|k1,1,1",15,2026197184107,B,noisy,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,4.0,4,0,2.865,3.875,1.1018989086151123
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",29,2026195484070,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.425,62.0,4.941194534301758
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.001|pi0|k1,1,1",16,2026197184108,B,noisy,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,6.5,7,1,2.6925,4.25,1.1935579776763916
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",107,2026193784097,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,12.45,36.18,4.411243677139282
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.001|pi0|k1,1,1",17,2026197184109,B,noisy,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,4.5,5,1,2.7625,3.955,1.24031662940979
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",97,2026192084036,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,1.0,1,0,5.21,16.17,6.6666905879974365
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.001|pi0|k1,1,1",18,2026197184110,B,noisy,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,1.5,2,1,2.74,3.93,1.210968017578125
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.001|pi0|k1,1,1",19,2026197184111,B,noisy,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,2.76,3.865,1.4636716842651367
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",30,2026195484071,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.375,63.7,5.456434965133667
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",108,2026193784098,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.09,35.58,4.949535608291626
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.001|pi0|k1,1,1",20,2026197184112,B,noisy,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,7.0,7,0,2.9825,3.925,1.3962233066558838
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.001|pi0|k1,1,1",21,2026197184113,B,noisy,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,7.5,8,1,2.9125,3.87,1.2459650039672852
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",98,2026192084037,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.93,16.205,5.750056266784668
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.001|pi0|k1,1,1",22,2026197184114,B,noisy,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,4.5,5,1,2.935,4.065,1.2117886543273926
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.001|pi0|k1,1,1",23,2026197184115,B,noisy,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,2.5,3,1,2.8525,4.095,1.2206401824951172
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",109,2026193784099,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,12.81,35.26,4.493918418884277
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.001|pi0|k1,1,1",24,2026197184116,B,noisy,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,2.5,3,1,2.855,3.895,1.5093660354614258
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",31,2026195484072,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.875,64.6,5.682687997817993
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.001|pi0|k1,1,1",25,2026197184117,B,noisy,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,2.5,3,1,2.84,4.0,1.2330515384674072
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",99,2026192084038,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.075,16.045,5.561096668243408
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.001|pi0|k1,1,1",26,2026197184118,B,noisy,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,9.0,9,0,2.7675,3.94,1.7672150135040283
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",110,2026193784100,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.83,36.46,4.988342523574829
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.001|pi0|k1,1,1",27,2026197184119,B,noisy,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,4.0,4,0,2.86,4.005,1.1478855609893799
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",32,2026195484073,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.65,64.15,4.6435253620147705
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.001|pi0|k1,1,1",28,2026197184120,B,noisy,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,5.5,6,1,2.7625,4.02,1.1218748092651367
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.001|pi0|k1,1,1",29,2026197184121,B,noisy,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,6.0,7,2,2.755,3.845,1.1170032024383545
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.001|pi0|k1,1,1",30,2026197184122,B,noisy,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,5.0,6,2,2.7175,4.105,0.9836893081665039
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",100,2026192084039,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,1.0,1,0,4.995,15.89,5.73075532913208
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",111,2026193784101,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.08,35.32,4.648927927017212
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.001|pi0|k1,1,1",31,2026197184123,B,noisy,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,5.0,5,0,2.8,4.08,1.612992286682129
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",33,2026195484074,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.8,63.15,5.128412961959839
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.001|pi0|k1,1,1",32,2026197184124,B,noisy,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,4.0,5,2,2.6975,3.91,1.2940075397491455
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.001|pi0|k1,1,1",33,2026197184125,B,noisy,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,10.0,11,2,2.7975,3.825,1.72996187210083
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",112,2026193784102,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.09,35.96,4.5736260414123535
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.001|pi0|k1,1,1",34,2026197184126,B,noisy,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,6.0,7,2,2.7075,4.08,1.266880989074707
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",101,2026192084040,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.0975,16.035,5.812774181365967
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",34,2026195484075,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.575,62.45,4.838258981704712
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",113,2026193784103,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.95,36.92,5.355122089385986
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",0,2026198884143,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.395,16.135,4.796993970870972
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",35,2026195484076,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.45,63.1,5.347147226333618
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",102,2026192084041,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.975,15.75,5.951885461807251
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",114,2026193784104,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.15,36.14,4.775311231613159
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",1,2026198884144,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,3.0,3,0,12.2075,15.725,5.806288719177246
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",36,2026195484077,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.8,63.45,5.1236913204193115
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",103,2026192084042,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.31,16.095,6.023036003112793
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",115,2026193784105,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.52,35.7,4.8708531856536865
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",2,2026198884145,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,4.5,5,1,12.11,15.895,5.350488185882568
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",37,2026195484078,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.975,63.35,4.9877238273620605
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",104,2026192084043,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,2.0,2,0,4.9975,16.335,5.275311708450317
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",116,2026193784106,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.61,35.9,5.232998609542847
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",38,2026195484079,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.25,63.8,4.738445043563843
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",3,2026198884146,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,7.5,8,1,12.13,15.89,6.196445465087891
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",105,2026192084044,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.28,16.175,5.17791748046875
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",117,2026193784107,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.46,36.12,4.10590386390686
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",39,2026195484080,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.65,63.95,5.441861391067505
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",106,2026192084045,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.105,16.61,5.482150316238403
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",118,2026193784108,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,12.99,36.38,4.531302452087402
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",4,2026198884147,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,3.5,4,1,12.465,16.145,6.46880316734314
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",40,2026195484081,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.425,63.6,4.658350944519043
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",119,2026193784109,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.4,35.86,4.226379632949829
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",5,2026198884148,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,3.5,4,1,12.23,16.05,5.33105731010437
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",107,2026192084046,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.9925,15.94,6.294402837753296
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",41,2026195484082,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.125,62.95,5.634289741516113
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",120,2026193784110,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.05,36.96,4.389668703079224
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",108,2026192084047,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.01,16.03,5.506845951080322
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",6,2026198884149,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,12.3325,15.895,6.183403491973877
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",42,2026195484083,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.875,66.95,4.006070137023926
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",121,2026193784111,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.42,35.82,5.073121547698975
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",109,2026192084048,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,1.0,1,0,5.09,15.845,5.390887498855591
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",7,2026198884150,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,12.0075,16.185,5.3320066928863525
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",43,2026195484084,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.7,64.7,5.263944864273071
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",122,2026193784112,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.58,35.62,5.373797416687012
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",44,2026195484085,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.7,63.35,4.537933111190796
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",8,2026198884151,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,2.5,3,1,12.18,16.03,5.223116636276245
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",110,2026192084049,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.135,15.96,6.04267430305481
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",123,2026193784113,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.02,35.58,5.697829008102417
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",9,2026198884152,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,5.0,5,0,12.065,15.8,5.097336053848267
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",45,2026195484086,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.95,63.3,5.4396655559539795
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",111,2026192084050,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.135,15.795,5.613066911697388
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",124,2026193784114,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.45,36.56,5.260976552963257
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",46,2026195484087,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,29.6,63.7,4.916375637054443
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",10,2026198884153,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,4.0,5,2,12.1,15.74,5.214547157287598
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",125,2026193784115,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.31,36.7,4.133531808853149
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",112,2026192084051,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.8575,16.33,6.004525899887085
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",11,2026198884154,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,4.0,4,0,12.025,16.115,4.790824890136719
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",126,2026193784116,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.84,35.6,4.78162956237793
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",47,2026195484088,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.35,64.75,6.3876261711120605
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",113,2026192084052,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.0075,15.845,5.99980092048645
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",12,2026198884155,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,12.0075,15.83,4.606678009033203
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",48,2026195484089,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.425,64.15,4.400723934173584
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",127,2026193784117,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.31,36.68,5.6982855796813965
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",114,2026192084053,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.0175,16.11,5.821410655975342
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",49,2026195484090,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,29.1,63.85,3.57389235496521
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",13,2026198884156,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,6.0,6,0,12.1975,15.755,5.5634095668792725
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",128,2026193784118,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.29,35.58,5.245914697647095
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",50,2026195484091,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.625,63.95,4.73453426361084
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",115,2026192084054,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.16,15.87,6.335689306259155
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",14,2026198884157,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,12.1925,15.885,5.4339377880096436
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",129,2026193784119,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.57,35.5,4.572428941726685
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",51,2026195484092,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.4,62.85,4.583955526351929
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",116,2026192084055,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,1.0,1,0,5.2475,16.08,5.072026968002319
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",15,2026198884158,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,3.5,4,1,11.915,16.115,5.434707880020142
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",130,2026193784120,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.08,35.9,5.057057619094849
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",52,2026195484093,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.2,65.7,4.664531469345093
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",117,2026192084056,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.9075,16.35,5.732624053955078
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",16,2026198884159,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,3.0,3,0,12.1375,15.885,5.696073055267334
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",131,2026193784121,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.45,35.74,5.0848610401153564
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",53,2026195484094,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.725,63.35,5.155717849731445
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",132,2026193784122,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.89,35.14,4.719185829162598
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",17,2026198884160,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,12.165,16.055,5.144467353820801
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",118,2026192084057,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.9725,16.085,5.822557687759399
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",54,2026195484095,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.375,63.5,4.682225227355957
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",119,2026192084058,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.1625,15.855,4.993446111679077
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",18,2026198884161,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,5.5,6,1,12.3225,15.91,5.543658018112183
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",133,2026193784123,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.42,35.68,5.742231845855713
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",55,2026195484096,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.425,63.3,6.27829384803772
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",134,2026193784124,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.71,35.14,4.237258672714233
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",120,2026192084059,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.835,16.055,5.406763315200806
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",19,2026198884162,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,3.5,4,1,12.16,16.04,5.601442337036133
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",56,2026195484097,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.375,62.8,5.9384729862213135
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",135,2026193784125,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.55,35.88,4.291005849838257
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",121,2026192084060,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.37,16.085,5.548710107803345
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",20,2026198884163,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,5.5,7,3,12.2075,15.98,5.934493780136108
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",57,2026195484098,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.65,63.0,4.5325706005096436
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",136,2026193784126,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.91,35.88,4.208822250366211
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",122,2026192084061,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.995,15.995,6.147245645523071
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",21,2026198884164,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,2.5,3,1,12.105,15.965,5.4730823040008545
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",137,2026193784127,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.21,36.24,4.756389379501343
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",58,2026195484099,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.175,61.95,5.074018239974976
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",59,2026195484100,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.675,64.1,4.414561748504639
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",138,2026193784128,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.18,36.02,4.771372556686401
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",123,2026192084062,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.09,15.995,5.983420372009277
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",22,2026198884165,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,2.0,3,2,11.9675,16.07,5.798550367355347
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",60,2026195484101,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.025,65.4,5.080702066421509
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",23,2026198884166,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,12.13,16.165,4.916630029678345
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",139,2026193784129,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.77,36.92,5.492020130157471
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",124,2026192084063,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.0725,15.895,5.834826231002808
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",61,2026195484102,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.5,62.7,5.311449289321899
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",140,2026193784130,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.67,36.1,4.729800701141357
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",24,2026198884167,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,3.5,4,1,12.1075,15.925,5.457811594009399
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",125,2026192084064,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.1375,15.915,5.413785219192505
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",141,2026193784131,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,12.96,35.68,4.872790575027466
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",62,2026195484103,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.125,62.85,5.5432281494140625
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",126,2026192084065,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.055,16.23,5.5010857582092285
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",25,2026198884168,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,2.5,3,1,12.05,15.89,6.4851579666137695
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",63,2026195484104,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.7,61.75,4.200492858886719
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",142,2026193784132,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.18,35.5,5.410869836807251
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",127,2026192084066,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.0875,16.075,5.187595844268799
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",26,2026198884169,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,12.25,16.175,5.016655445098877
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",64,2026195484105,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.9,64.7,4.91235089302063
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",143,2026193784133,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.69,37.18,5.678654432296753
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",128,2026192084067,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.045,15.83,4.998408317565918
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",27,2026198884170,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,5.5,6,1,12.435,16.21,6.5426928997039795
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",144,2026193784134,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.04,34.96,3.6015563011169434
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",65,2026195484106,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.525,64.2,6.506981611251831
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",129,2026192084068,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,1.5,2,1,5.0425,16.18,5.530960559844971
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",145,2026193784135,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.31,35.68,4.377307176589966
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",28,2026198884171,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,8.5,9,1,11.975,16.27,5.506779193878174
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",66,2026195484107,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.375,63.75,5.6782684326171875
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",146,2026193784136,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.58,35.7,4.751864910125732
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",130,2026192084069,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,1.0,1,0,5.0375,16.2,6.3606719970703125
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",29,2026198884172,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,5.0,5,0,12.1025,15.785,5.223914861679077
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",67,2026195484108,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.625,62.9,4.282247543334961
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",30,2026198884173,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.0175,15.955,4.970516920089722
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",131,2026192084070,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.9125,16.425,5.583864212036133
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",147,2026193784137,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.03,36.14,5.873158693313599
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",68,2026195484109,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.375,62.65,4.862362623214722
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",132,2026192084071,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,1.0,1,0,4.9125,15.81,5.057583570480347
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",148,2026193784138,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.08,36.68,5.320072412490845
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",69,2026195484110,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.95,63.35,3.9788622856140137
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",31,2026198884174,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,5.5,6,1,12.1325,16.255,6.062835693359375
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",70,2026195484111,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.85,64.0,4.40250563621521
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",149,2026193784139,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,12.7,35.76,4.516035079956055
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",133,2026192084072,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.99,15.895,5.69895601272583
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",32,2026198884175,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,3.5,4,1,12.2525,15.93,5.221593141555786
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",71,2026195484112,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,1.0,1,0,27.125,63.95,3.8048884868621826
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",150,2026193784140,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.52,37.22,5.505290746688843
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",33,2026198884176,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,2.0,2,0,12.2675,15.98,5.134802341461182
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",134,2026192084073,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.0975,16.035,5.84700608253479
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",72,2026195484113,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.025,64.3,4.950802803039551
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",151,2026193784141,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.5,1,1,13.04,35.58,4.834600925445557
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",135,2026192084074,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.1,16.0,5.140799283981323
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",34,2026198884177,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.5125,16.025,5.843519449234009
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",73,2026195484114,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.15,63.25,5.337946176528931
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",152,2026193784142,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.05,36.3,5.05354118347168
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",136,2026192084075,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.0275,16.11,4.957639455795288
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",35,2026198884178,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,4.5,6,3,12.2525,15.91,5.7092413902282715
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",74,2026195484115,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.3,62.95,4.0825514793396
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",153,2026193784143,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.67,35.78,5.2458271980285645
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",75,2026195484116,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.05,63.7,4.677747488021851
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",137,2026192084076,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.9975,16.145,6.180587291717529
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",36,2026198884179,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.5,3,3,12.225,15.78,6.092089653015137
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",154,2026193784144,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.4,34.64,4.487806797027588
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",76,2026195484117,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.6,64.85,5.199902296066284
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",138,2026192084077,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.895,15.675,5.237022876739502
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",37,2026198884180,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,2.0,2,0,12.2675,15.855,5.243113994598389
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",155,2026193784145,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.46,35.64,4.436673164367676
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",77,2026195484118,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,25.925,63.65,5.199662923812866
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",139,2026192084078,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.76,15.72,6.0226662158966064
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",156,2026193784146,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.74,36.1,4.850939989089966
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",38,2026198884181,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,2.0,2,0,12.065,16.345,5.365130186080933
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",78,2026195484119,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.85,63.75,5.939530849456787
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",39,2026198884182,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,2.5,3,1,12.195,15.865,4.7794189453125
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",157,2026193784147,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.3,36.22,5.397413015365601
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",140,2026192084079,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,1.0,1,0,5.0025,16.19,5.831350803375244
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",79,2026195484120,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.775,64.75,4.177251100540161
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",158,2026193784148,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.97,34.74,4.6982879638671875
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",141,2026192084080,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.0325,15.65,5.693714141845703
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",40,2026198884183,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,2.0,2,0,12.0325,16.015,6.477476358413696
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",80,2026195484121,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.05,66.5,5.0140955448150635
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",159,2026193784149,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.36,35.5,4.768765687942505
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",41,2026198884184,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,12.3375,15.955,5.55456018447876
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",142,2026192084081,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.2525,15.59,6.383786201477051
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",81,2026195484122,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.65,65.05,4.977938175201416
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",160,2026193784150,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.19,35.4,4.838658332824707
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",143,2026192084082,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.24,15.845,4.610438346862793
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",42,2026198884185,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,2.5,3,1,12.245,15.785,5.915984869003296
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",82,2026195484123,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.45,61.95,5.589550256729126
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",161,2026193784151,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,12.93,35.36,4.871660470962524
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",43,2026198884186,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,11.9025,16.11,4.9112629890441895
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",144,2026192084083,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,1.0,1,0,5.0825,16.055,5.971917152404785
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",162,2026193784152,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.53,35.76,4.193556308746338
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",83,2026195484124,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.15,64.15,5.755338907241821
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",145,2026192084084,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,2.0,2,0,4.91,15.955,5.061887502670288
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",163,2026193784153,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.23,37.14,5.107757806777954
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",44,2026198884187,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,11.9675,15.98,6.2366783618927
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",84,2026195484125,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.05,64.15,5.569822549819946
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",146,2026192084085,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.9925,15.91,4.641857624053955
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",164,2026193784154,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.11,35.72,5.59084153175354
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",45,2026198884188,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,2,2,12.2725,16.275,5.819348096847534
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",85,2026195484126,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.625,66.15,4.705899000167847
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",147,2026192084086,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.805,16.075,6.25139594078064
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",165,2026193784155,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.69,35.4,5.1693830490112305
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",86,2026195484127,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.15,63.7,4.213840961456299
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",46,2026198884189,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,2.5,3,1,12.325,16.28,5.437173128128052
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",166,2026193784156,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.5,35.78,4.659251689910889
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",87,2026195484128,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.55,64.35,5.0656960010528564
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",148,2026192084087,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.5,1,1,5.16,16.06,6.027052879333496
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",47,2026198884190,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,3.5,4,1,12.5175,16.365,5.919030427932739
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",167,2026193784157,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.16,35.9,4.805798768997192
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",88,2026195484129,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.15,64.05,4.73227071762085
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",149,2026192084088,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,1.0,1,0,5.125,16.195,5.769852638244629
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",48,2026198884191,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,3.0,3,0,12.4375,16.07,5.757631063461304
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",89,2026195484130,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.05,65.95,4.784613847732544
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",168,2026193784158,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.06,36.6,5.6309826374053955
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",150,2026192084089,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.8625,16.05,5.402833938598633
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",49,2026198884192,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,12.2825,15.825,5.476412773132324
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",169,2026193784159,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.51,35.68,4.739273309707642
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",90,2026195484131,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.65,63.25,5.5796613693237305
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",151,2026192084090,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,2.0,2,0,5.0875,16.1,5.705432415008545
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",50,2026198884193,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,12.4575,16.125,5.11285400390625
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",170,2026193784160,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.52,36.0,4.89965295791626
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",91,2026195484132,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.325,64.6,5.445740222930908
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",152,2026192084091,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.1675,15.66,5.235942840576172
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",51,2026198884194,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,12.075,16.03,5.748628616333008
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",171,2026193784161,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,12.42,36.02,4.684325456619263
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",92,2026195484133,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.125,62.7,6.098955392837524
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",153,2026192084092,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.075,16.025,5.290395259857178
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",52,2026198884195,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,2.5,3,1,12.24,16.025,5.201918363571167
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",172,2026193784162,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.14,35.74,5.307343244552612
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",93,2026195484134,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.775,63.35,5.324174880981445
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",53,2026198884196,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,6.5,7,1,12.4,16.27,4.823019742965698
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",154,2026192084093,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.0875,16.035,5.888883590698242
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",173,2026193784163,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.03,36.14,5.289541482925415
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",94,2026195484135,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.9,64.75,4.656208276748657
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",54,2026198884197,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,2.5,3,1,12.3475,15.93,4.806270599365234
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",174,2026193784164,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,12.99,35.52,4.868014097213745
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",155,2026192084094,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.1975,16.11,6.801560163497925
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",95,2026195484136,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.875,65.5,4.227843523025513
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",55,2026198884198,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,3.5,4,1,12.18,15.865,5.82738471031189
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",175,2026193784165,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.26,36.0,4.637934446334839
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",96,2026195484137,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,25.525,64.6,4.93114161491394
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",156,2026192084095,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.11,15.81,5.518888473510742
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",176,2026193784166,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.62,35.64,4.362297773361206
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",56,2026198884199,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,2.5,3,1,12.1625,16.4,5.818020343780518
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",97,2026195484138,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.25,65.7,4.850241661071777
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",157,2026192084096,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,1.0,1,0,4.88,15.875,5.312044382095337
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",177,2026193784167,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,1.0,1,0,13.19,34.62,4.358664274215698
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",57,2026198884200,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,4.5,5,1,12.235,15.915,5.735923767089844
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",98,2026195484139,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.825,64.15,5.35785984992981
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",158,2026192084097,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.9525,15.535,5.573969841003418
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",178,2026193784168,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.11,35.84,4.955041408538818
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",99,2026195484140,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.35,63.3,4.36151909828186
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",58,2026198884201,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,12.155,16.14,5.881602048873901
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",179,2026193784169,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.16,35.76,4.279131889343262
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",159,2026192084098,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.8225,16.04,6.527655839920044
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",100,2026195484141,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.925,66.7,4.3409504890441895
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",180,2026193784170,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,12.68,35.54,4.273885726928711
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.001|pi0|k1,1,1",59,2026198884202,B,noisy,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,4.0,4,0,12.33,16.245,5.513751029968262
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",160,2026192084099,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.175,16.235,5.275635242462158
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",101,2026195484142,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,1.0,1,0,28.475,62.9,4.5473411083221436
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",181,2026193784171,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,1.0,1,0,12.95,36.52,4.382699489593506
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",102,2026195484143,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.05,65.0,4.582977294921875
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",0,2026200584194,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,33.09,35.44,4.403694152832031
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",161,2026192084100,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.095,15.64,5.811587333679199
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",182,2026193784172,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.24,36.48,4.3928916454315186
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",103,2026195484144,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.475,64.25,4.250796794891357
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",183,2026193784173,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.49,35.88,3.9934942722320557
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",1,2026200584195,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,33.56,35.78,5.378276109695435
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",162,2026192084101,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,2.0,2,0,5.1575,15.975,7.199817419052124
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",104,2026195484145,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.3,62.95,5.371532917022705
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",184,2026193784174,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.57,35.46,4.226658821105957
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",2,2026200584196,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.46,36.54,4.286931991577148
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",185,2026193784175,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.38,35.26,3.774015188217163
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",105,2026195484146,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,29.225,63.35,4.564149379730225
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",163,2026192084102,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.5,1,1,5.01,15.775,6.608065366744995
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",3,2026200584197,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.5,2,1,32.51,35.64,4.792010545730591
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",186,2026193784176,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.3,36.6,4.874492406845093
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",106,2026195484147,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.2,65.0,5.0776755809783936
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",4,2026200584198,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.5,2,1,32.36,35.56,4.414855003356934
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",164,2026192084103,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.1975,16.035,5.46808385848999
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",187,2026193784177,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.79,35.0,5.107246398925781
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",5,2026200584199,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,31.5,36.34,4.498990535736084
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",107,2026195484148,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,29.075,62.85,5.207052230834961
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",165,2026192084104,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,1.0,1,0,5.055,15.925,5.237908601760864
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",6,2026200584200,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,2.5,3,1,32.98,35.56,4.111268043518066
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",188,2026193784178,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,12.79,36.66,4.904989957809448
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",108,2026195484149,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.35,63.3,3.8982527256011963
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",166,2026192084105,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.7125,16.29,5.844707250595093
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",7,2026200584201,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,32.38,35.56,4.7120866775512695
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",109,2026195484150,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.7,64.75,4.56964635848999
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",189,2026193784179,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.22,35.84,5.323693037033081
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",167,2026192084106,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.0075,15.815,5.4293928146362305
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",110,2026195484151,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.825,65.1,4.219155311584473
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",8,2026200584202,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,32.87,35.82,4.768094778060913
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",190,2026193784180,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,12.9,35.3,4.692057371139526
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",111,2026195484152,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,29.1,64.8,4.54451847076416
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",168,2026192084107,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.1325,16.175,5.610413551330566
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",9,2026200584203,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,4.0,4,0,32.6,34.58,5.250177383422852
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",191,2026193784181,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.54,36.48,5.453178405761719
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",112,2026195484153,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.625,65.25,5.535112142562866
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",10,2026200584204,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,32.31,35.62,4.433700799942017
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",169,2026192084108,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.1275,16.135,6.1333091259002686
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",192,2026193784182,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.79,36.94,4.862326145172119
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",113,2026195484154,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.45,63.7,4.23221755027771
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",11,2026200584205,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.12,35.48,5.134668350219727
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",170,2026192084109,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.0775,15.94,5.576037645339966
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",193,2026193784183,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,12.96,35.36,5.582883358001709
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",114,2026195484155,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,1.0,1,0,27.45,63.6,4.239930629730225
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",12,2026200584206,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,32.76,35.92,4.3805742263793945
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",194,2026193784184,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.95,35.08,4.570586204528809
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",115,2026195484156,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.75,66.45,4.452712297439575
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",171,2026192084110,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.145,15.86,6.51502799987793
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",13,2026200584207,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,2.0,2,0,31.85,36.46,5.18730616569519
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",195,2026193784185,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.08,36.12,4.165379285812378
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",116,2026195484157,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.425,62.25,4.6717445850372314
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",172,2026192084111,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.075,15.755,5.456705808639526
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",14,2026200584208,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,31.92,36.14,4.829726457595825
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",196,2026193784186,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.11,35.5,5.102433204650879
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",117,2026195484158,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.325,63.6,4.947145223617554
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",15,2026200584209,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,32.32,35.48,4.782881498336792
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",173,2026192084112,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.2525,16.21,6.557835102081299
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",197,2026193784187,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,12.92,35.8,4.418666839599609
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",118,2026195484159,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.75,64.95,5.337645053863525
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",16,2026200584210,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,2.0,2,0,32.98,35.58,5.003028869628906
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",174,2026192084113,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.1025,16.0,4.902716875076294
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",198,2026193784188,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.62,34.72,5.573550224304199
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",119,2026195484160,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.725,63.75,3.818161725997925
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",17,2026200584211,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,32.7,35.76,4.657101154327393
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",199,2026193784189,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.58,35.7,4.3683671951293945
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",175,2026192084114,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.06,15.585,6.2757768630981445
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",120,2026195484161,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.975,67.35,5.429280757904053
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",18,2026200584212,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.13,34.76,4.472441673278809
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",200,2026193784190,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.97,36.74,5.906913995742798
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",19,2026200584213,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,31.9,35.84,3.4867606163024902
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",121,2026195484162,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,1.0,1,0,28.475,64.75,4.94019341468811
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",176,2026192084115,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,1.0,1,0,4.9525,16.115,6.779048919677734
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",201,2026193784191,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.12,36.76,4.8951427936553955
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",122,2026195484163,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.225,65.4,4.710749864578247
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",20,2026200584214,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,32.55,36.66,4.913142442703247
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",177,2026192084116,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.5,1,1,5.025,16.025,4.917330265045166
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",123,2026195484164,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.25,62.8,4.303859233856201
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",202,2026193784192,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,1.0,1,0,12.94,37.04,5.224591255187988
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",21,2026200584215,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.47,36.3,4.872046947479248
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",178,2026192084117,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.095,15.95,6.024969100952148
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",124,2026195484165,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.65,63.7,4.505080223083496
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",22,2026200584216,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,32.5,35.24,4.397709608078003
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",203,2026193784193,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.57,36.54,5.358554124832153
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",125,2026195484166,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.05,62.1,3.351513624191284
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",179,2026192084118,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.9275,15.91,6.763140439987183
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",204,2026193784194,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,12.97,35.68,4.514000177383423
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",23,2026200584217,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,31.96,36.8,5.1964640617370605
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",126,2026195484167,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.65,64.55,4.105421543121338
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",24,2026200584218,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,33.23,36.86,3.895954132080078
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",205,2026193784195,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.42,35.3,5.4157702922821045
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",127,2026195484168,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.3,65.05,4.027856826782227
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",180,2026192084119,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.16,15.83,7.641772508621216
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",25,2026200584219,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,32.7,36.64,3.965787172317505
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",128,2026195484169,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.25,62.0,4.5552849769592285
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",206,2026193784196,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.27,35.54,5.728599548339844
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",26,2026200584220,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,33.68,35.72,3.8442089557647705
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",181,2026192084120,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.9575,15.82,5.864908218383789
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",129,2026195484170,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.175,61.85,4.394252061843872
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",207,2026193784197,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.17,35.24,4.457870244979858
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",27,2026200584221,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.37,35.92,4.047548294067383
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",130,2026195484171,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.275,64.75,4.638095378875732
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",208,2026193784198,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,12.96,36.26,4.044183254241943
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",182,2026192084121,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.0675,16.2,6.555537223815918
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",28,2026200584222,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,32.62,35.74,4.389417886734009
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",209,2026193784199,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.72,36.58,4.505229234695435
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",131,2026195484172,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.325,62.25,4.808660984039307
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",29,2026200584223,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.04,35.8,4.7435832023620605
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",183,2026192084122,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.02,16.03,6.350431203842163
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",132,2026195484173,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.475,61.2,4.2586989402771
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",30,2026200584224,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,32.7,34.8,3.820965051651001
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",210,2026193784200,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.13,36.16,5.552877187728882
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",184,2026192084123,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.0075,15.86,5.160610675811768
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",133,2026195484174,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.275,63.75,4.913088798522949
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",31,2026200584225,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,2.5,3,1,32.44,36.9,4.721496105194092
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",211,2026193784201,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.7,35.98,5.0354087352752686
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",185,2026192084124,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.1075,15.85,5.845380783081055
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",134,2026195484175,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.45,64.3,4.577255487442017
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",32,2026200584226,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,32.75,34.94,4.408761262893677
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",212,2026193784202,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.06,35.22,5.044511556625366
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",186,2026192084125,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.125,15.83,5.870164155960083
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",135,2026195484176,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.725,64.2,4.930863618850708
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",33,2026200584227,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,32.43,35.5,4.926833391189575
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",213,2026193784203,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.54,35.46,4.203275918960571
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",34,2026200584228,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,2.0,2,0,33.79,36.54,4.214757204055786
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",136,2026195484177,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.675,63.6,4.33608865737915
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",214,2026193784204,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.59,36.6,4.738132953643799
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",187,2026192084126,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.8875,16.14,7.18427848815918
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",35,2026200584229,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,32.5,35.52,4.30451512336731
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",137,2026195484178,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.375,62.85,5.353381395339966
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",215,2026193784205,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,12.58,35.74,4.659703969955444
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",188,2026192084127,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.105,15.68,5.287710428237915
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",216,2026193784206,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,1.0,1,0,13.62,35.66,3.866791009902954
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",36,2026200584230,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,32.98,36.34,5.822185754776001
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",138,2026195484179,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.825,65.2,5.155277967453003
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",189,2026192084128,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,1.5,2,1,5.125,16.095,5.6252195835113525
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",139,2026195484180,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,29.025,65.8,3.871842384338379
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",37,2026200584231,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,2.0,2,0,32.31,36.96,4.944855690002441
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",217,2026193784207,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.25,36.46,5.605761766433716
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",38,2026200584232,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,33.12,35.88,4.559178113937378
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",190,2026192084129,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,1.5,2,1,4.945,16.6,5.9843878746032715
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",140,2026195484181,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.85,63.55,5.352647542953491
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",218,2026193784208,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.55,37.18,4.7043116092681885
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",39,2026200584233,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,2.0,2,0,32.2,36.7,3.9185731410980225
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",141,2026195484182,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.325,62.85,4.556169509887695
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",219,2026193784209,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,1.0,1,0,13.36,34.98,4.6290223598480225
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",191,2026192084130,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.1575,16.13,7.203970909118652
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",40,2026200584234,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,32.41,35.34,4.882484674453735
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",142,2026195484183,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.25,63.4,4.364460229873657
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",220,2026193784210,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.09,35.74,4.306763410568237
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",143,2026195484184,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.375,64.2,4.217276096343994
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",192,2026192084131,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.245,15.865,6.075601100921631
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",221,2026193784211,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.44,36.4,4.3378424644470215
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",41,2026200584235,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,32.21,36.42,5.436929225921631
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",193,2026192084132,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,1.0,1,0,5.135,16.035,4.9987633228302
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",144,2026195484185,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.925,65.3,5.109718322753906
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",222,2026193784212,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.24,36.24,4.699222564697266
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",42,2026200584236,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,32.77,36.08,4.663517951965332
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",145,2026195484186,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.05,62.8,4.56218695640564
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",223,2026193784213,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.32,36.34,4.756148815155029
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",43,2026200584237,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,31.67,35.62,4.732002019882202
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",194,2026192084133,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.88,16.225,6.0260560512542725
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",146,2026195484187,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,1.0,1,0,27.575,63.5,4.865171670913696
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",224,2026193784214,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.65,35.22,5.058596849441528
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",44,2026200584238,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,32.86,35.24,4.948310375213623
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",195,2026192084134,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,1.0,1,0,5.13,15.77,5.0180275440216064
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",147,2026195484188,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.225,62.55,3.945446014404297
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",45,2026200584239,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.5,2,1,32.86,35.74,5.14289665222168
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",225,2026193784215,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.5,1,1,12.92,36.4,5.725264310836792
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",196,2026192084135,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.31,15.975,5.24687385559082
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",148,2026195484189,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.0,61.45,4.168456315994263
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",46,2026200584240,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,32.79,36.04,3.984961748123169
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",226,2026193784216,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.01,37.06,5.114257574081421
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",197,2026192084136,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.02,16.05,5.779822587966919
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",149,2026195484190,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,29.025,62.2,4.921562433242798
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",47,2026200584241,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,2.0,2,0,32.03,36.68,4.646598815917969
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",227,2026193784217,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.57,36.06,4.803702354431152
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",150,2026195484191,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.75,62.85,4.511523008346558
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",198,2026192084137,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.9775,16.235,5.889289140701294
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",48,2026200584242,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,32.7,36.08,5.120729923248291
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",228,2026193784218,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.27,35.72,4.573282718658447
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",151,2026195484192,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.875,60.75,4.5449302196502686
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",199,2026192084138,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.6575,15.92,5.154883861541748
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",49,2026200584243,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.09,36.36,5.248192071914673
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",229,2026193784219,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.97,35.92,4.298687934875488
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",152,2026195484193,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.625,64.95,4.035764694213867
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",200,2026192084139,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.93,15.93,5.788050174713135
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",230,2026193784220,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.29,36.2,4.397258043289185
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",50,2026200584244,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,32.8,35.0,6.028489589691162
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",153,2026195484194,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,1.0,1,0,27.375,64.1,4.924656391143799
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",231,2026193784221,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.24,36.72,4.808214902877808
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",201,2026192084140,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,1.0,1,0,5.215,16.03,5.032317638397217
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",154,2026195484195,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.925,64.55,4.613498210906982
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",51,2026200584245,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,32.8,35.18,5.197744607925415
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",232,2026193784222,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.28,35.2,5.277381658554077
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",202,2026192084141,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,1.0,1,0,5.065,15.815,5.454259872436523
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",155,2026195484196,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.525,63.0,4.298863410949707
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",52,2026200584246,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,2.0,2,0,33.84,35.94,4.046885967254639
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",233,2026193784223,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.34,36.58,4.696839809417725
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",156,2026195484197,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.35,67.1,4.388043403625488
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",203,2026192084142,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.5,1,1,4.8225,15.935,4.655987024307251
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",53,2026200584247,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,32.29,35.22,5.724294900894165
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",157,2026195484198,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.475,65.3,4.992964267730713
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",234,2026193784224,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.35,36.06,5.413304328918457
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",204,2026192084143,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,1.0,1,0,5.06,16.405,5.066128730773926
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",54,2026200584248,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,32.28,36.06,4.498192310333252
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",235,2026193784225,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.29,36.04,4.269009828567505
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",205,2026192084144,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.0975,16.105,4.731818675994873
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",158,2026195484199,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.1,64.7,6.1451733112335205
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",55,2026200584249,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,32.23,36.7,5.635148286819458
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",236,2026193784226,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,12.66,35.82,4.8351662158966064
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",206,2026192084145,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,2.0,2,0,4.9625,15.975,5.303553342819214
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",159,2026195484200,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.525,65.5,4.452242612838745
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",56,2026200584250,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,2.5,3,1,33.68,35.48,4.554562568664551
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",237,2026193784227,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,12.79,35.52,5.275240659713745
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",160,2026195484201,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.675,65.5,4.404850006103516
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",57,2026200584251,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,32.05,35.06,4.261425733566284
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",207,2026192084146,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.2175,15.7,6.179523468017578
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",161,2026195484202,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.55,62.95,4.511690616607666
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",238,2026193784228,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.48,35.42,4.965909481048584
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",58,2026200584252,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,32.77,36.18,5.500547409057617
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",208,2026192084147,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.9175,15.82,5.406895875930786
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",239,2026193784229,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,1.0,1,0,12.97,35.88,4.268218517303467
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",162,2026195484203,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.9,64.35,6.074140310287476
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",59,2026200584253,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,33.36,35.7,5.517909049987793
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",209,2026192084148,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.12,16.055,5.129583835601807
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",240,2026193784230,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.45,35.46,4.036161422729492
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",163,2026195484204,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.775,62.2,4.460220813751221
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",60,2026200584254,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,2.0,2,0,32.64,34.68,5.3195202350616455
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",210,2026192084149,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.88,16.18,5.3048388957977295
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",241,2026193784231,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.14,36.7,4.883694410324097
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",164,2026195484205,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.725,65.3,4.644346714019775
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",61,2026200584255,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,33.33,35.76,4.44820499420166
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",242,2026193784232,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,12.84,35.4,4.8142924308776855
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",211,2026192084150,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.805,16.08,5.5587944984436035
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",165,2026195484206,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.425,63.35,5.300520420074463
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",62,2026200584256,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.3,36.04,4.729832172393799
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",243,2026193784233,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.44,34.84,4.478916168212891
production-20261029T053925Z-29f79c,"A|noisy|d5|p0.0001|pi0.0001|k1,1,1",212,2026192084151,A,noisy,5,0.0001,0.0001,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.1,15.895,4.5056471824646
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",166,2026195484207,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,29.4,61.6,5.1999218463897705
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",63,2026200584257,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,32.38,36.38,5.036293029785156
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",244,2026193784234,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,12.92,36.56,4.253485441207886
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",167,2026195484208,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.775,63.3,4.708097457885742
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",64,2026200584258,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,33.81,37.12,4.7793662548065186
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",245,2026193784235,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.84,35.92,4.683087587356567
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",0,2026202284245,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,68.125,63.05,5.396847724914551
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",246,2026193784236,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.25,36.8,3.784226417541504
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",65,2026200584259,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,2.5,3,1,33.04,36.08,5.055830478668213
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",168,2026195484209,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.8,65.4,6.728811264038086
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",1,2026202284246,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,67.425,63.4,4.410704612731934
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",247,2026193784237,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,1.0,1,0,13.69,35.68,5.0746471881866455
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",66,2026200584260,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,32.44,36.88,4.6974852085113525
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",169,2026195484210,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.125,66.05,4.535056114196777
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",2,2026202284247,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,70.65,62.45,4.871093034744263
production-20261029T053925Z-29f79c,"A|noisy|d7|p0.0001|pi0.0001|k1,1,1",248,2026193784238,A,noisy,7,0.0001,0.0001,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.53,35.44,5.391012668609619
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",67,2026200584261,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,32.37,36.32,4.946049690246582
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",3,2026202284248,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,66.45,64.45,4.0168492794036865
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",170,2026195484211,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.95,63.45,4.922185182571411
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",0,2026203984296,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.5,2,1,1.4725,4.15,1.2919564247131348
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",1,2026203984297,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.32,4.09,0.9063377380371094
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",2,2026203984298,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,4.0,4,0,1.3475,3.85,1.0365984439849854
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",3,2026203984299,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.365,3.765,1.092118501663208
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",68,2026200584262,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.3,35.8,4.83504581451416
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",4,2026202284249,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,70.925,65.5,4.548758268356323
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",4,2026203984300,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,4.5,5,1,1.475,4.125,0.7832155227661133
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",171,2026195484212,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.7,64.25,4.8407371044158936
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",5,2026203984301,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,3.0,3,0,1.3075,4.13,0.9886057376861572
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",6,2026203984302,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,1.355,4.085,1.171766757965088
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",7,2026203984303,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.315,4.025,1.0766737461090088
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",69,2026200584263,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,2.0,2,0,32.37,35.04,4.489301681518555
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",5,2026202284250,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,2.0,2,0,68.8,63.7,4.779547691345215
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",8,2026203984304,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,2,2,1.3925,4.205,1.2810981273651123
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",172,2026195484213,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.6,64.4,4.262609481811523
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",9,2026203984305,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,3.0,3,0,1.32,3.985,1.2202084064483643
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",10,2026203984306,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,2.5,3,1,1.355,4.065,1.5880274772644043
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",173,2026195484214,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.25,62.25,3.7900729179382324
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",11,2026203984307,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,3.0,3,0,1.2725,3.88,1.3082334995269775
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",6,2026202284251,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.5,1,1,70.35,63.95,4.6702330112457275
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",70,2026200584264,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,2.0,2,0,33.48,36.3,5.452706336975098
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",12,2026203984308,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.475,4.05,1.382577896118164
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",13,2026203984309,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.3575,3.95,1.5867886543273926
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",14,2026203984310,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.34,3.8,1.0958123207092285
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",7,2026202284252,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,69.4,63.45,4.130892276763916
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",174,2026195484215,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.075,64.45,5.649954080581665
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",71,2026200584265,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,31.38,37.24,4.433175086975098
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",15,2026203984311,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.4475,4.07,1.5120160579681396
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",16,2026203984312,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,1.4675,4.06,1.5608885288238525
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",17,2026203984313,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.4025,3.97,1.3970913887023926
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",175,2026195484216,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.2,63.75,3.6708030700683594
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",8,2026202284253,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,67.775,63.95,4.488142013549805
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",18,2026203984314,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.3725,3.975,1.4000329971313477
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",72,2026200584266,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,32.86,36.72,5.039664268493652
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",19,2026203984315,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.3725,3.96,1.2777316570281982
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",20,2026203984316,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.36,4.02,1.1879265308380127
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",21,2026203984317,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.43,4.03,0.9150350093841553
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",9,2026202284254,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,67.45,64.75,4.637475967407227
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",176,2026195484217,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.275,64.8,4.933394193649292
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",22,2026203984318,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.4,3.93,1.356308937072754
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",73,2026200584267,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,33.03,35.46,4.873174667358398
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",23,2026203984319,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,4.0,4,0,1.335,3.89,1.3509447574615479
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",24,2026203984320,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.3725,3.825,1.2361187934875488
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",10,2026202284255,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,69.65,64.2,4.3910722732543945
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",177,2026195484218,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,1.0,1,0,27.2,63.9,4.527232646942139
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",25,2026203984321,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.5,2,1,1.365,4.105,1.2381260395050049
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",26,2026203984322,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,1.455,4.165,1.2540583610534668
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",74,2026200584268,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,33.7,36.66,5.285846710205078
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",27,2026203984323,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,1.315,3.945,0.8341050148010254
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",28,2026203984324,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.415,3.995,0.8464314937591553
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",11,2026202284256,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,69.325,65.8,4.294040679931641
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",29,2026203984325,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,1.355,3.885,1.1808245182037354
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",178,2026195484219,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.325,61.75,4.582081079483032
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",30,2026203984326,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.5,2,1,1.255,4.02,1.2359869480133057
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",75,2026200584269,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.23,37.22,4.2721545696258545
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",31,2026203984327,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.4875,4.045,1.3767247200012207
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",12,2026202284257,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,68.475,61.75,3.968996524810791
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",32,2026203984328,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.5,2,1,1.3525,3.96,1.493194341659546
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",33,2026203984329,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.365,3.835,1.421412467956543
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",76,2026200584270,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,2.0,2,0,31.58,35.68,3.6662065982818604
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",179,2026195484220,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.2,63.65,5.823245525360107
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",34,2026203984330,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.4025,4.17,0.8252949714660645
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",35,2026203984331,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.3475,4.185,1.2634141445159912
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",36,2026203984332,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.2925,3.84,0.9406116008758545
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",13,2026202284258,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,68.95,63.0,5.526832818984985
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",37,2026203984333,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.315,4.09,0.8644406795501709
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",77,2026200584271,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,2.5,3,1,33.48,36.68,4.1449713706970215
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",38,2026203984334,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,3.0,3,0,1.3875,4.075,1.458613395690918
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",180,2026195484221,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.7,63.8,4.781744480133057
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",39,2026203984335,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.3675,3.945,0.7732923030853271
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",40,2026203984336,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.37,4.08,1.1979000568389893
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",14,2026202284259,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,69.6,62.9,4.671195983886719
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",41,2026203984337,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.2825,3.935,1.1208739280700684
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",78,2026200584272,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,32.99,36.9,4.38247275352478
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",181,2026195484222,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.475,64.45,4.170886516571045
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",42,2026203984338,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,2.5,3,1,1.275,3.815,1.519662857055664
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",43,2026203984339,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.5,2,1,1.4125,4.07,1.7595415115356445
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",44,2026203984340,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.5475,3.975,0.993682861328125
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",79,2026200584273,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,33.46,35.9,4.414464950561523
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",45,2026203984341,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,2,2,1.415,4.19,0.7997126579284668
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",15,2026202284260,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,71.2,60.9,5.435947418212891
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",182,2026195484223,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.075,61.85,4.4798502922058105
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",46,2026203984342,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.3675,4.035,0.8009107112884521
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",47,2026203984343,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.5,2,1,1.3675,4.215,1.7516400814056396
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",48,2026203984344,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,2.0,3,2,1.3675,4.045,1.094846248626709
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",16,2026202284261,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,68.6,65.15,4.138598918914795
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",49,2026203984345,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,3.0,4,2,1.3775,4.14,1.010756492614746
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",80,2026200584274,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,32.99,35.48,5.1226208209991455
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",183,2026195484224,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.625,64.15,4.252360105514526
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",50,2026203984346,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.3875,3.95,0.8196275234222412
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",51,2026203984347,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.5,2,1,1.285,3.94,1.0095176696777344
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",52,2026203984348,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.5,2,1,1.3925,4.025,1.5516304969787598
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",17,2026202284262,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,68.375,63.2,4.12652587890625
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",184,2026195484225,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.0,64.05,4.444801568984985
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",81,2026200584275,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,33.3,36.26,4.860196828842163
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",53,2026203984349,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.3725,3.92,1.7441835403442383
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",54,2026203984350,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.3175,3.975,1.7612242698669434
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",18,2026202284263,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,66.85,65.75,3.8986921310424805
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",55,2026203984351,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.4175,3.93,1.112386703491211
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",56,2026203984352,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,4.0,5,2,1.35,3.935,1.177299976348877
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",185,2026195484226,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.525,64.6,4.786329984664917
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",82,2026200584276,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,32.28,36.82,4.674098968505859
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",57,2026203984353,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,1.305,3.85,0.8491261005401611
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",58,2026203984354,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.275,3.975,0.9739656448364258
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",19,2026202284264,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.5,1,1,71.45,67.2,3.6915781497955322
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",59,2026203984355,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,3.0,3,0,1.41,3.795,1.4015815258026123
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",60,2026203984356,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,3.0,3,0,1.445,3.91,1.0852296352386475
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",186,2026195484227,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.325,63.8,5.009885787963867
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",61,2026203984357,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.305,3.93,0.9935257434844971
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",83,2026200584277,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,2.0,2,0,32.91,36.48,5.262767553329468
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",62,2026203984358,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,4.0,4,0,1.35,3.845,1.2140958309173584
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",20,2026202284265,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,66.875,64.1,5.078498840332031
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",63,2026203984359,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.415,3.96,1.1206400394439697
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",64,2026203984360,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.5,2,1,1.415,4.055,1.1351032257080078
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",84,2026200584278,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,32.09,37.28,4.282090187072754
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",187,2026195484228,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.85,64.75,4.835628271102905
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",65,2026203984361,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.3775,3.94,1.3232781887054443
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",66,2026203984362,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.27,3.905,1.659865379333496
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",21,2026202284266,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,67.25,65.55,4.735327243804932
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",67,2026203984363,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.36,3.805,1.4671459197998047
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",188,2026195484229,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.8,64.65,3.936221122741699
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",85,2026200584279,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.2,35.36,4.031912803649902
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",68,2026203984364,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,2.5,3,1,1.3175,3.835,1.2660441398620605
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",69,2026203984365,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,3.0,3,0,1.38,4.085,1.4818751811981201
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",22,2026202284267,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,69.6,64.8,4.377953290939331
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",70,2026203984366,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.335,3.765,0.9782402515411377
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",71,2026203984367,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,4.0,4,0,1.3825,4.055,0.7743031978607178
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",86,2026200584280,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,2.0,2,0,33.79,35.62,4.287912130355835
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",72,2026203984368,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,3.0,3,0,1.4025,4.1,0.7670555114746094
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",189,2026195484230,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.55,65.3,4.507188320159912
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",73,2026203984369,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.36,3.815,1.0899016857147217
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",23,2026202284268,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,68.425,63.9,4.204033374786377
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",74,2026203984370,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.4725,3.89,1.7256782054901123
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",75,2026203984371,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.305,4.17,1.126469612121582
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",190,2026195484231,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.125,63.55,3.991990089416504
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",87,2026200584281,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,32.54,36.42,4.425880432128906
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",76,2026203984372,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,3.0,3,0,1.4175,3.875,0.9236438274383545
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",77,2026203984373,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.5,2,1,1.3925,3.92,0.9489667415618896
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",78,2026203984374,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,1.46,4.055,0.9024386405944824
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",79,2026203984375,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.4775,4.1,1.1509616374969482
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",24,2026202284269,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.5,2,1,69.3,65.35,5.815683126449585
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",88,2026200584282,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,32.57,36.8,4.328726291656494
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",80,2026203984376,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,3.0,3,0,1.3375,4.045,0.8570539951324463
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",191,2026195484232,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.725,64.3,4.874683856964111
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",81,2026203984377,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.4075,4.06,1.1493947505950928
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",82,2026203984378,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,4.5,5,1,1.4275,3.875,1.4657478332519531
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",83,2026203984379,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.5,2,1,1.36,3.92,1.0588409900665283
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",25,2026202284270,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,70.4,64.4,4.581885099411011
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",192,2026195484233,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.575,65.75,4.230129957199097
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",89,2026200584283,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,33.56,36.12,4.6868932247161865
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",84,2026203984380,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,2.5,3,1,1.355,4.045,0.907179594039917
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",85,2026203984381,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.3475,4.075,0.9343404769897461
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",86,2026203984382,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,2.5,3,1,1.4075,3.975,1.246016263961792
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",26,2026202284271,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,68.45,65.95,3.784658193588257
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",87,2026203984383,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,3.0,4,2,1.2525,4.165,1.5392615795135498
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",193,2026195484234,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.8,66.0,4.182827711105347
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",88,2026203984384,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,1.37,4.085,1.2624433040618896
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",90,2026200584284,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,32.55,35.2,5.744059085845947
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",89,2026203984385,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.335,4.05,1.1914317607879639
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",27,2026202284272,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,68.725,61.85,3.7079498767852783
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",194,2026195484235,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.3,63.95,3.5811071395874023
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",90,2026203984386,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,3.0,3,0,1.3375,4.005,1.4506161212921143
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",91,2026203984387,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,2.5,3,1,1.37,3.86,1.0421667098999023
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",92,2026203984388,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,2.5,3,1,1.3525,3.96,0.9673755168914795
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",93,2026203984389,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.3375,4.12,1.0106942653656006
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",94,2026203984390,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,3.0,4,2,1.3025,4.03,0.8643012046813965
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",195,2026195484236,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,29.55,65.4,4.151534080505371
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",91,2026200584285,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.07,35.56,6.094988107681274
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",28,2026202284273,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,67.625,65.85,5.012082815170288
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",95,2026203984391,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.5075,4.2,1.340501308441162
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",96,2026203984392,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.415,4.0,1.4373424053192139
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",196,2026195484237,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.7,64.5,3.928729772567749
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",97,2026203984393,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,3.0,3,0,1.3875,3.995,1.3669993877410889
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",29,2026202284274,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,68.9,63.05,4.531056642532349
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",98,2026203984394,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,2,2,1.35,3.935,1.0610613822937012
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",92,2026200584286,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,2.0,2,0,31.31,36.14,4.946418523788452
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",99,2026203984395,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.3175,3.87,1.0616109371185303
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",100,2026203984396,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.32,3.97,1.0161128044128418
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",101,2026203984397,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.3675,4.07,1.0765888690948486
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",30,2026202284275,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,67.25,62.4,4.035521507263184
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",197,2026195484238,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.125,65.0,4.925568342208862
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",93,2026200584287,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.09,35.7,4.455840826034546
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",102,2026203984398,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,3.5,4,1,1.4375,4.19,1.4254858493804932
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",103,2026203984399,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.36,4.0,1.256842851638794
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",104,2026203984400,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,2.5,3,1,1.3725,3.98,0.8280379772186279
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0005|pi0|k1,1,1",105,2026203984401,B,noisy,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,3.0,3,0,1.4375,3.83,0.7822971343994141
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",94,2026200584288,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,2.0,2,0,32.96,35.22,3.697115659713745
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",31,2026202284276,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,66.6,64.8,5.358035564422607
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",198,2026195484239,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.925,65.7,5.526547431945801
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",0,2026205684347,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,5.81,16.05,4.808599948883057
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",32,2026202284277,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,66.95,62.45,3.6704795360565186
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",95,2026200584289,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,2.0,2,0,32.11,35.12,5.086154937744141
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",199,2026195484240,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.65,60.6,4.445523262023926
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",1,2026205684348,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,5.945,16.06,4.365851640701294
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",200,2026195484241,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.825,64.25,4.0724945068359375
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",96,2026200584290,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.5,2,1,32.57,35.26,4.68986439704895
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",33,2026202284278,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,69.0,65.75,5.267291784286499
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",201,2026195484242,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.2,65.15,4.046803951263428
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",2,2026205684349,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,6.1475,15.985,4.883697986602783
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",97,2026200584291,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,32.9,36.72,5.020756006240845
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",34,2026202284279,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,70.25,62.7,5.299587726593018
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",202,2026195484243,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.8,66.0,4.0192530155181885
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",3,2026205684350,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,6.0525,15.96,4.87324595451355
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",98,2026200584292,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,2.0,2,0,32.19,35.82,4.23000168800354
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",35,2026202284280,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,69.55,65.2,4.957473278045654
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",203,2026195484244,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.45,61.8,4.4877777099609375
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",4,2026205684351,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,6.2225,16.06,4.835600852966309
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",99,2026200584293,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,33.72,36.2,4.874178886413574
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",36,2026202284281,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,2.0,2,0,66.3,63.75,4.782289505004883
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",204,2026195484245,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.075,63.6,4.531703948974609
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",5,2026205684352,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.89,15.74,4.7095255851745605
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",100,2026200584294,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,4.0,4,0,32.16,35.62,5.070302963256836
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",37,2026202284282,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,68.625,62.4,4.817088842391968
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",205,2026195484246,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.175,63.65,4.1909520626068115
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",101,2026200584295,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.5,2,1,33.62,35.92,4.255131483078003
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",6,2026205684353,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,6.045,16.06,5.101772308349609
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",38,2026202284283,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,69.55,64.65,4.937597036361694
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",206,2026195484247,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.775,64.6,4.161961555480957
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",7,2026205684354,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,6.0075,15.73,4.224425554275513
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",102,2026200584296,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,32.94,35.96,4.881714820861816
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",39,2026202284284,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,68.375,63.15,4.153405427932739
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",207,2026195484248,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.8,64.4,3.8863589763641357
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",8,2026205684355,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,4.5,5,1,6.1175,15.99,4.408111095428467
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",40,2026202284285,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,68.075,65.1,4.26653528213501
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",103,2026200584297,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,3.0,3,0,32.41,35.32,5.384256601333618
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",208,2026195484249,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.925,62.95,4.468750953674316
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",41,2026202284286,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,70.325,61.1,3.759850025177002
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",9,2026205684356,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,6.0725,15.925,4.97159481048584
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",209,2026195484250,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.1,62.7,4.527286767959595
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",104,2026200584298,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,32.25,35.82,5.79559588432312
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",10,2026205684357,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,2.0,2,0,6.1875,15.64,4.051158666610718
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",42,2026202284287,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,2.0,2,0,68.075,64.85,4.76511812210083
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",210,2026195484251,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.55,63.75,4.4021360874176025
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",105,2026200584299,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,32.37,35.58,3.9653263092041016
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",11,2026205684358,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,6.0625,16.01,4.857941150665283
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",211,2026195484252,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.625,63.35,3.801626205444336
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",43,2026202284288,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.5,1,1,69.65,61.5,5.881282806396484
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",106,2026200584300,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,32.7,35.98,5.398065567016602
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",12,2026205684359,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,3.0,3,0,5.975,16.015,4.717392683029175
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",212,2026195484253,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.725,63.85,4.802681922912598
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",44,2026202284289,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,68.2,64.45,3.8162100315093994
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",107,2026200584301,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,31.61,35.6,5.238655090332031
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",213,2026195484254,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.275,66.0,4.075684070587158
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",13,2026205684360,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,6.24,16.095,5.155957460403442
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",45,2026202284290,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,69.15,60.85,5.225419998168945
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",108,2026200584302,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,31.21,36.42,4.516091346740723
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",14,2026205684361,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,6.2375,16.035,4.510447025299072
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",214,2026195484255,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.525,64.05,5.538696050643921
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",46,2026202284291,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,68.7,64.6,5.037067651748657
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",109,2026200584303,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,32.94,36.36,4.724095106124878
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",15,2026205684362,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,3.0,3,0,6.2175,16.18,3.7735416889190674
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",215,2026195484256,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.7,64.95,4.892618417739868
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",47,2026202284292,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,67.65,61.5,4.1841957569122314
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",110,2026200584304,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,32.71,36.48,5.279695510864258
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",16,2026205684363,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.9875,15.88,4.785241603851318
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",48,2026202284293,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,70.125,65.65,3.8451614379882812
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",216,2026195484257,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.475,65.25,5.775066137313843
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",17,2026205684364,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,6.065,15.785,3.9540483951568604
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",111,2026200584305,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,32.84,36.68,4.648992538452148
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",49,2026202284294,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,68.85,63.55,4.890657424926758
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",217,2026195484258,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.1,63.85,4.762408971786499
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",18,2026205684365,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,2.0,2,0,6.425,15.91,4.2107093334198
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",112,2026200584306,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,3.0,3,0,32.94,36.24,5.229895114898682
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",50,2026202284295,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,68.05,66.95,5.076191663742065
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",218,2026195484259,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.3,63.2,4.181026458740234
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",19,2026205684366,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,6.01,15.625,4.732085227966309
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",113,2026200584307,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,32.58,36.06,4.904932260513306
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",51,2026202284296,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,68.5,62.35,4.185987234115601
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",219,2026195484260,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.4,62.0,4.4415318965911865
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",20,2026205684367,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,6.23,15.855,5.187983512878418
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",114,2026200584308,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,32.59,35.5,5.099437475204468
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",52,2026202284297,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,69.875,62.65,4.912708759307861
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",220,2026195484261,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.75,65.15,5.58609938621521
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",21,2026205684368,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,6.095,16.095,4.0400636196136475
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",115,2026200584309,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,32.7,35.76,4.877618789672852
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",53,2026202284298,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,69.275,64.9,5.003626346588135
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",221,2026195484262,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,1.0,1,0,28.625,64.55,4.677229404449463
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",22,2026205684369,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,6.0375,16.095,4.97241735458374
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",54,2026202284299,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,68.425,64.85,4.160027027130127
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",116,2026200584310,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,32.14,35.42,4.638195991516113
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",23,2026205684370,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,3.0,4,2,6.2075,16.295,4.655487537384033
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",222,2026195484263,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.225,64.5,5.5319623947143555
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",55,2026202284300,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,69.475,63.2,4.2839789390563965
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",117,2026200584311,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.5,2,1,33.2,35.9,4.934025764465332
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",223,2026195484264,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.1,65.45,4.669615268707275
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",24,2026205684371,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,5.9425,16.16,4.967562675476074
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",56,2026202284301,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,69.45,62.85,3.760782480239868
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",118,2026200584312,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,32.28,35.62,4.995687484741211
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",25,2026205684372,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,2.0,2,0,6.085,15.985,4.416936159133911
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",224,2026195484265,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.0,62.9,5.293342590332031
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",57,2026202284302,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,69.975,63.85,4.81110954284668
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",119,2026200584313,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,32.73,36.74,4.705524444580078
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",26,2026205684373,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,6.1,15.945,3.9821174144744873
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",225,2026195484266,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.85,62.95,3.9996063709259033
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",58,2026202284303,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,68.125,65.8,4.9496355056762695
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",120,2026200584314,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,2.0,2,0,33.3,36.8,5.0399909019470215
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",27,2026205684374,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,2.0,3,2,6.19,16.04,4.833592176437378
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",226,2026195484267,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.175,65.6,4.477926254272461
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",59,2026202284304,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,69.925,62.8,4.740624189376831
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",121,2026200584315,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,2.0,2,0,32.0,36.0,4.591574192047119
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",28,2026205684375,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,6.1675,16.105,5.029765844345093
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",227,2026195484268,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.3,65.75,5.1524763107299805
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",60,2026202284305,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.5,1,1,66.825,65.35,4.340775012969971
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",122,2026200584316,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.5,2,1,32.99,36.34,5.242254018783569
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",29,2026205684376,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,6.165,16.015,4.787161111831665
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",228,2026195484269,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.15,64.45,4.631967067718506
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",61,2026202284306,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.5,1,1,68.625,62.7,4.345665693283081
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",123,2026200584317,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.11,35.82,5.002906560897827
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",30,2026205684377,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,6.095,15.83,4.2481348514556885
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",62,2026202284307,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,70.625,63.95,4.7550084590911865
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",229,2026195484270,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.225,61.8,5.0517823696136475
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",124,2026200584318,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,31.6,34.94,3.733452320098877
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",31,2026205684378,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,6.145,16.19,4.59140419960022
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",230,2026195484271,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.7,62.1,4.113440275192261
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",63,2026202284308,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,69.45,63.85,5.293917179107666
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",125,2026200584319,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,2.0,3,2,32.94,36.42,4.205097436904907
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",231,2026195484272,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.15,63.75,3.4733669757843018
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",32,2026205684379,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,6.0275,15.975,5.054646730422974
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",64,2026202284309,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.5,1,1,70.975,62.3,5.637009620666504
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",126,2026200584320,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.56,36.6,4.786584377288818
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",232,2026195484273,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.375,65.1,3.8055291175842285
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",33,2026205684380,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,6.0425,15.655,4.628851890563965
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",127,2026200584321,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,32.06,35.46,4.308780193328857
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",233,2026195484274,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.625,63.4,4.463606595993042
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",65,2026202284310,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,66.825,62.45,5.504878759384155
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",34,2026205684381,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,6.125,15.625,5.365927696228027
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",128,2026200584322,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,32.68,35.02,4.42868971824646
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",66,2026202284311,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,70.375,62.7,3.8150274753570557
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",234,2026195484275,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.5,62.7,5.603817939758301
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",35,2026205684382,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,6.31,16.315,5.131161451339722
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",129,2026200584323,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.22,35.42,4.31593918800354
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",67,2026202284312,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,68.1,64.45,4.079503774642944
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",235,2026195484276,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.875,63.05,4.399761438369751
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",68,2026202284313,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,67.0,61.2,4.252669095993042
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",36,2026205684383,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,6.1825,15.96,4.94279146194458
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",130,2026200584324,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,2.0,2,0,31.64,36.02,5.534209489822388
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",236,2026195484277,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.175,66.05,4.253832817077637
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",69,2026202284314,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,66.725,66.9,5.039031744003296
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",131,2026200584325,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.5,2,1,31.46,37.28,4.384040594100952
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",37,2026205684384,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,6.295,15.88,5.8348448276519775
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",237,2026195484278,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,25.725,65.25,4.588411808013916
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",132,2026200584326,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.47,36.26,3.532956600189209
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",70,2026202284315,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,69.0,64.3,4.587863922119141
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",238,2026195484279,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.25,66.3,4.796553134918213
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",38,2026205684385,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,6.0875,16.095,5.3536458015441895
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",71,2026202284316,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,68.1,65.5,4.735275745391846
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",133,2026200584327,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,32.12,35.3,5.294048547744751
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",239,2026195484280,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.775,61.55,4.192351818084717
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",39,2026205684386,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,6.1,16.035,5.168458461761475
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",240,2026195484281,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.35,63.5,3.764310598373413
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",134,2026200584328,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.55,36.46,4.453921794891357
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",72,2026202284317,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,66.525,62.15,5.843940496444702
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",40,2026205684387,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,2.5,3,1,6.155,16.315,4.5944037437438965
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",135,2026200584329,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,3.0,3,0,32.75,35.68,4.429396152496338
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",241,2026195484282,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.325,63.5,4.711546897888184
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",73,2026202284318,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,69.875,63.55,4.713971376419067
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",41,2026205684388,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,5.8025,15.785,5.1315765380859375
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",136,2026200584330,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,32.38,35.88,4.745370388031006
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",242,2026195484283,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.7,63.85,4.651698350906372
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",74,2026202284319,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.5,2,1,72.7,62.8,5.151238203048706
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",42,2026205684389,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.9675,16.025,4.411006450653076
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",137,2026200584331,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,31.98,36.46,4.489640474319458
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",243,2026195484284,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,29.075,63.5,5.115691423416138
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",75,2026202284320,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,69.575,63.55,4.68722677230835
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",43,2026205684390,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,6.1875,16.14,4.965322971343994
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",138,2026200584332,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,32.23,35.38,4.7292869091033936
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",244,2026195484285,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.85,63.35,4.989423990249634
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",76,2026202284321,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,67.45,62.75,5.263253927230835
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",44,2026205684391,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,2.0,2,0,5.9575,16.285,4.820231914520264
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",139,2026200584333,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,32.5,36.36,4.4062769412994385
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",245,2026195484286,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.275,63.95,5.061175584793091
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",45,2026205684392,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,6.1225,16.115,4.577208995819092
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",77,2026202284322,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,71.375,65.5,4.933039665222168
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",140,2026200584334,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,32.57,36.7,5.28351902961731
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",246,2026195484287,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.75,64.7,5.387249231338501
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",78,2026202284323,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,69.0,62.55,4.340863943099976
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",46,2026205684393,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,5.97,15.78,4.998881578445435
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",141,2026200584335,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,2.0,2,0,33.08,35.9,4.980520963668823
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",247,2026195484288,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.55,64.55,4.929251432418823
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",47,2026205684394,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,6.1475,15.6,4.127439498901367
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",79,2026202284324,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,67.75,65.25,5.678791046142578
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",142,2026200584336,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,2.0,2,0,32.84,35.82,4.375699996948242
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",248,2026195484289,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.125,64.1,5.307374715805054
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",48,2026205684395,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,6.125,15.665,5.3504109382629395
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",80,2026202284325,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,67.025,63.2,4.6106390953063965
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",143,2026200584337,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,31.74,36.28,4.126123905181885
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",249,2026195484290,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,26.9,63.5,5.062649965286255
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",49,2026205684396,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,6.05,16.51,4.924591064453125
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",81,2026202284326,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,2.0,2,0,70.175,64.45,5.18674635887146
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",144,2026200584338,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.27,36.12,4.576737880706787
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",50,2026205684397,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,6.1425,16.18,4.196355819702148
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",250,2026195484291,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.675,63.65,4.66383957862854
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",145,2026200584339,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,33.41,35.96,4.123451232910156
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",82,2026202284327,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,67.35,62.0,6.435712099075317
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",251,2026195484292,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.825,63.2,4.235944986343384
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",146,2026200584340,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,32.88,36.88,5.092012882232666
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",51,2026205684398,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,6.1375,15.925,5.721416711807251
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",83,2026202284328,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,2.0,2,0,71.275,66.05,4.036997079849243
production-20261029T053925Z-29f79c,"A|noisy|d9|p0.0001|pi0.0001|k1,1,1",252,2026195484293,A,noisy,9,0.0001,0.0001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.125,63.05,3.4319088459014893
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",52,2026205684399,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,6.2075,15.605,4.462114572525024
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",147,2026200584341,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.5,2,1,32.57,35.5,4.627621412277222
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",84,2026202284329,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,67.95,64.65,5.092060327529907
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",53,2026205684400,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,2.0,2,0,6.1025,15.865,4.646148443222046
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",0,2026207384398,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.86,36.84,4.488463640213013
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",148,2026200584342,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,32.01,36.06,5.105706453323364
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",85,2026202284330,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,69.75,62.75,4.957914590835571
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",1,2026207384399,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.86,36.78,3.459014415740967
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",149,2026200584343,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,32.36,35.38,5.052754640579224
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",54,2026205684401,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,6.0625,15.82,6.194247722625732
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",86,2026202284331,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,68.7,62.45,4.511515378952026
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",2,2026207384400,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.56,35.54,3.7448015213012695
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",150,2026200584344,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,32.58,34.88,4.548649549484253
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",55,2026205684402,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,6.2275,15.725,4.496367931365967
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",3,2026207384401,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,16.41,35.8,4.4480249881744385
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",87,2026202284332,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,67.0,65.05,5.818670749664307
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",151,2026200584345,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,2.0,2,0,32.8,36.04,4.679003000259399
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",56,2026205684403,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,5.9925,15.955,4.820985555648804
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",4,2026207384402,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.63,36.28,3.8639349937438965
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",88,2026202284333,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,69.95,64.4,5.029990196228027
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",5,2026207384403,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,16.42,36.34,3.7933895587921143
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",152,2026200584346,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,33.21,36.06,5.084986925125122
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",57,2026205684404,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,6.4875,15.96,4.868090629577637
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",89,2026202284334,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,67.875,64.45,5.42097544670105
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",6,2026207384404,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,16.45,34.44,4.139154672622681
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",58,2026205684405,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,6.2825,16.125,4.307682275772095
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",153,2026200584347,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.67,35.86,5.193445205688477
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",90,2026202284335,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,67.55,63.45,4.7696216106414795
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",7,2026207384405,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.2,36.06,4.195825815200806
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",59,2026205684406,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.94,15.835,4.979597806930542
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",154,2026200584348,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,31.68,35.84,4.67555832862854
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",91,2026202284336,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,70.15,64.3,4.678295135498047
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",8,2026207384406,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.19,36.5,4.435716867446899
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",60,2026205684407,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,6.11,16.07,4.831939220428467
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",155,2026200584349,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,33.59,35.74,4.775342226028442
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",92,2026202284337,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,68.875,63.65,3.5945539474487305
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",9,2026207384407,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.52,36.2,5.43347430229187
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",61,2026205684408,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,6.18,16.215,4.4871790409088135
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",156,2026200584350,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,2.0,3,2,33.21,36.08,5.076587438583374
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",93,2026202284338,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,66.95,64.3,4.691240310668945
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",10,2026207384408,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.28,36.34,4.605764150619507
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",62,2026205684409,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,6.32,15.88,5.0896830558776855
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",157,2026200584351,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,32.76,35.44,4.381732702255249
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",94,2026202284339,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,69.4,63.55,4.084131956100464
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",11,2026207384409,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,16.26,37.1,4.051770925521851
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",95,2026202284340,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,66.775,63.3,4.115265130996704
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",63,2026205684410,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,6.0525,16.215,4.9771928787231445
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",158,2026200584352,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.29,36.08,5.23296856880188
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",12,2026207384410,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,15.69,35.06,4.513956308364868
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",96,2026202284341,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,67.0,66.75,4.4858973026275635
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",64,2026205684411,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,6.025,16.345,5.280932664871216
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",159,2026200584353,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,2.0,2,0,32.09,36.64,5.332549095153809
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",13,2026207384411,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.59,36.84,4.160285711288452
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",97,2026202284342,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,68.025,64.05,3.5643465518951416
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",65,2026205684412,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,6.2175,16.165,4.8101911544799805
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",14,2026207384412,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.18,35.24,4.439908981323242
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",160,2026200584354,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,32.67,36.22,4.928564786911011
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",98,2026202284343,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,69.25,64.45,4.523168087005615
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",15,2026207384413,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.55,35.98,4.137754678726196
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",66,2026205684413,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,5.9375,16.215,4.79623818397522
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",161,2026200584355,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,31.9,36.36,4.438833475112915
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",99,2026202284344,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,69.1,63.9,4.231089353561401
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",67,2026205684414,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,4.5,6,3,6.3125,16.4,4.019068717956543
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",16,2026207384414,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.5,2,1,15.75,35.18,4.721019983291626
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",162,2026200584356,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,33.6,36.08,5.167867422103882
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",100,2026202284345,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,69.125,63.95,4.61141562461853
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",17,2026207384415,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.08,35.04,3.7692008018493652
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",68,2026205684415,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,6.1575,15.985,5.477476596832275
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",163,2026200584357,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,32.07,34.78,4.513155221939087
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",101,2026202284346,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,68.75,65.35,4.947533845901489
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",18,2026207384416,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,15.63,36.78,3.918065071105957
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",69,2026205684416,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,5.92,15.84,4.191626310348511
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",164,2026200584358,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.17,36.42,4.813138961791992
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",102,2026202284347,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,70.525,65.85,4.877091646194458
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",19,2026207384417,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.05,36.9,5.211867094039917
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",70,2026205684417,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,6.1225,16.18,4.826991081237793
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",165,2026200584359,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,2.0,2,0,32.59,35.92,4.8744752407073975
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",103,2026202284348,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,69.1,63.45,4.2775185108184814
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",20,2026207384418,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.77,36.58,4.162564277648926
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",71,2026205684418,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,2.5,4,3,6.24,15.945,4.977212429046631
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",166,2026200584360,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,2.0,2,0,31.94,35.74,4.70445704460144
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",104,2026202284349,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,67.425,65.35,4.107116222381592
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",21,2026207384419,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.64,36.84,4.711271047592163
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",72,2026205684419,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,2.0,2,0,6.2275,15.615,4.658526182174683
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",105,2026202284350,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,68.525,64.7,4.857102632522583
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",167,2026200584361,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,32.05,35.22,5.582410573959351
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",22,2026207384420,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.43,35.62,3.992589235305786
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",73,2026205684420,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,3.5,5,3,6.1075,16.05,4.144495010375977
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",106,2026202284351,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,67.275,64.6,4.197720527648926
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",168,2026200584362,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,33.07,37.06,4.863717555999756
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",23,2026207384421,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.65,36.0,4.8785223960876465
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",74,2026205684421,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,5.99,16.165,4.424238204956055
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",107,2026202284352,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,66.7,64.0,4.742705583572388
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",24,2026207384422,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,15.93,35.38,3.70365309715271
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",169,2026200584363,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.5,2,1,32.32,35.72,5.519693374633789
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",75,2026205684422,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.94,15.625,4.344466924667358
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",108,2026202284353,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,68.775,66.15,3.935554265975952
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",25,2026207384423,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.21,36.22,4.344357013702393
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",170,2026200584364,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,2.0,2,0,32.19,36.0,5.235035419464111
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",76,2026205684423,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,6.1925,16.085,4.465852975845337
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",109,2026202284354,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,69.025,65.95,4.69691014289856
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",26,2026207384424,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,16.83,36.26,4.4625771045684814
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",77,2026205684424,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,6.215,15.675,4.332391738891602
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",110,2026202284355,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,70.075,64.05,4.388874292373657
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",171,2026200584365,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,33.55,35.94,5.916187763214111
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",27,2026207384425,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.69,36.74,4.566516637802124
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",78,2026205684425,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,6.2575,16.035,4.523704290390015
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",172,2026200584366,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,31.41,35.76,4.832431793212891
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",28,2026207384426,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.32,36.12,3.9730443954467773
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",111,2026202284356,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,67.275,65.05,5.094728469848633
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",79,2026205684426,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.9975,15.935,4.42576789855957
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",29,2026207384427,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,2.0,2,0,16.58,36.2,4.261503458023071
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",112,2026202284357,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,67.775,63.85,4.632450580596924
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",173,2026200584367,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.4,35.6,5.04269003868103
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",80,2026205684427,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,6.0675,15.885,4.692961692810059
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",30,2026207384428,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.17,36.52,4.577341794967651
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",113,2026202284358,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,67.9,64.2,4.128812789916992
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",174,2026200584368,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,32.74,34.64,4.864218711853027
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",31,2026207384429,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,15.94,34.96,3.7904012203216553
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",81,2026205684428,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,6.13,16.0,5.809030055999756
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",114,2026202284359,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,66.8,64.85,5.277639627456665
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",175,2026200584369,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,33.08,35.7,5.058608770370483
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",32,2026207384430,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.22,35.44,4.403347969055176
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",82,2026205684429,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,5.955,15.815,4.707526922225952
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",115,2026202284360,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,67.4,65.35,4.987001180648804
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",176,2026200584370,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,32.6,35.48,5.500680685043335
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",33,2026207384431,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.62,37.02,4.889174222946167
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",83,2026205684430,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,6.1225,16.25,4.95301079750061
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",116,2026202284361,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,67.175,64.55,4.345644474029541
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",177,2026200584371,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,32.55,35.68,5.164394855499268
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",34,2026207384432,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.66,35.74,4.337037801742554
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",117,2026202284362,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,69.925,62.3,4.391117095947266
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",84,2026205684431,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,6.2,16.165,5.4892730712890625
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",178,2026200584372,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.41,36.6,3.68316388130188
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",35,2026207384433,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.88,35.7,4.600503206253052
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",118,2026202284363,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.5,2,1,69.35,64.2,4.835850954055786
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",85,2026205684432,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,2.0,2,0,6.005,15.77,4.914034366607666
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",179,2026200584373,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,32.74,36.64,4.5119407176971436
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",36,2026207384434,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,17.24,35.74,5.121242523193359
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",86,2026205684433,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.9225,15.98,3.799957752227783
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",119,2026202284364,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.5,1,1,70.475,62.05,5.6112895011901855
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",180,2026200584374,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,33.64,36.22,5.2678680419921875
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",37,2026207384435,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,15.65,36.46,3.741725206375122
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",87,2026205684434,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,2.0,2,0,6.1425,16.335,3.8717751502990723
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",38,2026207384436,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,15.97,36.16,3.774059295654297
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",181,2026200584375,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,32.98,36.52,5.061093091964722
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",120,2026202284365,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,68.3,63.45,5.693125009536743
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",88,2026205684435,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,6.11,16.27,5.425787687301636
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",39,2026207384437,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,15.65,35.86,3.842576265335083
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",182,2026200584376,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,32.62,36.34,4.642435789108276
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",121,2026202284366,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,69.125,64.5,4.977221727371216
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",89,2026205684436,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,6.255,16.09,4.734073638916016
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",40,2026207384438,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,16.03,35.26,4.596613645553589
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",183,2026200584377,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,2.5,3,1,32.95,35.3,4.292424201965332
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",122,2026202284367,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,69.5,64.65,4.181684970855713
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",41,2026207384439,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,16.27,36.1,3.836897373199463
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",90,2026205684437,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.0,2,2,5.8525,16.38,5.5619752407073975
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",184,2026200584378,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,32.35,35.82,4.238757610321045
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",123,2026202284368,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,69.275,64.0,5.24639892578125
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",42,2026207384440,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.42,36.66,3.426124334335327
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",91,2026205684438,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,6.2475,16.145,4.833270311355591
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",124,2026202284369,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,67.375,65.45,4.299830675125122
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",185,2026200584379,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,32.58,36.28,5.8328938484191895
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",43,2026207384441,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,16.6,35.88,4.355148553848267
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",92,2026205684439,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,6.185,16.12,4.887444496154785
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",44,2026207384442,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.85,35.82,3.9531173706054688
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",125,2026202284370,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,69.05,65.65,5.062429666519165
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",186,2026200584380,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,32.21,35.86,5.205108880996704
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",93,2026205684440,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,6.29,15.81,4.546951055526733
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",187,2026200584381,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,2.0,2,0,33.22,36.34,4.103536128997803
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",45,2026207384443,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.85,36.64,5.433308124542236
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",126,2026202284371,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,68.3,62.9,5.603541135787964
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",94,2026205684441,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,6.03,16.245,4.957201719284058
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",46,2026207384444,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,17.34,34.96,4.443428039550781
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",188,2026200584382,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,33.53,36.58,5.275893211364746
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",127,2026202284372,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,69.625,64.0,4.5735132694244385
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",95,2026205684442,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,6.2175,15.9,3.6289031505584717
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",47,2026207384445,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,15.48,35.38,5.040721654891968
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",128,2026202284373,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,65.975,62.4,4.640383958816528
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",189,2026200584383,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,32.87,35.7,5.204074382781982
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",96,2026205684443,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,6.2775,16.105,4.2673938274383545
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",48,2026207384446,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.37,35.58,3.9588444232940674
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",129,2026202284374,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,68.125,64.3,4.0656960010528564
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",190,2026200584384,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,32.21,35.98,5.530672550201416
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",97,2026205684444,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.9775,16.01,4.979196310043335
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",49,2026207384447,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.07,35.74,4.293457508087158
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",130,2026202284375,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,67.8,64.3,4.951771974563599
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",191,2026200584385,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.01,36.16,4.192487716674805
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",98,2026205684445,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,6.2725,16.065,4.988615989685059
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",50,2026207384448,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,15.51,36.26,4.286360025405884
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",131,2026202284376,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,2.0,2,0,71.25,65.55,4.300664901733398
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",192,2026200584386,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,32.79,35.62,4.841429710388184
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",99,2026205684446,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,6.08,16.105,4.899872779846191
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",51,2026207384449,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.45,36.04,4.111211538314819
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",132,2026202284377,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,69.025,63.05,5.524331331253052
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",193,2026200584387,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,31.93,35.98,5.038639068603516
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",52,2026207384450,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,16.0,36.34,4.024231672286987
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",100,2026205684447,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,3.0,4,2,6.2625,16.22,4.567434787750244
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",133,2026202284378,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,68.5,62.75,4.094511032104492
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",194,2026200584388,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,32.29,35.76,4.346424341201782
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",101,2026205684448,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,2.5,3,1,5.955,15.99,4.508816957473755
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",53,2026207384451,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.73,36.5,5.535406827926636
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",134,2026202284379,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,68.55,62.0,4.544056177139282
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",195,2026200584389,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,2.5,3,1,33.3,35.38,5.471415996551514
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",102,2026205684449,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,5.9475,15.965,4.637418270111084
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",54,2026207384452,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.24,36.04,3.825206995010376
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",135,2026202284380,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.5,2,1,65.35,64.9,4.132232427597046
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",103,2026205684450,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,6.1625,16.31,3.5182485580444336
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",55,2026207384453,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.46,37.06,4.735435724258423
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",196,2026200584390,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.65,36.62,5.280557632446289
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",136,2026202284381,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,69.55,60.8,4.4073646068573
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",104,2026205684451,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,6.0,15.985,4.54420018196106
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",197,2026200584391,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.58,37.1,4.406691551208496
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",56,2026207384454,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.35,37.0,4.801882266998291
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",137,2026202284382,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,69.85,64.25,4.9729156494140625
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",105,2026205684452,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,5.9725,15.985,4.938612461090088
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",57,2026207384455,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.33,35.96,3.9179699420928955
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.001|pi0|k1,1,1",198,2026200584392,B,noisy,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,2.0,2,0,34.09,35.34,5.327659606933594
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",138,2026202284383,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,69.8,64.75,4.533238410949707
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",106,2026205684453,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,5.83,16.08,4.574813365936279
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",58,2026207384456,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.67,35.98,4.311831712722778
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",139,2026202284384,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.5,2,1,72.675,64.55,5.101898670196533
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",59,2026207384457,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.36,36.5,3.6563761234283447
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",107,2026205684454,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,6.2725,15.95,5.724770545959473
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",0,2026209084449,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.45,62.75,5.398221015930176
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",60,2026207384458,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.04,35.38,4.2765889167785645
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",140,2026202284385,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,2.0,2,0,68.9,59.9,5.467993259429932
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",108,2026205684455,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,6.0825,15.835,4.492632627487183
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",1,2026209084450,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,36.1,62.2,4.6466710567474365
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",61,2026207384459,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.62,37.54,4.452645301818848
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",141,2026202284386,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,69.65,64.7,5.389311790466309
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",109,2026205684456,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,6.015,15.87,4.806419610977173
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",62,2026207384460,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.53,35.38,4.218224763870239
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",2,2026209084451,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.525,65.5,4.615807771682739
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",142,2026202284387,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,69.15,62.75,4.475908041000366
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",110,2026205684457,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,2.0,2,0,5.99,16.315,4.413436651229858
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",63,2026207384461,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,2.5,3,1,16.82,36.64,4.861525535583496
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",3,2026209084452,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.875,63.55,5.007293939590454
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",143,2026202284388,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,68.125,62.15,4.6753928661346436
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",64,2026207384462,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.11,35.84,3.674260377883911
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",111,2026205684458,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,6.235,16.15,6.283365249633789
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",4,2026209084453,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.725,64.3,5.060487508773804
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",144,2026202284389,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,68.825,63.35,3.9856905937194824
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",65,2026207384463,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.05,35.36,3.4143970012664795
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",112,2026205684459,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,6.1825,15.765,5.622609376907349
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",5,2026209084454,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,35.75,62.1,5.026273012161255
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",145,2026202284390,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,67.45,63.65,4.932090759277344
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",66,2026207384464,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,16.1,36.18,4.745594024658203
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",113,2026205684460,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,6.0275,15.97,4.32182240486145
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",146,2026202284391,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,2.0,2,0,69.25,62.9,4.773643732070923
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",6,2026209084455,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.75,62.95,4.966672897338867
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",67,2026207384465,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,15.92,35.62,3.811110258102417
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",114,2026205684461,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,3.5,4,1,6.08,15.82,4.170673370361328
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",147,2026202284392,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,69.925,62.4,4.802968978881836
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",7,2026209084456,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,36.45,65.5,4.751035690307617
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",68,2026207384466,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,15.89,35.16,4.827877521514893
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",115,2026205684462,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,6.045,16.225,4.8472065925598145
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",8,2026209084457,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.65,62.05,4.44602370262146
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",148,2026202284393,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,68.35,66.3,4.64582371711731
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",69,2026207384467,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.18,36.28,4.668925523757935
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",116,2026205684463,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.9775,16.07,5.103117942810059
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",9,2026209084458,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,35.05,62.75,4.267452001571655
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",70,2026207384468,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.53,37.22,4.349849700927734
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",149,2026202284394,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,2.0,2,0,67.75,62.3,5.169574975967407
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",10,2026209084459,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.15,64.35,3.3255207538604736
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",117,2026205684464,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,2.5,3,1,6.1075,16.245,5.04324746131897
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",71,2026207384469,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,16.13,34.6,4.590555906295776
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",150,2026202284395,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,69.2,62.0,5.415885925292969
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",11,2026209084460,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,35.575,61.8,3.6860101222991943
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",118,2026205684465,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,5.92,15.715,4.578498363494873
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",72,2026207384470,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.37,36.86,4.636040687561035
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",12,2026209084461,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.35,63.5,3.5753026008605957
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",151,2026202284396,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,69.275,64.5,4.784689903259277
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",119,2026205684466,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,6.115,15.99,5.064147472381592
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",73,2026207384471,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,15.97,35.78,4.898622751235962
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",13,2026209084462,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,35.15,64.25,4.599404573440552
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",152,2026202284397,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,69.275,65.65,4.487030506134033
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",120,2026205684467,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.9925,15.87,4.589369297027588
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",74,2026207384472,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,15.94,36.38,4.1350672245025635
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",153,2026202284398,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,71.85,62.25,4.828654527664185
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",14,2026209084463,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,35.675,66.55,5.49432897567749
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",75,2026207384473,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,16.74,35.58,4.450300216674805
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",121,2026205684468,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,6.0475,16.145,5.037948846817017
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",154,2026202284399,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,68.9,65.05,4.314974784851074
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",15,2026209084464,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,35.4,63.9,5.733106374740601
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",122,2026205684469,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,5.965,16.06,3.954928398132324
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",76,2026207384474,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,15.75,35.86,4.842837572097778
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",155,2026202284400,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,67.625,62.6,4.801723480224609
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",16,2026209084465,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,35.15,62.85,4.506792783737183
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",123,2026205684470,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,2.0,3,2,6.1225,16.03,4.72911524772644
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",77,2026207384475,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,16.48,35.6,5.07466197013855
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",17,2026209084466,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.125,65.0,3.8832547664642334
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",156,2026202284401,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,67.15,65.1,5.68479323387146
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",124,2026205684471,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,6.1275,16.055,4.54785943031311
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",78,2026207384476,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,2,2,15.92,36.08,4.2636871337890625
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",18,2026209084467,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,33.6,61.6,3.8255362510681152
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",157,2026202284402,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.5,1,1,67.15,63.7,4.118499994277954
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",125,2026205684472,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,5.9925,16.005,5.066900014877319
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",79,2026207384477,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.55,36.22,4.241910696029663
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",19,2026209084468,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.95,63.7,3.340226650238037
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",158,2026202284403,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,70.4,63.65,5.422434568405151
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",126,2026205684473,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,6.2025,15.79,3.8307132720947266
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",20,2026209084469,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.45,64.5,4.020675897598267
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",80,2026207384478,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.14,35.28,5.5430333614349365
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",159,2026202284404,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,68.275,62.2,4.936580419540405
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",127,2026205684474,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,6.1325,16.225,4.60400128364563
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",21,2026209084470,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,35.025,64.3,4.4161152839660645
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",81,2026207384479,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,16.43,36.28,4.909499645233154
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",128,2026205684475,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,6.1175,15.705,4.140340089797974
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",22,2026209084471,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.575,64.7,3.8913631439208984
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",160,2026202284405,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,67.4,63.5,5.573659658432007
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",82,2026207384480,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.26,35.48,3.9740889072418213
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",129,2026205684476,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,6.25,15.96,4.029173135757446
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",23,2026209084472,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,35.325,64.1,5.054358243942261
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",161,2026202284406,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,69.675,63.35,5.255526304244995
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",83,2026207384481,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.08,35.06,4.552443027496338
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",130,2026205684477,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,2.0,2,0,6.0525,15.9,4.455625295639038
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",84,2026207384482,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.14,36.36,3.651477813720703
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",24,2026209084473,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,35.8,66.85,5.192607164382935
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",162,2026202284407,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,69.475,64.3,6.2209272384643555
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",131,2026205684478,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,6.115,15.925,4.802350759506226
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",25,2026209084474,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,32.075,63.9,3.7067039012908936
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",85,2026207384483,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.55,36.22,4.4380152225494385
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",163,2026202284408,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,68.175,64.1,4.644036531448364
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",132,2026205684479,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,5.97,16.195,4.6941938400268555
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",86,2026207384484,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.82,36.3,4.348430633544922
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",26,2026209084475,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.075,63.65,5.17907190322876
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",164,2026202284409,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.5,1,1,68.325,62.95,4.536248683929443
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",133,2026205684480,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,6.02,15.93,4.523098468780518
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",27,2026209084476,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.8,62.4,4.1109397411346436
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",87,2026207384485,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.14,36.22,4.648406505584717
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",165,2026202284410,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,69.15,64.7,4.338391065597534
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",134,2026205684481,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,6.085,16.38,4.310154914855957
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",88,2026207384486,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,15.83,36.42,4.359528541564941
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",28,2026209084477,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.575,62.05,5.844022512435913
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",166,2026202284411,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,4.0,4,0,67.975,63.75,3.9447031021118164
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",135,2026205684482,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,2.5,3,1,6.125,16.19,5.131077289581299
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",89,2026207384487,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.49,35.82,4.496141195297241
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",167,2026202284412,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,68.025,64.75,4.057399272918701
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",29,2026209084478,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.425,65.85,5.145201683044434
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",136,2026205684483,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,6.005,16.1,4.403384685516357
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",90,2026207384488,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,16.29,35.18,5.2007551193237305
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",30,2026209084479,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.325,64.35,4.453317403793335
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",168,2026202284413,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,68.0,64.5,5.203981876373291
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",137,2026205684484,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.84,16.33,4.744471788406372
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",91,2026207384489,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,16.01,35.14,4.393987655639648
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",31,2026209084480,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,35.8,63.65,3.986316442489624
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",169,2026202284414,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,67.95,64.3,4.3070104122161865
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",138,2026205684485,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,6.0275,16.155,4.808634519577026
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",32,2026209084481,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,35.325,62.35,3.470848560333252
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",92,2026207384490,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,15.63,35.84,4.519076108932495
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",170,2026202284415,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,67.5,63.4,4.486616849899292
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",93,2026207384491,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,16.45,36.38,3.3839144706726074
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",139,2026205684486,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,3.5,4,1,6.145,16.14,4.306520938873291
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",33,2026209084482,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,35.85,62.65,4.623655796051025
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",171,2026202284416,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,69.325,63.05,5.4374098777771
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",94,2026207384492,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.4,35.62,4.356842994689941
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",34,2026209084483,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,32.175,62.6,4.305109024047852
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",140,2026205684487,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.96,16.165,4.963496923446655
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",172,2026202284417,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,68.525,64.1,4.20642614364624
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",95,2026207384493,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,16.09,35.76,3.627432107925415
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",35,2026209084484,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,35.525,62.5,4.35163950920105
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",141,2026205684488,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,5.995,16.19,5.371679782867432
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",96,2026207384494,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.5,2,1,16.8,36.44,3.8137893676757812
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",173,2026202284418,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,2.0,2,0,68.775,61.8,4.653811693191528
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",36,2026209084485,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.95,64.6,4.322725296020508
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",142,2026205684489,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,6.2025,16.025,4.130844354629517
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",174,2026202284419,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,68.0,64.1,3.7470109462738037
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",97,2026207384495,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,16.13,37.4,5.011951684951782
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",37,2026209084486,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.05,63.45,4.835244655609131
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",143,2026205684490,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,3.5,4,1,5.985,15.89,4.317705154418945
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",98,2026207384496,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,15.58,35.8,3.9068381786346436
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",175,2026202284420,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,67.625,63.25,4.570894479751587
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",38,2026209084487,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.35,64.15,4.263794422149658
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",144,2026205684491,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,6.255,15.865,5.70665431022644
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",99,2026207384497,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,16.66,36.62,4.435619831085205
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",39,2026209084488,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.025,63.7,3.4467833042144775
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",176,2026202284421,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,70.575,61.85,5.8083014488220215
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",40,2026209084489,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,35.475,63.9,3.7175159454345703
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",145,2026205684492,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,5.9925,16.56,5.509725570678711
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",100,2026207384498,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.46,35.72,5.188751935958862
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",177,2026202284422,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,69.55,62.9,4.35202431678772
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",41,2026209084490,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.15,64.05,3.732069730758667
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",101,2026207384499,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,16.41,35.96,4.293869256973267
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",146,2026205684493,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,6.0325,16.005,4.872895956039429
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",178,2026202284423,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.5,1,1,69.35,63.45,4.603550434112549
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",42,2026209084491,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,35.875,64.7,4.940199851989746
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",102,2026207384500,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.87,35.48,4.73029088973999
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",179,2026202284424,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,68.3,64.25,3.9155147075653076
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",147,2026205684494,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.9075,15.61,5.341096878051758
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",43,2026209084492,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.05,61.9,3.28236985206604
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",103,2026207384501,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.87,35.16,4.772134304046631
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",148,2026205684495,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,6.0825,16.025,4.231641054153442
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",180,2026202284425,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,69.65,65.95,5.373584747314453
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",44,2026209084493,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,35.5,63.65,4.7248687744140625
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",104,2026207384502,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.5,2,1,15.99,36.16,4.595709323883057
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",181,2026202284426,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,69.675,64.6,4.119908332824707
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",149,2026205684496,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,6.0775,15.845,4.636070251464844
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",45,2026209084494,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,35.25,61.8,3.598984956741333
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",105,2026207384503,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.42,36.92,4.438534736633301
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",150,2026205684497,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,5.9625,16.085,4.760652542114258
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",182,2026202284427,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,68.9,60.5,5.674964666366577
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",46,2026209084495,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,35.425,64.6,4.606143951416016
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",106,2026207384504,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,16.12,36.28,4.666001796722412
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",151,2026205684498,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.0,2,2,5.92,16.24,4.524549722671509
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",47,2026209084496,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.625,65.15,4.426949739456177
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",183,2026202284428,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,68.2,61.5,4.905765533447266
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",107,2026207384505,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.33,36.88,4.586127519607544
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",152,2026205684499,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,5.8375,15.73,5.314725637435913
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",184,2026202284429,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,68.4,63.35,4.435319900512695
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",48,2026209084497,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.5,1,1,33.525,65.6,4.69648289680481
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",108,2026207384506,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,15.53,35.82,4.385744571685791
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",185,2026202284430,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,69.8,63.7,4.196571588516235
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",49,2026209084498,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,35.4,65.2,4.236015796661377
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",153,2026205684500,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,2.0,2,0,6.5625,16.21,6.257458209991455
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",109,2026207384507,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.6,36.04,4.404904127120972
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",50,2026209084499,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.875,65.25,4.2346837520599365
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",186,2026202284431,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,69.275,64.05,5.620491981506348
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",154,2026205684501,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,6.2425,16.205,3.718979597091675
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",110,2026207384508,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,16.71,36.06,5.165754079818726
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",51,2026209084500,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.425,63.55,4.397511959075928
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",155,2026205684502,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,6.08,15.715,4.5329084396362305
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",187,2026202284432,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,69.275,63.8,5.76244330406189
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",52,2026209084501,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.2,65.5,3.3777101039886475
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",111,2026207384509,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.17,35.64,4.465830326080322
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",156,2026205684503,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,3.0,3,0,6.1925,16.195,3.7522456645965576
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",112,2026207384510,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.65,35.7,3.5240519046783447
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",188,2026202284433,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,67.725,62.35,5.270034551620483
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",53,2026209084502,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.375,61.2,4.568494081497192
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",157,2026205684504,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,6.0425,15.805,4.8567352294921875
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",54,2026209084503,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,32.725,63.2,3.7935523986816406
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",113,2026207384511,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.58,35.52,5.533195734024048
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",189,2026202284434,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,69.075,62.9,5.92296028137207
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",158,2026205684505,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,6.055,16.04,4.678318738937378
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",55,2026209084504,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.45,64.5,4.134092092514038
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",114,2026207384512,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.5,37.12,4.588149070739746
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",190,2026202284435,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,67.625,64.05,4.141693353652954
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",159,2026205684506,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,6.3625,16.44,4.362374782562256
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",56,2026209084505,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.5,1,1,34.0,63.8,4.949514389038086
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",115,2026207384513,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.94,36.76,4.376530647277832
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",191,2026202284436,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,69.45,64.4,5.114950180053711
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",160,2026205684507,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,3.5,4,1,6.1325,16.235,4.932316303253174
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",57,2026209084506,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,35.175,65.75,4.070226192474365
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",116,2026207384514,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.73,36.14,4.839436054229736
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",192,2026202284437,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,70.575,64.3,4.637537717819214
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",161,2026205684508,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,6.09,15.975,5.006209850311279
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",58,2026209084507,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.4,63.8,4.1227076053619385
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",117,2026207384515,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.02,37.14,3.6905782222747803
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",193,2026202284438,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,69.95,64.2,5.134033203125
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",59,2026209084508,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,36.3,65.65,4.06085991859436
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",162,2026205684509,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,6.1225,16.025,4.958863258361816
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",118,2026207384516,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.5,2,1,16.29,36.16,5.015583038330078
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",194,2026202284439,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,66.55,62.75,4.609918117523193
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",60,2026209084509,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.85,64.5,4.74507212638855
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",163,2026205684510,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,6.275,16.575,4.399616718292236
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",119,2026207384517,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,15.71,35.32,3.84091854095459
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",195,2026202284440,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,2.0,2,0,71.35,62.9,4.533595085144043
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",61,2026209084510,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,33.25,66.75,4.27693247795105
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",164,2026205684511,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.8975,16.155,4.86547327041626
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",120,2026207384518,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,2.0,2,0,17.2,37.02,4.346253871917725
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",196,2026202284441,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,68.825,63.8,4.547048091888428
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",165,2026205684512,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,5.9825,16.08,3.669215440750122
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",62,2026209084511,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.325,64.6,5.351656913757324
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",121,2026207384519,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.49,36.12,4.674354553222656
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",197,2026202284442,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,69.5,61.3,4.407209634780884
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",122,2026207384520,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.17,35.16,3.7664756774902344
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",166,2026205684513,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,6.01,16.04,5.371432065963745
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",63,2026209084512,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.825,64.6,4.929701328277588
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",123,2026207384521,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.39,35.0,3.4995312690734863
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",198,2026202284443,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,69.05,63.1,4.673064470291138
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",64,2026209084513,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.3,64.3,4.301283597946167
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",167,2026205684514,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,5.7675,16.03,4.8680431842803955
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",124,2026207384522,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.55,36.48,3.84230375289917
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",65,2026209084514,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,34.175,64.0,4.426417112350464
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",168,2026205684515,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,6.0325,15.885,4.242745876312256
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",199,2026202284444,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,69.025,64.75,5.825345516204834
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",125,2026207384523,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,15.97,36.18,5.336182355880737
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",66,2026209084515,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.5,1,1,33.775,64.2,3.6940197944641113
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0005|pi0|k1,1,1",169,2026205684516,B,noisy,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,2.0,2,0,6.18,15.925,4.807828664779663
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",200,2026202284445,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,67.5,63.45,5.287184476852417
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",0,2026210784500,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.8675,3.915,1.2242999076843262
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",1,2026210784501,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,2.5,3,1,0.9,4.045,0.7986311912536621
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",126,2026207384524,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.23,36.46,4.018166780471802
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",2,2026210784502,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.9,3.915,1.2725155353546143
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",67,2026209084516,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,2.0,2,0,34.375,64.3,5.286555528640747
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",3,2026210784503,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.7875,3.885,1.0340449810028076
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",201,2026202284446,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,68.825,63.65,4.972667694091797
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",4,2026210784504,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.815,3.99,1.5595331192016602
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",127,2026207384525,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.19,36.98,4.028191566467285
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",5,2026210784505,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.5,2,1,0.765,3.855,1.3357956409454346
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",6,2026210784506,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.81,3.81,1.1162614822387695
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",68,2026209084517,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.825,64.05,4.703666925430298
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",7,2026210784507,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.5,2,1,0.83,3.885,0.9096746444702148
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",202,2026202284447,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,68.05,66.9,4.132783651351929
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",8,2026210784508,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.8375,4.085,1.0846712589263916
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",128,2026207384526,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,15.83,36.3,4.552668333053589
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",9,2026210784509,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.7825,4.02,0.7894248962402344
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",10,2026210784510,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.7675,4.095,1.1800312995910645
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",69,2026209084518,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.325,63.65,4.836676359176636
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",11,2026210784511,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.795,4.005,1.2099452018737793
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",203,2026202284448,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,71.3,64.75,4.683823347091675
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",12,2026210784512,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.7625,3.85,1.3239548206329346
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",129,2026207384527,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.36,35.62,4.258042335510254
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",13,2026210784513,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.835,4.03,0.9418811798095703
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",14,2026210784514,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8425,4.045,0.7417593002319336
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",15,2026210784515,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.865,3.92,0.816051721572876
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",70,2026209084519,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.775,63.6,4.4761059284210205
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",16,2026210784516,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.8225,3.95,0.7169966697692871
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",204,2026202284449,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,67.775,64.55,4.49887490272522
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",17,2026210784517,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.865,3.98,0.9344527721405029
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",130,2026207384528,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,17.34,35.58,5.10155725479126
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",18,2026210784518,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8625,4.025,1.334075927734375
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",71,2026209084520,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,35.175,62.4,3.2037532329559326
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",19,2026210784519,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.5,2,1,0.825,4.095,1.547440767288208
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",20,2026210784520,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,3.0,3,0,0.7975,3.79,1.0981130599975586
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",21,2026210784521,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.8125,3.905,0.9843342304229736
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",205,2026202284450,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,69.425,62.25,5.642960786819458
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",131,2026207384529,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.94,36.84,4.316791534423828
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",22,2026210784522,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.7975,3.97,1.194150447845459
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",72,2026209084521,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,32.625,64.8,4.224333763122559
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",23,2026210784523,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8575,4.0,1.2509403228759766
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",24,2026210784524,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.79,4.0,1.4509186744689941
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",132,2026207384530,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,2,2,16.2,35.88,4.108625650405884
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",25,2026210784525,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,2.5,3,1,0.8375,3.92,0.8796350955963135
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",206,2026202284451,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,68.95,62.35,4.5911805629730225
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",73,2026209084522,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,35.75,65.35,3.92520809173584
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",26,2026210784526,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8075,3.95,0.8983886241912842
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",27,2026210784527,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8425,4.115,1.6562950611114502
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",28,2026210784528,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,2.5,3,1,0.75,4.01,1.0983366966247559
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",133,2026207384531,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.27,35.78,3.910311222076416
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",74,2026209084523,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,34.575,62.9,3.8598461151123047
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",29,2026210784529,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.7675,4.06,0.7694847583770752
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",207,2026202284452,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,67.8,64.85,5.03922963142395
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",30,2026210784530,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.7475,3.98,1.118619680404663
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",31,2026210784531,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.9225,4.06,0.8609166145324707
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",32,2026210784532,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.7825,3.97,0.7645666599273682
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",75,2026209084524,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,35.1,62.75,3.27194881439209
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",33,2026210784533,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.825,3.965,1.3209776878356934
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",134,2026207384532,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.16,36.68,4.807931423187256
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",34,2026210784534,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.5,2,1,0.85,4.095,1.337416648864746
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",208,2026202284453,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,68.475,64.45,5.5965142250061035
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",35,2026210784535,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.86,3.97,1.1022801399230957
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",76,2026209084525,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.4,64.9,3.933830499649048
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",36,2026210784536,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8075,3.94,0.8125503063201904
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",135,2026207384533,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.62,35.28,3.870948553085327
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",37,2026210784537,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.5,2,1,0.885,3.92,1.2282092571258545
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",38,2026210784538,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8425,4.03,1.2682065963745117
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",39,2026210784539,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.8825,3.97,1.0782814025878906
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",77,2026209084526,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.225,61.9,4.268639087677002
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",209,2026202284454,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,66.325,64.2,5.023740530014038
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",40,2026210784540,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8675,3.945,1.2801384925842285
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",136,2026207384534,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,16.53,36.6,4.702912330627441
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",41,2026210784541,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.7875,4.105,1.349916934967041
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",42,2026210784542,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.825,4.065,1.028196096420288
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",78,2026209084527,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.925,64.2,3.6207940578460693
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",43,2026210784543,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.8475,3.945,0.9349005222320557
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",210,2026202284455,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.5,1,1,70.8,64.05,4.829827785491943
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",44,2026210784544,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,2.0,3,2,0.7525,4.18,0.9506440162658691
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",137,2026207384535,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.25,35.74,4.685936689376831
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",45,2026210784545,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.5,2,1,0.83,4.025,1.2123816013336182
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",46,2026210784546,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.7975,4.005,1.351776361465454
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",79,2026209084528,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.575,63.5,4.559621810913086
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",47,2026210784547,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8475,4.16,0.9243535995483398
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",48,2026210784548,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.845,4.025,0.7395946979522705
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",211,2026202284456,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,68.65,62.2,4.604046583175659
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",49,2026210784549,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.8525,4.145,0.7752556800842285
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",138,2026207384536,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,2.0,2,0,16.25,36.5,4.55397367477417
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",50,2026210784550,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.7925,3.995,0.9234962463378906
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",51,2026210784551,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8075,4.105,1.301637887954712
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",80,2026209084529,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.5,1,1,35.85,62.15,5.058658123016357
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",52,2026210784552,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.8925,3.965,1.5382764339447021
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",139,2026207384537,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,16.26,35.4,3.9770493507385254
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",212,2026202284457,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,70.025,65.7,5.2736852169036865
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",53,2026210784553,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8875,4.085,1.0512373447418213
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",54,2026210784554,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.815,3.94,1.0412611961364746
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",55,2026210784555,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.815,3.985,1.4047086238861084
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",81,2026209084530,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.325,64.5,4.401803255081177
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",56,2026210784556,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.8225,4.08,0.8890471458435059
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",140,2026207384538,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.1,36.14,4.421111106872559
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",213,2026202284458,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,70.375,65.15,4.549600124359131
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",57,2026210784557,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.77,4.11,1.4958252906799316
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",58,2026210784558,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.8375,4.205,0.8081345558166504
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",59,2026210784559,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.825,3.85,0.8621902465820312
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",82,2026209084531,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.875,64.3,4.634893178939819
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",60,2026210784560,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.8075,3.98,1.4900193214416504
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",214,2026202284459,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,68.825,62.9,3.7559475898742676
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",141,2026207384539,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.1,35.82,4.696132183074951
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",61,2026210784561,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8525,3.99,1.2182843685150146
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",62,2026210784562,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.9575,3.795,1.3370473384857178
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",83,2026209084532,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,35.075,63.7,3.7340495586395264
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",63,2026210784563,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.7825,3.995,0.8882467746734619
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",215,2026202284460,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,68.65,63.45,4.6028831005096436
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",142,2026207384540,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.43,36.72,3.997974395751953
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",64,2026210784564,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.8775,3.955,1.716559886932373
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",65,2026210784565,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.875,4.075,0.8762264251708984
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",66,2026210784566,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.8425,4.135,0.9939954280853271
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",84,2026209084533,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,35.475,63.65,4.843513488769531
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",67,2026210784567,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8,4.055,1.2770476341247559
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",143,2026207384541,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,16.44,36.36,3.9049978256225586
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",68,2026210784568,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,3.0,3,0,0.8975,4.125,1.494671106338501
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",216,2026202284461,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,2.0,2,0,66.85,64.2,5.242778778076172
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",69,2026210784569,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.785,4.01,1.1278915405273438
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",85,2026209084534,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,34.175,63.75,3.6671464443206787
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",70,2026210784570,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.83,4.065,0.8816189765930176
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",144,2026207384542,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.44,36.1,3.7861809730529785
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",71,2026210784571,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.715,3.98,0.9930648803710938
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",72,2026210784572,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,3.0,3,0,0.795,3.875,1.025381088256836
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",73,2026210784573,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.77,3.765,1.0794017314910889
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",217,2026202284462,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,69.2,62.65,4.984668970108032
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",86,2026209084535,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.6,62.9,4.201932668685913
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",74,2026210784574,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.5,2,1,0.7925,4.0,0.9454212188720703
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",145,2026207384543,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,2.0,2,0,16.72,36.3,4.240724563598633
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",75,2026210784575,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.9175,4.075,1.0654218196868896
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",76,2026210784576,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8175,4.035,0.7916824817657471
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",87,2026209084536,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,32.625,63.05,3.4163808822631836
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",77,2026210784577,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.8175,3.98,1.5428483486175537
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",218,2026202284463,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,68.275,64.5,5.377197027206421
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",78,2026210784578,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.83,4.095,1.0897483825683594
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",146,2026207384544,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,16.98,35.86,4.090240001678467
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",79,2026210784579,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.8375,4.075,1.0999321937561035
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",80,2026210784580,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,2.5,3,1,0.825,3.93,1.0431303977966309
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",88,2026209084537,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,32.625,63.45,4.128581285476685
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",81,2026210784581,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.8475,3.935,1.38258695602417
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",147,2026207384545,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,16.19,36.7,3.9744222164154053
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",219,2026202284464,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,68.725,63.95,4.63556694984436
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",82,2026210784582,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.8375,3.9,1.1770024299621582
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",83,2026210784583,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.8175,3.975,0.9565365314483643
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",84,2026210784584,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.82,3.955,1.1990725994110107
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",89,2026209084538,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.075,65.55,4.576982259750366
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",85,2026210784585,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.83,4.065,1.0249018669128418
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",148,2026207384546,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,16.75,36.52,3.6788763999938965
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",86,2026210784586,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.7625,3.97,1.1181859970092773
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",87,2026210784587,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.5,2,1,0.9175,3.885,0.7424790859222412
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",220,2026202284465,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,70.8,65.0,5.809709310531616
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",88,2026210784588,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8175,4.03,0.8342916965484619
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",89,2026210784589,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.885,3.88,1.1213817596435547
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",90,2026209084539,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,35.15,64.0,4.390027284622192
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",149,2026207384547,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.5,2,1,16.95,35.76,4.385894536972046
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",90,2026210784590,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8475,4.02,1.0386207103729248
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",91,2026210784591,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.835,3.905,1.3786530494689941
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",92,2026210784592,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.7875,4.065,1.4706625938415527
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",91,2026209084540,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.625,62.5,3.937500476837158
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",221,2026202284466,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,2.0,2,0,70.025,62.1,5.405215263366699
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",150,2026207384548,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,16.55,36.54,3.9179179668426514
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",93,2026210784593,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.7575,4.035,0.9987545013427734
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",94,2026210784594,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.7575,4.0,0.9569947719573975
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",95,2026210784595,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.845,4.22,0.7430353164672852
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",96,2026210784596,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.8525,3.95,0.7626760005950928
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",97,2026210784597,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.7375,3.88,0.8266944885253906
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",151,2026207384549,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.22,35.84,3.9972665309906006
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",92,2026209084541,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,33.5,62.15,4.847426652908325
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",98,2026210784598,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,3.0,3,0,0.9475,3.85,1.5929090976715088
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",222,2026202284467,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.5,1,1,67.025,64.15,6.01207971572876
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",99,2026210784599,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,2.5,5,5,0.91,4.22,1.153411865234375
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",100,2026210784600,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,2,2,0.8175,4.065,0.9370386600494385
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",101,2026210784601,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.7575,3.98,0.8274037837982178
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",152,2026207384550,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.76,36.82,4.4418745040893555
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",93,2026209084542,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.6,62.3,4.915203332901001
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",102,2026210784602,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,2.5,3,1,0.865,3.845,1.3383994102478027
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",223,2026202284468,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,71.025,63.55,4.373559951782227
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",103,2026210784603,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.7575,4.075,1.4118213653564453
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",104,2026210784604,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,3.0,3,0,0.835,3.945,1.2426674365997314
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",153,2026207384551,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.39,35.92,4.2030181884765625
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",105,2026210784605,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8375,4.07,0.9318509101867676
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",94,2026209084543,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.975,63.3,3.994617462158203
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",106,2026210784606,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.875,3.775,0.7887725830078125
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",224,2026202284469,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,70.4,64.15,4.413993835449219
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",107,2026210784607,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.865,3.99,1.3544442653656006
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",108,2026210784608,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.85,4.105,1.2107501029968262
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",95,2026209084544,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.275,63.8,3.537304401397705
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",109,2026210784609,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.7975,4.0,1.104651689529419
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",225,2026202284470,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,68.825,65.9,3.920969009399414
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",110,2026210784610,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.9175,3.985,0.7744317054748535
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",154,2026207384552,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,15.85,36.18,5.705901384353638
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",111,2026210784611,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.81,3.925,1.56776762008667
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",112,2026210784612,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.91,3.965,0.9681408405303955
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",113,2026210784613,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.8,4.035,0.7440645694732666
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",114,2026210784614,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.78,4.065,0.8701958656311035
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",155,2026207384553,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,15.77,35.6,4.101132392883301
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",96,2026209084545,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.6,65.5,5.915051698684692
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",226,2026202284471,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,67.2,67.05,4.6502602100372314
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",115,2026210784615,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8375,4.015,1.4554831981658936
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",116,2026210784616,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,2,2,0.895,4.08,1.3685524463653564
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",117,2026210784617,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8125,3.925,1.2685227394104004
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",97,2026209084546,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.675,63.35,3.9014267921447754
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",227,2026202284472,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,70.375,62.6,4.1624369621276855
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",156,2026207384554,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,15.94,36.54,4.686418771743774
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",118,2026210784618,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.8325,3.82,1.3287067413330078
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",119,2026210784619,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.845,3.985,0.8958232402801514
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",120,2026210784620,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,3.0,3,0,0.8825,4.145,1.0970094203948975
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",121,2026210784621,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8575,3.95,1.017653465270996
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",98,2026209084547,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.975,64.05,4.407852649688721
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",122,2026210784622,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.905,4.045,0.8252792358398438
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",228,2026202284473,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,66.5,63.35,5.2418012619018555
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",157,2026207384555,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.37,36.12,5.215923070907593
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",123,2026210784623,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.8275,3.96,1.553149938583374
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",124,2026210784624,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.7525,4.005,1.2685530185699463
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",99,2026209084548,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.95,65.2,3.604794502258301
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",125,2026210784625,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.7975,3.95,1.2420365810394287
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",229,2026202284474,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,72.575,62.65,4.532267093658447
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",126,2026210784626,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.5,2,1,0.8325,4.15,1.1322336196899414
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",158,2026207384556,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,15.67,35.22,5.578989744186401
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",127,2026210784627,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.7975,3.97,1.1792941093444824
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",100,2026209084549,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,33.475,63.8,4.453414440155029
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",128,2026210784628,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.8475,4.025,1.3368237018585205
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",129,2026210784629,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8625,4.19,1.3297038078308105
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",230,2026202284475,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,68.125,62.95,4.485901117324829
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",159,2026207384557,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.52,37.12,4.186320781707764
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",130,2026210784630,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,2.5,3,1,0.8075,3.98,1.4790220260620117
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",131,2026210784631,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.745,3.97,0.9365072250366211
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",101,2026209084550,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.275,63.95,4.762754201889038
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",132,2026210784632,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.7775,4.08,0.7885246276855469
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",133,2026210784633,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.83,4.015,1.257110357284546
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",160,2026207384558,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.5,35.58,3.612067222595215
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",134,2026210784634,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.855,3.935,1.2380046844482422
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",231,2026202284476,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,68.65,63.45,5.193126201629639
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",135,2026210784635,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8225,4.105,0.80623459815979
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",102,2026209084551,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,35.15,64.35,4.35298228263855
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",136,2026210784636,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.8175,4.115,1.192965030670166
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",137,2026210784637,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.775,4.01,1.1425836086273193
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",138,2026210784638,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,3.0,3,0,0.875,4.13,1.1001229286193848
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",232,2026202284477,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,69.5,63.05,4.2915990352630615
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",161,2026207384559,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.5,2,1,16.54,36.22,4.992669105529785
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",103,2026209084552,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,36.125,64.7,3.9536292552948
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",139,2026210784639,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9325,4.015,1.502526044845581
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",140,2026210784640,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8375,3.84,1.7423079013824463
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",141,2026210784641,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.805,4.145,0.8010730743408203
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",162,2026207384560,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,16.88,36.02,4.2946741580963135
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",233,2026202284478,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,68.225,62.75,4.387497425079346
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",142,2026210784642,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8175,4.065,1.2754621505737305
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",104,2026209084553,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.35,63.75,4.26811408996582
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",143,2026210784643,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.5,2,1,0.77,4.095,1.0674676895141602
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",144,2026210784644,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.865,3.85,0.812650203704834
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",145,2026210784645,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.9225,3.91,1.2283167839050293
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",163,2026207384561,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.67,36.22,4.4232118129730225
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",234,2026202284479,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,2.5,3,1,68.5,64.3,4.956018447875977
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",146,2026210784646,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.8125,4.01,1.2874071598052979
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",105,2026209084554,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,35.475,67.3,4.475249290466309
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",147,2026210784647,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8225,3.91,1.1699297428131104
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",148,2026210784648,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.8175,3.87,1.0936386585235596
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",149,2026210784649,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.5,2,1,0.85,3.885,1.2665114402770996
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",235,2026202284480,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,68.225,61.8,3.9496257305145264
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",106,2026209084555,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.25,63.3,3.358180522918701
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",150,2026210784650,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8475,4.0,1.1870810985565186
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",164,2026207384562,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,15.66,36.66,5.666083812713623
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",151,2026210784651,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.845,4.015,1.079329490661621
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",152,2026210784652,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.835,3.945,1.1428437232971191
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",236,2026202284481,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,70.1,66.15,4.052837610244751
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",107,2026209084556,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.0,65.2,4.064737558364868
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",153,2026210784653,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.785,3.955,1.274982213973999
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",165,2026207384563,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.59,35.46,4.378039360046387
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",154,2026210784654,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.85,3.87,1.2032063007354736
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",155,2026210784655,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.815,4.0,1.2151100635528564
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",156,2026210784656,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.7875,3.905,1.193657398223877
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",108,2026209084557,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,35.45,64.6,4.157088994979858
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",157,2026210784657,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.8875,4.115,0.8237957954406738
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",237,2026202284482,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,69.275,63.25,5.529197692871094
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",166,2026207384564,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,15.58,35.52,4.2931177616119385
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",158,2026210784658,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.7525,3.915,0.8270113468170166
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",159,2026210784659,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.845,4.04,1.360264539718628
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",160,2026210784660,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8725,4.21,1.0399370193481445
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",109,2026209084558,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,35.275,63.2,4.770031213760376
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",161,2026210784661,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.83,4.195,1.2239067554473877
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",167,2026207384565,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,15.92,36.48,4.098221302032471
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",238,2026202284483,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,68.1,62.55,4.952946186065674
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",162,2026210784662,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.795,4.0,1.1724562644958496
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",163,2026210784663,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.8625,3.93,0.8897666931152344
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",164,2026210784664,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.5,2,1,0.9575,3.975,0.8676073551177979
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",110,2026209084559,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,35.25,65.35,3.7853217124938965
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",165,2026210784665,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.79,4.01,0.8407533168792725
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",166,2026210784666,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.87,3.985,0.7656955718994141
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",168,2026207384566,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.88,36.16,4.572529077529907
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",167,2026210784667,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.85,3.915,0.7676396369934082
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",239,2026202284484,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,69.825,64.6,5.1705827713012695
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",168,2026210784668,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.825,3.985,1.2211720943450928
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",111,2026209084560,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.525,63.85,4.06271505355835
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",169,2026210784669,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.865,4.035,1.112077236175537
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",170,2026210784670,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.82,4.06,0.895103931427002
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",169,2026207384567,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.13,36.2,4.228957891464233
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",171,2026210784671,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.795,3.895,1.0823166370391846
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",240,2026202284485,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,67.05,64.65,4.350747346878052
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",172,2026210784672,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.8475,3.895,1.3734681606292725
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",173,2026210784673,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8075,4.125,0.7276194095611572
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",112,2026209084561,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,36.325,65.35,4.21450138092041
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",174,2026210784674,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.84,3.855,0.773627758026123
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",175,2026210784675,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,2.5,3,1,0.82,3.81,0.9821643829345703
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",170,2026207384568,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,16.37,36.28,5.189285039901733
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",176,2026210784676,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8825,4.065,1.2586755752563477
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",241,2026202284486,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,67.375,64.45,5.290077209472656
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",177,2026210784677,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9375,4.155,1.1408989429473877
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",113,2026209084562,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.0,65.05,4.702016592025757
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",178,2026210784678,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.77,3.965,0.7571539878845215
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",179,2026210784679,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.895,3.99,0.8970928192138672
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",180,2026210784680,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.765,4.05,1.1897952556610107
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",171,2026207384569,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.67,35.1,4.358928918838501
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",181,2026210784681,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.7625,3.91,1.345179796218872
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",242,2026202284487,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,67.75,65.4,4.3714141845703125
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",114,2026209084563,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.3,63.85,4.502093553543091
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",182,2026210784682,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.78,3.965,1.5757641792297363
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",183,2026210784683,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9375,3.965,0.9296360015869141
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",172,2026207384570,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,15.58,36.0,4.497403621673584
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",184,2026210784684,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.8225,3.98,1.1101696491241455
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",243,2026202284488,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,70.275,62.4,4.122581720352173
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",185,2026210784685,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.815,4.08,0.9243109226226807
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",115,2026209084564,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.925,64.25,4.2407450675964355
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",186,2026210784686,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8125,4.065,0.7654314041137695
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",187,2026210784687,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8125,4.055,0.8194880485534668
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",188,2026210784688,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.85,3.945,1.2330656051635742
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",173,2026207384571,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,15.99,37.24,4.352035999298096
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",189,2026210784689,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.8425,3.97,1.3489940166473389
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",244,2026202284489,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,67.2,65.6,4.5890045166015625
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",116,2026209084565,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.15,63.1,4.74306321144104
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0003|pi0|k1,1,1",190,2026210784690,B,noisy,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.5,2,1,0.805,3.865,1.569164752960205
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",174,2026207384572,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.43,36.14,4.017250299453735
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",117,2026209084566,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.175,62.8,4.192723512649536
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",245,2026202284490,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,67.575,65.95,5.439026594161987
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",0,2026212484551,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.7125,15.795,4.406181573867798
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",175,2026207384573,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.07,36.6,4.53312611579895
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",246,2026202284491,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.5,1,1,69.625,64.8,4.380922794342041
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",118,2026209084567,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.225,64.95,4.832017421722412
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",1,2026212484552,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.8875,15.92,4.785941123962402
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",176,2026207384574,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.93,36.44,4.277267694473267
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",247,2026202284492,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,68.525,65.3,4.314475774765015
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",2,2026212484553,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,2.0,2,0,3.5525,15.815,3.1122963428497314
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",119,2026209084568,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.5,1,1,36.75,64.35,5.256417274475098
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",177,2026207384575,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.68,36.98,4.81855845451355
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",3,2026212484554,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.5875,16.025,4.108078718185425
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",248,2026202284493,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,68.65,64.9,4.625187635421753
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",120,2026209084569,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.575,62.8,4.661882638931274
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",178,2026207384576,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.04,36.12,4.175684213638306
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",4,2026212484555,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.445,16.275,4.624111652374268
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",249,2026202284494,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.5,2,1,69.05,63.65,4.2467591762542725
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",179,2026207384577,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.63,34.96,3.993083953857422
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",121,2026209084570,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.85,67.05,4.540366888046265
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",5,2026212484556,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,3.6625,16.09,4.447566032409668
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",250,2026202284495,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,66.125,65.6,4.404958724975586
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",180,2026207384578,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.16,37.0,3.4481828212738037
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",122,2026209084571,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.8,63.8,4.732497453689575
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",181,2026207384579,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.24,35.9,4.07942533493042
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",6,2026212484557,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.71,15.93,5.446432828903198
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",123,2026209084572,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.65,63.1,3.7516236305236816
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",251,2026202284496,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,68.9,62.75,5.787353515625
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",182,2026207384580,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,2.0,2,0,16.79,36.22,3.423421621322632
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",124,2026209084573,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.125,63.85,4.279340744018555
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",7,2026212484558,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.605,15.995,4.57494330406189
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",252,2026202284497,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,68.325,65.2,5.565575361251831
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",183,2026207384581,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.07,35.36,4.967058181762695
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",8,2026212484559,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.7075,16.11,4.1430487632751465
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",125,2026209084574,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.15,64.3,4.5849738121032715
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.001|pi0|k1,1,1",253,2026202284498,B,noisy,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,69.65,64.25,4.40898323059082
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",184,2026207384582,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.79,35.82,4.286802530288696
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",126,2026209084575,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.025,64.55,4.10559344291687
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",9,2026212484560,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.8175,15.97,4.933339834213257
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",0,2026214184602,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.4,36.18,3.418945789337158
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",185,2026207384583,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,16.71,35.9,4.071810960769653
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",127,2026209084576,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.65,63.0,3.625763416290283
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",10,2026212484561,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.55,15.71,4.239184379577637
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",1,2026214184603,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.33,36.04,3.623217821121216
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",186,2026207384584,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.66,37.02,4.344040393829346
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",11,2026212484562,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.72,15.87,3.8145198822021484
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",128,2026209084577,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.9,62.7,5.145788192749023
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",2,2026214184604,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.51,35.42,4.23968768119812
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",187,2026207384585,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,16.92,36.72,3.765475273132324
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",129,2026209084578,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,35.05,65.25,3.8842673301696777
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",12,2026212484563,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.5575,15.675,4.325091361999512
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",3,2026214184605,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.75,37.34,3.3386895656585693
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",188,2026207384586,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,16.15,36.2,5.171443700790405
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",13,2026212484564,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.57,16.15,3.731128454208374
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",130,2026209084579,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.1,63.0,4.2644453048706055
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",4,2026214184606,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.83,35.94,3.6217281818389893
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",5,2026214184607,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.9,36.46,3.1181955337524414
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",189,2026207384587,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,16.86,34.56,4.4653160572052
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",14,2026212484565,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.6275,15.85,4.359211206436157
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",131,2026209084580,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.8,65.15,4.2617456912994385
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",6,2026214184608,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.79,35.18,4.134258270263672
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",190,2026207384588,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,16.37,36.6,4.033503532409668
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",15,2026212484566,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.7925,15.65,3.963157892227173
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",132,2026209084581,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.65,63.8,4.73628306388855
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",7,2026214184609,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,10.04,35.84,3.5353193283081055
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",16,2026212484567,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.715,15.86,4.0924999713897705
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",191,2026207384589,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,17.19,36.62,5.0016396045684814
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",133,2026209084582,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,35.55,64.1,5.055586576461792
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",8,2026214184610,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.83,35.8,3.7601985931396484
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",17,2026212484568,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.46,15.825,3.4008235931396484
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",134,2026209084583,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.5,1,1,34.825,63.9,3.3426575660705566
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",192,2026207384590,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.18,36.04,5.295134782791138
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",9,2026214184611,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.45,36.24,4.295395851135254
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",18,2026212484569,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.7075,16.185,3.686845541000366
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",135,2026209084584,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,34.225,64.6,4.048008918762207
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",10,2026214184612,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.42,35.48,3.185208320617676
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",193,2026207384591,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,17.06,35.98,4.406171798706055
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",19,2026212484570,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.6275,16.235,4.463042497634888
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",136,2026209084585,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.425,64.3,4.040120601654053
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",11,2026214184613,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,10.09,35.46,3.492988348007202
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",194,2026207384592,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.52,35.2,3.924557685852051
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",20,2026212484571,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.5575,16.255,4.231555700302124
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",12,2026214184614,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.87,35.04,4.245662689208984
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",137,2026209084586,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.375,63.45,4.804536819458008
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",195,2026207384593,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.54,35.96,4.20938515663147
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",21,2026212484572,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,2.0,2,0,3.7825,16.32,4.257784128189087
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",13,2026214184615,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,9.94,36.04,3.8440968990325928
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",196,2026207384594,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.26,36.96,3.9028160572052
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",138,2026209084587,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.275,60.8,4.360462665557861
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",22,2026212484573,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.6625,16.245,4.209537029266357
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",14,2026214184616,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.43,37.38,3.280937910079956
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",139,2026209084588,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.8,65.35,4.037008047103882
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",197,2026207384595,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,16.33,35.42,4.752014636993408
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",23,2026212484574,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.845,15.925,4.011220455169678
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",15,2026214184617,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,9.86,35.38,4.241534233093262
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",140,2026209084589,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,2.0,2,0,35.45,64.75,4.089277982711792
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",198,2026207384596,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.66,35.94,4.207239627838135
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",24,2026212484575,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.65,16.42,4.49098539352417
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",16,2026214184618,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,9.83,35.72,3.69724178314209
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",199,2026207384597,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,2.0,2,0,16.65,35.84,4.0747926235198975
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",141,2026209084590,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,32.95,62.65,4.6688148975372314
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",17,2026214184619,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.8,36.04,3.707538604736328
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",25,2026212484576,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,3.7075,15.39,4.600394248962402
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",200,2026207384598,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,15.78,36.28,3.987090826034546
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",142,2026209084591,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.6,63.95,4.6196393966674805
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",18,2026214184620,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.88,35.9,3.877603054046631
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",26,2026212484577,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.7375,16.085,3.656816005706787
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",143,2026209084592,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,35.3,64.85,3.364915609359741
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",201,2026207384599,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.59,36.78,4.870485544204712
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",19,2026214184621,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,1.0,2,2,10.34,36.68,3.8381683826446533
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",27,2026212484578,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.7225,16.1,4.616959810256958
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",20,2026214184622,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,9.65,35.5,3.4551820755004883
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",144,2026209084593,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.8,63.2,4.588731050491333
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",202,2026207384600,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,16.25,35.58,4.431027412414551
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",28,2026212484579,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.7075,15.795,3.5040721893310547
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",21,2026214184623,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.46,35.98,3.117469072341919
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",145,2026209084594,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.1,63.95,4.025236368179321
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",29,2026212484580,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.53,16.085,3.8275561332702637
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",203,2026207384601,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.41,36.06,4.806880950927734
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",22,2026214184624,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,9.78,36.3,3.579007148742676
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",30,2026212484581,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.68,15.65,3.796591281890869
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",146,2026209084595,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,32.875,64.3,5.011659145355225
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",23,2026214184625,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.95,35.02,3.3259520530700684
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",204,2026207384602,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,2.0,2,0,16.84,36.54,4.670240640640259
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",31,2026212484582,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.5575,16.065,3.5692272186279297
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",24,2026214184626,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.29,34.96,3.6102287769317627
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",147,2026209084596,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.75,62.9,4.764968395233154
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",205,2026207384603,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.56,36.94,4.445780038833618
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",32,2026212484583,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.7775,15.77,3.8245396614074707
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",25,2026214184627,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.81,35.36,3.442023277282715
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",206,2026207384604,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,2,2,15.94,35.86,4.04638934135437
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",148,2026209084597,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.325,62.05,5.71627402305603
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",33,2026212484584,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.59,16.065,3.410576105117798
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",26,2026214184628,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,2.0,2,0,9.31,35.88,4.36690878868103
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",207,2026207384605,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.06,36.32,4.068821668624878
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",34,2026212484585,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.59,15.825,4.292094469070435
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",149,2026209084598,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.9,65.2,5.810276508331299
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",27,2026214184629,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.92,36.58,4.2257750034332275
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",208,2026207384606,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.32,35.9,4.918281555175781
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",35,2026212484586,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.5825,15.565,4.58798885345459
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",28,2026214184630,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.74,35.38,3.9139437675476074
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",150,2026209084599,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.9,64.75,4.876115322113037
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",36,2026212484587,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.7525,16.63,3.0605006217956543
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",209,2026207384607,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,2.0,2,0,16.84,35.82,4.694602966308594
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",29,2026214184631,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.76,36.62,3.784008026123047
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",151,2026209084600,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.875,65.8,3.490046739578247
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",37,2026212484588,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,2.0,2,0,3.5475,16.19,3.7345244884490967
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",30,2026214184632,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.85,36.96,3.176496744155884
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",210,2026207384608,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,16.24,37.1,4.772385835647583
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",152,2026209084601,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.2,62.8,4.911761045455933
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",38,2026212484589,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.6775,16.01,3.679860830307007
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",31,2026214184633,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.44,36.5,3.3810064792633057
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",211,2026207384609,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,16.24,36.22,5.4231719970703125
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",39,2026212484590,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.64,15.905,3.8800430297851562
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",32,2026214184634,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.52,37.36,3.0682079792022705
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",153,2026209084602,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,35.275,63.85,4.656385183334351
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",40,2026212484591,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.5825,16.22,3.2684597969055176
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",154,2026209084603,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,32.3,63.9,3.837226152420044
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",33,2026214184635,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.67,35.38,4.313791751861572
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",212,2026207384610,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,16.72,36.44,5.315851926803589
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",41,2026212484592,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.875,16.195,4.441287994384766
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",34,2026214184636,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.9,36.94,3.853698968887329
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",155,2026209084604,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.375,64.8,4.436418771743774
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",213,2026207384611,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.71,35.82,3.7938344478607178
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",35,2026214184637,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.3,35.54,3.392704486846924
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",42,2026212484593,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.805,15.62,4.798092365264893
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",214,2026207384612,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.42,36.18,4.041508436203003
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",156,2026209084605,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,35.225,65.1,4.7498228549957275
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",36,2026214184638,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.63,36.66,3.1703078746795654
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",215,2026207384613,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.48,36.02,4.076815843582153
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",43,2026212484594,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,3.5,4,1,3.6775,15.92,4.409700870513916
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",157,2026209084606,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.075,65.45,4.09195876121521
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",37,2026214184639,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.22,34.88,3.5074760913848877
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",216,2026207384614,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.93,35.88,3.9554271697998047
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",44,2026212484595,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.56,15.695,4.115260362625122
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",158,2026209084607,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.225,64.3,4.055779457092285
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",38,2026214184640,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.66,35.78,3.3573226928710938
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",45,2026212484596,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.67,16.08,3.2817881107330322
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",39,2026214184641,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.54,35.3,3.1739375591278076
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",217,2026207384615,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,16.24,35.92,5.11834454536438
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",159,2026209084608,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.05,65.3,4.7729668617248535
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",46,2026212484597,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.7825,15.59,4.20560097694397
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",40,2026214184642,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,10.32,36.24,3.921635389328003
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",218,2026207384616,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.69,35.62,3.744994640350342
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",160,2026209084609,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.9,63.85,4.5246803760528564
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",47,2026212484598,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.6325,16.145,3.463578462600708
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",41,2026214184643,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.44,37.46,3.5063254833221436
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",219,2026207384617,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,15.57,36.78,4.6866607666015625
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",161,2026209084610,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,32.575,62.05,4.237947940826416
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",48,2026212484599,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,3.8675,16.31,3.7449228763580322
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",42,2026214184644,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,9.43,35.74,3.665081262588501
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",43,2026214184645,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.64,36.32,3.187431573867798
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",49,2026212484600,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.6525,16.065,3.5586626529693604
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",162,2026209084611,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.25,64.15,4.730101108551025
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",220,2026207384618,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.47,34.74,5.609739780426025
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",44,2026214184646,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.95,36.16,3.682631254196167
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",50,2026212484601,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.5875,15.915,4.6557769775390625
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",163,2026209084612,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,34.375,63.05,4.23945164680481
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",221,2026207384619,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,15.88,34.34,4.326059341430664
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",45,2026214184647,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.84,36.32,3.260256052017212
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",164,2026209084613,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.5,1,1,36.35,62.3,3.6361758708953857
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",222,2026207384620,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.4,36.36,3.7715439796447754
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",51,2026212484602,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,3.825,15.97,4.60889458656311
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",46,2026214184648,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.81,36.06,4.205638885498047
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",165,2026209084614,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.05,65.4,4.2873547077178955
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",52,2026212484603,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.725,15.875,3.5787770748138428
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",223,2026207384621,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,16.03,36.1,4.82429313659668
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",47,2026214184649,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.68,35.9,3.221085786819458
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",53,2026212484604,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.4825,15.755,3.637812376022339
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",166,2026209084615,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,32.65,60.8,4.483272075653076
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",224,2026207384622,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.7,36.26,4.201101779937744
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",48,2026214184650,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.85,35.54,4.095208644866943
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",54,2026212484605,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.54,15.92,3.8003087043762207
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",167,2026209084616,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.9,63.6,3.8181731700897217
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",49,2026214184651,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.84,36.74,3.1140761375427246
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",225,2026207384623,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.25,36.3,4.250169038772583
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",55,2026212484606,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,2.0,2,0,3.6225,16.04,4.482826232910156
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",50,2026214184652,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.01,35.82,3.444836139678955
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",168,2026209084617,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.775,63.7,4.5017218589782715
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",226,2026207384624,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.7,35.12,4.3710010051727295
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",51,2026214184653,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.5,36.4,3.016249895095825
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",56,2026212484607,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,3.735,15.84,3.9757745265960693
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",227,2026207384625,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,17.02,35.54,3.7105979919433594
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",169,2026209084618,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.35,63.15,4.900883913040161
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",52,2026214184654,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.88,37.22,3.3826346397399902
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",57,2026212484608,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.7875,15.83,4.3296027183532715
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",170,2026209084619,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.5,65.95,3.643127202987671
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",53,2026214184655,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.38,36.26,3.173952341079712
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",228,2026207384626,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.07,36.56,4.423013210296631
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",58,2026212484609,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,3.0,3,0,3.69,16.23,3.81038761138916
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",54,2026214184656,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.39,37.4,3.5682826042175293
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",229,2026207384627,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.8,35.04,4.407060384750366
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",171,2026209084620,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.0,64.25,5.0774009227752686
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",55,2026214184657,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.0,35.34,3.1893222332000732
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",59,2026212484610,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.62,16.245,4.54574728012085
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",230,2026207384628,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.72,36.0,3.7134644985198975
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",172,2026209084621,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.0,63.2,4.907217502593994
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",60,2026212484611,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,3.5975,15.83,4.085449457168579
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",56,2026214184658,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.57,35.82,4.684391736984253
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",231,2026207384629,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.68,36.02,4.0027570724487305
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",173,2026209084622,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,35.425,64.35,3.4918086528778076
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",61,2026212484612,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,3.7025,16.295,3.7059669494628906
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",57,2026214184659,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.68,36.14,3.704348087310791
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",232,2026207384630,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,2.0,2,0,16.64,35.42,4.30725359916687
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",174,2026209084623,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.75,64.35,4.550997972488403
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",58,2026214184660,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.21,35.72,3.350491523742676
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",62,2026212484613,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,3.5575,15.88,4.433753490447998
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",233,2026207384631,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,16.3,36.08,4.166893243789673
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",175,2026209084624,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.55,65.05,3.8694074153900146
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",59,2026214184661,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.91,35.84,4.049788951873779
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",63,2026212484614,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.78,16.115,4.330070972442627
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",234,2026207384632,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,16.63,35.46,4.254716634750366
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",60,2026214184662,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.04,36.44,3.998464584350586
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",176,2026209084625,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.675,64.25,5.10478663444519
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",64,2026212484615,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.61,15.93,3.2346956729888916
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",235,2026207384633,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.33,35.68,3.6318464279174805
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",61,2026214184663,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.76,35.34,3.562195062637329
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",177,2026209084626,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.675,65.85,4.417089462280273
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",65,2026212484616,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.7475,16.035,4.305727481842041
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",62,2026214184664,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.19,35.98,3.0846798419952393
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",236,2026207384634,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.4,35.34,4.899457216262817
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",178,2026209084627,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.5,1,1,35.625,63.6,4.003586530685425
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",66,2026212484617,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.7025,16.16,4.075581312179565
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",237,2026207384635,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.44,35.4,4.240826845169067
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",63,2026214184665,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.35,34.56,4.673141002655029
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",67,2026212484618,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.6175,15.81,3.7871787548065186
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",179,2026209084628,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.45,62.05,5.101875066757202
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",238,2026207384636,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,16.08,36.12,3.7146596908569336
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",64,2026214184666,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.06,35.2,3.797501802444458
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",68,2026212484619,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.66,16.165,3.922945499420166
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",239,2026207384637,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.14,35.18,3.2831568717956543
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",65,2026214184667,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.81,36.14,4.20845365524292
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",180,2026209084629,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,34.25,63.35,6.301182270050049
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",69,2026212484620,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.6025,15.91,3.804335594177246
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",240,2026207384638,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,15.54,36.06,4.186627626419067
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",70,2026212484621,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.65,15.93,2.976738691329956
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",66,2026214184668,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.69,34.96,4.169515371322632
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",181,2026209084630,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.875,64.55,4.242505788803101
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",71,2026212484622,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.5275,16.17,3.3847997188568115
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",241,2026207384639,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.78,36.22,4.461382865905762
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",67,2026214184669,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.3,35.86,3.6377482414245605
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",182,2026209084631,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,35.2,65.85,4.441410303115845
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",72,2026212484623,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,3.58,15.735,4.050777912139893
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",68,2026214184670,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.76,35.16,3.432295560836792
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",242,2026207384640,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.04,36.2,4.64017915725708
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",183,2026209084632,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,35.225,63.85,4.602949380874634
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",69,2026214184671,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.93,36.1,3.5842251777648926
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",73,2026212484624,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.7725,15.965,3.8302478790283203
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",70,2026214184672,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.01,35.04,2.8352537155151367
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",243,2026207384641,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,17.01,35.88,5.747253894805908
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",184,2026209084633,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.6,65.45,4.262748956680298
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",74,2026212484625,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.685,16.18,3.617112636566162
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",71,2026214184673,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.15,36.74,3.5624775886535645
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",244,2026207384642,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.31,35.54,4.090006589889526
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",185,2026209084634,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.625,64.45,4.51129937171936
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",75,2026212484626,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,3.6675,15.69,4.479972839355469
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",72,2026214184674,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,9.76,35.54,3.0804266929626465
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",186,2026209084635,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.65,63.25,3.7461960315704346
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",245,2026207384643,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,15.82,36.76,4.550418376922607
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",76,2026212484627,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.745,15.955,4.267544984817505
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",73,2026214184675,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,1.5,2,1,9.71,35.92,3.1425859928131104
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",187,2026209084636,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.425,61.45,3.843475341796875
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",74,2026214184676,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.0,36.64,3.498063325881958
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",246,2026207384644,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,16.41,36.18,4.550984621047974
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",77,2026212484628,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.6075,15.87,3.7955141067504883
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",75,2026214184677,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.63,36.08,2.818969964981079
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",188,2026209084637,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.525,62.65,4.16253399848938
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",247,2026207384645,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.36,36.48,4.667326211929321
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",78,2026212484629,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.7375,16.31,4.59748387336731
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",76,2026214184678,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.84,35.36,2.8871712684631348
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",189,2026209084638,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.15,63.75,3.811958074569702
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",79,2026212484630,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.685,15.87,3.870091676712036
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",248,2026207384646,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,15.84,36.42,3.9964215755462646
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",77,2026214184679,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.31,35.68,4.394906997680664
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",190,2026209084639,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.575,65.2,4.2856645584106445
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",249,2026207384647,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.45,35.14,3.9965720176696777
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",80,2026212484631,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.625,15.955,4.6964216232299805
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",78,2026214184680,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.54,35.74,3.4921631813049316
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",191,2026209084640,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.875,64.4,5.211487054824829
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",79,2026214184681,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.99,36.18,3.3468470573425293
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",250,2026207384648,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,2.0,2,0,16.55,36.42,4.454878091812134
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",81,2026212484632,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.4825,16.295,3.982898235321045
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",80,2026214184682,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.58,35.66,3.664029121398926
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",82,2026212484633,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,3.74,15.89,3.99690580368042
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",251,2026207384649,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.2,35.3,4.438544034957886
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",192,2026209084641,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.075,62.95,5.0527448654174805
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",83,2026212484634,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.735,15.905,3.0179922580718994
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",81,2026214184683,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.44,37.52,4.301907777786255
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",252,2026207384650,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.58,35.52,4.188174724578857
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",193,2026209084642,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.425,62.9,4.3273961544036865
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",82,2026214184684,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.68,37.48,3.235530138015747
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",84,2026212484635,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.575,16.035,3.8423521518707275
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",253,2026207384651,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,15.87,35.48,3.9527292251586914
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",194,2026209084643,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.85,65.15,4.057501316070557
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",83,2026214184685,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,9.7,35.72,3.6247456073760986
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",85,2026212484636,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.74,15.94,4.292088985443115
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",254,2026207384652,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,16.45,36.46,4.0785908699035645
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",195,2026209084644,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,34.15,64.0,4.529361963272095
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",84,2026214184686,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.92,36.22,3.049532413482666
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",86,2026212484637,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.835,15.725,4.272939682006836
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",196,2026209084645,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.375,64.2,3.4627792835235596
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",85,2026214184687,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.48,36.14,3.617065668106079
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",255,2026207384653,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.78,35.56,5.248719930648804
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",87,2026212484638,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,3.6075,16.015,3.631350517272949
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",86,2026214184688,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.09,36.46,3.3149383068084717
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",197,2026209084646,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.475,63.8,4.1472063064575195
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",256,2026207384654,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,15.98,34.84,4.287414789199829
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",88,2026212484639,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.735,15.67,3.7392239570617676
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",87,2026214184689,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.06,35.54,3.171441078186035
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",198,2026209084647,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,36.475,61.45,4.863385200500488
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",88,2026214184690,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,2.0,2,0,9.84,35.74,3.3013501167297363
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",89,2026212484640,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.6025,15.945,4.168859958648682
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",257,2026207384655,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.58,36.46,5.083038806915283
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",89,2026214184691,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.52,36.02,3.471531391143799
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",199,2026209084648,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,2.0,2,0,32.35,62.85,4.763017416000366
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",258,2026207384656,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,16.13,37.72,3.7512245178222656
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",90,2026212484641,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.64,16.055,4.535459041595459
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",90,2026214184692,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.44,35.08,3.457167863845825
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",259,2026207384657,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,16.57,36.34,3.9179344177246094
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",200,2026209084649,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.075,65.95,4.334740400314331
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",91,2026212484642,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,3.54,15.975,4.3361735343933105
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",91,2026214184693,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.48,35.76,3.233983039855957
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",260,2026207384658,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,15.66,35.12,3.923457622528076
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",201,2026209084650,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.425,62.9,4.278763055801392
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",92,2026212484643,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.6875,15.98,4.097020387649536
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",92,2026214184694,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.29,35.94,3.074383020401001
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",202,2026209084651,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.825,63.5,2.817558765411377
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",261,2026207384659,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,15.8,35.66,3.3570034503936768
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",93,2026214184695,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.59,36.14,2.598703145980835
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",93,2026212484644,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.635,16.155,3.2318506240844727
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",203,2026209084652,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.45,63.8,2.9278931617736816
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",94,2026214184696,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.78,35.3,2.52774715423584
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",262,2026207384660,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,16.51,36.76,3.4681055545806885
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",94,2026212484645,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.555,16.345,3.0086829662323
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",95,2026214184697,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.97,36.84,2.626804828643799
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",204,2026209084653,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.5,1,1,34.8,64.0,2.996528387069702
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",263,2026207384661,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.65,36.7,3.2948291301727295
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",95,2026212484646,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.68,15.97,3.139709949493408
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",96,2026214184698,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,9.52,35.66,2.844392776489258
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",205,2026209084654,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.75,61.15,3.232553243637085
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",264,2026207384662,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,15.52,36.38,3.190162420272827
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",96,2026212484647,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.6225,16.225,2.950232982635498
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",97,2026214184699,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.18,35.22,2.762502431869507
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",206,2026209084655,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.475,64.1,3.6291415691375732
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",97,2026212484648,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.66,15.95,3.0124545097351074
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",265,2026207384663,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,16.63,35.88,3.360360860824585
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",98,2026214184700,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.65,36.3,2.635080099105835
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",98,2026212484649,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.6175,15.85,3.041194438934326
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",207,2026209084656,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,34.85,62.75,3.637314558029175
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",266,2026207384664,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,15.94,35.76,3.2709202766418457
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",99,2026214184701,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.93,35.94,2.7635605335235596
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",99,2026212484650,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.6825,16.195,2.9557671546936035
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",100,2026214184702,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.67,37.34,2.683220148086548
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",267,2026207384665,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.0,35.68,3.2454490661621094
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",208,2026209084657,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.975,65.3,3.5524003505706787
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",100,2026212484651,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,3.77,15.745,2.9978296756744385
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",101,2026214184703,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.84,35.6,2.814530849456787
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",209,2026209084658,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.075,62.7,3.1989049911499023
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",268,2026207384666,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.13,36.9,3.335536241531372
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",101,2026212484652,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.745,15.67,2.992072105407715
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",102,2026214184704,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.43,36.22,2.9474711418151855
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",269,2026207384667,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,2.5,3,1,15.88,35.22,3.217862367630005
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",210,2026209084659,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.275,64.65,3.6711065769195557
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",103,2026214184705,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.39,35.8,2.759260416030884
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",102,2026212484653,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.54,15.805,2.847330093383789
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",270,2026207384668,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,16.67,35.9,3.3968639373779297
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",211,2026209084660,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.5,62.5,3.512671709060669
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",104,2026214184706,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.39,35.38,2.6296141147613525
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",103,2026212484654,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.6525,15.905,2.9391603469848633
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",271,2026207384669,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,16.21,36.88,3.293464183807373
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",105,2026214184707,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.58,35.3,2.6279895305633545
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",212,2026209084661,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,33.7,66.25,3.2234344482421875
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",104,2026212484655,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.7025,15.94,3.2453348636627197
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",106,2026214184708,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.09,35.28,2.8299849033355713
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",272,2026207384670,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,16.78,36.68,3.2643775939941406
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",213,2026209084662,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,35.525,64.7,2.8458917140960693
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",105,2026212484656,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.5475,15.86,2.8295435905456543
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",107,2026214184709,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,1.0,2,2,10.15,36.04,2.715550661087036
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",214,2026209084663,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,35.025,61.95,2.7676100730895996
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",273,2026207384671,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.22,36.84,3.1279752254486084
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",106,2026212484657,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.67,15.72,2.8507702350616455
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",108,2026214184710,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.01,34.86,2.7808687686920166
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",215,2026209084664,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,33.225,65.05,2.8472466468811035
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",274,2026207384672,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.11,35.38,3.180161476135254
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",107,2026212484658,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.53,16.1,3.1455512046813965
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",109,2026214184711,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.36,35.7,2.602445125579834
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",216,2026209084665,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.925,63.55,2.7664685249328613
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",108,2026212484659,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,2.0,2,0,3.6925,16.195,2.8370296955108643
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",275,2026207384673,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,17.02,35.6,3.2931110858917236
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",110,2026214184712,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.17,35.72,2.832200050354004
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",217,2026209084666,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.625,63.2,2.718850612640381
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",109,2026212484660,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.62,15.975,2.91637921333313
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",276,2026207384674,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.11,36.48,3.120509147644043
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",111,2026214184713,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.09,35.88,2.8802855014801025
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",218,2026209084667,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.075,63.8,2.896376848220825
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",110,2026212484661,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.5925,15.905,3.6793792247772217
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",277,2026207384675,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.22,34.58,4.024904727935791
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",112,2026214184714,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.7,36.38,3.2555980682373047
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",219,2026209084668,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.5,1,1,35.375,64.25,3.9876744747161865
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",111,2026212484662,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.59,15.955,3.817549467086792
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0005|pi0|k1,1,1",278,2026207384676,B,noisy,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,16.12,36.04,4.115000247955322
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",113,2026214184715,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.55,36.8,4.130389928817749
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",220,2026209084669,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.0,64.2,4.0444655418396
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",114,2026214184716,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.39,35.84,3.3556630611419678
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",112,2026212484663,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,2,2,3.77,15.725,4.512193918228149
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",221,2026209084670,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.825,64.1,4.54323410987854
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",115,2026214184717,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.07,35.7,3.2212769985198975
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",113,2026212484664,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.7,15.765,3.841383934020996
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",222,2026209084671,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,36.4,64.9,4.244830846786499
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",116,2026214184718,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.44,34.44,4.148564100265503
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",0,2026215884653,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.025,65.2,5.515338897705078
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",114,2026212484665,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.5825,15.995,3.705880880355835
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",1,2026215884654,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.075,65.0,3.203552007675171
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",115,2026212484666,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.5725,16.21,3.6615614891052246
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",117,2026214184719,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.81,34.7,4.330551862716675
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",223,2026209084672,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.575,63.8,5.189305067062378
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",2,2026215884655,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.175,64.35,3.5868611335754395
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",118,2026214184720,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.08,36.74,3.770191192626953
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",116,2026212484667,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,3.69,16.105,4.415511131286621
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",224,2026209084673,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.25,62.15,4.701237440109253
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",3,2026215884656,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.0,62.75,4.05330753326416
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",119,2026214184721,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.44,35.54,3.8372433185577393
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",117,2026212484668,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,3.6375,15.975,3.639638662338257
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",225,2026209084674,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.925,64.55,4.9201719760894775
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",4,2026215884657,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.325,65.35,4.227871417999268
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",120,2026214184722,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,10.13,35.88,3.151928186416626
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",118,2026212484669,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.47,15.935,4.382425308227539
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",121,2026214184723,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.27,36.02,2.7908291816711426
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",226,2026209084675,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.4,64.1,4.378046035766602
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",5,2026215884658,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.275,63.85,4.084877967834473
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",119,2026212484670,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.67,16.115,4.499112367630005
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",122,2026214184724,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.17,36.96,3.114088535308838
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",227,2026209084676,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.325,62.95,4.014817714691162
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",6,2026215884659,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.725,65.25,3.5917880535125732
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",123,2026214184725,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.79,36.38,3.6718897819519043
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",120,2026212484671,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,3.7425,16.035,3.8190064430236816
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",228,2026209084677,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,34.45,64.05,3.8875646591186523
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",7,2026215884660,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.65,64.05,4.190685033798218
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",124,2026214184726,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,9.44,36.1,3.1790599822998047
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",121,2026212484672,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,2.0,2,0,3.6375,15.92,5.531350612640381
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",229,2026209084678,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.45,64.7,4.218493223190308
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",8,2026215884661,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.775,63.2,3.648341655731201
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",125,2026214184727,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.81,36.68,3.546139717102051
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",122,2026212484673,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.74,16.12,3.9157021045684814
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",126,2026214184728,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.29,35.68,3.1266591548919678
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",9,2026215884662,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.6,62.9,4.355693817138672
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",230,2026209084679,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,32.8,64.75,5.71872878074646
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",127,2026214184729,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,9.79,36.1,3.379181146621704
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",123,2026212484674,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.4425,16.23,4.1446027755737305
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",231,2026209084680,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.85,63.7,3.6850433349609375
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",10,2026215884663,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.9,64.65,5.15532660484314
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",128,2026214184730,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.52,36.72,3.5575954914093018
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",124,2026212484675,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.6525,16.36,4.839995861053467
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",232,2026209084681,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.2,64.15,3.998601198196411
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",129,2026214184731,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.77,35.9,2.988237142562866
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",11,2026215884664,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.925,64.05,4.552598476409912
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",125,2026212484676,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.5925,15.92,4.031581401824951
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",233,2026209084682,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.9,64.6,4.039582967758179
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",130,2026214184732,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.51,36.16,4.461678743362427
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",12,2026215884665,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,20.55,67.15,4.7320709228515625
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",126,2026212484677,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.715,16.06,3.479008197784424
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",234,2026209084683,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,36.65,61.3,3.6565616130828857
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",131,2026214184733,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,10.19,35.6,3.7786855697631836
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",13,2026215884666,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,21.3,61.65,3.4697186946868896
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",127,2026212484678,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.65,16.14,3.957404851913452
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",235,2026209084684,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.325,64.95,3.7939372062683105
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",132,2026214184734,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.53,36.18,3.5103366374969482
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",14,2026215884667,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,22.225,63.25,4.190941572189331
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",128,2026212484679,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,3.7725,16.29,3.5231308937072754
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",133,2026214184735,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,9.61,36.14,3.466996669769287
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",236,2026209084685,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,1.5,2,1,34.125,65.1,4.621725559234619
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",15,2026215884668,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,21.175,62.4,4.583049774169922
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",129,2026212484680,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,2.0,2,0,3.6925,16.255,3.768923282623291
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",134,2026214184736,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.97,35.82,3.547454833984375
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",237,2026209084686,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.45,62.6,4.466177940368652
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",16,2026215884669,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.375,63.9,3.674025774002075
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",130,2026212484681,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.585,15.955,4.568054914474487
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",135,2026214184737,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.66,37.52,3.3453752994537354
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",238,2026209084687,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.275,63.4,4.468605995178223
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",17,2026215884670,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.475,61.85,3.9333279132843018
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",136,2026214184738,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.43,36.74,3.384855270385742
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",131,2026212484682,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.5675,15.975,3.747144937515259
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",18,2026215884671,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.15,60.1,3.5161123275756836
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",239,2026209084688,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,35.25,65.85,4.366466045379639
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",137,2026214184739,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,9.55,36.46,3.6715965270996094
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",132,2026212484683,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,3.6625,15.75,3.831796169281006
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",138,2026214184740,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.56,35.64,3.523183822631836
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",240,2026209084689,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.675,63.95,4.4060375690460205
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",19,2026215884672,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.25,60.65,5.082143306732178
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",133,2026212484684,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.5625,15.815,3.9991350173950195
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",139,2026214184741,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.97,36.16,3.26003360748291
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",241,2026209084690,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.3,60.8,4.412301301956177
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",134,2026212484685,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.7875,15.73,4.337818145751953
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",20,2026215884673,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.0,65.6,5.486954927444458
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",140,2026214184742,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.97,36.96,3.2928831577301025
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",21,2026215884674,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.225,64.75,3.830470085144043
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",141,2026214184743,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.54,36.72,3.5605640411376953
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",135,2026212484686,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,3.5825,15.92,4.828078031539917
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",242,2026209084691,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.55,64.15,5.143961429595947
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",142,2026214184744,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.8,36.18,3.2786612510681152
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",22,2026215884675,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.825,65.15,3.956922769546509
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",136,2026212484687,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.7475,16.06,4.1834797859191895
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",243,2026209084692,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,35.4,66.0,5.440778732299805
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",143,2026214184745,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.88,36.06,2.9654786586761475
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",23,2026215884676,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.725,63.25,3.967374801635742
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",137,2026212484688,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,3.71,15.75,3.932379722595215
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",144,2026214184746,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.96,36.04,3.535162925720215
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",244,2026209084693,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,35.375,65.15,4.341458082199097
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",138,2026212484689,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,3.66,15.83,3.419059991836548
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",24,2026215884677,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.275,61.95,4.149774074554443
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",245,2026209084694,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,35.35,60.55,4.18900203704834
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",145,2026214184747,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.56,36.64,4.456624746322632
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",139,2026212484690,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.445,16.28,3.537996530532837
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",25,2026215884678,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.2,66.0,3.728364944458008
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",146,2026214184748,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.87,37.14,3.727855920791626
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",140,2026212484691,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.68,16.285,3.7531211376190186
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",246,2026209084695,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,35.25,64.3,5.228648900985718
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",26,2026215884679,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.275,64.7,3.9965851306915283
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",147,2026214184749,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.37,36.44,2.578138828277588
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",141,2026212484692,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,3.8225,16.0,4.221119165420532
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",148,2026214184750,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.97,36.56,3.146468162536621
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",27,2026215884680,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.925,62.3,4.0556480884552
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",247,2026209084696,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.5,1,1,33.35,64.65,4.732893943786621
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",149,2026214184751,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.62,36.04,2.6320407390594482
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",142,2026212484693,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.645,15.815,4.80754828453064
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",28,2026215884681,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.05,60.7,4.306129217147827
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",248,2026209084697,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.875,62.75,4.197895288467407
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",150,2026214184752,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.67,36.88,3.103142499923706
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",143,2026212484694,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.72,16.025,3.759331703186035
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",249,2026209084698,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.375,63.1,4.176219463348389
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",29,2026215884682,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.225,65.4,4.335232734680176
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",151,2026214184753,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.95,36.94,3.5710885524749756
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",144,2026212484695,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.5175,15.945,3.8618783950805664
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",152,2026214184754,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,9.46,36.1,3.4648072719573975
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",30,2026215884683,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.625,62.9,4.1494786739349365
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",250,2026209084699,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.075,66.75,4.365806579589844
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",145,2026212484696,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.6175,16.455,3.720489501953125
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",153,2026214184755,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,9.76,36.04,2.9765982627868652
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",31,2026215884684,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.625,64.25,4.039183139801025
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",251,2026209084700,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.475,64.55,4.621324777603149
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",146,2026212484697,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,3.7975,15.995,3.503171443939209
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",154,2026214184756,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.4,36.24,3.4272634983062744
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",32,2026215884685,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.45,64.45,4.121101379394531
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",155,2026214184757,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.74,35.58,3.3585495948791504
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",147,2026212484698,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,3.695,15.87,3.626932382583618
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",252,2026209084701,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,37.4,61.95,5.917625904083252
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",33,2026215884686,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.25,63.8,4.358677625656128
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",156,2026214184758,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.83,36.58,3.781820774078369
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",148,2026212484699,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.66,15.805,4.173906326293945
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",253,2026209084702,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,35.65,63.75,5.2330482006073
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",34,2026215884687,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.025,63.05,3.7641892433166504
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",157,2026214184759,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.76,35.88,3.6572554111480713
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",149,2026212484700,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.6775,15.805,3.283313512802124
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",158,2026214184760,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.27,35.3,3.1681528091430664
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",35,2026215884688,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.825,64.4,4.0442774295806885
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",254,2026209084703,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.875,61.25,5.332985877990723
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",150,2026212484701,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.665,16.27,4.466116189956665
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",159,2026214184761,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.83,35.76,2.8352808952331543
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",36,2026215884689,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.075,63.8,4.174179315567017
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",151,2026212484702,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.665,15.77,4.018906354904175
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",160,2026214184762,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,2.0,2,0,9.69,35.66,2.7084689140319824
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",255,2026209084704,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,35.2,65.05,5.326808929443359
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",161,2026214184763,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.95,36.94,3.1357579231262207
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",37,2026215884690,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,22.15,64.6,5.04166579246521
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",256,2026209084705,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,35.8,64.6,3.2112393379211426
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",152,2026212484703,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.5425,16.14,4.296072721481323
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",162,2026214184764,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.64,36.38,3.910376787185669
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",153,2026212484704,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.735,15.695,3.457479476928711
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",38,2026215884691,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,2.0,2,0,19.425,63.15,4.3889641761779785
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",257,2026209084706,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.675,65.55,5.11591100692749
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",163,2026214184765,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.11,35.92,4.272928237915039
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",154,2026212484705,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.7,16.14,3.7043280601501465
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",39,2026215884692,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.4,65.05,3.2841358184814453
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",258,2026209084707,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.425,64.35,4.0474629402160645
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",40,2026215884693,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.85,64.6,3.351607084274292
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",164,2026214184766,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.61,36.28,3.485790729522705
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",155,2026212484706,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.52,16.015,4.034559726715088
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",259,2026209084708,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.625,63.9,3.9055020809173584
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",41,2026215884694,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.525,62.65,2.9895880222320557
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",165,2026214184767,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.67,35.46,3.214890718460083
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",156,2026212484707,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.6775,16.055,4.5423338413238525
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",260,2026209084709,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.5,1,1,35.025,65.4,4.102984666824341
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",42,2026215884695,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.775,64.2,3.6867828369140625
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",166,2026214184768,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.1,36.52,4.195622444152832
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",157,2026212484708,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,3.5525,15.975,3.329674243927002
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",261,2026209084710,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.45,64.45,3.518031120300293
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",43,2026215884696,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.9,64.55,3.974515199661255
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",167,2026214184769,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.75,36.66,4.124196290969849
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",158,2026212484709,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.67,15.93,4.0706541538238525
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",262,2026209084711,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,35.475,64.4,3.656954050064087
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",44,2026215884697,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.15,63.55,3.15757155418396
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",168,2026214184770,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.9,36.64,3.7902231216430664
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",159,2026212484710,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.6125,16.195,3.9496371746063232
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",263,2026209084712,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.4,65.5,3.9386332035064697
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",45,2026215884698,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.3,64.3,4.449378967285156
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",169,2026214184771,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.91,36.52,3.694472551345825
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",160,2026212484711,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.625,16.2,4.303050518035889
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",264,2026209084713,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,32.825,63.55,3.817255735397339
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",46,2026215884699,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.85,64.5,3.7490453720092773
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",170,2026214184772,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.3,36.66,3.6065938472747803
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",265,2026209084714,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.075,64.05,3.6175408363342285
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",161,2026212484712,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.73,16.21,4.708431243896484
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",171,2026214184773,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.98,35.3,3.1853742599487305
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",47,2026215884700,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,22.0,64.2,4.20081353187561
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",172,2026214184774,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.6,35.76,3.253369092941284
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",266,2026209084715,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.0,66.05,4.898400068283081
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",162,2026212484713,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.6375,16.085,4.331040143966675
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",48,2026215884701,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,20.4,65.3,4.220952272415161
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",173,2026214184775,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.31,35.44,3.2175652980804443
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",49,2026215884702,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.575,63.65,3.1426401138305664
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",163,2026212484714,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.6325,15.475,4.085622310638428
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",267,2026209084716,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.475,64.9,4.273773431777954
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",174,2026214184776,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,9.75,35.72,3.723235845565796
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",50,2026215884703,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.375,63.9,3.515153646469116
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",268,2026209084717,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,32.475,62.0,3.534579277038574
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",164,2026212484715,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.525,15.77,4.324563026428223
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",175,2026214184777,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.72,35.56,4.073129892349243
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",51,2026215884704,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.575,63.1,3.7500996589660645
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",165,2026212484716,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.51,16.35,3.9577953815460205
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",269,2026209084718,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,35.8,63.75,4.621976613998413
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",176,2026214184778,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.29,36.26,3.0057432651519775
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",52,2026215884705,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,22.25,63.85,3.6505415439605713
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",177,2026214184779,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.99,35.88,2.8355233669281006
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",166,2026212484717,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,3.5475,15.665,3.834582805633545
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",270,2026209084719,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,32.975,63.2,5.12289571762085
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",53,2026215884706,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.2,66.5,3.803694248199463
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",178,2026214184780,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,9.91,35.98,3.581193685531616
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",167,2026212484718,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.74,15.955,3.6442227363586426
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",271,2026209084720,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,35.45,63.15,4.53316593170166
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",54,2026215884707,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.15,64.75,4.666337728500366
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",168,2026212484719,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.725,15.995,3.678316831588745
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",179,2026214184781,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.29,36.56,3.9075047969818115
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",272,2026209084721,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.975,66.85,4.1742188930511475
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",180,2026214184782,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,1.5,2,1,9.68,36.46,2.787128448486328
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",55,2026215884708,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.425,64.4,4.757841110229492
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",169,2026212484720,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.6725,16.33,4.657601356506348
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",181,2026214184783,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.8,35.44,3.752153158187866
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",273,2026209084722,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.5,64.9,4.625064849853516
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",170,2026212484721,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.7875,16.01,4.141845703125
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",56,2026215884709,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.975,63.9,4.60521388053894
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",182,2026214184784,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.89,36.52,4.105008363723755
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",274,2026209084723,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,35.175,63.05,4.864047288894653
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",171,2026212484722,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.665,15.99,4.053349733352661
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",57,2026215884710,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.525,62.45,4.5059144496917725
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",183,2026214184785,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.74,36.04,3.5437352657318115
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",275,2026209084724,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,34.1,63.1,4.266511917114258
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",172,2026212484723,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.5075,15.725,4.165246248245239
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",58,2026215884711,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.75,65.35,4.119385719299316
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",184,2026214184786,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.08,35.56,4.360162973403931
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",173,2026212484724,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,3.7525,16.075,3.8014118671417236
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",276,2026209084725,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.6,63.55,4.354531288146973
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",59,2026215884712,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.825,62.05,4.381434917449951
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",185,2026214184787,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.48,35.62,3.781536340713501
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",174,2026212484725,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.8625,15.71,4.158869981765747
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",277,2026209084726,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.725,63.65,4.950634479522705
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",186,2026214184788,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.06,36.08,3.9694056510925293
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",60,2026215884713,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.175,64.1,4.8157243728637695
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",175,2026212484726,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.7275,15.83,3.7496140003204346
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",61,2026215884714,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.625,65.0,3.2754366397857666
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",187,2026214184789,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.38,35.68,3.96574068069458
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0005|pi0|k1,1,1",278,2026209084727,B,noisy,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,33.775,62.5,5.2269673347473145
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",176,2026212484727,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,3.5825,15.84,4.0832695960998535
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",0,2026217584704,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.54,3.85,1.438469648361206
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",188,2026214184790,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.94,35.88,2.7242774963378906
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",62,2026215884715,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.8,60.4,3.539536714553833
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",1,2026217584705,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,3.0,3,0,0.5075,3.85,1.1363794803619385
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",2,2026217584706,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5175,4.06,1.1015686988830566
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",3,2026217584707,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.5925,3.995,1.0337624549865723
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",177,2026212484728,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,2.5,3,1,3.6525,15.865,3.955050230026245
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",4,2026217584708,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5925,3.985,1.098006248474121
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",63,2026215884716,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.775,66.05,3.972984790802002
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",189,2026214184791,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.36,35.84,5.072537422180176
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",5,2026217584709,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.545,3.835,1.0925450325012207
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",6,2026217584710,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.43,3.825,1.0596528053283691
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",7,2026217584711,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5575,3.9,0.9153785705566406
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",178,2026212484729,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.695,15.835,4.110707759857178
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",190,2026214184792,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.81,35.66,3.3481459617614746
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",64,2026215884717,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.9,61.55,3.9680850505828857
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",8,2026217584712,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.65,4.07,1.4115879535675049
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",9,2026217584713,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5425,4.005,0.8503937721252441
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",10,2026217584714,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.545,3.9,1.3475162982940674
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",11,2026217584715,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.555,3.875,0.8008232116699219
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",191,2026214184793,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.81,35.36,3.7975900173187256
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",179,2026212484730,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,3.6075,15.95,4.469994783401489
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",65,2026215884718,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.5,1,1,21.45,64.2,4.299912929534912
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",12,2026217584716,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5475,4.015,1.28169584274292
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",13,2026217584717,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5525,3.93,1.0594089031219482
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",14,2026217584718,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5775,3.965,0.892317533493042
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",180,2026212484731,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,3.6125,15.995,3.415435552597046
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",15,2026217584719,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,2.5,3,1,0.54,4.005,0.8693544864654541
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",192,2026214184794,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.11,35.44,4.303255081176758
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",16,2026217584720,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.515,4.26,1.6714115142822266
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",66,2026215884719,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.1,65.7,4.994527578353882
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",17,2026217584721,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.51,4.11,0.878028154373169
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",181,2026212484732,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.5725,15.985,3.480415105819702
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",193,2026214184795,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.06,35.84,3.0211362838745117
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",18,2026217584722,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5625,4.005,1.1859543323516846
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",19,2026217584723,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.5975,4.075,0.795438289642334
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",20,2026217584724,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.585,4.005,0.9174792766571045
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",67,2026215884720,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.525,63.0,4.1552510261535645
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",21,2026217584725,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.525,3.94,1.0150237083435059
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",182,2026212484733,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.46,15.88,3.742988109588623
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",194,2026214184796,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.65,35.84,3.489759922027588
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",22,2026217584726,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.48,4.145,1.3699605464935303
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",23,2026217584727,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.575,3.99,0.7838976383209229
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",24,2026217584728,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.575,3.955,0.9643089771270752
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",25,2026217584729,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5375,4.1,0.6854243278503418
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",195,2026214184797,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.76,35.1,3.1539855003356934
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",68,2026215884721,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.525,64.7,4.662346839904785
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",183,2026212484734,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.6875,16.095,3.981431007385254
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",26,2026217584730,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.56,4.085,1.004390001296997
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",27,2026217584731,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5425,3.92,0.8930964469909668
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",28,2026217584732,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5625,3.9,1.254913568496704
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",196,2026214184798,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.86,35.16,3.459341526031494
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",29,2026217584733,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6025,3.795,0.8153715133666992
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",184,2026212484735,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.6325,15.77,3.7555556297302246
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",69,2026215884722,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.95,63.5,4.15392804145813
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",30,2026217584734,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,2.5,3,1,0.6125,3.91,1.6175920963287354
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",31,2026217584735,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.64,4.13,1.1938893795013428
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",197,2026214184799,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.43,35.88,3.4413673877716064
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",32,2026217584736,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.555,4.05,0.7136833667755127
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",33,2026217584737,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5875,4.005,0.9581263065338135
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",70,2026215884723,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.775,62.6,3.9065747261047363
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",185,2026212484736,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,2.0,3,2,3.55,15.685,4.297515630722046
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",34,2026217584738,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.555,3.935,0.9628171920776367
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",198,2026214184800,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.79,35.46,2.9262967109680176
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",35,2026217584739,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.515,3.92,1.214888334274292
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",36,2026217584740,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.49,3.935,1.0152513980865479
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",71,2026215884724,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.625,63.95,3.690915584564209
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",186,2026212484737,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.6475,16.09,3.740710973739624
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",37,2026217584741,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,3.5,4,1,0.5525,3.91,1.2807085514068604
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",199,2026214184801,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.04,35.62,4.117039203643799
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",38,2026217584742,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5625,3.825,1.243741750717163
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",39,2026217584743,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5675,4.21,0.8507575988769531
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",40,2026217584744,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5375,4.035,1.0521020889282227
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",72,2026215884725,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.625,66.45,4.339408874511719
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",187,2026212484738,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,3.6575,16.25,4.263752698898315
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",200,2026214184802,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.93,35.7,2.8939661979675293
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",41,2026217584745,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5325,3.97,0.8612174987792969
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",42,2026217584746,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5975,3.905,0.7794344425201416
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",43,2026217584747,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5375,4.115,0.9374880790710449
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",44,2026217584748,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5975,4.0,0.9488692283630371
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",45,2026217584749,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,3.0,3,0,0.5075,3.975,0.8195269107818604
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",188,2026212484739,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.73,16.195,4.12794828414917
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",73,2026215884726,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.775,67.7,4.438052415847778
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",201,2026214184803,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.93,36.34,4.1528613567352295
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",46,2026217584750,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.555,3.975,0.8091776371002197
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",47,2026217584751,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.535,3.905,1.1619906425476074
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",48,2026217584752,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5775,4.28,1.1606929302215576
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",202,2026214184804,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.42,36.02,3.0876858234405518
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",49,2026217584753,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,3.0,3,0,0.53,4.03,0.7016804218292236
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",74,2026215884727,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.4,64.75,3.8812878131866455
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",50,2026217584754,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.505,3.945,0.9302723407745361
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",189,2026212484740,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.5625,15.945,4.237787485122681
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",51,2026217584755,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.5775,3.985,1.3272902965545654
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",52,2026217584756,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.4525,3.945,1.0735418796539307
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",203,2026214184805,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.14,35.46,4.215886831283569
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",53,2026217584757,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.5325,4.05,0.8499255180358887
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",75,2026215884728,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.525,65.2,4.3596885204315186
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",190,2026212484741,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.63,16.22,4.3866965770721436
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",54,2026217584758,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.62,4.09,1.2715520858764648
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",55,2026217584759,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5025,3.915,0.8059341907501221
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",204,2026214184806,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.75,35.22,3.2659952640533447
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",56,2026217584760,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.62,3.905,1.2214105129241943
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",57,2026217584761,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6025,4.11,0.9808359146118164
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",76,2026215884729,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.55,64.55,4.41186261177063
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",191,2026212484742,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.755,16.04,4.238982439041138
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",58,2026217584762,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5625,4.105,1.2590267658233643
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",59,2026217584763,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.4675,3.905,0.6915140151977539
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",205,2026214184807,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,9.63,35.9,3.531686782836914
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",60,2026217584764,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.645,3.865,1.5478651523590088
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",61,2026217584765,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5325,4.01,0.9486250877380371
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",192,2026212484743,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.67,16.125,3.5943517684936523
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",77,2026215884730,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,21.125,64.95,4.456321954727173
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",206,2026214184808,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,10.29,36.46,3.051798105239868
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",62,2026217584766,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.485,4.055,0.9732201099395752
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",63,2026217584767,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.54,3.92,1.4911694526672363
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",64,2026217584768,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.5,2,1,0.53,4.04,0.9361932277679443
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",207,2026214184809,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.83,36.14,3.0107290744781494
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",78,2026215884731,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.0,64.7,3.506584644317627
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",65,2026217584769,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5425,4.045,1.0950546264648438
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",193,2026212484744,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.6975,15.82,4.431798696517944
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",66,2026217584770,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5625,4.06,0.8446781635284424
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",67,2026217584771,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.495,3.94,0.8487453460693359
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",208,2026214184810,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.43,35.9,3.170133590698242
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",68,2026217584772,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.545,3.94,1.076664686203003
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",194,2026212484745,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.6175,15.925,3.104938268661499
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",69,2026217584773,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5225,3.86,0.9881138801574707
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",70,2026217584774,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.585,4.01,0.9622092247009277
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",79,2026215884732,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.4,62.9,5.0263512134552
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",71,2026217584775,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.4975,3.855,1.1531214714050293
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",72,2026217584776,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5375,3.97,0.7293369770050049
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",209,2026214184811,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.95,35.52,4.31063175201416
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",195,2026212484746,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.66,15.875,4.0326738357543945
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",73,2026217584777,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5325,3.815,1.0980701446533203
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",74,2026217584778,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5625,4.01,0.7444138526916504
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",80,2026215884733,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,20.275,65.15,3.886883020401001
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",75,2026217584779,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.605,4.215,0.7694394588470459
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",76,2026217584780,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5825,3.89,1.2994797229766846
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",210,2026214184812,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.65,35.44,3.8914635181427
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",196,2026212484747,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.76,15.925,3.869776725769043
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",77,2026217584781,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.56,4.055,1.0788185596466064
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",81,2026215884734,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,21.1,63.15,3.5340240001678467
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",78,2026217584782,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.49,3.795,1.328364610671997
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",197,2026212484748,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.72,15.87,3.2874555587768555
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",79,2026217584783,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5475,3.9,1.5298588275909424
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",211,2026214184813,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,9.77,36.32,4.037145137786865
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",82,2026215884735,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.1,64.95,2.926861524581909
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",80,2026217584784,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.59,3.995,1.3949487209320068
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",81,2026217584785,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.56,4.065,1.2083866596221924
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",82,2026217584786,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.48,3.805,0.7542216777801514
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",198,2026212484749,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.565,16.225,3.539158821105957
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",83,2026217584787,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.5975,4.125,0.8079333305358887
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",212,2026214184814,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.17,36.06,4.062692403793335
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",83,2026215884736,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.525,66.25,3.955518960952759
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",84,2026217584788,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5625,4.05,1.2523951530456543
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",85,2026217584789,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5375,4.12,0.8484323024749756
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",86,2026217584790,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.545,4.14,0.7538235187530518
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",199,2026212484750,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.6675,16.03,3.9667069911956787
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",87,2026217584791,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5025,4.075,0.7853453159332275
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",84,2026215884737,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.05,64.1,3.7198784351348877
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",213,2026214184815,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,9.63,36.06,4.166736364364624
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",88,2026217584792,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6475,3.93,0.9649255275726318
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",89,2026217584793,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.5,2,1,0.5525,4.025,1.345660924911499
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",200,2026212484751,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.565,16.47,3.657055616378784
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",90,2026217584794,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5425,4.055,1.240386962890625
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",91,2026217584795,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5825,4.205,0.9480347633361816
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",85,2026215884738,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,21.475,64.5,3.8330862522125244
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",214,2026214184816,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,8.93,36.2,3.8479185104370117
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",92,2026217584796,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5625,3.965,0.9612345695495605
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",93,2026217584797,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.51,4.11,0.9342250823974609
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",201,2026212484752,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.69,15.92,3.5992672443389893
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",94,2026217584798,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.4975,4.11,1.1273412704467773
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",215,2026214184817,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.42,36.22,3.4894092082977295
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",95,2026217584799,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.54,3.92,0.716458797454834
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",96,2026217584800,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.575,3.92,0.8563306331634521
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",86,2026215884739,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.825,63.25,4.731981515884399
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",97,2026217584801,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.53,3.97,0.7570374011993408
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",202,2026212484753,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,2.0,2,0,3.6275,16.14,4.219061374664307
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",98,2026217584802,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.4975,4.185,1.3054161071777344
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",216,2026214184818,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.04,36.34,4.085429430007935
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",99,2026217584803,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.505,3.96,1.272383689880371
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",87,2026215884740,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.675,64.15,3.9565036296844482
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",100,2026217584804,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5725,3.905,1.2981343269348145
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",101,2026217584805,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.56,4.0,0.891728401184082
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",217,2026214184819,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.8,36.86,2.9905202388763428
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",203,2026212484754,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,3.595,15.88,4.471489906311035
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",102,2026217584806,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.505,3.99,1.2908656597137451
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",103,2026217584807,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.5,2,1,0.5575,4.185,1.1532542705535889
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",88,2026215884741,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.2,64.7,3.946259021759033
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",104,2026217584808,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5725,4.105,0.8293719291687012
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",218,2026214184820,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,9.92,35.58,3.4770283699035645
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",105,2026217584809,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.53,4.115,0.8160824775695801
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",204,2026212484755,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,3.5325,16.145,3.3801522254943848
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",106,2026217584810,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.565,3.945,0.8198046684265137
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",107,2026217584811,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5,3.905,0.7314376831054688
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",89,2026215884742,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.125,62.65,3.2321503162384033
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",108,2026217584812,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.55,3.84,0.8109340667724609
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",219,2026214184821,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.51,35.82,2.746687650680542
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",109,2026217584813,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.515,4.035,0.7457208633422852
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",205,2026212484756,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.6175,16.065,3.14101243019104
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",110,2026217584814,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.5625,4.045,0.958920955657959
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",90,2026215884743,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.375,63.3,2.933915138244629
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",111,2026217584815,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.54,4.05,0.7839150428771973
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",220,2026214184822,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.59,35.98,2.637160539627075
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",112,2026217584816,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5825,3.915,0.7241699695587158
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",113,2026217584817,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.545,4.055,0.7837255001068115
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",206,2026212484757,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,3.6425,15.93,2.975464344024658
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",114,2026217584818,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5225,4.205,0.7214698791503906
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",91,2026215884744,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.7,62.2,3.0957648754119873
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",115,2026217584819,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.58,4.005,0.7187783718109131
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",221,2026214184823,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,1.5,2,1,9.7,35.46,2.660407066345215
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",116,2026217584820,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5975,4.03,0.7136890888214111
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",207,2026212484758,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.5625,15.795,2.805617570877075
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",117,2026217584821,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.5075,3.86,0.6864008903503418
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",118,2026217584822,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.49,3.92,0.6939430236816406
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",222,2026214184824,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.71,35.6,2.383075475692749
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",119,2026217584823,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.5,2,1,0.5425,4.11,0.700089693069458
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",92,2026215884745,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.35,63.6,3.4314143657684326
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",120,2026217584824,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.525,3.925,0.6905059814453125
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",121,2026217584825,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.55,3.905,0.6937358379364014
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",208,2026212484759,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,3.7225,15.86,2.8247058391571045
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",122,2026217584826,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.54,3.97,0.7508306503295898
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",223,2026214184825,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.98,36.0,2.7407400608062744
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",123,2026217584827,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.52,4.115,0.7664620876312256
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",93,2026215884746,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.1,63.25,3.2040748596191406
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",124,2026217584828,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.545,4.03,0.8305959701538086
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",125,2026217584829,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.515,3.935,0.8004088401794434
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",209,2026212484760,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.63,16.1,3.205885887145996
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",224,2026214184826,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.94,35.8,2.7574188709259033
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",126,2026217584830,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.56,3.97,0.7720403671264648
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",127,2026217584831,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5,3.91,0.7489049434661865
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",94,2026215884747,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.025,65.0,2.5970237255096436
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",128,2026217584832,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5325,3.885,0.7637221813201904
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",129,2026217584833,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.565,4.05,0.8020308017730713
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",210,2026212484761,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.57,15.89,3.0820846557617188
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",225,2026214184827,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,10.14,36.14,2.71838116645813
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",130,2026217584834,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5725,4.08,0.7672395706176758
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",131,2026217584835,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5425,4.19,0.7569143772125244
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",95,2026215884748,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.325,64.05,3.327016592025757
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",132,2026217584836,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5025,4.05,0.7685353755950928
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",133,2026217584837,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.58,4.055,0.7635769844055176
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",226,2026214184828,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.69,36.58,2.7002029418945312
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",211,2026212484762,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.515,16.06,3.1574642658233643
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",134,2026217584838,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5225,4.0,0.7619507312774658
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",135,2026217584839,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5025,3.89,0.9043948650360107
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",96,2026215884749,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.15,64.9,3.0930280685424805
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",136,2026217584840,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.505,4.055,0.7781803607940674
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",227,2026214184829,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.43,37.38,2.7335503101348877
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",137,2026217584841,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.565,4.04,0.7270495891571045
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",212,2026212484763,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.6775,16.175,3.3102104663848877
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",138,2026217584842,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.58,3.98,0.729717493057251
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",139,2026217584843,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5775,4.085,0.728562593460083
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",97,2026215884750,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.5,1,1,20.375,63.45,2.7443573474884033
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",140,2026217584844,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.525,3.93,0.7709715366363525
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",228,2026214184830,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.95,35.88,2.7226524353027344
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",141,2026217584845,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.515,3.98,0.7648086547851562
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",213,2026212484764,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.7675,15.79,2.99950909614563
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",142,2026217584846,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.5,2,1,0.6175,4.105,0.7773692607879639
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",98,2026215884751,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.8,62.85,2.7443325519561768
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",143,2026217584847,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5525,4.225,0.7460741996765137
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",229,2026214184831,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,9.71,36.82,2.8260021209716797
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",144,2026217584848,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.54,4.05,0.7511472702026367
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",145,2026217584849,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.495,3.985,0.7830228805541992
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",214,2026212484765,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.6425,15.77,3.0779869556427
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",146,2026217584850,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,3.0,3,0,0.61,4.235,0.8574504852294922
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",99,2026215884752,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.4,63.05,3.0810916423797607
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",147,2026217584851,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.545,4.15,0.7669849395751953
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",230,2026214184832,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,9.7,35.76,2.6454355716705322
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",148,2026217584852,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5425,4.0,1.1020939350128174
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",149,2026217584853,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.5,2,1,0.5675,4.125,1.5627212524414062
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",215,2026212484766,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.6175,15.665,4.279993057250977
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",150,2026217584854,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.6,3.89,1.504990816116333
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",231,2026214184833,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.19,34.74,5.2551429271698
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",100,2026215884753,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.325,63.35,5.735621213912964
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",151,2026217584855,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.555,4.065,1.6975929737091064
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",152,2026217584856,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.555,3.87,1.5693879127502441
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",153,2026217584857,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.525,4.02,1.5547137260437012
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",216,2026212484767,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.5075,16.095,6.5675458908081055
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",154,2026217584858,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5175,4.0,1.552696943283081
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",232,2026214184834,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.3,35.6,5.682172060012817
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",101,2026215884754,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.1,64.35,5.915846586227417
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",155,2026217584859,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5225,3.925,1.633530855178833
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",156,2026217584860,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.55,3.875,1.7138679027557373
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",157,2026217584861,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.53,3.95,1.689514398574829
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",217,2026212484768,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,2.0,2,0,3.69,15.955,6.323490381240845
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",233,2026214184835,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.78,36.38,5.564270734786987
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",158,2026217584862,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5975,4.07,1.6100678443908691
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",102,2026215884755,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.225,64.6,6.489675045013428
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",159,2026217584863,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.515,3.97,1.6264081001281738
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",160,2026217584864,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5525,4.03,1.467421054840088
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",234,2026214184836,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.58,35.82,5.070284128189087
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",218,2026212484769,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.6075,16.105,6.131028652191162
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",161,2026217584865,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5475,3.935,1.8030414581298828
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",162,2026217584866,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.57,4.095,1.7300963401794434
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",103,2026215884756,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.825,61.9,6.085678815841675
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",163,2026217584867,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5625,4.04,1.7219023704528809
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",235,2026214184837,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.46,35.8,5.364573955535889
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",164,2026217584868,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.5,4.135,1.7513959407806396
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",219,2026212484770,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,2.0,2,0,3.555,15.61,6.486207723617554
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",165,2026217584869,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.54,4.005,1.6357195377349854
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",104,2026215884757,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.4,66.4,6.195813417434692
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",166,2026217584870,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.4825,3.795,1.6751227378845215
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",167,2026217584871,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.525,3.93,1.6261897087097168
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",236,2026214184838,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.97,36.58,5.431467056274414
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",168,2026217584872,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.585,4.015,1.65995192527771
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",220,2026212484771,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,3.63,16.375,5.874324560165405
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",169,2026217584873,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.54,4.225,1.7756876945495605
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",105,2026215884758,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.85,66.0,6.70676589012146
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",237,2026214184839,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.88,36.04,4.517447471618652
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",170,2026217584874,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,2,2,0.5475,4.005,1.6487491130828857
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",171,2026217584875,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5475,4.005,1.655411720275879
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",221,2026212484772,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.6525,16.425,6.282054662704468
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",172,2026217584876,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6375,4.15,1.7581498622894287
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",238,2026214184840,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.14,37.08,5.267248630523682
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",173,2026217584877,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.515,4.0,1.6734871864318848
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",106,2026215884759,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.15,64.95,6.623932361602783
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",174,2026217584878,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,2.5,3,1,0.5875,3.96,1.641477346420288
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",175,2026217584879,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.59,3.955,1.7235321998596191
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",222,2026212484773,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.5975,16.055,6.417081356048584
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",176,2026217584880,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5825,4.065,1.669569969177246
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",239,2026214184841,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.34,36.28,5.762909412384033
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",177,2026217584881,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.515,4.0,1.6794915199279785
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",107,2026215884760,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.8,63.85,6.776688814163208
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",178,2026217584882,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5375,3.945,1.6730000972747803
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",179,2026217584883,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5525,3.9,1.6649754047393799
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",223,2026212484774,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.73,15.97,6.222679376602173
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",240,2026214184842,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.49,35.76,5.619103670120239
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",180,2026217584884,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.5025,4.04,1.6466331481933594
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",181,2026217584885,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5925,4.1,1.6856701374053955
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",108,2026215884761,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.1,63.75,5.897036552429199
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",182,2026217584886,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.58,4.055,1.6737544536590576
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",241,2026214184843,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.56,36.38,5.522447109222412
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",183,2026217584887,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5425,3.945,1.6504578590393066
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",224,2026212484775,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.54,16.33,6.245987415313721
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",184,2026217584888,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.505,3.895,1.664224624633789
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",109,2026215884762,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.475,65.15,5.989112854003906
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",185,2026217584889,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5475,4.125,1.7859909534454346
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",186,2026217584890,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5175,4.225,1.590315580368042
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",242,2026214184844,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.7,35.64,5.769659519195557
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",225,2026212484776,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,3.5025,15.93,6.221098899841309
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",187,2026217584891,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.5475,3.88,1.5743722915649414
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",188,2026217584892,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6125,4.05,1.5946927070617676
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",110,2026215884763,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,22.2,63.6,6.214962482452393
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",189,2026217584893,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.515,4.165,1.7287395000457764
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",243,2026214184845,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.47,34.92,5.332171201705933
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",190,2026217584894,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.5,2,1,0.55,3.86,1.658658742904663
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",226,2026212484777,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.5725,15.735,6.525269985198975
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",191,2026217584895,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.54,3.93,1.6598095893859863
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",192,2026217584896,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5625,4.025,1.657245397567749
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",111,2026215884764,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.625,66.8,6.773498296737671
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",193,2026217584897,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.53,4.01,1.7081749439239502
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",244,2026214184846,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.71,36.02,5.896036624908447
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",194,2026217584898,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.555,4.08,1.7166593074798584
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",227,2026212484778,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.7475,16.09,6.372652292251587
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",195,2026217584899,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.5175,4.105,1.605757713317871
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",196,2026217584900,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.58,3.845,1.602034330368042
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",245,2026214184847,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.21,36.24,5.38515830039978
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",112,2026215884765,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.075,62.5,6.490401029586792
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",197,2026217584901,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5475,4.015,1.5952680110931396
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",198,2026217584902,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.58,3.99,1.6901776790618896
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",228,2026212484779,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.575,15.68,6.409250497817993
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",199,2026217584903,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5,3.915,1.6990232467651367
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",246,2026214184848,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.89,36.52,5.413312911987305
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",200,2026217584904,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5575,3.83,1.73077392578125
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",113,2026215884766,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.275,63.7,6.960413932800293
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",201,2026217584905,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5575,4.02,1.7263176441192627
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",202,2026217584906,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.5875,4.04,1.7323718070983887
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",229,2026212484780,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.6475,15.765,6.244168043136597
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",247,2026214184849,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.42,36.68,5.808332681655884
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",203,2026217584907,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.63,4.195,1.848355770111084
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",204,2026217584908,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5325,3.99,1.6357626914978027
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",114,2026215884767,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.5,1,1,20.225,62.1,7.01423454284668
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",205,2026217584909,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.6225,3.99,1.6877126693725586
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",206,2026217584910,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,2.5,3,1,0.575,4.03,1.624495029449463
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",230,2026212484781,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,3.5925,16.12,6.666304349899292
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",248,2026214184850,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.29,36.04,5.54617166519165
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",207,2026217584911,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.515,3.895,1.5492401123046875
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",208,2026217584912,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5175,3.98,1.6189401149749756
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",209,2026217584913,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.625,3.995,1.7964930534362793
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",115,2026215884768,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.35,62.35,7.270457029342651
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",249,2026214184851,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,9.23,34.36,5.968791246414185
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",210,2026217584914,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.555,3.955,1.6302595138549805
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",231,2026212484782,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.5975,16.22,6.769061326980591
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",211,2026217584915,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.56,3.935,1.747117280960083
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",212,2026217584916,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5125,4.245,1.721616268157959
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",213,2026217584917,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5025,4.005,1.738344430923462
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",116,2026215884769,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.85,63.85,7.22412896156311
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",250,2026214184852,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,9.81,37.28,5.886060953140259
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",232,2026212484783,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.6925,15.925,6.320071697235107
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",214,2026217584918,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.4575,4.1,1.6688055992126465
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",215,2026217584919,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.4575,4.095,1.7456705570220947
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",216,2026217584920,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6075,4.175,1.5457768440246582
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",251,2026214184853,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.86,37.06,5.534097671508789
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",217,2026217584921,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.57,4.135,1.5265212059020996
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",117,2026215884770,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.025,63.25,6.986990928649902
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",233,2026212484784,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.585,15.915,6.1932268142700195
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",218,2026217584922,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5275,3.955,1.5735409259796143
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",219,2026217584923,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5675,4.205,1.5238125324249268
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",220,2026217584924,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5125,4.11,1.5884482860565186
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",252,2026214184854,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.82,35.3,5.642707347869873
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",221,2026217584925,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5325,4.095,1.634855031967163
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",118,2026215884771,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,22.15,62.25,6.2160186767578125
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",234,2026212484785,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.725,16.075,6.431520462036133
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",222,2026217584926,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.505,4.005,1.5199840068817139
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",223,2026217584927,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.58,3.92,1.6711831092834473
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",253,2026214184855,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.67,36.06,5.733134508132935
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",224,2026217584928,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5625,3.885,1.532898187637329
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",225,2026217584929,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5675,3.945,1.5489695072174072
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",119,2026215884772,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.825,64.05,6.518070697784424
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",235,2026212484786,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.8,16.17,6.4088380336761475
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",226,2026217584930,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.535,4.22,1.7571120262145996
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",227,2026217584931,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.68,3.92,1.8703398704528809
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",254,2026214184856,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.1,35.6,5.700110197067261
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",228,2026217584932,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.485,4.015,1.7030575275421143
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",229,2026217584933,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.555,3.855,1.8002955913543701
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",236,2026212484787,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.7125,16.09,6.448001861572266
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",120,2026215884773,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.6,63.75,6.753538131713867
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",230,2026217584934,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.53,4.095,1.6538875102996826
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",255,2026214184857,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.08,36.54,5.317234754562378
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",231,2026217584935,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5075,4.185,1.6365654468536377
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",232,2026217584936,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.5225,4.06,1.7965211868286133
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",233,2026217584937,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5825,3.955,1.798520565032959
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",237,2026212484788,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,2.0,3,2,3.665,15.975,6.450859546661377
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",121,2026215884774,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.6,62.45,6.924661874771118
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",256,2026214184858,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.55,36.36,5.724154472351074
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",234,2026217584938,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6225,4.215,1.7181577682495117
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",235,2026217584939,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.54,3.865,1.6716442108154297
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",236,2026217584940,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.515,4.055,1.6979529857635498
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",257,2026214184859,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.81,36.22,5.3420093059539795
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",238,2026212484789,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.5,16.32,6.357919216156006
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",237,2026217584941,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,2.5,3,1,0.5675,4.055,1.7680654525756836
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",122,2026215884775,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.8,65.5,7.434214353561401
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",238,2026217584942,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5925,4.02,1.8217709064483643
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",239,2026217584943,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.505,3.895,1.6768460273742676
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",258,2026214184860,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.63,35.3,5.557934522628784
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",240,2026217584944,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6075,3.905,1.6525120735168457
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",239,2026212484790,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.6675,16.19,6.419561386108398
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",241,2026217584945,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5775,3.965,1.6650841236114502
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",123,2026215884776,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.125,65.3,7.034793853759766
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",242,2026217584946,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,2.5,3,1,0.5775,3.895,1.6628751754760742
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",243,2026217584947,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.515,4.0,1.5459110736846924
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",259,2026214184861,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.4,37.24,5.651749610900879
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",244,2026217584948,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.515,3.93,1.682569980621338
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",240,2026212484791,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.665,15.415,6.409040689468384
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",245,2026217584949,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.57,4.02,1.727778434753418
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",124,2026215884777,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.875,65.85,6.529229640960693
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",246,2026217584950,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.585,3.77,1.59602952003479
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",247,2026217584951,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5075,4.105,1.5580470561981201
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",260,2026214184862,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.84,36.22,5.825350284576416
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",248,2026217584952,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.535,3.93,1.6863977909088135
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",241,2026212484792,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.6,15.935,6.473063707351685
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",249,2026217584953,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.485,4.15,1.6236562728881836
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",125,2026215884778,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.575,61.9,6.112721920013428
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",250,2026217584954,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.54,4.13,1.6318731307983398
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",261,2026214184863,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.04,36.14,5.424854278564453
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",251,2026217584955,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.53,3.955,1.5584604740142822
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",252,2026217584956,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.5225,4.04,1.555624008178711
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",242,2026212484793,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,3.575,15.765,6.040794849395752
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",253,2026217584957,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5725,4.035,1.556190013885498
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",126,2026215884779,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.05,62.45,5.8085222244262695
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",262,2026214184864,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.81,35.72,5.247920036315918
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",254,2026217584958,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5725,3.925,1.522594690322876
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",255,2026217584959,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5225,4.135,1.5484204292297363
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",256,2026217584960,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.57,4.19,1.4914462566375732
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",243,2026212484794,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.6275,15.97,5.747074604034424
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",263,2026214184865,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.56,35.16,4.985318422317505
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",257,2026217584961,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5575,3.9,1.5254826545715332
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",127,2026215884780,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.625,61.4,5.61445951461792
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",258,2026217584962,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.58,4.2,1.5381853580474854
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",259,2026217584963,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5275,4.055,1.5102832317352295
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",244,2026212484795,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.7575,15.725,5.824105501174927
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",260,2026217584964,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,3.0,3,0,0.5525,4.105,1.567387580871582
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",264,2026214184866,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,9.75,34.7,5.3564512729644775
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",261,2026217584965,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.4925,4.055,1.4780795574188232
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",128,2026215884781,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,22.325,63.5,5.911491632461548
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",262,2026217584966,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.6075,3.955,1.4692840576171875
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",263,2026217584967,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.55,4.01,1.5099692344665527
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",245,2026212484796,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.6175,15.905,6.202956438064575
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",264,2026217584968,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.58,3.975,1.5187032222747803
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",265,2026214184867,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.8,35.82,5.8284993171691895
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",129,2026215884782,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.075,64.5,5.546315431594849
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",265,2026217584969,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.53,4.025,1.5188500881195068
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",266,2026217584970,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5275,3.945,1.5168864727020264
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",267,2026217584971,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.515,3.765,1.4879918098449707
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",268,2026217584972,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.525,4.01,1.4934287071228027
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",246,2026212484797,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.8725,15.885,6.510505676269531
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",266,2026214184868,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.71,36.76,6.146236419677734
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",130,2026215884783,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.45,63.8,5.737529277801514
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",269,2026217584973,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5075,3.815,1.5832655429840088
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",270,2026217584974,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5725,4.005,1.5270628929138184
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",271,2026217584975,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.58,3.86,1.5197064876556396
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",272,2026217584976,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5175,3.93,1.5103952884674072
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",267,2026214184869,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.04,35.84,5.725039005279541
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",131,2026215884784,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.6,64.65,5.657747745513916
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",247,2026212484798,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.6475,15.95,6.506833076477051
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",273,2026217584977,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5775,4.07,1.51637864112854
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",274,2026217584978,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5525,4.07,1.4797184467315674
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",275,2026217584979,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5325,3.995,1.5123960971832275
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",268,2026214184870,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.23,35.92,5.454724311828613
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",276,2026217584980,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.55,3.84,1.531982183456421
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",132,2026215884785,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.725,64.1,5.831952810287476
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",248,2026212484799,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.71,16.195,6.258746147155762
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",277,2026217584981,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.52,4.015,1.8975908756256104
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",278,2026217584982,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6125,3.885,1.8137106895446777
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",269,2026214184871,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,2.0,2,0,9.26,36.16,5.4609081745147705
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",279,2026217584983,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.525,3.91,1.7654249668121338
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",280,2026217584984,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.495,4.0,1.5300450325012207
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",133,2026215884786,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.85,62.95,7.246360540390015
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",249,2026212484800,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,3.8225,15.85,6.358222007751465
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",281,2026217584985,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5775,4.085,1.5043549537658691
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",282,2026217584986,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5075,4.12,1.5374314785003662
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",270,2026214184872,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.87,36.14,5.664721250534058
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",283,2026217584987,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5375,3.995,1.670301914215088
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",134,2026215884787,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.775,65.2,5.751231908798218
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",284,2026217584988,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.515,4.015,1.517303466796875
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",250,2026212484801,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,3.67,16.46,6.500629663467407
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",285,2026217584989,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5675,4.17,1.5334172248840332
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",271,2026214184873,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,9.88,35.72,5.546453237533569
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",286,2026217584990,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.605,3.905,1.5348687171936035
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",287,2026217584991,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.615,3.98,1.5676748752593994
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",135,2026215884788,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.575,63.25,6.327759265899658
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",288,2026217584992,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.4875,4.005,1.7014243602752686
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",251,2026212484802,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.66,15.995,6.131568670272827
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",289,2026217584993,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.5975,4.12,1.6748876571655273
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",272,2026214184874,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.09,36.44,5.4432532787323
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",290,2026217584994,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.54,4.09,1.6467864513397217
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",291,2026217584995,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5575,3.955,1.5998241901397705
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",252,2026212484803,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.73,16.245,6.015254497528076
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",292,2026217584996,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.525,3.965,1.5517981052398682
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",136,2026215884789,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.95,65.15,6.6806581020355225
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",273,2026214184875,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.0,36.26,5.357872009277344
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",293,2026217584997,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5925,3.91,1.5760767459869385
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",294,2026217584998,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5675,3.845,1.6845054626464844
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",295,2026217584999,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.58,3.95,1.6155643463134766
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",253,2026212484804,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.6025,15.78,5.938964366912842
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",137,2026215884790,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.825,65.9,6.051153659820557
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",274,2026214184876,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.84,36.3,5.21576714515686
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",296,2026217585000,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.61,4.115,1.600217342376709
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",297,2026217585001,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.4725,4.065,1.7161684036254883
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",298,2026217585002,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.4575,4.07,1.6991033554077148
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",275,2026214184877,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.54,36.2,4.85932993888855
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",254,2026212484805,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,2.0,2,0,3.5725,16.255,5.685019254684448
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",299,2026217585003,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5775,4.11,1.705160140991211
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",138,2026215884791,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.825,64.25,6.726988315582275
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",300,2026217585004,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.575,4.065,1.5685856342315674
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",301,2026217585005,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5625,3.91,1.500450611114502
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",302,2026217585006,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.6625,4.07,1.5042181015014648
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",276,2026214184878,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,9.73,36.26,5.111801385879517
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",255,2026212484806,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,3.6125,16.09,5.858720541000366
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",303,2026217585007,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6075,4.125,1.4925222396850586
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",139,2026215884792,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.725,64.65,5.564918756484985
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",304,2026217585008,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5325,4.02,1.455432653427124
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",305,2026217585009,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.57,3.99,1.4726836681365967
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",277,2026214184879,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.02,36.5,5.121057748794556
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",306,2026217585010,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.555,4.025,1.467500925064087
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",256,2026212484807,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.605,16.0,5.824216604232788
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",307,2026217585011,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.6025,4.025,1.4599096775054932
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",140,2026215884793,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.325,63.0,5.448674201965332
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",308,2026217585012,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5675,4.015,1.6690254211425781
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",278,2026214184880,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.53,35.92,5.076328277587891
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",309,2026217585013,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5125,4.085,1.550264835357666
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",310,2026217585014,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.5975,3.955,1.5228967666625977
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",257,2026212484808,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,2.0,2,0,3.635,15.825,6.021103858947754
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",311,2026217585015,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5525,4.145,1.7713322639465332
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",141,2026215884794,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.2,63.5,6.476538181304932
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",279,2026214184881,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.34,36.04,5.1626482009887695
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",312,2026217585016,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.62,3.84,1.622133493423462
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",313,2026217585017,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.4725,4.095,1.59464430809021
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",314,2026217585018,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5225,3.95,1.5995135307312012
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",258,2026212484809,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.81,15.735,6.0881187915802
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",315,2026217585019,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.6075,3.87,1.5390825271606445
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",142,2026215884795,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.2,65.5,6.45727014541626
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",280,2026214184882,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.61,36.12,5.231436491012573
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",316,2026217585020,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.4975,4.0,1.5056500434875488
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0002|pi0|k1,1,1",317,2026217585021,B,noisy,3,0.0002,0.0,1.0,1.0,1.0,3,3,3,400,2.0,3,2,0.4625,3.985,1.59055495262146
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",259,2026212484810,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.5325,16.145,6.137400150299072
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",281,2026214184883,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.87,35.62,5.617105484008789
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",143,2026215884796,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.675,60.85,6.449859142303467
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",0,2026219284755,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.355,15.67,5.859861612319946
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",260,2026212484811,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.675,15.94,6.4348976612091064
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",282,2026214184884,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.17,35.9,5.889523983001709
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",144,2026215884797,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.025,65.45,6.211498498916626
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",1,2026219284756,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.4875,15.855,5.473863124847412
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",261,2026212484812,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.55,15.635,6.26246452331543
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",283,2026214184885,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,9.56,36.2,6.057253360748291
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",145,2026215884798,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.35,62.45,6.100866079330444
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",2,2026219284757,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.5,16.06,5.4820146560668945
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",284,2026214184886,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.25,35.94,5.243431329727173
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",262,2026212484813,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.5,15.74,6.025879383087158
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",146,2026215884799,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.075,63.6,6.742473602294922
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",3,2026219284758,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.365,16.165,5.593872785568237
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",285,2026214184887,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.72,35.74,4.966125011444092
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",263,2026212484814,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.7925,16.205,6.095573425292969
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",4,2026219284759,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,2.0,3,2,2.38,16.01,5.485507011413574
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",147,2026215884800,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.325,62.5,6.765893936157227
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",286,2026214184888,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.22,35.46,5.945823907852173
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",264,2026212484815,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.555,16.135,6.083770990371704
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",5,2026219284760,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.4125,15.96,5.511596918106079
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",148,2026215884801,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.05,64.05,6.210346221923828
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",287,2026214184889,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.41,35.82,5.6989500522613525
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",265,2026212484816,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.685,15.96,6.8500635623931885
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",6,2026219284761,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.425,15.81,5.823156833648682
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",149,2026215884802,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.875,66.05,6.461870193481445
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",288,2026214184890,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.57,36.76,5.459052085876465
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",266,2026212484817,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.8375,15.915,6.5286455154418945
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",7,2026219284762,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.66,15.99,6.130779504776001
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",289,2026214184891,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.82,35.34,5.765838146209717
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",150,2026215884803,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.4,63.5,6.972002744674683
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",8,2026219284763,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,2.44,15.95,5.697152853012085
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",267,2026212484818,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.6525,16.15,6.473725080490112
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",290,2026214184892,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,9.47,37.24,5.975093364715576
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",151,2026215884804,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.675,63.95,6.494748115539551
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",9,2026219284764,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.2975,15.72,5.843615770339966
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",268,2026212484819,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.6525,15.93,6.233517169952393
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",291,2026214184893,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.53,35.22,5.848737716674805
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",152,2026215884805,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.7,65.95,7.554729461669922
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",10,2026219284765,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.465,16.27,5.831576824188232
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",292,2026214184894,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.5,36.32,5.787642240524292
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",269,2026212484820,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,3.6875,16.33,6.710803985595703
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",11,2026219284766,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,2.0,2,0,2.475,15.82,6.0274529457092285
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",153,2026215884806,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.475,64.2,7.611304521560669
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",293,2026214184895,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.52,37.0,5.306509017944336
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",270,2026212484821,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.8375,15.835,6.174945831298828
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",12,2026219284767,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.575,16.08,5.773850202560425
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",294,2026214184896,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.34,36.12,5.4425554275512695
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",271,2026212484822,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.52,16.13,6.375385761260986
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",154,2026215884807,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.0,63.7,7.70359468460083
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",13,2026219284768,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.375,16.235,5.967010974884033
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",295,2026214184897,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.27,36.34,5.531597375869751
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",272,2026212484823,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,2,2,3.78,15.595,6.17950439453125
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",155,2026215884808,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.075,63.85,7.161711931228638
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",14,2026219284769,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.4125,15.71,5.523616552352905
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",296,2026214184898,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.52,35.16,5.704326152801514
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0003|pi0|k1,1,1",273,2026212484824,B,noisy,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.75,16.025,6.546228885650635
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",156,2026215884809,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.375,64.6,6.421074151992798
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",15,2026219284770,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.5275,16.01,5.715625286102295
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",297,2026214184899,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,1.0,2,2,9.54,35.94,5.481029987335205
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",16,2026219284771,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,2.4175,16.075,5.432190179824829
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",157,2026215884810,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.35,65.5,6.04557466506958
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",298,2026214184900,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,10.02,36.16,6.1028242111206055
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",0,2026220984806,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.26,36.62,5.885727882385254
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",17,2026219284772,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.4625,15.875,5.463674068450928
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",299,2026214184901,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.64,35.72,5.181849002838135
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",158,2026215884811,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.825,63.5,6.2489893436431885
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",1,2026220984807,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.5,35.64,5.17923378944397
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",18,2026219284773,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.435,16.04,5.817004680633545
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",300,2026214184902,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.88,36.2,5.095204591751099
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",2,2026220984808,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.61,36.18,4.876275062561035
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",159,2026215884812,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.275,65.75,7.60173487663269
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",301,2026214184903,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.71,35.72,5.581190586090088
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",3,2026220984809,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.48,36.96,5.108555793762207
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",19,2026219284774,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.3525,16.12,5.795680284500122
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",160,2026215884813,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.875,62.95,7.059569835662842
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",20,2026219284775,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.35,16.06,5.499218702316284
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",4,2026220984810,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.75,36.02,5.734643220901489
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",302,2026214184904,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.78,35.92,6.022706031799316
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",5,2026220984811,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.54,35.94,5.477079153060913
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",21,2026219284776,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.4075,15.725,5.734167098999023
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",161,2026215884814,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.825,64.9,7.526946306228638
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",303,2026214184905,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.84,36.02,5.711899518966675
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",6,2026220984812,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.56,35.84,5.422309875488281
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0003|pi0|k1,1,1",304,2026214184906,B,noisy,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.87,36.02,5.583089590072632
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",22,2026219284777,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,2.395,15.83,5.951989412307739
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",162,2026215884815,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.625,64.15,7.558062314987183
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",7,2026220984813,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.4,36.12,5.550561904907227
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",23,2026219284778,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.2575,15.83,5.924792289733887
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",163,2026215884816,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.95,64.55,7.1222145557403564
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",8,2026220984814,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.76,37.22,5.652498483657837
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",24,2026219284779,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.4275,16.28,5.374679327011108
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",0,2026222684857,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,12.75,65.3,6.469575881958008
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",164,2026215884817,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.175,62.4,6.271275520324707
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",9,2026220984815,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.81,35.9,5.488616943359375
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",25,2026219284780,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.4675,16.045,5.709676504135132
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",1,2026222684858,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.025,64.45,6.059776544570923
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",10,2026220984816,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.52,35.5,5.620272874832153
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",165,2026215884818,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.2,64.7,6.762648105621338
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",26,2026219284781,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.315,16.05,5.750970125198364
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",2,2026222684859,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.5,62.85,5.999715328216553
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",11,2026220984817,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.47,35.56,5.639165163040161
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",27,2026219284782,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.4925,15.665,5.741247177124023
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",166,2026215884819,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.95,64.6,7.183960914611816
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",12,2026220984818,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.72,36.4,5.447220325469971
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",3,2026222684860,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,14.275,65.15,6.504347324371338
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",28,2026219284783,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.48,15.755,5.881561517715454
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",167,2026215884820,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.25,64.55,6.141657590866089
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",13,2026220984819,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.39,36.68,5.131104230880737
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",4,2026222684861,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.175,64.35,6.0639872550964355
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",29,2026219284784,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.3825,15.995,5.842134237289429
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",168,2026215884821,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.3,65.6,6.412933349609375
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",14,2026220984820,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,7.02,34.78,5.1809258460998535
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",5,2026222684862,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.05,64.8,6.188625812530518
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",30,2026219284785,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,2.405,16.325,5.836013317108154
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",169,2026215884822,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.6,63.95,6.275458335876465
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",15,2026220984821,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.54,36.12,4.924026250839233
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",31,2026219284786,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.48,16.3,5.647674798965454
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",6,2026222684863,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.575,61.65,6.102607488632202
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",16,2026220984822,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.87,35.58,5.54438042640686
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",170,2026215884823,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.1,64.65,6.141800880432129
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",7,2026222684864,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.375,64.85,5.382422924041748
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",32,2026219284787,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,2.5325,15.79,6.156285762786865
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",17,2026220984823,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,6.63,35.76,4.867847442626953
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",171,2026215884824,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.025,62.45,7.202590703964233
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",8,2026222684865,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.0,64.7,5.712381601333618
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",33,2026219284788,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,2.4,16.02,5.620628356933594
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",18,2026220984824,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.56,34.6,5.104224920272827
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",9,2026222684866,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.275,63.3,5.67767333984375
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",172,2026215884825,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.725,64.65,7.176612615585327
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",19,2026220984825,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.88,35.66,5.4147961139678955
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",34,2026219284789,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,2.4925,15.755,5.788932800292969
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",20,2026220984826,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.61,36.64,4.859953165054321
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",10,2026222684867,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.6,64.45,5.7954065799713135
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",173,2026215884826,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.65,65.85,6.00861382484436
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",35,2026219284790,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,2.475,16.045,5.306124687194824
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",21,2026220984827,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.27,35.34,4.830535173416138
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",174,2026215884827,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.525,64.6,5.4952232837677
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",36,2026219284791,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.4325,16.02,5.541334867477417
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",11,2026222684868,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,12.925,64.15,6.000180244445801
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",22,2026220984828,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,7.14,35.8,5.5584187507629395
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",37,2026219284792,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.43,16.36,5.443350076675415
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",175,2026215884828,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,21.475,62.7,5.868619918823242
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",12,2026222684869,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,12.575,65.55,5.745845317840576
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",23,2026220984829,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.85,37.0,5.287667274475098
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",38,2026219284793,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.315,16.08,5.082580804824829
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",176,2026215884829,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.35,63.65,5.389167070388794
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",13,2026222684870,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.375,66.55,5.503266096115112
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",24,2026220984830,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.36,36.48,4.950501441955566
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",39,2026219284794,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.44,15.975,5.355191707611084
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",177,2026215884830,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.975,65.6,5.727900505065918
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",14,2026222684871,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,12.575,64.2,5.57645058631897
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",25,2026220984831,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.08,35.68,5.074901342391968
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",40,2026219284795,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.3625,15.82,5.141162395477295
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",178,2026215884831,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,21.9,65.4,5.718914985656738
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",15,2026222684872,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.025,65.55,5.639533519744873
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",26,2026220984832,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.72,35.56,4.88599705696106
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",41,2026219284796,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,2.2475,16.02,5.670112371444702
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",16,2026222684873,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.9,64.6,5.400624513626099
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",27,2026220984833,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.49,37.32,5.747471809387207
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",179,2026215884832,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.85,64.0,7.630141735076904
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",42,2026219284797,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.3375,15.895,6.266385793685913
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",17,2026222684874,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.5,64.1,6.221395492553711
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",28,2026220984834,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,7.01,34.5,5.722588300704956
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",180,2026215884833,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.275,63.5,6.697417259216309
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",43,2026219284798,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,2.385,15.815,5.776348114013672
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",18,2026222684875,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.375,64.35,5.447894096374512
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",29,2026220984835,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.65,35.78,5.438742399215698
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",181,2026215884834,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.475,62.65,6.8946850299835205
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",44,2026219284799,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.3825,16.285,5.794140577316284
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",19,2026222684876,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.075,64.95,5.987037658691406
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",30,2026220984836,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.3,36.22,5.324362754821777
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",45,2026219284800,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,2.3175,15.85,6.030826091766357
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",182,2026215884835,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.025,63.15,7.57349967956543
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",20,2026222684877,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,12.9,65.15,5.8170084953308105
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",31,2026220984837,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,5.92,35.0,5.047253131866455
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",46,2026219284801,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.405,15.925,5.570576906204224
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",32,2026220984838,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,6.44,36.26,4.947677135467529
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",21,2026222684878,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.45,63.7,5.631664037704468
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",183,2026215884836,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.45,62.35,6.375875473022461
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",33,2026220984839,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,7.14,35.12,5.217631101608276
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",47,2026219284802,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.3,15.79,5.708232641220093
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",22,2026222684879,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.45,61.9,5.382875442504883
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",184,2026215884837,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.0,62.45,6.482034206390381
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",34,2026220984840,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.65,36.28,4.699549674987793
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",48,2026219284803,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.44,15.975,5.4656970500946045
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",23,2026222684880,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.3,61.4,5.72079062461853
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",185,2026215884838,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.8,64.8,6.39911150932312
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",35,2026220984841,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,7.31,36.2,5.596837520599365
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",49,2026219284804,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.345,16.125,5.419374465942383
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",24,2026222684881,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.675,63.05,5.6983888149261475
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",36,2026220984842,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.55,35.52,5.204861164093018
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",186,2026215884839,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.725,61.15,6.737611532211304
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",50,2026219284805,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,1.0,2,2,2.4325,16.07,5.599096298217773
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",25,2026222684882,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.2,64.55,5.809615850448608
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",37,2026220984843,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,7.04,36.82,6.177254915237427
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",51,2026219284806,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,2.3825,16.065,6.008243799209595
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",187,2026215884840,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.5,1,1,20.975,62.6,7.164830923080444
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",26,2026222684883,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.0,63.25,5.8644444942474365
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",38,2026220984844,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.38,35.18,5.245349884033203
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",52,2026219284807,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.5575,16.23,5.654297113418579
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",188,2026215884841,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.65,64.3,6.633456230163574
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",27,2026222684884,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.85,65.3,6.214962720870972
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",39,2026220984845,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.27,36.54,5.454651594161987
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",53,2026219284808,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,2.505,16.125,5.7964558601379395
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",189,2026215884842,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.5,65.85,6.534771203994751
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",28,2026222684885,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.05,63.15,6.219721555709839
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",40,2026220984846,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,6.55,36.1,4.973859786987305
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",54,2026219284809,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.5225,16.3,5.5145344734191895
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",190,2026215884843,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,21.325,61.95,5.8975160121917725
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",29,2026222684886,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.075,64.05,6.042306184768677
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",41,2026220984847,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.98,35.82,5.301318883895874
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",55,2026219284810,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,2.3875,16.185,5.348492383956909
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",191,2026215884844,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.95,64.35,5.704437732696533
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",42,2026220984848,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.24,35.38,5.493156671524048
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",30,2026222684887,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.675,63.6,6.257657766342163
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",56,2026219284811,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.4025,16.065,5.525009393692017
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",43,2026220984849,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.03,35.06,5.012498378753662
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",192,2026215884845,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.95,65.6,6.521953582763672
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",31,2026222684888,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.1,62.35,5.994390964508057
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",57,2026219284812,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,2.3525,15.74,5.606656074523926
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",44,2026220984850,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,2.0,2,0,6.5,35.48,5.744270324707031
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",32,2026222684889,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.325,65.0,5.98182487487793
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",193,2026215884846,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.8,63.9,7.575716018676758
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",58,2026219284813,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.4175,16.06,5.890641689300537
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",45,2026220984851,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,6.98,35.98,5.761781215667725
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",33,2026222684890,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.375,63.5,6.245052099227905
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",59,2026219284814,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,2.3575,15.93,5.498586177825928
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",194,2026215884847,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.5,1,1,20.225,64.55,5.7074830532073975
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",46,2026220984852,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,7.03,35.7,5.065977334976196
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",34,2026222684891,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.825,60.65,5.839815378189087
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",60,2026219284815,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.265,15.63,5.53362250328064
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",195,2026215884848,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.375,65.5,5.772543668746948
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",47,2026220984853,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,6.82,36.32,5.568547248840332
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",61,2026219284816,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.5625,16.03,5.575153350830078
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",35,2026222684892,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.5,65.0,6.22465968132019
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",196,2026215884849,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.175,63.75,6.163163661956787
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",48,2026220984854,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.96,36.08,5.7411792278289795
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",62,2026219284817,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,2.3425,15.875,5.377564907073975
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",36,2026222684893,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.225,63.25,5.5267014503479
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",49,2026220984855,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,7.07,35.38,5.406359672546387
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",197,2026215884850,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.85,60.5,7.17250919342041
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",63,2026219284818,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,2.37,16.065,5.6976237297058105
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",37,2026222684894,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.2,65.55,5.73413872718811
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",50,2026220984856,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,7.1,36.14,5.357161045074463
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",198,2026215884851,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.925,64.4,6.894100666046143
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",64,2026219284819,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.35,15.88,5.98348331451416
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",38,2026222684895,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,12.85,63.25,5.896729946136475
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",51,2026220984857,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.07,35.72,5.296159267425537
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",199,2026215884852,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.25,63.0,7.2837913036346436
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",65,2026219284820,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.4025,16.13,5.908086061477661
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",39,2026222684896,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.275,63.75,6.139423370361328
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",52,2026220984858,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,7.07,36.06,5.566713333129883
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",66,2026219284821,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.385,16.055,5.667364835739136
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",200,2026215884853,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.35,63.3,6.636723518371582
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",40,2026222684897,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,12.425,64.1,6.0075342655181885
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",53,2026220984859,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.59,35.74,5.70442008972168
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",67,2026219284822,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.355,15.795,5.745374917984009
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",54,2026220984860,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.74,36.26,5.445117235183716
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",41,2026222684898,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.8,65.35,5.639612436294556
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",201,2026215884854,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.775,63.2,7.146315813064575
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",55,2026220984861,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.04,36.46,4.888702154159546
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",68,2026219284823,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.42,15.94,6.183257579803467
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",42,2026222684899,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.025,63.25,5.781735420227051
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",202,2026215884855,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.35,66.4,6.988763332366943
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",56,2026220984862,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,6.66,37.1,5.352984666824341
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",69,2026219284824,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.36,16.005,5.602080345153809
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",43,2026222684900,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.0,67.25,5.338958501815796
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",203,2026215884856,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.95,65.7,6.09767746925354
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",57,2026220984863,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.28,35.94,4.9440672397613525
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",44,2026222684901,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.925,63.55,5.750667572021484
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",70,2026219284825,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.43,16.08,6.074258804321289
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",204,2026215884857,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.125,65.3,6.476890563964844
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",58,2026220984864,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.32,35.56,5.212386846542358
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",71,2026219284826,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.3975,16.16,5.307548522949219
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",45,2026222684902,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,12.8,64.7,5.582463264465332
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",59,2026220984865,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.6,36.16,4.939311265945435
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",205,2026215884858,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.25,63.9,5.883152008056641
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",72,2026219284827,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.53,16.175,5.411355972290039
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",46,2026222684903,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,12.9,63.7,5.461596488952637
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",60,2026220984866,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.43,36.12,4.929632902145386
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",206,2026215884859,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.025,65.25,5.982927083969116
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",73,2026219284828,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.41,16.32,5.307642698287964
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",47,2026222684904,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.45,62.6,5.576897621154785
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",61,2026220984867,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.61,36.26,4.9194676876068115
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",207,2026215884860,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.35,63.7,5.658159017562866
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",74,2026219284829,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,2.365,16.34,5.942836284637451
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",48,2026222684905,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.55,65.4,5.76624608039856
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",62,2026220984868,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.4,36.3,5.2689173221588135
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",75,2026219284830,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.505,15.985,5.5613014698028564
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",208,2026215884861,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.225,64.2,7.17427396774292
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",63,2026220984869,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.95,35.7,4.769758939743042
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",49,2026222684906,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.2,63.95,6.319067001342773
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",76,2026219284831,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,2.37,15.835,5.614022731781006
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",64,2026220984870,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.36,35.94,5.53623366355896
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",50,2026222684907,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.025,63.55,5.880409002304077
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",209,2026215884862,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.75,63.95,6.686387777328491
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",77,2026219284832,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.3725,15.74,5.411535024642944
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",65,2026220984871,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.32,34.92,5.024106979370117
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",51,2026222684908,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.825,63.15,5.993760108947754
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",210,2026215884863,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,22.125,65.9,6.779100656509399
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",66,2026220984872,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.4,35.28,5.642263650894165
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",78,2026219284833,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,2.38,16.11,5.983801603317261
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",52,2026222684909,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.425,62.45,6.2126305103302
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",211,2026215884864,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.425,66.1,6.5510265827178955
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",67,2026220984873,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.55,35.8,5.271159410476685
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",79,2026219284834,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.43,15.875,5.817038536071777
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",53,2026222684910,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,13.425,63.55,5.760501146316528
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",68,2026220984874,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.79,36.04,5.6060240268707275
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",212,2026215884865,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.65,63.4,7.638058423995972
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",80,2026219284835,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.485,15.98,5.69019341468811
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",54,2026222684911,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.5,1,1,12.9,64.35,6.057015419006348
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",69,2026220984875,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.58,36.36,5.310551404953003
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",81,2026219284836,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,2.3575,16.26,5.616680383682251
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",213,2026215884866,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.2,63.4,6.844810247421265
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",55,2026222684912,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.275,64.35,5.9404449462890625
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",70,2026220984876,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.59,36.62,5.6592793464660645
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",82,2026219284837,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.4775,15.825,5.760545969009399
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",214,2026215884867,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.875,64.1,6.349820613861084
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",56,2026222684913,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.1,61.7,5.615369558334351
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",71,2026220984877,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.74,35.76,5.663421630859375
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",83,2026219284838,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,2.4125,15.85,5.727027893066406
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",215,2026215884868,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.9,63.25,5.841033697128296
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",57,2026222684914,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.5,65.05,5.702852010726929
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",72,2026220984878,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.55,35.32,5.142164707183838
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",84,2026219284839,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.455,15.695,5.680839776992798
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",216,2026215884869,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.025,64.25,6.102254390716553
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",58,2026222684915,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.6,65.5,5.759778022766113
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",73,2026220984879,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.34,36.98,5.066251277923584
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",85,2026219284840,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.455,16.375,5.589211940765381
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",217,2026215884870,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.25,63.4,5.753011226654053
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",59,2026222684916,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.75,64.15,5.484734296798706
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",74,2026220984880,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.6,36.54,5.419606447219849
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",86,2026219284841,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.355,16.23,5.564161062240601
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",60,2026222684917,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.475,62.65,4.360060453414917
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",75,2026220984881,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.25,36.16,4.421462535858154
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",218,2026215884871,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,21.2,63.5,6.438419342041016
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",87,2026219284842,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,2.4775,15.95,3.126969337463379
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",61,2026222684918,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.15,63.15,2.9669930934906006
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",76,2026220984882,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.07,36.3,3.200786590576172
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",88,2026219284843,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.51,15.655,3.5974912643432617
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",219,2026215884872,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.5,65.45,4.836169004440308
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",77,2026220984883,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.37,35.68,3.3870699405670166
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",62,2026222684919,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.25,62.15,4.179912567138672
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",89,2026219284844,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.41,15.99,3.68643856048584
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",78,2026220984884,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.56,35.54,3.4822421073913574
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",220,2026215884873,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.2,64.0,4.353179693222046
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",63,2026222684920,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.725,60.8,4.121631145477295
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",79,2026220984885,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.26,36.68,2.5637171268463135
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",90,2026219284845,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.5275,16.02,4.694512605667114
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",64,2026222684921,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.9,63.55,3.790740966796875
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",221,2026215884874,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.6,63.95,5.073756217956543
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",80,2026220984886,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.32,35.34,3.490525007247925
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",91,2026219284846,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.385,16.12,3.187257766723633
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",65,2026222684922,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.025,61.8,3.6943421363830566
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",81,2026220984887,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.29,35.84,3.151911973953247
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",92,2026219284847,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.4275,16.325,3.401282787322998
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",222,2026215884875,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.825,64.85,5.428880214691162
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",66,2026222684923,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,12.925,64.3,3.0116400718688965
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",82,2026220984888,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.44,36.04,3.6914010047912598
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",93,2026219284848,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.465,15.96,3.820995569229126
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",223,2026215884876,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.725,62.95,3.9772775173187256
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",67,2026222684924,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,13.55,64.55,3.863469362258911
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",94,2026219284849,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.52,16.17,2.9715487957000732
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",83,2026220984889,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.74,35.9,3.706453800201416
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",224,2026215884877,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,20.675,66.05,3.780247688293457
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",68,2026222684925,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.1,63.95,4.317468166351318
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",84,2026220984890,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.53,35.58,2.8908205032348633
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",95,2026219284850,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.5175,15.84,4.6473424434661865
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",85,2026220984891,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.37,36.02,2.907883405685425
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0003|pi0|k1,1,1",225,2026215884878,B,noisy,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,22.5,66.05,4.2274439334869385
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",69,2026222684926,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.55,66.35,3.4480581283569336
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",0,2026224384908,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2375,4.135,1.0571248531341553
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",1,2026224384909,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.27,3.81,0.7063283920288086
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",86,2026220984892,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.87,36.58,2.9403014183044434
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",96,2026219284851,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.4925,15.79,4.464541673660278
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",2,2026224384910,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.2725,4.085,0.7214593887329102
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",3,2026224384911,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.285,3.955,0.8652148246765137
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",70,2026222684927,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.55,62.6,3.9805963039398193
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",4,2026224384912,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.2525,3.945,0.9859015941619873
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",5,2026224384913,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.305,4.08,0.7849910259246826
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",87,2026220984893,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.42,36.58,3.207669496536255
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",6,2026224384914,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2525,3.935,0.7485513687133789
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",97,2026219284852,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,2.4275,16.175,4.06526780128479
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",7,2026224384915,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,3.0,3,0,0.2975,3.805,0.6757354736328125
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",8,2026224384916,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.335,3.73,0.8593080043792725
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",71,2026222684928,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.85,62.75,4.4498395919799805
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",9,2026224384917,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.265,3.93,1.0712828636169434
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",88,2026220984894,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.51,35.88,3.2438840866088867
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",10,2026224384918,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.23,4.05,0.8936662673950195
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",11,2026224384919,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2225,4.015,0.9075217247009277
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",98,2026219284853,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.505,16.12,4.0614333152771
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",12,2026224384920,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2825,3.98,1.0731608867645264
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",72,2026222684929,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.35,64.0,3.655212879180908
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",13,2026224384921,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2575,4.23,1.2453134059906006
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",89,2026220984895,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.73,35.92,4.053105592727661
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",14,2026224384922,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.2525,3.865,0.9380786418914795
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",99,2026219284854,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.43,16.12,3.306772470474243
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",15,2026224384923,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.27,4.07,1.05647611618042
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",16,2026224384924,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.25,4.07,0.7877283096313477
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",73,2026222684930,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,12.775,67.2,3.9922187328338623
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",17,2026224384925,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.245,3.965,0.7810962200164795
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",90,2026220984896,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.73,34.32,3.4127140045166016
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",18,2026224384926,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2625,4.21,0.7520236968994141
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",19,2026224384927,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2925,3.89,1.091024398803711
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",100,2026219284855,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.3575,16.225,4.42052149772644
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",20,2026224384928,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3175,3.92,0.7836048603057861
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",21,2026224384929,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.2975,4.055,0.696662425994873
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",22,2026224384930,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.3025,4.125,0.7265017032623291
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",91,2026220984897,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.63,36.54,4.145488023757935
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",74,2026222684931,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.825,62.6,4.702142953872681
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",23,2026224384931,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2725,4.075,0.9570562839508057
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",101,2026219284856,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,2.5425,15.83,3.445221424102783
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",24,2026224384932,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2825,4.055,0.9786128997802734
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",25,2026224384933,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3,3.95,1.2161591053009033
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",92,2026220984898,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.13,36.94,3.7636547088623047
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",75,2026222684932,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.55,65.5,3.679002046585083
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",26,2026224384934,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3425,3.935,1.241976022720337
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",102,2026219284857,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.3875,15.77,3.6378564834594727
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",27,2026224384935,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.25,4.005,1.4802460670471191
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",93,2026220984899,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.63,35.8,3.1966371536254883
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",28,2026224384936,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.285,3.955,1.2532157897949219
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",76,2026222684933,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.2,64.5,3.744493246078491
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",29,2026224384937,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2825,3.925,0.6989903450012207
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",30,2026224384938,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.31,3.89,1.0304036140441895
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",103,2026219284858,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.6125,16.305,4.259732484817505
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",31,2026224384939,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2525,3.87,1.1423633098602295
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",94,2026220984900,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.29,35.6,3.309601306915283
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",32,2026224384940,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2825,3.99,0.7594430446624756
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",77,2026222684934,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.65,63.6,3.257081985473633
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",33,2026224384941,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3025,3.895,0.8173501491546631
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",34,2026224384942,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2825,3.935,0.7481768131256104
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",104,2026219284859,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.49,15.97,3.686603546142578
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",95,2026220984901,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,6.61,36.48,3.1296944618225098
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",35,2026224384943,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2775,4.03,1.6198139190673828
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",36,2026224384944,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.28,4.05,0.8336746692657471
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",37,2026224384945,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.3,4.065,0.8166763782501221
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",78,2026222684935,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.325,62.5,4.729674577713013
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",96,2026220984902,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.54,36.0,2.8460097312927246
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",105,2026219284860,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,2.3925,15.805,3.696556806564331
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",38,2026224384946,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.3125,4.04,1.3205351829528809
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",39,2026224384947,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2875,4.085,0.9820578098297119
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",40,2026224384948,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2675,3.985,1.1696326732635498
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",106,2026219284861,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.3225,15.95,2.7889833450317383
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",41,2026224384949,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.255,3.895,0.8134109973907471
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",79,2026222684936,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,15.4,64.2,4.68566370010376
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",97,2026220984903,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.51,35.62,4.071251153945923
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",42,2026224384950,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.295,4.005,0.7362124919891357
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",43,2026224384951,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3,3.895,1.018963098526001
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",44,2026224384952,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.23,4.005,1.0080585479736328
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",107,2026219284862,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.35,15.86,4.021894693374634
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",45,2026224384953,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.2925,3.86,1.2031729221343994
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",98,2026220984904,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.56,36.64,3.5834076404571533
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",80,2026222684937,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.5,64.25,3.8617446422576904
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",46,2026224384954,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2425,3.92,1.5318524837493896
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",47,2026224384955,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2875,4.075,1.0818986892700195
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",99,2026220984905,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.8,36.54,3.172354221343994
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",108,2026219284863,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,2.2775,16.25,3.8691468238830566
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",48,2026224384956,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.305,3.92,1.1315479278564453
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",81,2026222684938,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,12.4,64.55,3.531777858734131
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",49,2026224384957,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2925,3.965,1.1805589199066162
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",50,2026224384958,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2925,3.965,1.1664752960205078
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",51,2026224384959,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2425,3.98,1.0612664222717285
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",109,2026219284864,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.435,16.045,3.5598039627075195
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",82,2026222684939,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.275,64.8,3.249117612838745
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",100,2026220984906,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.47,36.22,3.9345781803131104
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",52,2026224384960,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2725,3.985,0.760688304901123
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",53,2026224384961,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2975,4.11,0.7648532390594482
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",54,2026224384962,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.32,3.745,0.86623215675354
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",110,2026219284865,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.4925,16.5,3.568521022796631
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",83,2026222684940,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,15.025,64.6,3.5623273849487305
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",55,2026224384963,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3275,3.96,1.407221794128418
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",101,2026220984907,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.59,36.26,3.95151424407959
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",56,2026224384964,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.255,3.975,0.8163368701934814
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",57,2026224384965,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2275,4.095,0.835761308670044
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",111,2026219284866,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,2.235,15.985,2.949125051498413
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",58,2026224384966,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3125,4.045,1.4984796047210693
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",84,2026222684941,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.7,63.15,3.867427110671997
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",59,2026224384967,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2825,3.85,0.7505481243133545
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",102,2026220984908,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.44,35.72,4.391798496246338
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",60,2026224384968,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3175,3.875,1.4949963092803955
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",61,2026224384969,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.28,3.88,1.0236740112304688
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",112,2026219284867,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.375,16.175,3.763399600982666
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",85,2026222684942,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.15,64.2,3.5910959243774414
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",62,2026224384970,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.27,4.27,0.9664380550384521
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",103,2026220984909,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.2,37.26,3.249403715133667
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",63,2026224384971,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.315,3.97,0.9917993545532227
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",64,2026224384972,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.2925,3.935,0.9984712600708008
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",113,2026219284868,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.41,16.14,3.739097833633423
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",65,2026224384973,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.25,4.03,0.8351421356201172
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",86,2026222684943,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.225,67.7,3.9527347087860107
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",66,2026224384974,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.255,4.095,1.0034010410308838
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",104,2026220984910,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.2,35.16,3.4272332191467285
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",67,2026224384975,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2825,3.995,0.7950661182403564
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",68,2026224384976,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.295,4.15,0.7881336212158203
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",69,2026224384977,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.275,4.1,0.7826805114746094
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",114,2026219284869,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,2.41,15.885,4.028401851654053
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",70,2026224384978,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.2725,4.045,0.7171330451965332
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",105,2026220984911,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.9,36.58,3.858240842819214
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",71,2026224384979,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2325,4.095,0.9022955894470215
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",87,2026222684944,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.2,64.75,4.08459734916687
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",72,2026224384980,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.335,3.965,0.7106351852416992
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",73,2026224384981,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.28,3.845,0.9558799266815186
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",74,2026224384982,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.275,3.85,0.7686004638671875
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",115,2026219284870,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.5075,16.095,4.00462794303894
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",75,2026224384983,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2275,3.99,1.1297929286956787
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",88,2026222684945,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,12.825,63.05,3.6379470825195312
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",106,2026220984912,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.22,36.1,3.757019281387329
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",76,2026224384984,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.275,3.845,1.2496728897094727
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",77,2026224384985,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.27,4.095,0.969602108001709
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",116,2026219284871,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.4,15.83,3.794438362121582
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",78,2026224384986,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.26,4.05,1.0894739627838135
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",89,2026222684946,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.05,65.6,3.4952287673950195
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",107,2026220984913,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.54,35.18,3.788698196411133
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",79,2026224384987,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2875,4.05,1.1345603466033936
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",80,2026224384988,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.265,3.97,0.7269911766052246
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",81,2026224384989,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.265,3.92,0.7830455303192139
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",82,2026224384990,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.235,4.04,0.9433209896087646
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",108,2026220984914,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.82,36.12,3.170966148376465
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",90,2026222684947,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.25,63.2,3.4787070751190186
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",117,2026219284872,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,2.4025,16.065,4.385301828384399
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",83,2026224384991,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2775,3.91,0.7915527820587158
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",84,2026224384992,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.26,4.07,0.9398248195648193
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",85,2026224384993,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.27,3.955,0.9176344871520996
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",86,2026224384994,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.27,3.925,0.7007639408111572
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",109,2026220984915,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,6.73,35.94,3.599412441253662
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",91,2026222684948,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.475,64.35,3.810183525085449
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",118,2026219284873,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.4275,16.435,3.7994461059570312
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",87,2026224384995,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.275,3.935,1.3109731674194336
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",88,2026224384996,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2925,4.02,1.221731185913086
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",89,2026224384997,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3175,3.9,1.1974449157714844
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",92,2026222684949,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.275,65.9,3.288450241088867
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",110,2026220984916,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,7.15,36.32,3.5843489170074463
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",119,2026219284874,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.4475,16.035,3.1620705127716064
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",90,2026224384998,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.29,3.745,0.9109675884246826
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",91,2026224384999,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.3125,4.07,0.9533085823059082
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",92,2026224385000,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3225,4.055,1.008734941482544
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",111,2026220984917,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.13,36.52,3.417980432510376
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",93,2026224385001,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3025,3.98,0.8390522003173828
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",93,2026222684950,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.15,62.9,3.7360527515411377
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",94,2026224385002,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2825,4.135,0.8746337890625
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",120,2026219284875,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,2.345,16.01,4.19413948059082
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",95,2026224385003,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.245,3.905,1.010347604751587
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",96,2026224385004,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.28,4.11,1.165100336074829
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",112,2026220984918,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.99,36.24,3.2844176292419434
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",97,2026224385005,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.22,4.155,0.7154150009155273
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",94,2026222684951,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,12.5,62.25,3.8289363384246826
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",98,2026224385006,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2725,4.03,0.701063871383667
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",121,2026219284876,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.45,15.795,4.104684591293335
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",99,2026224385007,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3075,3.985,1.2618951797485352
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",113,2026220984919,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,5.84,36.88,3.7215769290924072
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",100,2026224385008,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.285,3.745,1.591318130493164
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",95,2026222684952,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.6,62.55,3.744213819503784
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",122,2026219284877,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.5425,15.95,2.9268038272857666
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",101,2026224385009,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.265,3.945,1.1477222442626953
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",102,2026224385010,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.29,3.935,0.8887574672698975
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",114,2026220984920,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.53,36.32,2.4967615604400635
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",103,2026224385011,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.255,3.95,0.7376046180725098
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",104,2026224385012,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2925,3.965,0.7005794048309326
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",123,2026219284878,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.4225,15.69,3.470324993133545
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",105,2026224385013,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.2925,4.06,1.1412696838378906
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",96,2026222684953,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,12.975,62.35,4.705650091171265
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",106,2026224385014,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.245,3.935,0.8601834774017334
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",115,2026220984921,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.32,37.84,3.6838502883911133
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",107,2026224385015,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2575,3.835,0.8252894878387451
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",108,2026224385016,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.235,3.76,1.4155359268188477
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",124,2026219284879,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,2.5675,15.94,3.96374249458313
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",116,2026220984922,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.37,35.5,2.803391933441162
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",97,2026222684954,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.975,63.7,3.490666389465332
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",109,2026224385017,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.28,4.02,1.0288608074188232
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",110,2026224385018,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.235,4.01,0.7235457897186279
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",111,2026224385019,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3075,4.06,0.8295509815216064
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",112,2026224385020,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2675,4.02,0.9213778972625732
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",125,2026219284880,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.57,16.03,3.5737781524658203
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",117,2026220984923,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.6,36.16,3.54734206199646
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",113,2026224385021,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.26,4.04,1.109114408493042
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",98,2026222684955,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.95,64.95,3.904343366622925
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",114,2026224385022,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.275,4.05,0.8918054103851318
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",115,2026224385023,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2625,4.015,0.9617938995361328
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",126,2026219284881,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.45,15.795,3.0242743492126465
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",118,2026220984924,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.42,35.5,2.9363715648651123
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",116,2026224385024,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.21,4.07,1.136817216873169
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",117,2026224385025,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.2725,4.035,1.088172435760498
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",99,2026222684956,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.5,63.8,4.513301372528076
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",118,2026224385026,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3075,3.975,0.743966817855835
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",119,2026224385027,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2625,4.06,0.7421004772186279
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",127,2026219284882,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.3575,15.99,3.436105728149414
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",119,2026220984925,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.66,36.02,3.5625977516174316
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",120,2026224385028,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2725,4.065,1.1261322498321533
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",121,2026224385029,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.27,4.065,0.7548604011535645
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",122,2026224385030,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2675,3.855,0.9230892658233643
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",100,2026222684957,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.65,64.75,3.697287082672119
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",123,2026224385031,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.305,3.975,0.7666196823120117
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",128,2026219284883,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,2.465,16.05,3.829169511795044
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",120,2026220984926,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.61,35.18,3.3749265670776367
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",124,2026224385032,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.29,3.91,1.3361308574676514
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",125,2026224385033,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.285,3.86,1.2142620086669922
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",101,2026222684958,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.2,66.0,3.6440014839172363
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",126,2026224385034,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.27,4.115,1.2940599918365479
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",129,2026219284884,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.2775,16.11,3.3307266235351562
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",121,2026220984927,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,6.33,37.0,3.4615278244018555
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",127,2026224385035,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.2925,4.2,1.4797234535217285
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",128,2026224385036,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.255,4.07,0.9090981483459473
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",130,2026219284885,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.465,16.51,2.836073637008667
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",102,2026222684959,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.925,63.3,3.961674213409424
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",122,2026220984928,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.7,36.0,3.1812915802001953
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",129,2026224385037,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.2825,4.04,1.0759961605072021
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",130,2026224385038,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.26,4.07,1.0851762294769287
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",123,2026220984929,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.24,36.54,2.232600688934326
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",131,2026224385039,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.28,4.085,1.3748438358306885
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",131,2026219284886,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,2.31,16.2,3.3067209720611572
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",132,2026224385040,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2825,3.975,1.6083028316497803
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",103,2026222684960,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.85,64.0,4.905042886734009
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",124,2026220984930,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.57,36.68,2.6685397624969482
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",133,2026224385041,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.29,3.96,0.8226163387298584
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",134,2026224385042,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.245,3.96,0.996262788772583
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",132,2026219284887,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.4375,15.92,3.7334747314453125
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",135,2026224385043,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.265,3.905,0.8742060661315918
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",136,2026224385044,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.26,4.065,0.8826730251312256
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",104,2026222684961,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.95,63.8,3.314126491546631
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",125,2026220984931,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.83,35.8,4.023884296417236
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",137,2026224385045,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2525,3.96,1.2148394584655762
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",138,2026224385046,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2875,3.965,0.89727783203125
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",133,2026219284888,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.55,16.145,3.7117507457733154
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",139,2026224385047,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2475,3.945,1.0606136322021484
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",105,2026222684962,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.25,64.3,4.069646835327148
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",140,2026224385048,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2675,4.045,1.0788867473602295
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",126,2026220984932,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.92,36.14,3.6022441387176514
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",141,2026224385049,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2575,3.94,1.0851411819458008
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",134,2026219284889,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.345,16.405,3.296987771987915
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",142,2026224385050,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.29,3.85,0.9110281467437744
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",143,2026224385051,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2425,3.965,1.031214952468872
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",106,2026222684963,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.65,66.3,3.465930461883545
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",144,2026224385052,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.3125,4.15,1.0299551486968994
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",127,2026220984933,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.32,35.64,3.6701366901397705
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",145,2026224385053,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.2725,4.175,1.0027172565460205
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",135,2026219284890,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.2775,16.33,3.9273903369903564
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",146,2026224385054,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.275,4.035,1.005066156387329
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",107,2026222684964,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.025,64.1,2.998342514038086
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",147,2026224385055,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.2525,3.875,0.6919050216674805
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",128,2026220984934,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.95,35.86,3.4639878273010254
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",148,2026224385056,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.305,4.025,1.0617961883544922
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",149,2026224385057,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.5,2,1,0.265,3.915,1.4652175903320312
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",108,2026222684965,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.1,63.85,2.937983512878418
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",136,2026219284891,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,2.4325,15.83,4.485744476318359
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",129,2026220984935,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.75,36.2,2.4547319412231445
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",150,2026224385058,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2775,4.1,1.4892969131469727
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",151,2026224385059,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.32,4.01,1.4114303588867188
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",130,2026220984936,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,7.32,35.2,2.7578623294830322
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",152,2026224385060,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3,3.94,0.8344094753265381
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",137,2026219284892,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.6675,15.835,3.1358847618103027
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",109,2026222684966,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,12.375,63.7,4.254361391067505
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",153,2026224385061,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.295,4.12,0.9030683040618896
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",154,2026224385062,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.2175,3.915,1.0277037620544434
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",155,2026224385063,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.27,3.97,1.1622250080108643
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",131,2026220984937,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,6.54,36.8,3.674870491027832
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",138,2026219284893,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.49,15.765,3.918574571609497
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",156,2026224385064,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2775,4.095,1.0449976921081543
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",110,2026222684967,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.675,63.45,3.933511257171631
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",157,2026224385065,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.29,3.985,1.201559066772461
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",158,2026224385066,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.2775,3.84,0.7430105209350586
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",132,2026220984938,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.22,36.12,2.93325138092041
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",159,2026224385067,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.33,3.9,0.8394694328308105
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",139,2026219284894,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.375,15.885,3.7895476818084717
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",111,2026222684968,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.15,65.35,3.5913331508636475
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",160,2026224385068,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.295,3.875,1.2588739395141602
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",161,2026224385069,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.32,4.11,1.387218952178955
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",133,2026220984939,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,6.88,35.14,3.587374687194824
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",162,2026224385070,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.235,4.16,0.866847038269043
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",140,2026219284895,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.4225,15.725,3.5235698223114014
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",112,2026222684969,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.075,62.75,3.607077121734619
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",163,2026224385071,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2675,4.01,1.533353328704834
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",164,2026224385072,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.305,3.945,1.0965819358825684
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",134,2026220984940,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.23,36.24,3.619805097579956
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",165,2026224385073,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.32,3.91,0.7148799896240234
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",166,2026224385074,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3,3.83,1.0320446491241455
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",141,2026219284896,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.3325,16.185,3.8940234184265137
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",113,2026222684970,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,13.1,63.65,3.6912930011749268
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",167,2026224385075,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2425,4.005,1.2866668701171875
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",135,2026220984941,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.47,35.48,3.05844783782959
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",168,2026224385076,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.305,4.17,1.2640221118927002
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",169,2026224385077,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3,4.065,1.454575538635254
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",114,2026222684971,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.4,63.9,3.536874771118164
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",142,2026219284897,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.5,16.16,3.9072484970092773
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",136,2026220984942,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.93,36.68,2.8346920013427734
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",170,2026224385078,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.28,3.95,0.7052206993103027
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",171,2026224385079,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.26,4.155,1.0251028537750244
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",172,2026224385080,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3075,3.915,1.1173408031463623
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",115,2026222684972,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.875,62.1,3.4117558002471924
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",173,2026224385081,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3275,3.775,1.0455975532531738
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",137,2026220984943,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.51,35.48,3.240342378616333
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",143,2026219284898,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.4,16.11,3.9584083557128906
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",174,2026224385082,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3,4.075,1.2262444496154785
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",175,2026224385083,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2975,4.025,1.1935040950775146
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",116,2026222684973,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.375,64.6,3.1531577110290527
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",176,2026224385084,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.225,4.05,0.8579916954040527
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",144,2026219284899,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,2.3925,16.02,3.3037190437316895
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",177,2026224385085,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2425,4.135,0.6909370422363281
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",138,2026220984944,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.91,35.68,4.465848684310913
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",178,2026224385086,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.235,3.78,0.7078349590301514
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",179,2026224385087,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2975,3.975,0.9093024730682373
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",117,2026222684974,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.8,61.95,3.7077226638793945
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",180,2026224385088,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.265,3.925,0.9994938373565674
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",145,2026219284900,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.355,16.195,3.570341110229492
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",181,2026224385089,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.255,4.04,0.8623862266540527
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",139,2026220984945,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.44,36.54,3.168419599533081
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",182,2026224385090,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.3025,3.75,1.189300298690796
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",183,2026224385091,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.245,3.93,0.9003763198852539
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",184,2026224385092,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.29,4.03,0.681509256362915
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",118,2026222684975,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,12.275,63.55,3.944812297821045
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",140,2026220984946,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.48,36.78,2.941251754760742
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",146,2026219284901,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.3675,15.83,3.5214951038360596
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",185,2026224385093,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.27,3.94,0.924206018447876
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",186,2026224385094,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.275,4.105,0.82289719581604
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",187,2026224385095,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.3,4.015,1.0594894886016846
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",141,2026220984947,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.43,35.9,3.3895204067230225
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",188,2026224385096,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2875,4.105,0.958198070526123
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",119,2026222684976,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.05,63.95,3.785224199295044
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",147,2026219284902,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.455,16.205,3.5619351863861084
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",189,2026224385097,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.2825,4.01,1.1899933815002441
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",190,2026224385098,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.28,4.0,0.7411086559295654
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",191,2026224385099,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.285,4.0,0.7316460609436035
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",142,2026220984948,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.76,36.76,2.857013702392578
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",192,2026224385100,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.255,3.945,0.8839221000671387
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",193,2026224385101,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2925,3.95,0.9098296165466309
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",120,2026222684977,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.45,65.65,4.288002967834473
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",148,2026219284903,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,2.54,15.985,5.020376205444336
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",194,2026224385102,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.27,4.025,0.9686930179595947
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",143,2026220984949,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,7.41,36.3,3.851142168045044
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",195,2026224385103,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2425,4.05,1.2511978149414062
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",121,2026222684978,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,12.325,65.8,3.556962013244629
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",196,2026224385104,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2625,4.045,1.3936190605163574
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",149,2026219284904,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.4875,16.145,4.013275623321533
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",144,2026220984950,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.0,35.86,2.701772689819336
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",197,2026224385105,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2975,4.015,1.514765739440918
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",198,2026224385106,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2925,3.93,0.8801863193511963
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",122,2026222684979,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,15.2,64.15,2.899975061416626
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",199,2026224385107,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.25,4.005,0.7618539333343506
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",200,2026224385108,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.28,3.82,0.9738812446594238
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",145,2026220984951,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.59,36.92,3.6633048057556152
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",201,2026224385109,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2775,4.02,1.285243272781372
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",150,2026219284905,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.3725,16.01,4.490077257156372
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",202,2026224385110,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2275,3.985,0.9819440841674805
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",123,2026222684980,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.85,62.35,3.656144380569458
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",203,2026224385111,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.26,3.93,0.8930401802062988
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",204,2026224385112,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.325,3.965,1.3264544010162354
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",146,2026220984952,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,7.14,35.04,3.7440013885498047
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",151,2026219284906,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,2.4825,16.05,4.073822975158691
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",205,2026224385113,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.27,4.045,1.2231848239898682
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",206,2026224385114,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2975,3.975,0.732734203338623
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",124,2026222684981,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.775,60.8,4.495797157287598
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",207,2026224385115,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.23,3.805,0.7828006744384766
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",208,2026224385116,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.25,4.035,0.9899864196777344
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",147,2026220984953,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.62,35.36,3.801537275314331
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",152,2026219284907,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.4175,15.945,3.5348973274230957
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",209,2026224385117,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.2625,4.035,1.5181763172149658
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",125,2026222684982,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.8,63.5,3.3894481658935547
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",210,2026224385118,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.305,3.865,1.5246632099151611
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",148,2026220984954,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.02,36.22,3.123231887817383
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",211,2026224385119,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.25,4.04,0.9317309856414795
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",153,2026219284908,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.4125,15.975,3.875455856323242
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",126,2026222684983,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.25,66.05,3.3764984607696533
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",212,2026224385120,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.26,3.84,1.4751513004302979
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",213,2026224385121,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.295,4.08,1.127983570098877
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",149,2026220984955,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.56,35.66,4.071758031845093
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",214,2026224385122,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.255,4.2,1.0872678756713867
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",154,2026219284909,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.65,15.97,2.912071466445923
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",215,2026224385123,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2875,4.11,1.2935185432434082
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",127,2026222684984,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.975,64.25,3.8210394382476807
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",216,2026224385124,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.3,4.01,1.2655377388000488
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",155,2026219284910,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.4,16.175,3.0049099922180176
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",150,2026220984956,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.48,35.44,3.3930788040161133
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",217,2026224385125,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2425,4.02,1.004223108291626
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",218,2026224385126,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3175,3.975,0.7043545246124268
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",219,2026224385127,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.24,3.8,0.694434404373169
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",220,2026224385128,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2575,4.04,0.7310817241668701
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",156,2026219284911,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,2.535,16.125,2.6466448307037354
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",128,2026222684985,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.95,64.55,4.775676727294922
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",221,2026224385129,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.285,4.05,0.6894998550415039
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",222,2026224385130,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.27,3.985,0.7179937362670898
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",151,2026220984957,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.52,35.52,4.908143997192383
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",223,2026224385131,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2375,4.045,1.168128490447998
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",157,2026219284912,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.395,15.96,3.208528518676758
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",224,2026224385132,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.28,3.99,1.2523119449615479
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",129,2026222684986,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.4,63.5,3.683136224746704
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",225,2026224385133,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.295,4.065,1.2730600833892822
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",152,2026220984958,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.72,35.98,3.1088974475860596
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",226,2026224385134,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.245,3.96,0.7105197906494141
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",227,2026224385135,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.32,4.01,1.1531810760498047
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",158,2026219284913,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.3225,15.82,3.866748809814453
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",130,2026222684987,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.825,63.5,3.1482620239257812
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",228,2026224385136,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2625,4.07,1.3071346282958984
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",153,2026220984959,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.86,35.6,3.890385866165161
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",229,2026224385137,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2725,4.01,0.9934782981872559
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",230,2026224385138,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2625,4.13,0.9347579479217529
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",159,2026219284914,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.395,16.24,3.248284101486206
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",131,2026222684988,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,15.1,62.4,3.6890604496002197
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",231,2026224385139,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.265,3.885,1.2382433414459229
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",232,2026224385140,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.25,3.93,0.9427225589752197
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",233,2026224385141,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.28,4.09,0.6794886589050293
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",154,2026220984960,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.59,36.58,4.048859119415283
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",160,2026219284915,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,2.415,15.985,3.390515089035034
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",234,2026224385142,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2875,4.04,0.7885339260101318
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",132,2026222684989,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.75,63.45,3.8930022716522217
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",235,2026224385143,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2575,3.965,1.1975147724151611
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",236,2026224385144,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2475,4.06,1.2204930782318115
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",155,2026220984961,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.51,36.02,3.3796091079711914
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",237,2026224385145,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.3075,3.92,0.9545905590057373
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",161,2026219284916,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,2.4825,16.09,3.8055691719055176
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",238,2026224385146,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.26,4.22,0.9130959510803223
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",133,2026222684990,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,12.85,63.9,3.3578312397003174
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",239,2026224385147,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.27,3.94,0.941253662109375
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",156,2026220984962,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.57,36.92,3.2919859886169434
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",240,2026224385148,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.27,4.07,1.437837839126587
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",162,2026219284917,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.4875,16.065,3.6137423515319824
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",241,2026224385149,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.22,4.005,1.0933825969696045
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",134,2026222684991,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.0,62.4,3.9755213260650635
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",242,2026224385150,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2525,3.965,0.7691080570220947
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",243,2026224385151,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.32,3.98,0.7347795963287354
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",157,2026220984963,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.76,37.0,3.9049978256225586
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",244,2026224385152,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.285,3.855,0.9365439414978027
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",163,2026219284918,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.6275,15.89,3.4451024532318115
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",245,2026224385153,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.3125,3.915,1.0337247848510742
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",246,2026224385154,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2475,4.13,0.970613956451416
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",135,2026222684992,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.225,65.1,3.7625656127929688
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",247,2026224385155,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.3175,3.99,0.7518236637115479
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",158,2026220984964,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,7.11,37.12,3.4870924949645996
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",248,2026224385156,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.2975,4.025,0.9136652946472168
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",164,2026219284919,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.5425,16.025,3.917231559753418
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",249,2026224385157,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.265,3.98,1.4203860759735107
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",250,2026224385158,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2775,4.03,0.8754177093505859
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",136,2026222684993,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.625,65.15,3.986569404602051
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",159,2026220984965,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.46,36.14,3.447791814804077
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",251,2026224385159,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.245,3.97,0.9749135971069336
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",252,2026224385160,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.255,4.04,0.7938616275787354
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",165,2026219284920,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.3,16.275,3.4495468139648438
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",253,2026224385161,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2675,4.005,1.0601601600646973
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",254,2026224385162,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.275,3.915,0.9597291946411133
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",160,2026220984966,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.8,35.92,3.2130916118621826
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",137,2026222684994,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.4,63.7,4.29491662979126
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",255,2026224385163,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2925,3.95,1.3071272373199463
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",166,2026219284921,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.4125,16.25,3.942516803741455
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",256,2026224385164,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2625,3.98,1.0936977863311768
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",257,2026224385165,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.31,4.005,1.0768804550170898
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",138,2026222684995,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,14.4,62.65,3.5256950855255127
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",161,2026220984967,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.96,34.7,4.210234642028809
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",258,2026224385166,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.265,3.895,1.0654802322387695
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",259,2026224385167,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,2,2,0.2725,3.955,0.9697158336639404
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",167,2026219284922,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,2.345,16.085,3.9030203819274902
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",260,2026224385168,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.28,3.86,1.1222004890441895
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",162,2026220984968,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.4,36.04,3.05538272857666
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",139,2026222684996,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.325,63.9,3.970182418823242
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",261,2026224385169,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2675,4.1,1.5238049030303955
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",262,2026224385170,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.245,4.02,1.3370978832244873
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",168,2026219284923,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.5175,16.285,3.6519689559936523
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",263,2026224385171,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2775,4.105,0.780015230178833
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",163,2026220984969,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.02,36.06,2.942892551422119
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",264,2026224385172,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.2675,4.175,0.9811239242553711
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",140,2026222684997,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.925,64.85,3.6923890113830566
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",265,2026224385173,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2475,4.135,1.092904806137085
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",266,2026224385174,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2775,3.925,1.2849812507629395
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",164,2026220984970,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.49,35.44,3.3791961669921875
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",169,2026219284924,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.505,15.98,4.331745147705078
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",267,2026224385175,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3025,4.085,0.8705315589904785
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",141,2026222684998,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.475,64.05,3.4844233989715576
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",268,2026224385176,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.25,4.16,0.9304141998291016
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",269,2026224385177,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.275,3.95,1.0324535369873047
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",165,2026220984971,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.75,36.24,3.049203395843506
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",270,2026224385178,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2625,4.08,1.2793972492218018
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",170,2026219284925,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,2.0,2,0,2.3475,16.07,4.304825067520142
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",271,2026224385179,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.28,4.055,0.9944255352020264
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",142,2026222684999,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.05,64.8,3.799978256225586
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",272,2026224385180,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2825,4.065,0.8622770309448242
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",166,2026220984972,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.48,36.44,3.4305691719055176
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",273,2026224385181,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.27,3.86,0.7412662506103516
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",274,2026224385182,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.24,3.82,1.3264803886413574
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",171,2026219284926,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.445,16.135,4.251230239868164
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",143,2026222685000,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.35,62.6,3.49568247795105
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",275,2026224385183,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.27,4.08,1.383286952972412
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",167,2026220984973,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.32,35.04,3.288099527359009
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",276,2026224385184,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.265,4.155,0.7713072299957275
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",277,2026224385185,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.23,4.015,0.7586669921875
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",278,2026224385186,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.275,3.86,0.8881347179412842
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",144,2026222685001,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.1,61.85,3.557493209838867
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",279,2026224385187,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2925,3.89,0.8917102813720703
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",172,2026219284927,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.3675,16.31,4.090564489364624
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",168,2026220984974,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.57,36.06,3.7592062950134277
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",280,2026224385188,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.285,4.05,1.1827495098114014
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",281,2026224385189,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.27,4.205,0.7475621700286865
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",282,2026224385190,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.255,3.9,1.2389578819274902
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",145,2026222685002,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,12.85,64.9,3.9821736812591553
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",283,2026224385191,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.5,2,1,0.2725,4.095,0.853224515914917
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",173,2026219284928,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.42,16.22,3.9398865699768066
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",169,2026220984975,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.47,35.56,3.64744234085083
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",284,2026224385192,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.275,3.945,0.9369826316833496
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",285,2026224385193,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2875,4.0,1.020141839981079
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",286,2026224385194,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.25,4.14,0.7342031002044678
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",170,2026220984976,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.59,36.58,2.995237112045288
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",146,2026222685003,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.075,67.0,4.062473297119141
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",287,2026224385195,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.2675,3.965,1.241495132446289
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",174,2026219284929,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,2.3625,15.915,4.48841404914856
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",288,2026224385196,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.235,4.005,0.7189610004425049
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",289,2026224385197,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.28,3.97,0.9812498092651367
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",171,2026220984977,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.9,35.66,3.4250223636627197
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",290,2026224385198,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3,4.03,1.543461799621582
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",147,2026222685004,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.85,62.5,3.7784311771392822
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",291,2026224385199,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2925,3.96,0.7079341411590576
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",175,2026219284930,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.455,16.0,4.011534929275513
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",292,2026224385200,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2775,4.105,0.691756010055542
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",293,2026224385201,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.2375,3.945,0.6650211811065674
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",294,2026224385202,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.28,3.855,0.707007646560669
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",172,2026220984978,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.45,36.24,3.1629018783569336
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",295,2026224385203,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3075,4.055,1.1318933963775635
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",148,2026222685005,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,13.75,63.4,4.216431140899658
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",296,2026224385204,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3375,4.125,1.0235278606414795
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",176,2026219284931,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.38,16.195,4.154892444610596
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",297,2026224385205,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.32,3.785,1.1064043045043945
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",173,2026220984979,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.63,36.58,3.402292251586914
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",298,2026224385206,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.29,4.195,0.7502250671386719
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",299,2026224385207,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2575,4.075,1.3051986694335938
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",149,2026222685006,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.7,65.45,3.7147281169891357
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",300,2026224385208,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.255,4.01,0.8994040489196777
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",177,2026219284932,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,2.3925,16.175,3.5125463008880615
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",301,2026224385209,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2775,3.93,0.9288890361785889
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",174,2026220984980,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,6.32,35.96,4.034773111343384
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",302,2026224385210,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.2525,3.91,0.7123281955718994
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",303,2026224385211,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2175,3.985,0.7425696849822998
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",150,2026222685007,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.625,64.15,3.400731086730957
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",304,2026224385212,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3075,4.0,0.8841540813446045
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",305,2026224385213,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.28,4.095,0.7326719760894775
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",178,2026219284933,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.45,16.055,4.510780334472656
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",306,2026224385214,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.305,4.075,0.7306134700775146
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",175,2026220984981,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.6,35.42,4.394385814666748
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",307,2026224385215,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.305,4.235,1.1799144744873047
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",151,2026222685008,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.125,63.55,4.117509365081787
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",308,2026224385216,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2525,4.175,1.2815580368041992
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",179,2026219284934,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.3275,16.12,2.672333240509033
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",176,2026220984982,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.17,36.12,2.335999011993408
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",309,2026224385217,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.28,3.965,1.2090179920196533
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",310,2026224385218,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.295,4.105,1.0699679851531982
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",152,2026222685009,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,12.775,64.2,3.2039084434509277
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",311,2026224385219,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2675,4.005,0.9553837776184082
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",312,2026224385220,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.2775,4.015,0.7154855728149414
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",180,2026219284935,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.4525,15.85,4.147319793701172
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",177,2026220984983,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.55,37.02,3.497088670730591
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",313,2026224385221,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.305,4.03,0.7550911903381348
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",314,2026224385222,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2375,4.15,1.015148639678955
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",153,2026222685010,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.275,61.5,3.618215322494507
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",315,2026224385223,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.325,3.945,1.1852200031280518
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",178,2026220984984,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.58,35.04,2.816715717315674
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",316,2026224385224,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2825,4.21,0.773737907409668
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",317,2026224385225,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.22,4.02,0.7458047866821289
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",181,2026219284936,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,2.395,15.825,4.4978249073028564
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",318,2026224385226,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2375,4.075,1.2595250606536865
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",154,2026222685011,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.4,63.6,3.6334621906280518
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",179,2026220984985,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.06,34.88,3.161818265914917
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",319,2026224385227,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2425,4.005,1.5464820861816406
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",182,2026219284937,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.435,16.33,3.103846311569214
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",320,2026224385228,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2825,3.915,1.225386142730713
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",321,2026224385229,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.2725,3.995,0.7190141677856445
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",180,2026220984986,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.49,36.32,3.2830162048339844
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",155,2026222685012,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.025,63.4,3.8473281860351562
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",322,2026224385230,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.2675,3.89,0.9187064170837402
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",323,2026224385231,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2825,4.02,1.2860896587371826
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",183,2026219284938,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,2.47,16.04,3.8535070419311523
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",324,2026224385232,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2825,4.085,0.8961348533630371
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",325,2026224385233,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.26,3.875,1.1903131008148193
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",181,2026220984987,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.4,36.84,4.123376131057739
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",156,2026222685013,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.875,64.6,3.925032377243042
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",326,2026224385234,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.285,4.11,0.8890278339385986
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",184,2026219284939,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.3125,16.005,2.743319272994995
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",327,2026224385235,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.265,4.03,1.577329397201538
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",157,2026222685014,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,12.875,63.5,2.7092459201812744
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",185,2026219284940,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.34,16.13,2.62298321723938
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",328,2026224385236,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.25,4.09,1.4965980052947998
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",329,2026224385237,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2775,3.955,0.8993813991546631
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",182,2026220984988,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.57,36.62,4.65952467918396
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",330,2026224385238,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2425,3.865,0.8415570259094238
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",331,2026224385239,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.275,4.04,0.8472092151641846
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",158,2026222685015,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.4,66.3,3.8833024501800537
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",186,2026219284941,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.4425,15.985,3.5767080783843994
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",332,2026224385240,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3325,3.805,0.8319051265716553
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",183,2026220984989,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,6.42,36.22,2.968395709991455
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",333,2026224385241,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.25,4.025,0.9348905086517334
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",334,2026224385242,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.5,2,1,0.3,4.07,0.9880967140197754
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",335,2026224385243,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3275,3.995,1.1761915683746338
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",184,2026220984990,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.66,35.78,3.078270673751831
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",159,2026222685016,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.775,64.2,4.148479700088501
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",187,2026219284942,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.4225,15.925,4.103486061096191
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",336,2026224385244,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.245,4.095,1.0417532920837402
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",337,2026224385245,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.2575,3.935,0.9858553409576416
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",338,2026224385246,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.26,3.955,0.6811482906341553
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",339,2026224385247,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2525,4.065,0.6847507953643799
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",188,2026219284943,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.325,16.07,3.2029170989990234
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",340,2026224385248,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.29,4.055,0.7141261100769043
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",185,2026220984991,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.41,35.94,4.394904613494873
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",341,2026224385249,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3175,4.025,1.0271763801574707
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",160,2026222685017,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.375,66.05,4.8607916831970215
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",342,2026224385250,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.31,4.015,1.276038408279419
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",343,2026224385251,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2625,4.045,0.7631118297576904
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",189,2026219284944,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.3925,15.89,3.6850404739379883
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",186,2026220984992,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.81,36.48,2.7748732566833496
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",344,2026224385252,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2675,4.05,0.7415246963500977
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",345,2026224385253,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.275,4.075,0.755786657333374
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",346,2026224385254,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.25,3.945,0.9248723983764648
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",161,2026222685018,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.25,65.1,4.217453956604004
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",347,2026224385255,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3,3.975,0.9998159408569336
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",187,2026220984993,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.56,36.54,3.0673580169677734
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",190,2026219284945,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.375,15.915,4.250916004180908
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",348,2026224385256,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2525,3.955,1.3340704441070557
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",349,2026224385257,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3025,3.88,0.7585852146148682
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",162,2026222685019,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.9,64.75,3.999476432800293
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",350,2026224385258,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.31,3.885,1.1672389507293701
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",188,2026220984994,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.58,36.36,3.4344735145568848
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",191,2026219284946,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.4575,15.425,2.9928102493286133
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",351,2026224385259,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.265,3.935,1.17262864112854
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",352,2026224385260,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.275,3.975,0.887944221496582
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",353,2026224385261,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.28,4.19,0.9794621467590332
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",163,2026222685020,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.95,64.65,3.8890798091888428
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",354,2026224385262,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.31,3.88,0.7484402656555176
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",192,2026219284947,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.4975,15.97,3.3391520977020264
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",189,2026220984995,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.55,35.2,4.317629814147949
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",355,2026224385263,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2425,4.015,0.9075393676757812
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",356,2026224385264,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.27,4.12,1.091219425201416
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",357,2026224385265,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3075,3.88,1.1149013042449951
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",358,2026224385266,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.315,4.045,0.6921041011810303
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",190,2026220984996,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.28,36.54,3.1218461990356445
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",164,2026222685021,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.85,64.0,4.203015565872192
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",193,2026219284948,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,2.4875,15.96,3.669665813446045
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",359,2026224385267,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.27,3.86,0.7031149864196777
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",360,2026224385268,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.25,4.025,0.684373140335083
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",361,2026224385269,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.305,3.895,0.6965517997741699
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",362,2026224385270,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2575,4.07,1.0846269130706787
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",191,2026220984997,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,2.0,2,0,6.77,35.6,3.1697473526000977
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",194,2026219284949,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,2.365,15.89,3.930567979812622
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",363,2026224385271,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2975,4.0,1.4123501777648926
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",165,2026222685022,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.7,66.2,4.625920534133911
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",364,2026224385272,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2875,3.985,1.2272765636444092
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",192,2026220984998,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.68,34.48,3.031099796295166
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",365,2026224385273,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2675,3.955,1.1872732639312744
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",195,2026219284950,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.4625,15.99,3.028594970703125
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",366,2026224385274,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.2875,4.075,1.098637580871582
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",166,2026222685023,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.275,65.75,3.9915947914123535
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",193,2026220984999,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.28,35.24,2.9063613414764404
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",367,2026224385275,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2425,3.975,1.4009511470794678
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",368,2026224385276,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2725,3.965,0.9884955883026123
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",369,2026224385277,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2825,4.13,0.6785275936126709
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",194,2026220985000,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.23,35.78,2.3887691497802734
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",196,2026219284951,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.365,15.85,4.454174041748047
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",370,2026224385278,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3075,4.055,0.6934208869934082
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",371,2026224385279,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2975,3.905,0.6699554920196533
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",167,2026222685024,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,12.35,61.2,4.20712423324585
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",372,2026224385280,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.245,4.07,0.9950032234191895
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",195,2026220985001,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.24,35.86,2.727219820022583
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",373,2026224385281,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3025,3.94,1.026059627532959
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",374,2026224385282,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.2475,4.005,0.8957781791687012
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",197,2026219284952,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.4425,16.135,4.136532306671143
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",375,2026224385283,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.29,4.1,0.9002566337585449
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",168,2026222685025,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.225,63.9,3.919707775115967
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",376,2026224385284,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.245,4.07,0.9024078845977783
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",196,2026220985002,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,6.37,35.92,3.255516529083252
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",377,2026224385285,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2825,4.045,0.6847114562988281
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",378,2026224385286,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2775,3.775,0.7582132816314697
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",379,2026224385287,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2975,3.935,0.8517534732818604
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",198,2026219284953,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.595,16.13,3.6696226596832275
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",380,2026224385288,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.295,4.12,1.0445799827575684
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",381,2026224385289,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2225,3.955,0.7206735610961914
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",169,2026222685026,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.925,63.9,4.550126791000366
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",197,2026220985003,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.69,36.32,4.245975732803345
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",382,2026224385290,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2825,3.955,1.1518840789794922
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",199,2026219284954,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,2.4175,16.055,3.7038254737854004
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",383,2026224385291,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.265,3.97,0.8382713794708252
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",384,2026224385292,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.3025,4.155,1.051382303237915
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",198,2026220985004,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.46,35.08,3.380164861679077
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",385,2026224385293,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.26,3.935,1.247948408126831
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",170,2026222685027,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.325,64.55,4.110918045043945
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",386,2026224385294,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3075,3.975,0.9691760540008545
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",200,2026219284955,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.465,16.345,3.3401098251342773
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",387,2026224385295,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2425,4.12,1.4086694717407227
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",388,2026224385296,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2475,4.01,0.8933651447296143
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",199,2026220985005,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.58,36.58,3.7838361263275146
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",171,2026222685028,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.3,65.1,3.7540106773376465
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",389,2026224385297,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2525,4.17,0.7649555206298828
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",201,2026219284956,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,2.4675,16.0,3.4720747470855713
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",390,2026224385298,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.265,4.005,0.9964399337768555
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",391,2026224385299,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.27,3.815,1.094346284866333
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",392,2026224385300,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.235,4.1,0.7090780735015869
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",200,2026220985006,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.71,35.76,3.5208323001861572
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",172,2026222685029,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.05,64.05,3.5234827995300293
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",393,2026224385301,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.2325,3.88,0.8657212257385254
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",394,2026224385302,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2625,4.1,0.834730863571167
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",202,2026219284957,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.52,16.18,4.477185964584351
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",395,2026224385303,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.2325,4.025,0.6848597526550293
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",201,2026220985007,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.96,35.6,2.757944345474243
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",396,2026224385304,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2475,3.905,0.6805686950683594
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",397,2026224385305,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.3525,4.045,1.1141655445098877
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",173,2026222685030,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.25,63.0,4.5116705894470215
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",398,2026224385306,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.3025,4.12,0.9879848957061768
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",203,2026219284958,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,2.2875,16.08,3.304269313812256
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",202,2026220985008,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.8,35.66,3.1343579292297363
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",399,2026224385307,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2375,3.86,0.836749792098999
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",400,2026224385308,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.235,4.08,0.9953534603118896
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",401,2026224385309,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.27,4.005,1.063474178314209
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",174,2026222685031,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.225,66.3,3.5300724506378174
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",203,2026220985009,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.81,36.26,2.993420124053955
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",402,2026224385310,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.225,3.97,0.9961607456207275
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",204,2026219284959,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,2.47,16.05,4.016088962554932
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",403,2026224385311,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3,3.955,1.1761775016784668
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",404,2026224385312,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2275,4.09,1.1548497676849365
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",175,2026222685032,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.175,63.65,3.609382390975952
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",204,2026220985010,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.73,36.3,3.259699583053589
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",405,2026224385313,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.29,4.03,0.9384546279907227
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",205,2026219284960,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.525,15.605,3.7395870685577393
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",406,2026224385314,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.2775,3.86,0.9246411323547363
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",407,2026224385315,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2725,4.09,0.8699884414672852
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",408,2026224385316,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.2875,4.055,1.0286576747894287
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",409,2026224385317,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.295,4.085,0.7766108512878418
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",205,2026220985011,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.68,35.34,4.00792670249939
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",176,2026222685033,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.725,64.95,4.133891582489014
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",206,2026219284961,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,2.55,16.025,3.6647491455078125
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",410,2026224385318,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.29,4.08,1.0353128910064697
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",411,2026224385319,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2775,3.85,0.8559112548828125
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",412,2026224385320,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.27,3.985,0.771428108215332
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",206,2026220985012,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.61,35.64,3.3327910900115967
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",413,2026224385321,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.2425,4.09,1.0056581497192383
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",177,2026222685034,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.5,64.3,3.596052885055542
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",414,2026224385322,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.5,2,1,0.285,3.92,1.0297975540161133
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",207,2026219284962,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.275,16.095,4.115065097808838
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",415,2026224385323,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.3025,4.155,1.041966438293457
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",416,2026224385324,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.265,4.06,0.8912968635559082
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",207,2026220985013,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.3,36.2,3.441321849822998
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",417,2026224385325,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2825,4.16,0.7625610828399658
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",178,2026222685035,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.025,64.25,4.00095272064209
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",418,2026224385326,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.3075,4.08,0.8143508434295654
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",208,2026219284963,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.36,16.15,4.16978645324707
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",419,2026224385327,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.31,4.055,1.0499012470245361
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",420,2026224385328,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.29,3.94,0.9678194522857666
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",421,2026224385329,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.28,4.16,0.7362544536590576
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",208,2026220985014,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.83,35.06,4.707826375961304
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",179,2026222685036,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.325,64.75,3.9305260181427
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",422,2026224385330,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.29,3.85,0.9190256595611572
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",209,2026219284964,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.445,16.35,3.3626840114593506
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",423,2026224385331,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.275,3.935,0.679347038269043
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",424,2026224385332,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.245,3.845,1.2021563053131104
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",425,2026224385333,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2575,3.965,0.6927902698516846
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",180,2026222685037,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.5,1,1,13.125,64.7,3.1367387771606445
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",426,2026224385334,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.2925,4.025,0.6991348266601562
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",209,2026220985015,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.73,35.74,3.920034170150757
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",427,2026224385335,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2825,4.06,0.7140116691589355
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",210,2026219284965,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,2.47,16.12,3.8501219749450684
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",428,2026224385336,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.3,3.88,1.1106376647949219
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",429,2026224385337,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.305,4.08,1.2842824459075928
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",181,2026222685038,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.025,62.2,3.591771125793457
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",210,2026220985016,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.71,36.02,3.2722368240356445
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",430,2026224385338,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.26,4.04,1.3510701656341553
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",211,2026219284966,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,2.375,15.865,3.7818691730499268
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",431,2026224385339,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.265,3.89,1.1516149044036865
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",182,2026222685039,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.125,64.25,2.958735942840576
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",432,2026224385340,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2825,4.035,0.8053693771362305
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",433,2026224385341,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.19,3.86,0.7676668167114258
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",211,2026220985017,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.6,36.22,3.5830864906311035
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",212,2026219284967,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.5325,15.715,3.2785260677337646
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",434,2026224385342,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2525,3.925,1.5211918354034424
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",435,2026224385343,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.2925,3.88,1.00059175491333
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",436,2026224385344,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.275,4.015,1.0310115814208984
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",212,2026220985018,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.31,35.88,3.449347496032715
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",183,2026222685040,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.325,63.7,4.679592609405518
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",213,2026219284968,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,2.38,16.245,3.0492353439331055
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",437,2026224385345,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.24,4.09,0.9718139171600342
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",438,2026224385346,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.24,3.835,0.7881684303283691
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",439,2026224385347,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.285,4.095,0.7090184688568115
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",440,2026224385348,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.32,4.02,0.735344648361206
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",184,2026222685041,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.65,64.5,3.1702730655670166
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",214,2026219284969,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,2.475,16.175,3.5942275524139404
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",441,2026224385349,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.285,3.86,0.9544892311096191
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",213,2026220985019,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.66,35.4,4.399577856063843
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",442,2026224385350,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.285,3.885,1.1406900882720947
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",443,2026224385351,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.27,4.035,1.5329546928405762
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",214,2026220985020,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.91,36.5,2.829604387283325
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",185,2026222685042,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,14.25,62.4,3.8410720825195312
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",444,2026224385352,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2225,3.93,0.6993362903594971
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",215,2026219284970,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.31,15.87,3.7781786918640137
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",445,2026224385353,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.2925,4.02,0.9299125671386719
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",446,2026224385354,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2775,4.025,1.0291578769683838
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",447,2026224385355,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2575,3.95,1.0154519081115723
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",186,2026222685043,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.425,64.25,3.8369626998901367
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",215,2026220985021,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.66,35.12,4.0402045249938965
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",216,2026219284971,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.43,16.27,3.701982021331787
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",448,2026224385356,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.26,4.075,1.2967567443847656
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",449,2026224385357,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.27,3.995,1.1387481689453125
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",450,2026224385358,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.275,3.775,0.9522600173950195
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",216,2026220985022,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.37,36.62,3.1904819011688232
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",187,2026222685044,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.875,64.35,4.05247163772583
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",451,2026224385359,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.275,3.89,1.3731608390808105
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",217,2026219284972,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.4375,16.185,3.8316574096679688
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",452,2026224385360,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.29,4.21,1.604468822479248
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",453,2026224385361,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.255,3.88,1.019895076751709
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",217,2026220985023,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.7,36.36,3.525973081588745
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",188,2026222685045,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.6,62.65,3.364342451095581
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",454,2026224385362,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.27,3.955,1.019618272781372
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",218,2026219284973,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.4875,15.655,4.201311111450195
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",455,2026224385363,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.265,4.005,1.274789571762085
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",456,2026224385364,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2725,4.075,1.0961072444915771
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",218,2026220985024,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,7.31,36.18,3.5136895179748535
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",189,2026222685046,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.3,64.1,3.142775535583496
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",457,2026224385365,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.29,3.875,0.7457897663116455
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",458,2026224385366,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2275,4.18,0.7320442199707031
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",459,2026224385367,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3125,4.12,0.7545197010040283
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",460,2026224385368,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.25,3.955,0.7439169883728027
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",190,2026222685047,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,12.9,66.1,2.8735544681549072
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",219,2026219284974,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.3625,15.715,5.052669048309326
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",461,2026224385369,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.28,3.89,0.8072679042816162
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",219,2026220985025,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,6.73,35.5,4.558042526245117
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",462,2026224385370,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2875,4.17,1.312680959701538
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",463,2026224385371,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2625,4.06,1.0548138618469238
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",191,2026222685048,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.05,62.3,3.416442394256592
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",220,2026219284975,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.2425,15.94,3.718977212905884
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",464,2026224385372,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.295,3.93,1.1861493587493896
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",465,2026224385373,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3,4.085,0.9843711853027344
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",220,2026220985026,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.8,36.36,3.8507025241851807
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",466,2026224385374,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.28,3.88,1.220592737197876
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",221,2026219284976,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,2.46,15.65,3.328322172164917
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",192,2026222685049,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.725,61.9,4.046448469161987
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",467,2026224385375,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2675,3.97,1.370931625366211
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",221,2026220985027,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.78,35.68,3.452291488647461
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",468,2026224385376,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.2925,3.93,1.5771393775939941
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",222,2026219284977,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.39,15.83,2.521822452545166
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",469,2026224385377,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.27,4.055,0.7934274673461914
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",470,2026224385378,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.265,4.245,0.808762788772583
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",471,2026224385379,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2225,4.02,1.0366630554199219
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",222,2026220985028,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.58,34.9,3.136941909790039
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",193,2026222685050,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.7,62.9,4.6305012702941895
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",472,2026224385380,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.3375,3.915,0.8717379570007324
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",473,2026224385381,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.255,3.835,0.7806451320648193
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",474,2026224385382,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.26,3.73,0.7652668952941895
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",223,2026219284978,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.4125,16.265,5.186858415603638
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",475,2026224385383,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3125,4.0,0.8691091537475586
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",223,2026220985029,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,7.0,35.6,3.7143099308013916
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",194,2026222685051,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.15,64.6,3.5511839389801025
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",476,2026224385384,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2225,4.085,1.1615715026855469
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",477,2026224385385,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.335,3.875,0.998471736907959
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",478,2026224385386,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.27,4.1,0.7217700481414795
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",224,2026219284979,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,2.315,15.88,4.015357494354248
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",224,2026220985030,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.69,35.7,3.3411896228790283
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",479,2026224385387,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3025,4.03,1.1774885654449463
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",480,2026224385388,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.26,3.97,0.7148489952087402
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",195,2026222685052,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,12.725,62.6,4.4498560428619385
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",481,2026224385389,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.295,4.025,0.9844443798065186
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",482,2026224385390,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.295,4.165,0.6895594596862793
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",225,2026220985031,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.44,36.52,3.082304000854492
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",225,2026219284980,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.49,15.88,3.998352289199829
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",483,2026224385391,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.28,3.99,1.300351858139038
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",196,2026222685053,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.875,64.15,4.0383007526397705
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",484,2026224385392,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.25,3.995,1.4956395626068115
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",485,2026224385393,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2775,4.035,1.0342097282409668
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",226,2026220985032,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.38,36.6,3.6023354530334473
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",226,2026219284981,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.4625,15.55,3.0808393955230713
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",486,2026224385394,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2725,4.205,0.9486472606658936
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",487,2026224385395,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3025,4.03,0.8109781742095947
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",197,2026222685054,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.925,65.0,4.212857484817505
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",488,2026224385396,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3675,3.97,1.3266549110412598
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",227,2026220985033,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.76,36.44,3.4605119228363037
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",227,2026219284982,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,2.395,16.11,3.3787176609039307
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",489,2026224385397,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.295,4.14,0.8692169189453125
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",490,2026224385398,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.2425,4.035,1.3399035930633545
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",198,2026222685055,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.65,65.15,3.283684730529785
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",491,2026224385399,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2525,4.075,1.1893501281738281
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",492,2026224385400,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.2825,4.175,0.7118165493011475
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",228,2026220985034,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.54,35.56,3.7079296112060547
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",228,2026219284983,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.5975,15.95,3.7598884105682373
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",493,2026224385401,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2775,4.125,1.0855255126953125
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",494,2026224385402,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.235,4.065,1.2408292293548584
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",199,2026222685056,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.025,64.8,3.654897451400757
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",495,2026224385403,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.31,3.94,0.8754892349243164
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",229,2026220985035,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,7.04,35.18,3.147155523300171
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",496,2026224385404,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2175,3.875,0.8337490558624268
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",497,2026224385405,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.305,3.96,0.8994894027709961
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",229,2026219284984,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.36,15.935,4.684166431427002
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",498,2026224385406,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2325,4.12,1.0007591247558594
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",200,2026222685057,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.2,64.0,3.4010932445526123
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",230,2026220985036,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.63,36.42,3.607896089553833
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",499,2026224385407,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.265,3.82,1.050877332687378
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",500,2026224385408,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.305,3.925,1.2121541500091553
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",230,2026219284985,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.4825,16.05,3.3325977325439453
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",501,2026224385409,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.5,2,1,0.27,3.8,1.005366563796997
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",231,2026220985037,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.63,35.84,2.8315320014953613
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",201,2026222685058,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,12.925,68.6,4.176057577133179
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",502,2026224385410,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2275,3.77,1.135141372680664
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",503,2026224385411,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2825,3.915,1.5889859199523926
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",231,2026219284986,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.3875,15.91,4.159463405609131
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",504,2026224385412,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2525,3.95,0.9826352596282959
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",232,2026220985038,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.92,36.6,3.562595844268799
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",202,2026222685059,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.45,62.8,3.267686128616333
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",505,2026224385413,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2825,4.0,0.9683103561401367
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",506,2026224385414,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.28,4.03,0.8684453964233398
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",232,2026219284987,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.31,15.985,3.321540117263794
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",507,2026224385415,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3025,3.855,1.5954394340515137
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",203,2026222685060,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.85,64.15,3.359157085418701
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",233,2026220985039,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.38,36.26,4.143988132476807
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",508,2026224385416,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.22,3.775,1.4377670288085938
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",233,2026219284988,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,2.42,16.135,3.0528345108032227
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",509,2026224385417,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2725,4.015,1.197211742401123
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",234,2026220985040,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.39,36.02,2.912809133529663
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",510,2026224385418,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.24,4.01,1.3609733581542969
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",204,2026222685061,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,12.975,63.95,4.378556966781616
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",511,2026224385419,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.2725,4.07,1.1667964458465576
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",512,2026224385420,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2875,4.06,0.8514766693115234
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",234,2026219284989,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.415,15.95,3.576047897338867
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",513,2026224385421,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.26,4.11,0.8372013568878174
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",514,2026224385422,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2425,4.02,0.9706661701202393
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",235,2026220985041,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.27,35.94,3.894960641860962
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",205,2026222685062,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.8,62.2,3.446194648742676
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",515,2026224385423,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.28,4.09,1.2231788635253906
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",235,2026219284990,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.3825,16.05,3.370612144470215
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",516,2026224385424,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3125,3.975,0.7262499332427979
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",517,2026224385425,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.33,4.075,1.5715878009796143
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",206,2026222685063,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.55,63.4,3.078549861907959
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",236,2026220985042,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,7.07,35.98,4.154993772506714
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",518,2026224385426,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.3375,3.81,0.8462212085723877
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",519,2026224385427,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2925,3.935,1.1977839469909668
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",236,2026219284991,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.3375,15.905,4.091302394866943
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",207,2026222685064,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.425,64.95,2.863159418106079
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",520,2026224385428,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.28,3.695,1.0366878509521484
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",521,2026224385429,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2575,3.8,1.083115816116333
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",237,2026220985043,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.66,35.5,3.834786891937256
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",522,2026224385430,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2575,4.08,0.7157964706420898
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",208,2026222685065,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.55,65.8,3.1089894771575928
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",523,2026224385431,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2875,3.965,1.0972731113433838
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",237,2026219284992,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.4725,15.79,3.9312305450439453
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",524,2026224385432,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.28,4.115,1.212073564529419
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",525,2026224385433,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.31,4.145,0.7932155132293701
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",238,2026220985044,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,7.21,35.5,3.8297438621520996
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",526,2026224385434,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.285,3.995,1.0557007789611816
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",238,2026219284993,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.3375,15.895,3.895829916000366
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",527,2026224385435,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.265,4.13,1.1003828048706055
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",209,2026222685066,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.4,61.8,4.5336833000183105
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",528,2026224385436,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.26,4.075,0.6928386688232422
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",239,2026220985045,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.47,35.12,3.167562961578369
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",529,2026224385437,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.26,3.905,0.766758918762207
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",530,2026224385438,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.26,3.94,0.7158777713775635
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",531,2026224385439,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.28,4.02,1.2867367267608643
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",239,2026219284994,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.3475,15.89,4.060022592544556
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",240,2026220985046,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.78,34.34,2.806792974472046
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",210,2026222685067,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.1,66.1,4.337960481643677
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",532,2026224385440,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.2625,3.835,1.3657329082489014
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",533,2026224385441,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.295,3.985,0.7012434005737305
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",534,2026224385442,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.2975,4.065,0.6963555812835693
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",535,2026224385443,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.255,4.185,0.7133266925811768
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",241,2026220985047,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,6.49,35.36,3.249570369720459
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",536,2026224385444,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.29,4.115,0.8163836002349854
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",211,2026222685068,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.85,66.9,3.4399924278259277
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",240,2026219284995,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.35,15.835,4.1846184730529785
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",537,2026224385445,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.24,3.925,1.2432503700256348
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",538,2026224385446,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2625,3.945,0.8512001037597656
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",242,2026220985048,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.75,35.0,3.3056728839874268
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",539,2026224385447,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.28,3.905,1.3963711261749268
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",212,2026222685069,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.05,64.4,3.3744852542877197
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",241,2026219284996,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.4675,15.785,3.6790919303894043
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",540,2026224385448,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.27,4.165,0.9613778591156006
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",541,2026224385449,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.25,3.815,1.0812761783599854
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",243,2026220985049,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.42,35.9,2.708800792694092
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",542,2026224385450,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2925,4.1,0.749920129776001
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",213,2026222685070,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.6,64.75,3.6419527530670166
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",543,2026224385451,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2825,4.0,1.155778408050537
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",544,2026224385452,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.29,3.845,1.0493206977844238
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",244,2026220985050,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.51,35.46,2.8504745960235596
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",242,2026219284997,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.4275,15.96,4.47746729850769
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",545,2026224385453,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3325,4.09,1.134260892868042
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",546,2026224385454,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.29,3.975,0.7308619022369385
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",214,2026222685071,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.5,63.85,3.6174590587615967
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",547,2026224385455,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2175,4.065,0.6909725666046143
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",548,2026224385456,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3025,3.96,0.6977472305297852
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",245,2026220985051,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.14,35.82,3.9273433685302734
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",243,2026219284998,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.295,16.315,3.9660110473632812
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",549,2026224385457,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.26,3.95,0.8637604713439941
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",550,2026224385458,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.325,4.005,0.6928625106811523
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",551,2026224385459,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2525,4.04,0.9944543838500977
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",246,2026220985052,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.28,36.08,2.566265344619751
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",215,2026222685072,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.925,64.5,4.531553506851196
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",552,2026224385460,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2775,4.025,1.3925867080688477
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",244,2026219284999,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.545,16.35,3.682300567626953
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",553,2026224385461,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2975,4.105,0.7796990871429443
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",554,2026224385462,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.24,4.13,1.0092990398406982
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",247,2026220985053,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.76,35.8,3.363710641860962
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",555,2026224385463,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.255,4.12,1.140923023223877
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",245,2026219285000,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.4025,16.315,3.4031388759613037
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",556,2026224385464,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.3075,4.005,1.103198528289795
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",216,2026222685073,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.9,63.05,4.395420551300049
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",557,2026224385465,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2875,4.07,1.1966910362243652
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",558,2026224385466,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.295,3.905,0.673396110534668
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",559,2026224385467,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2875,3.905,0.6813910007476807
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",248,2026220985054,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,6.66,36.1,4.546634912490845
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",246,2026219285001,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,2.385,16.15,3.4051527976989746
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",217,2026222685074,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.725,64.9,3.2287771701812744
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",560,2026224385468,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.25,3.96,0.8986899852752686
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",561,2026224385469,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.305,3.985,0.6793041229248047
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",562,2026224385470,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.5,2,1,0.2525,3.995,1.0174975395202637
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",563,2026224385471,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.27,3.925,1.2406439781188965
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",247,2026219285002,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,2.43,16.14,3.3864827156066895
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",249,2026220985055,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,6.45,35.86,3.840301036834717
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",218,2026222685075,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.525,64.0,4.016422510147095
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",564,2026224385472,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.29,4.2,1.1350598335266113
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",565,2026224385473,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.29,3.97,1.1892056465148926
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",566,2026224385474,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.28,3.98,0.9553368091583252
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",248,2026219285003,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.42,16.04,3.5184836387634277
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",219,2026222685076,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.95,64.3,3.2858400344848633
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",250,2026220985056,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.46,36.68,3.767998218536377
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",567,2026224385475,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.2625,3.98,1.1347465515136719
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",568,2026224385476,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.305,3.89,1.545619010925293
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",249,2026219285004,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,2.45,15.655,2.730388879776001
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",569,2026224385477,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.2925,4.0,0.8009850978851318
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",570,2026224385478,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2775,3.89,0.7419998645782471
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",220,2026222685077,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.65,64.25,3.3317153453826904
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",251,2026220985057,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.6,36.42,4.0474207401275635
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",571,2026224385479,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2975,3.895,0.9423947334289551
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",572,2026224385480,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3075,3.82,0.6854555606842041
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",250,2026219285005,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,2.2875,15.94,3.171198844909668
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",573,2026224385481,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2825,4.11,1.1227772235870361
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",221,2026222685078,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.5,66.25,3.513875961303711
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",574,2026224385482,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2525,4.045,1.132430076599121
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",252,2026220985058,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.6,35.18,3.532909870147705
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",575,2026224385483,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.2475,4.015,0.6791141033172607
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",576,2026224385484,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.275,3.93,0.9340271949768066
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",251,2026219285006,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.2525,16.115,3.966771364212036
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",577,2026224385485,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.22,4.17,0.9149785041809082
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",578,2026224385486,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2675,4.035,0.8015761375427246
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",222,2026222685079,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.525,63.9,3.8431897163391113
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",579,2026224385487,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2875,4.125,0.9530982971191406
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",253,2026220985059,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.52,36.88,3.913997173309326
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",580,2026224385488,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2725,3.995,1.3011174201965332
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",252,2026219285007,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.6175,16.08,4.289117813110352
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",581,2026224385489,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.25,4.14,1.249101161956787
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",223,2026222685080,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.375,63.35,3.715461254119873
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",582,2026224385490,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.285,3.905,0.9058957099914551
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",254,2026220985060,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.75,35.74,3.683365821838379
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",583,2026224385491,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2525,4.005,0.9901764392852783
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",584,2026224385492,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.31,3.965,0.8527140617370605
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",253,2026219285008,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.53,15.845,3.88824462890625
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",585,2026224385493,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.25,3.895,1.0192594528198242
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",224,2026222685081,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.625,65.35,3.7785651683807373
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",586,2026224385494,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2725,3.82,0.7978038787841797
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",587,2026224385495,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2575,3.835,1.0372231006622314
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",255,2026220985061,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.22,36.08,4.488201379776001
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",588,2026224385496,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.27,4.14,0.9235014915466309
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",225,2026222685082,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.075,64.1,3.181715250015259
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",254,2026219285009,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.34,16.06,3.9414222240448
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",589,2026224385497,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2575,4.1,1.0137007236480713
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",590,2026224385498,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2425,4.135,0.6722185611724854
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",591,2026224385499,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2875,3.83,0.8284661769866943
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",592,2026224385500,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2775,4.045,0.6739187240600586
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",256,2026220985062,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.45,35.16,4.081200838088989
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",226,2026222685083,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.1,63.75,2.9135799407958984
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",593,2026224385501,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.32,4.05,1.155388593673706
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",255,2026219285010,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,2.32,16.05,3.9866857528686523
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",594,2026224385502,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.29,4.15,1.1564836502075195
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",227,2026222685084,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.25,65.85,2.61523175239563
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",595,2026224385503,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.26,4.0,1.1423792839050293
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",257,2026220985063,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.92,36.74,3.9770522117614746
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",596,2026224385504,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2725,3.97,0.9721448421478271
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",597,2026224385505,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.32,3.99,0.8979036808013916
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",256,2026219285011,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.4975,16.08,3.874079942703247
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",598,2026224385506,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2375,4.205,1.131305456161499
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",228,2026222685085,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.05,65.45,3.7117197513580322
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",599,2026224385507,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2525,4.06,0.7579038143157959
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",258,2026220985064,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,1.5,2,1,6.13,36.28,3.64067006111145
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",600,2026224385508,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.22,3.895,0.978715181350708
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",601,2026224385509,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.26,4.11,1.170750617980957
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",257,2026219285012,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.3325,15.94,3.856210470199585
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",602,2026224385510,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.5,2,1,0.2375,3.98,0.9731252193450928
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",259,2026220985065,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.24,36.18,2.6590988636016846
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",229,2026222685086,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.3,63.05,4.433836936950684
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",603,2026224385511,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2475,4.0,1.1115965843200684
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",604,2026224385512,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2725,3.795,1.0168330669403076
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",258,2026219285013,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.3775,16.165,3.507333517074585
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",260,2026220985066,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.35,36.1,2.850633382797241
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",605,2026224385513,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2625,4.025,1.0576324462890625
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",606,2026224385514,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.25,4.1,0.8373298645019531
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",230,2026222685087,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.375,66.3,3.7414920330047607
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",607,2026224385515,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.26,4.105,1.0796575546264648
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",259,2026219285014,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,2.0,2,0,2.4125,16.005,3.2688724994659424
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",608,2026224385516,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2625,3.995,1.3695940971374512
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",261,2026220985067,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.62,36.72,3.7362382411956787
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",609,2026224385517,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2475,3.92,0.9058077335357666
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",610,2026224385518,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.265,4.045,0.9074888229370117
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",231,2026222685088,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,12.625,62.75,4.04505467414856
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",611,2026224385519,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.265,4.09,0.9510867595672607
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",260,2026219285015,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,2.5425,16.23,3.5083112716674805
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",262,2026220985068,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.7,35.66,3.390035390853882
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",612,2026224385520,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2625,3.93,1.3265552520751953
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",613,2026224385521,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.24,4.125,1.2272512912750244
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",232,2026222685089,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.65,63.3,3.876028060913086
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",614,2026224385522,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.2725,4.18,1.1031043529510498
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",261,2026219285016,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.4875,16.16,3.660027265548706
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",263,2026220985069,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.94,35.5,3.239396095275879
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",615,2026224385523,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2975,3.94,1.244001865386963
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",616,2026224385524,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2575,3.995,1.098839282989502
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",233,2026222685090,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.6,61.25,3.3297691345214844
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",617,2026224385525,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.26,3.95,1.230147123336792
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",262,2026219285017,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.5,16.02,3.4024744033813477
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",264,2026220985070,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.61,36.64,3.8354012966156006
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",618,2026224385526,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2725,4.1,1.0744068622589111
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",619,2026224385527,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2475,4.055,1.4139823913574219
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",234,2026222685091,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.95,64.05,3.3005454540252686
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",620,2026224385528,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.2775,4.025,0.7141876220703125
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",263,2026219285018,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.3875,16.02,4.053907155990601
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",621,2026224385529,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2625,3.965,0.9128258228302002
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",265,2026220985071,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.63,36.4,3.5543978214263916
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",622,2026224385530,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.3,4.025,0.8210980892181396
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",623,2026224385531,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2675,4.015,0.6919674873352051
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",235,2026222685092,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.125,62.9,3.837125301361084
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",624,2026224385532,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.24,4.165,1.4000201225280762
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",264,2026219285019,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,2.4025,16.04,2.9805033206939697
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",625,2026224385533,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.285,3.92,1.1617999076843262
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",266,2026220985072,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.42,36.6,4.185819864273071
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",626,2026224385534,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.315,4.055,0.8046374320983887
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",627,2026224385535,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3075,4.19,0.7037420272827148
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",628,2026224385536,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.2875,4.03,0.7147014141082764
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",236,2026222685093,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.6,62.9,3.7961912155151367
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",629,2026224385537,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.245,4.125,0.6934523582458496
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",265,2026219285020,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.4825,16.095,4.67845892906189
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",267,2026220985073,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.26,36.54,3.4694082736968994
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",630,2026224385538,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.29,4.04,0.7512989044189453
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",631,2026224385539,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2825,4.05,1.2400026321411133
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",237,2026222685094,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.5,64.45,3.1871237754821777
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",632,2026224385540,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.27,4.0,1.1044299602508545
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",633,2026224385541,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.29,4.035,1.069136381149292
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",266,2026219285021,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,2.355,16.45,3.5934696197509766
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",268,2026220985074,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.4,35.02,4.125515699386597
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",634,2026224385542,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.3225,3.98,0.8658039569854736
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",238,2026222685095,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.325,64.25,3.456027030944824
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",635,2026224385543,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.29,3.91,1.0572433471679688
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",636,2026224385544,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.27,3.95,0.7013201713562012
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",637,2026224385545,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.2475,3.915,0.687201738357544
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",269,2026220985075,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.51,34.5,3.3776814937591553
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",267,2026219285022,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.5125,16.045,4.201153039932251
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",638,2026224385546,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2925,3.935,1.233109474182129
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",639,2026224385547,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3025,3.915,1.0706498622894287
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",239,2026222685096,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,12.65,64.3,4.074085235595703
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",640,2026224385548,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.2875,3.945,0.8093271255493164
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",270,2026220985076,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,6.6,36.16,3.269984006881714
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",641,2026224385549,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2625,4.005,1.5199594497680664
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",268,2026219285023,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.2525,16.115,4.153950452804565
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",642,2026224385550,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2625,4.055,1.3624908924102783
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",240,2026222685097,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.075,63.9,3.5135514736175537
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",643,2026224385551,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.23,3.825,1.0228376388549805
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",271,2026220985077,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.6,35.36,3.4820785522460938
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",644,2026224385552,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.26,3.815,0.8398277759552002
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",269,2026219285024,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.4275,16.015,3.579571008682251
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",645,2026224385553,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2775,4.045,1.0692808628082275
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",646,2026224385554,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2475,4.11,0.9688448905944824
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",272,2026220985078,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.13,35.82,2.9707212448120117
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",241,2026222685098,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.1,63.05,4.534172773361206
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",647,2026224385555,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2925,4.01,1.2912366390228271
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",270,2026219285025,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,2.3575,15.745,3.2083029747009277
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",648,2026224385556,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2725,3.97,0.9551270008087158
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",649,2026224385557,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.265,4.205,0.8662276268005371
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",273,2026220985079,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.79,35.5,3.2517237663269043
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",650,2026224385558,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2625,3.91,1.0426056385040283
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",242,2026222685099,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,12.95,64.05,4.291030168533325
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",271,2026219285026,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,2.565,16.055,3.2237401008605957
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",651,2026224385559,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.315,3.975,1.1961746215820312
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",652,2026224385560,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.245,4.045,0.8152110576629639
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",653,2026224385561,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2975,4.11,1.086026906967163
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",274,2026220985080,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.75,36.52,3.7347662448883057
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",654,2026224385562,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.245,3.71,0.8448464870452881
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",272,2026219285027,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.45,16.035,3.3652305603027344
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",655,2026224385563,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.255,3.85,1.2438831329345703
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",243,2026222685100,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.225,64.8,4.964179277420044
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",656,2026224385564,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.2775,4.11,0.6941680908203125
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",275,2026220985081,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.79,36.76,3.3651998043060303
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",273,2026219285028,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,2.34,16.06,2.630397319793701
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",657,2026224385565,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.21,4.135,1.2456467151641846
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",658,2026224385566,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2775,4.045,1.0219767093658447
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",659,2026224385567,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.295,4.14,1.161696195602417
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",244,2026222685101,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.75,64.15,3.6908764839172363
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",276,2026220985082,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.56,36.3,2.90091872215271
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",660,2026224385568,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.29,3.915,0.836967945098877
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",274,2026219285029,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.445,16.08,3.7165117263793945
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",661,2026224385569,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2525,3.905,1.0622336864471436
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",662,2026224385570,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.29,3.975,1.3839733600616455
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",277,2026220985083,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.55,37.18,3.53763747215271
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",245,2026222685102,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,13.9,63.25,3.999330520629883
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",663,2026224385571,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.295,3.955,1.2966408729553223
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",275,2026219285030,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.2575,15.9,3.901733636856079
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",664,2026224385572,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.29,3.975,1.1563622951507568
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",665,2026224385573,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.2475,3.895,1.0689115524291992
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",278,2026220985084,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.26,34.7,3.2782742977142334
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",666,2026224385574,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.25,4.065,0.8464276790618896
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",246,2026222685103,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.525,63.3,3.9935617446899414
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",667,2026224385575,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.3175,4.03,0.732029914855957
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",276,2026219285031,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,2.3075,15.975,3.7701525688171387
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",668,2026224385576,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3225,4.145,0.7270998954772949
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",669,2026224385577,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.28,3.925,0.7433998584747314
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",670,2026224385578,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2775,4.005,1.2132160663604736
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",247,2026222685104,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.175,65.4,2.9809234142303467
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",279,2026220985085,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,7.03,35.48,4.509913444519043
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",671,2026224385579,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2725,4.005,0.7372395992279053
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",672,2026224385580,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.2525,4.05,1.0154027938842773
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1,1",277,2026219285032,B,noisy,5,0.0002,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,2.36,16.14,4.110579252243042
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",673,2026224385581,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.2725,4.16,1.0025722980499268
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",248,2026222685105,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.35,65.35,3.0846052169799805
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",674,2026224385582,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2625,4.085,1.1864252090454102
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",280,2026220985086,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.54,35.32,3.7910094261169434
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",675,2026224385583,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.265,4.04,1.4532034397125244
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",676,2026224385584,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.26,3.9,1.1645421981811523
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",249,2026222685106,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.95,64.65,3.5801308155059814
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",0,2026226084959,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.19,16.075,4.121116399765015
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",281,2026220985087,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.57,35.42,3.4642956256866455
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",677,2026224385585,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3025,4.075,1.2742877006530762
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",1,2026226084960,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,2.0,2,0,1.2625,15.675,2.488211154937744
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",678,2026224385586,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.27,4.09,1.6478540897369385
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",250,2026222685107,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.825,64.5,3.1810154914855957
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",679,2026224385587,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2725,3.87,1.1864550113677979
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",2,2026226084961,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.21,16.13,2.2288119792938232
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",680,2026224385588,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.27,4.055,0.9415793418884277
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",282,2026220985088,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,7.0,36.12,4.084914922714233
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",681,2026224385589,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2775,4.01,0.8948781490325928
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",682,2026224385590,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2575,4.025,0.8580493927001953
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",251,2026222685108,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,12.9,66.3,4.179432153701782
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",3,2026226084962,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2575,16.015,2.7924883365631104
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",683,2026224385591,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.29,4.165,1.2743277549743652
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",283,2026220985089,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.88,36.3,3.4137697219848633
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",684,2026224385592,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.3325,3.95,1.0798406600952148
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",685,2026224385593,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.305,3.95,1.0187792778015137
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",686,2026224385594,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.27,4.21,0.9388961791992188
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",4,2026226084963,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.29,16.01,3.4440078735351562
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",252,2026222685109,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,15.35,65.4,3.859145164489746
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",284,2026220985090,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.54,36.16,3.2142727375030518
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",687,2026224385595,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.24,3.85,0.9548940658569336
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",688,2026224385596,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2825,3.97,1.5648632049560547
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",253,2026222685110,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.8,63.35,3.0758564472198486
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",689,2026224385597,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2675,4.19,0.8817274570465088
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",5,2026226084964,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2875,16.135,3.3373615741729736
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",285,2026220985091,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.83,35.06,3.569695234298706
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",690,2026224385598,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.32,3.93,1.1752278804779053
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",691,2026224385599,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.305,3.93,0.7597599029541016
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",692,2026224385600,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2325,3.915,1.2136433124542236
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",6,2026226084965,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2375,16.005,3.103588819503784
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",254,2026222685111,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.325,62.15,3.390648603439331
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",693,2026224385601,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.27,3.985,0.9671261310577393
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",286,2026220985092,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.62,35.94,3.6064672470092773
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",694,2026224385602,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.29,4.04,0.7333080768585205
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",695,2026224385603,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2625,4.07,0.8625049591064453
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",7,2026226084966,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1675,16.14,3.3776168823242188
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",696,2026224385604,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2525,4.08,1.1047632694244385
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",255,2026222685112,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.5,65.05,3.983774185180664
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",697,2026224385605,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.265,4.095,1.170118808746338
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",287,2026220985093,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.56,36.04,3.6519663333892822
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",698,2026224385606,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2875,3.835,1.0460295677185059
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",8,2026226084967,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.22,15.935,2.915679693222046
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",699,2026224385607,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2675,4.015,1.103283405303955
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",256,2026222685113,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.225,64.85,3.5037295818328857
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",700,2026224385608,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.2675,4.24,1.0449292659759521
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",701,2026224385609,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.3325,4.145,0.7679383754730225
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",288,2026220985094,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.83,36.14,3.7840094566345215
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",9,2026226084968,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.2075,15.505,2.9327566623687744
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",702,2026224385610,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.265,3.94,0.9071948528289795
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",703,2026224385611,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.275,3.94,0.685422420501709
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",704,2026224385612,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2825,4.03,0.9007689952850342
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",257,2026222685114,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.275,62.6,4.048070430755615
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",10,2026226084969,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.215,16.28,3.1900510787963867
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",705,2026224385613,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3325,4.175,1.1538033485412598
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",289,2026220985095,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.46,36.2,4.2009522914886475
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",706,2026224385614,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.23,3.98,1.5107543468475342
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",258,2026222685115,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.475,64.85,2.6960132122039795
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",707,2026224385615,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.26,4.19,0.7476053237915039
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",708,2026224385616,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.24,4.155,0.7256162166595459
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",11,2026226084970,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.165,15.96,3.194563627243042
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",709,2026224385617,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.305,3.98,0.7608470916748047
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",290,2026220985096,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,7.03,36.7,3.797651767730713
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",710,2026224385618,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.28,3.825,1.2585902214050293
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",259,2026222685116,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.325,61.45,4.004831075668335
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",12,2026226084971,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1975,15.625,2.9384286403656006
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",711,2026224385619,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3175,3.855,1.186039924621582
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",712,2026224385620,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.275,3.895,0.8831045627593994
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",291,2026220985097,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.35,35.48,2.9156901836395264
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",713,2026224385621,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.325,3.825,1.1119670867919922
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",13,2026226084972,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2025,15.78,2.85394287109375
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",714,2026224385622,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2575,4.08,1.1589338779449463
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",260,2026222685117,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.3,63.05,3.703827142715454
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",715,2026224385623,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2675,3.985,0.8729066848754883
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",292,2026220985098,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.58,36.64,3.600111961364746
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",716,2026224385624,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.255,3.97,0.8988165855407715
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",717,2026224385625,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2875,4.11,0.9613990783691406
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",14,2026226084973,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1475,15.805,3.323275327682495
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",718,2026224385626,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3275,4.075,1.1457393169403076
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",261,2026222685118,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.475,63.5,3.9919657707214355
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",719,2026224385627,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3,3.925,0.9113788604736328
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",293,2026220985099,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,7.26,35.86,3.3669424057006836
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",720,2026224385628,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.23,3.995,0.7301540374755859
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",721,2026224385629,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.225,3.93,0.7584202289581299
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",15,2026226084974,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.255,16.0,3.619046688079834
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",722,2026224385630,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.24,4.055,0.8241322040557861
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",723,2026224385631,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.225,3.82,0.6911349296569824
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",262,2026222685119,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.725,64.95,3.730762243270874
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",724,2026224385632,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2525,4.16,0.6779978275299072
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",294,2026220985100,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,6.29,36.72,4.089199542999268
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",725,2026224385633,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.295,3.84,1.0491864681243896
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",726,2026224385634,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2425,3.965,0.7466983795166016
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",16,2026226084975,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.285,16.025,3.811803102493286
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",727,2026224385635,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.26,3.985,1.364152193069458
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",263,2026222685120,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,15.3,63.4,3.9664723873138428
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",728,2026224385636,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.265,4.025,0.6660137176513672
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",295,2026220985101,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.22,35.92,3.6770901679992676
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",729,2026224385637,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2125,3.925,0.8761451244354248
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",17,2026226084976,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.315,15.955,3.456195116043091
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",730,2026224385638,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2625,4.03,0.6833004951477051
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",731,2026224385639,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2675,3.96,1.3848040103912354
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",296,2026220985102,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.81,36.24,2.644987106323242
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",264,2026222685121,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.6,64.45,4.163492679595947
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",732,2026224385640,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.265,4.095,1.1111440658569336
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",18,2026226084977,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2075,16.19,3.071509599685669
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",733,2026224385641,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2775,4.0,1.2529959678649902
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",297,2026220985103,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.09,36.1,2.760948657989502
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",734,2026224385642,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.2825,4.115,0.9255962371826172
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",735,2026224385643,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.2625,4.125,0.9246668815612793
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",265,2026222685122,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.025,63.75,3.353231191635132
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",736,2026224385644,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.2825,4.0,1.1022822856903076
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",19,2026226084978,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1575,15.715,3.8304362297058105
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",737,2026224385645,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2375,4.12,1.1107232570648193
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",298,2026220985104,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.51,34.9,3.5753543376922607
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",738,2026224385646,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.29,4.035,1.1179847717285156
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",266,2026222685123,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.525,63.3,3.3769800662994385
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",739,2026224385647,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2575,3.975,1.0781512260437012
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",20,2026226084979,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.2275,15.78,3.774932861328125
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",299,2026220985105,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.61,36.38,3.0108823776245117
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",740,2026224385648,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2575,4.16,0.9810829162597656
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",741,2026224385649,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.25,4.015,0.8125607967376709
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",267,2026222685124,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,12.4,62.85,3.7264342308044434
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",742,2026224385650,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2375,3.975,1.0894498825073242
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",743,2026224385651,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3125,4.0,0.8332819938659668
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",21,2026226084980,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.27,15.62,3.6979639530181885
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",300,2026220985106,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.47,36.22,3.6326708793640137
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",744,2026224385652,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.265,4.0,0.7877597808837891
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",745,2026224385653,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.305,3.855,0.7430448532104492
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",746,2026224385654,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2575,4.045,0.6795949935913086
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",268,2026222685125,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.65,63.4,4.036360263824463
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",747,2026224385655,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2325,4.01,1.0253331661224365
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",301,2026220985107,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.72,35.98,2.7913506031036377
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",22,2026226084981,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.235,16.115,3.232689619064331
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",748,2026224385656,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.285,4.055,1.0513644218444824
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",749,2026224385657,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.2575,4.185,0.8507659435272217
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",750,2026224385658,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.28,4.095,1.3052432537078857
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",269,2026222685126,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.6,65.1,3.454815149307251
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",302,2026220985108,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.56,34.92,3.004653215408325
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",23,2026226084982,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2325,15.72,3.2516562938690186
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",751,2026224385659,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.28,3.86,1.3245458602905273
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",752,2026224385660,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2525,4.015,1.0973515510559082
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",303,2026220985109,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.47,35.06,3.074925184249878
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",270,2026222685127,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.325,64.5,3.5126755237579346
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",24,2026226084983,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2575,16.16,3.1304309368133545
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",753,2026224385661,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2675,3.92,1.668365240097046
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",754,2026224385662,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.25,4.02,1.3456826210021973
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",304,2026220985110,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.62,36.24,2.821226119995117
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",755,2026224385663,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.27,4.035,1.0520086288452148
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",25,2026226084984,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.175,15.955,3.2149124145507812
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",271,2026222685128,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.375,64.05,4.056286811828613
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",756,2026224385664,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2625,4.03,1.3177924156188965
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",757,2026224385665,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3025,3.96,0.8098154067993164
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",305,2026220985111,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.38,35.42,2.7397522926330566
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",758,2026224385666,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2525,4.225,1.221595287322998
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",759,2026224385667,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2575,3.94,0.9953956604003906
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",26,2026226084985,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2525,15.99,3.9545578956604004
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",272,2026222685129,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.025,63.95,3.4519665241241455
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",760,2026224385668,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.29,4.125,1.0658469200134277
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1,1",306,2026220985112,B,noisy,7,0.0002,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,6.52,36.5,3.4993340969085693
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",761,2026224385669,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.28,4.06,0.9146480560302734
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",762,2026224385670,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2125,4.045,0.860438346862793
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",27,2026226084986,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2425,15.935,3.4780588150024414
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",763,2026224385671,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.215,3.895,0.8361952304840088
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",273,2026222685130,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.4,65.15,4.123625755310059
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",764,2026224385672,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.285,3.97,1.0410048961639404
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",765,2026224385673,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.2625,4.03,1.031754493713379
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",0,2026227785010,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.41,35.14,2.3474419116973877
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",766,2026224385674,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.27,4.0,0.8995673656463623
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",28,2026226084987,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.32,16.28,3.288085699081421
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",767,2026224385675,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.3125,3.93,1.1571810245513916
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",274,2026222685131,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.45,62.8,3.5580127239227295
production-20261029T053925Z-29f79c,"B|noisy|d3|p0.0001|pi0|k1,1,1",768,2026224385676,B,noisy,3,0.0001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.2975,4.13,1.0570011138916016
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",1,2026227785011,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.01,36.52,2.834866762161255
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",29,2026226084988,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2325,15.805,3.635781764984131
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",275,2026222685132,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,12.7,64.7,3.172600507736206
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",2,2026227785012,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.03,36.14,2.6232950687408447
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",30,2026226084989,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1925,15.95,3.504514694213867
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",3,2026227785013,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.41,35.24,2.7283003330230713
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",276,2026222685133,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.4,63.7,3.6872165203094482
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",31,2026226084990,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.23,15.845,2.734825372695923
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",0,2026229485061,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.15,62.4,3.3453588485717773
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",4,2026227785014,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,2.99,36.74,3.1022074222564697
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",277,2026222685134,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.775,64.65,4.164459705352783
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",5,2026227785015,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.21,35.66,2.4234492778778076
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",32,2026226084991,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2725,15.87,3.1921255588531494
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",1,2026229485062,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.225,62.95,3.729407548904419
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",33,2026226084992,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2275,15.74,2.633143901824951
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",6,2026227785016,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.18,36.66,2.9817967414855957
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",278,2026222685135,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.0,62.1,4.338788032531738
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",2,2026229485063,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.3,65.95,3.312486171722412
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",7,2026227785017,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.38,35.0,2.381882429122925
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",34,2026226084993,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.27,15.805,3.4787607192993164
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",279,2026222685136,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.9,64.8,3.1416263580322266
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",3,2026229485064,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.275,63.9,3.3476009368896484
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",8,2026227785018,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.06,35.68,2.935124158859253
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",35,2026226084994,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2625,16.02,3.3781776428222656
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",280,2026222685137,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,12.925,62.2,3.0932271480560303
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",4,2026229485065,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.65,65.3,3.3736722469329834
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",9,2026227785019,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.27,36.38,3.5804128646850586
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",36,2026226084995,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.3175,16.1,3.3966307640075684
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",281,2026222685138,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.875,63.45,4.18681001663208
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",5,2026229485066,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.925,64.1,3.3890535831451416
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",10,2026227785020,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.45,35.98,2.47684645652771
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",37,2026226084996,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.255,16.285,3.3307201862335205
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",11,2026227785021,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.42,35.26,1.9892418384552002
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",6,2026229485067,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.275,63.1,3.4229836463928223
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",282,2026222685139,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.475,65.9,4.001475811004639
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",38,2026226084997,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1725,15.9,2.599823236465454
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",12,2026227785022,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.16,36.48,2.762643337249756
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",7,2026229485068,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.55,63.0,2.6343367099761963
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",13,2026227785023,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.15,36.12,2.0228090286254883
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",283,2026222685140,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.75,63.65,4.293690919876099
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",39,2026226084998,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1275,16.31,3.709069013595581
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",8,2026229485069,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.45,64.1,3.0337471961975098
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",14,2026227785024,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,2.94,35.3,2.5913100242614746
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",40,2026226084999,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.335,15.5,2.8112552165985107
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",9,2026229485070,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.25,62.6,2.5897178649902344
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",284,2026222685141,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,14.775,62.45,4.125438928604126
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",15,2026227785025,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.18,34.42,3.475114107131958
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",10,2026229485071,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.025,63.3,2.596928834915161
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",41,2026226085000,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1975,15.86,3.8137333393096924
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",16,2026227785026,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.11,35.96,2.4271092414855957
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0002|pi0|k1,1,1",285,2026222685142,B,noisy,9,0.0002,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,13.225,63.9,4.171163082122803
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",11,2026229485072,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.3,64.7,3.0536646842956543
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",17,2026227785027,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.34,34.76,2.5669705867767334
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",0,2026231185112,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.895,16.175,2.162973642349243
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",42,2026226085001,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.195,16.03,4.061870098114014
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",18,2026227785028,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.14,35.76,2.5046019554138184
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",12,2026229485073,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.625,64.8,3.6552884578704834
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",43,2026226085002,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.245,15.95,2.5882174968719482
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",1,2026231185113,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.785,15.625,2.6863889694213867
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",19,2026227785029,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.55,36.28,2.387294054031372
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",2,2026231185114,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.7625,15.86,2.6964797973632812
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",13,2026229485074,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,7.825,62.6,3.760240316390991
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",20,2026227785030,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.25,35.96,2.4110724925994873
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",44,2026226085003,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.25,16.07,3.74627423286438
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",21,2026227785031,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.14,35.42,2.241410493850708
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",3,2026231185115,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.815,15.885,3.154547691345215
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",45,2026226085004,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1675,16.07,3.138035535812378
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",14,2026229485075,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,6.725,61.9,3.755613088607788
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",22,2026227785032,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.42,35.46,2.783196210861206
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",4,2026231185116,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.875,16.135,2.821427345275879
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",15,2026229485076,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.8,67.1,2.905880928039551
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",46,2026226085005,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.34,16.055,3.3240268230438232
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",5,2026231185117,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.85,15.92,2.4276609420776367
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",23,2026227785033,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.37,35.08,2.7012619972229004
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",47,2026226085006,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.23,16.18,3.0481584072113037
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",6,2026231185118,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,2.0,2,0,1.7625,16.145,2.2780070304870605
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",16,2026229485077,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.4,66.25,3.6803719997406006
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",24,2026227785034,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.35,36.98,2.814185619354248
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",7,2026231185119,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.815,16.03,2.5895044803619385
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",17,2026229485078,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.325,64.15,2.927865982055664
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",25,2026227785035,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.37,36.04,2.3634567260742188
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",48,2026226085007,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.255,15.94,3.2432141304016113
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",49,2026226085008,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.28,16.05,2.187211513519287
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",8,2026231185120,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.9,16.125,3.1346096992492676
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",18,2026229485079,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.675,62.2,2.9149668216705322
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",26,2026227785036,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.29,35.96,2.9248478412628174
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",50,2026226085009,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.225,16.33,2.8625829219818115
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",9,2026231185121,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.725,15.65,2.4318647384643555
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",27,2026227785037,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.66,35.78,2.7298524379730225
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",19,2026229485080,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,6.825,62.95,3.414062261581421
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",10,2026231185122,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.725,15.775,1.9799363613128662
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",28,2026227785038,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.3,36.14,2.7691218852996826
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",20,2026229485081,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,5.875,64.0,2.4988880157470703
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",51,2026226085010,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.29,16.295,4.276399850845337
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",11,2026231185123,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.7775,16.03,2.7257986068725586
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",29,2026227785039,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.31,36.8,2.5687365531921387
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",52,2026226085011,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.145,15.83,2.588073968887329
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",21,2026229485082,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.85,65.55,3.2988641262054443
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",12,2026231185124,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.8075,15.93,2.915656805038452
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",30,2026227785040,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.27,35.92,3.1738879680633545
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",53,2026226085012,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.195,15.97,2.542396306991577
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",13,2026231185125,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,2.0,2,0,1.6525,16.19,2.123549461364746
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",22,2026229485083,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.8,66.65,3.355705976486206
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",31,2026227785041,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.1,35.72,2.62746262550354
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",54,2026226085013,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.11,16.105,3.0851855278015137
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",23,2026229485084,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.825,62.55,2.332265853881836
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",14,2026231185126,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.8925,16.045,3.3140413761138916
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",32,2026227785042,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.55,37.02,2.442697286605835
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",15,2026231185127,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.8275,15.945,2.5280873775482178
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",55,2026226085014,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.27,16.18,3.4525465965270996
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",24,2026229485085,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.6,63.4,3.2966344356536865
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",33,2026227785043,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.15,35.8,2.7994415760040283
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",16,2026231185128,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,2.0,2,0,1.835,16.155,2.36448335647583
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",56,2026226085015,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.24,15.945,2.6625306606292725
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",34,2026227785044,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.5,36.92,2.037343740463257
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",17,2026231185129,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.89,16.02,2.500028371810913
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",25,2026229485086,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.325,62.95,5.069170236587524
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",35,2026227785045,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.1,35.84,2.4230704307556152
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",57,2026226085016,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.21,16.14,3.1522488594055176
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",26,2026229485087,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.825,64.3,2.6783852577209473
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",18,2026231185130,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.7775,15.74,3.404780387878418
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",36,2026227785046,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.17,36.44,2.94458270072937
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",58,2026226085017,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1675,16.15,2.8394510746002197
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",37,2026227785047,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.48,36.4,2.443732738494873
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",27,2026229485088,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.2,64.75,3.3697543144226074
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",19,2026231185131,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.78,16.025,3.283266067504883
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",59,2026226085018,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2175,15.79,3.001431703567505
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",20,2026231185132,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.82,15.74,2.391603469848633
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",38,2026227785048,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.08,35.56,3.010742425918579
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",28,2026229485089,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.925,65.95,3.149498462677002
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",60,2026226085019,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.245,15.775,2.8493945598602295
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",39,2026227785049,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.27,36.08,2.873598575592041
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",21,2026231185133,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,2.0,2,0,1.8125,15.56,3.2679433822631836
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",29,2026229485090,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.05,63.95,2.887511730194092
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",61,2026226085020,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.25,16.025,2.8847227096557617
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",22,2026231185134,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.7975,16.07,2.1475563049316406
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",40,2026227785050,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.48,35.78,2.5917463302612305
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",30,2026229485091,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,7.125,62.1,3.393054246902466
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",62,2026226085021,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.295,16.13,3.3488123416900635
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",41,2026227785051,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.41,35.62,2.248441696166992
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",23,2026231185135,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.8,16.015,3.1942107677459717
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",42,2026227785052,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.17,36.58,1.707529067993164
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",63,2026226085022,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.17,15.855,2.9414567947387695
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",31,2026229485092,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.325,64.3,3.2740657329559326
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",24,2026231185136,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.5,2,1,1.8525,15.7,2.7389469146728516
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",43,2026227785053,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.22,36.34,2.7596523761749268
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",32,2026229485093,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,6.875,63.9,2.792489767074585
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",64,2026226085023,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.115,16.125,3.5663111209869385
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",25,2026231185137,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,3.0,3,0,1.9175,16.16,2.9876813888549805
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",44,2026227785054,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.24,36.26,2.6642284393310547
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",33,2026229485094,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.65,65.5,3.3286709785461426
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",65,2026226085024,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1225,15.91,3.470627784729004
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",26,2026231185138,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.8675,16.0,2.8386600017547607
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",45,2026227785055,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.33,37.14,2.284923553466797
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",66,2026226085025,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.3,16.15,2.6909193992614746
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",27,2026231185139,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.8875,16.475,2.670124053955078
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",46,2026227785056,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.26,35.92,2.918396472930908
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",34,2026229485095,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.6,65.5,4.087780237197876
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",28,2026231185140,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.925,15.815,2.448061943054199
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",67,2026226085026,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1875,16.185,2.803985834121704
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",47,2026227785057,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,3.61,35.96,2.9722354412078857
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",35,2026229485096,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.5,66.85,3.741903066635132
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",29,2026231185141,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.81,15.995,2.9471187591552734
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",48,2026227785058,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.17,36.14,2.5444495677948
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",68,2026226085027,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1175,15.835,3.470844030380249
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",36,2026229485097,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.05,64.3,3.5220863819122314
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",30,2026231185142,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,2.0,2,0,1.7825,15.715,2.5889129638671875
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",49,2026227785059,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.39,35.04,2.1942358016967773
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",69,2026226085028,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.2175,16.26,3.328202247619629
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",31,2026231185143,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,2.5,3,1,1.735,16.17,2.3113226890563965
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",37,2026229485098,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.8,64.45,3.5086045265197754
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",50,2026227785060,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.25,36.2,3.434096574783325
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",70,2026226085029,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2525,15.89,2.608429431915283
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",32,2026231185144,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.8075,15.86,2.5234713554382324
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",38,2026229485099,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.9,64.75,3.1964144706726074
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",51,2026227785061,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.1,35.9,3.2344417572021484
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",33,2026231185145,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,2.0,2,0,1.8675,15.755,2.1453051567077637
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",71,2026226085030,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.2475,15.9,3.1538400650024414
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",39,2026229485100,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.675,64.75,3.0281574726104736
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",52,2026227785062,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.18,35.82,2.842743158340454
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",34,2026231185146,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.7975,16.085,2.97117280960083
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",72,2026226085031,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1975,16.21,2.9848766326904297
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",35,2026231185147,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.8225,16.145,2.4590766429901123
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",53,2026227785063,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.06,36.34,2.9424281120300293
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",40,2026229485101,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,5.85,64.55,3.1015586853027344
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",73,2026226085032,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2175,16.015,3.194399356842041
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",54,2026227785064,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.33,35.88,2.7443737983703613
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",36,2026231185148,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.915,16.16,3.195585250854492
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",41,2026229485102,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.95,62.55,3.058095932006836
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",74,2026226085033,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.15,16.095,3.6471214294433594
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",55,2026227785065,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,2.92,36.0,2.2521653175354004
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",37,2026231185149,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.9325,16.355,2.9944934844970703
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",42,2026229485103,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.85,62.85,3.151780366897583
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",75,2026226085034,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2475,15.58,2.9964230060577393
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",56,2026227785066,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.15,36.22,2.771829843521118
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",38,2026231185150,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.7525,16.06,2.583378553390503
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",43,2026229485104,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.425,64.15,3.7516350746154785
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",57,2026227785067,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,3.14,36.86,2.49654221534729
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",39,2026231185151,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,2.5,3,1,1.855,15.81,2.4030673503875732
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",76,2026226085035,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2125,15.895,3.7204995155334473
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",58,2026227785068,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.1,36.18,2.3093180656433105
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",40,2026231185152,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.91,16.155,2.851895809173584
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",77,2026226085036,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,1.255,15.815,2.9012880325317383
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",44,2026229485105,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.3,62.7,4.140669345855713
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",59,2026227785069,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.14,36.26,2.4466824531555176
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",41,2026231185153,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,2.0,2,0,1.8225,15.9,2.493298053741455
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",60,2026227785070,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.11,36.98,2.216390371322632
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",45,2026229485106,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.0,63.05,3.3135569095611572
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",78,2026226085037,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.1975,15.5,3.623687982559204
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",42,2026231185154,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.845,16.13,2.6628475189208984
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",61,2026227785071,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.32,35.92,2.797919273376465
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",79,2026226085038,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1525,15.86,2.8408098220825195
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",46,2026229485107,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.75,63.05,3.4841558933258057
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",43,2026231185155,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.805,16.135,3.139099359512329
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",62,2026227785072,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.48,35.96,2.5323851108551025
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",80,2026226085039,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2825,15.725,2.9437782764434814
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",47,2026229485108,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.5,65.1,3.1132285594940186
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",44,2026231185156,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.835,16.075,2.43166446685791
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",63,2026227785073,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.4,35.28,2.714405059814453
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",45,2026231185157,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.77,16.1,2.473188638687134
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",81,2026226085040,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.205,15.745,3.5305862426757812
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",48,2026229485109,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.9,63.5,3.9512076377868652
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",64,2026227785074,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.3,36.64,2.798680305480957
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",46,2026231185158,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.9275,15.99,2.0521068572998047
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",82,2026226085041,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,1.3,15.86,3.3213231563568115
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",49,2026229485110,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.35,65.2,3.0399889945983887
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",65,2026227785075,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.28,36.36,3.2275140285491943
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",47,2026231185159,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.5,2,1,1.7675,16.3,2.5163462162017822
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",83,2026226085042,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1375,16.09,3.2065155506134033
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",66,2026227785076,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.11,36.14,2.942648410797119
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",50,2026229485111,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.125,63.2,3.4226086139678955
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",48,2026231185160,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.9175,16.175,3.052074670791626
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",67,2026227785077,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,3.42,35.92,2.5470566749572754
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",84,2026226085043,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1675,15.835,3.3306097984313965
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",49,2026231185161,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.8,16.04,2.461402177810669
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",51,2026229485112,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.375,63.5,2.916184663772583
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",50,2026231185162,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.9225,15.805,2.804857015609741
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",68,2026227785078,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.36,36.94,3.142591714859009
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",85,2026226085044,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1425,16.09,3.1918137073516846
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",52,2026229485113,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.3,63.3,3.432270050048828
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",69,2026227785079,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.18,36.28,2.8025693893432617
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",51,2026231185163,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,4.0,4,0,1.885,16.0,3.0785937309265137
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",86,2026226085045,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.22,15.97,3.1875898838043213
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",53,2026229485114,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.275,64.55,3.042961359024048
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",70,2026227785080,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.23,35.2,2.7751073837280273
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",52,2026231185164,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,2.0,15.965,3.0046944618225098
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",87,2026226085046,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.17,15.735,3.2025630474090576
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",54,2026229485115,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.625,63.65,3.493062734603882
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",53,2026231185165,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.7975,15.895,2.4498283863067627
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",71,2026227785081,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.4,35.58,2.93515944480896
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",88,2026226085047,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1975,16.08,3.775465250015259
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",55,2026229485116,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.2,63.2,2.843838691711426
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",72,2026227785082,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.07,34.14,1.8536536693572998
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",54,2026231185166,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.855,16.65,3.1881818771362305
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",73,2026227785083,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.16,36.76,2.426687717437744
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",56,2026229485117,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,5.975,66.2,2.9634850025177
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",55,2026231185167,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.8,15.83,2.5345938205718994
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",89,2026226085048,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1925,16.06,3.964940309524536
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",74,2026227785084,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.13,35.34,2.869211196899414
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",57,2026229485118,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.275,64.7,2.6930463314056396
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",56,2026231185168,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,2.0,2,0,1.74,15.845,2.4113729000091553
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",90,2026226085049,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.205,15.765,3.013981580734253
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",75,2026227785085,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.6,36.52,2.3236217498779297
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",57,2026231185169,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.8225,16.005,2.922114849090576
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",58,2026229485119,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.6,62.7,4.013482332229614
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",91,2026226085050,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,1.21,15.84,3.2611639499664307
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",76,2026227785086,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.22,35.56,2.563707113265991
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",58,2026231185170,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.5,1,1,1.805,15.715,2.9973642826080322
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",77,2026227785087,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.25,36.12,2.386275053024292
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",92,2026226085051,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2725,16.08,2.5705902576446533
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",59,2026229485120,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.775,65.6,4.488749027252197
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",59,2026231185171,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.5,1,1,1.8825,15.94,2.444631338119507
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",78,2026227785088,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.14,35.78,2.9882915019989014
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",93,2026226085052,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1725,15.775,3.6334285736083984
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",60,2026229485121,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.1,63.15,2.616699695587158
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",60,2026231185172,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.5,1,1,1.915,16.005,2.567589282989502
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",79,2026227785089,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.59,37.02,2.6403675079345703
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",94,2026226085053,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2125,15.94,2.880530834197998
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",61,2026229485122,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.725,63.35,3.131286144256592
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",61,2026231185173,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.8075,15.87,2.8607370853424072
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",80,2026227785090,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.24,35.6,2.1014456748962402
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",62,2026231185174,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,3.0,3,0,1.755,15.905,2.123344898223877
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",95,2026226085054,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.23,15.81,3.253138542175293
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",81,2026227785091,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.39,35.96,2.7152647972106934
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",62,2026229485123,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.175,65.15,3.955018997192383
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",63,2026231185175,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,2.0,2,0,1.85,15.855,2.7248940467834473
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",96,2026226085055,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.245,15.74,3.128268241882324
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",82,2026227785092,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.05,36.46,2.8048195838928223
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",63,2026229485124,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.15,64.9,2.8073325157165527
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",64,2026231185176,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,3.0,3,0,1.85,15.91,3.435906410217285
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",97,2026226085056,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.15,15.87,2.6705431938171387
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",64,2026229485125,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.825,62.75,2.395798921585083
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",83,2026227785093,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,2.93,34.44,3.2388076782226562
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",65,2026231185177,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,2.0,2,0,1.7275,15.855,2.4047553539276123
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",65,2026229485126,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,5.925,63.2,2.4820716381073
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",98,2026226085057,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1625,16.055,3.4557652473449707
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",84,2026227785094,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.23,35.3,3.3005590438842773
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",66,2026231185178,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.7325,15.955,3.2630181312561035
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",66,2026229485127,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.175,62.2,2.988548517227173
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",85,2026227785095,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.09,35.96,2.4250361919403076
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",99,2026226085058,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.315,16.1,3.6593689918518066
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",67,2026231185179,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,3.0,3,0,1.8,15.97,2.968050479888916
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",86,2026227785096,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.25,35.96,2.5830037593841553
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",67,2026229485128,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.2,65.3,3.0189788341522217
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",100,2026226085059,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2725,15.94,3.301741361618042
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",68,2026231185180,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.7925,16.405,2.47040057182312
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",68,2026229485129,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.35,62.15,2.739246129989624
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",87,2026227785097,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.42,35.34,3.284227132797241
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",69,2026231185181,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.94,15.895,2.7179903984069824
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",101,2026226085060,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.175,15.945,3.6938295364379883
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",69,2026229485130,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.9,62.65,2.675813913345337
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",88,2026227785098,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.06,35.74,2.459507942199707
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",70,2026231185182,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,2.5,3,1,1.835,16.115,2.4390463829040527
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",70,2026229485131,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,6.95,62.55,2.7076587677001953
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",102,2026226085061,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.295,15.96,3.338681221008301
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",89,2026227785099,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.49,35.68,3.0956263542175293
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",71,2026231185183,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.7075,15.84,2.8968312740325928
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",103,2026226085062,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2075,16.08,2.7314553260803223
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",71,2026229485132,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.075,64.3,3.0710716247558594
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",90,2026227785100,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.42,36.08,2.8417131900787354
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",72,2026231185184,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,2.0,2,0,1.86,16.04,2.309891700744629
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",91,2026227785101,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.03,36.24,2.573261022567749
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",104,2026226085063,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2625,16.25,3.2130515575408936
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",72,2026229485133,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,8.275,63.85,3.329174518585205
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",73,2026231185185,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.84,16.04,3.0437123775482178
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",92,2026227785102,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.11,35.76,2.8677172660827637
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",105,2026226085064,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1325,16.475,3.1712241172790527
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",73,2026229485134,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.75,65.8,3.0392239093780518
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",74,2026231185186,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,2.0,2,0,1.74,16.325,2.828237771987915
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",93,2026227785103,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.49,35.76,2.7247424125671387
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",106,2026226085065,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2175,15.69,3.291389226913452
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",74,2026229485135,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.7,62.35,3.2070114612579346
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",75,2026231185187,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.905,15.93,2.709739923477173
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",94,2026227785104,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.47,35.14,1.8324594497680664
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",76,2026231185188,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.7925,16.025,2.6889874935150146
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",107,2026226085066,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.195,15.98,3.609020948410034
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",75,2026229485136,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.525,62.5,3.7426226139068604
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",95,2026227785105,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.01,36.24,2.9890623092651367
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",77,2026231185189,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,2.0,2,0,1.8725,15.63,2.333977460861206
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",76,2026229485137,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.8,64.65,2.5235445499420166
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",96,2026227785106,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.51,35.68,2.7443575859069824
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",108,2026226085067,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1875,16.135,3.9227726459503174
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",78,2026231185190,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,3.0,3,0,1.8025,16.325,3.5131397247314453
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",77,2026229485138,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.45,64.7,2.380613088607788
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",97,2026227785107,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.18,35.48,2.5091941356658936
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",109,2026226085068,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,1.1725,15.69,3.5611512660980225
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",98,2026227785108,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.21,35.02,2.2688984870910645
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",78,2026229485139,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.475,64.4,2.8263840675354004
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",79,2026231185191,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.7475,15.905,3.1450419425964355
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",99,2026227785109,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.09,36.9,2.3548331260681152
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",80,2026231185192,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.5,1,1,1.84,16.38,2.0071516036987305
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",110,2026226085069,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1775,15.895,3.441073417663574
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",79,2026229485140,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.5,63.7,3.5778913497924805
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",81,2026231185193,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.835,15.8,2.5281484127044678
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",100,2026227785110,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.27,36.6,2.739349365234375
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",111,2026226085070,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2275,15.91,2.998645067214966
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",82,2026231185194,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,2.0,2,0,1.8275,15.715,2.1485321521759033
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",80,2026229485141,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.15,64.25,3.4344542026519775
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",101,2026227785111,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.25,35.26,3.051213502883911
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",112,2026226085071,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.17,16.285,3.427140235900879
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",81,2026229485142,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.9,66.8,2.678528070449829
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",83,2026231185195,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.88,15.955,3.285789728164673
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",102,2026227785112,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.33,35.86,2.5866525173187256
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",113,2026226085072,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,1.215,15.785,2.3824269771575928
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",84,2026231185196,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,2.0,2,0,1.755,16.24,2.387392520904541
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",103,2026227785113,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.44,37.32,2.6028552055358887
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",82,2026229485143,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.975,63.75,4.270315647125244
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",114,2026226085073,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.135,15.66,2.566453218460083
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",104,2026227785114,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,2.96,34.9,2.520833730697632
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",85,2026231185197,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.75,15.75,3.0523269176483154
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",83,2026229485144,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.2,64.6,2.558803081512451
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",115,2026226085074,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.28,15.91,3.235718250274658
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",86,2026231185198,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.5,2,1,1.65,15.895,2.4829750061035156
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",105,2026227785115,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.48,37.28,2.8526949882507324
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",87,2026231185199,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,2.5,3,1,1.7625,16.325,2.058290481567383
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",116,2026226085075,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1225,16.025,2.9745845794677734
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",84,2026229485145,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.15,63.85,4.037544250488281
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",106,2026227785116,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.47,36.3,3.355275869369507
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",85,2026229485146,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.525,66.35,2.2252984046936035
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",117,2026226085076,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.25,15.905,2.3872499465942383
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",88,2026231185200,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.765,16.245,3.0948455333709717
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",107,2026227785117,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.38,35.38,3.039139747619629
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",86,2026229485147,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.475,64.2,2.2174105644226074
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",118,2026226085077,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2625,15.925,2.3962252140045166
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",89,2026231185201,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,2.0,2,0,1.97,15.84,3.5218424797058105
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",87,2026229485148,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.175,64.25,2.6374118328094482
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",108,2026227785118,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.13,36.24,2.7689476013183594
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",119,2026226085078,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2175,15.885,2.9102861881256104
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",109,2026227785119,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.62,35.34,1.9058551788330078
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",88,2026229485149,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.025,64.2,2.58381986618042
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",90,2026231185202,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.88,16.075,3.5739595890045166
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",120,2026226085079,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.2675,16.255,2.8183701038360596
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",110,2026227785120,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.27,35.82,2.9613349437713623
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",89,2026229485150,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.775,63.65,2.7508347034454346
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",91,2026231185203,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,2.04,16.065,2.7010653018951416
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",121,2026226085080,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,1.2275,15.84,3.540524482727051
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",111,2026227785121,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.21,35.62,2.3982632160186768
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",92,2026231185204,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,3.0,3,0,1.9325,15.925,2.6210267543792725
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",90,2026229485151,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.0,62.25,3.3047454357147217
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",122,2026226085081,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.25,16.39,2.413848876953125
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",112,2026227785122,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.29,35.9,2.832268238067627
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",93,2026231185205,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,3.0,3,0,1.8875,15.845,2.627986431121826
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",91,2026229485152,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.875,63.8,3.2767276763916016
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",123,2026226085082,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2,16.075,3.3475892543792725
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",113,2026227785123,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,2.98,35.52,2.6097896099090576
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",94,2026231185206,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.715,15.8,2.7591538429260254
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",114,2026227785124,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.59,36.5,2.3593156337738037
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",92,2026229485153,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.925,63.65,3.6121037006378174
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",95,2026231185207,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.85,16.14,2.5174317359924316
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",124,2026226085083,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,2.0,2,0,1.1575,16.02,3.3468871116638184
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",115,2026227785125,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.57,35.82,2.916923999786377
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",93,2026229485154,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,5.975,61.35,2.7525932788848877
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",96,2026231185208,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.89,16.215,2.575460910797119
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",125,2026226085084,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.18,15.625,2.9400198459625244
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",116,2026227785126,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.52,36.54,2.5392184257507324
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",94,2026229485155,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.525,64.4,2.816848039627075
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",97,2026231185209,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,2.0,2,0,1.855,16.005,3.356050968170166
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",126,2026226085085,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.22,15.895,3.631743907928467
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",117,2026227785127,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.37,36.02,2.1280436515808105
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",95,2026229485156,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.975,63.05,3.3427343368530273
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",98,2026231185210,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.7925,16.295,2.8729279041290283
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",127,2026226085086,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2575,16.18,3.0101265907287598
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",118,2026227785128,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.56,36.3,3.0701005458831787
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",99,2026231185211,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.82,15.71,2.2406532764434814
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",96,2026229485157,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.575,65.7,2.692328929901123
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",119,2026227785129,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.53,36.08,2.7442820072174072
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",128,2026226085087,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.17,15.56,3.2271862030029297
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",100,2026231185212,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.5,2,1,1.85,15.945,2.457935094833374
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",97,2026229485158,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.125,62.75,3.635030746459961
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",120,2026227785130,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.05,35.7,2.5827853679656982
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",101,2026231185213,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.8,15.695,2.262688398361206
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",129,2026226085088,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1875,16.615,3.354835271835327
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",98,2026229485159,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.775,64.35,2.865187883377075
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",102,2026231185214,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,2.0,2,0,1.7575,16.11,2.336958885192871
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",121,2026227785131,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.35,37.64,3.390164613723755
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",130,2026226085089,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.235,16.295,2.7985827922821045
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",99,2026229485160,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.85,62.75,3.4284324645996094
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",122,2026227785132,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.16,36.0,2.587073802947998
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",131,2026226085090,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2625,15.745,2.803478717803955
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",103,2026231185215,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.9375,16.0,3.6913063526153564
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",123,2026227785133,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.34,35.72,2.8719842433929443
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",100,2026229485161,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.375,64.5,2.99984073638916
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",104,2026231185216,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.93,15.715,2.9194161891937256
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",132,2026226085091,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1625,16.095,3.417963981628418
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",124,2026227785134,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.27,36.5,2.4310741424560547
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",101,2026229485162,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.125,64.65,3.255742311477661
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",105,2026231185217,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.8975,16.285,2.6843490600585938
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",133,2026226085092,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.19,15.96,2.974679946899414
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",125,2026227785135,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.16,35.84,2.852271795272827
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",106,2026231185218,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.7875,16.155,2.574911117553711
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",102,2026229485163,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.675,63.6,3.578009605407715
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",134,2026226085093,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.065,15.76,3.1743593215942383
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",126,2026227785136,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.14,36.24,2.928851366043091
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",107,2026231185219,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.82,15.71,2.573390245437622
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",103,2026229485164,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.5,63.8,3.138361930847168
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",135,2026226085094,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.165,16.125,3.0519955158233643
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",108,2026231185220,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.8225,16.2,2.4012749195098877
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",127,2026227785137,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.09,35.98,3.06881046295166
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",109,2026231185221,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.8025,15.89,2.2966647148132324
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",128,2026227785138,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.18,36.48,2.073397159576416
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",104,2026229485165,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.575,62.75,3.2874717712402344
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",136,2026226085095,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.3325,16.025,3.567927837371826
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",110,2026231185222,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.935,16.16,2.386136531829834
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",129,2026227785139,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,3.11,35.54,2.623729705810547
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",105,2026229485166,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.025,64.25,3.028073310852051
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",137,2026226085096,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.26,15.94,3.904670238494873
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",130,2026227785140,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.09,36.64,2.1694931983947754
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",111,2026231185223,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.8075,15.86,2.877140760421753
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",106,2026229485167,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.725,65.15,2.7260899543762207
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",131,2026227785141,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.29,35.58,2.291548013687134
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",107,2026229485168,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.125,64.35,2.2441580295562744
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",138,2026226085097,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.125,16.085,3.454026460647583
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",112,2026231185224,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.835,15.84,2.9670071601867676
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",132,2026227785142,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.48,35.6,1.832463264465332
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",108,2026229485169,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.85,63.75,2.195952892303467
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",133,2026227785143,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.62,35.9,2.304175853729248
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",113,2026231185225,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.78,15.715,3.1516809463500977
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",139,2026226085098,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.22,16.165,3.9306411743164062
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",114,2026231185226,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,2.0,2,0,1.7675,16.275,2.2026009559631348
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",134,2026227785144,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.08,36.48,2.6701600551605225
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",109,2026229485170,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.45,63.05,3.8758935928344727
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",140,2026226085099,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1025,16.28,4.051419019699097
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",115,2026231185227,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.8225,16.19,2.9332823753356934
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",135,2026227785145,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.3,35.88,2.86438250541687
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",110,2026229485171,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.5,62.3,3.501828908920288
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",141,2026226085100,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.11,16.205,2.9253628253936768
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",116,2026231185228,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.8275,15.93,3.0518321990966797
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",136,2026227785146,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,3.35,35.68,3.598933219909668
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",111,2026229485172,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.7,63.75,3.0365591049194336
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",142,2026226085101,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2675,16.22,2.7120816707611084
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",117,2026231185229,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.8525,16.1,3.1757354736328125
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",112,2026229485173,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.95,64.1,2.714113712310791
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",137,2026227785147,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,2.95,36.36,3.5357491970062256
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",143,2026226085102,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.215,15.935,2.964128255844116
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",118,2026231185230,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.93,16.345,3.0239028930664062
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",138,2026227785148,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,3.44,36.14,2.692985773086548
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",113,2026229485174,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.825,62.95,3.711965560913086
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",144,2026226085103,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.235,15.905,3.0336217880249023
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",119,2026231185231,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.9325,16.09,2.847362756729126
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",139,2026227785149,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.47,37.08,2.5045435428619385
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",114,2026229485175,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,7.825,66.25,3.2644546031951904
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",145,2026226085104,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.3,15.695,3.368919610977173
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",120,2026231185232,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.845,15.93,2.4976792335510254
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",140,2026227785150,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.1,35.22,2.6766417026519775
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",115,2026229485176,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.05,63.8,2.797952890396118
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",121,2026231185233,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.87,16.255,2.7160329818725586
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",146,2026226085105,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.16,16.2,3.639185905456543
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",141,2026227785151,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.02,36.1,2.9225893020629883
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",122,2026231185234,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.805,16.15,2.371424436569214
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",116,2026229485177,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.85,62.7,3.423292398452759
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",142,2026227785152,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.28,35.3,2.67063307762146
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",147,2026226085106,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2525,15.83,2.8829426765441895
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",123,2026231185235,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.5,2,1,1.8025,16.1,2.497598648071289
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",117,2026229485178,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.125,62.95,3.4632115364074707
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",143,2026227785153,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.36,35.82,3.0653076171875
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",148,2026226085107,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.275,16.315,3.1189868450164795
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",124,2026231185236,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.84,15.74,2.8559036254882812
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",118,2026229485179,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.675,64.1,2.8022568225860596
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",149,2026226085108,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.295,15.97,2.917802333831787
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",144,2026227785154,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.18,36.38,3.3168489933013916
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",125,2026231185237,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,2.0,2,0,1.815,15.595,2.98950457572937
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",119,2026229485180,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.475,62.95,2.498873710632324
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",145,2026227785155,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.24,35.04,2.5990653038024902
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",150,2026226085109,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.205,16.305,3.1344358921051025
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",126,2026231185238,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,2,2,1.8425,15.995,3.5789544582366943
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",120,2026229485181,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.525,62.6,3.075873613357544
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",146,2026227785156,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.23,36.04,2.888258695602417
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",151,2026226085110,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1275,16.46,2.717505931854248
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",127,2026231185239,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,2.0,2,0,1.7375,15.955,2.4878358840942383
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",147,2026227785157,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.42,36.0,2.1676113605499268
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",121,2026229485182,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.625,63.35,3.378143548965454
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",152,2026226085111,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.195,16.235,3.0101547241210938
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",128,2026231185240,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.8225,15.88,2.6701009273529053
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",148,2026227785158,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.15,36.16,2.8223941326141357
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",122,2026229485183,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.825,63.35,3.0239431858062744
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",153,2026226085112,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1975,16.05,3.2066121101379395
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",129,2026231185241,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,2.0,2,0,1.7825,16.235,2.898601531982422
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",149,2026227785159,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,2.93,35.5,3.1754348278045654
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",154,2026226085113,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2975,15.95,2.3357579708099365
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",123,2026229485184,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.625,62.8,3.7276909351348877
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",130,2026231185242,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,2.0,2,0,1.785,15.62,2.7459704875946045
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",150,2026227785160,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.32,36.18,2.479827642440796
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",155,2026226085114,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2425,15.845,3.212803840637207
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",131,2026231185243,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.8325,16.08,2.4319849014282227
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",124,2026229485185,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.9,66.2,3.2688183784484863
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",151,2026227785161,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.34,34.74,2.9836130142211914
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",156,2026226085115,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1775,15.67,2.520477056503296
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",132,2026231185244,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,3.0,4,2,1.7225,15.945,2.6249547004699707
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",125,2026229485186,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.375,66.85,3.340806722640991
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",152,2026227785162,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.45,36.5,3.2068138122558594
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",157,2026226085116,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.11,15.965,2.435279369354248
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",133,2026231185245,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.5,1,1,1.84,16.13,3.1605474948883057
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",126,2026229485187,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.875,63.9,3.1990065574645996
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",153,2026227785163,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.34,34.68,2.7029874324798584
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",134,2026231185246,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,2.0,2,0,1.895,15.925,2.4580559730529785
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",158,2026226085117,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2375,15.83,3.6174933910369873
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",154,2026227785164,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.52,35.0,2.630527973175049
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",127,2026229485188,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,8.05,64.85,3.9946601390838623
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",135,2026231185247,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.5,2,1,1.885,16.19,2.874300479888916
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",159,2026226085118,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.215,16.14,3.554763078689575
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",155,2026227785165,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.27,35.92,2.5469768047332764
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",136,2026231185248,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.7625,15.725,2.7905945777893066
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",128,2026229485189,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.325,65.85,3.9877562522888184
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",156,2026227785166,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.23,36.48,2.4556541442871094
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",160,2026226085119,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2225,16.095,3.167802333831787
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",137,2026231185249,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.7375,15.94,2.6888482570648193
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",157,2026227785167,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.32,36.24,2.9040772914886475
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",129,2026229485190,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.125,64.25,3.6380155086517334
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",161,2026226085120,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.3525,15.795,3.627533435821533
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",138,2026231185250,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.785,16.13,2.2786152362823486
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",158,2026227785168,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.53,35.42,2.3827874660491943
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",139,2026231185251,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.82,15.885,2.9845399856567383
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",162,2026226085121,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2625,15.91,3.0694143772125244
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",130,2026229485191,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.425,62.7,3.8536741733551025
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",159,2026227785169,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.26,35.44,2.6984236240386963
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",163,2026226085122,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2175,16.29,2.3972327709198
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",140,2026231185252,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.815,16.1,2.599632978439331
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",131,2026229485192,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.925,64.35,3.0092265605926514
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",160,2026227785170,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,2.98,35.92,2.750314474105835
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",164,2026226085123,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.3425,15.885,2.945885181427002
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",141,2026231185253,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.9,16.1,2.943577766418457
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",132,2026229485193,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.0,65.35,3.836615562438965
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",161,2026227785171,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.25,35.12,3.5965845584869385
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",142,2026231185254,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.93,16.265,2.476247787475586
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",165,2026226085124,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2,15.97,2.994983196258545
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",162,2026227785172,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.16,36.44,2.445516586303711
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",143,2026231185255,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.8325,15.875,2.5787389278411865
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",133,2026229485194,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,5.95,62.6,3.1234490871429443
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",166,2026226085125,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.1675,15.84,3.08651065826416
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",163,2026227785173,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.52,35.6,3.006021499633789
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",134,2026229485195,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.525,63.35,2.874882459640503
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",144,2026231185256,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,2.0,2,0,1.8325,16.075,2.8872625827789307
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",167,2026226085126,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.225,15.88,2.8970038890838623
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",164,2026227785174,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.18,36.78,2.7728590965270996
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",145,2026231185257,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.87,15.96,2.9914214611053467
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",135,2026229485196,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.6,63.9,3.3759536743164062
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",168,2026226085127,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.16,15.945,2.8260159492492676
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",165,2026227785175,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.17,35.88,2.8745551109313965
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",146,2026231185258,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.795,15.685,3.0357205867767334
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",169,2026226085128,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1775,16.22,3.307950973510742
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",136,2026229485197,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.975,63.8,3.7782015800476074
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",166,2026227785176,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.41,36.02,2.15045428276062
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",167,2026227785177,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,3.22,35.74,2.344416379928589
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",147,2026231185259,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.7825,15.705,3.7056753635406494
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",137,2026229485198,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.05,61.8,2.6257662773132324
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",170,2026226085129,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2075,16.0,3.044135093688965
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",168,2026227785178,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.33,35.8,2.2739622592926025
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",171,2026226085130,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1875,15.775,2.8438844680786133
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",148,2026231185260,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.5,2,1,1.79,15.77,3.1733803749084473
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",138,2026229485199,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.7,63.55,4.24754524230957
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",169,2026227785179,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.22,35.2,2.6384730339050293
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",149,2026231185261,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.82,15.83,2.7386062145233154
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",172,2026226085131,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.325,16.24,2.90204119682312
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",139,2026229485200,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.475,63.15,3.358901023864746
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",170,2026227785180,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,3.5,35.22,2.926936388015747
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",150,2026231185262,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.8875,15.68,2.9508962631225586
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",173,2026226085132,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.285,16.05,2.9202775955200195
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",171,2026227785181,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.1,35.64,2.4534506797790527
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",140,2026229485201,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.7,63.4,3.3141045570373535
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",151,2026231185263,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.5,1,1,1.8225,16.04,3.25018572807312
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",174,2026226085133,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.285,16.01,3.257096290588379
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",172,2026227785182,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,3.23,35.24,2.3128011226654053
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",141,2026229485202,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.05,62.7,3.589674472808838
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",173,2026227785183,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.07,35.88,2.6888740062713623
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",152,2026231185264,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,2.0,2,0,1.6775,15.79,3.174229860305786
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",175,2026226085134,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2525,15.85,3.2839667797088623
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",174,2026227785184,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,2.97,35.5,2.4089972972869873
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",153,2026231185265,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.8275,16.105,2.5929017066955566
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",142,2026229485203,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.125,65.7,3.638944149017334
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",176,2026226085135,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.2675,16.115,3.6724231243133545
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",175,2026227785185,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.29,35.32,1.9645118713378906
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",154,2026231185266,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.7725,15.98,2.7478768825531006
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",143,2026229485204,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.625,65.5,3.5665998458862305
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",177,2026226085136,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2075,15.795,2.733924150466919
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",176,2026227785186,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.05,35.58,2.787949800491333
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",155,2026231185267,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,2.0,2,0,1.86,16.295,2.707062005996704
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",177,2026227785187,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.5,36.76,2.6434600353240967
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",144,2026229485205,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,7.35,62.1,3.5246431827545166
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",178,2026226085137,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.23,15.75,3.83512806892395
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",156,2026231185268,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.8475,15.87,2.4349136352539062
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",178,2026227785188,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.34,35.72,2.7539403438568115
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",179,2026226085138,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2275,16.165,2.5128958225250244
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",157,2026231185269,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,2.0,2,0,1.8325,15.735,3.052018880844116
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",145,2026229485206,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.475,64.55,4.200971841812134
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",179,2026227785189,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.27,36.24,2.351510524749756
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",158,2026231185270,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.8425,15.835,2.818193197250366
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",146,2026229485207,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,8.0,63.7,2.820756196975708
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",180,2026227785190,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.17,36.12,2.2719786167144775
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",180,2026226085139,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,2,2,1.2125,15.825,4.314609050750732
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",159,2026231185271,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.8125,16.225,2.743671417236328
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",181,2026227785191,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,2.95,37.12,2.7543723583221436
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",181,2026226085140,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2825,15.795,3.1620826721191406
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",147,2026229485208,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,6.75,64.05,3.4894156455993652
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",160,2026231185272,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.815,16.05,2.8491930961608887
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",182,2026227785192,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.35,36.34,2.375589370727539
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",182,2026226085141,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,1.1475,16.3,2.774894952774048
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",148,2026229485209,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.7,63.7,2.9042985439300537
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",183,2026227785193,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.0,36.0,2.1052732467651367
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",161,2026231185273,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.8375,15.715,3.287804365158081
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",183,2026226085142,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2675,16.115,3.1837003231048584
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",149,2026229485210,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.9,64.95,3.106318473815918
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",184,2026227785194,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.04,35.36,2.8315439224243164
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",162,2026231185274,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.5,1,1,1.945,16.09,2.5454440116882324
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",150,2026229485211,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.1,63.4,2.967268466949463
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",163,2026231185275,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.7525,15.755,2.5231423377990723
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",184,2026226085143,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.215,15.785,3.941009283065796
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",185,2026227785195,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.31,36.06,3.398343086242676
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",151,2026229485212,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,5.925,64.05,2.875880479812622
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",186,2026227785196,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.42,35.76,2.2329771518707275
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",164,2026231185276,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.9,16.02,2.915501832962036
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",185,2026226085144,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2675,15.825,3.7298314571380615
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",187,2026227785197,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.47,35.58,2.688302993774414
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",152,2026229485213,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,5.8,63.95,3.355470657348633
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",165,2026231185277,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.88,15.86,3.0359082221984863
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",186,2026226085145,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1325,15.8,3.1036179065704346
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",153,2026229485214,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.75,64.2,2.4604365825653076
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",166,2026231185278,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.84,16.285,2.4642558097839355
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",188,2026227785198,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,2.97,36.16,3.2277116775512695
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",187,2026226085146,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.225,15.905,3.6517281532287598
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",189,2026227785199,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.34,36.08,2.3191192150115967
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",167,2026231185279,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.895,15.735,2.7426531314849854
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",154,2026229485215,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.025,65.75,3.3670217990875244
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",188,2026226085147,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.21,16.21,2.9879355430603027
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",168,2026231185280,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.905,15.825,2.586289167404175
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",190,2026227785200,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.22,34.06,3.346149444580078
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",155,2026229485216,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.25,64.35,3.798830270767212
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",169,2026231185281,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.7275,15.88,2.5331597328186035
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",191,2026227785201,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.44,36.18,2.167421340942383
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",189,2026226085148,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.28,15.76,3.414001703262329
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",156,2026229485217,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.725,64.45,2.6003775596618652
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",192,2026227785202,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.38,34.94,2.6199305057525635
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",157,2026229485218,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.65,61.8,2.1810598373413086
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",170,2026231185282,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.9275,15.935,3.7772152423858643
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",193,2026227785203,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,2.85,35.46,1.7437553405761719
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",190,2026226085149,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2275,16.145,4.1859130859375
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",158,2026229485219,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.775,66.6,2.929553747177124
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",171,2026231185283,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,2.0,2,0,1.895,15.935,2.877455711364746
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",194,2026227785204,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,2.79,36.7,2.812736749649048
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",191,2026226085150,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.265,15.77,3.1777002811431885
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",172,2026231185284,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,2.0,2,0,1.85,15.81,2.1939010620117188
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",195,2026227785205,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.35,36.14,2.352348804473877
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",159,2026229485220,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.5,67.0,3.270702838897705
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",192,2026226085151,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2,15.975,3.4011099338531494
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",173,2026231185285,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.7925,16.33,2.647007703781128
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",196,2026227785206,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.2,36.54,3.2051665782928467
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",160,2026229485221,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.175,63.5,3.133845806121826
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",174,2026231185286,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.7875,16.24,2.405102252960205
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",193,2026226085152,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1725,15.84,3.022447347640991
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",197,2026227785207,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.36,36.26,2.522589683532715
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",175,2026231185287,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.825,16.0,2.5836904048919678
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",161,2026229485222,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.55,64.15,3.978471040725708
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",194,2026226085153,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1975,16.335,2.995224952697754
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",198,2026227785208,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.52,36.9,2.6324124336242676
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",176,2026231185288,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.875,15.87,2.2148661613464355
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",195,2026226085154,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2275,15.845,3.0399177074432373
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",177,2026231185289,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.5,1,1,1.8225,16.105,2.0675289630889893
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",199,2026227785209,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.57,36.16,2.883760452270508
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",162,2026229485223,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.55,62.55,4.001386880874634
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",200,2026227785210,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.45,37.1,2.496300458908081
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",196,2026226085155,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.26,15.945,3.012773036956787
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",178,2026231185290,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.8725,16.08,2.740487575531006
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",163,2026229485224,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.075,64.1,3.5894548892974854
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",201,2026227785211,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.29,35.48,2.7732937335968018
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",179,2026231185291,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.5,2,1,1.8675,15.965,2.6538681983947754
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",197,2026226085156,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2,16.21,3.312316417694092
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",164,2026229485225,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.95,62.65,3.5162315368652344
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",202,2026227785212,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.46,36.24,2.829286575317383
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",180,2026231185292,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.765,15.805,3.2211666107177734
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",198,2026226085157,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.105,16.155,2.7718608379364014
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",165,2026229485226,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.25,66.3,2.8974366188049316
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",203,2026227785213,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,2.92,35.96,3.0477957725524902
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",199,2026226085158,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1975,15.855,3.0485267639160156
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",181,2026231185293,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.5,2,1,1.6775,15.97,3.5509281158447266
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",166,2026229485227,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.5,65.65,3.227064847946167
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",204,2026227785214,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.43,35.62,2.4209227561950684
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",200,2026226085159,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.365,16.19,3.0605509281158447
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",182,2026231185294,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.7825,15.935,3.073362112045288
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",205,2026227785215,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.13,35.78,2.8246219158172607
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",167,2026229485228,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.05,65.2,3.5673344135284424
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",183,2026231185295,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.5,1,1,1.73,16.23,2.6255300045013428
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",201,2026226085160,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.165,16.15,3.073598861694336
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",206,2026227785216,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.23,36.18,2.6066088676452637
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",168,2026229485229,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.825,63.7,3.178278923034668
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",184,2026231185296,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.89,16.18,2.9700281620025635
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",207,2026227785217,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.12,36.22,1.9979348182678223
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",202,2026226085161,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.165,16.065,3.7360072135925293
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",208,2026227785218,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.18,36.06,2.4382519721984863
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",169,2026229485230,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.775,62.5,3.625271797180176
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",185,2026231185297,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.5,1,1,1.85,16.135,2.969292402267456
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",203,2026226085162,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1775,16.495,3.3943264484405518
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",209,2026227785219,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.22,36.06,2.334548234939575
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",186,2026231185298,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.825,15.8,2.6564505100250244
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",170,2026229485231,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.6,62.3,3.5785467624664307
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",204,2026226085163,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.22,16.035,3.3667447566986084
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",187,2026231185299,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.94,16.145,2.4106802940368652
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",210,2026227785220,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.37,36.18,3.2493886947631836
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",171,2026229485232,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,5.875,65.2,2.3069217205047607
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",188,2026231185300,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.5,1,1,1.7875,16.145,2.1426076889038086
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",172,2026229485233,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.075,62.0,2.3409111499786377
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",205,2026226085164,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,1.2825,15.965,3.6189053058624268
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",211,2026227785221,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.34,36.08,3.191417932510376
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",189,2026231185301,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.875,16.1,3.0213425159454346
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",212,2026227785222,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.63,35.26,2.707261562347412
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",206,2026226085165,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2475,16.255,2.862353801727295
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",173,2026229485234,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.375,66.2,3.79245924949646
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",190,2026231185302,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,3.0,3,0,1.86,16.22,1.9451696872711182
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",191,2026231185303,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,2.0,2,0,1.81,15.505,2.288504123687744
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",174,2026229485235,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.325,65.0,2.6713547706604004
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",213,2026227785223,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.36,35.48,3.6113312244415283
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",207,2026226085166,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1225,15.975,3.635546922683716
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",192,2026231185304,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.83,16.115,2.465116262435913
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",214,2026227785224,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.18,36.54,2.6480417251586914
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",175,2026229485236,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.975,60.65,2.8650588989257812
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",208,2026226085167,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2875,15.945,4.161248207092285
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",193,2026231185305,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.8075,15.95,2.9168832302093506
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",215,2026227785225,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.35,35.84,2.6215271949768066
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",176,2026229485237,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.925,63.65,3.3577301502227783
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",209,2026226085168,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2625,16.215,3.171278238296509
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",194,2026231185306,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.8425,15.91,2.4647865295410156
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",216,2026227785226,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.27,36.1,2.908355712890625
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",177,2026229485238,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.15,64.85,2.9308578968048096
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",195,2026231185307,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.9,15.675,2.4180562496185303
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",210,2026226085169,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.16,15.715,3.8078956604003906
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",217,2026227785227,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.26,36.3,3.047847270965576
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",196,2026231185308,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,2.0,2,0,1.8325,16.16,2.526930809020996
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",178,2026229485239,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.35,64.8,3.602147340774536
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",218,2026227785228,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.17,35.98,2.0331413745880127
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",211,2026226085170,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2525,16.305,2.8469080924987793
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",219,2026227785229,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.34,34.6,2.5313596725463867
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",197,2026231185309,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,3.0,3,0,1.8525,16.0,3.463487148284912
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",179,2026229485240,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.4,64.55,3.5758213996887207
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",212,2026226085171,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.27,16.02,3.6929099559783936
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",220,2026227785230,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.38,36.1,2.0207583904266357
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",198,2026231185310,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,1.0,1,0,1.7975,16.17,2.49299693107605
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",180,2026229485241,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.925,63.2,3.2190966606140137
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",213,2026226085172,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.14,15.71,2.76983642578125
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",221,2026227785231,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.62,35.2,2.961637258529663
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",199,2026231185311,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.8375,16.195,3.6135265827178955
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",222,2026227785232,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.2,36.36,2.286017656326294
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",181,2026229485242,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.35,64.45,3.7292864322662354
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",214,2026226085173,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.23,15.67,2.817805767059326
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",223,2026227785233,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.52,35.94,2.4332427978515625
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",200,2026231185312,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,0.0,0,0,1.87,16.345,3.6209914684295654
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",182,2026229485243,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.575,64.7,3.446282148361206
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",215,2026226085174,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2675,16.245,3.666527032852173
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",224,2026227785234,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.33,36.12,2.21461820602417
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.4,1",201,2026231185313,B,noisy,5,0.0002,0.0,1.0,0.4,1.0,5,2,5,400,2.0,2,0,1.855,16.195,2.7772979736328125
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",216,2026226085175,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2125,16.015,2.610474109649658
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",183,2026229485244,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.875,61.9,3.124995708465576
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",225,2026227785235,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.38,35.12,2.0822036266326904
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",0,2026232885163,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,1.9675,15.87,2.2178351879119873
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",226,2026227785236,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.15,35.82,1.9407308101654053
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",217,2026226085176,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.185,15.82,2.3414201736450195
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",184,2026229485245,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.8,66.15,2.479001522064209
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",1,2026232885164,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.025,15.85,2.1624529361724854
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",227,2026227785237,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.3,36.54,2.1614651679992676
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",218,2026226085177,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.24,15.81,2.3870880603790283
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",185,2026229485246,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.225,62.45,2.411533832550049
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",2,2026232885165,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0,16.31,2.246760845184326
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",228,2026227785238,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.2,35.46,2.1236109733581543
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",219,2026226085178,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1975,15.895,2.295238733291626
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",186,2026229485247,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.975,66.3,2.4419350624084473
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",3,2026232885166,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9625,16.23,2.1481637954711914
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",229,2026227785239,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.39,36.52,1.9389474391937256
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",220,2026226085179,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2775,16.09,2.2792532444000244
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",187,2026229485248,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.375,63.65,2.4795849323272705
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",230,2026227785240,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.49,36.62,1.9970133304595947
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",4,2026232885167,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0825,16.105,2.4558370113372803
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",221,2026226085180,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,1.1625,16.06,2.5631978511810303
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",231,2026227785241,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.35,36.46,2.2302253246307373
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",188,2026229485249,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.125,60.8,2.8493027687072754
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",5,2026232885168,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.005,15.96,2.524327278137207
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",222,2026226085181,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.0775,16.135,2.586580991744995
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",232,2026227785242,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.76,36.3,2.1641788482666016
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",6,2026232885169,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.065,16.165,2.369614839553833
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",189,2026229485250,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.05,64.8,2.7492623329162598
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",233,2026227785243,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.31,35.96,1.944687843322754
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",223,2026226085182,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.13,16.085,2.3309617042541504
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",7,2026232885170,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.1275,15.96,2.0530014038085938
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",190,2026229485251,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.55,64.65,2.225127696990967
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",234,2026227785244,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.35,36.18,1.9755313396453857
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",224,2026226085183,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1725,15.855,2.4162187576293945
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",8,2026232885171,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.1,16.215,2.1088223457336426
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",191,2026229485252,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.675,61.9,2.323317050933838
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",235,2026227785245,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.28,35.16,2.0017919540405273
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",225,2026226085184,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2175,15.975,2.41117525100708
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",9,2026232885172,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.0475,15.79,2.196803569793701
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",192,2026229485253,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.35,62.65,2.217209815979004
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",236,2026227785246,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.23,36.32,1.8458211421966553
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",226,2026226085185,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1425,15.955,2.3176045417785645
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",10,2026232885173,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9975,16.375,2.297574043273926
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",237,2026227785247,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.08,36.56,1.9125065803527832
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",193,2026229485254,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.975,61.35,2.293849229812622
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",11,2026232885174,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9725,15.62,2.236055374145508
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",238,2026227785248,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.28,36.36,1.9679906368255615
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",227,2026226085186,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.17,15.655,2.485358953475952
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",194,2026229485255,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.275,63.35,2.4092016220092773
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",239,2026227785249,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.35,36.48,2.0043513774871826
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",12,2026232885175,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0475,16.21,2.3489811420440674
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",228,2026226085187,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.19,16.0,2.428457736968994
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",195,2026229485256,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.95,62.7,2.4236223697662354
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",240,2026227785250,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.42,35.54,2.21911883354187
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",13,2026232885176,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.5,1,1,2.0325,15.74,2.2903265953063965
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",229,2026226085188,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.2075,15.785,2.487175464630127
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",196,2026229485257,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.125,62.8,2.524334192276001
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",241,2026227785251,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.06,35.36,1.9796314239501953
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",14,2026232885177,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9825,15.89,2.119594097137451
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",230,2026226085189,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1725,16.43,2.339426279067993
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",242,2026227785252,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.3,36.74,1.8945419788360596
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",197,2026229485258,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.4,65.45,2.7483108043670654
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",15,2026232885178,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0325,15.895,2.287175178527832
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",231,2026226085190,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2275,15.62,2.353919744491577
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",243,2026227785253,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.13,35.22,2.020643711090088
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",198,2026229485259,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.85,66.45,2.1591954231262207
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",16,2026232885179,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.055,16.255,2.3791847229003906
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",244,2026227785254,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.19,35.98,1.7940690517425537
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",232,2026226085191,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1725,15.835,2.2985596656799316
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",199,2026229485260,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.925,64.1,2.2790439128875732
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",17,2026232885180,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0125,15.725,2.074846029281616
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",245,2026227785255,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.22,37.02,1.73478364944458
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",233,2026226085192,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2075,16.195,2.2486488819122314
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",200,2026229485261,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.2,64.05,2.261003017425537
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",246,2026227785256,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,3.05,34.74,1.954148769378662
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",18,2026232885181,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9875,15.87,2.1985270977020264
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",234,2026226085193,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1425,16.155,2.362462282180786
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",201,2026229485262,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.625,63.5,2.3886830806732178
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",247,2026227785257,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.08,35.62,1.952026128768921
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",19,2026232885182,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,1.9475,15.995,2.223640203475952
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",235,2026226085194,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.295,15.545,2.402552366256714
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",248,2026227785258,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.06,36.42,1.9680416584014893
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",202,2026229485263,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.6,62.7,2.4740841388702393
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",20,2026232885183,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0825,15.97,2.228757381439209
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",236,2026226085195,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2425,15.99,2.3570361137390137
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",249,2026227785259,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.63,36.18,2.014885663986206
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",203,2026229485264,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.375,65.1,2.3812386989593506
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",21,2026232885184,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.155,15.82,2.2001516819000244
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",250,2026227785260,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.46,36.98,1.7610564231872559
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",237,2026226085196,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.27,15.735,2.2814669609069824
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",204,2026229485265,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.975,64.5,2.055976390838623
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",22,2026232885185,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,2.5,3,1,2.06,16.12,2.1199710369110107
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",251,2026227785261,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.29,35.78,1.8890669345855713
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",238,2026226085197,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.19,15.81,2.2271974086761475
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",23,2026232885186,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.115,16.225,2.1130869388580322
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",205,2026229485266,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.3,65.05,2.3904008865356445
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",252,2026227785262,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.18,35.02,2.0150041580200195
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",239,2026226085198,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.26,15.785,2.421353340148926
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",24,2026232885187,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.1475,16.125,2.2049665451049805
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",206,2026229485267,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.15,62.95,2.4946706295013428
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",253,2026227785263,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.29,36.1,2.0771164894104004
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",240,2026226085199,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2475,16.345,2.4501426219940186
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",25,2026232885188,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.5,1,1,2.0425,15.965,2.1373512744903564
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",254,2026227785264,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.26,36.68,1.9428071975708008
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",207,2026229485268,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.825,61.85,2.4807674884796143
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",26,2026232885189,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.1325,15.81,2.1487462520599365
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",255,2026227785265,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.1,35.36,1.8020763397216797
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",241,2026226085200,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,1.3375,15.94,2.4895856380462646
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",208,2026229485269,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.75,63.7,2.4292986392974854
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",256,2026227785266,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.21,36.74,1.8143954277038574
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",27,2026232885190,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.97,16.14,2.0960161685943604
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",242,2026226085201,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.295,16.075,2.292569160461426
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",209,2026229485270,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,7.25,63.85,2.489865779876709
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",257,2026227785267,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.49,36.96,1.8466997146606445
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",28,2026232885191,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0375,15.825,2.1301159858703613
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",243,2026226085202,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1225,16.02,2.208324432373047
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",258,2026227785268,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.15,35.84,2.0191292762756348
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",210,2026229485271,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.125,63.45,2.3782432079315186
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",29,2026232885192,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.0,15.855,2.2817182540893555
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",244,2026226085203,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.28,15.93,2.4169209003448486
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",259,2026227785269,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.15,36.46,1.8938415050506592
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",211,2026229485272,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.75,64.65,2.4945168495178223
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",30,2026232885193,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.5,1,1,1.9425,16.21,2.0677573680877686
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",245,2026226085204,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2775,16.1,2.306262254714966
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",260,2026227785270,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,2.99,35.96,2.147050619125366
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",31,2026232885194,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9225,16.335,2.0544493198394775
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",212,2026229485273,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.075,63.7,2.380575180053711
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",261,2026227785271,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.02,34.98,1.9478893280029297
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",246,2026226085205,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.2925,16.225,2.318493366241455
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",32,2026232885195,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.5,1,1,2.01,16.03,2.2768890857696533
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",213,2026229485274,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.125,64.6,2.594057559967041
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",262,2026227785272,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.31,35.34,1.8962311744689941
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",247,2026226085206,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1325,15.82,2.35707426071167
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",33,2026232885196,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0375,15.965,2.143028974533081
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",263,2026227785273,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.22,36.0,1.963097333908081
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",214,2026229485275,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.7,64.25,2.6105129718780518
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",248,2026226085207,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.125,16.415,2.3491342067718506
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",34,2026232885197,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.1125,15.89,2.0867886543273926
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",264,2026227785274,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,2.88,36.28,1.8416180610656738
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",249,2026226085208,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2725,16.18,2.277489185333252
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",215,2026229485276,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.675,64.45,2.647610664367676
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",265,2026227785275,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.17,35.84,1.942044734954834
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",35,2026232885198,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.1,15.91,2.362147331237793
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",250,2026226085209,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.185,15.84,2.3345117568969727
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",216,2026229485277,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.975,64.4,2.4503567218780518
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",266,2026227785276,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.06,36.46,1.8789668083190918
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",36,2026232885199,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9275,16.125,2.0882503986358643
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",267,2026227785277,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.12,35.58,1.7456881999969482
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",217,2026229485278,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.675,63.9,2.3342342376708984
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",251,2026226085210,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.2775,15.825,2.4849743843078613
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",37,2026232885200,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0675,16.24,2.1045827865600586
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",268,2026227785278,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.37,35.7,1.97088623046875
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",252,2026226085211,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1675,15.765,2.352694034576416
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",38,2026232885201,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.1275,16.05,2.228928565979004
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",218,2026229485279,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.975,63.2,2.8670594692230225
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",269,2026227785279,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,3.11,36.68,1.878605604171753
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",39,2026232885202,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0325,16.11,2.136486291885376
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",253,2026226085212,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1025,15.92,2.284825086593628
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",270,2026227785280,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.1,35.84,1.954127311706543
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",219,2026229485280,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.75,64.35,2.5456273555755615
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",40,2026232885203,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.0225,16.03,2.3012900352478027
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",254,2026226085213,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.19,15.855,2.3479530811309814
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",271,2026227785281,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,3.49,36.02,2.2682714462280273
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",220,2026229485281,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.25,63.3,2.3976595401763916
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",41,2026232885204,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0325,15.93,2.3224806785583496
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",255,2026226085214,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1975,15.58,2.2573466300964355
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",272,2026227785282,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.36,36.3,2.0583157539367676
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",221,2026229485282,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.0,64.55,2.526989459991455
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",42,2026232885205,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.0425,15.735,2.2451913356781006
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",256,2026226085215,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2125,16.09,2.2968637943267822
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",273,2026227785283,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.52,35.66,2.163698673248291
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",222,2026229485283,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.475,64.4,2.3150346279144287
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",257,2026226085216,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1975,16.17,2.186204433441162
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",43,2026232885206,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.5,1,1,2.125,15.865,2.3358824253082275
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",274,2026227785284,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.53,36.42,2.081237554550171
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",223,2026229485284,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.925,63.9,2.2717201709747314
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",275,2026227785285,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.19,36.16,1.8371105194091797
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",44,2026232885207,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.04,15.9,2.071237802505493
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",258,2026226085217,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.24,15.965,2.2306864261627197
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",224,2026229485285,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,7.4,62.65,2.5082051753997803
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",276,2026227785286,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.22,35.6,1.8390491008758545
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",45,2026232885208,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9325,15.82,2.1289303302764893
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",259,2026226085218,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.19,16.165,2.1730740070343018
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",277,2026227785287,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.23,35.88,1.973970651626587
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",225,2026229485286,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.525,63.6,2.4482357501983643
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",46,2026232885209,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,1.9625,15.745,2.102911949157715
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",260,2026226085219,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.25,15.66,2.3163270950317383
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",278,2026227785288,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,3.75,36.54,1.8725612163543701
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",47,2026232885210,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.035,16.135,2.1424145698547363
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",226,2026229485287,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.5,63.65,2.5357506275177
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",261,2026226085220,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.2075,16.115,2.241020441055298
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",279,2026227785289,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,2.97,36.48,1.9008338451385498
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",48,2026232885211,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.08,15.915,2.196857213973999
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",227,2026229485288,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.95,62.15,2.1870410442352295
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",262,2026226085221,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1475,15.99,2.3687422275543213
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",280,2026227785290,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.29,37.56,1.863572597503662
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",49,2026232885212,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.965,15.97,2.2845613956451416
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",228,2026229485289,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.35,64.55,2.340198278427124
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",281,2026227785291,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.01,36.2,1.9721596240997314
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",263,2026226085222,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2875,15.785,2.321791648864746
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",50,2026232885213,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.035,16.075,2.23209547996521
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",282,2026227785292,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.3,35.6,2.000157356262207
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",229,2026229485290,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.2,63.4,2.5426206588745117
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",264,2026226085223,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.36,16.09,2.2868492603302
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",283,2026227785293,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,2.97,35.22,2.0355474948883057
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",51,2026232885214,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0475,16.065,2.273996114730835
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",230,2026229485291,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.825,63.3,2.3071694374084473
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",265,2026226085224,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.2225,16.31,2.436349391937256
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",284,2026227785294,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,3.43,35.54,2.225257635116577
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",52,2026232885215,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.5,3,3,2.015,16.125,2.1513285636901855
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",231,2026229485292,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.65,65.3,2.290135145187378
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",266,2026226085225,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2225,15.96,2.2853057384490967
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",285,2026227785295,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.02,36.36,1.7701666355133057
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",53,2026232885216,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.1475,16.5,2.253156900405884
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",232,2026229485293,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.225,62.7,2.345140218734741
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",267,2026226085226,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1875,16.19,2.4773030281066895
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",286,2026227785296,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.19,35.88,1.9125902652740479
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",54,2026232885217,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,2.0,2,0,2.055,16.13,2.171696186065674
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",233,2026229485294,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.35,64.75,2.1819183826446533
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",287,2026227785297,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.32,36.0,1.889622449874878
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",268,2026226085227,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.16,15.695,2.200208902359009
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",55,2026232885218,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9825,16.075,2.123398780822754
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",234,2026229485295,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.65,62.8,2.220813035964966
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",288,2026227785298,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.11,36.7,1.9947845935821533
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",269,2026226085228,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.13,15.84,2.185044288635254
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",56,2026232885219,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0875,15.72,2.285651922225952
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",289,2026227785299,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.42,36.26,1.835949420928955
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",235,2026229485296,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.975,63.35,2.6028881072998047
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",270,2026226085229,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.3475,16.13,2.5111021995544434
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",57,2026232885220,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9025,16.24,2.4290435314178467
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",290,2026227785300,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.1,36.62,2.0473735332489014
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",236,2026229485297,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.35,65.3,2.095458507537842
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",271,2026226085230,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.1625,16.175,2.3139572143554688
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",58,2026232885221,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9725,16.13,2.1904165744781494
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",291,2026227785301,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.02,35.88,2.044477939605713
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",237,2026229485298,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.775,62.85,2.465151309967041
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",272,2026226085231,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1175,16.015,2.133751392364502
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",292,2026227785302,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.57,36.5,1.9642574787139893
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",59,2026232885222,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,2.0,2,0,2.0875,15.835,2.211390972137451
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",238,2026229485299,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.625,61.95,2.5036087036132812
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",273,2026226085232,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2225,16.125,2.311039686203003
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",293,2026227785303,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.29,36.64,1.9071533679962158
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",60,2026232885223,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9525,16.355,2.1114163398742676
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",274,2026226085233,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.3175,15.765,2.4162631034851074
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",294,2026227785304,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.46,35.52,2.2540178298950195
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",239,2026229485300,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.275,63.1,2.717348098754883
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",61,2026232885224,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0275,15.82,2.193678855895996
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",295,2026227785305,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.23,36.14,2.0043771266937256
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",275,2026226085234,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2525,15.755,2.254154920578003
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",62,2026232885225,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.5,1,1,1.9975,16.055,2.14680552482605
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",240,2026229485301,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.975,63.8,2.5606629848480225
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",296,2026227785306,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.28,35.32,2.0347721576690674
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",276,2026226085235,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.3,15.74,2.393803596496582
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",63,2026232885226,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.12,16.05,2.3135201930999756
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",241,2026229485302,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.125,65.2,2.5092670917510986
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",297,2026227785307,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.35,37.24,1.9742200374603271
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",277,2026226085236,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2,16.28,2.536841869354248
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",64,2026232885227,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.5,1,1,2.2025,15.95,2.4563450813293457
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",242,2026229485303,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.325,64.3,2.5808322429656982
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",298,2026227785308,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.31,35.88,2.0688393115997314
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",278,2026226085237,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.23,16.105,2.507333993911743
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",65,2026232885228,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.1825,16.02,2.521843433380127
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",299,2026227785309,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.2,36.1,1.9620890617370605
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",243,2026229485304,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.95,62.5,2.6324312686920166
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",66,2026232885229,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0975,15.9,2.131192207336426
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",300,2026227785310,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.23,36.16,1.9869358539581299
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",279,2026226085238,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.24,15.94,2.482072591781616
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",244,2026229485305,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.35,64.45,2.6986382007598877
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",301,2026227785311,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.04,36.28,2.113018035888672
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",67,2026232885230,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0425,15.815,2.264103651046753
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",280,2026226085239,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1375,16.135,2.533590078353882
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",245,2026229485306,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.025,63.5,2.7975170612335205
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",302,2026227785312,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.15,34.92,2.057356119155884
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",68,2026232885231,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.5,1,1,2.0225,15.92,2.2358903884887695
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",281,2026226085240,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.23,15.88,2.369962692260742
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",303,2026227785313,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.22,36.26,2.088475465774536
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",246,2026229485307,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.925,62.25,2.432924270629883
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",69,2026232885232,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.995,16.415,2.3052897453308105
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",282,2026226085241,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.265,15.905,2.335308790206909
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",304,2026227785314,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.25,36.86,2.0985584259033203
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",247,2026229485308,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.125,63.2,2.5862646102905273
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",70,2026232885233,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.07,16.265,2.338331699371338
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",283,2026226085242,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2075,15.935,2.3968043327331543
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",305,2026227785315,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.61,35.6,2.047940254211426
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",71,2026232885234,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.995,16.095,2.3099942207336426
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",248,2026229485309,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.25,61.35,2.4848968982696533
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",284,2026226085243,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2625,15.78,2.3542773723602295
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",306,2026227785316,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.36,36.26,1.8068630695343018
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",72,2026232885235,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0525,15.76,2.189279556274414
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",307,2026227785317,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.13,36.28,1.8954644203186035
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",249,2026229485310,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.675,63.85,2.68042254447937
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",285,2026226085244,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.22,16.095,2.192208766937256
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",73,2026232885236,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.0725,15.965,2.1978847980499268
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",308,2026227785318,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.15,34.98,1.9052135944366455
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",250,2026229485311,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.525,66.25,2.4496896266937256
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",286,2026226085245,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.23,16.28,2.415419101715088
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",74,2026232885237,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0875,16.18,2.2899413108825684
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",309,2026227785319,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.24,35.44,2.0069591999053955
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",287,2026226085246,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1375,15.995,2.2487761974334717
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",251,2026229485312,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.975,63.25,2.63665771484375
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",310,2026227785320,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.58,36.48,2.128279447555542
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",75,2026232885238,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0925,15.72,2.4103856086730957
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",288,2026226085247,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,1.2375,15.685,2.141382932662964
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",252,2026229485313,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.925,64.4,2.2490408420562744
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",311,2026227785321,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.14,36.7,2.066117286682129
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",76,2026232885239,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.98,16.045,2.077211618423462
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",289,2026226085248,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.285,15.88,2.284947633743286
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",253,2026229485314,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.85,63.4,2.5325074195861816
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",312,2026227785322,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.37,35.34,1.9080591201782227
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",77,2026232885240,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.5,1,1,2.0825,16.35,2.2240970134735107
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",290,2026226085249,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.225,16.145,2.1858084201812744
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",313,2026227785323,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.69,36.94,2.00646710395813
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",254,2026229485315,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.25,65.35,2.449744701385498
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",78,2026232885241,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.07,16.035,2.1297290325164795
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",291,2026226085250,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2875,16.065,2.3590641021728516
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",314,2026227785324,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.15,35.62,2.020085573196411
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",255,2026229485316,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.55,61.9,2.2360074520111084
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",79,2026232885242,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.1075,15.805,2.1810741424560547
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",292,2026226085251,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.165,15.965,2.075910806655884
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",315,2026227785325,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.55,35.98,2.122802972793579
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",80,2026232885243,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0375,16.105,2.0524723529815674
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",256,2026229485317,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.4,63.95,2.5059661865234375
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",293,2026226085252,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2425,16.26,2.1659908294677734
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",316,2026227785326,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.21,36.18,2.1354708671569824
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",81,2026232885244,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.0,16.21,2.17854380607605
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",257,2026229485318,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.075,64.45,2.1809539794921875
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",317,2026227785327,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,2.87,35.86,1.8176913261413574
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",294,2026226085253,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1725,16.305,2.1579318046569824
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",82,2026232885245,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9275,15.705,2.04728102684021
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",258,2026229485319,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.375,65.95,2.5387966632843018
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",318,2026227785328,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.32,35.72,1.8942861557006836
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",295,2026226085254,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.34,15.825,2.306752920150757
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",83,2026232885246,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0025,15.74,2.2942118644714355
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",319,2026227785329,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.35,35.88,1.9764690399169922
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",259,2026229485320,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.75,64.25,2.616469621658325
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",296,2026226085255,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2825,15.94,2.4036049842834473
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",84,2026232885247,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9825,16.07,2.110880136489868
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",320,2026227785330,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.51,36.2,2.0934455394744873
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",260,2026229485321,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.65,66.6,2.4043729305267334
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",297,2026226085256,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.22,15.795,2.2965571880340576
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",321,2026227785331,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.66,36.32,1.9319555759429932
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",85,2026232885248,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0575,15.85,2.425881862640381
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",261,2026229485322,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.175,63.4,2.3013858795166016
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",298,2026226085257,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.28,15.845,2.285555601119995
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",322,2026227785332,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.04,35.9,2.20758318901062
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",86,2026232885249,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,1.98,15.97,2.249160051345825
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",262,2026229485323,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.725,65.65,2.370415210723877
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",299,2026226085258,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.21,16.065,2.3095850944519043
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",323,2026227785333,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.21,36.5,1.7777047157287598
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",87,2026232885250,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.5,1,1,2.005,16.5,2.4302289485931396
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",324,2026227785334,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.24,35.36,1.9372460842132568
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",263,2026229485324,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.9,60.1,2.5281169414520264
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",300,2026226085259,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.25,15.65,2.420433521270752
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",88,2026232885251,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0275,15.805,2.271519422531128
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",325,2026227785335,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,2.0,2,0,3.16,36.56,1.9112050533294678
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",264,2026229485325,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.525,65.2,2.1468148231506348
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",301,2026226085260,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.205,16.23,2.491178512573242
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",89,2026232885252,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.11,15.985,2.1961843967437744
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",326,2026227785336,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.27,35.38,1.8443074226379395
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",302,2026226085261,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2775,15.945,2.2170450687408447
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",265,2026229485326,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.825,63.95,2.7777936458587646
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",90,2026232885253,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.99,15.95,2.103996992111206
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",327,2026227785337,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.39,35.32,2.0338566303253174
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",303,2026226085262,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2425,15.895,2.2080366611480713
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",266,2026229485327,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.375,65.2,2.2888894081115723
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",91,2026232885254,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,1.98,15.68,2.14560604095459
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",328,2026227785338,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,3.05,36.58,1.933997631072998
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",304,2026226085263,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.11,15.97,2.411430835723877
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",267,2026229485328,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.15,62.0,2.6137986183166504
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",329,2026227785339,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.24,36.6,2.2812998294830322
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",92,2026232885255,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.19,15.865,2.5851075649261475
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",305,2026226085264,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1075,15.98,2.230222463607788
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",330,2026227785340,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.2,36.92,1.8659586906433105
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",93,2026232885256,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.065,15.65,2.2879951000213623
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",268,2026229485329,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.5,61.4,2.4100935459136963
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",331,2026227785341,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,3.3,36.08,1.9382164478302002
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",306,2026226085265,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.255,15.84,2.201390504837036
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",94,2026232885257,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9575,16.105,2.1668155193328857
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",269,2026229485330,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.9,62.45,2.5750732421875
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",332,2026227785342,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.07,36.28,2.0592565536499023
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",307,2026226085266,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.18,15.835,2.2402665615081787
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",95,2026232885258,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.1625,16.16,2.207150459289551
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",270,2026229485331,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.55,61.65,2.3017497062683105
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",333,2026227785343,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.12,36.5,2.090487480163574
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",308,2026226085267,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1625,16.095,2.396970748901367
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",96,2026232885259,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.1125,16.315,2.1456856727600098
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",271,2026229485332,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.775,63.2,2.407414197921753
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",334,2026227785344,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.27,35.94,2.021256685256958
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",309,2026226085268,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.185,15.91,2.3900811672210693
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",97,2026232885260,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0475,16.085,2.182446241378784
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",272,2026229485333,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.2,64.85,2.3349435329437256
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",335,2026227785345,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.4,36.84,2.111908435821533
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",98,2026232885261,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.97,15.89,2.172079086303711
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",310,2026226085269,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2275,16.135,2.403658151626587
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",336,2026227785346,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.19,36.06,2.0855140686035156
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",273,2026229485334,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.4,64.4,2.756072759628296
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",99,2026232885262,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0275,15.9,2.30397891998291
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",311,2026226085270,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.21,16.07,2.586940288543701
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",337,2026227785347,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.06,36.7,2.1605026721954346
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",274,2026229485335,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.475,65.65,2.7732644081115723
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",100,2026232885263,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.995,15.92,2.2561686038970947
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",338,2026227785348,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.27,35.84,2.0849111080169678
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",312,2026226085271,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.3175,16.09,2.561203718185425
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",101,2026232885264,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9875,16.025,2.3840136528015137
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",275,2026229485336,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.35,65.05,2.78342866897583
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",339,2026227785349,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.5,35.7,2.11384916305542
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",313,2026226085272,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,1.235,15.68,2.3433802127838135
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",102,2026232885265,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.0975,15.755,2.406527519226074
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",340,2026227785350,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.2,36.42,2.0733418464660645
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",276,2026229485337,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.575,64.9,2.6437735557556152
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",314,2026226085273,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.3225,16.005,2.317307710647583
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",341,2026227785351,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.2,35.58,1.949303150177002
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",103,2026232885266,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.5,1,1,2.105,16.005,2.3558332920074463
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",315,2026226085274,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1875,16.065,2.242802381515503
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",277,2026229485338,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.6,62.0,2.638573408126831
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",342,2026227785352,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.34,36.34,1.9883131980895996
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",104,2026232885267,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0825,16.36,2.2910900115966797
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",316,2026226085275,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.255,15.91,2.3377628326416016
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",278,2026229485339,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.375,64.6,2.312431573867798
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",343,2026227785353,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.45,35.26,2.1448357105255127
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",105,2026232885268,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.0475,15.845,2.3090455532073975
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",317,2026226085276,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.255,15.88,2.3805596828460693
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",279,2026229485340,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,7.15,64.5,2.660369634628296
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",344,2026227785354,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.62,36.84,2.289893865585327
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",106,2026232885269,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.01,16.11,2.20796537399292
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",318,2026226085277,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,1.1175,15.805,2.382087469100952
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",345,2026227785355,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.32,36.68,1.9197983741760254
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",280,2026229485341,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.9,63.25,2.5589540004730225
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",107,2026232885270,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.97,15.785,2.2550504207611084
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",319,2026226085278,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1925,15.82,2.2912635803222656
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",346,2026227785356,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.2,35.4,2.0353987216949463
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",281,2026229485342,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.6,65.15,2.5058417320251465
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",108,2026232885271,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,1.9275,16.035,2.0826728343963623
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",347,2026227785357,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.4,35.56,2.063647747039795
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",320,2026226085279,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.185,15.865,2.246826648712158
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",282,2026229485343,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.275,62.55,2.6325559616088867
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",109,2026232885272,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.5,1,1,2.15,15.625,2.2537083625793457
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",348,2026227785358,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.03,35.46,2.102503776550293
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",321,2026226085280,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1225,15.96,2.3963570594787598
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",110,2026232885273,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.8775,16.185,2.2072649002075195
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",283,2026229485344,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.45,62.85,2.5097250938415527
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",349,2026227785359,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.06,36.78,1.81247878074646
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",322,2026226085281,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,1.27,15.99,2.5279829502105713
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",111,2026232885274,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0875,16.03,2.2578773498535156
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",350,2026227785360,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.33,35.74,1.9922583103179932
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",284,2026229485345,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.5,63.95,2.6335034370422363
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",323,2026226085282,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2975,16.17,2.354187488555908
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",351,2026227785361,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.23,35.58,1.8110535144805908
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",112,2026232885275,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,1.9525,16.105,2.3584086894989014
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",285,2026229485346,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.6,62.4,2.4925546646118164
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",352,2026227785362,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.32,35.72,1.9942560195922852
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",324,2026226085283,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.1325,16.08,2.3002469539642334
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",113,2026232885276,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0375,16.225,2.2679409980773926
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",286,2026229485347,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.625,62.9,2.363893985748291
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",353,2026227785363,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,2.94,35.06,1.9617383480072021
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",325,2026226085284,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2,16.13,2.1733407974243164
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",114,2026232885277,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9975,16.11,2.184882640838623
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",354,2026227785364,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.25,35.1,2.072418689727783
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",287,2026229485348,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.15,62.7,2.52358341217041
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",326,2026226085285,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,1.17,15.93,2.3259084224700928
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",115,2026232885278,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,1.895,15.975,2.135190010070801
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",355,2026227785365,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.57,35.74,2.1378419399261475
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",288,2026229485349,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.075,64.3,2.8195178508758545
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",327,2026226085286,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.3175,15.53,2.390425443649292
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",116,2026232885279,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9625,16.29,2.1924550533294678
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",356,2026227785366,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.4,35.48,2.1304268836975098
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",328,2026226085287,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.3125,15.57,2.22416353225708
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",117,2026232885280,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.08,16.16,2.084421157836914
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",289,2026229485350,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.675,62.2,2.4810070991516113
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",357,2026227785367,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.07,36.12,1.9607365131378174
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",329,2026226085288,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2025,16.09,2.2194864749908447
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",118,2026232885281,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.0425,16.22,2.170133113861084
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",290,2026229485351,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.55,64.75,2.441861867904663
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",358,2026227785368,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.35,35.46,2.209197521209717
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",330,2026226085289,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.225,16.13,2.378795862197876
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",119,2026232885282,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.0425,15.75,2.3121612071990967
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",359,2026227785369,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.55,36.38,2.004059076309204
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",291,2026229485352,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.65,66.75,2.742882490158081
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",331,2026226085290,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1,15.55,2.2794675827026367
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",120,2026232885283,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9575,15.77,2.2694077491760254
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",360,2026227785370,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.13,35.88,1.9872207641601562
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",292,2026229485353,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.225,64.85,2.8006463050842285
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",121,2026232885284,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.145,16.2,2.1493027210235596
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",361,2026227785371,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.4,35.7,1.991267204284668
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",332,2026226085291,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,1.25,16.135,2.460474967956543
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",293,2026229485354,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.45,63.75,2.4125618934631348
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",122,2026232885285,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.5,1,1,2.15,16.13,2.1228930950164795
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",362,2026227785372,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.28,35.28,2.0964083671569824
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",333,2026226085292,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.105,15.895,2.456310987472534
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",294,2026229485355,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.825,62.65,2.5303072929382324
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",123,2026232885286,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.5,1,1,2.0375,16.015,2.223921060562134
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",363,2026227785373,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.3,36.76,2.0928633213043213
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",334,2026226085293,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.215,16.055,2.304837703704834
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",364,2026227785374,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.18,35.34,2.0252747535705566
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",124,2026232885287,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.96,16.165,2.110280990600586
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",295,2026229485356,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.3,64.45,2.7287375926971436
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",335,2026226085294,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.3325,15.995,2.273319959640503
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",125,2026232885288,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.035,16.515,2.0878145694732666
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",365,2026227785375,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.3,36.0,2.2068073749542236
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",336,2026226085295,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,1.2,16.035,2.2252752780914307
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",296,2026229485357,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.525,62.45,2.701003074645996
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",366,2026227785376,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.31,36.48,1.9190852642059326
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",126,2026232885289,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.025,16.055,2.339794397354126
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",337,2026226085296,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2375,16.0,2.426769971847534
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",297,2026229485358,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.85,64.35,2.7602012157440186
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",367,2026227785377,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.43,34.98,2.2330057621002197
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",127,2026232885290,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.935,15.565,2.390573740005493
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",338,2026226085297,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.145,16.07,2.346456289291382
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",368,2026227785378,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.02,36.22,1.9929261207580566
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",298,2026229485359,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.875,66.15,2.7299013137817383
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",128,2026232885291,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,2,2,2.005,16.37,2.367097854614258
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",369,2026227785379,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.51,36.2,2.086601495742798
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",339,2026226085298,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,1.155,15.97,2.606820821762085
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",299,2026229485360,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,6.15,66.4,2.5481550693511963
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",129,2026232885292,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0725,16.105,2.3940672874450684
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",370,2026227785380,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.04,37.6,1.9234530925750732
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",340,2026226085299,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2275,16.095,2.4537148475646973
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",300,2026229485361,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.45,63.0,2.5627634525299072
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",130,2026232885293,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.01,15.885,2.2392725944519043
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",371,2026227785381,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.19,36.02,1.9262101650238037
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",341,2026226085300,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.17,16.08,2.3803980350494385
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",131,2026232885294,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.055,15.59,2.2518763542175293
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",372,2026227785382,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.32,35.9,2.001985549926758
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",301,2026229485362,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.075,63.3,2.7420451641082764
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",342,2026226085301,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2125,16.105,2.527951240539551
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",373,2026227785383,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.15,36.1,2.195608377456665
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",132,2026232885295,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0075,15.83,2.4261224269866943
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",302,2026229485363,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.175,66.15,2.4527149200439453
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",343,2026226085302,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.235,16.155,2.352778911590576
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",374,2026227785384,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.1,36.52,2.1444549560546875
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",133,2026232885296,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9075,15.53,2.297053575515747
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",303,2026229485364,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.525,64.9,2.5131020545959473
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",344,2026226085303,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.21,16.265,2.303875684738159
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",375,2026227785385,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.32,35.44,2.1469483375549316
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",134,2026232885297,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0425,15.985,2.3309335708618164
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",304,2026229485365,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.775,63.9,2.6694161891937256
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",376,2026227785386,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.49,35.88,2.245943069458008
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",345,2026226085304,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.26,15.975,2.5932936668395996
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",135,2026232885298,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.2075,15.88,2.3080813884735107
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",305,2026229485366,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.875,65.55,2.7841172218322754
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",377,2026227785387,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.35,35.76,2.1667721271514893
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",346,2026226085305,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.235,16.13,2.362621784210205
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",136,2026232885299,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,2.0,2,0,2.065,16.11,2.2223925590515137
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",378,2026227785388,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.16,35.48,2.2499072551727295
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",306,2026229485367,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.8,63.95,2.6157140731811523
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",347,2026226085306,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,1.235,15.875,2.4055356979370117
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",137,2026232885300,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0925,16.025,2.4980692863464355
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",379,2026227785389,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.15,35.2,2.0022521018981934
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",348,2026226085307,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.225,15.885,2.378861665725708
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",307,2026229485368,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.8,64.55,2.835174083709717
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",138,2026232885301,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.0875,16.025,2.298649549484253
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",380,2026227785390,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.39,36.08,2.1033530235290527
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",349,2026226085308,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,1.2425,15.735,2.34260630607605
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",139,2026232885302,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9875,16.02,2.237602710723877
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",308,2026229485369,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.75,64.05,2.666353940963745
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",381,2026227785391,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.43,36.88,2.050266742706299
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",350,2026226085309,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.285,15.735,2.2420437335968018
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",140,2026232885303,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9925,15.84,2.139098644256592
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",309,2026229485370,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.0,66.5,2.672426700592041
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",382,2026227785392,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.5,34.64,2.06107759475708
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",351,2026226085310,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.195,15.71,2.3787176609039307
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",141,2026232885304,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0725,16.075,2.424891948699951
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",383,2026227785393,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.15,36.76,2.15539288520813
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",310,2026229485371,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.85,64.05,2.71423077583313
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",352,2026226085311,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.16,16.07,2.2793684005737305
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",142,2026232885305,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.04,15.875,2.322946310043335
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",384,2026227785394,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,3.25,35.88,2.166196346282959
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",311,2026229485372,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.175,62.4,2.422767400741577
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",353,2026226085312,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2425,15.92,2.20527720451355
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",143,2026232885306,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.045,16.175,2.2359981536865234
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",385,2026227785395,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.75,35.42,2.3062942028045654
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",312,2026229485373,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.75,61.15,2.219209671020508
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",354,2026226085313,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.23,16.015,2.2278456687927246
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",144,2026232885307,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9275,16.325,2.2785494327545166
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",386,2026227785396,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.24,35.86,1.973827838897705
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",313,2026229485374,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.625,65.25,2.4761343002319336
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",387,2026227785397,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.33,35.16,1.9675366878509521
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",355,2026226085314,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.265,16.255,2.3958756923675537
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",145,2026232885308,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.5,1,1,1.8425,15.93,2.2558059692382812
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",314,2026229485375,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.05,62.15,2.6615710258483887
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",388,2026227785398,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.49,35.64,1.9336705207824707
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",146,2026232885309,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0075,15.94,2.188413619995117
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",356,2026226085315,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.165,16.545,2.4434871673583984
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",389,2026227785399,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.08,35.74,1.8887670040130615
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",315,2026229485376,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.775,61.95,2.4832570552825928
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",147,2026232885310,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.5,1,1,2.065,15.62,2.294647693634033
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",357,2026226085316,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2,15.815,2.427373170852661
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",390,2026227785400,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.32,35.98,1.8654956817626953
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",316,2026229485377,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.425,63.65,2.414877414703369
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",148,2026232885311,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9975,15.995,2.333073854446411
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",358,2026226085317,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1175,16.01,2.275866985321045
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",391,2026227785401,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.33,36.84,1.8966174125671387
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",317,2026229485378,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.925,64.15,2.2246975898742676
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",149,2026232885312,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,1.9225,16.215,2.2369277477264404
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",359,2026226085318,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2,15.875,2.5556445121765137
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",392,2026227785402,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.2,36.1,2.414729118347168
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",150,2026232885313,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0,16.005,2.230678081512451
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",318,2026229485379,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.35,62.75,2.4428954124450684
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",360,2026226085319,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.205,15.635,2.2740535736083984
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",393,2026227785403,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.36,36.24,2.1570827960968018
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",151,2026232885314,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.0375,16.18,2.1978330612182617
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",394,2026227785404,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.26,36.06,1.9346606731414795
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",361,2026226085320,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.205,16.265,2.23805832862854
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",319,2026229485380,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.7,63.5,2.865800380706787
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",152,2026232885315,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.8975,15.96,2.2720441818237305
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",395,2026227785405,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.31,36.92,2.003617763519287
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",362,2026226085321,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.0975,15.905,2.406688928604126
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",320,2026229485381,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.25,64.35,2.706233501434326
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",153,2026232885316,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.0725,16.145,2.2649247646331787
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",396,2026227785406,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,3.43,36.26,2.171215534210205
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",363,2026226085322,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2725,16.095,2.2816944122314453
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",321,2026229485382,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.2,68.05,2.500204086303711
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",397,2026227785407,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,3.16,36.8,1.829246997833252
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",154,2026232885317,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.075,15.985,2.353226661682129
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",364,2026226085323,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.19,16.085,2.3153748512268066
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",322,2026229485383,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.1,62.45,2.268388509750366
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",398,2026227785408,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.44,35.06,1.87563157081604
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",155,2026232885318,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0675,15.99,2.1214442253112793
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",365,2026226085324,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.3325,15.77,2.3117170333862305
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",399,2026227785409,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.3,35.98,1.8460536003112793
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",323,2026229485384,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.15,64.8,2.216686725616455
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",156,2026232885319,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.5,1,1,1.94,16.01,2.2097256183624268
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",400,2026227785410,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.51,34.98,2.1202518939971924
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",366,2026226085325,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.25,16.06,2.3748950958251953
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",324,2026229485385,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.275,63.1,2.5511155128479004
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",157,2026232885320,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.04,15.775,2.293771505355835
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",401,2026227785411,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.37,36.78,1.9627878665924072
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",367,2026226085326,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.2125,16.045,2.396789073944092
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",325,2026229485386,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.475,63.3,2.296666145324707
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",158,2026232885321,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.5,1,1,1.945,15.905,2.276560068130493
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",402,2026227785412,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.03,36.18,1.8067572116851807
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",368,2026226085327,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1775,16.05,2.3895368576049805
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",326,2026229485387,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.95,65.65,2.662461519241333
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",159,2026232885322,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.1025,16.395,2.3493599891662598
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",403,2026227785413,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.28,36.44,2.042421579360962
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",369,2026226085328,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.15,15.92,2.217360496520996
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",327,2026229485388,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.25,64.3,2.325143575668335
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",404,2026227785414,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.42,35.1,2.265653133392334
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",160,2026232885323,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0625,15.865,2.325917959213257
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",370,2026226085329,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.26,16.215,2.3782005310058594
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",405,2026227785415,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.17,35.94,2.1931509971618652
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",328,2026229485389,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.85,65.55,2.60077166557312
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",161,2026232885324,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0825,16.1,2.397693157196045
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",371,2026226085330,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.1625,16.44,2.435011148452759
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",406,2026227785416,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.13,35.72,1.7878448963165283
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",162,2026232885325,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.005,15.98,2.2109529972076416
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",329,2026229485390,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.425,62.2,2.589801788330078
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",407,2026227785417,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.29,36.28,2.0567057132720947
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",372,2026226085331,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.245,16.21,2.4535694122314453
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",163,2026232885326,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.5,2,1,2.0975,16.395,2.454523801803589
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",330,2026229485391,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.8,63.0,2.534722089767456
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",408,2026227785418,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.09,35.2,1.8556888103485107
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",373,2026226085332,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.24,15.515,2.3497281074523926
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",164,2026232885327,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.1,16.075,2.2169642448425293
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",409,2026227785419,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,3.13,36.26,1.9014918804168701
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",331,2026229485392,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.675,64.2,2.5787651538848877
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",374,2026226085333,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.185,16.325,2.4426305294036865
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",165,2026232885328,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.5,1,1,1.99,15.885,2.312638998031616
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",410,2026227785420,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.11,35.52,1.9825289249420166
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",332,2026229485393,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.8,63.35,2.454741954803467
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",375,2026226085334,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2375,15.875,2.6655166149139404
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",411,2026227785421,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.7,36.0,2.0512781143188477
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",166,2026232885329,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0725,16.15,2.289074182510376
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",333,2026229485394,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.3,62.15,2.516737937927246
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",412,2026227785422,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.36,35.3,2.0283925533294678
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",376,2026226085335,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2025,15.73,2.3876099586486816
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",167,2026232885330,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0225,15.825,2.192781448364258
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",334,2026229485395,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.9,65.75,2.2127926349639893
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",413,2026227785423,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.31,36.36,1.8159420490264893
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",168,2026232885331,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.0075,15.96,2.056305408477783
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",377,2026226085336,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.165,15.95,2.284799575805664
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",414,2026227785424,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,3.13,36.22,1.9203319549560547
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",335,2026229485396,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.8,63.45,2.4632699489593506
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",169,2026232885332,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9875,16.26,2.0999221801757812
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",378,2026226085337,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.27,15.785,2.2530014514923096
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",415,2026227785425,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,3.42,36.52,1.921959638595581
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",336,2026229485397,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.925,64.35,2.339900493621826
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",170,2026232885333,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.1175,15.745,2.0913302898406982
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",379,2026226085338,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2225,15.83,2.337541103363037
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",416,2026227785426,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.39,35.88,1.969048023223877
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",171,2026232885334,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,1.96,15.94,2.0757079124450684
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",337,2026229485398,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.625,66.15,2.7150397300720215
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",380,2026226085339,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1975,15.695,2.553366184234619
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",417,2026227785427,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.34,36.02,2.1222400665283203
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",172,2026232885335,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0575,16.08,2.1278576850891113
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",418,2026227785428,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.1,36.0,1.9090533256530762
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",338,2026229485399,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.45,64.55,2.692183256149292
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",381,2026226085340,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.205,15.885,2.2263243198394775
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",173,2026232885336,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9375,16.065,2.0661416053771973
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",419,2026227785429,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.1,36.24,1.9218225479125977
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",382,2026226085341,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1975,16.08,2.171757936477661
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",339,2026229485400,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.875,62.95,2.40297532081604
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",174,2026232885337,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.13,15.72,2.3837730884552
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",420,2026227785430,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.53,36.12,2.0624639987945557
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",383,2026226085342,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.29,16.085,2.3001694679260254
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",340,2026229485401,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.4,63.1,2.4288129806518555
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",175,2026232885338,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.07,16.01,2.2129855155944824
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",421,2026227785431,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.67,36.02,2.0435383319854736
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",384,2026226085343,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1375,16.06,2.3338046073913574
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",341,2026229485402,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.8,62.1,2.666339874267578
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",176,2026232885339,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.5,2,1,1.9675,16.395,2.205308437347412
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",422,2026227785432,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.02,36.62,1.9491238594055176
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",385,2026226085344,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1875,16.015,2.473695755004883
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",423,2026227785433,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.49,35.84,1.9714179039001465
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",177,2026232885340,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.085,16.0,2.259115695953369
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",342,2026229485403,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.45,62.4,2.8057150840759277
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",386,2026226085345,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,2.0,2,0,1.2525,15.86,2.476138114929199
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",424,2026227785434,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.25,36.22,2.0565977096557617
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",178,2026232885341,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.1025,16.1,2.3168933391571045
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",343,2026229485404,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.225,66.05,2.4935152530670166
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",425,2026227785435,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.25,35.8,1.9598431587219238
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",387,2026226085346,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.25,15.9,2.4947550296783447
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",179,2026232885342,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.1375,15.945,2.3310670852661133
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",344,2026229485405,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.325,63.65,2.518594264984131
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",426,2026227785436,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,3.27,36.18,1.93735671043396
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",388,2026226085347,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1275,15.805,2.409167528152466
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",180,2026232885343,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0425,15.805,2.365767240524292
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",427,2026227785437,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,2.95,36.42,1.9714975357055664
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",345,2026229485406,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.725,64.15,2.6785778999328613
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",389,2026226085348,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.2,16.105,2.2151005268096924
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",181,2026232885344,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.05,16.0,2.3715810775756836
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",428,2026227785438,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.18,35.54,1.9788269996643066
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",346,2026229485407,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.65,62.8,2.5995590686798096
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",390,2026226085349,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.175,16.315,2.3180973529815674
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",182,2026232885345,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.5,1,1,2.035,15.925,2.2954561710357666
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",429,2026227785439,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.36,36.34,2.0314013957977295
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",347,2026229485408,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.425,65.7,2.5930161476135254
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",391,2026226085350,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2025,15.98,2.388061761856079
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",430,2026227785440,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.39,36.48,2.054058074951172
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",183,2026232885346,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.03,15.785,2.1953203678131104
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",348,2026229485409,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.325,65.45,2.525059938430786
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",431,2026227785441,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.32,36.54,2.0073037147521973
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",392,2026226085351,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.185,15.955,2.2421505451202393
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",184,2026232885347,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0675,16.06,2.301483631134033
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",432,2026227785442,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.14,35.7,2.1077475547790527
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",349,2026229485410,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.275,63.5,2.678445816040039
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",393,2026226085352,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.28,15.91,2.5116214752197266
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",185,2026232885348,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.1425,16.34,2.3143701553344727
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",433,2026227785443,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.22,36.12,2.1760003566741943
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",394,2026226085353,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,1.25,16.24,2.351780652999878
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",186,2026232885349,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9975,15.95,2.251559257507324
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",350,2026229485411,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.85,64.75,2.6879971027374268
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",434,2026227785444,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.44,36.06,2.275517225265503
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",187,2026232885350,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0325,15.675,2.2176034450531006
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",395,2026226085354,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1925,16.05,2.3764030933380127
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",351,2026229485412,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.7,63.85,2.737807035446167
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",435,2026227785445,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.2,35.98,2.175898790359497
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",188,2026232885351,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.1025,16.28,2.2487633228302
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",396,2026226085355,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2625,16.025,2.3988523483276367
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",352,2026229485413,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.725,64.45,2.198578357696533
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",436,2026227785446,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.16,36.82,1.8357250690460205
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",189,2026232885352,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.5,1,1,2.09,16.005,2.2418088912963867
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",397,2026226085356,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.215,16.005,2.4292919635772705
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",437,2026227785447,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.13,35.92,2.053380250930786
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",353,2026229485414,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.075,65.2,2.7069995403289795
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",190,2026232885353,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9275,15.82,2.1815905570983887
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",398,2026226085357,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2325,16.025,2.2370059490203857
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",438,2026227785448,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.38,35.84,1.9264037609100342
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",354,2026229485415,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.575,64.35,2.416475534439087
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",191,2026232885354,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0475,15.87,2.2296483516693115
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",399,2026226085358,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.135,15.97,2.266737699508667
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",439,2026227785449,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.28,36.26,2.0747597217559814
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",355,2026229485416,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.025,64.5,2.3445515632629395
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",192,2026232885355,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.095,16.025,2.2876875400543213
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",440,2026227785450,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.17,35.46,2.0151000022888184
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",400,2026226085359,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2675,15.79,2.469501256942749
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",356,2026229485417,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.075,66.55,2.4791274070739746
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",193,2026232885356,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,1.97,15.79,2.2027571201324463
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",441,2026227785451,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,3.2,36.88,2.1004531383514404
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",401,2026226085360,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.185,16.08,2.169743776321411
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",357,2026229485418,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.9,62.45,2.459751605987549
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",442,2026227785452,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.52,36.58,1.8254952430725098
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",194,2026232885357,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.955,16.145,2.227027177810669
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",402,2026226085361,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.205,16.025,2.3808536529541016
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",443,2026227785453,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.44,36.6,2.147244691848755
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",358,2026229485419,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.25,62.35,2.448664426803589
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",195,2026232885358,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.03,16.255,2.4206466674804688
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",403,2026226085362,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1975,15.87,2.281991958618164
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",444,2026227785454,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.11,35.42,1.8822050094604492
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",196,2026232885359,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0125,16.045,2.227855682373047
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",359,2026229485420,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.4,64.0,2.5907912254333496
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",404,2026226085363,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.2275,16.28,2.356759548187256
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",445,2026227785455,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.37,36.46,2.0014209747314453
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",197,2026232885360,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9275,16.515,2.274141550064087
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",360,2026229485421,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.3,64.7,2.3957901000976562
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",405,2026226085364,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.2525,15.865,2.255579710006714
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",446,2026227785456,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.27,35.96,1.9928655624389648
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",198,2026232885361,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.97,16.075,2.1778981685638428
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",361,2026229485422,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.95,63.65,2.4897587299346924
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",447,2026227785457,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.24,37.58,2.026341438293457
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",406,2026226085365,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1075,16.225,2.553006649017334
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",199,2026232885362,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,1.99,16.115,2.1140949726104736
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",448,2026227785458,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,3.31,35.14,2.0988895893096924
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",362,2026229485423,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.65,60.8,2.276339530944824
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",407,2026226085366,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.12,15.91,2.2988243103027344
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",200,2026232885363,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.075,16.085,2.0515530109405518
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",449,2026227785459,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.3,35.54,2.0897281169891357
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",363,2026229485424,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.6,62.35,2.483030080795288
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",408,2026226085367,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,1.2125,16.065,2.301663875579834
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",201,2026232885364,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,1.9475,15.75,2.0593831539154053
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",450,2026227785460,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.48,35.66,2.0550642013549805
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",364,2026229485425,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.6,66.05,2.2840895652770996
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",409,2026226085368,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.265,15.95,2.400130271911621
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",202,2026232885365,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9375,16.31,2.311475992202759
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",451,2026227785461,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.29,35.84,1.9249024391174316
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",365,2026229485426,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.325,60.85,2.6107850074768066
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",410,2026226085369,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2675,16.415,2.4541850090026855
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",203,2026232885366,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9875,15.82,2.241020679473877
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",452,2026227785462,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.41,36.5,1.9653854370117188
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",366,2026229485427,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.225,63.45,2.5248446464538574
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",453,2026227785463,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.38,36.34,1.8768112659454346
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",411,2026226085370,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.15,15.95,2.3161253929138184
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",204,2026232885367,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0625,15.885,2.1614105701446533
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",454,2026227785464,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.21,35.52,1.914602279663086
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",367,2026229485428,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.225,63.3,2.263439655303955
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",205,2026232885368,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,1.9875,15.85,2.122732639312744
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",412,2026226085371,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1575,16.0,2.318568706512451
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",455,2026227785465,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,2.95,36.46,1.989272117614746
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",206,2026232885369,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.105,15.81,2.273052215576172
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",368,2026229485429,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.975,64.3,2.5634360313415527
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",413,2026226085372,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.225,16.485,2.291203498840332
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",456,2026227785466,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.39,36.96,1.870392084121704
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",207,2026232885370,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.145,15.69,2.265994071960449
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",414,2026226085373,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1775,16.17,2.3023581504821777
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",369,2026229485430,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,5.975,63.5,2.4959230422973633
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",457,2026227785467,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.27,35.28,1.8489551544189453
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",208,2026232885371,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.055,15.85,2.281390428543091
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",415,2026226085374,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.21,16.16,2.4056472778320312
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",370,2026229485431,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.3,62.2,2.39561128616333
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",458,2026227785468,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,3.2,36.82,2.1332015991210938
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",209,2026232885372,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.125,15.995,2.452446460723877
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",459,2026227785469,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.5,35.96,2.0362448692321777
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",416,2026226085375,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2,15.915,2.4323360919952393
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",371,2026229485432,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,5.75,66.2,2.3918204307556152
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",460,2026227785470,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,2.97,36.06,1.8747622966766357
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",210,2026232885373,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0125,15.665,2.322444200515747
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",417,2026226085376,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1925,16.2,2.462907075881958
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",372,2026229485433,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.0,63.65,2.592471122741699
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",461,2026227785471,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.33,35.3,2.173551321029663
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",211,2026232885374,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.03,16.115,2.328963041305542
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",418,2026226085377,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.205,15.81,2.5758917331695557
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",373,2026229485434,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.475,63.05,2.38102388381958
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",462,2026227785472,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.28,36.26,1.9505693912506104
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",212,2026232885375,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.06,15.715,2.277285099029541
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",419,2026226085378,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2225,15.775,2.4553112983703613
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",374,2026229485435,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.725,61.15,2.5187249183654785
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",463,2026227785473,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.29,36.38,1.9157872200012207
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",213,2026232885376,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.99,15.6,2.181689977645874
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",420,2026226085379,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.215,15.72,2.2669246196746826
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",464,2026227785474,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.02,36.3,2.1878416538238525
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",375,2026229485436,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.15,62.8,2.431381940841675
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",214,2026232885377,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0625,15.785,2.2237942218780518
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",465,2026227785475,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.32,36.74,1.9315788745880127
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",421,2026226085380,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.1875,15.78,2.211773157119751
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",376,2026229485437,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.3,62.0,2.4161548614501953
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",215,2026232885378,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.5,1,1,2.1875,16.13,2.262944459915161
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",466,2026227785476,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.47,35.88,2.136441946029663
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",422,2026226085381,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.28,16.235,2.2400972843170166
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",377,2026229485438,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.275,65.1,2.4545013904571533
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",216,2026232885379,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0225,15.795,2.2588610649108887
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",467,2026227785477,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.34,35.78,2.2607035636901855
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",423,2026226085382,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2725,16.165,2.2999062538146973
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",378,2026229485439,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.7,66.4,2.503990650177002
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",217,2026232885380,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0925,15.925,2.2131354808807373
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",468,2026227785478,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.13,35.58,1.979149580001831
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",424,2026226085383,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2175,16.105,2.2593555450439453
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",379,2026229485440,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.85,65.3,2.416311502456665
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",218,2026232885381,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.025,15.79,2.25544810295105
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",469,2026227785479,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.24,34.96,1.9231257438659668
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",425,2026226085384,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.31,16.04,2.3280835151672363
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",380,2026229485441,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.325,64.4,2.1305739879608154
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",470,2026227785480,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.0,37.2,1.993711233139038
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",219,2026232885382,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9475,15.875,2.2454025745391846
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",426,2026226085385,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.235,16.18,2.2581989765167236
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",471,2026227785481,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.55,36.44,2.1275453567504883
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",220,2026232885383,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.98,15.9,2.101386308670044
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",381,2026229485442,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.05,63.2,2.5972933769226074
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",427,2026226085386,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.175,16.185,2.269796848297119
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",472,2026227785482,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.35,35.78,2.0430190563201904
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",221,2026232885384,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.105,16.095,2.285226821899414
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",382,2026229485443,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.25,64.9,2.6826086044311523
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",428,2026226085387,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2525,15.985,2.479362964630127
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",473,2026227785483,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.24,36.06,1.9592232704162598
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",222,2026232885385,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9875,16.255,2.149679183959961
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",383,2026229485444,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.8,66.05,2.493499279022217
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",474,2026227785484,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.2,35.56,2.060521125793457
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",429,2026226085388,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,1.3175,15.74,2.3346059322357178
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",223,2026232885386,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,1.9425,15.695,2.062812566757202
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",384,2026229485445,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.575,63.0,2.3863627910614014
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",475,2026227785485,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.63,35.96,1.9528393745422363
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",430,2026226085389,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.2425,16.03,2.4304418563842773
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",224,2026232885387,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.04,15.81,2.4739973545074463
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",476,2026227785486,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.08,36.46,1.949711799621582
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",385,2026229485446,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.9,64.45,2.6225197315216064
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",431,2026226085390,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.15,15.955,2.393838405609131
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",225,2026232885388,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.045,15.955,2.2263031005859375
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",477,2026227785487,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.25,35.32,2.089313507080078
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",386,2026229485447,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.55,63.45,2.515516996383667
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0001|pi0|k1,1,1",432,2026226085391,B,noisy,5,0.0001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,1.3325,15.835,2.446631669998169
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",226,2026232885389,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.125,16.08,2.266003131866455
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",478,2026227785488,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,2.99,35.46,2.0838136672973633
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",387,2026229485448,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.5,62.75,2.328019857406616
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",227,2026232885390,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.07,15.59,2.295314311981201
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",479,2026227785489,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.16,35.96,2.108499765396118
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",0,2026234585214,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,2,2,2.9925,16.195,3.2900891304016113
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",388,2026229485449,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.425,66.4,2.312668800354004
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",228,2026232885391,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.175,16.275,2.425137996673584
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",480,2026227785490,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.28,35.82,2.135305404663086
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",389,2026229485450,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.675,65.7,2.7019991874694824
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",229,2026232885392,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.075,15.905,2.2102200984954834
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",481,2026227785491,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.39,37.1,2.089545965194702
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",1,2026234585215,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0875,15.875,3.2608213424682617
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",482,2026227785492,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.32,35.56,2.096446990966797
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",230,2026232885393,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0475,15.89,2.322068452835083
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",390,2026229485451,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,7.45,60.85,2.4081318378448486
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",2,2026234585216,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,3.01,16.01,3.2501943111419678
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",483,2026227785493,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.02,35.22,1.937178373336792
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",231,2026232885394,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.06,16.045,2.1391093730926514
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",391,2026229485452,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.075,64.45,2.5438623428344727
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",484,2026227785494,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.33,35.72,1.9736692905426025
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",232,2026232885395,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.5,2,1,2.0425,16.37,2.1697959899902344
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",3,2026234585217,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0325,16.015,3.2597575187683105
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",392,2026229485453,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.075,65.25,2.5443544387817383
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",485,2026227785495,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.41,36.9,1.9837822914123535
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",233,2026232885396,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0775,16.07,2.1429121494293213
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",393,2026229485454,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.6,63.05,2.5086452960968018
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",486,2026227785496,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.1,36.08,2.0034141540527344
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",4,2026234585218,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.1275,15.89,3.569566249847412
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",234,2026232885397,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,2.0,3,2,2.0425,16.05,2.207176685333252
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",487,2026227785497,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.5,36.6,1.9870145320892334
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",394,2026229485455,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.675,64.3,2.4453017711639404
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",235,2026232885398,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.1075,16.185,2.324624538421631
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",5,2026234585219,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.05,15.62,3.299137592315674
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",488,2026227785498,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.47,36.02,2.0569281578063965
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",395,2026229485456,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.35,63.1,2.3969969749450684
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",236,2026232885399,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0225,15.835,2.186668634414673
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",489,2026227785499,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.46,36.16,1.7681140899658203
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",6,2026234585220,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.165,15.785,3.3604094982147217
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",396,2026229485457,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.4,64.2,2.297292470932007
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",237,2026232885400,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,2.0,2,0,1.955,16.415,2.0397374629974365
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",490,2026227785500,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.06,35.16,1.9434897899627686
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",397,2026229485458,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.475,67.3,2.371467113494873
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",238,2026232885401,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.045,16.13,2.1742210388183594
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",491,2026227785501,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,3.01,35.32,2.0936315059661865
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",7,2026234585221,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.9725,16.16,3.40529203414917
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",239,2026232885402,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.035,15.99,2.1602184772491455
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",492,2026227785502,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.14,35.62,2.0256900787353516
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",398,2026229485459,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.5,66.0,2.555651903152466
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",8,2026234585222,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0875,15.82,3.5337934494018555
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",240,2026232885403,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9625,16.01,2.143176555633545
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",493,2026227785503,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.33,35.96,2.18648099899292
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",399,2026229485460,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.4,61.65,2.4870665073394775
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",241,2026232885404,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.0725,15.67,2.0870821475982666
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",494,2026227785504,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,3.16,36.22,2.0223641395568848
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",400,2026229485461,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.475,64.65,2.3255341053009033
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",9,2026234585223,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.8525,16.01,3.261403799057007
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",495,2026227785505,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.33,36.52,1.7881519794464111
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",242,2026232885405,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0375,15.845,2.1663575172424316
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",401,2026229485462,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.525,63.5,2.5732924938201904
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",496,2026227785506,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,3.12,35.88,1.8620891571044922
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",243,2026232885406,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.1125,16.03,2.1939175128936768
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",10,2026234585224,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.9975,15.98,3.5641849040985107
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",497,2026227785507,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.36,36.9,2.037602424621582
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",402,2026229485463,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.45,62.25,2.5758190155029297
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",244,2026232885407,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.98,16.055,2.2221851348876953
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",498,2026227785508,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,3.3,35.72,1.8829436302185059
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",11,2026234585225,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.8775,16.205,3.2131423950195312
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",403,2026229485464,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.625,65.35,2.2475686073303223
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",245,2026232885408,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0,16.06,2.006481647491455
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",499,2026227785509,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.48,35.52,1.8564419746398926
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",404,2026229485465,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.25,65.3,2.408803939819336
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",246,2026232885409,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9725,15.73,2.087125778198242
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",12,2026234585226,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.1925,15.81,3.4071993827819824
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",500,2026227785510,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.31,36.14,1.9775242805480957
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",247,2026232885410,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9675,16.14,2.0652427673339844
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",405,2026229485466,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.0,65.25,2.4578680992126465
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",501,2026227785511,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.4,35.24,2.051776647567749
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",13,2026234585227,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,3.195,15.91,3.4578981399536133
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",248,2026232885411,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0225,16.015,2.230090856552124
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",406,2026229485467,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.225,66.05,2.388423204421997
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",502,2026227785512,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.03,36.94,2.040337324142456
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",249,2026232885412,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.1075,15.9,2.399779796600342
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",503,2026227785513,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,2.8,36.04,1.9032812118530273
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",407,2026229485468,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.275,65.15,2.3078773021698
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",14,2026234585228,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0275,15.915,3.7330100536346436
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",504,2026227785514,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.38,36.38,1.8710403442382812
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",250,2026232885413,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,1.9475,15.875,2.0406382083892822
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",408,2026229485469,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.725,61.1,2.4470973014831543
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",505,2026227785515,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.21,35.94,1.7967939376831055
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",251,2026232885414,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.5,1,1,2.0175,16.07,2.110424518585205
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",15,2026234585229,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.015,16.04,3.4693994522094727
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",409,2026229485470,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.875,65.3,2.3671891689300537
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",506,2026227785516,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,2.88,35.08,2.009187698364258
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",252,2026232885415,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.1,15.84,2.315014123916626
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",410,2026229485471,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.825,64.05,2.203655958175659
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",507,2026227785517,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.38,36.16,1.967137336730957
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",16,2026234585230,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,3.1,15.68,3.401742935180664
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",253,2026232885416,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.09,16.525,2.464573621749878
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",411,2026229485472,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.8,61.55,2.1571884155273438
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",508,2026227785518,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.26,35.24,1.9523487091064453
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",17,2026234585231,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.5,1,1,2.95,16.335,3.1808230876922607
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",254,2026232885417,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.1475,16.13,2.1843512058258057
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",509,2026227785519,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.3,36.78,1.9528529644012451
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",412,2026229485473,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.6,64.1,2.5293662548065186
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",255,2026232885418,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.5,1,1,2.07,15.88,2.127765655517578
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",510,2026227785520,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.22,37.2,1.9185009002685547
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",413,2026229485474,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.75,64.65,2.303314685821533
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",18,2026234585232,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,3.1625,16.075,3.32768177986145
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",511,2026227785521,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.13,35.88,1.8383054733276367
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",256,2026232885419,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.98,16.18,2.2144687175750732
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",414,2026229485475,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.6,66.5,2.465874671936035
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",512,2026227785522,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.19,36.38,2.0733087062835693
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",257,2026232885420,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.1325,15.65,2.2731781005859375
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",19,2026234585233,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0925,15.895,3.6196234226226807
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",513,2026227785523,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.36,36.42,1.9687650203704834
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",415,2026229485476,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.4,63.85,2.8528661727905273
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",258,2026232885421,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0225,16.205,2.2925407886505127
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",514,2026227785524,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.12,35.78,1.78652024269104
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",20,2026234585234,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.005,15.885,3.531661033630371
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",416,2026229485477,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.825,64.65,2.3907418251037598
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",259,2026232885422,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.5,1,1,2.125,15.715,2.1418206691741943
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",515,2026227785525,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.34,36.4,1.943345308303833
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",260,2026232885423,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.1375,16.195,2.2183146476745605
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",417,2026229485478,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.225,64.15,2.645906448364258
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",21,2026234585235,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,2,2,2.9375,16.49,3.391477346420288
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",516,2026227785526,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.29,35.64,2.3039467334747314
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",261,2026232885424,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.5,2,1,2.15,16.12,2.214970350265503
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",418,2026229485479,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.625,63.75,2.5597708225250244
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",517,2026227785527,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.35,35.9,1.9903345108032227
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",22,2026234585236,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.985,15.82,3.558155059814453
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",262,2026232885425,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,1.96,16.335,2.1612906455993652
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",518,2026227785528,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,3.29,35.34,1.9836928844451904
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",419,2026229485480,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.1,63.7,2.6075799465179443
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",263,2026232885426,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,1.9475,16.07,2.118140697479248
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",519,2026227785529,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.16,36.62,2.00740647315979
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",23,2026234585237,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.5,1,1,3.1075,16.055,3.5385076999664307
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",420,2026229485481,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.25,65.2,2.5413434505462646
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",264,2026232885427,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.0275,15.91,2.292363166809082
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",520,2026227785530,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.46,36.98,1.945038080215454
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",421,2026229485482,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.5,63.35,2.5327847003936768
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",24,2026234585238,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,3.0075,15.85,3.265789270401001
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",265,2026232885428,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.1,16.085,2.2827539443969727
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",521,2026227785531,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.42,35.8,2.1110899448394775
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",522,2026227785532,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.07,36.2,1.8783066272735596
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",422,2026229485483,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.475,65.45,2.53033185005188
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",266,2026232885429,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,1.945,16.205,2.3281400203704834
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",25,2026234585239,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,3.18,15.84,3.422816753387451
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",523,2026227785533,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.22,36.02,2.1001012325286865
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",267,2026232885430,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.025,16.035,2.1335110664367676
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",423,2026229485484,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.475,63.9,2.5480904579162598
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0001|pi0|k1,1,1",524,2026227785534,B,noisy,7,0.0001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,3.46,35.1,1.9410738945007324
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",268,2026232885431,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.01,16.005,2.121961832046509
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",26,2026234585240,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,2.9225,15.885,3.3703911304473877
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",424,2026229485485,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.025,62.4,2.2744710445404053
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",269,2026232885432,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.005,15.965,2.1939432621002197
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",0,2026236285265,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.24,35.66,1.809929370880127
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",425,2026229485486,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,7.125,65.0,2.851302146911621
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",27,2026234585241,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.5,1,1,3.0375,16.0,3.7146053314208984
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",1,2026236285266,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.21,36.3,1.7941904067993164
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",270,2026232885433,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.5,1,1,1.97,15.915,2.2537715435028076
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",426,2026229485487,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.575,62.8,2.4319186210632324
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",2,2026236285267,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.85,36.24,1.545802354812622
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",271,2026232885434,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0625,15.88,2.265927791595459
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",28,2026234585242,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,2.985,16.03,3.5894055366516113
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",3,2026236285268,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.96,36.16,1.787862777709961
production-20261029T053925Z-29f79c,"B|noisy|d9|p0.0001|pi0|k1,1,1",427,2026229485488,B,noisy,9,0.0001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,6.5,64.7,2.360931396484375
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",272,2026232885435,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0,16.375,2.310471534729004
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",4,2026236285269,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.17,35.22,1.7135379314422607
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",29,2026234585243,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,3.0025,15.985,3.426809310913086
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",5,2026236285270,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.11,35.12,1.726130723953247
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",273,2026232885436,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0725,16.04,2.179267406463623
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",0,2026237985316,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.2,36.18,2.0960657596588135
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",6,2026236285271,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.84,35.32,1.5664074420928955
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",274,2026232885437,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,1.9725,16.175,2.191521644592285
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",1,2026237985317,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.32,35.94,1.9722959995269775
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",7,2026236285272,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,5.35,36.94,1.6190166473388672
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",30,2026234585244,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.945,16.155,3.6499698162078857
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",2,2026237985318,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.54,37.0,2.072985887527466
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",275,2026232885438,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.145,15.965,2.3348019123077393
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",8,2026236285273,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.14,35.42,2.0254099369049072
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",3,2026237985319,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.03,36.12,2.0319952964782715
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",9,2026236285274,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,4.61,36.62,1.540194034576416
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",276,2026232885439,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.0475,16.015,2.1417245864868164
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",31,2026234585245,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.03,16.365,3.704493761062622
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",10,2026236285275,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.89,35.8,1.6097462177276611
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",4,2026237985320,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.82,35.82,2.33664870262146
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",277,2026232885440,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0025,15.84,2.365762233734131
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",11,2026236285276,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.59,35.54,1.526674747467041
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",32,2026234585246,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0175,16.335,3.58172607421875
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",5,2026237985321,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.44,35.12,2.1394104957580566
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",278,2026232885441,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.065,16.33,2.1556997299194336
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",12,2026236285277,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.78,36.2,1.5459997653961182
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",6,2026237985322,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.94,36.76,2.007812976837158
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",13,2026236285278,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.5,1,1,5.1,35.1,1.9255931377410889
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",279,2026232885442,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0525,16.085,2.289494276046753
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",33,2026234585247,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.9925,16.28,3.5824687480926514
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",7,2026237985323,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,1.0,1,0,5.83,35.7,2.0115301609039307
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",14,2026236285279,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.06,35.52,1.8068218231201172
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",280,2026232885443,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9075,15.99,2.1872076988220215
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",15,2026236285280,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.04,36.72,1.8066396713256836
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",8,2026237985324,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.69,35.98,2.1851906776428223
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",34,2026234585248,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,3.005,16.065,3.306475877761841
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",281,2026232885444,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.03,15.955,2.297895908355713
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",16,2026236285281,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.84,36.16,1.640571117401123
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",9,2026237985325,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.83,36.98,1.9365766048431396
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",282,2026232885445,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9625,15.915,2.1078479290008545
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",17,2026236285282,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.8,36.72,1.7244434356689453
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",35,2026234585249,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.18,15.86,3.3239688873291016
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",10,2026237985326,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.73,36.1,1.8756906986236572
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",18,2026236285283,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.01,37.04,1.8243300914764404
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",283,2026232885446,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9775,16.295,2.136430501937866
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",11,2026237985327,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.84,35.14,1.9877796173095703
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",19,2026236285284,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.71,35.86,1.684514045715332
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",36,2026234585250,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,3.0825,16.045,3.4173953533172607
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",284,2026232885447,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0375,15.89,2.3737871646881104
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",12,2026237985328,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.71,35.96,1.9938156604766846
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",20,2026236285285,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.88,35.26,1.6121973991394043
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",285,2026232885448,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.01,16.04,2.136655330657959
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",13,2026237985329,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.57,36.78,1.8791325092315674
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",21,2026236285286,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.85,35.48,1.47121000289917
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",37,2026234585251,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.8275,16.12,3.4955928325653076
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",22,2026236285287,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.91,36.58,1.5618746280670166
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",14,2026237985330,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.91,36.82,1.8509981632232666
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",286,2026232885449,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.96,15.68,2.200709104537964
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",23,2026236285288,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.61,35.72,1.6196775436401367
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",15,2026237985331,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,1.0,1,0,6.35,35.88,2.096250057220459
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",38,2026234585252,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.1625,16.07,3.4443647861480713
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",287,2026232885450,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.065,16.025,2.2801156044006348
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",24,2026236285289,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.9,35.92,1.768970251083374
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",16,2026237985332,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.47,37.0,1.907383918762207
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",25,2026236285290,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.99,35.94,1.5562160015106201
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",288,2026232885451,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9825,15.795,2.196577548980713
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",39,2026234585253,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,2.9275,15.675,3.4138851165771484
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",17,2026237985333,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.21,36.98,2.017099142074585
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",26,2026236285291,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.89,35.32,1.7122433185577393
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",289,2026232885452,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9625,15.91,2.299800157546997
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",18,2026237985334,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,1.0,1,0,5.25,36.14,1.7970936298370361
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",27,2026236285292,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.28,36.08,1.6814162731170654
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",40,2026234585254,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.045,16.155,3.3262875080108643
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",290,2026232885453,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.025,16.155,2.2105937004089355
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",19,2026237985335,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.84,35.68,1.8127920627593994
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",28,2026236285293,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.91,35.98,1.824760913848877
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",20,2026237985336,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,1.0,1,0,5.97,36.32,1.858245849609375
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",291,2026232885454,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.08,16.205,2.2427978515625
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",29,2026236285294,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.09,35.3,1.6002719402313232
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",41,2026234585255,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.995,15.95,3.1508474349975586
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",30,2026236285295,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.86,36.28,1.605290412902832
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",21,2026237985337,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.6,35.86,1.8452560901641846
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",292,2026232885455,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0575,16.11,2.2314937114715576
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",31,2026236285296,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.7,34.28,1.547926425933838
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",22,2026237985338,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.69,35.74,1.9907219409942627
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",42,2026234585256,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.9275,16.235,3.427213430404663
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",293,2026232885456,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.1075,16.035,2.3203325271606445
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",32,2026236285297,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.19,35.66,1.6786112785339355
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",23,2026237985339,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.5,1,1,5.77,36.6,1.837653398513794
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",33,2026236285298,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.82,36.32,1.6723308563232422
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",294,2026232885457,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.925,15.91,2.2780532836914062
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",24,2026237985340,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.6,35.7,1.9619724750518799
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",43,2026234585257,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.5,1,1,3.0725,16.105,3.5146713256835938
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",34,2026236285299,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.88,35.7,1.7483134269714355
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",295,2026232885458,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0275,16.115,2.3398237228393555
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",25,2026237985341,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.75,36.1,1.858900785446167
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",35,2026236285300,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.08,35.56,1.6775426864624023
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",44,2026234585258,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,3.0675,15.86,3.3578941822052
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",26,2026237985342,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.43,36.7,1.9965507984161377
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",296,2026232885459,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0075,16.035,2.379624843597412
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",36,2026236285301,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.1,35.68,1.6853368282318115
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",37,2026236285302,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.5,35.24,1.7266242504119873
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",27,2026237985343,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.74,36.44,2.028388023376465
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",297,2026232885460,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.955,15.475,2.289829969406128
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",45,2026234585259,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.1575,16.03,3.405622959136963
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",38,2026236285303,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.84,36.56,1.8452110290527344
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",28,2026237985344,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.93,34.98,1.961538314819336
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",298,2026232885461,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.0675,15.98,2.2083754539489746
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",39,2026236285304,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.17,35.84,1.6437625885009766
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",29,2026237985345,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.09,36.32,1.82440185546875
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",46,2026234585260,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.1,16.105,3.477435827255249
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",299,2026232885462,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.09,16.045,2.2059268951416016
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",40,2026236285305,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.85,36.78,1.6254360675811768
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",30,2026237985346,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.71,35.38,1.9265918731689453
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",41,2026236285306,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.54,36.6,1.7468149662017822
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",300,2026232885463,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0,16.04,2.3739967346191406
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",31,2026237985347,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.28,36.1,2.0162696838378906
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",47,2026234585261,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.005,15.81,3.500563859939575
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",42,2026236285307,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.06,36.46,1.7652649879455566
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",301,2026232885464,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.87,15.855,2.2294466495513916
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",32,2026237985348,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.98,36.12,2.183537721633911
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",43,2026236285308,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,5.37,36.5,1.7074944972991943
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",48,2026234585262,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.1,15.805,3.560023307800293
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",302,2026232885465,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.07,16.06,2.246976137161255
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",33,2026237985349,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.94,36.9,1.8682103157043457
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",44,2026236285309,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.77,36.24,1.6312460899353027
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",45,2026236285310,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.66,36.48,1.5057003498077393
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",34,2026237985350,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,1.0,1,0,5.42,36.14,1.8418233394622803
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",303,2026232885466,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.5,1,1,2.03,16.225,2.1193418502807617
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",46,2026236285311,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.2,36.72,1.5729243755340576
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",49,2026234585263,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,2.97,16.025,3.34248423576355
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",35,2026237985351,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.5,36.16,1.8690822124481201
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",304,2026232885467,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0225,15.745,2.2177608013153076
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",47,2026236285312,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.96,36.7,1.4989876747131348
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",36,2026237985352,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.78,36.2,1.7901651859283447
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",305,2026232885468,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.995,15.975,2.1314404010772705
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",48,2026236285313,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.63,35.08,1.6834337711334229
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",50,2026234585264,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.1125,16.175,3.493967056274414
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",37,2026237985353,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.52,36.08,1.8909039497375488
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",49,2026236285314,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.69,36.04,1.7611165046691895
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",306,2026232885469,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.5,2,1,2.0525,16.155,2.209167003631592
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",38,2026237985354,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.62,36.6,1.8743879795074463
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",50,2026236285315,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,4.82,36.26,1.8202283382415771
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",51,2026234585265,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.19,15.855,3.396462917327881
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",307,2026232885470,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.1275,15.945,2.303440570831299
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",39,2026237985355,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.21,35.72,1.964879035949707
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",51,2026236285316,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.13,36.34,1.8819761276245117
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",308,2026232885471,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.5,1,1,1.9975,15.72,2.245326280593872
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",40,2026237985356,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,1.0,1,0,5.31,35.74,1.9409332275390625
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",52,2026236285317,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,4.9,36.24,1.7657153606414795
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",52,2026234585266,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.1825,16.225,3.692563772201538
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",41,2026237985357,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.84,36.0,2.0037636756896973
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",309,2026232885472,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.095,15.745,2.2036688327789307
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",53,2026236285318,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.89,36.7,1.7595188617706299
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",54,2026236285319,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.13,36.34,1.694350242614746
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",42,2026237985358,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.91,35.76,2.0117251873016357
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",53,2026234585267,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,3.035,16.05,3.5198276042938232
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",310,2026232885473,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9825,16.07,2.263319253921509
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",55,2026236285320,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.04,36.38,1.6362028121948242
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",43,2026237985359,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.59,36.24,1.9520063400268555
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",311,2026232885474,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.03,16.13,2.2005605697631836
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",56,2026236285321,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.26,35.96,1.671783208847046
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",54,2026234585268,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0275,15.81,3.3707258701324463
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",44,2026237985360,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.89,35.4,1.9550690650939941
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",312,2026232885475,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0775,16.075,2.1056406497955322
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",57,2026236285322,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.77,36.36,1.5749084949493408
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",45,2026237985361,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.66,35.96,1.9439976215362549
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",58,2026236285323,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.12,36.32,1.6013412475585938
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",55,2026234585269,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0975,15.835,3.3887617588043213
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",313,2026232885476,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.95,16.0,2.266948938369751
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",46,2026237985362,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.5,1,1,5.77,34.9,1.8852713108062744
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",59,2026236285324,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.98,37.28,1.5720279216766357
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",314,2026232885477,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0075,16.24,2.2892704010009766
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",60,2026236285325,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.5,1,1,4.95,35.14,1.5654876232147217
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",47,2026237985363,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.72,36.58,1.8771896362304688
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",56,2026234585270,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,3.0975,15.935,3.3595330715179443
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",61,2026236285326,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.89,35.48,1.6743674278259277
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",315,2026232885478,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.935,15.88,2.11015248298645
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",48,2026237985364,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,1.0,1,0,5.9,35.58,1.8566670417785645
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",62,2026236285327,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.86,35.8,1.6846668720245361
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",49,2026237985365,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.7,34.56,1.9101860523223877
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",316,2026232885479,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.095,15.965,2.0544567108154297
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",57,2026234585271,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.13,15.76,3.3635787963867188
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",63,2026236285328,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.02,36.36,1.6512904167175293
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",50,2026237985366,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.92,36.12,1.8991777896881104
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",317,2026232885480,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0925,15.74,2.10478138923645
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",64,2026236285329,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.23,35.6,1.775268793106079
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",58,2026234585272,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.9825,15.79,3.182053565979004
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",51,2026237985367,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.5,1,1,5.58,35.62,1.83925199508667
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",318,2026232885481,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0525,15.725,2.118863105773926
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",65,2026236285330,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.97,36.74,1.6849536895751953
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",52,2026237985368,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.56,36.82,1.8600492477416992
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",66,2026236285331,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,4.8,35.62,1.764627456665039
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",319,2026232885482,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9275,16.18,2.273209810256958
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",59,2026234585273,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.5,1,1,3.0175,15.86,3.4835803508758545
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",53,2026237985369,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.88,37.22,2.060485601425171
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",67,2026236285332,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.5,1,1,5.04,35.34,1.6947004795074463
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",320,2026232885483,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.09,15.73,2.2463529109954834
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",54,2026237985370,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,1.0,1,0,5.81,35.38,1.932077407836914
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",68,2026236285333,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,5.1,35.56,1.7343783378601074
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",60,2026234585274,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.5,2,1,3.165,16.045,3.6400184631347656
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",321,2026232885484,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0225,15.8,2.1250486373901367
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",69,2026236285334,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.2,36.04,1.6457126140594482
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",55,2026237985371,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.78,35.48,2.0262413024902344
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",70,2026236285335,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.33,36.86,1.7398602962493896
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",322,2026232885485,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.0975,15.94,2.2947607040405273
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",56,2026237985372,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.97,36.42,1.934027910232544
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",61,2026234585275,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.5,2,1,3.0925,15.865,3.4884989261627197
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",71,2026236285336,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,4.86,36.14,1.6089811325073242
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",323,2026232885486,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.05,15.815,2.1755614280700684
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",57,2026237985373,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.64,35.78,2.036945104598999
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",72,2026236285337,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.2,36.34,1.640164852142334
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",62,2026234585276,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.955,15.925,3.276261806488037
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",58,2026237985374,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.75,36.02,1.8445994853973389
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",324,2026232885487,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0175,15.935,2.1341850757598877
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",73,2026236285338,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.04,36.62,1.7515416145324707
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",59,2026237985375,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.53,36.22,1.9074087142944336
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",325,2026232885488,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.99,15.955,2.195751428604126
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",74,2026236285339,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.69,36.52,1.7655892372131348
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",63,2026234585277,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.13,15.645,3.6351099014282227
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",60,2026237985376,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.89,37.08,1.8901753425598145
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",75,2026236285340,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,4.69,35.06,1.8301424980163574
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",326,2026232885489,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0725,16.07,2.081509828567505
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",61,2026237985377,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.52,35.78,1.8595330715179443
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",76,2026236285341,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.1,37.22,1.6745386123657227
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",327,2026232885490,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.12,15.8,2.0193116664886475
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",64,2026234585278,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0475,16.07,3.3623743057250977
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",62,2026237985378,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.46,35.28,1.9470276832580566
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",77,2026236285342,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.87,36.06,1.6376347541809082
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",328,2026232885491,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.975,15.905,2.1222105026245117
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",78,2026236285343,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.54,35.64,1.681682825088501
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",63,2026237985379,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.1,36.06,2.106574773788452
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",65,2026234585279,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0325,15.975,3.3316545486450195
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",329,2026232885492,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.5,1,1,1.99,16.27,2.2516226768493652
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",79,2026236285344,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.86,36.26,1.706446886062622
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",64,2026237985380,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.57,35.3,2.0687713623046875
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",80,2026236285345,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,5.18,35.82,1.8265910148620605
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",330,2026232885493,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0125,15.865,2.168487548828125
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",66,2026234585280,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0325,15.67,3.125800609588623
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",65,2026237985381,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.89,35.38,1.9807276725769043
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",81,2026236285346,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.79,35.56,1.8184616565704346
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",331,2026232885494,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0175,15.8,2.1147918701171875
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",66,2026237985382,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.42,36.46,1.899092197418213
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",82,2026236285347,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.63,36.08,1.5906689167022705
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",67,2026234585281,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.99,15.955,3.1510205268859863
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",332,2026232885495,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9275,16.04,2.1655235290527344
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",67,2026237985383,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.27,36.22,2.014981985092163
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",83,2026236285348,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.93,35.32,1.6236283779144287
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",333,2026232885496,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.97,15.9,2.1479909420013428
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",84,2026236285349,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.11,35.8,1.6092283725738525
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",68,2026237985384,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.53,35.64,1.9495790004730225
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",68,2026234585282,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,3.06,16.17,3.479330062866211
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",85,2026236285350,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.05,34.4,1.812392234802246
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",334,2026232885497,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.1425,15.835,2.1352832317352295
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",69,2026237985385,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.69,35.94,1.930772304534912
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",86,2026236285351,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.78,36.28,1.7187721729278564
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",69,2026234585283,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.5,1,1,2.8475,16.085,3.3442904949188232
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",70,2026237985386,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.76,36.46,1.9530699253082275
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",335,2026232885498,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.065,15.72,2.15118145942688
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",87,2026236285352,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.88,35.94,1.520707607269287
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",71,2026237985387,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.48,36.8,1.9366531372070312
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",336,2026232885499,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.105,15.955,2.178208589553833
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",88,2026236285353,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.08,35.72,1.593353271484375
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",70,2026234585284,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.94,15.785,3.483029365539551
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",72,2026237985388,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.35,36.0,1.8084149360656738
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",89,2026236285354,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,5.12,37.26,1.600822925567627
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",337,2026232885500,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9375,16.045,2.085078001022339
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",73,2026237985389,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.79,35.92,2.1168646812438965
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",90,2026236285355,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.95,36.2,1.7868661880493164
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",338,2026232885501,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.1375,15.88,2.228379726409912
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",71,2026234585285,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.025,15.965,3.6741652488708496
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",74,2026237985390,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,1.0,1,0,5.83,36.68,1.9260268211364746
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",91,2026236285356,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,5.1,36.06,1.8195044994354248
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",339,2026232885502,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.1825,16.12,2.103196382522583
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",92,2026236285357,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,4.93,36.44,1.6068511009216309
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",75,2026237985391,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.68,35.84,2.0306477546691895
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",72,2026234585286,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.135,16.235,3.433166742324829
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",340,2026232885503,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.1075,16.24,2.0930216312408447
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",93,2026236285358,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.03,36.42,1.6128242015838623
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",76,2026237985392,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,1.0,1,0,5.73,34.84,2.0543036460876465
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",94,2026236285359,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.83,36.32,1.7184205055236816
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",341,2026232885504,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,2.0,2,0,2.035,15.73,2.2081058025360107
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",73,2026234585287,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.9225,16.145,3.3894946575164795
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",77,2026237985393,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.97,35.56,2.1346194744110107
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",95,2026236285360,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.75,36.16,1.7199256420135498
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",342,2026232885505,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.1325,15.885,2.232682704925537
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",96,2026236285361,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.11,35.82,1.5464451313018799
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",78,2026237985394,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.06,36.06,2.0144288539886475
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",74,2026234585288,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.1,16.065,3.5556774139404297
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",343,2026232885506,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.03,15.755,2.2986562252044678
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",97,2026236285362,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.5,1,1,5.27,36.22,1.8143389225006104
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",79,2026237985395,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.9,35.76,2.0313117504119873
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",98,2026236285363,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.74,35.74,1.5669898986816406
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",344,2026232885507,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.1675,16.315,2.2968342304229736
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",80,2026237985396,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.62,35.92,2.064324378967285
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",75,2026234585289,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.9175,15.77,3.615553379058838
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",99,2026236285364,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.71,35.26,1.7353167533874512
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",345,2026232885508,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.5,2,1,2.005,16.15,2.2045645713806152
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",81,2026237985397,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.61,36.64,1.9388628005981445
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",100,2026236285365,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,4.79,35.68,1.678178310394287
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",82,2026237985398,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.73,37.2,1.904717206954956
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",346,2026232885509,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0475,15.885,2.1147067546844482
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",76,2026234585290,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,2,2,3.12,15.98,3.465996503829956
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",101,2026236285366,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.03,36.3,1.5527098178863525
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",83,2026237985399,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.19,36.02,1.9385132789611816
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",102,2026236285367,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.99,37.1,1.5784804821014404
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",347,2026232885510,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0225,15.855,2.2553791999816895
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",77,2026234585291,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.065,16.15,3.22395396232605
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",103,2026236285368,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.23,35.38,1.751439094543457
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",84,2026237985400,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.53,36.62,1.9873170852661133
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",348,2026232885511,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.5,1,1,1.965,15.93,2.446657419204712
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",104,2026236285369,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.83,35.76,1.6246001720428467
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",85,2026237985401,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.12,36.54,2.1381406784057617
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",78,2026234585292,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,2.0,2,0,3.0925,16.125,3.5742855072021484
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",105,2026236285370,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.92,35.46,1.9423372745513916
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",349,2026232885512,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9625,15.69,2.4892642498016357
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",86,2026237985402,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.27,36.4,1.8035848140716553
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",106,2026236285371,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.19,35.8,1.5145525932312012
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",350,2026232885513,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,2.0,2,0,2.035,16.02,2.186129331588745
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",87,2026237985403,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.89,36.48,1.9043686389923096
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",107,2026236285372,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.23,34.92,1.5651588439941406
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",79,2026234585293,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.5,1,1,3.17,16.085,3.455517530441284
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",88,2026237985404,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.68,36.9,1.99900484085083
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",351,2026232885514,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.5,2,1,2.0,15.9,2.246561050415039
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",108,2026236285373,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.9,35.4,1.6711845397949219
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",89,2026237985405,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.55,35.62,1.9046053886413574
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",109,2026236285374,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.45,36.9,1.6513566970825195
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",352,2026232885515,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.09,16.35,2.244323492050171
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",80,2026234585294,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.945,15.73,3.6624345779418945
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",110,2026236285375,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.82,35.56,1.544888973236084
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",90,2026237985406,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.11,35.08,2.0689244270324707
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",353,2026232885516,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.925,16.46,2.0826826095581055
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",111,2026236285376,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.33,35.92,1.6545755863189697
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",81,2026234585295,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.5,2,1,3.025,15.925,3.4000868797302246
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",91,2026237985407,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,1.0,1,0,5.63,36.58,2.035221815109253
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",354,2026232885517,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.5,1,1,1.995,16.065,2.132063388824463
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",112,2026236285377,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.84,36.2,1.784595012664795
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",92,2026237985408,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.17,35.88,1.941688060760498
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",113,2026236285378,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.03,35.38,1.5917229652404785
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",355,2026232885518,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0325,16.225,2.1589138507843018
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",82,2026234585296,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,2.0,2,0,3.0725,16.065,3.5369322299957275
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",93,2026237985409,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.97,35.7,2.0638201236724854
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",114,2026236285379,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.62,35.92,1.6462900638580322
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",356,2026232885519,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9325,16.135,2.2222352027893066
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",115,2026236285380,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.99,35.48,1.5270028114318848
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",94,2026237985410,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.57,37.06,2.0136830806732178
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",357,2026232885520,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0825,15.945,2.1960065364837646
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",83,2026234585297,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.8475,16.07,3.669660806655884
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",116,2026236285381,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.02,34.98,1.4307856559753418
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",95,2026237985411,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.65,36.32,2.0026237964630127
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",117,2026236285382,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.86,34.94,1.5066730976104736
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",358,2026232885521,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.025,16.275,2.139434337615967
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",96,2026237985412,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.88,36.1,2.018630027770996
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",118,2026236285383,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.0,36.58,1.6896634101867676
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",84,2026234585298,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0175,16.09,3.5901079177856445
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",359,2026232885522,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.5,1,1,2.005,16.17,2.128667116165161
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",119,2026236285384,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.74,35.94,1.631836175918579
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",97,2026237985413,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.53,36.78,1.9030165672302246
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",360,2026232885523,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0425,15.87,2.0591444969177246
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",120,2026236285385,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.72,36.84,1.5828919410705566
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",98,2026237985414,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.33,35.46,1.8287103176116943
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",85,2026234585299,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,3.0275,15.92,3.498013734817505
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",121,2026236285386,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.88,35.16,1.865750789642334
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",361,2026232885524,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.5,2,1,2.1075,16.04,2.351032018661499
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",99,2026237985415,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,1.0,1,0,5.44,35.92,2.1478254795074463
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",122,2026236285387,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.31,35.22,1.741154432296753
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",86,2026234585300,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,2.9225,15.985,3.655388593673706
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",100,2026237985416,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.62,36.16,2.1627590656280518
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",362,2026232885525,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.155,15.935,2.3944411277770996
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",123,2026236285388,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.37,36.1,1.6101830005645752
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",363,2026232885526,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.08,16.1,2.1926488876342773
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",101,2026237985417,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.8,35.28,2.271662473678589
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",124,2026236285389,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.09,36.86,1.8046355247497559
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",87,2026234585301,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.19,15.93,3.5340328216552734
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",125,2026236285390,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.84,35.78,1.6962387561798096
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",102,2026237985418,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.87,35.8,2.011871576309204
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",364,2026232885527,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.99,15.77,2.236787796020508
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",126,2026236285391,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.68,36.84,1.5319833755493164
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",103,2026237985419,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.97,36.3,2.1422996520996094
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",365,2026232885528,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.04,16.01,2.2695412635803223
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",88,2026234585302,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.08,16.07,3.608173131942749
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",127,2026236285392,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.58,36.86,1.6171162128448486
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",104,2026237985420,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.15,35.86,2.0595169067382812
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",366,2026232885529,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.12,16.265,2.2149715423583984
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",128,2026236285393,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.22,36.34,1.9269640445709229
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",89,2026234585303,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.965,16.19,3.499154567718506
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",105,2026237985421,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.53,36.06,1.9959807395935059
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",129,2026236285394,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.5,1,1,4.97,35.54,1.6522676944732666
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",367,2026232885530,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0225,15.935,2.311940908432007
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",106,2026237985422,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.66,36.42,1.9563465118408203
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",130,2026236285395,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.89,36.86,1.6357417106628418
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",368,2026232885531,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.1925,16.035,2.4095942974090576
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",90,2026234585304,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.155,16.195,3.654799461364746
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",131,2026236285396,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.82,35.7,1.5741424560546875
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",107,2026237985423,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.52,35.8,2.0178616046905518
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",132,2026236285397,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.74,35.18,1.7066478729248047
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",369,2026232885532,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.01,16.08,2.2855048179626465
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",108,2026237985424,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.81,36.36,2.2376577854156494
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",133,2026236285398,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.8,36.36,1.4930176734924316
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",91,2026234585305,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0,16.04,3.4231760501861572
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",370,2026232885533,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.5,1,1,2.105,16.11,2.2241694927215576
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",109,2026237985425,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.8,35.6,1.93845534324646
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",134,2026236285399,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.89,36.8,1.5908787250518799
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",135,2026236285400,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.74,36.48,1.5575931072235107
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",371,2026232885534,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.1175,15.78,2.276566743850708
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",110,2026237985426,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.72,36.62,1.998533010482788
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",92,2026234585306,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.5,1,1,2.9075,15.945,3.322129249572754
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",136,2026236285401,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.73,36.92,1.4735069274902344
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",111,2026237985427,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,1.0,1,0,5.66,37.02,1.9015579223632812
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",372,2026232885535,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.5,1,1,1.9375,15.615,2.3398330211639404
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",137,2026236285402,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,4.89,36.82,1.714709997177124
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",93,2026234585307,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.5,1,1,3.055,16.015,3.5045173168182373
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",112,2026237985428,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.81,36.12,2.0966877937316895
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",373,2026232885536,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.0925,16.115,2.3563590049743652
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",138,2026236285403,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.18,35.74,1.600426197052002
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",113,2026237985429,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.86,35.86,2.1293458938598633
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",139,2026236285404,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.81,36.86,1.8026487827301025
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",374,2026232885537,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0625,16.08,2.4229986667633057
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",94,2026234585308,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0975,15.995,3.3938419818878174
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",140,2026236285405,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.55,35.52,1.7673592567443848
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",114,2026237985430,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.81,34.88,2.153088092803955
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",375,2026232885538,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,1.9575,16.48,2.214737892150879
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",141,2026236285406,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.0,36.44,1.6971466541290283
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",95,2026234585309,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.5,1,1,3.095,16.055,3.3698997497558594
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",115,2026237985431,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.25,36.04,2.1815555095672607
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",376,2026232885539,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.1125,15.81,2.1596758365631104
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",142,2026236285407,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.85,35.22,1.697786808013916
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",116,2026237985432,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.05,35.4,1.955841302871704
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",143,2026236285408,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,4.55,35.24,1.6427054405212402
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",377,2026232885540,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.0375,15.795,2.148756265640259
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",96,2026234585310,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0275,16.125,3.3355050086975098
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",117,2026237985433,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.91,35.9,2.290576219558716
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",144,2026236285409,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.26,35.92,1.767557144165039
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",378,2026232885541,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.07,15.95,2.29226016998291
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",145,2026236285410,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.83,37.14,1.8684520721435547
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",118,2026237985434,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,1.0,1,0,5.55,36.14,2.295928955078125
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",97,2026234585311,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.5,1,1,3.0925,15.63,3.734287977218628
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",379,2026232885542,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9825,16.07,2.2660279273986816
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",146,2026236285411,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.77,35.18,1.696406364440918
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",119,2026237985435,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.89,36.38,2.0347795486450195
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",380,2026232885543,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.01,16.135,2.1637156009674072
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",147,2026236285412,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.98,35.98,1.6882665157318115
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",98,2026234585312,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.03,16.335,3.603884220123291
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",120,2026237985436,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.12,35.8,2.116954803466797
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",148,2026236285413,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.01,35.64,1.7972657680511475
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",381,2026232885544,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.085,16.205,2.435789108276367
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",121,2026237985437,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.65,36.36,2.0151102542877197
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",149,2026236285414,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.87,35.38,1.666788101196289
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",99,2026234585313,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.9475,15.93,3.7151424884796143
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",382,2026232885545,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.08,16.025,2.432805061340332
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",122,2026237985438,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.62,35.94,2.082568407058716
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",150,2026236285415,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,4.92,35.96,1.749849557876587
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",151,2026236285416,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,5.07,34.86,1.7078149318695068
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",123,2026237985439,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,1.0,1,0,6.08,36.26,2.0995349884033203
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",383,2026232885546,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9325,15.935,2.4583070278167725
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",100,2026234585314,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.9225,15.715,3.7429816722869873
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",152,2026236285417,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.86,35.84,1.8589482307434082
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",124,2026237985440,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.74,36.78,2.0911006927490234
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",384,2026232885547,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.02,15.855,2.3415749073028564
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",153,2026236285418,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,4.87,36.4,1.7485272884368896
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",125,2026237985441,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.78,36.68,2.142669677734375
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",385,2026232885548,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.105,16.16,2.2214488983154297
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",101,2026234585315,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.05,15.745,3.6683032512664795
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",154,2026236285419,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.84,35.92,1.7285401821136475
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",126,2026237985442,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.93,35.18,2.202693462371826
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",155,2026236285420,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.27,35.42,1.6690294742584229
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",386,2026232885549,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0425,15.87,2.260568141937256
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",102,2026234585316,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0925,16.225,3.3646674156188965
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",156,2026236285421,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.02,34.66,1.8838605880737305
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",127,2026237985443,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.87,36.36,2.272127389907837
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",387,2026232885550,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.04,15.965,2.355935573577881
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",157,2026236285422,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.04,35.88,1.7835290431976318
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",128,2026237985444,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.77,36.7,2.0282819271087646
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",388,2026232885551,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,1.9825,16.525,2.216216802597046
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",103,2026234585317,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.5,2,1,2.965,15.925,3.589817762374878
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",158,2026236285423,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.7,33.74,1.8337371349334717
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",129,2026237985445,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.5,1,1,5.63,34.9,2.0646228790283203
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",389,2026232885552,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.995,15.99,2.119901657104492
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",159,2026236285424,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.92,34.82,1.5871655941009521
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",130,2026237985446,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,1.0,1,0,5.81,35.64,2.0327162742614746
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",104,2026234585318,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.2225,16.26,3.395387649536133
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",160,2026236285425,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.75,35.84,1.6039838790893555
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",390,2026232885553,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9875,16.255,2.103827953338623
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",131,2026237985447,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.69,36.92,2.003182888031006
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",161,2026236285426,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.89,35.92,1.8539767265319824
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",391,2026232885554,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.5,1,1,2.105,15.84,2.258302927017212
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",132,2026237985448,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,1.0,1,0,5.83,34.9,2.030466079711914
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",105,2026234585319,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.865,16.05,3.670863389968872
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",162,2026236285427,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.97,36.14,1.7984144687652588
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",392,2026232885555,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.99,15.83,2.2514286041259766
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",163,2026236285428,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.1,36.2,1.6006767749786377
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",133,2026237985449,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.02,36.4,2.0924344062805176
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",164,2026236285429,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.18,35.1,1.7143893241882324
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",106,2026234585320,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0275,15.695,3.6210594177246094
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",393,2026232885556,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.1225,15.725,2.2155678272247314
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",134,2026237985450,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.67,34.96,1.995558500289917
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",165,2026236285430,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.83,36.08,1.5492615699768066
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",394,2026232885557,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.905,15.975,2.065685749053955
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",135,2026237985451,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.82,36.04,1.9317970275878906
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",166,2026236285431,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.0,34.78,1.7701597213745117
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",107,2026234585321,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.135,16.115,3.4713401794433594
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",395,2026232885558,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.13,15.77,2.0586462020874023
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",136,2026237985452,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.83,35.54,1.9200611114501953
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",167,2026236285432,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.55,35.64,1.7297632694244385
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",396,2026232885559,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0225,15.93,2.2532248497009277
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",137,2026237985453,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.93,34.76,2.145977735519409
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",168,2026236285433,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.99,35.48,1.7632808685302734
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",108,2026234585322,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,3.01,15.645,3.534069776535034
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",169,2026236285434,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.91,35.54,1.7872037887573242
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",138,2026237985454,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.04,35.08,2.1936028003692627
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",397,2026232885560,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,1.9025,16.135,2.2893576622009277
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",170,2026236285435,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.77,36.0,1.794623851776123
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",109,2026234585323,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0975,16.235,3.5878684520721436
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",139,2026237985455,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.65,36.14,2.1542325019836426
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",398,2026232885561,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0325,15.795,2.255596399307251
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",171,2026236285436,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.49,36.16,1.931502103805542
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",140,2026237985456,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.59,36.52,2.156075954437256
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",399,2026232885562,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.05,16.055,2.263434410095215
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",172,2026236285437,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.99,37.38,1.737440586090088
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",110,2026234585324,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,2.0,2,0,3.1225,16.16,3.5513830184936523
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",141,2026237985457,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.62,36.1,2.0553395748138428
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",400,2026232885563,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.005,15.935,2.20306396484375
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",173,2026236285438,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.69,36.28,1.6621689796447754
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",142,2026237985458,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.71,35.88,2.1180191040039062
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",174,2026236285439,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.55,36.18,1.6455316543579102
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",401,2026232885564,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.8925,16.24,2.143000364303589
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",111,2026234585325,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.99,16.02,3.5131709575653076
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",175,2026236285440,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.86,35.86,1.7079474925994873
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",143,2026237985459,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.89,35.78,2.1027274131774902
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",402,2026232885565,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.1175,15.7,2.254077911376953
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",176,2026236285441,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.5,1,1,4.72,35.26,1.6305317878723145
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",144,2026237985460,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.7,35.98,2.1104602813720703
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",112,2026234585326,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0425,15.965,3.8359992504119873
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",403,2026232885566,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.11,15.99,2.217674732208252
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",177,2026236285442,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.83,35.56,1.8183109760284424
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",145,2026237985461,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.89,35.42,1.9093711376190186
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",404,2026232885567,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9375,15.825,2.080888271331787
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",178,2026236285443,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.17,34.3,1.694493055343628
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",113,2026234585327,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.98,16.195,3.6201040744781494
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",146,2026237985462,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.86,36.3,1.9922194480895996
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",179,2026236285444,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,5.26,36.24,1.8986504077911377
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",405,2026232885568,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.5,1,1,1.9375,15.555,2.2101545333862305
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",147,2026237985463,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.64,36.38,2.0083065032958984
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",180,2026236285445,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.54,36.06,1.5297472476959229
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",406,2026232885569,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.06,15.93,2.2626678943634033
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",114,2026234585328,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,2.8975,15.96,3.603738307952881
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",181,2026236285446,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.87,36.66,1.546705722808838
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",148,2026237985464,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.54,36.52,2.0328619480133057
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",407,2026232885570,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9425,15.96,2.24507474899292
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",182,2026236285447,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,3.0,3,0,5.1,36.18,1.781923770904541
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",149,2026237985465,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.5,1,1,5.86,35.88,2.1233205795288086
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",115,2026234585329,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.135,15.875,3.56962251663208
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",183,2026236285448,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.63,35.62,1.645414113998413
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",408,2026232885571,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.1475,15.94,2.2563083171844482
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",150,2026237985466,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.24,36.28,2.1453146934509277
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",184,2026236285449,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.84,35.04,1.798701524734497
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",409,2026232885572,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.98,15.995,2.334745407104492
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",151,2026237985467,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.5,1,1,5.5,36.28,1.9989280700683594
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",116,2026234585330,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.1125,16.05,3.5406315326690674
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",185,2026236285450,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.86,35.6,1.819199562072754
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",410,2026232885573,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.01,15.75,2.177401065826416
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",152,2026237985468,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.93,36.46,2.0409696102142334
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",186,2026236285451,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,4.54,36.58,1.5867664813995361
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",117,2026234585331,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,2.0,2,0,2.9925,15.715,3.573550224304199
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",153,2026237985469,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.51,36.74,1.9751594066619873
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",187,2026236285452,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,4.83,35.88,1.7672266960144043
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",411,2026232885574,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.065,15.835,2.090341329574585
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",188,2026236285453,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.49,35.92,1.731701374053955
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",154,2026237985470,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.92,36.52,2.1466360092163086
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",412,2026232885575,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.0325,16.065,2.186957359313965
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",118,2026234585332,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.16,15.935,3.6484365463256836
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",189,2026236285454,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.53,36.08,1.7878835201263428
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",155,2026237985471,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.91,36.52,2.0640249252319336
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",413,2026232885576,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.0375,15.79,2.1990842819213867
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",190,2026236285455,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.94,35.42,1.714184045791626
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",156,2026237985472,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.42,36.54,2.2832603454589844
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",414,2026232885577,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9525,15.81,2.1678061485290527
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",191,2026236285456,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.57,36.7,1.597109079360962
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",119,2026234585333,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.9275,15.535,3.6919195652008057
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",192,2026236285457,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.93,34.9,1.6653451919555664
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",157,2026237985473,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.66,35.64,2.075014114379883
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",415,2026232885578,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,1.9225,16.165,2.1081058979034424
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",193,2026236285458,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.06,35.42,1.5535500049591064
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",158,2026237985474,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.61,35.94,1.9728007316589355
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",120,2026234585334,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,2.975,15.965,3.5669643878936768
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",416,2026232885579,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9875,16.345,2.2912464141845703
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",194,2026236285459,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.73,35.52,1.5761840343475342
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",159,2026237985475,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.72,34.78,1.7791447639465332
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",417,2026232885580,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.5,1,1,1.9175,15.815,2.232147216796875
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",195,2026236285460,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.75,35.86,1.6812763214111328
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",121,2026234585335,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.105,15.805,3.480191946029663
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",160,2026237985476,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.61,36.26,1.9611220359802246
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",196,2026236285461,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.02,36.14,1.6754529476165771
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",418,2026232885581,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.5,1,1,1.9625,16.22,2.2391326427459717
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",161,2026237985477,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.71,35.24,2.008237361907959
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",197,2026236285462,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.06,35.72,1.761399269104004
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",122,2026234585336,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.1325,16.1,3.5199573040008545
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",419,2026232885582,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.975,15.755,2.2904937267303467
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",162,2026237985478,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.54,35.82,2.058297634124756
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",198,2026236285463,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.41,36.58,1.7467877864837646
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",420,2026232885583,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.1325,15.705,2.102536678314209
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",199,2026236285464,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.59,36.0,1.728682041168213
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",163,2026237985479,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.3,36.46,1.944251537322998
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",123,2026234585337,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0875,15.875,3.4094507694244385
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",200,2026236285465,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.04,35.78,1.6715903282165527
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",421,2026232885584,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.1225,16.055,2.1723854541778564
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",164,2026237985480,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.18,35.56,1.9843440055847168
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",201,2026236285466,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.73,35.38,1.8404064178466797
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",422,2026232885585,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.08,15.79,2.175305128097534
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",165,2026237985481,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.76,34.8,1.990910291671753
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",124,2026234585338,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,3.0725,16.19,3.2177207469940186
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",202,2026236285467,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.65,35.94,1.6665196418762207
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",166,2026237985482,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.87,36.66,2.071866035461426
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",423,2026232885586,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0925,15.75,2.1353936195373535
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",203,2026236285468,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.05,37.1,1.7431941032409668
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",125,2026234585339,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.99,16.24,3.579711675643921
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",167,2026237985483,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.82,36.5,1.9989097118377686
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",424,2026232885587,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.955,16.24,2.165445327758789
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",204,2026236285469,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.36,36.02,1.8275301456451416
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",168,2026237985484,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.7,35.92,1.9532968997955322
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",425,2026232885588,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,1.955,16.525,2.129216432571411
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",205,2026236285470,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.63,35.58,1.7202565670013428
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",126,2026234585340,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.5,2,1,3.2825,16.045,3.850438117980957
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",169,2026237985485,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.68,37.0,1.9850873947143555
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",206,2026236285471,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.39,35.4,1.8995978832244873
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",426,2026232885589,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.19,15.935,2.1510562896728516
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",170,2026237985486,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.71,36.66,1.85160231590271
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",207,2026236285472,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.75,36.24,1.7197823524475098
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",427,2026232885590,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9675,16.14,2.197394371032715
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",127,2026234585341,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0625,15.76,3.6290111541748047
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",171,2026237985487,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.77,35.9,1.9009079933166504
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",208,2026236285473,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.34,36.1,1.809471845626831
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",428,2026232885591,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.855,15.865,2.1590771675109863
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",209,2026236285474,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.78,35.46,1.6207668781280518
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",172,2026237985488,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.33,36.66,1.9157063961029053
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",128,2026234585342,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.2225,15.915,3.6745452880859375
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",210,2026236285475,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.99,34.66,1.733076810836792
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",429,2026232885592,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9775,15.68,2.305544137954712
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",173,2026237985489,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.09,35.78,2.0189762115478516
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",211,2026236285476,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.17,35.84,1.803288459777832
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",430,2026232885593,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0025,15.84,2.208054304122925
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",174,2026237985490,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.86,35.52,1.8668701648712158
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",129,2026234585343,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0425,16.035,3.5921764373779297
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",212,2026236285477,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.9,36.68,1.6214141845703125
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",175,2026237985491,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.55,36.42,1.9266397953033447
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",431,2026232885594,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.5,1,1,1.95,15.9,2.4177708625793457
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",213,2026236285478,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.29,35.88,1.5451552867889404
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",176,2026237985492,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.4,36.8,1.8348093032836914
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",130,2026234585344,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0725,16.015,3.3584232330322266
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",214,2026236285479,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.73,36.58,1.6424474716186523
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",432,2026232885595,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.5,1,1,2.035,15.995,2.334432363510132
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",177,2026237985493,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.54,36.14,1.8259496688842773
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",215,2026236285480,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.18,37.12,1.876784086227417
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",433,2026232885596,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.95,15.875,2.2707560062408447
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",131,2026234585345,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.5,1,1,3.11,15.855,3.381810188293457
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",178,2026237985494,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.66,35.68,2.05895733833313
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",216,2026236285481,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.93,35.32,1.6580853462219238
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",434,2026232885597,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.97,16.02,2.1334407329559326
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",179,2026237985495,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.97,34.6,2.0160794258117676
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",217,2026236285482,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,5.31,35.92,1.9047796726226807
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",132,2026234585346,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,2.9925,16.065,3.5752224922180176
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",218,2026236285483,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.75,35.9,1.5204246044158936
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",435,2026232885598,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0625,15.565,2.280243396759033
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",180,2026237985496,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.0,35.76,2.0421574115753174
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",219,2026236285484,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.55,36.66,1.6169497966766357
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",436,2026232885599,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.135,15.87,2.2435061931610107
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",181,2026237985497,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.07,35.74,2.0563461780548096
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",133,2026234585347,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,3.2175,16.165,3.4323322772979736
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",220,2026236285485,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.85,35.6,1.834810495376587
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",437,2026232885600,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.8775,16.415,2.2138822078704834
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",182,2026237985498,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.66,35.44,2.1708781719207764
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",221,2026236285486,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.22,36.92,1.6238601207733154
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",222,2026236285487,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.59,35.58,1.5023486614227295
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",438,2026232885601,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.015,15.95,2.209479570388794
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",183,2026237985499,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.86,35.92,2.0970466136932373
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",134,2026234585348,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.1775,15.925,3.485633611679077
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",223,2026236285488,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.65,37.26,1.584423303604126
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",184,2026237985500,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.04,36.12,2.0498414039611816
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",439,2026232885602,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.12,16.205,2.23439884185791
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",224,2026236285489,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.27,35.98,1.5008468627929688
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",135,2026234585349,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,3.01,16.02,3.5014562606811523
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",185,2026237985501,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.93,36.14,2.1161670684814453
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",440,2026232885603,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.115,15.84,2.338015079498291
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",225,2026236285490,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.14,36.14,1.7213730812072754
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",186,2026237985502,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.98,35.98,2.072361707687378
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",226,2026236285491,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.62,35.54,1.7214112281799316
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",441,2026232885604,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.5,1,1,2.165,15.96,2.400113344192505
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",136,2026234585350,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.01,15.895,3.6170709133148193
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",227,2026236285492,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.81,35.64,1.7770614624023438
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",187,2026237985503,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.79,36.58,2.151738166809082
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",442,2026232885605,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.5,2,1,1.9775,15.98,2.0935046672821045
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",228,2026236285493,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.75,36.16,1.6223816871643066
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",188,2026237985504,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.61,37.06,2.0322399139404297
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",137,2026234585351,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.1375,15.91,3.3316261768341064
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",443,2026232885606,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0175,16.13,2.1462044715881348
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",229,2026236285494,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.57,35.94,1.6328861713409424
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",189,2026237985505,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.26,35.4,2.010080575942993
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",230,2026236285495,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.91,35.16,1.7539865970611572
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",444,2026232885607,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9775,15.8,2.187181234359741
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",138,2026234585352,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.945,15.855,3.209104299545288
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",190,2026237985506,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.61,35.88,1.9094657897949219
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",231,2026236285496,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.82,36.22,1.7463343143463135
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",445,2026232885608,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0825,15.625,2.1544787883758545
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",191,2026237985507,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.21,35.96,1.9820404052734375
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",232,2026236285497,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.64,34.96,1.7606053352355957
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",139,2026234585353,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0175,16.18,3.5303585529327393
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",446,2026232885609,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,1.8925,16.025,2.1497931480407715
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",233,2026236285498,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.88,35.72,1.7830188274383545
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",192,2026237985508,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.79,35.52,2.0930120944976807
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",447,2026232885610,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0925,15.54,2.1159045696258545
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",234,2026236285499,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.08,36.72,1.8905751705169678
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",193,2026237985509,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.73,36.68,2.148362398147583
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",140,2026234585354,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0975,15.65,3.557182788848877
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",448,2026232885611,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0025,16.455,2.0854320526123047
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",235,2026236285500,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,4.87,35.56,1.8003077507019043
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",194,2026237985510,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.67,37.58,2.0680789947509766
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",236,2026236285501,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.06,36.22,1.657862901687622
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",449,2026232885612,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.1725,16.42,2.183405876159668
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",141,2026234585355,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.1,15.87,3.2638320922851562
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",195,2026237985511,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,1.0,1,0,5.83,36.52,1.9906470775604248
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",237,2026236285502,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.95,35.76,1.5882742404937744
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",450,2026232885613,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0,15.765,2.1672379970550537
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",196,2026237985512,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.66,35.42,2.1646344661712646
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",238,2026236285503,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.83,35.72,1.5737431049346924
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",142,2026234585356,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.5,2,1,3.02,16.265,3.530799627304077
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",451,2026232885614,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9425,15.99,2.1008224487304688
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",239,2026236285504,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.0,36.56,1.7105283737182617
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",197,2026237985513,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.58,36.42,1.975318431854248
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",240,2026236285505,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.02,36.6,1.6191420555114746
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",452,2026232885615,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.1375,16.055,2.158630847930908
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",198,2026237985514,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.69,35.5,1.9099106788635254
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",143,2026234585357,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.885,15.76,3.48598313331604
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",241,2026236285506,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.57,35.96,1.599379539489746
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",453,2026232885616,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.5,1,1,1.9625,16.04,2.0362253189086914
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",199,2026237985515,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.88,35.04,1.8667452335357666
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",242,2026236285507,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.79,36.26,1.6286969184875488
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",144,2026234585358,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.1025,16.205,3.4256432056427
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",200,2026237985516,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.06,35.42,1.964641809463501
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",454,2026232885617,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.125,16.465,2.07675838470459
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",243,2026236285508,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.93,36.44,1.5353567600250244
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",201,2026237985517,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.99,36.06,2.0187203884124756
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",244,2026236285509,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.36,35.54,1.6077728271484375
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",455,2026232885618,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,1.9575,16.025,2.084362268447876
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",145,2026234585359,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.8425,16.215,3.4044506549835205
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",245,2026236285510,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.75,37.4,1.5249037742614746
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",202,2026237985518,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.56,35.86,1.9718303680419922
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",456,2026232885619,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.085,16.225,2.1396303176879883
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",246,2026236285511,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.05,36.06,1.5451312065124512
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",203,2026237985519,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.74,35.2,1.8712377548217773
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",457,2026232885620,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.5,2,1,2.0825,15.965,2.0945675373077393
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",247,2026236285512,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.08,36.16,1.506366491317749
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",146,2026234585360,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,3.095,15.89,3.650317907333374
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",204,2026237985520,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.78,35.46,1.9159114360809326
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",248,2026236285513,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.87,35.58,1.5643022060394287
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",458,2026232885621,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,1.89,15.765,2.0735602378845215
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",249,2026236285514,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.2,35.7,1.5123803615570068
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",205,2026237985521,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.16,35.8,2.0277862548828125
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",147,2026234585361,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.9525,15.655,3.343512773513794
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",459,2026232885622,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.99,16.255,2.111678123474121
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",250,2026236285515,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.31,36.52,1.6930158138275146
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",206,2026237985522,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.57,36.82,1.865675926208496
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",460,2026232885623,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.0675,15.765,2.14249849319458
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",251,2026236285516,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.04,35.98,1.6794781684875488
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",207,2026237985523,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.62,35.82,1.918722152709961
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",148,2026234585362,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.1375,16.115,3.436615467071533
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",252,2026236285517,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.5,1,1,5.15,35.76,1.531459093093872
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",461,2026232885624,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0875,16.205,2.1356234550476074
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",208,2026237985524,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.25,36.7,2.043701171875
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",253,2026236285518,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.92,35.56,1.554603099822998
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",462,2026232885625,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.1625,15.94,2.1892616748809814
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",149,2026234585363,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.075,15.975,3.4015557765960693
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",209,2026237985525,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.8,34.74,2.022170305252075
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",254,2026236285519,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.75,37.14,1.5654118061065674
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",463,2026232885626,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.5,1,1,2.02,15.97,2.09175443649292
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",255,2026236285520,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.51,35.92,1.5770328044891357
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",210,2026237985526,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.01,37.04,2.055999994277954
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",150,2026234585364,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.035,15.915,3.366839647293091
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",256,2026236285521,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.5,1,1,4.9,36.82,1.5181646347045898
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",464,2026232885627,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,2.0,2,0,1.9425,16.29,2.0991950035095215
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",211,2026237985527,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.89,35.5,2.222043752670288
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",257,2026236285522,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.64,36.04,1.5815303325653076
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",465,2026232885628,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0075,15.925,2.1501026153564453
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",151,2026234585365,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,2.0,2,0,3.0175,15.655,3.399042844772339
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",212,2026237985528,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.63,35.94,2.187626838684082
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",258,2026236285523,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.96,36.0,1.8242309093475342
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",466,2026232885629,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.5,1,1,2.01,15.665,2.255255937576294
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",259,2026236285524,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.98,35.46,1.9067988395690918
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",213,2026237985529,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.14,35.98,2.190660238265991
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",152,2026234585366,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.9675,15.94,3.395571708679199
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",260,2026236285525,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.96,35.86,1.5728282928466797
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",467,2026232885630,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.5,1,1,2.08,16.44,2.2636196613311768
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",214,2026237985530,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.74,36.62,2.025176763534546
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",261,2026236285526,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.07,36.72,1.6870601177215576
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",468,2026232885631,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.065,16.185,2.1587939262390137
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",215,2026237985531,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.1,35.7,1.974670648574829
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",153,2026234585367,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0325,16.245,3.4742789268493652
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",262,2026236285527,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.95,36.14,1.6932451725006104
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",469,2026232885632,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0375,16.115,2.272571086883545
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",216,2026237985532,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.42,36.7,2.1459357738494873
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",263,2026236285528,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.09,35.74,1.6948189735412598
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",154,2026234585368,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.995,15.895,3.5620169639587402
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",264,2026236285529,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.78,35.68,1.616506814956665
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",470,2026232885633,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,1.97,16.165,2.1740188598632812
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",217,2026237985533,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.88,36.1,2.0783462524414062
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",265,2026236285530,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,4.75,36.0,1.7126617431640625
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",218,2026237985534,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.85,36.18,2.0658321380615234
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",471,2026232885634,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.5,1,1,2.055,15.885,2.2799112796783447
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",266,2026236285531,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.94,36.14,1.6136181354522705
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",155,2026234585369,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.5,1,1,2.9775,16.055,3.7564141750335693
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",219,2026237985535,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.61,35.78,2.0184082984924316
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",472,2026232885635,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.5,1,1,1.93,15.91,2.166764259338379
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",267,2026236285532,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.19,36.0,1.5395987033843994
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",220,2026237985536,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.99,35.86,1.9273490905761719
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",268,2026236285533,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.92,35.72,1.5858852863311768
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",473,2026232885636,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.2875,15.83,2.265822172164917
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",156,2026234585370,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,2.0,2,0,2.98,15.86,3.4121389389038086
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",221,2026237985537,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,1.0,1,0,5.54,36.52,1.9380362033843994
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",269,2026236285534,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,4.9,35.66,1.6344654560089111
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",474,2026232885637,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.055,15.975,2.2933707237243652
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",270,2026236285535,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.82,36.1,1.7688825130462646
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",222,2026237985538,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.77,34.94,2.107752799987793
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",157,2026234585371,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0375,16.135,3.5729312896728516
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",475,2026232885638,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9425,15.835,2.167609691619873
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",271,2026236285536,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.98,35.8,1.6797897815704346
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",223,2026237985539,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.63,36.16,2.0409791469573975
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",272,2026236285537,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.3,36.76,1.7976100444793701
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",476,2026232885639,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0525,15.99,2.14693546295166
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",158,2026234585372,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.5,1,1,3.13,15.96,3.2359468936920166
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",224,2026237985540,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.79,35.24,1.9877383708953857
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",273,2026236285538,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.81,35.64,1.7076208591461182
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",477,2026232885640,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.065,15.67,2.095989465713501
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",225,2026237985541,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.5,1,1,5.69,36.0,2.0089848041534424
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",274,2026236285539,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.57,35.2,1.7024507522583008
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",159,2026234585373,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.5,1,1,3.135,15.935,3.2079989910125732
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",478,2026232885641,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.1775,16.0,2.186028003692627
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",226,2026237985542,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.99,35.96,1.996194839477539
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",275,2026236285540,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.78,35.86,1.6483659744262695
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",479,2026232885642,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9625,15.88,2.1278629302978516
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",276,2026236285541,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.01,35.88,1.6430821418762207
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",227,2026237985543,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.5,36.94,2.046718120574951
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",160,2026234585374,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.1575,15.655,3.2475016117095947
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",277,2026236285542,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.98,35.66,1.6076388359069824
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",480,2026232885643,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.95,16.145,2.1348187923431396
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",228,2026237985544,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.21,34.8,2.0396838188171387
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",278,2026236285543,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.78,36.18,1.6100332736968994
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",161,2026234585375,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.5,2,1,3.0025,16.15,3.2242040634155273
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",481,2026232885644,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.065,16.28,2.19797420501709
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",229,2026237985545,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.04,35.22,2.1377172470092773
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",279,2026236285544,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.02,35.26,1.684723138809204
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",482,2026232885645,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.94,15.935,2.0681819915771484
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",280,2026236285545,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.14,36.74,1.5032367706298828
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",230,2026237985546,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.63,36.92,2.13893723487854
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",162,2026234585376,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0,15.975,3.572892189025879
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",281,2026236285546,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.06,36.2,1.649364709854126
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",483,2026232885646,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.04,15.67,2.236189842224121
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",231,2026237985547,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.67,35.78,2.145104169845581
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",282,2026236285547,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.9,35.82,1.701338291168213
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",484,2026232885647,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.1725,15.445,2.280573606491089
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",232,2026237985548,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.45,36.38,2.0476367473602295
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",163,2026234585377,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0225,16.04,3.662783145904541
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",283,2026236285548,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.84,36.24,1.6079418659210205
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",233,2026237985549,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.87,36.44,2.1814799308776855
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",485,2026232885648,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,2.0,2,0,2.02,16.06,2.3032546043395996
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",284,2026236285549,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.13,35.5,1.616852045059204
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",164,2026234585378,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0675,15.98,3.701998710632324
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",285,2026236285550,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.67,35.88,1.7226803302764893
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",234,2026237985550,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.72,36.28,1.9890503883361816
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",486,2026232885649,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0975,15.82,2.2651383876800537
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",286,2026236285551,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.1,36.1,1.6132786273956299
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",235,2026237985551,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.75,35.52,2.0769941806793213
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",487,2026232885650,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.045,16.03,2.178987979888916
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",287,2026236285552,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.97,35.64,1.8191583156585693
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",165,2026234585379,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0375,16.11,3.6758811473846436
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",236,2026237985552,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.55,34.8,2.148611307144165
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",488,2026232885651,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0775,16.59,2.1936585903167725
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",288,2026236285553,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,4.98,36.04,1.7247686386108398
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",237,2026237985553,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.81,36.3,2.161123037338257
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",289,2026236285554,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.96,35.78,1.6940813064575195
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",489,2026232885652,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9425,15.815,2.03825044631958
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",166,2026234585380,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,3.005,15.97,3.6071372032165527
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",238,2026237985554,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.28,36.0,2.1762568950653076
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",290,2026236285555,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,5.05,35.6,1.8845391273498535
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",490,2026232885653,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.0625,15.585,2.342470407485962
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",291,2026236285556,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,4.86,34.96,1.7137634754180908
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",167,2026234585381,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.5,2,1,3.105,16.425,3.4756622314453125
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",239,2026237985555,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,1.0,1,0,5.55,35.82,1.9961841106414795
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",491,2026232885654,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,2.0825,15.775,2.3688600063323975
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",292,2026236285557,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.5,1,1,5.5,35.6,1.7466413974761963
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",240,2026237985556,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.65,36.14,2.000624179840088
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",492,2026232885655,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,1.9675,15.79,2.1708312034606934
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",293,2026236285558,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.85,36.14,1.6592469215393066
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",168,2026234585382,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.96,15.905,3.675772190093994
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",241,2026237985557,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.65,35.6,2.074934482574463
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",294,2026236285559,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.72,35.44,1.7083747386932373
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",493,2026232885656,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.12,15.95,2.227323055267334
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",242,2026237985558,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.7,34.94,2.0218803882598877
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",295,2026236285560,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.88,36.36,1.7186298370361328
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",169,2026234585383,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.5,1,1,2.945,15.5,3.464979648590088
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",494,2026232885657,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0725,15.485,2.289482355117798
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",243,2026237985559,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.82,35.2,1.9678304195404053
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",296,2026236285561,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.76,35.12,1.758033275604248
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",495,2026232885658,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.025,15.975,2.08644700050354
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",297,2026236285562,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.07,36.42,1.7808949947357178
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",170,2026234585384,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,3.09,15.82,3.2030537128448486
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",244,2026237985560,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.26,35.9,2.1073336601257324
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",298,2026236285563,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.09,35.72,1.7383246421813965
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",496,2026232885659,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,1.945,16.09,2.155501127243042
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",245,2026237985561,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.5,1,1,5.64,35.62,2.117689847946167
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",299,2026236285564,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.21,35.14,1.7525372505187988
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",171,2026234585385,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.17,15.93,3.5896527767181396
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",497,2026232885660,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.8775,15.865,2.2090344429016113
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",246,2026237985562,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.45,35.08,1.8363127708435059
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",300,2026236285565,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.37,35.76,1.686758279800415
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",498,2026232885661,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.0075,15.65,2.107377052307129
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",247,2026237985563,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.97,35.48,2.154442071914673
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",301,2026236285566,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.77,35.86,1.5921483039855957
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",172,2026234585386,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,2.9,15.99,3.479968786239624
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",499,2026232885662,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,2.11,16.425,2.064098834991455
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",302,2026236285567,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.79,35.76,1.576885461807251
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",248,2026237985564,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.95,36.78,2.1968417167663574
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",303,2026236285568,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.05,35.76,1.5654010772705078
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",500,2026232885663,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,0.0,0,0,1.9575,16.09,2.237922430038452
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",249,2026237985565,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,1.0,1,0,5.69,35.6,2.0395095348358154
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",173,2026234585387,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.9825,15.93,3.4514081478118896
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",304,2026236285569,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.03,36.24,1.7574973106384277
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",250,2026237985566,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.03,35.86,2.060529947280884
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",501,2026232885664,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,1.0,1,0,1.9225,16.03,2.366527557373047
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",305,2026236285570,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.21,36.34,1.673125982284546
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",174,2026234585388,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,3.0325,16.095,3.213918685913086
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",251,2026237985567,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.43,36.46,1.9487662315368652
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,0.6,1",502,2026232885665,B,noisy,5,0.0002,0.0,1.0,0.6,1.0,5,3,5,400,2.5,3,1,2.0575,15.935,2.1086862087249756
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",306,2026236285571,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.31,37.74,1.627397060394287
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",252,2026237985568,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.16,38.32,1.9442534446716309
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",307,2026236285572,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.07,36.42,1.7407164573669434
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",175,2026234585389,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,3.0675,15.76,3.255396604537964
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",253,2026237985569,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.11,36.34,1.8769738674163818
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",308,2026236285573,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.15,35.68,1.5762667655944824
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",309,2026236285574,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.52,35.84,1.6736478805541992
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",0,2026239685367,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.58,35.32,3.4437174797058105
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",254,2026237985570,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.42,37.24,1.991523027420044
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",176,2026234585390,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.9975,16.11,3.3496017456054688
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",310,2026236285575,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.5,1,1,4.94,35.12,1.6548657417297363
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",255,2026237985571,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.84,35.26,2.1143476963043213
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",1,2026239685368,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.49,36.16,3.1259474754333496
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",311,2026236285576,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.82,35.64,1.7853777408599854
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",177,2026234585391,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.9425,16.145,3.3673579692840576
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",256,2026237985572,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.93,36.08,2.0500006675720215
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",312,2026236285577,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.28,35.7,1.5809123516082764
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",2,2026239685369,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.15,35.56,3.1586010456085205
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",257,2026237985573,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.28,35.56,2.0387532711029053
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",313,2026236285578,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.7,36.88,1.7871041297912598
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",178,2026234585392,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,3.14,16.315,3.360389232635498
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",314,2026236285579,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.5,1,1,4.63,37.26,1.5998096466064453
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",258,2026237985574,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.5,36.46,2.0593318939208984
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",3,2026239685370,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.33,36.36,3.1885623931884766
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",315,2026236285580,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.03,36.04,1.6886529922485352
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",259,2026237985575,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.76,36.22,1.896503210067749
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",179,2026234585393,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.5,1,1,3.0325,15.985,3.335571765899658
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",316,2026236285581,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.97,35.3,1.6302845478057861
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",260,2026237985576,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.69,36.0,2.050084114074707
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",4,2026239685371,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.74,36.84,3.260521411895752
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",317,2026236285582,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.1,34.72,1.7030260562896729
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",180,2026234585394,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.02,16.35,3.4076766967773438
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",261,2026237985577,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.45,35.44,2.021056890487671
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",318,2026236285583,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.51,35.52,1.703265905380249
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",5,2026239685372,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.97,35.84,3.5383129119873047
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",262,2026237985578,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.42,36.64,1.9467790126800537
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",319,2026236285584,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.1,36.92,1.8375918865203857
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",181,2026234585395,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0725,16.04,3.312325954437256
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",263,2026237985579,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.04,37.62,2.0849838256835938
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",320,2026236285585,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.19,35.14,1.7135827541351318
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",6,2026239685373,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.33,36.72,3.4876463413238525
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",321,2026236285586,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.22,35.62,1.5864450931549072
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",264,2026237985580,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.48,35.66,1.9402883052825928
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",182,2026234585396,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.5,1,1,2.9125,15.96,3.5537123680114746
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",322,2026236285587,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.09,35.4,1.5505800247192383
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",265,2026237985581,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.09,36.14,2.074793577194214
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",323,2026236285588,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.62,36.12,1.6716163158416748
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",7,2026239685374,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.26,35.76,3.6955060958862305
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",183,2026234585397,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.9725,15.895,3.624166250228882
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",266,2026237985582,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.82,35.94,2.147162675857544
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",324,2026236285589,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.13,37.14,1.7082078456878662
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",267,2026237985583,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.41,35.16,1.935354471206665
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",325,2026236285590,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.5,1,1,4.89,36.32,1.694500207901001
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",8,2026239685375,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.14,35.86,3.4425597190856934
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",184,2026234585398,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.095,15.92,3.5800068378448486
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",326,2026236285591,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.76,35.92,1.7354729175567627
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",268,2026237985584,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,1.0,1,0,5.76,36.14,2.056676149368286
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",9,2026239685376,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,1.0,1,0,8.13,36.86,3.2758023738861084
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",327,2026236285592,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.15,36.04,1.7338805198669434
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",269,2026237985585,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.48,36.1,2.0130484104156494
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",185,2026234585399,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.5,1,1,3.0075,15.905,3.4716796875
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",328,2026236285593,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.59,35.3,1.5774943828582764
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",270,2026237985586,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.29,36.38,1.9272654056549072
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",10,2026239685377,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.12,36.0,3.154808282852173
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",329,2026236285594,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.97,36.02,1.6764445304870605
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",271,2026237985587,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.8,35.4,1.9400033950805664
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",186,2026234585400,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.5,1,1,2.945,16.07,3.2815442085266113
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",330,2026236285595,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.52,35.88,1.6083784103393555
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",11,2026239685378,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.98,36.76,3.1090731620788574
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",272,2026237985588,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.66,35.78,1.954216480255127
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",331,2026236285596,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.26,35.9,1.5137441158294678
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",187,2026234585401,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,3.15,15.925,3.102771282196045
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",273,2026237985589,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.38,36.8,1.807619333267212
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",332,2026236285597,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,5.08,35.64,1.7030999660491943
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",12,2026239685379,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.99,35.5,3.091287136077881
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",333,2026236285598,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.24,36.24,1.6970419883728027
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",274,2026237985590,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.14,35.86,1.8802385330200195
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",188,2026234585402,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.1975,15.795,3.2574706077575684
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",334,2026236285599,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.78,35.84,1.820641279220581
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",275,2026237985591,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.86,36.02,2.084195852279663
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",13,2026239685380,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.99,36.88,3.1590933799743652
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",335,2026236285600,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.96,36.26,1.80430006980896
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",276,2026237985592,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.95,36.3,2.097351312637329
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",189,2026234585403,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.145,15.635,3.5530097484588623
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",336,2026236285601,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.0,36.3,1.7088754177093506
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",14,2026239685381,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.82,35.7,3.339404344558716
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",277,2026237985593,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,1.0,1,0,5.59,35.6,2.0121800899505615
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",337,2026236285602,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.28,37.18,1.6109082698822021
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",190,2026234585404,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.895,16.18,3.584669351577759
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",278,2026237985594,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.22,35.66,1.837632417678833
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",338,2026236285603,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,5.35,35.4,1.6382925510406494
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",15,2026239685382,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.99,35.78,3.360708475112915
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",279,2026237985595,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.68,35.56,1.8486099243164062
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",339,2026236285604,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.66,36.12,1.4861838817596436
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",191,2026234585405,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.06,16.33,3.475255250930786
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",340,2026236285605,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.6,36.22,1.7604284286499023
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",280,2026237985596,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.5,1,1,5.86,35.26,1.9085192680358887
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",16,2026239685383,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.36,36.34,3.3868308067321777
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",341,2026236285606,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.04,36.98,1.7590413093566895
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",281,2026237985597,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.83,35.84,1.8303430080413818
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",192,2026234585406,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.015,16.24,3.605954885482788
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",342,2026236285607,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.82,34.94,1.6412248611450195
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",282,2026237985598,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,1.0,1,0,5.77,37.1,2.09621000289917
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",17,2026239685384,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.1,35.72,3.3442482948303223
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",343,2026236285608,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.04,35.2,1.6028378009796143
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",283,2026237985599,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.56,36.48,1.879751443862915
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",344,2026236285609,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.39,36.28,1.681957483291626
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",193,2026234585407,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.05,16.005,3.5640318393707275
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",18,2026239685385,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.37,35.82,3.538544178009033
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",284,2026237985600,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.67,35.76,2.0818514823913574
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",345,2026236285610,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.75,36.0,1.591491937637329
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",285,2026237985601,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.36,35.66,2.1669695377349854
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",346,2026236285611,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.11,35.58,1.827735424041748
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",194,2026234585408,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,2.91,16.125,3.3445425033569336
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",19,2026239685386,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.09,35.64,3.688009023666382
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",347,2026236285612,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.35,36.36,1.6947863101959229
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",286,2026237985602,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.71,35.9,2.120779514312744
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",348,2026236285613,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.2,36.4,1.6198561191558838
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",195,2026234585409,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.045,15.99,3.2546839714050293
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",287,2026237985603,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.38,36.02,2.032268762588501
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",20,2026239685387,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,1.0,1,0,7.92,36.14,3.178596258163452
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",349,2026236285614,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.47,36.5,1.718428611755371
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",288,2026237985604,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.83,35.04,2.056630849838257
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",350,2026236285615,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.99,36.38,1.6886630058288574
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",196,2026234585410,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.5,1,1,2.9975,16.15,3.448852777481079
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",21,2026239685388,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.24,35.8,3.0517332553863525
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",289,2026237985605,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.75,35.46,1.952653169631958
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",351,2026236285616,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.92,36.06,1.7085931301116943
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",352,2026236285617,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.89,36.56,1.628746747970581
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",290,2026237985606,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.0,36.42,2.066136598587036
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",197,2026234585411,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.5,1,1,3.1,15.85,3.433196544647217
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",22,2026239685389,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.79,36.1,3.5308430194854736
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",353,2026236285618,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.5,1,1,4.95,35.18,1.6894094944000244
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",291,2026237985607,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.8,36.1,1.9731242656707764
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",354,2026236285619,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.07,36.16,1.6400413513183594
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",198,2026234585412,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,3.03,16.09,3.7849507331848145
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",292,2026237985608,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.9,37.12,2.071563243865967
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",23,2026239685390,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.76,34.86,3.4772987365722656
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",355,2026236285620,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.71,35.88,1.6568458080291748
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",293,2026237985609,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.78,36.06,2.021195650100708
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",356,2026236285621,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.82,34.88,1.6787493228912354
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",199,2026234585413,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.945,16.03,3.3356266021728516
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",24,2026239685391,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.85,35.4,3.1656699180603027
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",294,2026237985610,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.89,36.26,1.9569029808044434
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",357,2026236285622,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.5,1,1,5.11,37.12,1.704350233078003
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",358,2026236285623,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.91,36.44,1.7639563083648682
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",295,2026237985611,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.72,35.72,2.1693434715270996
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",200,2026234585414,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.9475,16.11,3.4244930744171143
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",25,2026239685392,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.89,35.68,3.403695821762085
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",359,2026236285624,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.83,35.36,1.5999889373779297
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",296,2026237985612,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.45,36.94,1.9199225902557373
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",360,2026236285625,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.02,34.98,1.5473518371582031
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",201,2026234585415,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.1225,16.14,3.344444751739502
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",297,2026237985613,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.01,36.32,1.9739580154418945
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",26,2026239685393,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.45,35.46,3.2603299617767334
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",361,2026236285626,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.77,34.72,1.5987553596496582
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",298,2026237985614,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.92,36.14,1.9308476448059082
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",362,2026236285627,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.77,36.76,1.433237075805664
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",202,2026234585416,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,3.07,16.13,3.3625590801239014
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",27,2026239685394,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.47,35.98,3.441112995147705
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",363,2026236285628,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.08,36.24,1.7253139019012451
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",299,2026237985615,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.99,35.98,2.1157875061035156
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",364,2026236285629,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.83,35.98,1.8634076118469238
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",300,2026237985616,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.55,35.64,2.1096572875976562
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",203,2026234585417,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.9025,16.06,3.5534160137176514
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",28,2026239685395,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.38,34.46,3.7359962463378906
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",365,2026236285630,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.12,36.3,1.7312731742858887
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",301,2026237985617,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.97,34.88,2.119373083114624
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",366,2026236285631,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.82,35.22,1.7026035785675049
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",302,2026237985618,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.83,35.6,1.958308458328247
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",204,2026234585418,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.5,1,1,3.0375,16.145,3.6063549518585205
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",29,2026239685396,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.89,36.66,3.2772538661956787
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",367,2026236285632,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.06,35.68,1.5630772113800049
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",368,2026236285633,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.18,36.08,1.6682713031768799
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",303,2026237985619,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.6,35.34,2.2565524578094482
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",369,2026236285634,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.89,35.2,1.568669080734253
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",30,2026239685397,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.17,36.84,3.3555426597595215
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",205,2026234585419,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,2.9275,15.625,3.687340259552002
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",304,2026237985620,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.58,35.52,2.0328805446624756
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",370,2026236285635,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,5.32,36.32,1.6737899780273438
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",305,2026237985621,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.82,36.78,2.219407320022583
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",371,2026236285636,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.5,1,1,5.35,36.86,1.7836389541625977
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",31,2026239685398,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.56,35.68,3.7343828678131104
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",206,2026234585420,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,3.0175,15.785,3.654794692993164
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",306,2026237985622,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,1.0,1,0,5.71,35.02,2.0766141414642334
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",372,2026236285637,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,4.99,36.3,1.5218875408172607
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",373,2026236285638,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.02,35.22,1.6427631378173828
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",307,2026237985623,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.26,35.64,2.1110618114471436
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",207,2026234585421,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.94,16.13,3.3362576961517334
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",32,2026239685399,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.74,36.1,3.641152858734131
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",374,2026236285639,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.14,36.28,1.7421088218688965
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",308,2026237985624,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.54,35.36,2.08420729637146
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",375,2026236285640,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.61,35.44,1.536346673965454
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",208,2026234585422,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.995,16.13,3.2526652812957764
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",33,2026239685400,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.35,35.68,3.493054151535034
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",309,2026237985625,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.5,1,1,5.8,36.34,1.9621796607971191
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",376,2026236285641,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.16,36.02,1.697312355041504
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",310,2026237985626,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.78,36.66,1.8505520820617676
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",377,2026236285642,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.45,36.42,1.7225768566131592
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",209,2026234585423,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.9375,16.045,3.2235448360443115
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",34,2026239685401,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.08,36.36,3.5312302112579346
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",311,2026237985627,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.65,37.22,1.9497120380401611
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",378,2026236285643,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.29,36.3,1.654775857925415
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",379,2026236285644,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,5.08,35.96,1.814422369003296
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",312,2026237985628,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.01,35.44,1.958996295928955
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",210,2026234585424,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.1475,16.085,3.35878849029541
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",35,2026239685402,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.25,35.12,3.869629383087158
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",380,2026236285645,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.91,35.56,1.6061859130859375
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",313,2026237985629,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.08,35.68,1.8935439586639404
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",381,2026236285646,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.37,35.0,1.5835256576538086
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",211,2026234585425,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.5,1,1,2.9675,16.185,3.093343734741211
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",314,2026237985630,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.7,34.98,1.9986968040466309
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",382,2026236285647,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.83,35.8,1.6036767959594727
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",36,2026239685403,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.68,34.76,3.488529682159424
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",315,2026237985631,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.92,35.62,2.1488914489746094
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",212,2026234585426,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.1175,15.89,3.1715712547302246
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",383,2026236285648,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.69,36.1,1.7556002140045166
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",316,2026237985632,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.7,36.88,2.009220838546753
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",384,2026236285649,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.19,36.52,1.6048707962036133
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",37,2026239685404,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.93,36.06,3.2134623527526855
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",213,2026234585427,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.05,16.43,3.177067518234253
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",385,2026236285650,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.09,35.42,1.6776254177093506
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",317,2026237985633,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.42,35.24,1.9090745449066162
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",38,2026239685405,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.96,35.76,3.3082125186920166
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",386,2026236285651,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.06,37.06,1.696045160293579
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",318,2026237985634,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.67,36.1,2.0565216541290283
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",214,2026234585428,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.05,15.73,3.440092086791992
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",387,2026236285652,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.79,36.42,1.529820442199707
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",319,2026237985635,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,1.0,1,0,5.59,36.14,2.074160575866699
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",39,2026239685406,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,1.0,1,0,8.1,36.32,3.3648481369018555
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",388,2026236285653,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.02,36.32,1.8022832870483398
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",320,2026237985636,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.62,36.94,2.091301441192627
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",215,2026234585429,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.015,15.955,3.593458652496338
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",389,2026236285654,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.8,36.6,1.8690507411956787
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",321,2026237985637,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.88,34.48,1.9289166927337646
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",40,2026239685407,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.47,35.8,3.6021368503570557
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",390,2026236285655,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.16,36.76,1.7186851501464844
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",391,2026236285656,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.67,36.46,1.6032962799072266
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",216,2026234585430,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,3.0075,16.0,3.4152517318725586
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",322,2026237985638,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.21,35.44,2.121875762939453
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",392,2026236285657,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.87,36.74,1.4266598224639893
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",323,2026237985639,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.86,35.36,1.854649543762207
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",41,2026239685408,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.37,36.26,3.774664878845215
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",393,2026236285658,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.32,35.7,1.4819867610931396
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",217,2026234585431,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0475,15.885,3.300628900527954
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",324,2026237985640,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.68,35.54,1.8662288188934326
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",394,2026236285659,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.95,35.7,1.4872264862060547
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",42,2026239685409,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,1.0,1,0,7.78,36.14,3.5076277256011963
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",325,2026237985641,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.86,36.58,2.0543854236602783
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",395,2026236285660,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.03,35.4,1.6323246955871582
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",218,2026234585432,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,2.8575,16.09,3.1542580127716064
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",396,2026236285661,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.02,36.42,1.429337501525879
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",326,2026237985642,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.57,35.84,1.9396450519561768
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",43,2026239685410,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.21,35.98,3.4757652282714844
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",397,2026236285662,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.21,35.26,1.8490939140319824
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",219,2026234585433,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.065,16.135,3.4798319339752197
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",327,2026237985643,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.71,35.64,2.14507794380188
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",398,2026236285663,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.5,1,1,5.06,35.24,1.7614681720733643
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",328,2026237985644,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.39,36.02,2.193150520324707
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",44,2026239685411,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.16,35.46,3.4458627700805664
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",399,2026236285664,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.13,36.9,1.7963922023773193
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",220,2026234585434,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.9675,15.965,3.3848445415496826
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",329,2026237985645,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.17,36.02,2.0498263835906982
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",400,2026236285665,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.0,35.48,1.6251535415649414
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",45,2026239685412,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.32,35.6,3.3494479656219482
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",401,2026236285666,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.82,36.18,1.7564544677734375
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",330,2026237985646,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.67,34.88,1.9958958625793457
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",221,2026234585435,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,2.0,2,0,2.8675,15.865,3.498424530029297
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",402,2026236285667,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.0,35.84,1.76711106300354
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",331,2026237985647,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.96,36.58,1.917137622833252
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",46,2026239685413,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.19,35.68,3.361363410949707
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",403,2026236285668,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.79,34.8,1.630765438079834
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",332,2026237985648,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.39,37.2,1.7876331806182861
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",222,2026234585436,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.045,15.885,3.4249820709228516
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",404,2026236285669,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.93,36.52,1.6384656429290771
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",333,2026237985649,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.65,36.64,2.052187919616699
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",47,2026239685414,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.89,35.7,3.57175612449646
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",405,2026236285670,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.5,1,1,5.25,36.18,1.8184237480163574
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",223,2026234585437,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0425,15.9,3.342695474624634
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",334,2026237985650,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.37,35.42,2.0848629474639893
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",406,2026236285671,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.86,35.62,1.7139883041381836
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",48,2026239685415,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.57,36.46,3.2117176055908203
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",335,2026237985651,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.16,36.38,1.967904806137085
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",407,2026236285672,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.07,35.42,1.7522504329681396
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",224,2026234585438,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.1075,16.195,3.6356310844421387
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",336,2026237985652,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.53,36.86,2.1595165729522705
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",408,2026236285673,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.33,36.96,1.7641150951385498
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",49,2026239685416,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.74,36.16,3.1340906620025635
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",409,2026236285674,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.15,35.8,1.7150259017944336
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",337,2026237985653,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.97,34.58,2.1505179405212402
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",225,2026234585439,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.15,15.775,3.543449878692627
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",410,2026236285675,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.65,36.84,1.4973807334899902
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",50,2026239685417,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.96,37.42,3.0723764896392822
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",338,2026237985654,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.81,36.52,2.0412635803222656
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",411,2026236285676,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,5.43,36.14,1.5068533420562744
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",226,2026234585440,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.5,1,1,2.985,16.005,3.6995506286621094
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",339,2026237985655,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.92,36.5,2.0352094173431396
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",412,2026236285677,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.54,36.5,1.715548038482666
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",51,2026239685418,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.16,36.9,3.5632102489471436
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",413,2026236285678,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.19,35.66,1.746321201324463
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",340,2026237985656,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.05,36.4,2.110971450805664
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",227,2026234585441,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,3.11,16.06,3.629824161529541
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",414,2026236285679,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,5.01,35.88,1.7514159679412842
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",341,2026237985657,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.66,36.82,2.042070150375366
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",52,2026239685419,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.32,35.18,3.5848021507263184
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",415,2026236285680,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,4.64,35.98,1.6581826210021973
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",342,2026237985658,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.88,36.78,1.94362473487854
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",228,2026234585442,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,3.0,15.94,3.449289321899414
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",416,2026236285681,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.96,35.4,1.639289140701294
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",343,2026237985659,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.14,36.26,2.035541296005249
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",53,2026239685420,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.24,36.12,3.7702133655548096
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",417,2026236285682,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.1,36.18,1.7166481018066406
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",344,2026237985660,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.14,36.56,2.022775650024414
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",418,2026236285683,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.99,36.44,1.737076759338379
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",229,2026234585443,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.915,15.605,3.5743465423583984
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",54,2026239685421,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.42,36.52,3.3085882663726807
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",419,2026236285684,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.69,36.18,1.5823781490325928
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",345,2026237985661,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.94,36.22,2.1067728996276855
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",420,2026236285685,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.9,36.96,1.6795141696929932
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",230,2026234585444,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.845,16.01,3.23002290725708
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",346,2026237985662,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.88,35.32,2.090372085571289
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",55,2026239685422,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.15,35.18,3.317934036254883
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",421,2026236285686,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.2,35.08,1.7254250049591064
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",347,2026237985663,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.53,36.34,2.0448153018951416
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",422,2026236285687,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.73,35.08,1.6508781909942627
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",231,2026234585445,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0325,16.15,3.457496404647827
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",348,2026237985664,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.61,36.58,1.940004587173462
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",56,2026239685423,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.66,36.26,3.3037915229797363
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",423,2026236285688,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.16,36.04,1.6802401542663574
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",349,2026237985665,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.83,35.84,1.8958406448364258
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",424,2026236285689,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.61,36.54,1.7189264297485352
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",232,2026234585446,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.1175,16.14,3.52189040184021
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",425,2026236285690,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.77,36.94,1.5489959716796875
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",350,2026237985666,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.5,1,1,5.82,36.76,1.9371228218078613
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",57,2026239685424,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.0,36.6,3.8238329887390137
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",426,2026236285691,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.97,34.82,1.7103807926177979
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",351,2026237985667,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.89,35.86,1.8951046466827393
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",233,2026234585447,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,3.0175,16.035,3.531346559524536
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",427,2026236285692,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.77,35.84,1.5718162059783936
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",58,2026239685425,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.9,35.36,3.3364644050598145
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",352,2026237985668,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.32,35.96,2.160205841064453
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",428,2026236285693,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.57,37.2,1.6123676300048828
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",234,2026234585448,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.16,16.165,3.288224697113037
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",353,2026237985669,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.82,36.74,2.1090285778045654
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",429,2026236285694,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.3,36.48,1.6146917343139648
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",59,2026239685426,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.12,36.86,3.333792209625244
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",430,2026236285695,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.89,35.68,1.7717921733856201
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",354,2026237985670,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.28,36.16,2.0629165172576904
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",235,2026234585449,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.955,16.02,3.6010565757751465
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",431,2026236285696,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.49,37.0,1.6322405338287354
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",355,2026237985671,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.44,35.52,2.052218437194824
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",60,2026239685427,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.67,36.08,3.5376014709472656
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",432,2026236285697,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.18,36.88,1.641899585723877
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",356,2026237985672,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.69,36.52,1.8550059795379639
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",236,2026234585450,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,3.1525,15.87,3.43730092048645
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",433,2026236285698,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.81,36.32,1.446232557296753
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",61,2026239685428,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.94,35.86,3.1970326900482178
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",357,2026237985673,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,1.0,1,0,5.82,35.76,2.0873119831085205
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",434,2026236285699,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.95,35.98,1.4930617809295654
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",237,2026234585451,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.035,16.115,3.312676191329956
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",435,2026236285700,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.78,36.48,1.5216293334960938
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",358,2026237985674,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.99,36.52,1.883763074874878
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",62,2026239685429,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.99,36.18,3.316117525100708
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",436,2026236285701,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.61,35.16,1.5032598972320557
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",359,2026237985675,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.25,35.72,2.0963294506073
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",437,2026236285702,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.1,36.78,1.5961127281188965
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",238,2026234585452,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,2.98,15.975,3.6631035804748535
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",63,2026239685430,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.12,35.66,3.0286145210266113
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",360,2026237985676,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.68,36.8,1.8863255977630615
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",438,2026236285703,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,5.12,35.44,1.6249306201934814
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",361,2026237985677,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.64,35.9,1.8869116306304932
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",439,2026236285704,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.87,36.54,1.6787161827087402
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",239,2026234585453,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.5,1,1,3.06,16.07,3.429720401763916
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",64,2026239685431,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.66,35.06,3.1958718299865723
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",362,2026237985678,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.87,35.92,1.9303631782531738
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",440,2026236285705,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.49,35.54,1.646144151687622
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",441,2026236285706,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.72,35.9,1.643092393875122
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",363,2026237985679,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.42,35.62,1.8575594425201416
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",65,2026239685432,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.88,35.76,3.067669630050659
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",240,2026234585454,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0475,15.92,3.590885877609253
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",442,2026236285707,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.27,35.76,1.7974693775177002
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",364,2026237985680,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,1.0,1,0,5.63,36.34,2.0489842891693115
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",443,2026236285708,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.11,36.3,1.6490941047668457
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",66,2026239685433,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.21,35.42,3.2509348392486572
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",365,2026237985681,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.59,36.56,2.1242549419403076
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",241,2026234585455,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.5,1,1,3.1975,16.135,3.7935962677001953
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",444,2026236285709,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.58,35.74,1.7855849266052246
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",366,2026237985682,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.59,36.88,1.84334397315979
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",445,2026236285710,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.2,36.08,1.7923905849456787
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",67,2026239685434,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.86,37.44,3.5306851863861084
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",242,2026234585456,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0825,16.275,3.62435245513916
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",367,2026237985683,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,1.0,1,0,5.91,36.7,2.1098709106445312
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",446,2026236285711,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.11,34.96,1.7003226280212402
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",447,2026236285712,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.59,35.06,1.450334072113037
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",368,2026237985684,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.51,36.36,2.035146713256836
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",68,2026239685435,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.56,36.46,3.5860557556152344
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",243,2026234585457,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0075,16.08,3.280078411102295
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",448,2026236285713,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.84,36.58,1.47676420211792
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",369,2026237985685,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,1.0,1,0,6.0,36.06,2.057790517807007
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",449,2026236285714,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.77,36.74,1.7760748863220215
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",370,2026237985686,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.59,35.38,1.8670573234558105
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",69,2026239685436,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.85,35.9,3.7250542640686035
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",244,2026234585458,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,2.9375,16.17,3.5918338298797607
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",450,2026236285715,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.01,35.76,1.6904633045196533
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",371,2026237985687,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.81,35.36,2.085346221923828
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",451,2026236285716,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,2,2,5.02,35.38,1.6861763000488281
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",372,2026237985688,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.65,35.82,2.042384386062622
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",70,2026239685437,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.36,35.1,3.754920482635498
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",452,2026236285717,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.73,35.56,1.7347464561462402
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",245,2026234585459,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,3.0175,16.12,3.467474937438965
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",453,2026236285718,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.87,36.04,1.646618127822876
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",373,2026237985689,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.64,36.14,1.9327332973480225
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",454,2026236285719,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.71,35.7,1.742110252380371
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",246,2026234585460,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.1725,16.06,3.5609655380249023
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",71,2026239685438,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.24,37.58,3.7513844966888428
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",374,2026237985690,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.64,35.98,2.0579748153686523
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",455,2026236285720,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.97,35.32,1.865354061126709
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",375,2026237985691,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.28,36.18,1.997133731842041
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",456,2026236285721,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.94,36.02,1.545128345489502
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",247,2026234585461,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.9525,16.04,3.5158674716949463
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",72,2026239685439,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.42,35.62,3.4670910835266113
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",376,2026237985692,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.82,35.34,1.9737017154693604
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",457,2026236285722,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.82,36.0,1.6082510948181152
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",377,2026237985693,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.68,35.34,1.9698450565338135
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",458,2026236285723,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.46,37.04,1.77079439163208
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",73,2026239685440,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.56,36.68,3.199355363845825
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",248,2026234585462,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0075,16.06,3.5926527976989746
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",378,2026237985694,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,1.0,1,0,6.09,36.46,2.1473140716552734
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",459,2026236285724,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.05,35.94,1.7387547492980957
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",460,2026236285725,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.8,36.2,1.8028435707092285
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",74,2026239685441,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,1.0,1,0,8.24,36.82,3.4597933292388916
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",379,2026237985695,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.96,34.92,2.120300769805908
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",249,2026234585463,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0925,16.405,3.628877878189087
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",461,2026236285726,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.02,35.1,1.6757235527038574
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",380,2026237985696,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.75,35.98,2.009587049484253
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",462,2026236285727,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.71,35.96,1.7427845001220703
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",75,2026239685442,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.39,35.02,3.6143765449523926
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",250,2026234585464,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.105,16.205,3.4789724349975586
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",381,2026237985697,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.39,36.46,1.9946999549865723
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",463,2026236285728,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.92,35.96,1.6187770366668701
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",382,2026237985698,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.97,35.96,2.028064489364624
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",464,2026236285729,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.95,36.0,1.6991498470306396
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",76,2026239685443,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.52,37.08,3.337385654449463
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",251,2026234585465,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.965,16.22,3.7571051120758057
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",383,2026237985699,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.76,35.44,2.0053300857543945
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",465,2026236285730,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.06,36.1,1.6620497703552246
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",466,2026236285731,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.11,35.9,1.6735680103302002
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",77,2026239685444,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.14,35.26,3.2309646606445312
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",384,2026237985700,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.66,36.56,1.9761443138122559
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",252,2026234585466,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0025,16.375,3.615177631378174
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",467,2026236285732,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.51,36.28,1.8416752815246582
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",385,2026237985701,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.7,35.3,1.9497952461242676
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",468,2026236285733,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.27,36.62,1.5732226371765137
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",78,2026239685445,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.14,35.94,3.4609360694885254
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",386,2026237985702,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.53,35.62,1.8681175708770752
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",253,2026234585467,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,3.04,16.17,3.505603313446045
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",469,2026236285734,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.68,36.18,1.5874903202056885
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",387,2026237985703,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.89,35.12,2.012632131576538
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",470,2026236285735,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.5,1,1,4.87,35.78,1.5082411766052246
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",79,2026239685446,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.69,37.04,3.130614757537842
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",388,2026237985704,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.63,35.7,1.9807007312774658
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",471,2026236285736,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.8,36.06,1.6486835479736328
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",254,2026234585468,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,3.1375,16.15,3.6553876399993896
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",472,2026236285737,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.34,37.28,1.6658153533935547
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",389,2026237985705,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.82,35.24,1.923543930053711
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",80,2026239685447,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.8,36.02,3.6369223594665527
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",473,2026236285738,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,4.82,36.14,1.6591196060180664
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",390,2026237985706,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.41,36.14,1.771552562713623
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",255,2026234585469,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,2.775,15.78,3.49776291847229
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",474,2026236285739,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.89,36.18,1.4802205562591553
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",391,2026237985707,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.69,35.8,1.834665060043335
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",81,2026239685448,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,1.0,1,0,8.25,35.96,3.5040998458862305
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",475,2026236285740,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.83,36.4,1.5066702365875244
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",392,2026237985708,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.47,37.12,1.8549854755401611
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",256,2026234585470,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.135,15.895,3.566262722015381
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",476,2026236285741,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.01,35.14,1.489750623703003
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",82,2026239685449,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.28,34.52,3.48940372467041
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",393,2026237985709,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.05,36.12,2.0561399459838867
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",477,2026236285742,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.01,35.46,1.6406183242797852
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",257,2026234585471,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.01,15.94,3.170808792114258
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",478,2026236285743,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.75,35.66,1.565976858139038
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",394,2026237985710,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.65,36.5,1.8516712188720703
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",479,2026236285744,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.87,36.4,1.7889933586120605
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",83,2026239685450,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.19,35.32,3.577570676803589
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",395,2026237985711,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.57,35.4,1.964491367340088
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",258,2026234585472,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.5,2,1,3.0175,15.7,3.636735200881958
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",480,2026236285745,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.12,36.56,1.9011659622192383
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",396,2026237985712,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.72,35.72,1.8356194496154785
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",481,2026236285746,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,5.06,35.52,1.7668213844299316
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",84,2026239685451,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.12,36.64,3.8641040325164795
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",397,2026237985713,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.67,34.8,2.0763635635375977
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",259,2026234585473,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.5,2,1,3.0675,15.56,3.5723602771759033
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",482,2026236285747,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.79,36.72,1.598304271697998
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",398,2026237985714,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.06,36.22,1.8831064701080322
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",483,2026236285748,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.96,35.22,1.5260543823242188
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",85,2026239685452,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.42,35.92,3.507727861404419
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",399,2026237985715,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.8,36.1,2.1847238540649414
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",484,2026236285749,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.5,1,1,5.02,35.5,1.6586170196533203
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",260,2026234585474,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.135,16.025,3.6619985103607178
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",485,2026236285750,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.68,36.26,1.6951782703399658
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",400,2026237985716,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.75,36.04,2.0064361095428467
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",86,2026239685453,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.42,37.34,3.6018729209899902
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",486,2026236285751,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.65,36.62,1.692415475845337
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",261,2026234585475,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,3.065,16.065,3.389371395111084
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",401,2026237985717,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.5,1,1,5.53,36.84,2.00712251663208
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",487,2026236285752,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.5,1,1,5.31,36.2,1.6010935306549072
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",87,2026239685454,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.5,1,1,8.48,36.1,3.1906135082244873
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",402,2026237985718,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.59,35.52,1.9558098316192627
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",488,2026236285753,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.42,35.28,1.5007662773132324
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",262,2026234585476,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.05,16.075,3.489534616470337
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",403,2026237985719,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.96,34.8,2.003127098083496
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",489,2026236285754,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.92,35.9,1.6276860237121582
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",88,2026239685455,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.18,35.6,3.272894859313965
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",490,2026236285755,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.06,36.02,1.6168286800384521
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",404,2026237985720,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.5,1,1,6.05,36.84,2.130300521850586
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",263,2026234585477,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.135,15.925,3.3838813304901123
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",491,2026236285756,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.07,36.16,1.5721991062164307
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",405,2026237985721,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.52,36.08,2.1220920085906982
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",89,2026239685456,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.38,36.36,3.4631831645965576
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",492,2026236285757,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.71,35.9,1.7307755947113037
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",264,2026234585478,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.5,3,3,2.98,15.775,3.3920400142669678
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",406,2026237985722,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.84,35.98,1.9219141006469727
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",493,2026236285758,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.99,36.26,1.7282395362854004
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",90,2026239685457,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,1.0,1,0,8.61,35.6,3.388169288635254
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",407,2026237985723,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.74,35.22,2.021035671234131
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",494,2026236285759,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.71,34.9,1.599012851715088
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",265,2026234585479,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.88,15.79,3.361363172531128
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",408,2026237985724,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.01,37.02,1.939509391784668
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",495,2026236285760,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.24,34.72,1.6248433589935303
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",91,2026239685458,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.08,35.6,3.248262882232666
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",496,2026236285761,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.62,36.06,1.685931921005249
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",409,2026237985725,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.67,35.92,2.038907051086426
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",266,2026234585480,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.02,16.07,3.579543113708496
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",497,2026236285762,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.9,36.36,1.6994054317474365
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",410,2026237985726,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.36,36.32,2.057314157485962
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",92,2026239685459,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,1.0,1,0,8.71,36.52,3.795048475265503
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",498,2026236285763,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.79,35.86,1.7370152473449707
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",411,2026237985727,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.5,36.58,1.9954288005828857
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",267,2026234585481,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.065,16.115,3.2985875606536865
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",499,2026236285764,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.74,35.0,1.6530711650848389
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",412,2026237985728,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.32,36.04,1.9932715892791748
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",500,2026236285765,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.1,35.84,1.6055164337158203
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",93,2026239685460,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.92,35.5,3.7963168621063232
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",268,2026234585482,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.015,16.185,3.2630138397216797
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",501,2026236285766,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.07,35.88,1.5918281078338623
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",413,2026237985729,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.76,35.92,1.9659104347229004
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",502,2026236285767,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.37,36.4,1.6992065906524658
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",414,2026237985730,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.07,36.76,2.0528407096862793
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",94,2026239685461,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.47,36.0,3.6356093883514404
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",269,2026234585483,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,2.99,16.285,3.4226560592651367
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",503,2026236285768,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.25,35.08,1.9139008522033691
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",415,2026237985731,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.48,35.74,2.0292935371398926
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",504,2026236285769,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.11,35.92,1.5527315139770508
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",95,2026239685462,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.85,35.9,3.4384090900421143
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",416,2026237985732,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.75,36.48,1.9901249408721924
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",270,2026234585484,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.9675,15.785,3.3607304096221924
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",505,2026236285770,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,5.29,35.4,1.644866704940796
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",417,2026237985733,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.92,36.32,1.9682862758636475
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",506,2026236285771,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.83,36.78,1.6498849391937256
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",96,2026239685463,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.0,35.26,3.483987808227539
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",271,2026234585485,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.5,1,1,2.955,16.085,3.223703622817993
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",418,2026237985734,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.57,35.72,1.8367533683776855
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",507,2026236285772,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.07,36.1,1.5565872192382812
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",508,2026236285773,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.78,35.64,1.6028130054473877
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",419,2026237985735,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.32,36.5,2.0128185749053955
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",97,2026239685464,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.6,35.06,3.4383959770202637
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",272,2026234585486,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.065,16.085,3.477346658706665
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",509,2026236285774,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.97,34.9,1.5579862594604492
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",420,2026237985736,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.61,36.14,2.103900194168091
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",510,2026236285775,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.35,36.96,1.9191555976867676
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",98,2026239685465,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.13,35.48,3.3475823402404785
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",421,2026237985737,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.24,36.04,2.1515488624572754
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",273,2026234585487,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,2.895,16.245,3.6035735607147217
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",511,2026236285776,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.72,36.18,1.8174171447753906
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",422,2026237985738,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.92,36.58,1.8678944110870361
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",512,2026236285777,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.07,36.08,1.5902011394500732
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",99,2026239685466,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.26,35.78,3.2356514930725098
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",274,2026234585488,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,2.9425,15.98,3.2125892639160156
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",423,2026237985739,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.59,36.2,2.0403482913970947
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",513,2026236285778,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.78,36.18,1.7318179607391357
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",514,2026236285779,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.71,36.12,1.6316180229187012
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",424,2026237985740,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.8,35.38,2.12732195854187
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",100,2026239685467,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.32,36.44,3.330089569091797
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",275,2026234585489,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.9525,15.585,3.554570198059082
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",515,2026236285780,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.92,36.22,1.8580832481384277
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",425,2026237985741,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.7,35.92,2.118241786956787
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",516,2026236285781,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.13,36.38,1.7956795692443848
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",101,2026239685468,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.38,36.66,3.522336006164551
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",426,2026237985742,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.27,36.24,1.8117542266845703
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",276,2026234585490,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.5,1,1,2.9225,15.995,3.4858057498931885
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",517,2026236285782,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.88,35.68,1.6597537994384766
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",427,2026237985743,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.86,36.32,1.845811128616333
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",518,2026236285783,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.1,35.68,1.4986131191253662
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",102,2026239685469,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.42,35.98,3.478586196899414
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",428,2026237985744,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.73,35.04,1.8030011653900146
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",277,2026234585491,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,2.9825,16.215,3.4579155445098877
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",519,2026236285784,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.16,36.56,1.7183384895324707
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",429,2026237985745,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,1.0,1,0,5.4,36.02,1.910400152206421
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",520,2026236285785,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.04,36.04,1.6506290435791016
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",103,2026239685470,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.17,35.64,3.5572853088378906
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",430,2026237985746,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.52,36.16,1.8696222305297852
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",278,2026234585492,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0925,16.29,3.451612710952759
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",521,2026236285786,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.99,35.78,1.6268229484558105
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",522,2026236285787,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.65,34.48,1.557511806488037
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",431,2026237985747,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.46,34.94,1.8045194149017334
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",523,2026236285788,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.89,34.74,1.6914098262786865
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",104,2026239685471,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.45,35.52,3.788013458251953
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",279,2026234585493,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.235,15.75,3.547057628631592
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",432,2026237985748,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.78,34.96,1.8775451183319092
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",524,2026236285789,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.8,36.02,1.6715586185455322
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",433,2026237985749,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.75,35.64,1.9653651714324951
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",525,2026236285790,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.27,36.16,1.5569140911102295
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",280,2026234585494,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.9975,16.035,3.4138827323913574
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",105,2026239685472,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.18,36.38,3.6557164192199707
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",434,2026237985750,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.86,36.92,2.064629554748535
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",526,2026236285791,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.82,35.36,1.799255609512329
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",435,2026237985751,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.63,36.02,2.23294734954834
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",527,2026236285792,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.85,36.32,1.6670124530792236
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",281,2026234585495,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.085,15.8,3.862002372741699
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",106,2026239685473,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.53,37.02,3.888963460922241
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",528,2026236285793,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.42,36.14,1.6704778671264648
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",436,2026237985752,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.89,35.64,2.212853193283081
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",529,2026236285794,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.16,35.12,1.8146557807922363
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",282,2026234585496,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.975,15.92,3.3322174549102783
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",437,2026237985753,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.06,36.2,2.0679244995117188
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",107,2026239685474,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.94,36.56,3.379990577697754
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",530,2026236285795,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.97,35.82,1.8139894008636475
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",438,2026237985754,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.52,35.84,2.037808656692505
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",531,2026236285796,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.65,35.12,1.7217357158660889
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",283,2026234585497,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.5,1,1,3.0875,15.795,3.6802589893341064
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",108,2026239685475,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.84,36.82,4.056641578674316
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",439,2026237985755,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.96,36.64,2.3605780601501465
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",532,2026236285797,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.13,36.56,1.8644802570343018
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",440,2026237985756,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.61,36.48,1.9793450832366943
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",533,2026236285798,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.89,34.98,1.7841784954071045
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",284,2026234585498,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0225,16.155,3.7691776752471924
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",109,2026239685476,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.01,35.42,3.748504161834717
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",534,2026236285799,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.83,36.9,1.5291285514831543
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",441,2026237985757,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.91,35.8,2.003690719604492
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",535,2026236285800,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.46,35.04,1.5471272468566895
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",442,2026237985758,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,1.0,1,0,5.51,35.78,2.06614351272583
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",285,2026234585499,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0875,15.89,3.4080514907836914
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",536,2026236285801,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.73,34.84,1.483421802520752
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",110,2026239685477,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.15,37.08,3.470275640487671
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",443,2026237985759,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.52,36.3,1.9037106037139893
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",537,2026236285802,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.44,36.68,1.4653148651123047
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",538,2026236285803,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.92,36.28,1.6698071956634521
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",286,2026234585500,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0675,16.195,3.35009765625
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",444,2026237985760,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.11,34.96,2.0246481895446777
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",111,2026239685478,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.93,36.58,3.3479080200195312
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",539,2026236285804,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.07,36.92,1.8902454376220703
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",445,2026237985761,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.44,36.16,1.909816026687622
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",287,2026234585501,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,2.995,15.93,3.3779423236846924
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",540,2026236285805,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.86,35.5,1.7333259582519531
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",112,2026239685479,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.62,35.24,3.5882070064544678
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",446,2026237985762,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.52,36.1,2.0657429695129395
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",541,2026236285806,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.55,36.9,1.7330522537231445
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",447,2026237985763,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.41,36.1,1.9598462581634521
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",288,2026234585502,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.9875,16.03,3.6689915657043457
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",542,2026236285807,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.41,35.64,1.7891499996185303
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",113,2026239685480,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.93,36.58,3.7325398921966553
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",448,2026237985764,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.53,35.28,2.091144561767578
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",543,2026236285808,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,4.86,35.46,1.6696491241455078
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",449,2026237985765,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.71,36.74,2.1104135513305664
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",289,2026234585503,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0925,16.025,3.434633731842041
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",544,2026236285809,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.01,35.26,1.715463399887085
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",114,2026239685481,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.03,36.24,3.4873476028442383
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",450,2026237985766,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.5,1,1,5.92,36.04,1.9934520721435547
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",545,2026236285810,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,4.79,34.9,1.8377742767333984
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",546,2026236285811,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.7,36.12,1.5404741764068604
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",290,2026234585504,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.1275,16.045,3.592097520828247
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",451,2026237985767,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.45,35.42,1.9114596843719482
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",115,2026239685482,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.52,36.64,3.3107478618621826
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",547,2026236285812,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.86,36.28,1.443206787109375
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",452,2026237985768,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.85,35.22,2.0026402473449707
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",548,2026236285813,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.64,35.66,1.6205358505249023
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",291,2026234585505,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.1425,15.515,3.7085747718811035
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",116,2026239685483,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.46,35.1,3.1992318630218506
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",453,2026237985769,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.95,36.72,1.957223653793335
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",549,2026236285814,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.04,36.56,1.6234889030456543
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",454,2026237985770,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.56,35.88,2.028244972229004
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",550,2026236285815,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.83,36.0,1.6721322536468506
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",117,2026239685484,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.38,36.92,3.3455076217651367
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",292,2026234585506,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,2,2,3.1725,16.005,3.504878520965576
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",551,2026236285816,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.75,35.74,1.7189667224884033
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",455,2026237985771,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.69,35.86,2.1569039821624756
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",552,2026236285817,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.37,35.7,1.7892088890075684
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",456,2026237985772,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.03,35.22,1.8704736232757568
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",118,2026239685485,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.3,35.84,3.2325854301452637
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",293,2026234585507,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.1625,16.215,3.665325403213501
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",553,2026236285818,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.72,36.22,1.6586177349090576
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",457,2026237985773,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.86,36.26,1.8818857669830322
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",554,2026236285819,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.85,35.48,1.598576545715332
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",119,2026239685486,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.41,35.96,3.093986749649048
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",458,2026237985774,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.67,36.28,1.821396827697754
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",294,2026234585508,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.06,15.49,3.6150925159454346
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",555,2026236285820,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,5.19,36.58,1.6083424091339111
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",459,2026237985775,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.44,35.1,1.8550755977630615
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",556,2026236285821,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.74,36.54,1.6133720874786377
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",120,2026239685487,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.99,35.84,3.2210230827331543
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",460,2026237985776,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.9,36.3,2.09409236907959
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",557,2026236285822,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.82,36.2,1.701188087463379
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",295,2026234585509,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,3.07,16.04,3.5491905212402344
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",558,2026236285823,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.5,1,1,5.02,37.14,1.7521271705627441
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",121,2026239685488,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.35,36.18,3.182530164718628
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",461,2026237985777,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.72,35.4,2.076580286026001
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",559,2026236285824,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.37,35.98,1.7019679546356201
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",296,2026234585510,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.07,16.25,3.437915802001953
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",462,2026237985778,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.91,35.3,1.9517638683319092
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",560,2026236285825,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.23,36.0,1.6568412780761719
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",122,2026239685489,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.25,35.86,3.2595582008361816
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",463,2026237985779,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.67,36.76,1.9707093238830566
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",561,2026236285826,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,4.97,35.8,1.6519958972930908
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",297,2026234585511,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,3.015,16.25,3.3910346031188965
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",464,2026237985780,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,2.0,2,0,5.83,36.78,1.8769538402557373
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",562,2026236285827,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.15,36.82,1.7066020965576172
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",123,2026239685490,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.2,36.64,3.423030376434326
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",465,2026237985781,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.05,35.84,2.052673101425171
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",563,2026236285828,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.86,35.54,1.512814998626709
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",298,2026234585512,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.9875,15.8,3.370638608932495
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",124,2026239685491,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.83,35.8,3.0673301219940186
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",564,2026236285829,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.4,36.82,1.747020959854126
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",466,2026237985782,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.52,34.96,1.9723148345947266
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",565,2026236285830,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.47,35.24,1.677948236465454
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",467,2026237985783,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,1.0,1,0,5.5,35.26,1.9793355464935303
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",299,2026234585513,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0975,15.865,3.503335952758789
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",566,2026236285831,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.65,35.76,1.5988285541534424
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",125,2026239685492,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.04,35.94,3.472198486328125
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",468,2026237985784,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.19,35.88,2.124957799911499
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",567,2026236285832,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.09,37.1,1.697556495666504
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",300,2026234585514,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.5,1,1,3.0375,15.86,3.493887424468994
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",469,2026237985785,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.03,34.78,1.9488976001739502
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",568,2026236285833,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.07,35.5,1.7181482315063477
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",126,2026239685493,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.86,35.16,3.272279977798462
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",470,2026237985786,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.93,35.92,2.038573980331421
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",569,2026236285834,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.83,36.32,1.6060912609100342
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",301,2026234585515,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.94,16.38,3.5489370822906494
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",127,2026239685494,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.26,36.2,3.246882200241089
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",570,2026236285835,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.18,36.08,1.671015739440918
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",471,2026237985787,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.61,34.98,1.9742317199707031
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",571,2026236285836,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.5,1,1,5.04,35.96,1.5287072658538818
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",472,2026237985788,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.85,37.08,1.9662444591522217
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",302,2026234585516,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.8975,16.03,3.6086716651916504
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",128,2026239685495,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.18,35.82,3.223641872406006
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",572,2026236285837,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.82,36.1,1.6888079643249512
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",473,2026237985789,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.37,35.42,1.9977014064788818
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",573,2026236285838,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.13,36.58,1.661111831665039
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",474,2026237985790,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.29,35.64,1.965008020401001
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",303,2026234585517,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,2.93,16.27,3.556851387023926
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",129,2026239685496,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.29,36.28,3.2365190982818604
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",574,2026236285839,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.96,35.5,1.7143442630767822
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",475,2026237985791,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.49,36.16,2.019334554672241
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",575,2026236285840,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.93,35.86,1.6545953750610352
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",130,2026239685497,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.37,35.52,3.3876407146453857
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",576,2026236285841,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.9,35.74,1.5968878269195557
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",476,2026237985792,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.79,35.06,1.823880672454834
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",304,2026234585518,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.055,15.79,3.7501273155212402
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",577,2026236285842,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.93,35.86,1.713045358657837
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",477,2026237985793,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.66,35.5,2.0615737438201904
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",131,2026239685498,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.15,36.56,3.213409900665283
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",578,2026236285843,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.9,36.52,1.6154148578643799
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",305,2026234585519,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,2.5,3,1,3.12,15.76,3.5957462787628174
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",478,2026237985794,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.66,36.06,2.009441375732422
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",579,2026236285844,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.23,36.64,1.622157335281372
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",132,2026239685499,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.41,36.6,3.0620555877685547
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",479,2026237985795,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.74,35.76,2.15159010887146
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",580,2026236285845,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.85,35.8,1.6454315185546875
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",306,2026234585520,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.1525,15.97,3.5116281509399414
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",581,2026236285846,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.94,36.86,1.5620968341827393
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",480,2026237985796,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.67,36.4,2.0470402240753174
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",133,2026239685500,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.24,35.5,3.101860761642456
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",582,2026236285847,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,5.07,35.72,1.7844874858856201
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",481,2026237985797,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.42,36.68,1.943488359451294
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",307,2026234585521,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.045,15.54,3.248616933822632
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",583,2026236285848,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.07,35.44,1.8587865829467773
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",134,2026239685501,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.76,35.14,2.959169864654541
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",482,2026237985798,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.25,36.3,2.0934133529663086
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",584,2026236285849,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.65,35.06,1.7147963047027588
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",308,2026234585522,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,2,2,3.105,16.06,3.325432777404785
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",483,2026237985799,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.05,35.64,1.9922270774841309
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",585,2026236285850,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.48,36.38,1.7115561962127686
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",135,2026239685502,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.27,36.62,3.7586655616760254
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",484,2026237985800,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,1.0,1,0,5.49,35.36,1.9652996063232422
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",586,2026236285851,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.91,36.04,1.6447408199310303
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",309,2026234585523,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0025,16.0,3.5153305530548096
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",485,2026237985801,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.15,37.16,2.0429415702819824
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",587,2026236285852,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.74,36.46,1.719878911972046
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",136,2026239685503,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.22,36.44,3.161430597305298
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",588,2026236285853,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.01,35.56,1.7429983615875244
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",486,2026237985802,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.56,35.96,1.9323217868804932
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",310,2026234585524,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.2275,15.9,3.5618252754211426
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",589,2026236285854,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.59,36.74,1.8860304355621338
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",487,2026237985803,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.7,36.44,2.1038033962249756
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",137,2026239685504,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.5,1,1,7.75,36.76,3.6330153942108154
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",590,2026236285855,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.98,36.42,1.7230279445648193
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",311,2026234585525,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.5,1,1,2.9425,16.22,3.4569406509399414
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",488,2026237985804,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.45,36.2,2.0067098140716553
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",591,2026236285856,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.05,35.2,1.6667702198028564
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",138,2026239685505,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.44,35.62,3.5497395992279053
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",489,2026237985805,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,1.0,1,0,5.68,36.08,1.942061424255371
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",592,2026236285857,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.78,35.62,1.7885301113128662
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",312,2026234585526,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,2.0,3,2,3.1575,16.065,3.6454784870147705
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",490,2026237985806,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.71,36.04,1.7927031517028809
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",593,2026236285858,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.06,36.42,1.6801366806030273
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",139,2026239685506,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.74,35.76,3.6866207122802734
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",491,2026237985807,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.54,36.88,1.9677762985229492
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",594,2026236285859,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.04,36.76,1.5287518501281738
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",313,2026234585527,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.1775,16.01,3.4839510917663574
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",595,2026236285860,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.05,35.76,1.540722131729126
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",492,2026237985808,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.12,36.78,1.9819326400756836
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",140,2026239685507,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.73,36.26,3.115243434906006
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",596,2026236285861,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.93,35.28,1.6959137916564941
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",493,2026237985809,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,1.0,1,0,5.84,34.98,1.965888261795044
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",314,2026234585528,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.5,1,1,3.1625,15.99,3.535311460494995
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",597,2026236285862,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,5.06,36.12,1.671438455581665
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",494,2026237985810,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.97,35.12,1.939366340637207
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",141,2026239685508,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.81,35.2,3.2825803756713867
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",598,2026236285863,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.05,35.9,1.6915779113769531
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",495,2026237985811,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.67,36.9,2.0173604488372803
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",315,2026234585529,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.5,1,1,2.935,16.13,3.3973443508148193
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",599,2026236285864,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.96,35.24,1.695984125137329
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",142,2026239685509,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.5,1,1,7.86,35.54,3.273846387863159
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",496,2026237985812,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,1.0,1,0,5.84,35.98,2.1013376712799072
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",600,2026236285865,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.98,36.26,1.6400997638702393
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",316,2026234585530,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.1025,16.05,3.5543694496154785
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",497,2026237985813,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.36,35.7,1.9590721130371094
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",601,2026236285866,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.13,36.92,1.7178723812103271
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",143,2026239685510,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.9,35.1,3.484726905822754
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",602,2026236285867,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,5.26,36.74,1.6114113330841064
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",498,2026237985814,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,1.0,1,0,5.96,35.48,2.0808792114257812
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",317,2026234585531,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.005,16.02,3.4847676753997803
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",603,2026236285868,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.89,35.62,1.7075684070587158
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",499,2026237985815,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.5,1,1,5.27,36.24,1.961500883102417
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",144,2026239685511,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.29,35.68,3.5538485050201416
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",604,2026236285869,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.14,35.34,1.9333672523498535
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",500,2026237985816,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.88,34.9,1.9851398468017578
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",318,2026234585532,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.035,16.29,3.3136956691741943
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",605,2026236285870,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.98,36.82,1.6129441261291504
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",145,2026239685512,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.48,36.9,3.3717455863952637
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",501,2026237985817,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.95,36.06,2.0309855937957764
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",606,2026236285871,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,4.8,36.66,1.804847240447998
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",502,2026237985818,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.85,34.8,1.934577465057373
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",319,2026234585533,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0225,16.215,3.345102548599243
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",607,2026236285872,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.7,36.62,1.6721179485321045
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",146,2026239685513,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.61,36.16,3.4387855529785156
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",503,2026237985819,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.81,36.04,1.964597225189209
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",608,2026236285873,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.04,36.66,1.61458158493042
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",609,2026236285874,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.83,35.44,1.5678648948669434
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",320,2026234585534,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0575,15.8,3.6022861003875732
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",504,2026237985820,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.34,36.2,1.888824224472046
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",147,2026239685514,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.06,36.04,3.170407295227051
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",610,2026236285875,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,5.16,35.7,1.9277563095092773
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",505,2026237985821,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.78,35.52,1.9580333232879639
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",321,2026234585535,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,3.0,3,0,3.19,16.09,3.472714900970459
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",611,2026236285876,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.39,35.7,1.631943941116333
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",506,2026237985822,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.38,35.44,1.9592931270599365
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",148,2026239685515,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.68,35.74,3.174494743347168
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",612,2026236285877,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.33,35.98,1.7397186756134033
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",507,2026237985823,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.45,36.66,1.921790361404419
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",613,2026236285878,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,5.23,35.84,1.7220735549926758
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",322,2026234585536,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.165,16.24,3.546645402908325
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",149,2026239685516,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.05,36.54,3.4351625442504883
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",508,2026237985824,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.29,36.26,1.8813202381134033
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",614,2026236285879,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.64,36.38,1.638153314590454
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",509,2026237985825,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.58,36.0,2.071401357650757
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",615,2026236285880,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.0,34.88,1.780740737915039
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",323,2026234585537,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.5,2,1,3.0,15.87,3.5728883743286133
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",150,2026239685517,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.42,35.42,3.393052816390991
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",510,2026237985826,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.73,37.26,2.107052803039551
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",616,2026236285881,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.06,36.52,1.7433338165283203
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",617,2026236285882,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.82,35.52,1.5994501113891602
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",324,2026234585538,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.5,1,1,2.84,15.895,3.4366142749786377
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",511,2026237985827,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.53,35.52,1.9969558715820312
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",151,2026239685518,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,1.0,1,0,8.2,36.5,3.468956232070923
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",618,2026236285883,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.89,36.08,1.514833688735962
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",512,2026237985828,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.74,36.22,2.0929718017578125
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",619,2026236285884,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.05,36.26,1.5100724697113037
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",325,2026234585539,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,2.9325,16.21,3.351550579071045
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",152,2026239685519,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.86,34.74,3.6136386394500732
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",513,2026237985829,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.88,35.42,2.0781753063201904
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",620,2026236285885,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.04,37.1,1.4845249652862549
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",621,2026236285886,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.9,37.1,1.6351215839385986
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",514,2026237985830,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.96,36.44,2.165823221206665
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",326,2026234585540,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.5,1,1,3.005,15.645,3.3974735736846924
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",622,2026236285887,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.95,35.82,1.6894316673278809
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",153,2026239685520,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.26,35.88,3.8882555961608887
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",515,2026237985831,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.13,35.26,2.248553991317749
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",623,2026236285888,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.72,36.24,1.657202959060669
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",327,2026234585541,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.9825,15.995,3.6580698490142822
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",516,2026237985832,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.22,35.38,1.8350036144256592
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",624,2026236285889,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.16,35.4,1.5790455341339111
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",154,2026239685521,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.0,35.62,3.5644659996032715
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",625,2026236285890,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.85,35.44,1.542163372039795
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",517,2026237985833,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.96,35.86,2.1238996982574463
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",626,2026236285891,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.71,35.44,1.5942325592041016
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",328,2026234585542,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,3.01,16.08,3.7130115032196045
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",518,2026237985834,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,1.0,1,0,5.93,35.22,1.904869556427002
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",155,2026239685522,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.55,36.5,3.431523561477661
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",627,2026236285892,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.74,36.86,1.6425457000732422
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",519,2026237985835,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.78,36.04,1.8878929615020752
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",628,2026236285893,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.38,36.82,1.8542134761810303
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",329,2026234585543,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.14,15.995,3.564628839492798
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",156,2026239685523,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.19,35.86,3.4061834812164307
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",520,2026237985836,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.51,35.74,2.127298355102539
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",629,2026236285894,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,4.94,36.04,1.6564738750457764
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",521,2026237985837,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,1.0,1,0,5.23,35.36,1.9542028903961182
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",630,2026236285895,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.09,35.98,1.6008262634277344
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",330,2026234585544,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.16,15.935,3.3533477783203125
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",157,2026239685524,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.66,36.66,3.587754249572754
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",631,2026236285896,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.96,35.26,1.6037983894348145
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",522,2026237985838,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.43,37.06,2.022224187850952
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",632,2026236285897,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,4.8,36.58,1.384864091873169
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",331,2026234585545,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.07,15.885,3.383352041244507
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",523,2026237985839,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.65,35.6,1.9725403785705566
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",633,2026236285898,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.07,36.36,1.5661835670471191
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",158,2026239685525,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.49,35.48,3.5002243518829346
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",524,2026237985840,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.18,36.76,2.213430166244507
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",634,2026236285899,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.84,37.04,1.521650791168213
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",332,2026234585546,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.5,1,1,2.9375,16.155,3.204303026199341
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",635,2026236285900,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.03,36.84,1.6560399532318115
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",525,2026237985841,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.44,35.86,1.988227128982544
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",159,2026239685526,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.07,35.68,3.3866891860961914
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",636,2026236285901,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.91,36.52,1.517669677734375
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",333,2026234585547,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.905,16.075,3.2700188159942627
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",526,2026237985842,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.67,36.08,2.2376010417938232
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",637,2026236285902,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.12,36.62,1.4902875423431396
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",160,2026239685527,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.64,35.44,3.3531885147094727
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",527,2026237985843,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.72,35.6,1.954057216644287
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",638,2026236285903,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,4.69,37.42,1.564112663269043
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",334,2026234585548,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.1775,16.215,3.607159376144409
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",639,2026236285904,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.86,36.6,1.7006583213806152
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",528,2026237985844,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.5,1,1,5.84,35.3,1.972607135772705
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",161,2026239685528,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.04,34.94,3.202698230743408
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",640,2026236285905,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.73,35.36,1.8119966983795166
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",529,2026237985845,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.76,35.34,2.140491247177124
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",335,2026234585549,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,2.92,16.09,3.4552879333496094
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",641,2026236285906,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,4.92,35.04,1.727790117263794
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",162,2026239685529,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.45,36.84,3.2879347801208496
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",530,2026237985846,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.39,36.42,2.0183093547821045
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",642,2026236285907,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.86,36.04,1.8248169422149658
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",531,2026237985847,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.69,35.66,2.05291485786438
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",336,2026234585550,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.5,1,1,3.0925,16.12,3.25758695602417
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",643,2026236285908,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.58,36.08,1.5848627090454102
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",163,2026239685530,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.58,36.36,3.4340121746063232
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",532,2026237985848,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.02,37.14,1.852149486541748
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",644,2026236285909,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.1,36.56,1.6370949745178223
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",337,2026234585551,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.085,16.15,3.474238157272339
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",645,2026236285910,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.5,1,1,4.46,35.68,1.4288194179534912
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",533,2026237985849,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.93,35.94,2.042036533355713
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",164,2026239685531,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.55,35.5,3.2780892848968506
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",646,2026236285911,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.83,35.54,1.6544604301452637
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",534,2026237985850,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.15,35.72,1.9069771766662598
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",647,2026236285912,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.11,35.76,1.7123503684997559
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",338,2026234585552,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,3.135,15.75,3.653183698654175
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",165,2026239685532,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.62,35.48,3.4333415031433105
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",535,2026237985851,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.63,35.84,2.099989891052246
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",648,2026236285913,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.01,36.42,1.581117868423462
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",536,2026237985852,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.71,36.48,1.983656883239746
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",649,2026236285914,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.29,35.38,1.6237943172454834
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",339,2026234585553,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.0825,15.815,3.354309320449829
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",166,2026239685533,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.44,35.72,3.0623624324798584
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",537,2026237985853,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.27,34.8,1.834970474243164
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",650,2026236285915,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.52,35.08,1.4996111392974854
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",651,2026236285916,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.5,1,1,5.13,36.4,1.6323254108428955
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",538,2026237985854,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.76,36.52,2.030853509902954
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",167,2026239685534,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.5,1,1,8.57,35.72,3.2604947090148926
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",340,2026234585554,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,3.015,15.94,3.5336124897003174
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",652,2026236285917,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.64,35.54,1.523139238357544
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",539,2026237985855,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.85,36.34,2.1591193675994873
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",653,2026236285918,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.09,35.96,1.7421801090240479
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",168,2026239685535,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.58,36.7,3.407132625579834
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",341,2026234585555,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.065,15.915,3.552978515625
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",540,2026237985856,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.61,36.16,2.069387197494507
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",654,2026236285919,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.59,36.28,1.5910735130310059
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",655,2026236285920,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.32,36.42,1.4855587482452393
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",541,2026237985857,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.8,36.16,2.082721710205078
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",169,2026239685536,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.3,36.6,3.3086442947387695
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",656,2026236285921,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.64,34.56,1.660078525543213
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",342,2026234585556,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,1.0,1,0,3.21,16.08,3.7159011363983154
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",542,2026237985858,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.98,35.48,1.9622368812561035
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",657,2026236285922,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.06,35.4,1.5194292068481445
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",170,2026239685537,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.23,36.16,3.199183702468872
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",543,2026237985859,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.74,35.48,2.0387492179870605
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",658,2026236285923,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.49,35.28,1.5229272842407227
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",343,2026234585557,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.1725,16.105,3.4429996013641357
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",659,2026236285924,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.74,35.98,1.491412878036499
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",544,2026237985860,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.43,36.8,2.0446646213531494
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",171,2026239685538,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.31,34.58,3.387430191040039
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",660,2026236285925,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.04,37.16,1.585641622543335
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",545,2026237985861,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.94,35.78,1.9732897281646729
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",344,2026234585558,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.17,16.19,3.529764175415039
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",661,2026236285926,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.36,35.8,1.7115895748138428
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",546,2026237985862,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.28,36.16,1.831559419631958
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",172,2026239685539,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.77,36.3,3.1293230056762695
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",662,2026236285927,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.17,36.3,1.5711274147033691
production-20261029T053925Z-29f79c,"B|noisy|d5|p0.0002|pi0|k1,1.5,1",345,2026234585559,B,noisy,5,0.0002,0.0,1.0,1.5,1.0,5,8,5,400,0.0,0,0,3.1075,15.955,3.304900884628296
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",547,2026237985863,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.69,36.9,1.8327763080596924
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",663,2026236285928,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.79,36.76,1.5242938995361328
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.001|pi0.001|k1,1,1",0,2026241385418,A,quiet,3,0.001,0.001,1.0,1.0,1.0,3,3,3,400,32.0,32,0,7.6825,3.91,1.255563497543335
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",664,2026236285929,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.1,36.32,1.505359172821045
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",548,2026237985864,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.64,36.64,2.096891403198242
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",173,2026239685540,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.56,34.96,3.492380380630493
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.001|pi0.001|k1,1,1",1,2026241385419,A,quiet,3,0.001,0.001,1.0,1.0,1.0,3,3,3,400,31.0,32,2,8.3475,3.96,1.3802859783172607
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",665,2026236285930,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.23,34.74,1.602735996246338
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",549,2026237985865,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.04,36.98,2.025336980819702
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.001|pi0.001|k1,1,1",2,2026241385420,A,quiet,3,0.001,0.001,1.0,1.0,1.0,3,3,3,400,32.5,34,3,8.3225,3.97,1.492396593093872
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",666,2026236285931,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.5,1,1,4.97,35.66,1.6253330707550049
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.001|pi0.001|k1,1,1",3,2026241385421,A,quiet,3,0.001,0.001,1.0,1.0,1.0,3,3,3,400,26.5,27,1,8.3475,3.92,1.337766170501709
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",174,2026239685541,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.8,35.36,3.698286294937134
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",550,2026237985866,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.65,36.04,2.02404522895813
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",667,2026236285932,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.2,36.54,1.554833173751831
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",551,2026237985867,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.5,1,1,5.62,36.4,1.990572452545166
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",668,2026236285933,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.69,34.64,1.5780270099639893
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",175,2026239685542,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.83,36.44,3.541224718093872
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",552,2026237985868,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.98,35.36,1.9369423389434814
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",669,2026236285934,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.85,35.56,1.6410706043243408
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.001|pi0.001|k1,1,1",0,2026243085469,A,quiet,5,0.001,0.001,1.0,1.0,1.0,5,5,5,400,26.5,27,1,40.4075,15.855,5.405729293823242
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",670,2026236285935,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.04,37.04,1.589573860168457
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",553,2026237985869,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,1.0,1,0,5.88,36.26,2.130659818649292
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",176,2026239685543,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.06,35.9,3.523879289627075
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",671,2026236285936,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,4.94,35.1,1.5577893257141113
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",554,2026237985870,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.37,35.64,2.0092527866363525
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",672,2026236285937,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.99,35.24,1.6267855167388916
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",555,2026237985871,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.47,36.88,2.0199267864227295
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",177,2026239685544,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.3,35.36,3.3814659118652344
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",673,2026236285938,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.44,35.86,1.5595340728759766
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.001|pi0.001|k1,1,1",1,2026243085470,A,quiet,5,0.001,0.001,1.0,1.0,1.0,5,5,5,400,29.0,30,2,40.98,15.62,5.5735557079315186
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",674,2026236285939,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.33,35.34,1.664252758026123
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",556,2026237985872,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.92,35.1,2.0131354331970215
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",178,2026239685545,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.56,35.92,3.142845869064331
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",675,2026236285940,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.93,36.18,1.4799189567565918
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",557,2026237985873,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.46,36.48,1.8805382251739502
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",676,2026236285941,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.18,36.54,1.5512959957122803
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",558,2026237985874,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.88,36.14,2.21114182472229
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",179,2026239685546,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.22,36.52,3.1203830242156982
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.001|pi0.001|k1,1,1",2,2026243085471,A,quiet,5,0.001,0.001,1.0,1.0,1.0,5,5,5,400,27.0,28,2,40.325,15.73,5.491424322128296
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",677,2026236285942,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.85,35.42,1.609952688217163
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",678,2026236285943,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.5,1,1,4.81,35.72,1.5835597515106201
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",559,2026237985875,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,1.0,1,0,6.19,33.7,2.225860357284546
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",679,2026236285944,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.83,34.82,1.6175520420074463
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",180,2026239685547,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.24,36.16,3.5349860191345215
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",560,2026237985876,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.21,36.74,2.123800754547119
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",680,2026236285945,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.47,34.62,1.5003256797790527
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.001|pi0.001|k1,1,1",3,2026243085472,A,quiet,5,0.001,0.001,1.0,1.0,1.0,5,5,5,400,32.0,33,2,40.305,16.11,5.580376863479614
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",561,2026237985877,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.95,35.58,1.9915895462036133
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",681,2026236285946,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.09,36.02,1.6428477764129639
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",181,2026239685548,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.76,35.12,3.505855083465576
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",562,2026237985878,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.22,36.86,1.759490728378296
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",682,2026236285947,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.97,35.5,1.5824918746948242
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",563,2026237985879,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.02,36.76,1.8949663639068604
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",683,2026236285948,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.17,35.9,1.7726078033447266
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",182,2026239685549,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.86,35.48,3.30682373046875
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",684,2026236285949,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.56,37.66,1.5307469367980957
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.001|pi0.001|k1,1,1",0,2026244785520,A,quiet,7,0.001,0.001,1.0,1.0,1.0,7,7,7,100,6.5,7,1,115.06,37.08,3.995452642440796
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",564,2026237985880,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.52,35.36,1.924225091934204
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",685,2026236285950,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.52,35.66,1.6005370616912842
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",565,2026237985881,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.81,35.52,2.0028858184814453
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",183,2026239685550,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.82,36.06,3.6183464527130127
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",686,2026236285951,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.0,35.56,1.586122751235962
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.001|pi0.001|k1,1,1",1,2026244785521,A,quiet,7,0.001,0.001,1.0,1.0,1.0,7,7,7,100,6.5,8,3,113.82,35.22,4.1523597240448
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",566,2026237985882,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.73,35.66,2.0631093978881836
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",687,2026236285952,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.03,35.98,1.5945146083831787
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",184,2026239685551,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,1.0,1,0,7.83,37.04,3.3514420986175537
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",688,2026236285953,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.83,35.68,1.6243395805358887
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",567,2026237985883,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.02,35.68,2.064136028289795
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",689,2026236285954,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.84,35.06,1.6114614009857178
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.001|pi0.001|k1,1,1",2,2026244785522,A,quiet,7,0.001,0.001,1.0,1.0,1.0,7,7,7,100,8.0,8,0,116.12,36.04,3.935666799545288
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",568,2026237985884,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.47,35.1,1.8903586864471436
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",185,2026239685552,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.86,36.54,3.4069628715515137
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",690,2026236285955,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.99,35.28,1.5592460632324219
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",569,2026237985885,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.85,36.36,2.2413768768310547
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",691,2026236285956,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.96,36.02,1.6827354431152344
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.001|pi0.001|k1,1,1",3,2026244785523,A,quiet,7,0.001,0.001,1.0,1.0,1.0,7,7,7,100,13.0,13,0,114.94,36.36,4.278435707092285
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",570,2026237985886,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.27,36.12,2.093216896057129
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",186,2026239685553,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.01,35.98,3.5375969409942627
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",692,2026236285957,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.89,36.62,1.7099628448486328
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",571,2026237985887,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.87,35.3,1.968846082687378
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",693,2026236285958,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.09,34.9,1.694535255432129
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",694,2026236285959,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.62,35.96,1.4533917903900146
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",187,2026239685554,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.45,35.64,3.575953722000122
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",572,2026237985888,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.5,1,1,5.74,35.3,1.9075536727905273
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.001|pi0.001|k1,1,1",4,2026244785524,A,quiet,7,0.001,0.001,1.0,1.0,1.0,7,7,7,100,5.0,5,0,113.63,34.24,4.450418472290039
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",695,2026236285960,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.41,36.52,1.4743938446044922
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",573,2026237985889,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.48,36.48,1.9782147407531738
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",696,2026236285961,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.06,35.84,1.5168206691741943
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",188,2026239685555,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.45,34.74,3.694202184677124
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",574,2026237985890,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.96,36.64,2.035696506500244
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",697,2026236285962,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.76,36.56,1.7514524459838867
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.001|pi0.001|k1,1,1",5,2026244785525,A,quiet,7,0.001,0.001,1.0,1.0,1.0,7,7,7,100,2.5,3,1,112.16,36.72,4.314778089523315
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",698,2026236285963,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.95,35.84,1.6915428638458252
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",575,2026237985891,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.71,35.32,2.121818780899048
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",189,2026239685556,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,1.0,1,0,7.67,35.76,3.4071359634399414
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",699,2026236285964,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.53,36.56,1.528350591659546
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",576,2026237985892,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.8,36.4,1.9404540061950684
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",700,2026236285965,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.25,36.92,1.5298922061920166
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.001|pi0.001|k1,1,1",6,2026244785526,A,quiet,7,0.001,0.001,1.0,1.0,1.0,7,7,7,100,7.5,8,1,114.2,36.32,4.357839107513428
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",577,2026237985893,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.61,36.38,1.980029821395874
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",701,2026236285966,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.59,35.86,1.5130252838134766
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",190,2026239685557,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.11,36.64,3.5686285495758057
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",702,2026236285967,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.57,36.14,1.4218931198120117
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",578,2026237985894,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.6,34.92,2.1020169258117676
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.001|pi0.001|k1,1,1",7,2026244785527,A,quiet,7,0.001,0.001,1.0,1.0,1.0,7,7,7,100,5.0,5,0,114.67,36.0,4.140870571136475
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",703,2026236285968,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.88,35.88,1.6489534378051758
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",191,2026239685558,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.23,35.56,3.634525775909424
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",579,2026237985895,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.2,36.18,2.094716787338257
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",704,2026236285969,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.76,35.46,1.7624342441558838
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",580,2026237985896,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.68,36.0,2.077221155166626
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",705,2026236285970,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.85,35.7,1.7222046852111816
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.001|pi0.001|k1,1,1",8,2026244785528,A,quiet,7,0.001,0.001,1.0,1.0,1.0,7,7,7,100,11.0,12,2,114.53,36.26,4.197889804840088
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",192,2026239685559,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.1,37.42,3.2887306213378906
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",581,2026237985897,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.94,35.94,2.1491479873657227
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",706,2026236285971,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,4.63,36.24,1.6486496925354004
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",707,2026236285972,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.26,36.82,1.792506217956543
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",582,2026237985898,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.5,1,1,5.62,36.08,2.1537065505981445
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",193,2026239685560,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.52,35.28,3.2840960025787354
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.001|pi0.001|k1,1,1",9,2026244785529,A,quiet,7,0.001,0.001,1.0,1.0,1.0,7,7,7,100,8.5,9,1,112.92,36.06,3.939683675765991
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",708,2026236285973,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,4.81,36.68,1.7186427116394043
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",583,2026237985899,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.61,36.36,1.922640323638916
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",709,2026236285974,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.17,35.4,1.7031784057617188
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",194,2026239685561,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.92,36.04,3.536250352859497
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",584,2026237985900,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,1.0,1,0,5.9,36.2,1.8509280681610107
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.001|pi0.001|k1,1,1",10,2026244785530,A,quiet,7,0.001,0.001,1.0,1.0,1.0,7,7,7,100,9.0,9,0,114.28,34.56,4.36649489402771
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",710,2026236285975,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.02,36.84,1.9134087562561035
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",585,2026237985901,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.6,36.1,1.785632610321045
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",711,2026236285976,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.18,36.56,1.7259199619293213
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",195,2026239685562,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.35,35.9,3.5602691173553467
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",586,2026237985902,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,1.0,1,0,6.1,36.58,1.9090440273284912
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",712,2026236285977,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,5.17,35.92,1.860978126525879
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",587,2026237985903,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.1,36.36,1.903970718383789
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.001|pi0.001|k1,1,1",11,2026244785531,A,quiet,7,0.001,0.001,1.0,1.0,1.0,7,7,7,100,4.5,5,1,115.4,35.26,5.052510499954224
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",713,2026236285978,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.78,36.0,1.6675634384155273
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",196,2026239685563,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.39,35.98,3.451371192932129
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",588,2026237985904,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.12,37.0,1.9755747318267822
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",714,2026236285979,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.05,35.06,1.743530035018921
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",589,2026237985905,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.39,35.84,1.9675664901733398
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",197,2026239685564,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.3,35.8,3.149879217147827
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",715,2026236285980,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.26,35.98,1.7135217189788818
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.001|pi0.001|k1,1,1",12,2026244785532,A,quiet,7,0.001,0.001,1.0,1.0,1.0,7,7,7,100,8.5,9,1,112.59,35.42,4.4574103355407715
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",590,2026237985906,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.01,37.18,2.07004714012146
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",716,2026236285981,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,4.84,36.32,1.7367596626281738
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",591,2026237985907,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.42,35.88,1.8960630893707275
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",717,2026236285982,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,1.0,1,0,4.75,36.16,1.6024820804595947
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",198,2026239685565,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.27,35.78,3.657688617706299
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",718,2026236285983,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.04,36.2,1.6091039180755615
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",592,2026237985908,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.03,36.18,1.9803242683410645
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.4,1",719,2026236285984,B,noisy,7,0.0002,0.0,1.0,0.4,1.0,7,3,7,100,0.0,0,0,5.1,34.32,1.6968448162078857
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",199,2026239685566,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.0,36.12,3.485037088394165
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",593,2026237985909,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.87,34.38,1.8579649925231934
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0005|pi0.0005|k1,1,1",0,2026248185622,A,quiet,3,0.0005,0.0005,1.0,1.0,1.0,3,3,3,400,4.5,5,1,3.8875,4.165,1.2549421787261963
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",594,2026237985910,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.84,36.38,1.9015660285949707
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0005|pi0.0005|k1,1,1",1,2026248185623,A,quiet,3,0.0005,0.0005,1.0,1.0,1.0,3,3,3,400,4.5,5,1,3.895,4.145,1.2723109722137451
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.001|pi0.001|k1,1,1",0,2026246485571,A,quiet,9,0.001,0.001,1.0,1.0,1.0,9,9,9,40,4.5,5,1,247.075,62.6,4.219998598098755
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",200,2026239685567,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.68,34.96,3.2580618858337402
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0005|pi0.0005|k1,1,1",2,2026248185624,A,quiet,3,0.0005,0.0005,1.0,1.0,1.0,3,3,3,400,9.0,9,0,4.085,3.915,1.2271227836608887
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",595,2026237985911,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,6.01,37.12,2.1179420948028564
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0005|pi0.0005|k1,1,1",3,2026248185625,A,quiet,3,0.0005,0.0005,1.0,1.0,1.0,3,3,3,400,5.0,5,0,4.0375,4.055,1.1070215702056885
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0005|pi0.0005|k1,1,1",4,2026248185626,A,quiet,3,0.0005,0.0005,1.0,1.0,1.0,3,3,3,400,6.0,6,0,3.9825,4.11,1.0878992080688477
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",596,2026237985912,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.56,36.1,1.8387956619262695
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",201,2026239685568,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.99,35.76,3.201362133026123
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.001|pi0.001|k1,1,1",1,2026246485572,A,quiet,9,0.001,0.001,1.0,1.0,1.0,9,9,9,40,1.5,2,1,244.7,63.55,3.973302125930786
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0005|pi0.0005|k1,1,1",5,2026248185627,A,quiet,3,0.0005,0.0005,1.0,1.0,1.0,3,3,3,400,3.0,3,0,3.9125,4.065,1.113675594329834
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,0.6,1",597,2026237985913,B,noisy,7,0.0002,0.0,1.0,0.6,1.0,7,5,7,100,0.0,0,0,5.66,36.4,2.084743022918701
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0005|pi0.0005|k1,1,1",6,2026248185628,A,quiet,3,0.0005,0.0005,1.0,1.0,1.0,3,3,3,400,10.0,10,0,4.04,4.06,1.3677423000335693
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0005|pi0.0005|k1,1,1",7,2026248185629,A,quiet,3,0.0005,0.0005,1.0,1.0,1.0,3,3,3,400,7.0,7,0,4.07,4.19,1.4613690376281738
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",202,2026239685569,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.14,36.64,3.7281785011291504
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0005|pi0.0005|k1,1,1",8,2026248185630,A,quiet,3,0.0005,0.0005,1.0,1.0,1.0,3,3,3,400,6.5,7,1,4.0,3.92,1.2128803730010986
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.001|pi0.001|k1,1,1",2,2026246485573,A,quiet,9,0.001,0.001,1.0,1.0,1.0,9,9,9,40,5.0,5,0,244.775,63.45,4.999445676803589
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0005|pi0.0005|k1,1,1",9,2026248185631,A,quiet,3,0.0005,0.0005,1.0,1.0,1.0,3,3,3,400,8.0,8,0,4.135,4.075,1.2875292301177979
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0005|pi0.0005|k1,1,1",10,2026248185632,A,quiet,3,0.0005,0.0005,1.0,1.0,1.0,3,3,3,400,5.0,5,0,4.1,3.95,1.2202084064483643
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",203,2026239685570,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.79,36.62,3.370361804962158
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0005|pi0.0005|k1,1,1",0,2026249885673,A,quiet,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,5.0,5,0,20.225,16.13,5.317366123199463
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0005|pi0.0005|k1,1,1",11,2026248185633,A,quiet,3,0.0005,0.0005,1.0,1.0,1.0,3,3,3,400,10.0,10,0,4.1425,4.05,1.2410190105438232
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0005|pi0.0005|k1,1,1",12,2026248185634,A,quiet,3,0.0005,0.0005,1.0,1.0,1.0,3,3,3,400,7.0,7,0,3.945,3.89,1.082613468170166
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.001|pi0.001|k1,1,1",3,2026246485574,A,quiet,9,0.001,0.001,1.0,1.0,1.0,9,9,9,40,5.0,5,0,251.0,63.65,4.3879804611206055
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",204,2026239685571,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.19,35.84,3.0769996643066406
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0005|pi0.0005|k1,1,1",13,2026248185635,A,quiet,3,0.0005,0.0005,1.0,1.0,1.0,3,3,3,400,6.0,6,0,3.8925,3.83,1.1445262432098389
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0005|pi0.0005|k1,1,1",14,2026248185636,A,quiet,3,0.0005,0.0005,1.0,1.0,1.0,3,3,3,400,2.0,2,0,4.21,4.11,1.25752592086792
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0005|pi0.0005|k1,1,1",15,2026248185637,A,quiet,3,0.0005,0.0005,1.0,1.0,1.0,3,3,3,400,6.0,6,0,4.1325,4.02,1.1821315288543701
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0005|pi0.0005|k1,1,1",1,2026249885674,A,quiet,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,2.0,2,0,19.9575,15.905,5.252014875411987
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",205,2026239685572,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.0,35.16,3.3723764419555664
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.001|pi0.001|k1,1,1",4,2026246485575,A,quiet,9,0.001,0.001,1.0,1.0,1.0,9,9,9,40,3.0,3,0,247.2,62.25,4.477006912231445
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",206,2026239685573,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.65,36.02,3.583143949508667
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.001|pi0.001|k1,1,1",5,2026246485576,A,quiet,9,0.001,0.001,1.0,1.0,1.0,9,9,9,40,3.0,3,0,242.8,62.85,4.252655267715454
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0005|pi0.0005|k1,1,1",2,2026249885675,A,quiet,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,3.0,3,0,20.24,15.88,5.4441773891448975
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",0,2026251585724,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.27,35.74,3.9024658203125
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",207,2026239685574,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.61,35.58,3.6524360179901123
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.001|pi0.001|k1,1,1",6,2026246485577,A,quiet,9,0.001,0.001,1.0,1.0,1.0,9,9,9,40,5.0,5,0,247.5,64.85,4.173346281051636
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",1,2026251585725,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,56.92,36.5,3.696484327316284
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0005|pi0.0005|k1,1,1",3,2026249885676,A,quiet,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,3.0,3,0,20.22,15.905,5.2648468017578125
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",208,2026239685575,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.06,35.1,3.3087263107299805
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",2,2026251585726,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.5,35.84,3.851053476333618
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.001|pi0.001|k1,1,1",7,2026246485578,A,quiet,9,0.001,0.001,1.0,1.0,1.0,9,9,9,40,4.0,4,0,250.325,64.1,4.311274766921997
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",209,2026239685576,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.17,37.18,3.2315170764923096
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0005|pi0.0005|k1,1,1",4,2026249885677,A,quiet,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,3.0,4,2,20.0575,15.72,5.4168925285339355
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",3,2026251585727,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.05,35.42,3.689220428466797
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",210,2026239685577,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.29,36.16,3.4335784912109375
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.001|pi0.001|k1,1,1",8,2026246485579,A,quiet,9,0.001,0.001,1.0,1.0,1.0,9,9,9,40,3.0,3,0,248.175,65.9,4.1482460498809814
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",4,2026251585728,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.91,35.36,3.6673977375030518
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",211,2026239685578,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.5,35.3,3.5195508003234863
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0005|pi0.0005|k1,1,1",5,2026249885678,A,quiet,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,1.0,1,0,20.185,15.61,5.3048012256622314
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.001|pi0.001|k1,1,1",9,2026246485580,A,quiet,9,0.001,0.001,1.0,1.0,1.0,9,9,9,40,3.0,3,0,248.55,63.2,3.9653282165527344
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",5,2026251585729,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.9,35.52,3.6573801040649414
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",212,2026239685579,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.32,36.56,3.493306875228882
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.001|pi0.001|k1,1,1",10,2026246485581,A,quiet,9,0.001,0.001,1.0,1.0,1.0,9,9,9,40,4.0,4,0,246.275,64.35,3.9760313034057617
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0005|pi0.0005|k1,1,1",6,2026249885679,A,quiet,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,0.0,0,0,20.4775,16.16,5.516738176345825
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",213,2026239685580,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.36,36.1,3.249098777770996
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",6,2026251585730,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,58.15,37.16,3.700955629348755
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.001|pi0.001|k1,1,1",11,2026246485582,A,quiet,9,0.001,0.001,1.0,1.0,1.0,9,9,9,40,3.0,3,0,246.525,65.95,3.959423542022705
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",214,2026239685581,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.04,35.44,3.3435299396514893
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",7,2026251585731,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,56.16,35.68,3.7788515090942383
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0005|pi0.0005|k1,1,1",7,2026249885680,A,quiet,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,1.5,2,1,20.275,16.045,5.344223499298096
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.001|pi0.001|k1,1,1",12,2026246485583,A,quiet,9,0.001,0.001,1.0,1.0,1.0,9,9,9,40,1.0,1,0,245.35,63.5,4.479056119918823
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",215,2026239685582,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.89,36.8,3.7295210361480713
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",8,2026251585732,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.5,1,1,56.76,36.2,3.7566277980804443
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0005|pi0.0005|k1,1,1",8,2026249885681,A,quiet,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,4.0,4,0,19.8575,15.995,5.310583114624023
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",216,2026239685583,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.09,35.86,3.085010528564453
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.001|pi0.001|k1,1,1",13,2026246485584,A,quiet,9,0.001,0.001,1.0,1.0,1.0,9,9,9,40,1.0,1,0,245.8,64.05,4.354550838470459
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",9,2026251585733,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.97,35.18,3.891451358795166
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",217,2026239685584,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.02,36.84,3.4959487915039062
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.001|pi0.001|k1,1,1",14,2026246485585,A,quiet,9,0.001,0.001,1.0,1.0,1.0,9,9,9,40,2.0,2,0,249.65,66.55,4.419156074523926
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0005|pi0.0005|k1,1,1",9,2026249885682,A,quiet,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,1.0,1,0,20.6925,16.235,5.437156677246094
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",10,2026251585734,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,59.09,35.5,3.8577539920806885
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",218,2026239685585,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.51,36.2,3.3853511810302734
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",11,2026251585735,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.5,2,1,56.85,34.84,3.8255696296691895
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.001|pi0.001|k1,1,1",15,2026246485586,A,quiet,9,0.001,0.001,1.0,1.0,1.0,9,9,9,40,6.5,7,1,243.675,62.85,4.320447683334351
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0005|pi0.0005|k1,1,1",10,2026249885683,A,quiet,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,1.0,1,0,19.775,16.02,5.28613018989563
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",219,2026239685586,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.74,36.84,3.310279607772827
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",12,2026251585736,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,56.42,36.64,3.966688394546509
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.001|pi0.001|k1,1,1",16,2026246485587,A,quiet,9,0.001,0.001,1.0,1.0,1.0,9,9,9,40,3.0,3,0,244.025,65.7,4.348395347595215
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",220,2026239685587,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.23,36.12,3.5240681171417236
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0005|pi0.0005|k1,1,1",11,2026249885684,A,quiet,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,3.5,4,1,20.335,15.88,5.341750144958496
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",13,2026251585737,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.5,1,1,55.95,36.36,3.815479040145874
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",221,2026239685588,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.16,36.14,3.670623540878296
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.001|pi0.001|k1,1,1",17,2026246485588,A,quiet,9,0.001,0.001,1.0,1.0,1.0,9,9,9,40,3.5,4,1,249.6,65.15,4.418375492095947
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",222,2026239685589,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.26,36.06,3.171729326248169
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",14,2026251585738,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,58.47,35.64,3.806320905685425
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0005|pi0.0005|k1,1,1",12,2026249885685,A,quiet,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,4.0,4,0,20.6175,16.285,5.2753212451934814
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.001|pi0.001|k1,1,1",18,2026246485589,A,quiet,9,0.001,0.001,1.0,1.0,1.0,9,9,9,40,3.0,3,0,247.425,64.5,4.325119256973267
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",223,2026239685590,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.23,36.44,3.3323237895965576
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",15,2026251585739,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,58.0,35.54,3.946378231048584
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0005|pi0.0005|k1,1,1",13,2026249885686,A,quiet,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,0.5,1,1,20.225,16.525,5.367832899093628
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.001|pi0.001|k1,1,1",19,2026246485590,A,quiet,9,0.001,0.001,1.0,1.0,1.0,9,9,9,40,7.0,7,0,248.225,64.55,4.25626540184021
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",224,2026239685591,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.95,36.62,3.5764920711517334
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",16,2026251585740,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.0,1,0,56.12,35.7,3.9016029834747314
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",225,2026239685592,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.15,36.88,3.081357717514038
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.001|pi0.001|k1,1,1",20,2026246485591,A,quiet,9,0.001,0.001,1.0,1.0,1.0,9,9,9,40,8.0,8,0,246.7,64.85,4.340848922729492
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0005|pi0.0005|k1,1,1",14,2026249885687,A,quiet,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,2.5,3,1,20.59,15.78,5.432461738586426
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",17,2026251585741,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,58.11,36.36,3.986967086791992
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",226,2026239685593,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.01,35.6,3.1101837158203125
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.001|pi0.001|k1,1,1",21,2026246485592,A,quiet,9,0.001,0.001,1.0,1.0,1.0,9,9,9,40,4.5,5,1,246.775,66.5,4.652388572692871
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",18,2026251585742,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,56.81,35.54,3.9643619060516357
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0005|pi0.0005|k1,1,1",15,2026249885688,A,quiet,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,5.5,6,1,20.3025,16.36,5.507394313812256
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",227,2026239685594,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.27,35.32,3.9260075092315674
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.001|pi0.001|k1,1,1",22,2026246485593,A,quiet,9,0.001,0.001,1.0,1.0,1.0,9,9,9,40,4.0,4,0,246.125,67.4,4.546398162841797
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",19,2026251585743,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.02,37.1,3.7124829292297363
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",228,2026239685595,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.2,35.66,3.714057207107544
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0005|pi0.0005|k1,1,1",16,2026249885689,A,quiet,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,2.0,3,2,20.865,15.975,5.1696085929870605
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",20,2026251585744,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.5,1,1,57.33,35.68,3.753708839416504
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.001|pi0.001|k1,1,1",23,2026246485594,A,quiet,9,0.001,0.001,1.0,1.0,1.0,9,9,9,40,0.0,0,0,243.875,65.4,4.268567800521851
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",229,2026239685596,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.01,35.34,3.312354564666748
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0005|pi0.0005|k1,1,1",17,2026249885690,A,quiet,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,2.5,3,1,20.335,15.91,5.272916555404663
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",21,2026251585745,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,58.16,35.18,3.8150761127471924
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",230,2026239685597,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.36,35.2,3.633286952972412
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.001|pi0.001|k1,1,1",24,2026246485595,A,quiet,9,0.001,0.001,1.0,1.0,1.0,9,9,9,40,3.0,3,0,248.6,64.35,4.289936304092407
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",22,2026251585746,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.0,1,0,57.78,36.42,3.7184031009674072
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",231,2026239685598,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.35,35.88,3.664476156234741
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0005|pi0.0005|k1,1,1",18,2026249885691,A,quiet,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,2.5,3,1,20.46,15.735,5.2510130405426025
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.001|pi0.001|k1,1,1",25,2026246485596,A,quiet,9,0.001,0.001,1.0,1.0,1.0,9,9,9,40,2.0,2,0,244.4,62.9,4.430378198623657
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",23,2026251585747,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.51,37.14,4.025593996047974
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",232,2026239685599,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.41,35.84,3.565338134765625
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.001|pi0.001|k1,1,1",26,2026246485597,A,quiet,9,0.001,0.001,1.0,1.0,1.0,9,9,9,40,2.0,2,0,242.975,64.95,4.622483015060425
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0005|pi0.0005|k1,1,1",19,2026249885692,A,quiet,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,4.0,4,0,20.21,15.865,5.394403457641602
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",233,2026239685600,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.57,36.38,3.7162492275238037
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",24,2026251585748,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,56.98,35.88,3.957105875015259
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.001|pi0.001|k1,1,1",27,2026246485598,A,quiet,9,0.001,0.001,1.0,1.0,1.0,9,9,9,40,4.0,4,0,244.0,64.65,4.520833969116211
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",234,2026239685601,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.02,35.22,3.2524514198303223
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",25,2026251585749,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.8,35.6,3.804044246673584
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0005|pi0.0005|k1,1,1",20,2026249885693,A,quiet,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,3.0,4,2,20.3825,16.34,5.408693552017212
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",235,2026239685602,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.35,35.0,3.607051372528076
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",26,2026251585750,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.5,2,1,56.06,35.88,3.7213950157165527
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0005|pi0.0005|k1,1,1",21,2026249885694,A,quiet,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,0.5,1,1,20.235,15.97,5.3099894523620605
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",236,2026239685603,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.27,36.1,3.540224075317383
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",27,2026251585751,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,2.0,2,0,56.49,37.2,3.811563730239868
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",0,2026253285775,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,125.7,63.0,4.603646993637085
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",237,2026239685604,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.08,36.26,3.7777106761932373
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0005|pi0.0005|k1,1,1",22,2026249885695,A,quiet,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,3.0,3,0,20.6275,15.965,5.579693794250488
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",28,2026251585752,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.5,1,1,56.73,35.92,3.9200704097747803
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",1,2026253285776,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,122.8,63.95,4.052194118499756
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",238,2026239685605,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.81,35.22,3.6842222213745117
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",29,2026251585753,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.5,34.9,3.879697322845459
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0005|pi0.0005|k1,1,1",23,2026249885696,A,quiet,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,5.0,5,0,20.1675,15.88,5.393385171890259
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",2,2026253285777,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,121.475,65.3,4.034940004348755
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",239,2026239685606,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.23,36.82,3.369584321975708
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",30,2026251585754,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.62,36.24,3.8125970363616943
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",240,2026239685607,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.58,35.34,3.13781476020813
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",3,2026253285778,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,126.5,65.9,4.279139280319214
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0005|pi0.0005|k1,1,1",24,2026249885697,A,quiet,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,2.0,2,0,20.6875,15.765,5.498764276504517
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",31,2026251585755,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.5,2,1,58.07,36.1,4.04196310043335
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",241,2026239685608,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.5,1,1,8.31,36.38,3.4390017986297607
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",4,2026253285779,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,123.4,63.5,3.7796671390533447
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",242,2026239685609,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.0,36.52,3.2756733894348145
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",32,2026251585756,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,54.43,36.0,3.94242262840271
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0005|pi0.0005|k1,1,1",25,2026249885698,A,quiet,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,2.0,2,0,20.445,15.78,5.4523608684539795
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",5,2026253285780,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,120.95,64.05,3.60280179977417
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",243,2026239685610,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.29,36.28,3.21630859375
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",33,2026251585757,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,58.34,35.24,3.854989528656006
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",6,2026253285781,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,128.25,62.9,4.368982791900635
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0005|pi0.0005|k1,1,1",26,2026249885699,A,quiet,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,2.0,2,0,20.4075,15.925,5.3751020431518555
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",244,2026239685611,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,1.0,1,0,8.36,36.32,3.5174055099487305
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",34,2026251585758,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,58.56,35.98,3.841704845428467
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",7,2026253285782,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,120.325,63.95,4.190539121627808
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",245,2026239685612,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,1.0,1,0,8.14,35.9,3.7019131183624268
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0005|pi0.0005|k1,1,1",27,2026249885700,A,quiet,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,2.5,3,1,20.625,15.7,5.497985124588013
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",35,2026251585759,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.36,36.14,3.873326539993286
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",246,2026239685613,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.48,36.18,3.7952582836151123
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",8,2026253285783,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,124.225,61.15,4.598423480987549
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",36,2026251585760,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,56.57,35.04,3.8857028484344482
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0005|pi0.0005|k1,1,1",28,2026249885701,A,quiet,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,2.0,2,0,20.69,16.025,5.371570825576782
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",247,2026239685614,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.26,35.56,3.7473678588867188
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",9,2026253285784,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,121.45,62.55,4.125291585922241
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",37,2026251585761,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,56.45,36.14,3.8551788330078125
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",248,2026239685615,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.19,35.84,3.487820625305176
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0005|pi0.0005|k1,1,1",29,2026249885702,A,quiet,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,0.0,0,0,20.295,16.035,5.527009963989258
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",10,2026253285785,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,125.225,62.5,4.397963523864746
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",38,2026251585762,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,56.78,35.16,3.817028760910034
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",249,2026239685616,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.5,1,1,8.36,36.64,3.752398729324341
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",39,2026251585763,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.0,1,0,56.44,36.38,3.8757026195526123
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",11,2026253285786,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,123.025,63.65,4.245453357696533
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0005|pi0.0005|k1,1,1",30,2026249885703,A,quiet,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,1.0,1,0,20.51,15.805,5.2320544719696045
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",250,2026239685617,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.21,35.6,3.642418384552002
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",40,2026251585764,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.0,1,0,58.09,36.2,3.7636077404022217
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",12,2026253285787,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,123.725,65.15,3.5989761352539062
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",251,2026239685618,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.94,35.54,3.3384618759155273
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0005|pi0.0005|k1,1,1",31,2026249885704,A,quiet,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,5.0,5,0,20.6025,15.975,5.268737554550171
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",41,2026251585765,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,2.0,2,0,58.01,36.22,3.746387004852295
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",13,2026253285788,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,123.775,63.6,3.7279412746429443
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",252,2026239685619,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,1.0,1,0,8.08,35.64,3.383650779724121
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0005|pi0.0005|k1,1,1",32,2026249885705,A,quiet,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,4.0,4,0,20.385,15.735,5.5157365798950195
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",42,2026251585766,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,56.37,35.5,3.789187431335449
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",253,2026239685620,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.81,36.04,3.288039445877075
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",14,2026253285789,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,122.15,66.15,4.719140529632568
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",43,2026251585767,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.95,35.7,3.870457410812378
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",254,2026239685621,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.44,37.78,3.7076125144958496
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0005|pi0.0005|k1,1,1",33,2026249885706,A,quiet,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,0.0,0,0,20.175,15.865,5.491065502166748
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",15,2026253285790,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,124.45,64.05,4.6494460105896
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",44,2026251585768,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,56.91,36.22,3.932886838912964
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",255,2026239685622,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.37,34.86,3.98700213432312
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",16,2026253285791,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,119.3,66.0,4.710836410522461
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0005|pi0.0005|k1,1,1",34,2026249885707,A,quiet,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,0.0,0,0,20.2675,16.04,5.59577751159668
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",256,2026239685623,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.0,35.54,3.324289560317993
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",45,2026251585769,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.5,1,1,57.84,35.98,4.0598273277282715
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",17,2026253285792,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,123.05,64.8,3.719816207885742
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",257,2026239685624,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.17,35.54,3.292055368423462
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",46,2026251585770,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.0,1,0,56.46,36.62,3.8367111682891846
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0005|pi0.0005|k1,1,1",35,2026249885708,A,quiet,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,4.0,4,0,20.235,15.985,5.446315050125122
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",18,2026253285793,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,123.4,63.75,3.933196544647217
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",258,2026239685625,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.24,36.44,3.6309385299682617
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",47,2026251585771,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,58.92,35.38,3.8414933681488037
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",259,2026239685626,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.83,36.52,3.080634593963623
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0005|pi0.0005|k1,1,1",36,2026249885709,A,quiet,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,3.0,3,0,20.4625,15.94,5.1755945682525635
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",19,2026253285794,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,123.225,65.4,4.567467451095581
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",48,2026251585772,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,58.45,36.16,3.8866658210754395
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",260,2026239685627,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.59,35.96,3.246586322784424
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",20,2026253285795,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,121.625,65.6,4.561504602432251
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0005|pi0.0005|k1,1,1",37,2026249885710,A,quiet,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,1.0,1,0,20.14,16.27,5.5235066413879395
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",49,2026251585773,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,58.18,37.36,4.040179491043091
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",261,2026239685628,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,1.0,1,0,8.47,36.28,3.328033924102783
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",262,2026239685629,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.91,36.56,3.149275779724121
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",50,2026251585774,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.0,1,0,57.51,36.5,3.8988237380981445
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",21,2026253285796,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,121.575,64.15,4.421386003494263
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0005|pi0.0005|k1,1,1",38,2026249885711,A,quiet,5,0.0005,0.0005,1.0,1.0,1.0,5,5,5,400,4.0,4,0,20.3125,16.32,5.452453136444092
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",0,2026254985826,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,3.0,3,0,2.3975,4.04,1.0920536518096924
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",263,2026239685630,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.24,36.62,3.2505762577056885
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",1,2026254985827,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,0.0,0,0,2.4425,4.095,1.0852890014648438
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",51,2026251585775,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.25,34.76,4.046701908111572
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",2,2026254985828,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,1.0,1,0,2.5775,4.02,1.0797626972198486
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",22,2026253285797,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,123.7,64.7,4.720662832260132
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",3,2026254985829,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,1.0,1,0,2.44,4.04,1.1259684562683105
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",264,2026239685631,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.13,36.16,3.503709554672241
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",4,2026254985830,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,3.0,3,0,2.415,4.15,1.0801639556884766
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",52,2026251585776,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.05,35.64,3.81994891166687
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",5,2026254985831,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,1.0,1,0,2.4875,3.965,1.1121551990509033
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",6,2026254985832,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,2.0,2,0,2.6325,3.96,1.180164098739624
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",23,2026253285798,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,121.05,63.15,4.455151796340942
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",7,2026254985833,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,3.0,3,0,2.405,3.995,1.1176705360412598
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",265,2026239685632,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.5,1,1,8.08,36.02,3.6784558296203613
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",8,2026254985834,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,1.0,1,0,2.5,3.965,1.1771447658538818
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",53,2026251585777,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.0,1,0,56.92,36.1,3.8975493907928467
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",9,2026254985835,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,2.0,2,0,2.3225,3.95,1.1152188777923584
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",24,2026253285799,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,125.4,63.75,4.191483736038208
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",10,2026254985836,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,3.0,3,0,2.39,3.96,1.1159493923187256
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",266,2026239685633,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.95,36.02,3.4935595989227295
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",11,2026254985837,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,1.0,1,0,2.4625,4.0,1.1459693908691406
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",54,2026251585778,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.38,35.88,3.8654205799102783
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",12,2026254985838,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,1.0,1,0,2.4075,3.83,1.1266119480133057
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",13,2026254985839,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,1.0,1,0,2.48,4.075,1.0501248836517334
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",267,2026239685634,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.22,36.34,3.3360812664031982
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",14,2026254985840,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,3.0,3,0,2.43,3.95,1.0102031230926514
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",25,2026253285800,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,124.325,63.75,4.441180229187012
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",15,2026254985841,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,2.0,2,0,2.4,3.89,1.0377295017242432
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",55,2026251585779,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.54,36.12,3.9392223358154297
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",16,2026254985842,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,2.0,2,0,2.5425,3.97,1.0286478996276855
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",268,2026239685635,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.03,36.42,3.255023717880249
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",17,2026254985843,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,3.0,3,0,2.445,4.05,1.0741727352142334
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",26,2026253285801,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,121.675,65.45,3.633960247039795
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",18,2026254985844,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,2.0,2,0,2.5125,3.96,1.0377089977264404
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",19,2026254985845,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,1.0,1,0,2.4175,3.905,0.9941668510437012
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",56,2026251585780,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.0,1,0,56.3,36.18,3.7449135780334473
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",269,2026239685636,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.39,35.38,3.442854166030884
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",20,2026254985846,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,5.0,5,0,2.4975,4.12,1.0794553756713867
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",27,2026253285802,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,125.7,63.35,3.6126527786254883
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",21,2026254985847,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,1.0,1,0,2.5225,3.975,1.1140496730804443
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",22,2026254985848,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,1.5,2,1,2.3825,4.095,1.1923573017120361
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",270,2026239685637,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,1.0,1,0,7.98,35.78,3.401066303253174
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",57,2026251585781,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.8,37.28,3.9829483032226562
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",23,2026254985849,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,2.0,2,0,2.5325,4.045,1.0998494625091553
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",24,2026254985850,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,1.0,1,0,2.4225,3.95,1.165374517440796
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",28,2026253285803,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,120.575,65.45,3.6845195293426514
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",25,2026254985851,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,2.0,2,0,2.485,4.1,1.0574369430541992
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",271,2026239685638,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.17,36.7,3.176107406616211
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",26,2026254985852,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,2.0,2,0,2.5625,4.025,1.0153617858886719
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",58,2026251585782,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.32,35.64,3.6732425689697266
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",27,2026254985853,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,3.0,3,0,2.505,3.89,1.0398623943328857
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",29,2026253285804,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,122.175,63.65,3.6046559810638428
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",28,2026254985854,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,2.0,2,0,2.505,3.96,1.0772721767425537
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",272,2026239685639,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.16,35.94,3.4235448837280273
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",29,2026254985855,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,1.0,1,0,2.4375,3.885,1.0941340923309326
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",59,2026251585783,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,56.91,35.7,3.7695751190185547
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",30,2026254985856,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,1.0,1,0,2.425,4.1,1.10670804977417
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",30,2026253285805,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,124.8,62.8,3.642495632171631
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",31,2026254985857,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,3.5,4,1,2.48,3.99,1.0457823276519775
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",32,2026254985858,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,3.0,3,0,2.53,4.175,1.0397138595581055
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",273,2026239685640,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.86,36.4,3.736227035522461
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",33,2026254985859,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,1.0,1,0,2.355,4.085,0.9938900470733643
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",60,2026251585784,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,58.46,36.34,3.7488903999328613
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",34,2026254985860,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,2.0,2,0,2.2975,3.93,1.035348892211914
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",31,2026253285806,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,125.825,63.2,3.6241235733032227
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",35,2026254985861,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,2.0,2,0,2.38,4.085,1.025064468383789
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",274,2026239685641,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.94,34.84,3.6027045249938965
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",36,2026254985862,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,1.0,1,0,2.675,4.005,1.0697989463806152
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",61,2026251585785,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.5,1,1,56.61,35.7,3.741594076156616
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",37,2026254985863,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,6.0,6,0,2.3425,4.05,1.0363304615020752
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",32,2026253285807,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,126.575,65.1,3.5776333808898926
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",38,2026254985864,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,1.0,1,0,2.475,3.905,1.0520069599151611
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",275,2026239685642,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.21,36.6,3.2557034492492676
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",39,2026254985865,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,1.0,1,0,2.48,3.94,1.1483819484710693
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",40,2026254985866,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,1.0,1,0,2.3275,3.865,1.116250991821289
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",62,2026251585786,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,55.7,37.34,3.9693801403045654
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",33,2026253285808,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,126.0,65.1,3.571732997894287
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",41,2026254985867,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,1.0,1,0,2.415,3.865,1.1698999404907227
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",276,2026239685643,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.6,36.46,3.6009721755981445
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",42,2026254985868,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,4.0,4,0,2.3375,3.995,1.0622882843017578
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",43,2026254985869,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,2.0,2,0,2.31,3.94,1.0411241054534912
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",63,2026251585787,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.01,35.24,3.7316548824310303
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",44,2026254985870,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,1.0,1,0,2.4025,4.065,1.0482468605041504
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",34,2026253285809,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,122.95,63.85,3.954224109649658
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",45,2026254985871,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,0.0,0,0,2.385,3.92,1.0181493759155273
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",277,2026239685644,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.97,36.06,3.4944260120391846
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",46,2026254985872,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,2.0,2,0,2.4825,4.125,1.0295331478118896
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",47,2026254985873,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,2.0,2,0,2.5225,4.075,1.0700223445892334
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",64,2026251585788,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,56.81,36.26,3.7539525032043457
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",48,2026254985874,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,1.0,1,0,2.45,4.145,1.019423246383667
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",35,2026253285810,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,123.25,60.45,4.106578350067139
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",278,2026239685645,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.3,35.28,3.591198444366455
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",49,2026254985875,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,1.0,1,0,2.4575,4.055,1.0164508819580078
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",50,2026254985876,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,0.0,0,0,2.5275,4.1,1.143916368484497
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",65,2026251585789,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.23,36.76,3.8697805404663086
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",51,2026254985877,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,3.0,3,0,2.4375,4.115,1.1138083934783936
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",279,2026239685646,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.39,35.26,3.326765775680542
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",52,2026254985878,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,1.0,1,0,2.335,4.135,1.0781970024108887
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",36,2026253285811,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,127.25,63.2,4.370246171951294
production-20261029T053925Z-29f79c,"A|quiet|d3|p0.0003|pi0.0003|k1,1,1",53,2026254985879,A,quiet,3,0.0003,0.0003,1.0,1.0,1.0,3,3,3,400,2.0,2,0,2.4675,4.195,1.192821979522705
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",66,2026251585790,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.26,35.46,3.9174654483795166
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",280,2026239685647,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.02,35.84,3.316722869873047
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",37,2026253285812,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,123.2,64.35,4.175464391708374
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",281,2026239685648,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.22,35.28,3.3399603366851807
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",67,2026251585791,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,56.43,35.76,4.001251697540283
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",0,2026256685877,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.0625,16.3,4.955570697784424
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",38,2026253285813,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,119.925,65.65,4.611668348312378
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",282,2026239685649,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.06,36.14,3.243619918823242
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",68,2026251585792,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.61,35.06,3.8908040523529053
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",1,2026256685878,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.5,1,1,12.1725,16.05,4.997389554977417
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",283,2026239685650,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.09,35.82,3.4650168418884277
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",39,2026253285814,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,123.2,65.2,4.768891096115112
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",69,2026251585793,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,58.81,35.78,4.010473966598511
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",284,2026239685651,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.03,36.14,3.241823196411133
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",2,2026256685879,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.32,16.13,4.823524713516235
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",70,2026251585794,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.0,1,0,57.19,36.14,3.8089654445648193
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",40,2026253285815,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,123.85,63.8,4.295865297317505
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",285,2026239685652,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.5,1,1,7.59,36.28,3.134838104248047
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",3,2026256685880,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.5,1,1,12.075,15.915,4.598156929016113
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",71,2026251585795,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.0,1,0,57.91,35.74,3.8548004627227783
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",286,2026239685653,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.84,36.34,3.277252435684204
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",41,2026253285816,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,122.25,64.85,4.351597309112549
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",287,2026239685654,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.22,35.58,3.236328125
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",4,2026256685881,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.5,1,1,12.0425,15.635,4.793456077575684
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",72,2026251585796,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.12,35.52,3.8462154865264893
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",42,2026253285817,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,120.725,63.7,3.7299082279205322
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",288,2026239685655,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.09,36.68,3.1669275760650635
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",73,2026251585797,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,56.01,35.94,3.8435487747192383
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",43,2026253285818,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,121.85,63.95,3.5695505142211914
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",5,2026256685882,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.3325,16.02,4.7663609981536865
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",289,2026239685656,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.94,35.76,3.172938585281372
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",44,2026253285819,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,121.825,62.15,3.637869119644165
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",74,2026251585798,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.42,35.16,3.885794162750244
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",290,2026239685657,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.9,37.2,3.2801921367645264
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",6,2026256685883,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.305,15.68,4.941816568374634
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",45,2026253285820,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,127.825,62.55,3.901549816131592
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",75,2026251585799,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,2.0,2,0,58.03,35.5,3.9748406410217285
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",291,2026239685658,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.84,35.18,3.655531406402588
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",7,2026256685884,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.1175,15.855,4.708232879638672
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",76,2026251585800,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.14,36.36,3.8957133293151855
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",46,2026253285821,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,121.975,63.05,4.331260681152344
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",292,2026239685659,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.83,36.26,3.474748134613037
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",77,2026251585801,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.5,1,1,55.09,34.8,3.8682093620300293
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",8,2026256685885,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.0,1,0,12.41,15.99,5.197628021240234
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",47,2026253285822,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,123.875,64.1,4.239353179931641
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",293,2026239685660,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.78,36.7,3.9434494972229004
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",78,2026251585802,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.75,36.1,3.8987040519714355
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",294,2026239685661,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.21,35.96,3.19864559173584
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",9,2026256685886,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.5,1,1,11.775,15.62,4.851337432861328
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",48,2026253285823,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,124.5,62.8,4.161957025527954
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",295,2026239685662,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.5,1,1,8.14,36.28,3.318798780441284
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",79,2026251585803,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.53,36.56,3.995090961456299
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",49,2026253285824,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,124.3,66.1,4.406179189682007
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",10,2026256685887,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.235,16.18,4.908402442932129
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",296,2026239685663,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.53,36.46,3.5562150478363037
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",80,2026251585804,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.44,36.98,3.9810431003570557
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",50,2026253285825,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,122.775,63.8,4.599281072616577
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",297,2026239685664,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.58,35.92,3.0657153129577637
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",11,2026256685888,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.0,1,0,11.915,16.155,4.964303493499756
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",81,2026251585805,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,56.51,35.92,3.815258502960205
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",298,2026239685665,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.85,36.24,3.2969236373901367
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",51,2026253285826,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,122.15,65.1,3.7131330966949463
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",12,2026256685889,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.0,1,0,12.145,15.955,4.928550958633423
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",82,2026251585806,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,58.29,35.94,4.212416648864746
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",299,2026239685666,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.04,34.4,3.2220702171325684
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",52,2026253285827,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,123.675,65.25,4.208435297012329
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",83,2026251585807,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,56.71,35.3,3.800241708755493
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",300,2026239685667,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.21,37.06,3.3334126472473145
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",13,2026256685890,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.0875,16.17,4.839764356613159
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",53,2026253285828,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,124.5,66.4,3.861787796020508
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",301,2026239685668,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.11,34.78,3.444288492202759
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",84,2026251585808,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.52,36.98,3.835858106613159
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",14,2026256685891,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,11.975,15.86,4.81022834777832
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",54,2026253285829,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,125.075,65.75,3.8301053047180176
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",302,2026239685669,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.57,35.92,3.368962049484253
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",85,2026251585809,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.31,35.88,3.7150232791900635
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",55,2026253285830,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,123.725,62.65,4.043554067611694
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",303,2026239685670,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,2.0,2,0,7.9,36.06,3.272320032119751
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",15,2026256685892,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.0,1,0,12.2525,16.14,4.918671131134033
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",86,2026251585810,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.75,35.86,3.724015712738037
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",56,2026253285831,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,124.7,64.4,3.607879877090454
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",304,2026239685671,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.67,35.78,3.1853580474853516
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",87,2026251585811,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.55,36.48,3.7580368518829346
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",16,2026256685893,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.0,1,0,12.41,16.185,4.83193826675415
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",305,2026239685672,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.15,36.7,3.398803949356079
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",57,2026253285832,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,125.3,64.3,4.073462009429932
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",88,2026251585812,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.5,1,1,55.63,36.3,3.8150391578674316
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",17,2026256685894,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.39,15.74,4.9840312004089355
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",306,2026239685673,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.5,1,1,8.09,36.76,3.420022964477539
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",58,2026253285833,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,120.0,63.15,4.569710969924927
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",89,2026251585813,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.5,1,1,58.66,36.42,3.8761708736419678
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",307,2026239685674,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,1.0,1,0,8.43,35.72,3.453984022140503
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",18,2026256685895,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.4425,16.1,4.799057245254517
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",59,2026253285834,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,124.175,64.95,4.192124605178833
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",90,2026251585814,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,56.97,37.46,3.866544723510742
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",308,2026239685675,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.15,36.56,3.503237009048462
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",60,2026253285835,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,122.325,62.75,3.539621353149414
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",19,2026256685896,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.0425,15.58,4.82079553604126
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",91,2026251585815,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,56.99,35.3,3.734058380126953
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",309,2026239685676,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.21,36.56,3.164480209350586
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",61,2026253285836,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,123.275,62.6,3.577751636505127
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",310,2026239685677,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,9.09,37.8,3.0931525230407715
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",92,2026251585816,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.63,36.42,3.6488587856292725
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",20,2026256685897,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.5,1,1,12.0925,15.715,4.936812162399292
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",311,2026239685678,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.01,37.0,3.117051362991333
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",62,2026253285837,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,124.55,64.75,3.674741744995117
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",93,2026251585817,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.64,35.36,3.96368145942688
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",21,2026256685898,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.13,15.83,5.012712478637695
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",312,2026239685679,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.54,35.98,3.3867642879486084
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",63,2026253285838,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,124.125,63.05,4.470686197280884
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",94,2026251585818,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.83,36.28,3.883584499359131
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",313,2026239685680,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.84,35.52,3.3198611736297607
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",22,2026256685899,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.19,16.215,4.8501296043396
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",64,2026253285839,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,124.25,64.05,3.812150239944458
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",95,2026251585819,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,56.55,36.02,3.9056296348571777
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",314,2026239685681,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.28,36.1,3.484304189682007
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",65,2026253285840,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,124.625,66.5,3.8722219467163086
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",23,2026256685900,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.0,1,0,12.33,16.125,4.9333014488220215
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",96,2026251585820,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.17,35.38,4.0382981300354
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",315,2026239685682,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.62,35.82,3.6303489208221436
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",66,2026253285841,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,126.55,64.75,4.129145622253418
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",97,2026251585821,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,56.45,35.7,3.8860366344451904
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",316,2026239685683,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.73,35.96,3.5355331897735596
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",24,2026256685901,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,2.0,2,0,12.4625,15.695,4.79207706451416
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",67,2026253285842,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,123.35,65.55,4.156112432479858
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",98,2026251585822,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,56.5,34.8,3.9529905319213867
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",317,2026239685684,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.42,37.38,3.5587120056152344
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",25,2026256685902,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.0475,16.28,4.901576519012451
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",318,2026239685685,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.37,36.16,3.622175455093384
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",99,2026251585823,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.0,1,0,57.16,35.42,3.7795424461364746
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",68,2026253285843,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,121.175,63.15,4.505866527557373
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",26,2026256685903,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,2.0,2,0,12.3525,15.995,4.793202638626099
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",319,2026239685686,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.89,35.22,3.4530856609344482
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",100,2026251585824,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.0,1,0,57.74,35.72,3.7402100563049316
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",69,2026253285844,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,122.65,66.1,4.481389760971069
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",320,2026239685687,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.3,36.16,3.4436488151550293
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",101,2026251585825,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,56.91,36.22,3.864600896835327
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",27,2026256685904,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.5,1,1,12.585,15.93,5.031829357147217
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",70,2026253285845,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,124.875,63.25,3.9010708332061768
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",321,2026239685688,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.22,36.76,3.2979187965393066
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",102,2026251585826,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,56.31,36.2,3.861888885498047
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",28,2026256685905,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.2775,15.97,5.13289213180542
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",71,2026253285846,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,120.1,65.55,4.12488055229187
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",322,2026239685689,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.54,35.72,3.5181350708007812
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",103,2026251585827,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,2.0,2,0,56.32,36.0,3.9800217151641846
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",323,2026239685690,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.86,36.22,3.582204580307007
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",72,2026253285847,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,125.2,64.85,4.530472278594971
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",29,2026256685906,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.2725,16.28,5.045179843902588
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",104,2026251585828,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,56.97,36.3,3.889202117919922
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",324,2026239685691,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.11,37.4,3.266653537750244
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",73,2026253285848,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,123.625,63.6,4.589518070220947
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",30,2026256685907,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,11.8975,15.985,4.839653730392456
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",105,2026251585829,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,56.63,35.76,3.825756788253784
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",325,2026239685692,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.36,35.22,3.488280773162842
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",74,2026253285849,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,119.25,64.85,4.371948480606079
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",106,2026251585830,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,56.45,35.74,3.9371089935302734
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",326,2026239685693,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.77,36.12,3.1702077388763428
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",31,2026256685908,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.195,15.865,4.854341268539429
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",327,2026239685694,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.57,35.74,3.1732051372528076
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",75,2026253285850,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,122.675,65.25,3.9646005630493164
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",107,2026251585831,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,55.06,36.16,3.925520896911621
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",32,2026256685909,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.085,15.785,4.712381839752197
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",328,2026239685695,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.01,36.6,3.3898141384124756
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",76,2026253285851,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,122.475,64.5,4.239683628082275
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",108,2026251585832,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,56.64,36.3,3.8604226112365723
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",33,2026256685910,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,11.995,15.74,4.786127090454102
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",329,2026239685696,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.88,36.2,3.7122514247894287
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",109,2026251585833,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.0,1,0,57.19,36.02,3.7002620697021484
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",77,2026253285852,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,125.975,65.6,4.116695404052734
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",330,2026239685697,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.46,36.1,3.800375461578369
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",34,2026256685911,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.0,1,0,12.38,15.865,4.8172926902771
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",110,2026251585834,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.15,34.92,3.8493525981903076
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",78,2026253285853,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,122.1,63.65,4.486764192581177
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",331,2026239685698,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.84,36.28,3.491184949874878
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",111,2026251585835,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.5,1,1,57.09,36.24,3.8663666248321533
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",35,2026256685912,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.57,15.99,4.769436359405518
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",79,2026253285854,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,121.575,63.4,3.862208604812622
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",332,2026239685699,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.19,34.52,3.3395235538482666
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",112,2026251585836,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,56.72,35.46,3.7625324726104736
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",80,2026253285855,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,121.875,62.65,3.666454553604126
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",36,2026256685913,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.0,1,0,11.9825,15.735,4.7071778774261475
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",333,2026239685700,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.5,1,1,8.27,36.42,3.434600830078125
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",113,2026251585837,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.24,36.26,3.8415842056274414
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",334,2026239685701,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.87,35.28,3.0374691486358643
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",81,2026253285856,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,121.45,64.2,3.7455999851226807
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",37,2026256685914,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.0,1,0,12.2825,16.42,4.780172348022461
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",114,2026251585838,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,2.5,3,1,57.47,36.18,3.7891528606414795
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",335,2026239685702,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.48,36.1,3.38948130607605
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",82,2026253285857,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,121.65,62.15,4.30916166305542
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",336,2026239685703,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.86,36.24,3.0738894939422607
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",38,2026256685915,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.0,1,0,12.295,15.96,4.980712890625
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",115,2026251585839,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.82,36.22,3.670304775238037
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",83,2026253285858,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,125.05,64.85,4.493366956710815
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",337,2026239685704,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.34,34.8,3.325181722640991
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",116,2026251585840,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.0,1,0,57.28,36.2,3.882192373275757
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",39,2026256685916,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,2.0,2,0,12.335,16.04,4.968048095703125
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",338,2026239685705,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.17,35.58,3.4108357429504395
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",84,2026253285859,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,123.7,65.25,4.43615460395813
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",117,2026251585841,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.31,35.8,3.8760929107666016
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",339,2026239685706,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.41,35.78,3.3707897663116455
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",40,2026256685917,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.545,16.1,4.933713674545288
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",85,2026253285860,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,121.825,62.55,4.374093055725098
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",118,2026251585842,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.17,36.44,3.756331443786621
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",340,2026239685707,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.5,1,1,8.03,35.76,3.395909309387207
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",41,2026256685918,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.24,16.07,4.802000522613525
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",86,2026253285861,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,124.675,66.1,3.935903787612915
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",119,2026251585843,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,56.67,35.74,3.776082992553711
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",341,2026239685708,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.7,36.78,3.4598639011383057
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",120,2026251585844,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,58.02,36.74,3.8198370933532715
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",87,2026253285862,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,124.9,63.5,4.372078895568848
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",42,2026256685919,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.5,1,1,12.175,16.185,4.899373769760132
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",342,2026239685709,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.09,36.0,3.3074183464050293
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",121,2026251585845,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,56.97,35.76,3.937044143676758
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",343,2026239685710,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.61,35.56,3.3785178661346436
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",88,2026253285863,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,120.425,65.8,4.227854251861572
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",43,2026256685920,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.1575,16.375,4.925332307815552
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",344,2026239685711,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.03,35.2,3.4065585136413574
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",122,2026251585846,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.31,35.52,3.8733279705047607
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",89,2026253285864,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,121.35,65.05,3.667013645172119
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",44,2026256685921,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.085,16.185,5.088076591491699
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",345,2026239685712,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.75,35.98,3.4060399532318115
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",123,2026251585847,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.0,1,0,56.49,35.66,3.915351390838623
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",90,2026253285865,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,121.4,63.8,4.554833889007568
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",346,2026239685713,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,7.9,35.16,3.437488079071045
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",45,2026256685922,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.0,1,0,12.1675,16.095,5.020786285400391
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",124,2026251585848,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.0,1,0,56.48,34.9,3.841081380844116
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",91,2026253285866,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,122.675,65.9,4.507981777191162
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",347,2026239685714,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.23,36.96,3.7599823474884033
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",125,2026251585849,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.5,1,1,55.01,35.52,3.9112157821655273
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",46,2026256685923,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,2.0,2,0,11.9925,15.71,4.872059106826782
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",348,2026239685715,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.16,35.7,3.4459760189056396
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",92,2026253285867,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,122.625,62.25,4.586621046066284
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",126,2026251585850,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.16,36.04,3.8118231296539307
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",47,2026256685924,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.0,1,0,12.2825,15.535,5.16489052772522
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",349,2026239685716,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.54,35.36,3.8739190101623535
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",93,2026253285868,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,121.55,63.1,4.32654881477356
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",127,2026251585851,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.5,1,1,57.5,36.02,3.8251378536224365
production-20261029T053925Z-29f79c,"B|noisy|d7|p0.0002|pi0|k1,1.5,1",350,2026239685717,B,noisy,7,0.0002,0.0,1.0,1.5,1.0,7,11,7,100,0.0,0,0,8.43,35.68,3.6874513626098633
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",48,2026256685925,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.0,1,0,12.2425,15.965,4.983658790588379
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",94,2026253285869,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,122.325,62.7,4.4657816886901855
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",128,2026251585852,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.87,36.4,3.7308499813079834
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",129,2026251585853,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.0,1,0,56.78,36.06,3.7773752212524414
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",95,2026253285870,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,122.65,62.05,4.217763900756836
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",0,2026258385928,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.49,36.32,4.159423112869263
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",49,2026256685926,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.0,1,0,12.4425,15.99,5.059116363525391
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",130,2026251585854,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,56.76,36.34,3.7923686504364014
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",96,2026253285871,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,125.5,64.05,4.438942193984985
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",1,2026258385929,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.32,36.08,3.8933448791503906
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",50,2026256685927,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.0,1,0,12.275,16.11,4.938873529434204
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",131,2026251585855,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,58.22,36.72,3.8914268016815186
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",2,2026258385930,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.5,1,1,33.88,36.24,3.9213709831237793
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",97,2026253285872,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,121.0,62.65,4.506681203842163
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",51,2026256685928,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.3175,16.02,5.08552098274231
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",132,2026251585856,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,58.11,35.36,3.82116436958313
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",3,2026258385931,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.79,36.38,3.7868587970733643
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",98,2026253285873,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,120.65,64.05,4.578648328781128
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",133,2026251585857,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.21,35.04,3.5962116718292236
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",52,2026256685929,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.5,2,1,12.2075,16.34,4.919363498687744
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",4,2026258385932,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.34,35.36,3.635542154312134
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",99,2026253285874,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,122.2,62.0,4.452620029449463
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",134,2026251585858,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,58.32,35.7,3.575594425201416
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",5,2026258385933,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.74,36.54,3.9662749767303467
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",53,2026256685930,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,11.905,16.01,4.800745964050293
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",135,2026251585859,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,56.13,36.08,3.6730425357818604
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",100,2026253285875,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,123.075,64.8,4.41606593132019
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",6,2026258385934,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.39,35.22,3.77687931060791
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",54,2026256685931,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.0,1,0,11.885,15.925,4.89797830581665
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",136,2026251585860,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.14,35.4,3.641810655593872
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",101,2026253285876,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,124.95,61.35,4.379157781600952
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",7,2026258385935,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.56,36.3,3.9177205562591553
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",137,2026251585861,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,2.0,2,0,56.48,36.78,3.6322238445281982
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",55,2026256685932,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.0,1,0,12.195,16.135,5.004056930541992
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",102,2026253285877,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,120.05,65.3,3.923109769821167
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",8,2026258385936,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.48,35.6,3.9407601356506348
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",138,2026251585862,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,56.97,36.52,3.77522611618042
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",103,2026253285878,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,123.325,61.6,3.606924057006836
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",9,2026258385937,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.5,1,1,35.45,36.78,3.612551212310791
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",56,2026256685933,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.0875,15.86,4.7907490730285645
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",139,2026251585863,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.27,35.12,3.6613900661468506
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",104,2026253285879,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,124.45,61.9,3.4720311164855957
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",10,2026258385938,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.9,36.92,3.5419821739196777
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",57,2026256685934,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.0,1,0,12.0775,16.25,4.943545579910278
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",140,2026251585864,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,56.28,37.0,3.664539098739624
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",105,2026253285880,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,122.1,64.15,3.5787909030914307
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",11,2026258385939,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.89,35.76,3.7621548175811768
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",141,2026251585865,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.09,36.88,3.740787982940674
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",106,2026253285881,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,125.5,64.95,3.4836246967315674
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",58,2026256685935,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.5,1,1,12.2125,15.815,4.938614130020142
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",12,2026258385940,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.03,35.58,3.98205304145813
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",142,2026251585866,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.0,1,0,58.62,34.94,3.6576716899871826
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",107,2026253285882,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,120.875,61.4,3.4830946922302246
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",59,2026256685936,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.145,15.975,4.770226716995239
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",13,2026258385941,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,35.44,35.94,3.9972777366638184
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",108,2026253285883,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,123.625,63.25,3.4534106254577637
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",143,2026251585867,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.0,1,0,57.25,36.08,3.8234806060791016
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",14,2026258385942,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.86,35.7,3.913466215133667
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",60,2026256685937,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.0,1,0,12.04,16.085,4.669321775436401
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",109,2026253285884,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,123.05,64.25,3.4300696849823
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",144,2026251585868,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,56.52,35.7,3.6218698024749756
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",15,2026258385943,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,35.12,34.12,3.9721975326538086
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",110,2026253285885,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,122.375,63.05,3.4637246131896973
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",61,2026256685938,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.5,1,1,12.315,15.48,4.593321800231934
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",145,2026251585869,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,56.33,36.44,3.6966614723205566
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",16,2026258385944,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.39,36.06,3.8003838062286377
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",111,2026253285886,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,121.225,63.45,3.764103889465332
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",146,2026251585870,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.0,1,0,58.08,35.5,3.9857265949249268
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",62,2026256685939,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.0,1,0,11.945,16.005,4.848400831222534
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",17,2026258385945,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,35.81,35.7,3.764248847961426
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",112,2026253285887,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,120.075,63.2,3.6782212257385254
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",147,2026251585871,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.89,35.7,3.7280075550079346
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",63,2026256685940,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.42,15.835,4.804420471191406
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",18,2026258385946,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.35,35.38,4.037880182266235
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",113,2026253285888,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,122.5,62.85,4.477647066116333
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",148,2026251585872,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,58.14,36.22,3.8417508602142334
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",64,2026256685941,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.3325,15.66,4.762694835662842
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",19,2026258385947,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.39,36.4,3.9900176525115967
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",149,2026251585873,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.0,1,0,56.92,35.28,3.7081546783447266
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",114,2026253285889,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,121.45,63.05,4.438894987106323
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",20,2026258385948,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.49,36.16,3.9540510177612305
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",150,2026251585874,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,58.17,36.14,3.653170108795166
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",65,2026256685942,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.285,15.875,4.744271516799927
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",115,2026253285890,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,124.3,65.3,4.414313316345215
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",21,2026258385949,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.96,35.96,3.9389455318450928
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",151,2026251585875,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.0,1,0,56.75,36.0,3.8244571685791016
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",116,2026253285891,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,121.425,64.75,3.6150999069213867
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",66,2026256685943,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,11.905,16.125,4.983380556106567
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",152,2026251585876,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.55,35.62,3.721891403198242
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",22,2026258385950,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.4,36.14,4.01527214050293
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",117,2026253285892,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,125.275,63.2,3.458759069442749
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",67,2026256685944,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.15,15.855,4.736544370651245
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",153,2026251585877,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.0,1,0,57.31,37.3,3.8431315422058105
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",23,2026258385951,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,32.57,36.18,3.906043291091919
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",118,2026253285893,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,124.625,64.75,3.840009927749634
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",68,2026256685945,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.37,16.14,4.9832048416137695
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",154,2026251585878,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,1.0,1,0,56.81,35.56,3.8003923892974854
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",24,2026258385952,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.81,35.2,3.590698480606079
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",119,2026253285894,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,122.75,65.95,3.5251693725585938
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",25,2026258385953,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.03,36.68,3.6289865970611572
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",155,2026251585879,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.92,35.86,3.775984525680542
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",120,2026253285895,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,123.6,64.95,3.526912212371826
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",69,2026256685946,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.0,1,0,12.5175,16.215,4.57651424407959
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",26,2026258385954,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.57,37.1,3.8281378746032715
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0005|pi0.0005|k1,1,1",156,2026251585880,A,quiet,7,0.0005,0.0005,1.0,1.0,1.0,7,7,7,100,0.0,0,0,57.62,35.1,3.838236093521118
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",121,2026253285896,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,125.175,67.2,4.274939298629761
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",70,2026256685947,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.3475,16.095,4.874911308288574
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",27,2026258385955,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.74,35.72,3.7419564723968506
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",122,2026253285897,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,123.6,64.5,4.303418874740601
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",71,2026256685948,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.4075,15.88,4.797527074813843
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",28,2026258385956,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.09,36.18,3.6664528846740723
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",123,2026253285898,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,122.375,65.25,3.843561887741089
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",0,2026260085979,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,74.95,64.8,4.3704304695129395
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",29,2026258385957,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.9,36.1,3.7506134510040283
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",72,2026256685949,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.07,16.2,4.960637807846069
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",124,2026253285899,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,121.9,65.5,3.8799946308135986
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",1,2026260085980,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,72.5,64.95,3.6314504146575928
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",30,2026258385958,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.92,35.4,3.5625643730163574
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",73,2026256685950,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.19,15.67,4.6595330238342285
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",125,2026253285900,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,123.025,64.9,3.581430673599243
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",2,2026260085981,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,75.725,66.35,3.582219362258911
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",31,2026258385959,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.47,36.1,3.8835947513580322
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",3,2026260085982,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,71.7,63.25,3.600998640060425
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",126,2026253285901,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,121.825,62.7,4.0875303745269775
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",74,2026256685951,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.0025,16.02,4.991053342819214
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",32,2026258385960,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.97,35.98,4.162268161773682
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",4,2026260085983,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,77.725,62.35,3.3866076469421387
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",127,2026253285902,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,123.525,64.55,3.979107141494751
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",75,2026256685952,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.0,1,0,12.02,15.81,5.0402233600616455
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",33,2026258385961,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.64,36.6,3.8798670768737793
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",5,2026260085984,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,73.175,63.5,3.6008083820343018
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",128,2026253285903,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,123.725,64.75,4.411701679229736
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",34,2026258385962,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.23,36.54,3.965182065963745
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",76,2026256685953,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.4775,16.355,4.94635272026062
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",6,2026260085985,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,70.65,66.55,4.063519239425659
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",129,2026253285904,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,122.175,62.9,3.9553232192993164
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",35,2026258385963,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.8,35.5,4.129299640655518
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",7,2026260085986,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,75.275,64.05,3.9859213829040527
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",77,2026256685954,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.195,16.145,5.036892414093018
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",130,2026253285905,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,122.025,63.25,4.500860691070557
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",36,2026258385964,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,35.19,36.5,3.8254952430725098
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",8,2026260085987,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,72.75,63.75,4.3334455490112305
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",78,2026256685955,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.37,16.14,4.881595611572266
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",131,2026253285906,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,125.975,64.25,4.376821279525757
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",37,2026258385965,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.56,35.42,3.8535852432250977
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",9,2026260085988,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,72.5,63.6,4.157315015792847
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",132,2026253285907,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,121.025,64.2,3.6684327125549316
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",79,2026256685956,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.5,1,1,12.535,16.08,4.916539669036865
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",38,2026258385966,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.5,1,1,34.58,36.66,3.9280335903167725
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",10,2026260085989,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,73.425,66.75,4.263415575027466
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",133,2026253285908,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,122.575,64.55,3.8566834926605225
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",39,2026258385967,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.28,35.54,4.044721364974976
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",80,2026256685957,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,11.9875,16.29,4.783539295196533
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",11,2026260085990,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,70.475,64.4,4.298836708068848
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",134,2026253285909,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,120.125,64.0,3.611124038696289
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",40,2026258385968,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.52,35.9,3.8849050998687744
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",81,2026256685958,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.25,16.335,4.950384140014648
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",135,2026253285910,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,123.425,62.85,3.613401174545288
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",12,2026260085991,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,75.1,65.55,4.339826345443726
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",41,2026258385969,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.73,35.96,3.773106336593628
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",136,2026253285911,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,123.125,63.3,3.888490676879883
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",82,2026256685959,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.365,16.185,4.983250141143799
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",13,2026260085992,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,72.325,63.3,4.214600324630737
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",42,2026258385970,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.45,36.04,3.8894383907318115
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",137,2026253285912,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,124.025,64.0,3.821682929992676
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",14,2026260085993,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,73.7,62.45,3.6659698486328125
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",83,2026256685960,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.0,1,0,12.4875,16.13,4.810563325881958
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",43,2026258385971,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.44,36.06,3.573103666305542
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",138,2026253285913,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,1.0,1,0,126.925,65.05,3.5891470909118652
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",15,2026260085994,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,73.775,63.9,3.5371780395507812
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",44,2026258385972,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,35.7,34.92,3.6852188110351562
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",84,2026256685961,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.1175,16.105,4.689007759094238
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",139,2026253285914,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,124.725,64.15,3.9542558193206787
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",16,2026260085995,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,73.85,65.9,3.9894540309906006
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",45,2026258385973,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.86,36.24,3.6119303703308105
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",85,2026256685962,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.12,16.405,4.786763668060303
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",140,2026253285915,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,119.55,61.15,3.7597463130950928
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",17,2026260085996,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,76.125,63.9,4.067550182342529
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",46,2026258385974,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.43,36.04,3.606017589569092
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",141,2026253285916,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,123.15,64.65,3.507997512817383
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",86,2026256685963,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.0,1,0,12.0275,15.88,4.773930072784424
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",47,2026258385975,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.89,36.86,3.5254757404327393
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",18,2026260085997,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,72.875,63.1,3.9887537956237793
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",142,2026253285917,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,124.0,64.65,4.339250802993774
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",48,2026258385976,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.82,36.6,3.768078088760376
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",19,2026260085998,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,72.8,65.0,3.9093315601348877
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",87,2026256685964,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,11.9275,15.8,4.829213380813599
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",49,2026258385977,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.32,36.14,3.6160056591033936
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",143,2026253285918,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,126.325,61.95,4.154194593429565
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",20,2026260085999,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,72.1,64.65,3.74395489692688
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",88,2026256685965,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,2.0,2,0,12.035,16.285,5.026570796966553
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",50,2026258385978,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.5,1,1,33.18,37.0,3.9155988693237305
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",21,2026260086000,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,75.75,63.75,4.0937535762786865
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",144,2026253285919,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,125.9,63.75,4.404521703720093
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",89,2026256685966,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,11.905,15.745,4.88492751121521
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",51,2026258385979,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.52,37.94,3.921461582183838
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",22,2026260086001,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,73.975,64.2,3.6427135467529297
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",145,2026253285920,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,121.75,65.1,4.338656902313232
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",52,2026258385980,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.97,35.4,3.938697099685669
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",90,2026256685967,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.1575,16.05,4.842466354370117
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",23,2026260086002,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,73.25,64.1,4.233516454696655
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",146,2026253285921,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,124.6,62.65,4.600343704223633
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",53,2026258385981,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.14,35.1,3.6451621055603027
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",91,2026256685968,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.3575,16.225,4.686616897583008
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",24,2026260086003,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,74.6,67.15,4.269824028015137
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0005|pi0.0005|k1,1,1",147,2026253285922,A,quiet,9,0.0005,0.0005,1.0,1.0,1.0,9,9,9,40,0.0,0,0,121.55,64.55,3.736628770828247
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",0,2026261786030,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.93,3.905,0.8098859786987305
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",54,2026258385982,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.0,36.4,3.568326234817505
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",1,2026261786031,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,3.0,3,0,1.93,4.095,0.8128173351287842
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",2,2026261786032,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,2.0275,3.99,0.9721558094024658
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",25,2026260086004,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,73.8,64.3,3.7500014305114746
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",3,2026261786033,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.955,3.81,0.8591709136962891
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",92,2026256685969,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.17,16.06,4.908284425735474
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",4,2026261786034,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,2.5,3,1,1.89,3.915,0.9578611850738525
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",55,2026258385983,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.75,35.9,3.8556809425354004
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",5,2026261786035,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,1.835,4.11,0.8056464195251465
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",6,2026261786036,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.935,4.09,0.8392705917358398
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",26,2026260086005,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,72.825,64.5,3.431487798690796
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",7,2026261786037,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,2.01,4.115,0.9907832145690918
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",8,2026261786038,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.8925,4.015,0.9081029891967773
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",93,2026256685970,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.5,1,1,12.2075,15.785,4.967578411102295
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",9,2026261786039,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.9425,4.11,0.8536787033081055
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",56,2026258385984,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.05,36.72,4.039813041687012
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",10,2026261786040,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.9275,3.835,0.8438193798065186
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",27,2026260086006,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,72.825,66.1,3.978487968444824
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",11,2026261786041,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.8475,4.16,0.9377291202545166
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",12,2026261786042,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.945,4.025,0.8770358562469482
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",13,2026261786043,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.8725,3.97,0.9554519653320312
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",57,2026258385985,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.08,36.74,3.9733026027679443
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",14,2026261786044,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,2.005,3.92,0.8454554080963135
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",94,2026256685971,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.165,16.265,4.904977321624756
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",15,2026261786045,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.8275,3.92,0.9029526710510254
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",28,2026260086007,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,74.375,65.55,3.916351318359375
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",16,2026261786046,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,1.5,2,1,1.81,3.995,0.8019888401031494
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",17,2026261786047,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.86,4.12,0.8436191082000732
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",18,2026261786048,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.9,4.185,0.940016508102417
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",58,2026258385986,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.93,36.04,4.0353429317474365
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",19,2026261786049,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.8375,3.985,0.9704134464263916
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",29,2026260086008,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,73.775,64.6,3.7005014419555664
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",20,2026261786050,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.9625,3.815,0.9184281826019287
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",95,2026256685972,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.5,1,1,12.605,16.14,5.002864360809326
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",21,2026261786051,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,1.95,4.085,0.9355373382568359
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",22,2026261786052,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.8975,4.08,0.9105300903320312
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",59,2026258385987,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.28,35.04,3.683351993560791
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",30,2026260086009,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,1.0,1,0,74.6,64.05,3.389387607574463
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",23,2026261786053,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.9075,3.88,0.910531759262085
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",24,2026261786054,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,1.5,2,1,2.02,3.9,1.0031459331512451
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",25,2026261786055,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.8475,4.165,0.988330602645874
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",96,2026256685973,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.0,1,0,11.9925,15.85,4.952581167221069
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",26,2026261786056,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,2.0425,3.91,1.0179853439331055
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",60,2026258385988,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.5,1,1,34.11,35.94,3.902489185333252
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",31,2026260086010,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,72.575,63.9,3.560926914215088
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",27,2026261786057,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.98,3.935,0.8692669868469238
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",28,2026261786058,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,2.025,3.82,0.8478634357452393
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",29,2026261786059,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.875,4.02,0.8229165077209473
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",30,2026261786060,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.81,4.005,0.8374731540679932
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",32,2026260086011,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,73.725,64.65,3.3464457988739014
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",97,2026256685974,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.28,16.12,4.768143892288208
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",61,2026258385989,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,34.21,35.54,3.6586155891418457
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",31,2026261786061,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.9125,4.065,0.8869690895080566
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",32,2026261786062,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,2.0275,4.15,0.8564627170562744
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",33,2026261786063,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.8575,3.855,0.797649621963501
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",34,2026261786064,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.92,3.905,0.8317718505859375
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",33,2026260086012,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,72.0,64.45,3.2728235721588135
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",35,2026261786065,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.9275,3.885,0.8445532321929932
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",62,2026258385990,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,35.17,36.7,3.725642681121826
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",36,2026261786066,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.89,3.785,0.8468239307403564
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",98,2026256685975,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.0,1,0,12.25,16.11,4.742077350616455
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",37,2026261786067,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.9125,3.975,0.9123966693878174
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",38,2026261786068,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.885,4.04,0.8342554569244385
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",34,2026260086013,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,74.625,63.9,3.842501640319824
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",39,2026261786069,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,1.99,4.025,0.864020586013794
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",63,2026258385991,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.42,35.78,3.77363920211792
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",40,2026261786070,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.9075,3.935,0.8650956153869629
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",41,2026261786071,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.8125,4.22,1.0476067066192627
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",99,2026256685976,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,11.9675,16.235,4.998723268508911
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",42,2026261786072,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.9725,4.04,0.9265923500061035
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",43,2026261786073,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.9025,4.095,0.8761661052703857
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",35,2026260086014,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,73.65,64.95,4.153888463973999
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",64,2026258385992,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.6,35.92,3.860504388809204
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",44,2026261786074,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,2.0475,3.885,0.9041478633880615
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",45,2026261786075,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.885,3.96,0.897150993347168
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",46,2026261786076,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.9225,4.125,0.8676126003265381
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",47,2026261786077,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.9225,4.17,0.908942461013794
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",100,2026256685977,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.0,1,0,12.03,16.105,4.819295883178711
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",36,2026260086015,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,74.375,65.75,4.137427091598511
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",48,2026261786078,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.9875,4.07,0.8966166973114014
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",65,2026258385993,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.9,35.84,4.026479721069336
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",49,2026261786079,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.87,3.865,0.8808844089508057
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",50,2026261786080,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.91,4.01,0.8912355899810791
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",51,2026261786081,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.92,4.04,0.8881256580352783
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",52,2026261786082,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.9075,4.015,0.8901524543762207
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",37,2026260086016,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,73.425,65.75,4.112311363220215
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",101,2026256685978,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.1825,16.225,4.845708131790161
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",66,2026258385994,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.35,36.48,4.183195352554321
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",53,2026261786083,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.8775,4.055,0.8650188446044922
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",54,2026261786084,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,2.0825,3.985,0.8908367156982422
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",55,2026261786085,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.8675,4.095,0.8768947124481201
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",56,2026261786086,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,3.0,3,0,1.9625,3.91,0.8753721714019775
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",57,2026261786087,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.9175,3.78,0.8895969390869141
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",38,2026260086017,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,73.675,63.9,4.128835201263428
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",67,2026258385995,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.41,36.02,4.000994443893433
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",102,2026256685979,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.2875,16.025,4.5475547313690186
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",58,2026261786088,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,2.1375,3.9,0.893364667892456
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",59,2026261786089,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.8075,4.215,0.8754277229309082
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",60,2026261786090,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.92,4.155,0.9131689071655273
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",61,2026261786091,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.9525,3.985,0.9300601482391357
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",39,2026260086018,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,73.2,63.7,3.652559280395508
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",68,2026258385996,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.14,36.7,3.9246253967285156
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",62,2026261786092,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.86,4.015,0.938624382019043
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",103,2026256685980,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.0,1,0,12.2,15.905,4.73453688621521
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",63,2026261786093,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,2.035,3.945,0.948338508605957
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",64,2026261786094,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.9375,4.075,0.9208352565765381
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",65,2026261786095,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.8225,4.01,0.8776252269744873
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",40,2026260086019,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,75.85,64.35,3.687856674194336
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",69,2026258385997,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.17,35.86,3.8475992679595947
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",66,2026261786096,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.8975,4.05,0.8810586929321289
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",67,2026261786097,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.91,3.93,0.8855044841766357
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",68,2026261786098,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.87,4.075,0.8878898620605469
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",104,2026256685981,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.27,16.065,4.737051725387573
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",69,2026261786099,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.9075,3.88,0.9680917263031006
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",41,2026260086020,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,74.45,63.95,3.6410882472991943
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",70,2026261786100,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,2.0325,3.955,0.8809232711791992
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",70,2026258385998,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.32,35.66,3.6515891551971436
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",71,2026261786101,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.8975,4.04,0.8208224773406982
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",72,2026261786102,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,2.02,4.135,0.8968226909637451
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",73,2026261786103,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.92,4.095,0.8168745040893555
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",105,2026256685982,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.0,1,0,12.1625,15.655,4.729154825210571
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",42,2026260086021,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,73.275,64.25,3.7300126552581787
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",74,2026261786104,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.915,3.97,0.8243656158447266
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",71,2026258385999,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,35.16,35.94,3.568551778793335
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",75,2026261786105,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.92,4.085,0.8300247192382812
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",76,2026261786106,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.93,3.995,0.8736538887023926
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",77,2026261786107,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.84,4.045,0.8293600082397461
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",78,2026261786108,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.8975,3.975,0.8252663612365723
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",43,2026260086022,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,69.75,66.3,4.333420753479004
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",72,2026258386000,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,35.57,36.42,3.9052670001983643
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",79,2026261786109,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.8775,3.99,0.9111247062683105
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",106,2026256685983,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.14,15.86,4.846308708190918
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",80,2026261786110,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.935,3.745,0.9083635807037354
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",81,2026261786111,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.9525,4.04,0.8975207805633545
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",82,2026261786112,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,1.5,2,1,1.8825,3.93,0.9641008377075195
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",83,2026261786113,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,2.0,4.14,0.9710032939910889
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",73,2026258386001,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.98,35.62,4.049725532531738
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",44,2026260086023,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,72.8,66.0,4.376605987548828
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",84,2026261786114,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.9625,4.245,0.8921704292297363
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",107,2026256685984,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.28,15.915,5.078036308288574
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",85,2026261786115,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.9725,4.165,0.9020111560821533
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",86,2026261786116,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.9525,4.105,0.940899133682251
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",87,2026261786117,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.9325,3.995,0.9027230739593506
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",74,2026258386002,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.7,37.08,3.8742525577545166
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",88,2026261786118,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.975,4.16,0.8946843147277832
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",45,2026260086024,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,73.225,60.35,4.066962242126465
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",89,2026261786119,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.8075,3.925,0.8393621444702148
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",90,2026261786120,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.72,3.965,0.8717262744903564
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",108,2026256685985,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.23,16.75,5.008713483810425
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",91,2026261786121,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.925,4.095,0.8676974773406982
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",75,2026258386003,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.47,35.34,3.735607624053955
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",92,2026261786122,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.935,3.97,0.9239826202392578
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",46,2026260086025,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,73.15,65.55,4.086642742156982
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",93,2026261786123,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.9475,3.94,0.8707339763641357
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",94,2026261786124,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.905,3.97,0.9418444633483887
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",95,2026261786125,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.945,3.97,0.9312570095062256
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",109,2026256685986,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.0,1,0,12.475,16.365,4.780324935913086
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",76,2026258386004,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.72,35.6,3.9296159744262695
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",96,2026261786126,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.965,3.88,0.928403377532959
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",97,2026261786127,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.85,4.155,0.8989832401275635
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",47,2026260086026,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,72.775,63.4,4.0558180809021
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",98,2026261786128,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.94,3.91,0.9450464248657227
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",99,2026261786129,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.97,3.995,0.8336448669433594
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",100,2026261786130,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.88,4.07,0.8151233196258545
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",77,2026258386005,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.7,37.2,3.902916193008423
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",101,2026261786131,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.93,4.005,0.8634777069091797
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",110,2026256685987,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.2825,15.965,4.94822096824646
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",48,2026260086027,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,73.825,64.9,4.0560290813446045
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",102,2026261786132,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.985,3.89,0.877655029296875
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",103,2026261786133,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.93,3.925,0.8161273002624512
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",104,2026261786134,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.9925,3.89,0.9138922691345215
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",78,2026258386006,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.55,35.64,3.917191743850708
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",105,2026261786135,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.985,3.94,0.8983595371246338
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",106,2026261786136,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,2.0225,3.995,0.8592383861541748
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",49,2026260086028,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,72.325,62.4,3.8876566886901855
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",111,2026256685988,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.23,15.66,4.791445970535278
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",107,2026261786137,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.88,3.925,0.9131643772125244
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",108,2026261786138,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.885,3.975,0.8524324893951416
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",109,2026261786139,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.8725,4.0,0.9588899612426758
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",79,2026258386007,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,35.0,36.1,3.935354709625244
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",110,2026261786140,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.875,4.02,0.8985300064086914
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",50,2026260086029,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,72.875,64.55,3.725794792175293
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",111,2026261786141,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,2.0375,3.91,0.8447525501251221
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",112,2026261786142,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.8625,4.075,0.8482625484466553
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",112,2026256685989,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.225,15.9,4.667361736297607
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",113,2026261786143,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.985,4.05,0.823549747467041
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",80,2026258386008,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.51,35.86,3.9353232383728027
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",114,2026261786144,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.815,4.065,0.8257474899291992
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",51,2026260086030,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.5,1,1,74.775,63.4,3.8026740550994873
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",115,2026261786145,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.955,4.09,0.9821262359619141
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",116,2026261786146,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.7775,4.08,0.9014546871185303
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",117,2026261786147,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.895,4.08,0.9385240077972412
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",113,2026256685990,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.2225,16.01,4.9859514236450195
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",118,2026261786148,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.89,4.16,0.9284358024597168
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",81,2026258386009,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.96,36.3,4.1591796875
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",52,2026260086031,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,74.775,63.4,3.831063985824585
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",119,2026261786149,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.8575,4.03,0.9509556293487549
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",120,2026261786150,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.8475,4.03,0.8764607906341553
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",121,2026261786151,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,2.005,3.965,0.8941097259521484
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",122,2026261786152,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,3.0,3,0,2.1325,3.915,0.937999963760376
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",53,2026260086032,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,74.0,62.55,3.356219530105591
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",82,2026258386010,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.66,35.5,4.013505220413208
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",114,2026256685991,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.5,2,1,11.965,16.06,4.803884029388428
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",123,2026261786153,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,2.005,4.005,1.0042638778686523
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",124,2026261786154,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.8575,3.93,0.8996396064758301
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",125,2026261786155,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,1.975,3.95,0.910292387008667
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",126,2026261786156,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.8575,4.01,0.8992190361022949
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",54,2026260086033,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.5,1,1,74.0,61.9,3.416844129562378
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",83,2026258386011,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.8,35.62,3.859283447265625
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.001|pi0|k1,1,1",127,2026261786157,B,quiet,3,0.001,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.8975,3.9,0.9023892879486084
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",115,2026256685992,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.5025,15.9,4.864692449569702
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",55,2026260086034,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,71.5,63.5,3.491696834564209
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",84,2026258386012,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.87,36.3,3.8726301193237305
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",0,2026263486081,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,9.65,16.19,3.9673473834991455
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",116,2026256685993,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.1225,16.155,5.00758695602417
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",56,2026260086035,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,75.3,65.0,3.5040981769561768
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",85,2026258386013,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,35.2,37.54,4.059887886047363
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",1,2026263486082,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.855,15.76,3.979736328125
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",57,2026260086036,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,74.075,63.85,3.619525909423828
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",117,2026256685994,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,1.0,1,0,12.1275,15.915,4.919116973876953
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",86,2026258386014,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.92,35.18,3.989424705505371
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",2,2026263486083,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,9.975,15.745,3.9929656982421875
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",58,2026260086037,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,74.875,61.8,3.437445878982544
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",87,2026258386015,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.43,36.48,3.545957088470459
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",118,2026256685995,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.0425,15.995,4.635181665420532
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",3,2026263486084,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.9725,15.7,3.8762824535369873
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",59,2026260086038,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,70.475,62.55,3.369856834411621
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",88,2026258386016,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.02,35.98,3.9541923999786377
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",60,2026260086039,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,76.0,63.75,3.3715102672576904
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",4,2026263486085,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.83,15.505,4.084091663360596
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",119,2026256685996,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.4475,15.95,4.788841009140015
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",89,2026258386017,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.93,36.68,3.965129852294922
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",61,2026260086040,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,74.825,62.1,3.6644906997680664
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",5,2026263486086,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.8,15.815,4.293980121612549
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",120,2026256685997,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.2325,16.17,4.630567312240601
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",90,2026258386018,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.43,36.9,3.830887794494629
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",62,2026260086041,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,77.1,64.05,3.575969934463501
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",6,2026263486087,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,9.805,16.015,3.9909398555755615
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",121,2026256685998,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.2375,16.06,4.853011846542358
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",91,2026258386019,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.87,36.2,3.9300363063812256
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",63,2026260086042,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,76.4,63.85,4.067565441131592
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",7,2026263486088,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.7575,15.655,4.084304332733154
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",122,2026256685999,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.385,15.795,4.918114900588989
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",92,2026258386020,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.95,36.28,3.717991590499878
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",64,2026260086043,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,74.925,62.55,3.594219207763672
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",8,2026263486089,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.68,16.175,3.9497830867767334
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",65,2026260086044,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,71.4,64.75,3.198462963104248
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",93,2026258386021,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.11,36.82,3.876620054244995
production-20261029T053925Z-29f79c,"A|quiet|d5|p0.0003|pi0.0003|k1,1,1",123,2026256686000,A,quiet,5,0.0003,0.0003,1.0,1.0,1.0,5,5,5,400,0.0,0,0,12.075,16.155,4.733182907104492
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",9,2026263486090,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.8025,15.965,3.820120334625244
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",66,2026260086045,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,74.5,63.85,3.3922696113586426
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",94,2026258386022,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.99,36.18,3.6731436252593994
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",10,2026263486091,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.7325,16.265,3.5604193210601807
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",0,2026265186132,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,27.67,36.12,3.346468925476074
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",67,2026260086046,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,74.35,62.75,3.642514705657959
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",95,2026258386023,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.64,35.76,3.880540132522583
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",1,2026265186133,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.46,36.06,3.1871440410614014
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",11,2026263486092,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.9225,16.115,3.9650299549102783
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",68,2026260086047,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,73.325,62.1,3.5975406169891357
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",96,2026258386024,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,34.61,35.4,3.7653496265411377
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",2,2026265186134,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.69,35.32,3.530534505844116
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",12,2026263486093,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.84,15.825,3.986844778060913
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",69,2026260086048,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,74.85,65.15,3.8652992248535156
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",97,2026258386025,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.56,35.42,3.968552827835083
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",3,2026265186135,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.73,36.26,3.424450635910034
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",13,2026263486094,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.7125,16.2,3.8305106163024902
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",70,2026260086049,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,72.725,65.85,4.120736360549927
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",98,2026258386026,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.93,35.6,4.008805990219116
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",4,2026265186136,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.52,35.94,3.456265449523926
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",14,2026263486095,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,9.6875,16.025,3.88142991065979
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",71,2026260086050,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,73.625,60.85,3.552492141723633
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",5,2026265186137,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.42,36.78,3.2695233821868896
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",99,2026258386027,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.3,37.08,4.059043884277344
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",15,2026263486096,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.94,15.9,4.0509796142578125
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",72,2026260086051,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,75.475,62.2,3.733052968978882
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",6,2026265186138,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.0,36.3,3.3093812465667725
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",100,2026258386028,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,35.2,35.12,3.991105079650879
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",16,2026263486097,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,9.95,16.37,3.8712074756622314
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",7,2026265186139,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.25,35.84,3.353569984436035
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",73,2026260086052,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,75.4,64.05,3.9833462238311768
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",101,2026258386029,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.37,37.3,3.980234146118164
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",8,2026265186140,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.99,34.42,3.481985330581665
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",17,2026263486098,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.8325,15.76,4.0171191692352295
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",74,2026260086053,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,71.45,66.35,3.885484457015991
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",102,2026258386030,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,35.51,35.3,4.0777809619903564
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",9,2026265186141,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.65,37.42,3.4747703075408936
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",75,2026260086054,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,74.0,62.75,3.5550577640533447
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",18,2026263486099,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.795,16.12,3.9415218830108643
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",103,2026258386031,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.14,36.32,4.153695821762085
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",10,2026265186142,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.37,36.18,3.448817253112793
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",76,2026260086055,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,73.9,63.7,3.3931686878204346
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",19,2026263486100,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.78,16.09,4.108121871948242
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",11,2026265186143,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.99,36.32,3.558997631072998
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",104,2026258386032,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.68,37.16,4.0693066120147705
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",77,2026260086056,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,72.5,63.7,4.067436456680298
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",20,2026263486101,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.0725,15.685,4.080943822860718
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",12,2026265186144,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.14,36.7,3.461428642272949
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",105,2026258386033,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.67,36.34,3.6688222885131836
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",78,2026260086057,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,75.05,63.8,4.1401379108428955
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",21,2026263486102,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.71,15.685,3.823538064956665
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",13,2026265186145,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.45,37.14,3.4687509536743164
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",106,2026258386034,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,35.41,36.74,3.6029281616210938
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",79,2026260086058,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,74.075,63.3,4.107494354248047
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",22,2026263486103,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.2125,15.685,3.8806464672088623
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",14,2026265186146,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.54,36.06,3.529052495956421
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",107,2026258386035,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.75,36.1,3.6514735221862793
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",80,2026260086059,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,73.025,63.45,4.191864252090454
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",23,2026263486104,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,9.785,16.215,4.015765905380249
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",15,2026265186147,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.03,36.58,3.524301767349243
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",108,2026258386036,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.02,36.46,3.7299344539642334
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",16,2026265186148,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.05,36.2,3.6284914016723633
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",81,2026260086060,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,73.15,65.6,4.133638381958008
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",24,2026263486105,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.8525,16.135,4.1488800048828125
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",109,2026258386037,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.48,36.22,3.7462174892425537
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",17,2026265186149,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.21,36.24,3.3204407691955566
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",25,2026263486106,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.9975,16.36,4.032320737838745
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",82,2026260086061,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,73.925,61.85,4.128127813339233
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",110,2026258386038,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,35.2,37.28,4.06342887878418
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",18,2026265186150,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.78,36.0,3.41557240486145
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",26,2026263486107,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.8425,16.315,3.9940996170043945
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",83,2026260086062,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,75.3,64.7,4.050300121307373
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",111,2026258386039,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,35.07,35.92,3.953521251678467
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",19,2026265186151,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,29.1,36.36,3.5183002948760986
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",84,2026260086063,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,74.175,64.95,3.561901807785034
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",27,2026263486108,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,9.98,16.175,4.027329683303833
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",112,2026258386040,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.31,36.2,3.952669620513916
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",20,2026265186152,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.17,37.12,3.626831293106079
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",85,2026260086064,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,73.925,65.75,3.343327760696411
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",28,2026263486109,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,9.8925,15.97,3.8731913566589355
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",113,2026258386041,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.5,1,1,34.03,35.86,3.952073335647583
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",21,2026265186153,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.31,36.02,3.5758516788482666
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",86,2026260086065,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,72.525,62.2,3.313713788986206
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",29,2026263486110,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.8525,16.025,4.0516862869262695
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",114,2026258386042,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.46,36.44,3.9723563194274902
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",22,2026265186154,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.85,37.04,3.6328248977661133
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",87,2026260086066,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,73.25,66.0,3.415529727935791
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",30,2026263486111,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.16,15.545,4.12213134765625
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",23,2026265186155,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.16,35.88,3.4551100730895996
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",115,2026258386043,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.68,35.84,4.028647422790527
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",88,2026260086067,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,75.475,65.9,4.050497055053711
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",24,2026265186156,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.58,35.46,3.3893959522247314
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",31,2026263486112,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.02,16.12,3.9195263385772705
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",116,2026258386044,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.74,37.5,4.044168710708618
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",89,2026260086068,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,73.125,65.0,3.949293851852417
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",25,2026265186157,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.07,36.86,3.4941251277923584
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",32,2026263486113,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.7675,16.125,3.9213531017303467
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",117,2026258386045,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,1.0,1,0,34.08,35.9,4.001374244689941
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",90,2026260086069,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,72.925,65.95,4.093275308609009
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",26,2026265186158,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.05,35.58,3.4366283416748047
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",33,2026263486114,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,9.9025,16.025,3.976046562194824
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",118,2026258386046,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.79,37.46,3.9982869625091553
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",91,2026260086070,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,73.375,64.3,3.621152877807617
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",27,2026265186159,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.54,36.1,3.366683006286621
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",34,2026263486115,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.895,15.855,3.999863386154175
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",119,2026258386047,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,32.95,36.6,3.9403719902038574
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",92,2026260086071,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,72.35,64.85,4.012123346328735
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",28,2026265186160,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.62,36.14,3.507920742034912
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",35,2026263486116,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.085,15.74,3.923647880554199
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",120,2026258386048,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.05,35.98,4.055936098098755
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",29,2026265186161,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.39,36.78,3.464928150177002
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",93,2026260086072,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,74.775,61.9,4.251054763793945
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",36,2026263486117,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.06,16.155,3.8158986568450928
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",30,2026265186162,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.05,36.28,3.291348695755005
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",121,2026258386049,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.22,35.76,3.9166901111602783
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",94,2026260086073,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,72.2,66.65,3.673405647277832
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",37,2026263486118,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,9.97,16.17,3.967031478881836
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",31,2026265186163,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.7,35.46,3.572338819503784
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",122,2026258386050,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.36,37.36,3.6776232719421387
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",95,2026260086074,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,74.075,64.55,4.147355318069458
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",32,2026265186164,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.71,36.1,3.5023298263549805
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",38,2026263486119,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.85,16.005,3.869929552078247
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",123,2026258386051,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,35.24,36.18,3.926288604736328
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",96,2026260086075,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,74.075,64.9,4.129213333129883
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",33,2026265186165,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.03,36.0,3.4442758560180664
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",39,2026263486120,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.0675,16.195,4.048243761062622
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",124,2026258386052,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,35.73,35.26,4.055696725845337
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",97,2026260086076,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,73.575,64.05,4.093417167663574
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",34,2026265186166,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.06,35.82,3.4741220474243164
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",40,2026263486121,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.71,15.665,4.000514984130859
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",125,2026258386053,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.28,35.7,3.9357199668884277
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",98,2026260086077,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,74.85,63.45,3.6761627197265625
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",35,2026265186167,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,28.35,35.6,3.593700408935547
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",41,2026263486122,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.885,16.035,3.9261281490325928
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",126,2026258386054,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,35.43,36.36,3.9877941608428955
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",99,2026260086078,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,76.5,63.45,3.7744123935699463
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",36,2026265186168,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,29.08,35.62,3.660632610321045
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",42,2026263486123,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,9.975,16.045,4.229555368423462
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",127,2026258386055,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.33,35.84,4.0626442432403564
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",37,2026265186169,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.73,36.8,3.574472427368164
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",100,2026260086079,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,74.775,62.95,4.0259528160095215
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",43,2026263486124,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.0025,16.09,4.108544826507568
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",128,2026258386056,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.78,35.86,4.038462162017822
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",38,2026265186170,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.07,35.02,3.4912962913513184
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",101,2026260086080,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,73.475,64.7,4.411966562271118
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",44,2026263486125,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,9.91,15.895,3.9323699474334717
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",39,2026265186171,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,29.41,36.42,3.5342679023742676
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",129,2026258386057,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.8,36.96,3.9600770473480225
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",102,2026260086081,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,75.475,61.2,4.027917861938477
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",45,2026263486126,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,10.015,15.835,3.9405782222747803
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",40,2026265186172,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.19,35.66,3.6434388160705566
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",130,2026258386058,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.81,35.48,3.6491854190826416
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",103,2026260086082,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,73.6,62.45,3.2820942401885986
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",41,2026265186173,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.19,35.6,3.4558091163635254
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",46,2026263486127,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.1875,16.29,3.939847230911255
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",131,2026258386059,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.59,35.12,3.7243807315826416
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",104,2026260086083,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,73.5,64.2,3.3699045181274414
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",42,2026265186174,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.67,36.16,3.5571653842926025
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",47,2026263486128,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.92,16.375,3.860049247741699
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",105,2026260086084,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,75.475,64.5,3.279254674911499
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",132,2026258386060,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.1,35.74,3.8863131999969482
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",43,2026265186175,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,29.61,36.6,3.5401411056518555
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",106,2026260086085,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,73.8,63.0,3.3795459270477295
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",48,2026263486129,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,9.9325,16.08,3.8609633445739746
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",133,2026258386061,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.8,35.94,4.058910846710205
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",44,2026265186176,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.59,36.22,3.5802125930786133
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",107,2026260086086,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,75.225,62.05,3.847604513168335
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",49,2026263486130,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,10.1,15.675,4.025177478790283
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",134,2026258386062,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.53,34.88,3.9812381267547607
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",45,2026265186177,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.8,35.8,3.4819047451019287
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",108,2026260086087,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,75.025,64.0,3.9850716590881348
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",50,2026263486131,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.8325,16.02,4.079933166503906
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",135,2026258386063,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.33,35.88,3.5562384128570557
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",46,2026265186178,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,28.16,36.78,3.3579659461975098
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",109,2026260086088,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,72.275,65.7,3.5631320476531982
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",136,2026258386064,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.59,34.6,3.54282283782959
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",51,2026263486132,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,9.8975,15.715,3.8200929164886475
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",47,2026265186179,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.86,36.32,3.324100971221924
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",110,2026260086089,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,75.25,62.6,3.9472219944000244
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",137,2026258386065,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.71,35.74,3.6476383209228516
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",52,2026263486133,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.9275,16.125,3.9776244163513184
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",48,2026265186180,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.11,36.04,3.5184056758880615
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",111,2026260086090,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,74.35,63.85,3.6779322624206543
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",138,2026258386066,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.62,36.22,3.9994852542877197
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",49,2026265186181,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.41,36.68,3.41606068611145
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",53,2026263486134,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,9.765,15.8,4.032005071640015
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",112,2026260086091,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,75.4,63.45,3.531834363937378
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",50,2026265186182,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,28.03,34.82,3.3858001232147217
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",139,2026258386067,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.76,35.68,4.024946451187134
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",54,2026263486135,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.87,16.175,3.9567201137542725
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",113,2026260086092,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,71.875,61.6,3.9554202556610107
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",51,2026265186183,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.89,35.58,3.424665689468384
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",140,2026258386068,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.83,36.18,3.6953465938568115
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",55,2026263486136,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.795,15.765,3.9525485038757324
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",52,2026265186184,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.53,35.12,3.3386576175689697
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",114,2026260086093,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,74.95,66.35,3.8182477951049805
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",141,2026258386069,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.37,36.02,3.5849132537841797
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",56,2026263486137,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,10.105,16.03,3.997627019882202
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",53,2026265186185,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.36,36.3,3.3022007942199707
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",115,2026260086094,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,74.525,64.3,3.8714184761047363
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",142,2026258386070,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.83,35.54,3.7976763248443604
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",57,2026263486138,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,2,2,9.74,15.695,3.9999570846557617
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",54,2026265186186,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.99,35.6,3.247922897338867
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",116,2026260086095,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,73.075,63.75,3.4460582733154297
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",143,2026258386071,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.56,35.56,3.8203134536743164
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",58,2026263486139,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.85,16.155,3.809469223022461
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",55,2026265186187,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.79,36.16,3.309147357940674
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",117,2026260086096,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,71.525,64.9,3.426419496536255
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",144,2026258386072,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.44,36.32,3.809174060821533
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",56,2026265186188,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.2,36.14,3.5068612098693848
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",59,2026263486140,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.88,15.97,4.18916130065918
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",118,2026260086097,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,74.975,65.2,3.9466936588287354
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",145,2026258386073,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,35.39,35.3,3.959794521331787
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",57,2026265186189,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.67,36.4,3.4334819316864014
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",60,2026263486141,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.7925,16.14,4.114853858947754
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",119,2026260086098,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,75.1,63.25,3.294264316558838
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",146,2026258386074,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.36,36.46,3.8711440563201904
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",58,2026265186190,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.15,36.4,3.2688848972320557
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",120,2026260086099,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,76.075,62.6,3.3481321334838867
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",61,2026263486142,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.02,16.055,4.055410146713257
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",59,2026265186191,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.4,37.5,3.5854732990264893
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",147,2026258386075,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.58,33.8,4.1024298667907715
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",121,2026260086100,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,72.925,62.2,3.399888753890991
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",62,2026263486143,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.7525,15.98,4.037347793579102
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",60,2026265186192,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.46,36.2,3.7447268962860107
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",148,2026258386076,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.96,36.44,3.7392632961273193
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",122,2026260086101,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,75.975,64.25,3.454340696334839
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",63,2026263486144,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.9225,16.025,3.7578067779541016
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",61,2026265186193,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.96,36.44,3.4784882068634033
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",149,2026258386077,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.83,36.28,3.633629083633423
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",123,2026260086102,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,73.1,64.75,3.472668170928955
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",64,2026263486145,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.7825,16.07,3.819300889968872
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",62,2026265186194,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.59,36.56,3.6400601863861084
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",150,2026258386078,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.49,36.44,3.696385622024536
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",124,2026260086103,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,75.025,65.0,3.48618221282959
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",65,2026263486146,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,9.8525,15.84,4.075954914093018
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",63,2026265186195,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.84,36.66,3.6184003353118896
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",125,2026260086104,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,72.5,65.7,3.490804672241211
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",151,2026258386079,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.59,37.06,3.6973047256469727
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",64,2026265186196,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.08,36.96,3.5382425785064697
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",66,2026263486147,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,10.0425,15.88,3.9369137287139893
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",152,2026258386080,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,33.06,36.74,3.672785520553589
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",126,2026260086105,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,76.225,67.1,4.312760591506958
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",65,2026265186197,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,29.27,35.96,3.5977494716644287
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",67,2026263486148,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,9.85,15.935,3.9921653270721436
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",153,2026258386081,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.0,36.16,4.00462007522583
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",127,2026260086106,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,74.65,63.65,4.399050951004028
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",66,2026265186198,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.19,36.96,3.530050039291382
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",68,2026263486149,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.7125,16.1,3.8964202404022217
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",154,2026258386082,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,34.8,35.56,3.796159505844116
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",128,2026260086107,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,75.8,61.65,4.2504870891571045
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",67,2026265186199,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,29.09,36.44,3.5866129398345947
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",69,2026263486150,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,9.7725,15.35,3.8640215396881104
production-20261029T053925Z-29f79c,"A|quiet|d7|p0.0003|pi0.0003|k1,1,1",155,2026258386083,A,quiet,7,0.0003,0.0003,1.0,1.0,1.0,7,7,7,100,0.0,0,0,35.48,36.42,3.7396416664123535
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",129,2026260086108,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,72.55,61.95,4.018116474151611
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",68,2026265186200,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.83,36.8,3.5413074493408203
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",70,2026263486151,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.9275,15.9,3.9871761798858643
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",69,2026265186201,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.41,36.6,3.5150861740112305
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",130,2026260086109,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,74.95,63.7,3.9261841773986816
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",71,2026263486152,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,10.2625,15.84,3.9536194801330566
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",0,2026266886183,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.05,64.35,3.4549551010131836
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",70,2026265186202,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.64,35.74,3.4809231758117676
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",131,2026260086110,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,73.125,65.55,3.6166324615478516
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",1,2026266886184,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,60.175,64.15,3.296673536300659
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",72,2026263486153,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.855,15.965,3.975022315979004
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",71,2026265186203,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.64,35.26,3.4647204875946045
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",132,2026260086111,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,74.25,65.4,3.7612390518188477
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",2,2026266886185,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,60.25,62.1,3.508115768432617
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",73,2026263486154,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.6925,16.005,3.8868422508239746
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",72,2026265186204,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.65,36.4,3.616239547729492
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",133,2026260086112,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,72.0,63.55,3.8552985191345215
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",3,2026266886186,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,63.325,62.35,3.0726866722106934
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",74,2026263486155,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.79,15.675,3.9201364517211914
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",73,2026265186205,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.98,36.48,3.533226728439331
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",134,2026260086113,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,72.975,61.6,3.54421329498291
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",4,2026266886187,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,59.375,64.5,3.459423303604126
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",75,2026263486156,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,9.69,16.355,3.9817023277282715
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",74,2026265186206,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.14,35.58,3.6711366176605225
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",135,2026260086114,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,74.3,65.25,3.5279924869537354
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",5,2026266886188,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.275,62.9,3.533106803894043
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",75,2026265186207,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.72,36.1,3.6122214794158936
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",76,2026263486157,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.5,2,1,9.9925,15.975,3.9409897327423096
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",6,2026266886189,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,62.2,62.4,3.4499850273132324
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",136,2026260086115,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,70.55,64.55,3.8519647121429443
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",76,2026265186208,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.56,36.26,3.5251352787017822
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",77,2026263486158,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,9.955,16.035,3.902085065841675
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",7,2026266886190,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,60.25,65.2,3.427673101425171
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",137,2026260086116,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,72.5,65.85,3.8326570987701416
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",77,2026265186209,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.97,35.82,3.6109261512756348
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",8,2026266886191,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,63.05,65.2,3.0519134998321533
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",78,2026263486159,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.095,15.795,3.994584798812866
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",138,2026260086117,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,72.825,62.75,3.47273850440979
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",9,2026266886192,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,62.925,63.6,3.2086479663848877
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",78,2026265186210,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.14,35.58,3.6813266277313232
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",79,2026263486160,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.985,15.75,4.130621671676636
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",139,2026260086118,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,74.45,63.2,4.2392659187316895
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",10,2026266886193,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,60.25,64.9,3.174025535583496
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",79,2026265186211,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.1,36.98,3.5987775325775146
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",140,2026260086119,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,72.85,64.8,3.457050323486328
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",80,2026263486161,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,9.87,16.11,3.8986141681671143
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",11,2026266886194,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,58.6,61.9,3.6171789169311523
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",80,2026265186212,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.86,36.36,3.39249587059021
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",141,2026260086120,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,72.025,64.55,4.053677558898926
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",81,2026263486162,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,9.72,15.595,3.930129289627075
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",12,2026266886195,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.775,63.25,3.4964942932128906
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",81,2026265186213,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.58,34.38,3.423452615737915
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",82,2026263486163,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.8175,16.185,3.97562837600708
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",142,2026260086121,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,72.8,63.75,4.299794912338257
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",13,2026266886196,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.35,63.0,3.4987082481384277
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",82,2026265186214,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.8,35.6,3.647059917449951
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",14,2026266886197,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.45,63.1,3.5227363109588623
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",83,2026263486164,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.7025,15.67,4.109502553939819
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",83,2026265186215,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.61,35.52,3.4569857120513916
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",143,2026260086122,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,75.225,63.95,4.34390926361084
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",15,2026266886198,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,62.05,63.1,3.3935492038726807
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",84,2026265186216,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,29.19,35.76,3.3996071815490723
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",84,2026263486165,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.9925,15.745,3.86950421333313
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",144,2026260086123,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,71.45,62.8,3.6196751594543457
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",16,2026266886199,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,59.075,64.3,3.3659303188323975
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",85,2026265186217,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.72,36.7,3.554455041885376
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",145,2026260086124,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,71.875,61.45,3.594486713409424
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",85,2026263486166,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.9325,15.98,4.073659181594849
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",17,2026266886200,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,60.95,64.55,3.561312437057495
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",86,2026265186218,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.03,35.86,3.7041101455688477
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",146,2026260086125,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,72.9,63.65,3.567833185195923
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",86,2026263486167,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.91,16.215,4.128170490264893
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",18,2026266886201,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,59.7,63.25,3.4226553440093994
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",87,2026265186219,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,28.14,35.16,3.7202227115631104
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",147,2026260086126,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,73.875,64.2,3.9352612495422363
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",87,2026263486168,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.965,16.355,4.0879621505737305
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",19,2026266886202,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,62.1,63.5,3.496760845184326
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",88,2026265186220,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.49,35.68,3.564326286315918
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",148,2026260086127,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,1.0,1,0,74.1,62.15,3.7610883712768555
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",88,2026263486169,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.9725,16.125,3.9434361457824707
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",20,2026266886203,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,62.15,61.95,3.4479002952575684
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",89,2026265186221,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.49,36.5,3.4368743896484375
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",149,2026260086128,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,75.675,62.75,3.841738224029541
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",21,2026266886204,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,62.0,60.6,3.4727025032043457
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",89,2026263486170,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,9.7625,16.065,4.040849924087524
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",90,2026265186222,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.69,35.64,3.587923049926758
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",150,2026260086129,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,73.125,64.65,4.034315347671509
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",22,2026266886205,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,59.6,63.5,3.4228575229644775
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",90,2026263486171,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.8575,16.055,4.1444008350372314
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",91,2026265186223,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.65,35.56,3.636730194091797
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",23,2026266886206,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,60.225,65.45,3.3769447803497314
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",151,2026260086130,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,75.275,62.55,4.302109956741333
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",92,2026265186224,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.64,36.04,3.4053544998168945
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",91,2026263486172,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,9.945,16.26,4.052111864089966
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",24,2026266886207,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,59.6,65.25,3.3580915927886963
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",152,2026260086131,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,74.9,62.65,4.0203258991241455
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",93,2026265186225,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,27.87,36.94,3.456550359725952
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",92,2026263486173,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.055,16.285,4.028176784515381
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",25,2026266886208,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,60.275,61.95,3.535693645477295
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",153,2026260086132,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,74.75,67.05,3.3829362392425537
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",94,2026265186226,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.8,36.3,3.5760679244995117
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",93,2026263486174,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.805,15.7,3.6738903522491455
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",26,2026266886209,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.8,63.85,2.9472804069519043
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",154,2026260086133,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,75.8,62.1,3.3333544731140137
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",95,2026265186227,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,29.08,36.06,3.5483639240264893
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",94,2026263486175,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,9.9325,16.105,3.857675075531006
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",27,2026266886210,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,60.625,65.35,3.4975244998931885
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",155,2026260086134,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,75.1,63.15,4.245468378067017
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",96,2026265186228,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.62,36.08,3.520005464553833
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",28,2026266886211,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,62.025,63.85,3.6332173347473145
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",95,2026263486176,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.795,16.035,3.9555952548980713
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",97,2026265186229,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.86,35.04,3.477055072784424
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",156,2026260086135,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,72.875,61.7,4.326194524765015
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",29,2026266886212,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,60.1,64.15,3.4182791709899902
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",96,2026263486177,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.68,15.575,3.9590370655059814
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",98,2026265186230,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.28,35.88,3.4395925998687744
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",30,2026266886213,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.3,63.1,3.2266528606414795
production-20261029T053925Z-29f79c,"A|quiet|d9|p0.0003|pi0.0003|k1,1,1",157,2026260086136,A,quiet,9,0.0003,0.0003,1.0,1.0,1.0,9,9,9,40,0.0,0,0,75.175,64.4,4.149684906005859
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",97,2026263486178,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,10.05,15.8,3.801513910293579
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",0,2026268586234,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.92,3.76,0.8451249599456787
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",1,2026268586235,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.995,3.95,0.886258602142334
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",99,2026265186231,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.92,35.6,3.55362868309021
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",2,2026268586236,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9775,4.04,0.8565552234649658
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",31,2026266886214,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.7,64.8,3.107135057449341
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",3,2026268586237,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.935,4.01,0.8246526718139648
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",4,2026268586238,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.9725,3.96,0.8226172924041748
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",98,2026263486179,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.065,15.975,4.005393028259277
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",5,2026268586239,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.905,4.12,0.8206429481506348
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",100,2026265186232,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.49,34.92,3.4960684776306152
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",32,2026266886215,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.425,65.1,3.1264467239379883
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",6,2026268586240,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.935,3.91,0.8832433223724365
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",7,2026268586241,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.955,4.13,0.7500286102294922
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",8,2026268586242,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.905,4.065,0.7586030960083008
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",9,2026268586243,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.955,4.125,0.7452294826507568
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",99,2026263486180,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.78,16.25,3.970710039138794
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",101,2026265186233,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.0,35.74,3.588447332382202
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",10,2026268586244,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.04,4.055,0.767939567565918
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",33,2026266886216,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,63.475,64.85,3.537445545196533
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",11,2026268586245,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.045,3.965,0.7668242454528809
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",12,2026268586246,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.9725,4.045,0.7867088317871094
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",13,2026268586247,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.96,4.13,0.8381168842315674
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",14,2026268586248,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9925,4.16,0.8222594261169434
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",102,2026265186234,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.52,35.84,3.5625851154327393
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",100,2026263486181,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,9.815,15.72,4.063191890716553
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",34,2026266886217,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,60.825,62.95,3.558692455291748
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",15,2026268586249,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.955,4.11,0.8279528617858887
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",16,2026268586250,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.945,3.91,0.855039119720459
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",17,2026268586251,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9175,4.105,0.8175535202026367
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",18,2026268586252,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.905,4.14,0.792677640914917
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",35,2026266886218,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.175,64.3,3.0646235942840576
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",103,2026265186235,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.92,36.1,3.536381959915161
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",19,2026268586253,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9225,3.895,0.808300256729126
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",101,2026263486182,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.935,15.965,3.914834499359131
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",20,2026268586254,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.96,3.965,0.8382563591003418
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",21,2026268586255,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0075,4.145,0.8268136978149414
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",22,2026268586256,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9975,3.975,0.8199257850646973
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",36,2026266886219,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.225,64.4,3.0285556316375732
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",23,2026268586257,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.945,4.03,0.7619407176971436
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",104,2026265186236,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.37,35.94,3.5060367584228516
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",24,2026268586258,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9875,4.045,0.7573730945587158
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",102,2026263486183,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,9.8375,15.785,4.017214775085449
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",25,2026268586259,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9175,3.98,0.7584409713745117
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",26,2026268586260,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.135,4.03,0.7544939517974854
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",37,2026266886220,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,63.35,65.1,3.1796391010284424
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",27,2026268586261,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9475,3.88,0.8027102947235107
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",105,2026265186237,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.36,35.18,3.6122379302978516
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",28,2026268586262,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.97,3.94,0.8256986141204834
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",29,2026268586263,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9825,4.17,0.8233819007873535
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",103,2026263486184,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,9.805,16.07,4.090027570724487
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",30,2026268586264,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.895,3.94,0.824688196182251
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",38,2026266886221,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.05,65.75,3.006871461868286
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",31,2026268586265,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.9375,4.16,0.8472645282745361
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",106,2026265186238,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,29.06,36.44,3.4976420402526855
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",32,2026268586266,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.02,4.025,0.7636010646820068
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",33,2026268586267,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.9075,3.85,0.7390809059143066
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",39,2026266886222,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,62.575,64.0,3.030341386795044
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",34,2026268586268,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8575,3.99,0.8149142265319824
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",104,2026263486185,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,10.06,15.93,3.86791729927063
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",35,2026268586269,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.84,4.02,0.8204548358917236
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",36,2026268586270,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9775,3.94,0.8603849411010742
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",107,2026265186239,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.75,35.86,3.505093812942505
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",37,2026268586271,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.9925,3.97,0.851489782333374
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",40,2026266886223,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.775,60.95,3.264883279800415
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",38,2026268586272,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.935,3.87,0.8513197898864746
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",39,2026268586273,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0075,4.025,0.8387243747711182
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",105,2026263486186,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.0575,15.73,4.131119966506958
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",40,2026268586274,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.905,4.055,0.8298008441925049
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",108,2026265186240,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.71,36.1,3.5889649391174316
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",41,2026268586275,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.975,4.13,0.9399895668029785
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",41,2026266886224,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,59.95,64.25,3.4520459175109863
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",42,2026268586276,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.035,3.87,0.8271870613098145
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",43,2026268586277,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.975,4.04,0.7470278739929199
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",44,2026268586278,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.97,3.995,0.8515241146087646
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",106,2026263486187,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.005,15.765,4.09941291809082
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",109,2026265186241,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.69,36.6,3.5390775203704834
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",45,2026268586279,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.8525,3.985,0.8173360824584961
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",46,2026268586280,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.8625,3.98,0.8233311176300049
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",42,2026266886225,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.75,61.45,3.5773513317108154
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",47,2026268586281,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.015,4.0,0.8870017528533936
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",48,2026268586282,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.9625,4.215,0.8280258178710938
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",110,2026265186242,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.92,34.74,3.546621799468994
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",49,2026268586283,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.91,3.97,0.8881845474243164
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",107,2026263486188,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.8675,15.88,4.042756080627441
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",50,2026268586284,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0,4.055,0.7719566822052002
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",43,2026266886226,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,62.2,64.0,3.7089431285858154
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",51,2026268586285,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.9875,4.155,0.7833762168884277
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",52,2026268586286,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.935,4.07,0.7646853923797607
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",53,2026268586287,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0175,3.95,0.7723660469055176
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",111,2026265186243,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.36,35.62,3.7099974155426025
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",54,2026268586288,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.01,3.975,0.7748231887817383
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",108,2026263486189,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.8925,16.075,4.26588249206543
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",55,2026268586289,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.975,4.12,0.7539353370666504
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",44,2026266886227,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,62.275,65.2,3.592298746109009
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",56,2026268586290,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.895,4.055,0.7708714008331299
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",57,2026268586291,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.95,4.035,0.8011369705200195
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",58,2026268586292,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.935,3.995,0.9536771774291992
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",112,2026265186244,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.23,35.06,3.653271436691284
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",59,2026268586293,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.0175,4.12,0.8222689628601074
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",109,2026263486190,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.7775,15.825,3.881754159927368
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",45,2026266886228,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,62.7,65.45,3.5105154514312744
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",60,2026268586294,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.905,4.06,0.8622739315032959
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",61,2026268586295,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.98,3.945,0.9357483386993408
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",62,2026268586296,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0075,4.17,0.7958705425262451
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",113,2026265186245,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.58,35.78,3.813021421432495
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",63,2026268586297,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9475,3.95,0.8132376670837402
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",46,2026266886229,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,59.475,63.95,3.5457582473754883
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",64,2026268586298,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0075,3.885,0.7924458980560303
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",110,2026263486191,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.845,15.89,4.1522603034973145
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",65,2026268586299,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8975,4.045,0.7501754760742188
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",66,2026268586300,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.02,3.9,0.8143491744995117
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",114,2026265186246,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.49,35.56,3.4013187885284424
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",67,2026268586301,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.905,4.03,0.7379591464996338
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",68,2026268586302,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9375,4.04,0.750800609588623
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",47,2026266886230,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,60.8,63.5,3.4938888549804688
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",69,2026268586303,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.965,3.97,0.8091034889221191
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",111,2026263486192,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.06,15.76,4.279038190841675
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",70,2026268586304,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.96,3.875,0.8926115036010742
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",115,2026265186247,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.02,36.72,3.2895936965942383
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",71,2026268586305,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.905,3.84,0.8361990451812744
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",72,2026268586306,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8975,4.105,0.8084464073181152
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",48,2026266886231,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.35,62.85,3.580825090408325
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",73,2026268586307,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.92,3.815,0.7381751537322998
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",74,2026268586308,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9075,3.93,0.7593443393707275
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",112,2026263486193,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,9.9925,15.865,4.067309856414795
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",75,2026268586309,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9775,4.03,0.7562899589538574
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",116,2026265186248,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.15,35.38,3.5505142211914062
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",76,2026268586310,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.03,3.935,0.7364330291748047
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",77,2026268586311,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0,4.0,0.7615194320678711
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",49,2026266886232,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.025,65.3,3.4347217082977295
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",78,2026268586312,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.9,4.11,0.7526509761810303
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",79,2026268586313,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.955,3.885,0.7548775672912598
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",80,2026268586314,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9225,4.05,0.7424399852752686
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",113,2026263486194,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.7575,15.945,4.1687095165252686
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",117,2026265186249,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,29.11,35.98,3.549868106842041
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",81,2026268586315,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9675,3.82,0.7566390037536621
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",50,2026266886233,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,59.25,64.5,3.482281446456909
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",82,2026268586316,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.995,4.01,0.739769458770752
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",83,2026268586317,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0375,3.98,0.7619709968566895
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",84,2026268586318,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9625,4.105,0.7536609172821045
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",118,2026265186250,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,29.23,36.3,3.423783302307129
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",85,2026268586319,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8675,3.94,0.7276597023010254
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",114,2026263486195,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.7375,16.075,3.965266466140747
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",86,2026268586320,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.965,3.96,0.7274234294891357
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",51,2026266886234,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,60.45,62.85,3.5654942989349365
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",87,2026268586321,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9675,4.11,0.8150818347930908
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",88,2026268586322,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9825,4.04,0.755518913269043
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",89,2026268586323,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.935,4.02,0.7437489032745361
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",119,2026265186251,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.89,35.44,3.5500667095184326
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",90,2026268586324,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.91,3.955,0.7408676147460938
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",115,2026263486196,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,9.6025,16.355,3.755810260772705
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",52,2026266886235,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.625,63.6,3.1540699005126953
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",91,2026268586325,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.015,4.02,0.748619794845581
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",92,2026268586326,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0375,3.965,0.8419418334960938
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",93,2026268586327,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.98,3.87,0.803760290145874
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",120,2026265186252,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,28.13,35.58,3.4025204181671143
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",94,2026268586328,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0425,4.04,0.8571774959564209
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",53,2026266886236,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.5,1,1,60.45,65.05,3.490403890609741
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",95,2026268586329,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.9625,4.165,0.8527665138244629
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",116,2026263486197,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.5825,15.695,3.8288967609405518
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",96,2026268586330,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.86,3.865,0.8714649677276611
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",97,2026268586331,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.9975,3.965,0.825023889541626
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",121,2026265186253,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.73,35.32,3.44927716255188
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",98,2026268586332,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0075,3.97,0.819136381149292
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",99,2026268586333,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0025,3.995,0.860051155090332
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",54,2026266886237,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.625,64.4,3.7573018074035645
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",100,2026268586334,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9475,4.005,0.8404490947723389
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",117,2026263486198,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.89,15.405,4.128519296646118
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",101,2026268586335,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.84,3.895,0.8622453212738037
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",102,2026268586336,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9125,3.925,0.8285524845123291
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",122,2026265186254,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.09,36.42,3.634625196456909
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",103,2026268586337,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.02,3.935,0.8410642147064209
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",55,2026266886238,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.7,62.85,3.44673752784729
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",104,2026268586338,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0425,3.965,0.8769328594207764
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",118,2026263486199,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.8725,15.885,3.844085931777954
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",105,2026268586339,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9175,4.025,0.8034403324127197
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",106,2026268586340,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9625,3.845,0.8140966892242432
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",123,2026265186255,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.72,35.46,3.4951534271240234
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",107,2026268586341,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.935,3.98,0.7523617744445801
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",56,2026266886239,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,63.95,64.55,3.55007004737854
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",108,2026268586342,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.875,3.87,0.8339338302612305
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",109,2026268586343,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.02,3.93,0.8810017108917236
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",119,2026263486200,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.675,16.095,3.9040615558624268
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",110,2026268586344,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.075,4.01,0.8350403308868408
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",124,2026265186256,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.54,37.6,3.520688533782959
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",111,2026268586345,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9025,3.92,0.8512694835662842
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",57,2026266886240,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,59.075,63.5,3.538214683532715
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",112,2026268586346,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.94,3.995,0.7277400493621826
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",113,2026268586347,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.93,3.955,0.7823584079742432
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",114,2026268586348,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0025,4.005,0.8218879699707031
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",120,2026263486201,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,2.0,2,0,9.8675,15.68,4.073879718780518
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",115,2026268586349,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9425,3.92,0.7241926193237305
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",125,2026265186257,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.25,38.0,3.4655795097351074
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",116,2026268586350,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.98,3.845,0.739112138748169
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",58,2026266886241,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,60.75,64.05,3.4163460731506348
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",117,2026268586351,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9925,3.91,0.7444508075714111
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",118,2026268586352,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0275,4.005,0.7428209781646729
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",119,2026268586353,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9225,4.01,0.7220981121063232
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",126,2026265186258,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.75,35.86,3.2801930904388428
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",121,2026263486202,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,9.795,15.76,3.8466806411743164
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",120,2026268586354,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8975,4.035,0.7310671806335449
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",59,2026266886242,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,62.675,64.65,3.0150656700134277
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",121,2026268586355,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8825,3.92,0.7961628437042236
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",122,2026268586356,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0175,3.945,0.7313659191131592
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",123,2026268586357,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.905,4.025,0.7898681163787842
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",124,2026268586358,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9925,3.98,0.9913051128387451
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",127,2026265186259,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.93,36.9,3.7039403915405273
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",122,2026263486203,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,9.98,16.095,4.039325952529907
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",60,2026266886243,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,63.825,63.35,3.490414619445801
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",125,2026268586359,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.9425,3.955,0.7700655460357666
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",126,2026268586360,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.985,4.12,0.7445557117462158
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",127,2026268586361,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9925,4.135,0.7542457580566406
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",128,2026268586362,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.005,3.985,0.7501487731933594
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",128,2026265186260,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.27,36.42,3.494302749633789
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",129,2026268586363,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9225,4.01,0.73356032371521
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",61,2026266886244,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,62.1,63.45,3.39145565032959
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",123,2026263486204,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.92,15.755,3.7546117305755615
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",130,2026268586364,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.985,4.05,0.7575008869171143
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",131,2026268586365,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.925,4.07,0.7398934364318848
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",132,2026268586366,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.935,4.025,0.7492156028747559
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",133,2026268586367,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0425,4.05,0.7515254020690918
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",129,2026265186261,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.28,36.66,3.3873672485351562
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",62,2026266886245,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,62.05,64.6,3.4203593730926514
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",134,2026268586368,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0325,4.16,0.7528479099273682
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",124,2026263486205,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,9.825,15.775,3.9013073444366455
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",135,2026268586369,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.015,3.87,0.7418098449707031
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",136,2026268586370,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9125,3.925,0.8281259536743164
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",137,2026268586371,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9225,4.105,0.8484315872192383
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",138,2026268586372,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.8975,4.05,0.7313871383666992
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",130,2026265186262,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.48,35.78,3.727173328399658
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",63,2026266886246,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,60.775,63.9,3.574105978012085
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",139,2026268586373,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.915,4.085,0.7331588268280029
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",140,2026268586374,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.9425,4.01,0.7428348064422607
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",125,2026263486206,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.8725,15.67,4.5241618156433105
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",141,2026268586375,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.965,3.94,0.7566225528717041
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",142,2026268586376,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9725,4.055,0.7698383331298828
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",64,2026266886247,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.225,63.3,3.4352097511291504
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",143,2026268586377,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9775,3.88,0.8729898929595947
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",131,2026265186263,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.25,35.58,3.92067813873291
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",144,2026268586378,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9875,3.905,0.8157095909118652
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",145,2026268586379,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.915,4.0,0.8204605579376221
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",126,2026263486207,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.9175,16.35,4.230702638626099
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",146,2026268586380,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.94,4.02,0.8703131675720215
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",65,2026266886248,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,60.75,64.75,3.4061942100524902
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",147,2026268586381,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.06,3.965,0.8544180393218994
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",132,2026265186264,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.42,35.16,3.7873902320861816
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",148,2026268586382,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9775,3.915,0.8452413082122803
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",149,2026268586383,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9075,3.85,0.7510123252868652
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",150,2026268586384,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9675,4.025,0.7856228351593018
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",127,2026263486208,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,10.035,15.785,4.146466493606567
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",151,2026268586385,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9425,3.935,0.7826583385467529
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",66,2026266886249,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.55,67.85,3.6026251316070557
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",152,2026268586386,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.025,4.13,0.848527193069458
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",133,2026265186265,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,29.0,35.44,3.5685739517211914
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",153,2026268586387,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8825,3.95,0.7649731636047363
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",154,2026268586388,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.0575,4.045,0.7801547050476074
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",155,2026268586389,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.965,4.025,0.744945764541626
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",128,2026263486209,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,9.9475,15.84,3.9794867038726807
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",156,2026268586390,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.0175,4.235,0.818091630935669
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",67,2026266886250,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.175,63.2,3.700810432434082
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",134,2026265186266,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.86,35.66,3.4866414070129395
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",157,2026268586391,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9675,4.085,0.8212556838989258
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",158,2026268586392,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.935,4.125,0.8137803077697754
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",159,2026268586393,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0375,3.925,0.7722461223602295
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",160,2026268586394,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9525,4.085,0.8221921920776367
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",68,2026266886251,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,60.65,64.1,3.5788216590881348
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",135,2026265186267,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.19,35.74,3.5395665168762207
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",161,2026268586395,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.945,3.905,0.8231699466705322
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",129,2026263486210,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.78,16.085,4.1205220222473145
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",162,2026268586396,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0275,4.07,0.763408899307251
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",163,2026268586397,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0475,4.0,0.7677340507507324
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",164,2026268586398,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.955,4.04,0.7468664646148682
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",165,2026268586399,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.01,4.095,0.7545862197875977
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",69,2026266886252,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.425,63.45,3.4725875854492188
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",136,2026265186268,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.46,35.52,3.6662135124206543
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",166,2026268586400,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.86,4.24,0.7459115982055664
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",130,2026263486211,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,10.0525,15.95,4.000564813613892
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",167,2026268586401,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.88,4.035,0.744457483291626
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",168,2026268586402,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.92,4.055,0.7414836883544922
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",169,2026268586403,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.975,4.02,0.7420923709869385
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",70,2026266886253,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,60.825,62.65,3.456700563430786
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",170,2026268586404,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.945,3.955,0.7470333576202393
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",137,2026265186269,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.85,36.14,3.450554132461548
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",171,2026268586405,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0175,4.14,0.7534642219543457
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",131,2026263486212,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.8625,16.275,4.056193590164185
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",172,2026268586406,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9775,4.05,0.7597289085388184
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",173,2026268586407,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0025,4.005,0.7490339279174805
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",174,2026268586408,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.985,4.04,0.8514547348022461
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",71,2026266886254,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,63.3,63.4,3.584493398666382
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",138,2026265186270,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.32,35.2,3.5589358806610107
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",175,2026268586409,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.9475,3.91,0.8111889362335205
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",176,2026268586410,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0275,3.84,0.8729696273803711
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",132,2026263486213,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.7425,16.085,4.010098457336426
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",177,2026268586411,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0325,3.905,0.866614818572998
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",178,2026268586412,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.915,3.94,0.8128561973571777
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",72,2026266886255,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.175,61.3,3.4704856872558594
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",179,2026268586413,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.9825,4.175,0.7630705833435059
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",139,2026265186271,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.19,35.22,3.7038426399230957
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",180,2026268586414,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.82,3.88,0.8491997718811035
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",181,2026268586415,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9675,3.96,0.7499439716339111
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",133,2026263486214,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.73,15.665,4.104723930358887
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",182,2026268586416,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.02,3.925,0.8259530067443848
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",73,2026266886256,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,60.775,64.6,3.501986265182495
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",183,2026268586417,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9225,4.005,0.7662680149078369
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",140,2026265186272,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.42,35.96,3.5627763271331787
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",184,2026268586418,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.915,4.06,0.7395415306091309
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",185,2026268586419,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.88,3.89,0.725499153137207
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",186,2026268586420,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9725,3.86,0.7740228176116943
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",134,2026263486215,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.925,16.31,4.058978080749512
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",187,2026268586421,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.0025,3.905,0.9327116012573242
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",74,2026266886257,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.45,64.0,3.650374412536621
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",188,2026268586422,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.975,3.885,0.8766968250274658
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",141,2026265186273,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.7,35.02,3.5330162048339844
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",189,2026268586423,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.975,3.9,0.8184854984283447
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",190,2026268586424,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9775,4.01,0.8152730464935303
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",191,2026268586425,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.995,4.045,0.8287875652313232
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",135,2026263486216,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.005,16.365,3.882439136505127
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",75,2026266886258,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,59.7,62.45,3.480830192565918
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",192,2026268586426,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0025,4.045,0.7980778217315674
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",142,2026265186274,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,29.19,34.84,3.5543806552886963
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",193,2026268586427,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9275,3.985,0.816110372543335
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",194,2026268586428,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.96,4.0,0.8727471828460693
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",195,2026268586429,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9875,4.04,0.8578310012817383
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",76,2026266886259,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.0,63.1,3.5285117626190186
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",196,2026268586430,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9075,4.105,0.8393771648406982
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",136,2026263486217,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.6475,16.32,3.8965091705322266
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",143,2026265186275,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,29.4,36.7,3.4617056846618652
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",197,2026268586431,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0425,3.95,0.8512446880340576
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",198,2026268586432,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.91,3.93,0.8197813034057617
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",199,2026268586433,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.93,4.17,0.7258763313293457
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",200,2026268586434,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8925,3.93,0.7560615539550781
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",77,2026266886260,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,62.475,63.55,3.4869933128356934
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",144,2026265186276,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.1,36.08,3.439262628555298
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",201,2026268586435,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.035,3.975,0.7569670677185059
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",137,2026263486218,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.9075,15.88,4.018313884735107
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",202,2026268586436,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9375,3.95,0.725426197052002
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",203,2026268586437,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.015,4.155,0.7459571361541748
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",204,2026268586438,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.9675,4.055,0.7566561698913574
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",78,2026266886261,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,62.325,63.95,3.460298538208008
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",205,2026268586439,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9175,4.085,0.7347159385681152
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",145,2026265186277,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,27.39,36.78,3.421454906463623
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",206,2026268586440,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.855,4.08,0.7262794971466064
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",138,2026263486219,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.18,16.08,4.147451639175415
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",207,2026268586441,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9925,3.98,0.750838041305542
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",208,2026268586442,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.91,4.08,0.7447619438171387
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",209,2026268586443,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.985,4.035,0.7274527549743652
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",79,2026266886262,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,60.25,66.15,3.552637815475464
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",210,2026268586444,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0,4.045,0.7614116668701172
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",146,2026265186278,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.1,35.82,3.5168182849884033
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",211,2026268586445,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.9575,3.96,0.8257803916931152
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",212,2026268586446,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.03,4.025,0.7522768974304199
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",139,2026263486220,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,9.73,15.74,3.9357433319091797
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",213,2026268586447,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8825,4.035,0.8128583431243896
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",80,2026266886263,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,63.625,64.7,3.293750762939453
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",214,2026268586448,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.95,4.09,0.7707626819610596
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",147,2026265186279,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.2,36.4,3.6106956005096436
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",215,2026268586449,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.985,4.105,0.77573561668396
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",216,2026268586450,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.905,4.01,0.7715167999267578
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",217,2026268586451,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9775,4.08,0.7622718811035156
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",140,2026263486221,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.845,15.955,4.0653157234191895
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",81,2026266886264,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,60.975,64.75,3.125749111175537
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",218,2026268586452,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8725,4.045,0.7681725025177002
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",219,2026268586453,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9775,4.1,0.7882795333862305
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",148,2026265186280,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.35,36.28,3.617997646331787
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",220,2026268586454,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9375,4.02,0.76955246925354
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",221,2026268586455,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.9225,4.125,0.785259485244751
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",222,2026268586456,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.98,3.855,0.7651786804199219
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",82,2026266886265,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,60.9,62.55,3.4543139934539795
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",141,2026263486222,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.735,15.845,4.168316841125488
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",223,2026268586457,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.9225,3.985,0.773280143737793
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",224,2026268586458,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.995,3.87,0.8383224010467529
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",149,2026265186281,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.49,36.98,3.818380832672119
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",225,2026268586459,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.95,3.855,0.7535419464111328
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",226,2026268586460,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9775,3.965,0.7535402774810791
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",83,2026266886266,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,60.15,63.25,3.476871967315674
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",227,2026268586461,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.865,3.985,0.7860515117645264
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",142,2026263486223,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.905,15.92,4.244392156600952
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",228,2026268586462,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8975,3.93,0.8853909969329834
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",150,2026265186282,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.53,35.38,3.680413246154785
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",229,2026268586463,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.9375,4.175,0.9375958442687988
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",230,2026268586464,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8675,4.025,0.8478033542633057
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",84,2026266886267,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,60.375,63.05,3.3191428184509277
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",231,2026268586465,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9675,3.955,0.8691110610961914
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",232,2026268586466,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.9275,4.065,0.8232574462890625
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",143,2026263486224,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,9.9675,15.675,4.207014799118042
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",233,2026268586467,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9825,3.94,0.832953929901123
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",151,2026265186283,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.69,36.32,3.6917316913604736
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",234,2026268586468,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9925,4.125,0.8749263286590576
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",85,2026266886268,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,60.975,64.25,3.459757089614868
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",235,2026268586469,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0975,4.025,0.8973937034606934
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",236,2026268586470,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0325,3.99,0.8160326480865479
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",237,2026268586471,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.94,3.875,0.7834551334381104
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",152,2026265186284,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,28.76,36.1,3.7695119380950928
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",144,2026263486225,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.7625,15.9,4.187664985656738
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",238,2026268586472,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9575,3.905,0.8833436965942383
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",239,2026268586473,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.93,4.085,0.766002893447876
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",86,2026266886269,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,63.65,61.5,3.715935707092285
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",240,2026268586474,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.965,4.085,0.742825984954834
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",241,2026268586475,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9325,3.915,0.7629451751708984
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",242,2026268586476,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.95,3.86,0.7399129867553711
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",153,2026265186285,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.22,36.24,3.5558676719665527
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",243,2026268586477,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.985,3.875,0.7324025630950928
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",145,2026263486226,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,10.0375,16.16,4.0961973667144775
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",87,2026266886270,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,62.25,64.3,3.288013458251953
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",244,2026268586478,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.0075,4.015,0.7384662628173828
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",245,2026268586479,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0225,4.035,0.7675902843475342
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",246,2026268586480,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.975,3.97,0.8055610656738281
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",154,2026265186286,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,29.08,35.84,3.5643398761749268
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",247,2026268586481,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.005,3.925,0.7442424297332764
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",248,2026268586482,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0175,4.0,0.7622766494750977
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",88,2026266886271,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,59.875,64.1,3.5150234699249268
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",146,2026263486227,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,9.995,16.045,4.268397808074951
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",249,2026268586483,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.9425,3.94,0.7676122188568115
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",250,2026268586484,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.98,3.945,0.8644375801086426
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",251,2026268586485,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.92,4.215,0.829404354095459
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",155,2026265186287,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.15,36.04,3.5237581729888916
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",252,2026268586486,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.905,4.0,0.8670918941497803
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",89,2026266886272,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,62.425,63.9,3.572988510131836
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",253,2026268586487,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.005,4.1,0.7896640300750732
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",147,2026263486228,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.6875,16.125,3.9243004322052
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",254,2026268586488,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0125,4.135,0.7481887340545654
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",255,2026268586489,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9225,4.15,0.7946946620941162
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",156,2026265186288,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.71,35.56,3.515125274658203
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",256,2026268586490,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.92,3.77,0.8508861064910889
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",90,2026266886273,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,59.925,65.45,3.238673686981201
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",257,2026268586491,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9675,4.03,0.7748012542724609
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",258,2026268586492,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9425,3.935,0.8038325309753418
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",148,2026263486229,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.965,16.09,3.846283435821533
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",259,2026268586493,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9925,3.94,0.772057294845581
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",260,2026268586494,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.5,2,1,1.0175,3.93,0.7900094985961914
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",157,2026265186289,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.45,38.08,3.544266939163208
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",261,2026268586495,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9,3.955,0.7534034252166748
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",91,2026266886274,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,62.025,63.8,3.582026720046997
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",262,2026268586496,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9375,4.09,0.7422587871551514
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",263,2026268586497,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9775,3.865,0.7923104763031006
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",149,2026263486230,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,9.9875,15.96,3.9068593978881836
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",264,2026268586498,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.01,4.045,0.8317699432373047
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",158,2026265186290,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.46,36.68,3.6307644844055176
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",265,2026268586499,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0575,3.98,0.848705530166626
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",92,2026266886275,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,60.925,63.8,3.6790709495544434
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",266,2026268586500,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.9825,4.09,0.8361971378326416
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",267,2026268586501,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9825,3.935,0.8388397693634033
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.001|pi0|k1,1,1",150,2026263486231,B,quiet,5,0.001,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,9.7675,16.05,3.8343796730041504
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",268,2026268586502,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.99,3.945,0.8761458396911621
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",159,2026265186291,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.85,35.9,3.498595952987671
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",269,2026268586503,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9325,3.855,0.8425514698028564
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",93,2026266886276,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.475,64.6,3.6126599311828613
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",270,2026268586504,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.9625,4.02,0.8860807418823242
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",271,2026268586505,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8975,3.99,0.8329265117645264
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",272,2026268586506,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.86,4.035,0.8718357086181641
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",0,2026270286285,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.0025,16.135,3.4903111457824707
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",160,2026265186292,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.14,36.08,3.5547702312469482
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",273,2026268586507,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9575,4.025,0.8720030784606934
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",94,2026266886277,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,63.0,63.35,3.273127555847168
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",274,2026268586508,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0525,3.995,0.865703821182251
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",275,2026268586509,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.995,3.91,0.8696184158325195
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",276,2026268586510,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9375,3.88,0.8405551910400391
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",1,2026270286286,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.7375,16.39,3.328089714050293
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",277,2026268586511,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.895,4.04,0.8176350593566895
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",161,2026265186293,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.77,34.96,3.565725803375244
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",95,2026266886278,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,59.3,62.95,3.6139419078826904
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",278,2026268586512,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.995,4.06,0.8442661762237549
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",279,2026268586513,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0075,4.065,0.8691518306732178
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",280,2026268586514,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9225,3.915,0.8436121940612793
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",2,2026270286287,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.9475,16.065,3.3098037242889404
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",281,2026268586515,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9025,3.825,0.865668535232544
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",162,2026265186294,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.66,36.74,3.4595155715942383
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",282,2026268586516,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0225,3.93,0.8780684471130371
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",96,2026266886279,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,62.1,65.45,3.714630603790283
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",283,2026268586517,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.885,3.875,0.8495051860809326
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",284,2026268586518,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0225,3.865,0.8458225727081299
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",3,2026270286288,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.8625,16.235,3.2958409786224365
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",285,2026268586519,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9,4.095,0.8603088855743408
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",163,2026265186295,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.53,36.38,3.5811591148376465
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",286,2026268586520,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8875,4.155,0.8658018112182617
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",97,2026266886280,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,58.7,64.9,3.5499894618988037
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",287,2026268586521,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.03,4.045,0.9843955039978027
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",4,2026270286289,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.9,16.16,3.4770150184631348
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",288,2026268586522,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0575,4.08,0.8786685466766357
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",289,2026268586523,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9425,4.11,0.7295019626617432
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",164,2026265186296,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.08,37.14,3.5928144454956055
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",290,2026268586524,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.9475,3.86,0.7400252819061279
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",98,2026266886281,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.8,62.8,3.3517251014709473
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",291,2026268586525,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.945,3.79,0.7323307991027832
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",5,2026270286290,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.9825,15.625,2.988854169845581
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",292,2026268586526,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.975,4.05,0.7327888011932373
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",293,2026268586527,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.95,3.99,0.7376630306243896
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",294,2026268586528,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.99,4.0,0.7582979202270508
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",165,2026265186297,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.77,36.04,3.528148889541626
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",295,2026268586529,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.0125,4.085,0.7445919513702393
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",99,2026266886282,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,60.5,64.0,3.4050750732421875
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",296,2026268586530,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.885,3.82,0.7300610542297363
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",6,2026270286291,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.8825,15.915,3.100011110305786
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",297,2026268586531,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.01,4.13,0.7805132865905762
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",298,2026268586532,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.985,4.06,0.7839393615722656
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",299,2026268586533,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8925,4.145,0.737863302230835
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",166,2026265186298,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,28.52,36.92,3.5354371070861816
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",100,2026266886283,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,59.675,62.95,3.495468854904175
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",300,2026268586534,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.995,3.915,0.818162202835083
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",7,2026270286292,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.9875,16.11,3.181696891784668
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",301,2026268586535,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9675,4.1,0.7254636287689209
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",302,2026268586536,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9675,3.98,0.7302141189575195
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",303,2026268586537,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.975,4.015,0.7372837066650391
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",167,2026265186299,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.29,35.66,3.5805881023406982
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",304,2026268586538,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.055,4.045,0.7616288661956787
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",101,2026266886284,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.625,64.0,3.3770880699157715
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",8,2026270286293,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.0825,16.075,3.0325636863708496
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",305,2026268586539,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0025,3.82,0.7308704853057861
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",306,2026268586540,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9375,4.12,0.7313599586486816
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",307,2026268586541,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.955,4.11,0.7855658531188965
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",308,2026268586542,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9825,3.875,0.8154206275939941
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",9,2026270286294,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.8475,15.575,3.104851722717285
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",168,2026265186300,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,28.43,36.26,3.6152167320251465
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",102,2026266886285,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,60.95,62.0,3.398235321044922
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",309,2026268586543,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9325,4.125,0.822678804397583
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",310,2026268586544,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9125,4.155,0.8152892589569092
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",311,2026268586545,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.985,4.185,0.8782181739807129
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",312,2026268586546,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9475,3.92,0.818143367767334
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",10,2026270286295,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.76,16.05,3.228743553161621
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",103,2026266886286,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.5,1,1,62.05,62.65,3.446873188018799
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",169,2026265186301,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,27.99,35.92,3.566394567489624
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",313,2026268586547,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9525,4.045,0.7747008800506592
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",314,2026268586548,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.0175,3.8,0.8990890979766846
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",315,2026268586549,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8725,3.925,0.8205752372741699
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",316,2026268586550,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.99,3.91,0.8412754535675049
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",11,2026270286296,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.9175,15.895,3.292482852935791
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.001|pi0|k1,1,1",170,2026265186302,B,quiet,7,0.001,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,26.58,36.04,3.4041318893432617
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",317,2026268586551,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.035,4.25,0.8597269058227539
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",104,2026266886287,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,59.6,63.45,3.7510788440704346
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",318,2026268586552,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9975,3.965,0.8728249073028564
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",319,2026268586553,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.88,3.97,0.8244545459747314
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",320,2026268586554,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9425,4.06,0.8796098232269287
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",12,2026270286297,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.875,16.12,3.3811705112457275
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",321,2026268586555,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9875,4.0,0.8831837177276611
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",105,2026266886288,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,64.175,65.25,3.7523252964019775
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",0,2026271986336,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.32,35.86,2.921630382537842
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",322,2026268586556,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.895,4.08,0.7431235313415527
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",323,2026268586557,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.965,3.985,0.7576620578765869
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",324,2026268586558,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9625,4.06,0.7874996662139893
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",13,2026270286298,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.145,16.16,3.3116202354431152
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",325,2026268586559,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9275,4.0,0.750469446182251
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",326,2026268586560,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0025,3.955,0.751162052154541
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",1,2026271986337,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.45,35.82,3.1658804416656494
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",106,2026266886289,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.5,1,1,61.55,62.55,3.6229255199432373
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",327,2026268586561,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.04,4.115,0.7445714473724365
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",328,2026268586562,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8975,4.145,0.7386727333068848
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",14,2026270286299,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.015,16.165,3.3631820678710938
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",329,2026268586563,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.0375,3.86,0.7608942985534668
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",330,2026268586564,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.0175,3.93,0.7534909248352051
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",2,2026271986338,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.31,34.24,2.956428289413452
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",331,2026268586565,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.955,3.895,0.7432386875152588
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",107,2026266886290,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,62.85,64.3,3.603231430053711
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",332,2026268586566,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.95,3.865,0.7435071468353271
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",333,2026268586567,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9425,3.995,0.7463572025299072
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",15,2026270286300,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.8075,15.99,3.3444156646728516
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",334,2026268586568,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.9,3.945,0.738490104675293
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",3,2026271986339,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.93,35.06,2.9377634525299072
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",335,2026268586569,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.87,4.095,0.726919412612915
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",108,2026266886291,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,62.725,64.2,3.4278197288513184
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",336,2026268586570,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.95,3.93,0.741673469543457
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",337,2026268586571,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0525,3.96,0.7534477710723877
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",16,2026270286301,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.8075,15.855,3.144094705581665
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",338,2026268586572,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.925,3.965,0.7351009845733643
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",4,2026271986340,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.21,35.38,3.113304615020752
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",339,2026268586573,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9525,4.07,0.7403318881988525
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",340,2026268586574,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.905,4.12,0.7462410926818848
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",109,2026266886292,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.7,64.25,3.432422637939453
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",341,2026268586575,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.015,4.055,0.7661869525909424
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",17,2026270286302,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.89,16.045,3.301248550415039
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",342,2026268586576,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.97,3.975,0.7426855564117432
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",5,2026271986341,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.31,37.34,3.0733230113983154
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",343,2026268586577,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.905,4.19,0.7372288703918457
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",344,2026268586578,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.005,3.81,0.7447326183319092
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",110,2026266886293,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,59.85,65.3,3.3832714557647705
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",345,2026268586579,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.97,4.03,0.7478110790252686
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",346,2026268586580,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.005,3.99,0.7460119724273682
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",18,2026270286303,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.845,16.07,3.275261402130127
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",6,2026271986342,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.44,35.18,3.0643770694732666
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",347,2026268586581,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9775,3.685,0.8238677978515625
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",348,2026268586582,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.96,3.88,0.8724572658538818
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",111,2026266886294,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,62.175,65.2,3.0996506214141846
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",349,2026268586583,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.0175,4.125,0.8354408740997314
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",350,2026268586584,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.9875,3.995,0.7865087985992432
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",7,2026271986343,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.01,36.62,3.1330976486206055
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",19,2026270286304,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.265,16.15,3.4342200756073
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",351,2026268586585,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9775,4.06,0.7740097045898438
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",352,2026268586586,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.935,3.935,0.7399418354034424
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",353,2026268586587,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0425,4.245,0.7380232810974121
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",112,2026266886295,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,60.5,64.0,3.634758472442627
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",354,2026268586588,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9125,4.01,0.7413427829742432
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",8,2026271986344,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.99,35.56,3.0018324851989746
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",20,2026270286305,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.8925,16.225,3.220547676086426
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",355,2026268586589,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.92,4.175,0.7203290462493896
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",356,2026268586590,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.965,4.005,0.7512636184692383
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",357,2026268586591,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0725,3.93,0.7432377338409424
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",358,2026268586592,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9475,4.015,0.7506546974182129
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",113,2026266886296,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,60.725,65.65,3.5362679958343506
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",9,2026271986345,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.72,36.06,3.1672120094299316
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",21,2026270286306,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.1575,15.995,3.0639002323150635
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",359,2026268586593,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8875,4.225,0.7094607353210449
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",360,2026268586594,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.9425,3.97,0.7801942825317383
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",361,2026268586595,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0625,4.15,0.7984564304351807
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",362,2026268586596,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,0.995,4.335,0.7366225719451904
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",10,2026271986346,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.32,36.06,3.2831618785858154
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",114,2026266886297,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,62.5,63.9,3.582045078277588
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",363,2026268586597,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.93,3.915,0.8027844429016113
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",22,2026270286307,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.9375,15.545,3.248802423477173
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",364,2026268586598,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.96,3.905,0.8196752071380615
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",365,2026268586599,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.065,4.19,0.7561159133911133
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",366,2026268586600,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.96,4.3,0.7718472480773926
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",11,2026271986347,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.66,36.64,3.1426663398742676
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",367,2026268586601,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.94,3.965,0.7682662010192871
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",115,2026266886298,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,60.25,63.5,3.441706418991089
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",23,2026270286308,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.0375,16.27,3.414525270462036
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",368,2026268586602,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.0475,4.045,0.7654380798339844
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",369,2026268586603,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.9775,3.865,0.7636480331420898
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",370,2026268586604,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9075,3.945,0.7561118602752686
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",12,2026271986348,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.97,34.62,3.153308629989624
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",371,2026268586605,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9125,4.07,0.7657136917114258
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",116,2026266886299,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,60.8,65.25,3.1198015213012695
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",24,2026270286309,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.655,15.865,3.3424787521362305
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",372,2026268586606,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9475,3.86,0.7764568328857422
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",373,2026268586607,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.11,4.12,0.7735464572906494
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",374,2026268586608,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.05,4.01,0.7553997039794922
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",13,2026271986349,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.37,37.08,2.9332120418548584
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",375,2026268586609,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.965,4.145,0.7686765193939209
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",117,2026266886300,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,62.175,64.9,3.383906602859497
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",376,2026268586610,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.0625,4.06,0.7574961185455322
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",25,2026270286310,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.985,16.25,3.253826856613159
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",377,2026268586611,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.95,4.0,0.7368452548980713
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",378,2026268586612,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.91,4.0,0.7390599250793457
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",14,2026271986350,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.88,36.92,3.1089625358581543
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",379,2026268586613,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.835,4.02,0.7356739044189453
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",380,2026268586614,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.005,4.07,0.7788875102996826
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",26,2026270286311,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.965,15.865,3.397284746170044
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",118,2026266886301,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,62.6,64.15,3.754439115524292
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",381,2026268586615,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9325,4.085,0.7665247917175293
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",382,2026268586616,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.93,4.145,0.7605798244476318
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",15,2026271986351,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.0,36.24,3.10967755317688
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",383,2026268586617,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9725,4.145,0.7554628849029541
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",384,2026268586618,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,2.0,2,0,1.0075,3.89,0.7480387687683105
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",27,2026270286312,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.9025,15.805,3.139817953109741
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",385,2026268586619,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9025,3.86,0.7266883850097656
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",119,2026266886302,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,60.725,63.55,3.378488302230835
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",386,2026268586620,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.905,3.895,0.7526309490203857
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",387,2026268586621,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.92,3.96,0.7355930805206299
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",16,2026271986352,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.28,35.66,3.1671276092529297
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",388,2026268586622,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.935,4.035,0.7481729984283447
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",389,2026268586623,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.88,3.99,0.7290933132171631
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",28,2026270286313,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.7825,16.41,3.3336379528045654
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",390,2026268586624,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.03,3.95,0.7507433891296387
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",120,2026266886303,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,59.05,65.25,3.5884857177734375
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",391,2026268586625,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.045,4.135,0.8131325244903564
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",17,2026271986353,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.74,36.94,3.237938165664673
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",392,2026268586626,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9975,3.985,0.772540807723999
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",393,2026268586627,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9575,4.17,0.7614812850952148
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",29,2026270286314,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.7575,15.71,3.507176399230957
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",394,2026268586628,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.035,4.02,0.7734122276306152
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",121,2026266886304,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.0,64.65,3.720532178878784
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",395,2026268586629,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.0125,3.8,0.7820937633514404
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",18,2026271986354,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.63,34.94,3.2473456859588623
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",396,2026268586630,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.94,4.04,0.7776196002960205
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",397,2026268586631,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9575,3.905,0.7727704048156738
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",30,2026270286315,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.7725,15.86,3.09258770942688
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",398,2026268586632,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.95,4.07,0.7398059368133545
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",399,2026268586633,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9625,3.955,0.7639024257659912
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",122,2026266886305,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,63.65,63.75,3.638559103012085
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",19,2026271986355,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.36,37.04,3.170508623123169
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",400,2026268586634,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8875,4.11,0.7440831661224365
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",401,2026268586635,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9375,3.855,0.7792718410491943
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",31,2026270286316,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.825,15.925,3.1076600551605225
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",402,2026268586636,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.97,4.165,0.8482751846313477
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",403,2026268586637,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9475,4.05,0.8221945762634277
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",123,2026266886306,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,63.325,65.65,3.5169837474823
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",404,2026268586638,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9075,4.005,0.8160319328308105
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",20,2026271986356,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.25,35.32,3.412787914276123
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",405,2026268586639,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0225,3.995,0.8321988582611084
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",32,2026270286317,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.045,16.11,3.354482650756836
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",406,2026268586640,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0025,4.19,0.8569226264953613
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",407,2026268586641,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8875,3.96,0.8519387245178223
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",21,2026271986357,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,15.07,35.64,3.274832010269165
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",124,2026266886307,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.15,61.85,3.4846174716949463
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",408,2026268586642,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.96,4.025,0.8841454982757568
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",409,2026268586643,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0375,3.955,0.8422086238861084
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",33,2026270286318,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,2.0,2,0,4.93,15.98,3.2304489612579346
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",410,2026268586644,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9175,3.89,0.7703630924224854
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",411,2026268586645,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.035,4.025,0.8178737163543701
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",22,2026271986358,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.15,36.04,3.172899007797241
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",412,2026268586646,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.905,4.22,0.748309850692749
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",125,2026266886308,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.325,62.45,3.49458646774292
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",413,2026268586647,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8925,4.0,0.7900784015655518
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",34,2026270286319,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.05,15.875,3.4613471031188965
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",414,2026268586648,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9625,4.06,0.8178198337554932
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",415,2026268586649,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0625,3.825,0.8997907638549805
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",23,2026271986359,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.46,37.12,3.3227922916412354
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",416,2026268586650,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0175,3.96,0.864346981048584
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",126,2026266886309,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.05,62.95,3.7708864212036133
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",417,2026268586651,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.94,4.125,0.7547166347503662
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",418,2026268586652,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9275,3.87,0.751227617263794
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",35,2026270286320,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.9725,15.92,3.5023961067199707
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",419,2026268586653,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8875,4.03,0.771747350692749
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",420,2026268586654,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.99,3.955,0.8666126728057861
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",24,2026271986360,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.96,36.72,3.3671274185180664
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",421,2026268586655,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9825,3.99,0.8234720230102539
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",127,2026266886310,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.925,64.2,3.6132795810699463
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",422,2026268586656,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.92,3.995,0.7570164203643799
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",36,2026270286321,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.845,15.625,3.3993337154388428
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",423,2026268586657,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9675,3.775,0.8030648231506348
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",424,2026268586658,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9025,3.8,0.7751779556274414
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",25,2026271986361,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.45,36.54,3.178990602493286
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",425,2026268586659,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.935,4.0,0.7615501880645752
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",128,2026266886311,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.575,63.75,3.523385524749756
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",426,2026268586660,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9825,3.84,0.8427460193634033
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",37,2026270286322,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.0825,15.97,3.4478185176849365
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",427,2026268586661,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9875,3.95,0.8339545726776123
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",428,2026268586662,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9525,4.07,0.8231916427612305
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",26,2026271986362,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,15.02,35.42,3.396595001220703
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",429,2026268586663,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.955,4.03,0.8189480304718018
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",129,2026266886312,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.225,64.65,3.186521053314209
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",430,2026268586664,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9725,3.955,0.8173182010650635
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",38,2026270286323,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.855,15.82,3.442671775817871
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",431,2026268586665,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.0225,4.06,0.8308162689208984
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",27,2026271986363,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.1,36.28,3.1274094581604004
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",432,2026268586666,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.0075,4.0,0.8752696514129639
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",433,2026268586667,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9675,4.135,0.8359489440917969
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",130,2026266886313,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,60.625,65.1,3.021890640258789
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",434,2026268586668,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.94,3.98,0.735154390335083
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",435,2026268586669,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0575,4.14,0.8249735832214355
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",39,2026270286324,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.14,16.045,3.4346425533294678
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",28,2026271986364,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.75,36.82,3.1229593753814697
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",436,2026268586670,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9375,3.785,0.7389967441558838
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",437,2026268586671,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9725,3.925,0.7481107711791992
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",438,2026268586672,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.955,3.935,0.7859041690826416
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",131,2026266886314,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.8,64.45,3.6069555282592773
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",439,2026268586673,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.88,4.055,0.743175745010376
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",40,2026270286325,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.9975,15.875,3.476431131362915
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",440,2026268586674,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,1.0175,4.08,0.8321900367736816
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",29,2026271986365,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.27,36.06,3.2282001972198486
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",441,2026268586675,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0,4.03,0.8885540962219238
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",442,2026268586676,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.9075,3.955,0.8582766056060791
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",132,2026266886315,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,60.45,63.8,3.5096540451049805
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",443,2026268586677,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.955,3.955,0.7822983264923096
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",41,2026270286326,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,5.1275,15.95,3.480038642883301
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",30,2026271986366,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.31,36.78,3.2236952781677246
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",444,2026268586678,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.9425,4.145,0.755800724029541
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",445,2026268586679,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0025,3.895,0.7933261394500732
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",446,2026268586680,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.01,3.925,0.8203983306884766
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",133,2026266886316,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,62.2,65.55,3.4740824699401855
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",447,2026268586681,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9875,4.01,0.7671012878417969
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",31,2026271986367,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.18,36.16,3.0324628353118896
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",42,2026270286327,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.97,16.07,3.1828558444976807
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",448,2026268586682,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0,3.86,0.7977786064147949
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",449,2026268586683,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.975,4.09,0.8239338397979736
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",450,2026268586684,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.945,4.005,0.8067772388458252
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",451,2026268586685,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.95,3.765,0.8384952545166016
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",134,2026266886317,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,62.825,64.55,3.4808754920959473
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",32,2026271986368,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.72,36.32,3.106929302215576
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",43,2026270286328,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.9075,15.665,3.2225232124328613
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",452,2026268586686,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.06,4.17,0.8508000373840332
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",453,2026268586687,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9,4.09,0.7517364025115967
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",454,2026268586688,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9725,3.935,0.7595226764678955
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",455,2026268586689,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.9125,4.16,0.8058538436889648
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",33,2026271986369,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.08,34.84,3.0254781246185303
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",135,2026266886318,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,62.7,62.5,3.7846415042877197
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",456,2026268586690,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9225,4.07,0.7475955486297607
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",44,2026270286329,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.775,15.905,3.3189518451690674
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",457,2026268586691,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.97,4.025,0.7850570678710938
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",458,2026268586692,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8975,3.97,0.7209062576293945
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",459,2026268586693,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.0025,3.96,0.8474972248077393
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",34,2026271986370,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.54,36.46,3.0576114654541016
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",460,2026268586694,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.915,4.125,0.8403604030609131
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",45,2026270286330,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.9975,15.84,3.333098888397217
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",136,2026266886319,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,63.85,64.1,3.6702792644500732
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",461,2026268586695,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.08,4.155,0.8402924537658691
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",462,2026268586696,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9825,3.965,0.8976278305053711
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",35,2026271986371,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.83,36.96,3.0389420986175537
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",463,2026268586697,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.85,4.005,0.8240571022033691
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",464,2026268586698,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.085,4.1,0.8222765922546387
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",46,2026270286331,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.01,15.755,3.2911055088043213
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",137,2026266886320,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.0,64.35,3.6813817024230957
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",465,2026268586699,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.94,3.835,0.8111963272094727
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",466,2026268586700,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,1.065,3.93,0.8484151363372803
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",36,2026271986372,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.29,35.82,3.1924123764038086
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",467,2026268586701,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.875,3.94,0.8495693206787109
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",468,2026268586702,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.885,4.055,0.8184185028076172
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",47,2026270286332,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.9625,16.045,3.4679017066955566
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",469,2026268586703,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.905,3.975,0.8097062110900879
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",138,2026266886321,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,60.0,66.1,3.6185665130615234
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",470,2026268586704,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.8875,3.925,0.8113572597503662
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",37,2026271986373,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.01,35.88,3.1806886196136475
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",471,2026268586705,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.9,4.0,0.796194314956665
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0005|pi0|k1,1,1",472,2026268586706,B,quiet,3,0.0005,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.99,3.975,0.8139092922210693
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",48,2026270286333,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.8575,15.86,3.4190306663513184
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",139,2026266886322,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.8,62.45,3.7880566120147705
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",38,2026271986374,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.37,35.92,3.2940826416015625
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",49,2026270286334,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.7675,16.195,3.362384557723999
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",140,2026266886323,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,62.1,63.8,3.5996062755584717
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",39,2026271986375,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.95,36.1,3.289966106414795
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",50,2026270286335,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.98,15.88,3.3020365238189697
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",0,2026273686387,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.85,64.75,3.3316378593444824
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",141,2026266886324,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.925,65.65,3.4559783935546875
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",40,2026271986376,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.99,36.06,3.243346691131592
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",51,2026270286336,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.995,15.885,3.1044809818267822
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",1,2026273686388,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,32.15,62.0,3.1059019565582275
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",41,2026271986377,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.69,36.22,2.9700164794921875
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",142,2026266886325,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.575,65.1,3.2781598567962646
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",52,2026270286337,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.0675,15.925,3.2267725467681885
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",2,2026273686389,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.875,64.1,3.2425827980041504
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",42,2026271986378,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.07,36.46,3.2520570755004883
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",143,2026266886326,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.075,63.65,3.6719155311584473
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",3,2026273686390,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.225,63.65,3.2231903076171875
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",53,2026270286338,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.8625,15.895,3.4731860160827637
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",43,2026271986379,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.02,35.62,3.082984209060669
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",144,2026266886327,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,62.075,63.85,3.6388168334960938
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",4,2026273686391,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.05,63.7,3.112302541732788
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",54,2026270286339,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.08,16.17,3.4920308589935303
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",44,2026271986380,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.21,34.82,3.244351625442505
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",145,2026266886328,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.1,63.8,3.5903756618499756
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",5,2026273686392,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.475,63.1,3.1260292530059814
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",55,2026270286340,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.785,16.015,3.3585846424102783
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",45,2026271986381,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.05,35.26,3.185835123062134
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",6,2026273686393,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,29.75,65.2,3.148237705230713
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",146,2026266886329,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,60.875,61.6,3.612682342529297
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",56,2026270286341,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.8775,16.06,3.227813720703125
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",46,2026271986382,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.16,35.14,3.1265463829040527
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",7,2026273686394,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,32.825,61.45,3.1303837299346924
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",147,2026266886330,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,60.75,65.05,3.489494800567627
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",57,2026270286342,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.925,15.91,3.4889118671417236
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",47,2026271986383,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.78,36.96,3.0157458782196045
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",8,2026273686395,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.9,64.8,3.0915069580078125
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",148,2026266886331,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,60.45,63.95,3.563570261001587
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",48,2026271986384,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.5,35.6,3.1477644443511963
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",58,2026270286343,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.83,16.155,3.407562732696533
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",9,2026273686396,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.125,63.9,3.2142300605773926
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",149,2026266886332,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,59.725,62.75,3.28202486038208
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",49,2026271986385,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.98,36.7,3.0513668060302734
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",59,2026270286344,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.9725,16.315,3.3931610584259033
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",10,2026273686397,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.625,63.25,3.1037347316741943
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",50,2026271986386,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.31,34.94,3.081437349319458
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",150,2026266886333,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,63.5,64.2,3.329803466796875
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",60,2026270286345,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.74,16.105,3.4124739170074463
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",11,2026273686398,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,29.7,64.55,2.989590644836426
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",51,2026271986387,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.39,36.58,3.051295518875122
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",151,2026266886334,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.975,67.95,3.5000431537628174
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",61,2026270286346,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.0,16.005,3.409073829650879
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",12,2026273686399,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,32.2,64.9,3.2637174129486084
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",52,2026271986388,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.37,35.3,3.2002744674682617
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",152,2026266886335,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.325,65.35,3.801421642303467
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",62,2026270286347,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.0,15.99,3.264953374862671
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",13,2026273686400,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.275,65.2,3.114694595336914
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",53,2026271986389,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.96,36.0,3.324843406677246
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",63,2026270286348,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.03,16.135,3.0916147232055664
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",153,2026266886336,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,63.575,64.65,3.7261509895324707
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",14,2026273686401,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,29.125,64.25,3.212390422821045
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",54,2026271986390,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.77,35.72,3.264145612716675
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",64,2026270286349,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.8475,16.13,3.1268837451934814
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",15,2026273686402,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.175,63.8,3.317331314086914
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",154,2026266886337,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,63.325,63.45,3.674760341644287
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",55,2026271986391,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.1,36.74,3.189932346343994
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",65,2026270286350,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.8725,15.82,3.3682146072387695
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",16,2026273686403,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,31.95,62.2,3.2333133220672607
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",155,2026266886338,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,60.625,62.75,3.6446642875671387
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",56,2026271986392,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.24,35.98,3.2759087085723877
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",66,2026270286351,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.0975,16.1,3.293572187423706
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",17,2026273686404,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.025,65.3,3.2005815505981445
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",156,2026266886339,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,59.075,63.4,3.663808584213257
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",57,2026271986393,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.07,35.14,3.06075382232666
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",67,2026270286352,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.0225,16.01,3.3407037258148193
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",18,2026273686405,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.725,65.4,3.270061731338501
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",157,2026266886340,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,62.125,63.75,3.4652769565582275
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",58,2026271986394,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.6,36.02,3.3027591705322266
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",68,2026270286353,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.9725,15.61,3.2056148052215576
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",19,2026273686406,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.4,63.35,3.120281934738159
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",59,2026271986395,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.69,35.26,3.2132887840270996
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",158,2026266886341,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.725,66.6,3.2699947357177734
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",69,2026270286354,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.9425,15.855,2.9999914169311523
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",20,2026273686407,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.175,62.95,3.148291826248169
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",60,2026271986396,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.66,36.32,3.178553819656372
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",159,2026266886342,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,60.375,65.45,3.3117473125457764
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",70,2026270286355,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.1425,16.135,3.1032485961914062
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",21,2026273686408,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.125,64.5,3.2903130054473877
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",61,2026271986397,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.75,36.16,3.3282108306884766
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",160,2026266886343,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,60.8,62.3,3.6043550968170166
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",71,2026270286356,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.095,16.13,3.1184158325195312
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",22,2026273686409,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.925,63.65,3.276766777038574
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",62,2026271986398,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.79,36.62,3.262427806854248
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",72,2026270286357,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.935,16.125,3.263741970062256
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",161,2026266886344,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.35,63.2,3.732090711593628
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",23,2026273686410,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.325,65.05,3.239651918411255
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",63,2026271986399,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.92,36.38,3.1004679203033447
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",73,2026270286358,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.83,15.655,3.148406982421875
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",162,2026266886345,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,58.9,63.05,3.7665164470672607
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",24,2026273686411,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.625,63.65,3.151337146759033
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",64,2026271986400,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,15.04,36.48,3.2844560146331787
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",74,2026270286359,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.8725,15.86,3.1540091037750244
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",163,2026266886346,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,60.9,62.9,3.511601686477661
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",25,2026273686412,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.6,65.25,3.3640658855438232
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",65,2026271986401,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.26,35.94,3.50048565864563
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",75,2026270286360,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.925,15.815,3.324582099914551
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",164,2026266886347,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,61.05,64.7,3.333638906478882
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",26,2026273686413,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.95,64.3,3.1930508613586426
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",66,2026271986402,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.94,35.76,3.22597599029541
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",76,2026270286361,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.9275,16.085,3.3713645935058594
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",27,2026273686414,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.35,64.4,3.074885368347168
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",165,2026266886348,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,62.4,61.9,3.9301505088806152
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",67,2026271986403,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.09,35.86,3.3795888423919678
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",77,2026270286362,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.9375,15.985,3.4208266735076904
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",28,2026273686415,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.1,60.75,3.261235237121582
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",166,2026266886349,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,59.375,63.45,3.684370279312134
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",68,2026271986404,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.86,36.56,3.184455394744873
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",78,2026270286363,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.005,15.655,3.193713426589966
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",29,2026273686416,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.65,62.65,3.2428951263427734
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",69,2026271986405,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.45,35.44,3.047701120376587
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",167,2026266886350,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,60.0,63.8,3.6565628051757812
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",79,2026270286364,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.0625,15.975,3.2554967403411865
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",30,2026273686417,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.725,63.1,3.074164390563965
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",70,2026271986406,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.92,35.04,3.028209686279297
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",168,2026266886351,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.125,63.6,3.6911487579345703
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",80,2026270286365,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.005,16.035,3.390159845352173
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",31,2026273686418,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.85,63.4,3.1530792713165283
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",71,2026271986407,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.56,36.78,3.1089069843292236
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",169,2026266886352,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,61.575,66.2,3.631309747695923
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",81,2026270286366,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.96,15.93,3.2457804679870605
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",32,2026273686419,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.9,62.95,3.184319019317627
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",72,2026271986408,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.81,36.24,3.1644814014434814
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",82,2026270286367,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,5.005,16.13,3.31564998626709
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",33,2026273686420,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.875,64.1,3.200988292694092
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",170,2026266886353,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,62.225,66.3,3.6950738430023193
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",73,2026271986409,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.88,36.1,3.3017654418945312
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",83,2026270286368,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,4.9475,16.025,3.2602436542510986
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",34,2026273686421,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.0,61.8,3.2151012420654297
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",171,2026266886354,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,59.95,65.15,3.6607699394226074
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",74,2026271986410,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.55,35.68,3.08132266998291
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",84,2026270286369,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.0275,16.06,3.2884535789489746
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",35,2026273686422,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.7,63.95,3.171278715133667
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.001|pi0|k1,1,1",172,2026266886355,B,quiet,9,0.001,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,60.65,63.9,3.528599739074707
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",75,2026271986411,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.27,35.8,3.16062593460083
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",0,2026275386438,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5575,4.095,0.8028595447540283
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",1,2026275386439,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.61,4.17,0.8122785091400146
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",85,2026270286370,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.8175,16.165,3.1159541606903076
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",36,2026273686423,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.675,67.15,3.118860960006714
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",2,2026275386440,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.555,3.98,0.7982146739959717
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",3,2026275386441,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.57,3.945,0.9212396144866943
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",76,2026271986412,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.06,36.7,3.1570146083831787
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",4,2026275386442,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5575,3.995,0.728327751159668
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",5,2026275386443,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5425,3.93,0.7519187927246094
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",37,2026273686424,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.525,66.35,3.252563238143921
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",86,2026270286371,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.9025,16.415,3.407646417617798
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",6,2026275386444,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5625,4.14,0.7964198589324951
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",7,2026275386445,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6075,4.115,0.7288563251495361
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",77,2026271986413,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.62,35.62,3.195131778717041
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",8,2026275386446,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6025,3.93,0.7298295497894287
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",9,2026275386447,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.535,3.945,0.7462782859802246
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",38,2026273686425,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.85,64.05,3.157773017883301
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",10,2026275386448,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.555,4.0,0.7355577945709229
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",87,2026270286372,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.945,16.105,3.308741807937622
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",11,2026275386449,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.575,3.94,0.733508825302124
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",78,2026271986414,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.15,36.26,3.232065200805664
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",12,2026275386450,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.595,3.84,0.7371087074279785
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",13,2026275386451,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6225,3.985,0.7219855785369873
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",14,2026275386452,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.6075,3.95,0.7726960182189941
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",39,2026273686426,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,29.45,63.55,3.1478443145751953
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",88,2026270286373,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.875,15.795,3.343116283416748
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",15,2026275386453,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5625,4.06,0.7228679656982422
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",16,2026275386454,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.61,4.01,0.7238554954528809
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",79,2026271986415,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.18,35.88,3.13134503364563
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",17,2026275386455,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.52,4.11,0.7550816535949707
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",18,2026275386456,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5925,4.065,0.7861752510070801
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",40,2026273686427,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.05,66.1,3.1446175575256348
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",89,2026270286374,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.845,16.245,3.0687434673309326
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",19,2026275386457,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5775,4.075,0.7810256481170654
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",20,2026275386458,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6225,4.0,0.7280533313751221
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",80,2026271986416,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.08,36.68,3.2499794960021973
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",21,2026275386459,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6575,4.1,0.7419922351837158
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",22,2026275386460,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.575,3.935,0.715303897857666
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",41,2026273686428,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.0,63.85,3.0926623344421387
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",23,2026275386461,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5575,4.065,0.7099792957305908
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",90,2026270286375,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.83,15.975,3.1077890396118164
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",24,2026275386462,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5675,4.095,0.7529394626617432
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",81,2026271986417,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.61,35.84,3.332960367202759
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",25,2026275386463,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5725,4.045,0.7029862403869629
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",26,2026275386464,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6575,3.955,0.7601516246795654
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",42,2026273686429,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,29.925,63.45,3.0856897830963135
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",27,2026275386465,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5775,3.985,0.7362589836120605
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",91,2026270286376,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.9175,16.08,3.1369595527648926
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",28,2026275386466,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.4925,3.945,0.7187709808349609
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",29,2026275386467,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5925,4.01,0.861738920211792
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",82,2026271986418,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.29,36.1,3.4467358589172363
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",30,2026275386468,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5575,3.86,0.823554515838623
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",43,2026273686430,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,29.475,65.05,3.1318793296813965
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",31,2026275386469,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5425,4.07,0.7932446002960205
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",92,2026270286377,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.755,15.985,3.309926748275757
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",32,2026275386470,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.575,3.765,0.7101459503173828
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",33,2026275386471,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.635,3.86,0.7952089309692383
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",83,2026271986419,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.77,35.28,3.2499523162841797
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",34,2026275386472,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5925,3.94,0.7810933589935303
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",44,2026273686431,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.375,64.95,3.178769588470459
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",35,2026275386473,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5925,3.99,0.7522263526916504
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",93,2026270286378,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.87,16.025,3.415219306945801
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",36,2026275386474,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5825,3.97,0.7354683876037598
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",37,2026275386475,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.565,4.14,0.7269532680511475
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",84,2026271986420,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.82,35.82,3.1882338523864746
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",38,2026275386476,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.65,3.915,0.7495372295379639
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",45,2026273686432,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.1,65.8,3.0497794151306152
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",39,2026275386477,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5825,3.915,0.7844283580780029
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",40,2026275386478,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5675,3.985,0.7966339588165283
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",94,2026270286379,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.8625,16.225,3.355485439300537
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",41,2026275386479,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5475,3.995,0.7721612453460693
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",85,2026271986421,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.97,35.62,3.1588706970214844
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",42,2026275386480,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5625,4.09,0.7736063003540039
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",46,2026273686433,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.925,64.95,3.1038384437561035
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",43,2026275386481,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.655,3.825,0.7982468605041504
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",44,2026275386482,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.61,4.02,0.7645671367645264
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",95,2026270286380,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.725,16.08,3.2766382694244385
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",45,2026275386483,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5775,3.995,0.7786812782287598
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",46,2026275386484,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5175,4.13,0.7530601024627686
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",86,2026271986422,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.09,36.44,3.333216905593872
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",47,2026273686434,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.6,63.45,3.1503641605377197
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",47,2026275386485,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.55,3.995,0.7382767200469971
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",48,2026275386486,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.59,4.16,0.7493183612823486
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",96,2026270286381,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.07,15.945,3.100328207015991
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",49,2026275386487,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.62,3.92,0.744802713394165
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",50,2026275386488,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.515,4.26,0.7256665229797363
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",87,2026271986423,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.96,35.52,3.2773425579071045
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",51,2026275386489,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.61,3.875,0.7560617923736572
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",48,2026273686435,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.125,63.6,3.2305026054382324
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",52,2026275386490,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.605,4.09,0.7404360771179199
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",53,2026275386491,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6,3.99,0.7380800247192383
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",97,2026270286382,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,4.97,16.275,3.370340347290039
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",54,2026275386492,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5775,3.92,0.7317521572113037
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",88,2026271986424,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.69,35.98,3.1577107906341553
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",55,2026275386493,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5675,4.13,0.8240013122558594
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",49,2026273686436,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.275,66.0,3.1104633808135986
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",56,2026275386494,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5875,3.965,0.745201826095581
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",57,2026275386495,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.575,4.13,0.8438842296600342
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",98,2026270286383,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.085,15.86,3.1693320274353027
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",58,2026275386496,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.6675,4.005,0.8380262851715088
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",89,2026271986425,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.02,37.34,3.351152181625366
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",59,2026275386497,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.595,4.095,0.8001902103424072
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",50,2026273686437,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,29.075,63.85,3.3096306324005127
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",60,2026275386498,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.58,4.065,0.7789714336395264
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",61,2026275386499,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5275,4.06,0.7647311687469482
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",99,2026270286384,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.83,15.895,3.1224308013916016
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",62,2026275386500,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.58,3.99,0.7731039524078369
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",90,2026271986426,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.23,37.44,3.091139316558838
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",63,2026275386501,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.505,4.17,0.7834415435791016
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",51,2026273686438,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,29.525,63.7,3.2707839012145996
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",64,2026275386502,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5375,4.11,0.784555196762085
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",65,2026275386503,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5225,3.91,0.7723309993743896
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",100,2026270286385,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.9575,15.715,3.0662903785705566
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",66,2026275386504,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.585,4.01,0.8051764965057373
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",67,2026275386505,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.57,3.975,0.7156846523284912
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",91,2026271986427,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.49,35.76,3.2487289905548096
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",52,2026273686439,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.55,64.55,3.1324098110198975
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",68,2026275386506,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.605,3.98,0.7271952629089355
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",69,2026275386507,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.585,3.995,0.7384133338928223
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",101,2026270286386,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.01,15.665,3.6216776371002197
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",70,2026275386508,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.58,4.045,0.7619516849517822
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",71,2026275386509,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5525,4.0,0.7234067916870117
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",92,2026271986428,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.16,36.14,3.3087871074676514
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",53,2026273686440,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,29.15,63.7,3.0693130493164062
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",72,2026275386510,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.62,3.925,0.7190368175506592
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",73,2026275386511,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.575,4.2,0.7349653244018555
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",74,2026275386512,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5125,4.01,0.7208564281463623
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",102,2026270286387,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.9925,15.935,3.3400564193725586
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",75,2026275386513,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.6175,4.185,0.7319424152374268
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",76,2026275386514,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.56,4.13,0.7259156703948975
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",93,2026271986429,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.86,35.74,3.3120763301849365
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",54,2026273686441,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.1,63.25,3.2077019214630127
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",77,2026275386515,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.58,3.96,0.7725043296813965
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",78,2026275386516,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.57,3.965,0.7485339641571045
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",79,2026275386517,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.685,4.02,0.7570416927337646
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",103,2026270286388,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.0125,16.085,3.3942809104919434
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",80,2026275386518,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.53,3.915,0.7251482009887695
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",55,2026273686442,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.975,65.1,3.1747612953186035
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",94,2026271986430,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.29,35.88,3.3477814197540283
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",81,2026275386519,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5025,3.985,0.7306685447692871
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",82,2026275386520,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.605,4.0,0.7480626106262207
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",83,2026275386521,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.555,4.045,0.7309443950653076
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",104,2026270286389,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.945,16.06,3.351653575897217
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",84,2026275386522,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.58,3.99,0.7406148910522461
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",56,2026273686443,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.725,65.9,3.178783893585205
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",95,2026271986431,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.13,35.18,3.1880509853363037
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",85,2026275386523,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.625,4.23,0.7360119819641113
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",86,2026275386524,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6025,3.96,0.7540767192840576
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",87,2026275386525,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.595,4.195,0.749354362487793
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",105,2026270286390,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.825,16.295,3.2597804069519043
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",88,2026275386526,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.555,4.05,0.73685622215271
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",89,2026275386527,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5875,4.105,0.7414753437042236
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",96,2026271986432,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.81,35.56,3.12532901763916
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",57,2026273686444,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.675,62.85,3.1873795986175537
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",90,2026275386528,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.56,4.085,0.788590669631958
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",91,2026275386529,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5675,4.035,0.7554619312286377
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",92,2026275386530,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6325,4.015,0.769127607345581
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",106,2026270286391,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.0,16.255,3.452932596206665
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",93,2026275386531,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5925,3.93,0.7494561672210693
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",97,2026271986433,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.78,36.6,3.2195212841033936
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",58,2026273686445,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,32.4,65.85,3.2453789710998535
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",94,2026275386532,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5925,3.9,0.7834722995758057
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",95,2026275386533,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5725,3.96,0.8258302211761475
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",96,2026275386534,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6025,4.02,0.8372740745544434
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",107,2026270286392,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.08,16.11,3.1665568351745605
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",97,2026275386535,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6525,3.99,0.7767429351806641
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",98,2026271986434,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.12,36.06,3.165658473968506
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",59,2026273686446,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.9,65.3,3.1508090496063232
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",98,2026275386536,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5775,4.01,0.8507070541381836
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",99,2026275386537,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6325,3.945,0.7562446594238281
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",100,2026275386538,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5275,3.98,0.7094964981079102
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",108,2026270286393,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.925,15.96,3.1718790531158447
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",101,2026275386539,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5925,4.045,0.8240368366241455
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",60,2026273686447,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.325,62.6,3.1178650856018066
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",99,2026271986435,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.16,35.84,3.1630606651306152
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",102,2026275386540,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5675,4.095,0.7315857410430908
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",103,2026275386541,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5725,3.99,0.7357625961303711
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",104,2026275386542,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.515,4.095,0.7660667896270752
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",109,2026270286394,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.945,16.255,2.972703218460083
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",105,2026275386543,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.535,3.98,0.7213451862335205
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",61,2026273686448,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,32.275,63.1,3.0713164806365967
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",100,2026271986436,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.18,35.52,3.167003631591797
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",106,2026275386544,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.54,3.915,0.768200159072876
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",107,2026275386545,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5375,3.975,0.7192790508270264
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",108,2026275386546,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5875,4.015,0.715430736541748
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",110,2026270286395,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,5.0025,15.97,3.330533266067505
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",109,2026275386547,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.56,3.955,0.7442398071289062
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",62,2026273686449,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.7,65.5,3.0273427963256836
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",110,2026275386548,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.64,3.93,0.7354047298431396
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",101,2026271986437,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.24,36.58,3.3330342769622803
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",111,2026275386549,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.65,4.195,0.7386393547058105
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",112,2026275386550,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5725,4.005,0.7233009338378906
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",113,2026275386551,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5775,3.96,0.7214725017547607
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",111,2026270286396,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.905,16.145,3.3114986419677734
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",63,2026273686450,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,32.05,64.35,3.0608959197998047
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",114,2026275386552,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5775,3.925,0.7184598445892334
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",102,2026271986438,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.83,36.52,3.052640438079834
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",115,2026275386553,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.57,4.065,0.7653274536132812
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",116,2026275386554,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.6525,4.085,0.784977912902832
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",117,2026275386555,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6375,3.975,0.7730646133422852
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",64,2026273686451,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.2,63.4,3.0464656352996826
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",112,2026270286397,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.0675,16.165,3.3766825199127197
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",118,2026275386556,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.625,3.97,0.7750813961029053
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",103,2026271986439,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.22,35.02,3.1921310424804688
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",119,2026275386557,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5625,4.0,0.7313475608825684
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",120,2026275386558,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5375,4.04,0.717463493347168
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",121,2026275386559,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.58,4.04,0.7054481506347656
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",65,2026273686452,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.65,64.05,3.0897860527038574
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",122,2026275386560,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.575,4.015,0.7263166904449463
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",113,2026270286398,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.8875,16.17,3.2458550930023193
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",104,2026271986440,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.82,36.02,3.316617727279663
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",123,2026275386561,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.4925,4.035,0.7297852039337158
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",124,2026275386562,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6175,4.055,0.7664210796356201
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",125,2026275386563,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.53,4.065,0.7402904033660889
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",66,2026273686453,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.4,63.5,3.0841829776763916
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",126,2026275386564,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5825,3.995,0.8303916454315186
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",114,2026270286399,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.0675,16.215,3.368288516998291
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",127,2026275386565,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.56,3.92,0.7287671566009521
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",105,2026271986441,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.24,35.78,3.134050130844116
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",128,2026275386566,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5275,4.045,0.7283060550689697
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",129,2026275386567,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6,4.01,0.7242441177368164
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",130,2026275386568,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5425,4.12,0.7199246883392334
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",67,2026273686454,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.425,63.55,3.116425037384033
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",115,2026270286400,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.755,16.595,2.985589027404785
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",131,2026275386569,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.565,3.865,0.7469601631164551
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",106,2026271986442,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.25,34.96,3.2427845001220703
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",132,2026275386570,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5525,4.095,0.7775626182556152
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",133,2026275386571,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.575,4.1,0.7877781391143799
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",134,2026275386572,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.55,3.785,0.7896604537963867
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",68,2026273686455,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,29.875,65.45,3.1195662021636963
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",116,2026270286401,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.895,16.095,3.1479976177215576
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",135,2026275386573,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.56,3.915,0.7220101356506348
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",107,2026271986443,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.05,36.02,3.1073079109191895
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",136,2026275386574,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.59,4.02,0.7318534851074219
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",137,2026275386575,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6025,4.035,0.7366008758544922
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",138,2026275386576,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6175,3.925,0.753441333770752
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",69,2026273686456,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.85,64.05,3.1592330932617188
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",139,2026275386577,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5325,4.095,0.7399694919586182
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",117,2026270286402,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.9375,16.06,3.2773234844207764
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",108,2026271986444,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.19,36.12,3.344773054122925
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",140,2026275386578,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.63,4.02,0.7942752838134766
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",141,2026275386579,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.565,4.095,0.780022144317627
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",142,2026275386580,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.58,4.06,0.8286571502685547
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",70,2026273686457,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,29.625,64.1,3.157181978225708
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",143,2026275386581,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5525,3.96,0.7596640586853027
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",118,2026270286403,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.8625,16.24,3.3014590740203857
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",144,2026275386582,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6275,4.04,0.8072435855865479
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",109,2026271986445,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.98,36.78,3.2263083457946777
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",145,2026275386583,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6575,4.06,0.7815606594085693
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",146,2026275386584,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5525,4.105,0.7860620021820068
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",71,2026273686458,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.25,63.85,3.162306070327759
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",147,2026275386585,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.54,3.975,0.787076473236084
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",119,2026270286404,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.0525,16.355,3.4298574924468994
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",148,2026275386586,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.575,4.1,0.8199677467346191
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",110,2026271986446,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.24,35.44,3.3068857192993164
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",149,2026275386587,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6675,4.025,0.7871181964874268
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",150,2026275386588,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5525,3.965,0.7640395164489746
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",72,2026273686459,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.475,63.4,3.384627342224121
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",151,2026275386589,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.57,4.03,0.768996000289917
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",152,2026275386590,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5975,3.86,0.7553620338439941
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",120,2026270286405,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.8675,16.37,3.4146008491516113
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",111,2026271986447,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.68,35.58,3.2540109157562256
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",153,2026275386591,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.53,3.91,0.7863016128540039
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",154,2026275386592,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5975,3.99,0.8430385589599609
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",155,2026275386593,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.59,4.115,0.8231489658355713
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",73,2026273686460,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,29.775,64.05,3.371260404586792
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",156,2026275386594,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6075,3.9,0.7940921783447266
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",121,2026270286406,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.95,15.64,3.3596956729888916
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",112,2026271986448,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.28,36.14,3.231117010116577
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",157,2026275386595,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5725,3.9,0.774975061416626
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",158,2026275386596,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5675,3.93,0.8053538799285889
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",159,2026275386597,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.62,3.985,0.8277809619903564
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",74,2026273686461,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.6,64.5,3.263338088989258
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",160,2026275386598,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5225,3.88,0.7674350738525391
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",113,2026271986449,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.34,35.34,3.301218032836914
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",122,2026270286407,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.02,15.95,3.576848030090332
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",161,2026275386599,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5325,4.01,0.7055573463439941
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",162,2026275386600,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6425,4.03,0.7326841354370117
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",163,2026275386601,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.535,4.09,0.7236354351043701
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",75,2026273686462,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.05,61.4,3.177314043045044
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",164,2026275386602,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5225,4.14,0.7684786319732666
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",114,2026271986450,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.61,36.54,3.103395462036133
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",165,2026275386603,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5825,4.015,0.7231297492980957
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",123,2026270286408,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.7975,16.16,3.0941219329833984
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",166,2026275386604,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.535,3.935,0.7170631885528564
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",167,2026275386605,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6225,3.98,0.7594237327575684
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",76,2026273686463,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,27.6,63.8,3.337879180908203
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",168,2026275386606,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6675,4.01,0.7358815670013428
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",169,2026275386607,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6175,4.1,0.7193295955657959
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",115,2026271986451,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.51,35.4,3.170870780944824
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",124,2026270286409,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.7575,15.955,3.123425245285034
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",170,2026275386608,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6225,4.065,0.7237038612365723
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",171,2026275386609,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6475,3.835,0.7242021560668945
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",172,2026275386610,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.56,3.9,0.7253601551055908
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",77,2026273686464,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,29.875,63.5,3.2784862518310547
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",173,2026275386611,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.52,4.065,0.7148122787475586
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",116,2026271986452,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.66,35.78,3.2695119380950928
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",174,2026275386612,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5125,3.995,0.7302870750427246
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",125,2026270286410,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.9975,15.8,3.4259324073791504
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",175,2026275386613,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.58,4.095,0.718745231628418
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",176,2026275386614,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.68,3.995,0.7431380748748779
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",78,2026273686465,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.175,62.95,3.2761263847351074
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",177,2026275386615,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5575,3.835,0.7461671829223633
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",178,2026275386616,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6225,3.845,0.738072395324707
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",117,2026271986453,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.11,35.96,3.200838088989258
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",126,2026270286411,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.18,15.785,3.523245096206665
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",179,2026275386617,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5175,3.945,0.7096590995788574
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",180,2026275386618,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.595,3.88,0.7080543041229248
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",181,2026275386619,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5875,3.85,0.7214136123657227
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",79,2026273686466,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.9,65.7,3.3216071128845215
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",182,2026275386620,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.555,3.99,0.78322434425354
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",118,2026271986454,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.5,35.04,3.2809650897979736
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",183,2026275386621,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5975,4.025,0.7422821521759033
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",127,2026270286412,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.0375,15.945,3.3100838661193848
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",184,2026275386622,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6125,3.975,0.7411127090454102
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",185,2026275386623,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5,3.9,0.7261126041412354
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",186,2026275386624,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5725,3.945,0.7759158611297607
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",80,2026273686467,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,29.525,64.85,3.424293279647827
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",119,2026271986455,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.36,35.46,3.103395700454712
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",187,2026275386625,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.6075,3.895,0.8229677677154541
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",128,2026270286413,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,4.8475,15.86,3.1531403064727783
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",188,2026275386626,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6825,3.775,0.8044991493225098
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",189,2026275386627,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.58,4.02,0.7775661945343018
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",190,2026275386628,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5925,3.845,0.7903785705566406
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",81,2026273686468,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.35,64.85,3.3897464275360107
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",120,2026271986456,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.31,35.92,3.159609794616699
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",191,2026275386629,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5775,4.085,0.8331038951873779
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",129,2026270286414,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.92,16.05,3.2565975189208984
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",192,2026275386630,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.52,4.135,0.7643711566925049
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",193,2026275386631,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.57,4.11,0.7790133953094482
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",194,2026275386632,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.585,3.935,0.7270400524139404
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",82,2026273686469,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,29.525,62.85,3.268528699874878
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",121,2026271986457,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.19,35.3,3.113715887069702
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",195,2026275386633,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.52,3.97,0.7210235595703125
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",130,2026270286415,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.7725,16.245,3.2561421394348145
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",196,2026275386634,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5725,4.055,0.7746984958648682
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",197,2026275386635,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.525,4.195,0.738560676574707
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",198,2026275386636,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.605,4.17,0.7634162902832031
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",122,2026271986458,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.88,36.56,3.0935373306274414
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",83,2026273686470,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.85,61.7,3.3166017532348633
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",199,2026275386637,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5325,4.0,0.7481997013092041
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",200,2026275386638,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.625,3.945,0.7562456130981445
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",131,2026270286416,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,4.9075,15.975,3.2144956588745117
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",201,2026275386639,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.645,3.795,0.754974365234375
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",202,2026275386640,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.6,4.17,0.8053092956542969
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",123,2026271986459,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.14,36.32,3.2887682914733887
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",203,2026275386641,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.58,3.88,0.7804903984069824
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",84,2026273686471,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.45,66.7,3.1637110710144043
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",204,2026275386642,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.645,4.015,0.7295913696289062
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",132,2026270286417,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.895,15.925,3.59427809715271
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",205,2026275386643,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5775,3.975,0.7646811008453369
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",206,2026275386644,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.555,4.07,0.7888870239257812
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",207,2026275386645,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.585,3.9,0.8073611259460449
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",124,2026271986460,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.19,36.5,3.306896448135376
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",85,2026273686472,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,33.175,63.8,3.4399521350860596
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",208,2026275386646,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.595,3.925,0.7317109107971191
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",209,2026275386647,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5675,3.88,0.7319786548614502
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",133,2026270286418,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.9175,16.37,3.3698527812957764
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",210,2026275386648,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6175,3.885,0.724628210067749
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",211,2026275386649,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.605,4.05,0.7099599838256836
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",125,2026271986461,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.11,35.64,3.0342540740966797
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",86,2026273686473,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.375,62.2,3.195791482925415
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",212,2026275386650,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5975,4.06,0.711561918258667
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",213,2026275386651,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5275,3.975,0.7849335670471191
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",134,2026270286419,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.8825,15.655,3.3162312507629395
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",214,2026275386652,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.6025,3.94,0.7316603660583496
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",215,2026275386653,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5475,3.92,0.7300379276275635
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",126,2026271986462,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.46,34.98,3.0756704807281494
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",216,2026275386654,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.505,4.005,0.7490406036376953
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",87,2026273686474,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.025,64.7,3.3236441612243652
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",217,2026275386655,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5725,4.135,0.7432682514190674
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",218,2026275386656,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5775,4.02,0.7582075595855713
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",135,2026270286420,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.965,15.895,3.3833115100860596
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",219,2026275386657,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.56,4.09,0.7463512420654297
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",127,2026271986463,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.44,36.12,3.273648977279663
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",220,2026275386658,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5975,3.94,0.7390124797821045
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",88,2026273686475,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.3,64.65,3.1246180534362793
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",221,2026275386659,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6025,3.97,0.7357079982757568
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",222,2026275386660,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5975,3.875,0.7801313400268555
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",136,2026270286421,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.96,16.155,3.239806652069092
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",223,2026275386661,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6375,3.935,0.8012902736663818
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",224,2026275386662,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.565,3.96,0.7363436222076416
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",128,2026271986464,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.76,35.52,3.203650712966919
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",89,2026273686476,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.85,66.85,3.2936177253723145
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",225,2026275386663,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.62,3.905,0.7473547458648682
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",226,2026275386664,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.595,3.98,0.7414858341217041
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",137,2026270286422,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.915,15.925,3.125824451446533
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",227,2026275386665,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.675,3.915,0.7363288402557373
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",228,2026275386666,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.565,4.01,0.7158336639404297
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",129,2026271986465,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.88,35.5,3.039060115814209
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",229,2026275386667,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.5,2,1,0.6325,3.905,0.7531478404998779
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",90,2026273686477,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.075,64.45,3.216507911682129
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",230,2026275386668,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.63,4.075,0.7268493175506592
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",231,2026275386669,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.565,4.14,0.7197344303131104
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",138,2026270286423,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.98,16.01,3.1823997497558594
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",232,2026275386670,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.605,4.02,0.7315692901611328
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",130,2026271986466,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.29,36.0,3.1893763542175293
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",233,2026275386671,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5825,4.025,0.7312767505645752
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",91,2026273686478,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.65,64.45,3.1527836322784424
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",234,2026275386672,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.625,3.81,0.7311568260192871
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",235,2026275386673,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6275,4.14,0.7425780296325684
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",139,2026270286424,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.0175,16.155,3.1720898151397705
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",236,2026275386674,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.64,4.13,0.7289791107177734
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",237,2026275386675,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.625,3.88,0.7485578060150146
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",131,2026271986467,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.74,35.36,3.258852958679199
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",92,2026273686479,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.475,63.05,3.1934096813201904
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",238,2026275386676,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.615,3.955,0.7218058109283447
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",239,2026275386677,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.59,4.015,0.7378082275390625
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",140,2026270286425,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.9175,15.585,3.19583797454834
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",240,2026275386678,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5925,3.845,0.8225786685943604
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",241,2026275386679,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5225,3.885,0.7528197765350342
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",132,2026271986468,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.46,36.5,3.2378928661346436
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",242,2026275386680,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5,3.845,0.7493462562561035
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",93,2026273686480,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,29.225,65.5,3.153395891189575
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",243,2026275386681,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.565,4.06,0.7704861164093018
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",141,2026270286426,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.1425,15.92,3.1357662677764893
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",244,2026275386682,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.57,4.085,0.7874517440795898
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",245,2026275386683,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5625,3.935,0.7617385387420654
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",133,2026271986469,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.25,36.52,3.1605331897735596
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",246,2026275386684,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5575,3.975,0.772864580154419
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",94,2026273686481,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,29.625,62.75,3.1304244995117188
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",247,2026275386685,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6325,3.955,0.8522605895996094
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",142,2026270286427,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.94,15.67,3.1307246685028076
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",248,2026275386686,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.54,3.9,0.7810642719268799
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",249,2026275386687,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.52,3.885,0.746532678604126
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",134,2026271986470,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.42,36.16,3.14168381690979
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",250,2026275386688,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.535,4.085,0.755629301071167
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",95,2026273686482,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.475,64.05,3.1196250915527344
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",251,2026275386689,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.635,3.825,0.7668297290802002
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",252,2026275386690,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.575,4.045,0.7938504219055176
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",143,2026270286428,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.11,15.77,3.292231798171997
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",253,2026275386691,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5975,4.18,0.8951201438903809
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",135,2026271986471,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.19,36.32,3.2585246562957764
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",96,2026273686483,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.175,64.85,3.136691093444824
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",254,2026275386692,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.52,3.915,0.8230781555175781
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",255,2026275386693,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6,3.995,0.8159492015838623
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",256,2026275386694,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5575,4.02,0.771662712097168
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",144,2026270286429,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.945,16.175,3.509059190750122
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",257,2026275386695,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5175,4.19,0.74772047996521
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",136,2026271986472,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.07,35.42,3.0588157176971436
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",258,2026275386696,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.56,4.04,0.7935914993286133
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",97,2026273686484,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.275,66.6,3.281057357788086
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",259,2026275386697,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.555,3.96,0.822303295135498
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",260,2026275386698,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6,4.08,0.7340753078460693
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",145,2026270286430,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.88,16.21,3.248380184173584
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",261,2026275386699,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.53,4.09,0.7432971000671387
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",137,2026271986473,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.11,36.98,3.143855333328247
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",262,2026275386700,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6025,3.94,0.7568645477294922
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",98,2026273686485,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.225,65.0,3.237973928451538
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",263,2026275386701,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.545,3.865,0.8322267532348633
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",264,2026275386702,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.57,3.96,0.7560336589813232
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",146,2026270286431,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.845,16.155,3.199941635131836
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",265,2026275386703,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.6075,3.99,0.7496199607849121
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",138,2026271986474,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.08,34.84,3.284444808959961
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",266,2026275386704,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.595,3.935,0.7876496315002441
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",99,2026273686486,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.25,65.5,3.3255746364593506
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",267,2026275386705,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5675,4.025,0.8159916400909424
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",268,2026275386706,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.53,3.875,0.7857539653778076
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",269,2026275386707,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.565,4.2,0.785341739654541
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",147,2026270286432,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.9125,15.65,3.486672878265381
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",139,2026271986475,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.11,36.72,3.3635237216949463
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",270,2026275386708,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5975,4.225,0.7772314548492432
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",100,2026273686487,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.375,61.95,3.2078139781951904
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",271,2026275386709,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6,4.03,0.8011786937713623
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",272,2026275386710,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5925,3.95,0.7792162895202637
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",273,2026275386711,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.64,3.88,0.7569591999053955
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",148,2026270286433,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.9825,15.935,3.3929169178009033
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",140,2026271986476,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.91,36.12,3.0970427989959717
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",274,2026275386712,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5925,4.08,0.754234790802002
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",101,2026273686488,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.275,63.9,3.1257474422454834
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",275,2026275386713,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.49,4.1,0.750802755355835
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",276,2026275386714,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5075,3.98,0.7387893199920654
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",277,2026275386715,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5025,4.175,0.7457890510559082
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",149,2026270286434,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.985,16.0,3.133455514907837
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",141,2026271986477,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.97,35.66,2.984766721725464
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",278,2026275386716,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.505,4.0,0.7483658790588379
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",102,2026273686489,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.35,62.05,3.2117812633514404
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",279,2026275386717,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5675,3.91,0.748692512512207
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",280,2026275386718,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.59,4.075,0.7713677883148193
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",281,2026275386719,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6275,3.85,0.8154230117797852
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",282,2026275386720,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.59,3.94,0.7752504348754883
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",150,2026270286435,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.9425,15.715,3.536379337310791
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",142,2026271986478,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.82,36.82,3.3777480125427246
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",283,2026275386721,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.52,3.835,0.8039383888244629
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",103,2026273686490,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,29.95,64.5,3.3893978595733643
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",284,2026275386722,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.555,4.145,0.737666130065918
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",285,2026275386723,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.525,4.005,0.7515604496002197
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",286,2026275386724,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5725,4.2,0.7491960525512695
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",143,2026271986479,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.77,35.46,3.0196163654327393
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",151,2026270286436,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.015,15.835,3.268921375274658
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",287,2026275386725,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5275,4.03,0.7588574886322021
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",104,2026273686491,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.5,60.9,3.1630821228027344
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",288,2026275386726,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5925,4.04,0.8266768455505371
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",289,2026275386727,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.63,4.005,0.8013699054718018
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",290,2026275386728,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.55,3.92,0.8071551322937012
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",144,2026271986480,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.66,36.14,3.141277551651001
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",152,2026270286437,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.0,15.88,3.199491024017334
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",291,2026275386729,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.64,4.095,0.8650691509246826
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",105,2026273686492,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,29.2,64.65,3.1609342098236084
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",292,2026275386730,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5725,4.045,0.8533322811126709
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",293,2026275386731,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.525,3.915,0.8377108573913574
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",294,2026275386732,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5525,3.91,0.8156836032867432
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",145,2026271986481,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.94,35.62,3.3630619049072266
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",153,2026270286438,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.785,16.215,3.2748639583587646
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",295,2026275386733,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.56,3.775,0.7685470581054688
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",106,2026273686493,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.125,64.5,3.1680428981781006
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",296,2026275386734,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.62,4.03,0.8168518543243408
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",297,2026275386735,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.55,4.125,0.7856268882751465
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",146,2026271986482,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.91,36.24,3.0656819343566895
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",298,2026275386736,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5675,3.845,0.7764327526092529
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",154,2026270286439,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.04,16.15,3.37528133392334
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",107,2026273686494,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.55,63.7,2.9542949199676514
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",299,2026275386737,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6025,3.865,0.8331794738769531
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",300,2026275386738,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6075,4.235,0.7380754947662354
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",301,2026275386739,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5975,3.895,0.7353239059448242
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",302,2026275386740,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.585,3.99,0.729865312576294
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",147,2026271986483,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.99,37.02,3.100553035736084
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",108,2026273686495,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.2,65.3,3.063149929046631
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",303,2026275386741,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5775,4.08,0.7214746475219727
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",155,2026270286440,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,4.9675,15.765,3.451470136642456
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",304,2026275386742,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5275,4.05,0.7194874286651611
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",305,2026275386743,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5825,4.085,0.8424232006072998
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",306,2026275386744,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5325,4.095,0.7940566539764404
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",148,2026271986484,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.95,35.22,3.241244077682495
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",307,2026275386745,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5925,3.86,0.7210121154785156
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",109,2026273686496,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.125,63.95,3.1175589561462402
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",156,2026270286441,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.02,16.255,3.3236231803894043
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",308,2026275386746,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.635,4.165,0.7396340370178223
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",309,2026275386747,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6475,3.935,0.8092939853668213
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",310,2026275386748,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.53,3.905,0.7079012393951416
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",149,2026271986485,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.81,35.26,3.1509897708892822
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",311,2026275386749,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.575,3.88,0.7243180274963379
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",110,2026273686497,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.55,64.5,3.010579824447632
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",157,2026270286442,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.845,15.795,3.253241539001465
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",312,2026275386750,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5075,3.8,0.70577073097229
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",313,2026275386751,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5325,3.845,0.7857615947723389
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",314,2026275386752,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.615,4.07,0.7543618679046631
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",150,2026271986486,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.21,36.7,3.2464635372161865
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",315,2026275386753,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5925,3.99,0.747779130935669
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",111,2026273686498,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.7,62.9,3.016286849975586
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",316,2026275386754,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.58,4.065,0.7396609783172607
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",158,2026270286443,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.84,16.35,3.147150993347168
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",317,2026275386755,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5625,4.15,0.8686504364013672
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",318,2026275386756,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5975,3.955,0.8170585632324219
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",151,2026271986487,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.55,35.86,3.2030107975006104
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",112,2026273686499,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.9,62.9,3.1414084434509277
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",319,2026275386757,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6325,4.12,0.8096709251403809
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",320,2026275386758,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.625,4.08,0.8525474071502686
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",159,2026270286444,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.925,16.135,3.327573537826538
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",321,2026275386759,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5675,3.945,0.786919116973877
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",322,2026275386760,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.505,3.995,0.7032716274261475
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",152,2026271986488,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.56,36.64,3.0690689086914062
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",323,2026275386761,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5625,4.145,0.7366623878479004
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",113,2026273686500,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.75,62.9,3.2843945026397705
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",324,2026275386762,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.6325,3.925,0.7808947563171387
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",160,2026270286445,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.8775,16.19,3.363898992538452
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",325,2026275386763,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.655,3.895,0.8067193031311035
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",326,2026275386764,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5625,3.965,0.7847023010253906
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",153,2026271986489,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.69,35.9,3.04240345954895
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",327,2026275386765,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6225,3.96,0.8049120903015137
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",114,2026273686501,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.125,65.0,3.2880303859710693
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",328,2026275386766,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.545,4.11,0.8443865776062012
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",161,2026270286446,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.0075,16.0,3.4277260303497314
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",329,2026275386767,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6625,4.115,0.7426586151123047
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",330,2026275386768,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5625,4.11,0.7635722160339355
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",154,2026271986490,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.65,35.5,3.207430839538574
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",331,2026275386769,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5575,4.11,0.7708725929260254
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",115,2026273686502,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.65,64.7,3.202965497970581
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",332,2026275386770,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.6025,4.0,0.7871637344360352
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",333,2026275386771,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.4575,3.905,0.7685613632202148
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",162,2026270286447,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.855,15.76,3.2953977584838867
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",334,2026275386772,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5275,3.935,0.7668991088867188
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",155,2026271986491,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.58,36.56,3.2627146244049072
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",335,2026275386773,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5125,3.99,0.7630870342254639
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",116,2026273686503,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,1.0,1,0,30.975,60.55,3.1929123401641846
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",336,2026275386774,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.57,4.03,0.7675602436065674
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",337,2026275386775,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5375,4.09,0.7711737155914307
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",163,2026270286448,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.97,16.215,3.2892494201660156
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",338,2026275386776,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6175,4.155,0.7720649242401123
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",156,2026271986492,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.5,36.24,3.1058835983276367
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",339,2026275386777,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5325,4.09,0.7992911338806152
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",117,2026273686504,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,29.175,60.0,2.9808638095855713
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",340,2026275386778,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.61,4.07,0.8167574405670166
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",341,2026275386779,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5825,4.075,0.7088868618011475
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",164,2026270286449,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.81,16.07,3.1114604473114014
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",342,2026275386780,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5875,3.94,0.7409152984619141
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",343,2026275386781,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5725,3.875,0.7604629993438721
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",157,2026271986493,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.5,36.28,3.1486141681671143
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",118,2026273686505,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,29.975,64.8,3.115530490875244
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",344,2026275386782,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5275,4.24,0.7250874042510986
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",345,2026275386783,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6075,4.05,0.7965435981750488
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",165,2026270286450,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.655,16.225,3.172016143798828
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",346,2026275386784,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5375,4.185,0.7016410827636719
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",347,2026275386785,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.6525,4.125,0.7114512920379639
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",158,2026271986494,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.87,35.86,3.0369622707366943
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",348,2026275386786,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6175,4.07,0.7418699264526367
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",119,2026273686506,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.8,64.9,3.3602325916290283
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",349,2026275386787,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.565,4.01,0.7221412658691406
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",350,2026275386788,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.605,4.045,0.7067286968231201
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",166,2026270286451,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.0275,15.73,3.499282121658325
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",351,2026275386789,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.61,4.135,0.7561185359954834
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",159,2026271986495,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.21,36.42,3.1084697246551514
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",352,2026275386790,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6425,4.175,0.7774016857147217
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",120,2026273686507,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.1,62.8,3.266839027404785
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",353,2026275386791,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.595,3.99,0.7275233268737793
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",354,2026275386792,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.605,4.05,0.7209045886993408
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",167,2026270286452,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.9575,15.84,3.2531135082244873
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",355,2026275386793,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.545,3.995,0.7209148406982422
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",160,2026271986496,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.82,34.94,3.194617509841919
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",356,2026275386794,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5275,4.12,0.7171621322631836
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",121,2026273686508,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,32.625,64.3,3.1575770378112793
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",357,2026275386795,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.57,3.945,0.7467310428619385
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",358,2026275386796,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.58,4.12,0.8577897548675537
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",359,2026275386797,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6075,4.075,0.7957117557525635
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",168,2026270286453,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.735,15.745,3.2985386848449707
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",161,2026271986497,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.23,35.8,3.313035488128662
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",360,2026275386798,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.605,3.935,0.7433209419250488
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",361,2026275386799,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.525,4.015,0.7043488025665283
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",122,2026273686509,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.2,61.05,3.2778587341308594
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",362,2026275386800,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.61,3.865,0.7075293064117432
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",363,2026275386801,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.525,3.775,0.75620436668396
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",169,2026270286454,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.985,15.955,3.127467393875122
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",364,2026275386802,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6025,4.045,0.7082729339599609
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",162,2026271986498,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.64,36.34,3.1192476749420166
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",365,2026275386803,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.615,4.095,0.7336404323577881
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",123,2026273686510,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.625,63.45,3.1841957569122314
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",366,2026275386804,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6675,3.905,0.7793397903442383
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",367,2026275386805,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5575,4.035,0.7057461738586426
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",170,2026270286455,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.0325,15.82,3.1333770751953125
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",368,2026275386806,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.525,3.92,0.703467845916748
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",163,2026271986499,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.5,1,1,14.38,36.5,3.2107675075531006
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",369,2026275386807,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.555,3.99,0.7091913223266602
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",370,2026275386808,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.565,4.135,0.7503106594085693
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",124,2026273686511,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.8,67.2,3.3206875324249268
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",371,2026275386809,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5825,3.99,0.7387127876281738
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",372,2026275386810,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5475,4.02,0.860379695892334
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",171,2026270286456,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.765,16.305,3.3514697551727295
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",164,2026271986500,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.98,36.56,3.3085415363311768
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",373,2026275386811,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.575,4.055,0.7852358818054199
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",374,2026275386812,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5925,3.96,0.8305559158325195
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",125,2026273686512,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.25,63.5,3.229942560195923
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",375,2026275386813,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5725,4.1,0.8016180992126465
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",376,2026275386814,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.56,4.0,0.7710182666778564
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",172,2026270286457,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.8525,15.99,3.4440414905548096
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",165,2026271986501,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.6,35.9,3.1047329902648926
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",377,2026275386815,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.55,3.91,0.7977259159088135
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",378,2026275386816,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.575,4.135,0.7951986789703369
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",126,2026273686513,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,29.4,65.0,3.1278271675109863
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",379,2026275386817,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5125,3.92,0.7822868824005127
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",380,2026275386818,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6125,4.035,0.7733767032623291
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",173,2026270286458,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.975,16.305,3.386622905731201
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",166,2026271986502,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.03,35.98,3.209625720977783
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",381,2026275386819,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6325,3.935,0.7763395309448242
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",382,2026275386820,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6,3.785,0.7505998611450195
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",127,2026273686514,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.475,62.4,3.3040947914123535
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",383,2026275386821,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.66,3.905,0.7737820148468018
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",384,2026275386822,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5925,4.09,0.7926831245422363
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",167,2026271986503,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.72,36.02,3.087299108505249
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",385,2026275386823,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.605,3.94,0.779334306716919
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",174,2026270286459,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,4.8925,16.365,3.503714084625244
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",386,2026275386824,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.565,4.005,0.7849142551422119
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",128,2026273686515,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,29.75,64.25,3.2543985843658447
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",387,2026275386825,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5625,4.05,0.7511696815490723
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",388,2026275386826,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5975,3.905,0.7486202716827393
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",389,2026275386827,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.53,4.01,0.710089921951294
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",168,2026271986504,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.34,35.34,3.12436842918396
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",175,2026270286460,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.0625,16.45,3.1917073726654053
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",390,2026275386828,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.6125,3.905,0.7388875484466553
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",129,2026273686516,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.075,64.05,3.0749075412750244
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",391,2026275386829,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5425,4.045,0.7390947341918945
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",392,2026275386830,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5575,4.0,0.7143888473510742
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",393,2026275386831,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5675,4.03,0.7238454818725586
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",169,2026271986505,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.59,35.94,3.3668417930603027
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",176,2026270286461,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.07,15.91,3.2964272499084473
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",394,2026275386832,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.565,4.135,0.7323265075683594
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",130,2026273686517,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.325,62.75,3.1679775714874268
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",395,2026275386833,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5425,3.945,0.7330355644226074
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",396,2026275386834,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.505,3.955,0.9321248531341553
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",397,2026275386835,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5675,4.025,0.8174784183502197
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",170,2026271986506,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,1.0,1,0,14.56,35.86,3.3752634525299072
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",398,2026275386836,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.535,3.99,0.8048374652862549
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",177,2026270286462,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.1225,16.325,3.320666790008545
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",131,2026273686518,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.675,64.75,3.2285547256469727
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",399,2026275386837,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5675,4.255,0.7721133232116699
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",400,2026275386838,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.59,4.0,0.7119312286376953
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",401,2026275386839,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.605,3.945,0.7224349975585938
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",402,2026275386840,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5575,4.15,0.7232508659362793
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",171,2026271986507,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.68,34.88,3.1672933101654053
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",178,2026270286463,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,5.1175,16.12,3.1217050552368164
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",403,2026275386841,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.61,4.05,0.7139673233032227
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",132,2026273686519,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.175,65.7,3.136279344558716
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",404,2026275386842,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.53,4.145,0.7094502449035645
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",405,2026275386843,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.615,3.885,0.7364861965179443
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",406,2026275386844,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5675,3.915,0.7126076221466064
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",179,2026270286464,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.97,16.335,3.2252910137176514
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",172,2026271986508,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.08,35.5,3.3718700408935547
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",407,2026275386845,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6125,4.065,0.70473313331604
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",133,2026273686520,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.225,65.6,3.1695165634155273
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",408,2026275386846,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.575,4.145,0.7056548595428467
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",409,2026275386847,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5875,3.99,0.7422537803649902
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",410,2026275386848,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6125,4.01,0.8007988929748535
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",411,2026275386849,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5975,4.055,0.7792520523071289
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",173,2026271986509,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.05,35.78,3.3819751739501953
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",180,2026270286465,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.93,15.495,3.4952290058135986
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",134,2026273686521,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.5,61.85,3.239553451538086
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",412,2026275386850,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5825,3.905,0.7990550994873047
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",413,2026275386851,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5525,3.905,0.735191822052002
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",414,2026275386852,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6025,4.055,0.7223215103149414
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",415,2026275386853,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5875,4.175,0.7847263813018799
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",174,2026271986510,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.46,35.96,3.0861759185791016
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",181,2026270286466,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.7625,16.505,3.1623291969299316
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",416,2026275386854,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5825,4.09,0.7339425086975098
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",135,2026273686522,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.925,65.4,3.1452717781066895
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",417,2026275386855,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5225,4.15,0.8325903415679932
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",418,2026275386856,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.57,4.02,0.7893486022949219
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",419,2026275386857,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5825,4.135,0.7709531784057617
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",175,2026271986511,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.93,35.72,3.1262197494506836
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0005|pi0|k1,1,1",182,2026270286467,B,quiet,5,0.0005,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,4.81,16.175,3.2344601154327393
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",420,2026275386858,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5575,3.995,0.7579526901245117
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",136,2026273686523,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,29.05,62.65,3.1330323219299316
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",421,2026275386859,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6275,3.775,0.8935716152191162
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",422,2026275386860,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.615,4.025,0.8557941913604736
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",423,2026275386861,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.605,3.81,0.7572212219238281
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",176,2026271986512,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.95,35.58,3.3310067653656006
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",0,2026277086489,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.9375,16.195,2.876176118850708
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",137,2026273686524,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.625,64.45,3.1373467445373535
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",424,2026275386862,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5925,3.865,0.8628206253051758
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",425,2026275386863,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.665,3.97,0.8019368648529053
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",426,2026275386864,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.605,3.9,0.7799527645111084
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",427,2026275386865,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.615,4.045,0.9003136157989502
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",1,2026277086490,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.065,15.915,2.8232293128967285
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",177,2026271986513,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.14,35.56,3.153834581375122
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",138,2026273686525,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.15,63.9,3.06007719039917
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",428,2026275386866,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.53,4.02,0.6949059963226318
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",429,2026275386867,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.635,4.085,0.7845702171325684
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",430,2026275386868,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5825,4.025,0.7840993404388428
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",2,2026277086491,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.8225,16.155,2.600830078125
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",431,2026275386869,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.5625,3.96,0.7807450294494629
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",178,2026271986514,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.7,36.38,3.1134893894195557
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",139,2026273686526,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,29.725,65.6,2.985346555709839
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",432,2026275386870,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.59,3.91,0.797839879989624
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",433,2026275386871,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.57,4.0,0.7688536643981934
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",434,2026275386872,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.58,3.985,0.7673442363739014
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",3,2026277086492,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.98,16.085,2.905648708343506
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",435,2026275386873,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6075,4.03,0.7801914215087891
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",140,2026273686527,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,29.4,65.3,3.08284854888916
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",179,2026271986515,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.28,35.86,3.2976315021514893
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",436,2026275386874,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.535,3.99,0.7785980701446533
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",437,2026275386875,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.64,4.08,0.7353708744049072
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",4,2026277086493,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.8525,15.92,2.900587797164917
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",438,2026275386876,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.605,3.885,0.7859001159667969
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",439,2026275386877,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5525,3.87,0.789684534072876
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",141,2026273686528,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.15,61.8,3.0945069789886475
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",180,2026271986516,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.57,35.96,3.285567283630371
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",440,2026275386878,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.62,3.95,0.8092854022979736
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",441,2026275386879,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.53,4.07,0.7880532741546631
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",5,2026277086494,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.0,16.115,2.512894630432129
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",442,2026275386880,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5525,4.055,0.8067276477813721
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",443,2026275386881,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.52,4.165,0.7901670932769775
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",142,2026273686529,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,32.175,63.85,3.2970311641693115
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",181,2026271986517,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.57,35.48,3.2680163383483887
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",444,2026275386882,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6325,3.96,0.7684717178344727
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",6,2026277086495,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.9275,15.895,2.8073055744171143
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",445,2026275386883,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5325,3.735,0.7913515567779541
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",446,2026275386884,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.56,3.88,0.7135105133056641
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",447,2026275386885,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.565,3.765,0.7770683765411377
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",143,2026273686530,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.3,64.0,3.173207998275757
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",448,2026275386886,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5325,4.015,0.7261166572570801
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",182,2026271986518,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,13.51,35.02,3.3045520782470703
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",7,2026277086496,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.9025,15.77,2.7076778411865234
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",449,2026275386887,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.655,4.045,0.7728192806243896
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",450,2026275386888,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5425,3.975,0.7109315395355225
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",451,2026275386889,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5425,4.01,0.7042624950408936
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",144,2026273686531,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.15,65.0,3.068763256072998
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",8,2026277086497,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.87,15.935,2.6458239555358887
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",452,2026275386890,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5925,4.065,0.6959652900695801
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",183,2026271986519,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.08,34.92,3.156461000442505
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",453,2026275386891,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.57,4.015,0.6944212913513184
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",454,2026275386892,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5425,3.94,0.7802565097808838
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",455,2026275386893,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5875,3.96,0.7049643993377686
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",9,2026277086498,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.0025,15.82,2.605949878692627
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",456,2026275386894,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6075,3.925,0.7098629474639893
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",145,2026273686532,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.45,61.95,2.9922573566436768
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",457,2026275386895,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.55,3.81,0.7068536281585693
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",184,2026271986520,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.76,35.84,3.333127737045288
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",458,2026275386896,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5725,3.975,0.7012939453125
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",459,2026275386897,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.65,4.01,0.7931897640228271
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",10,2026277086499,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.1275,16.065,2.8041415214538574
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",460,2026275386898,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.555,4.015,0.8039810657501221
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",146,2026273686533,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,32.0,63.1,3.1920714378356934
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",461,2026275386899,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.515,4.085,0.7995240688323975
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",185,2026271986521,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.31,37.46,3.30141544342041
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",462,2026275386900,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.6275,4.04,0.7809674739837646
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",11,2026277086500,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.8725,15.95,2.7966151237487793
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",463,2026275386901,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.575,3.99,0.788726806640625
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",464,2026275386902,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.61,4.04,0.7818353176116943
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",147,2026273686534,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.275,63.1,3.168536901473999
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",465,2026275386903,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.56,4.17,0.8038580417633057
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",186,2026271986522,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.09,36.3,3.3160736560821533
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",466,2026275386904,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.525,4.11,0.7762229442596436
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",12,2026277086501,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.95,16.31,2.7699222564697266
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",467,2026275386905,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5925,4.16,0.7983191013336182
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",468,2026275386906,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.61,3.975,0.7785625457763672
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",148,2026273686535,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.55,63.7,3.075359582901001
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",469,2026275386907,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.61,4.09,0.7677767276763916
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0005|pi0|k1,1,1",187,2026271986523,B,quiet,7,0.0005,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,14.5,36.12,3.148725748062134
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",13,2026277086502,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.12,16.26,2.7629969120025635
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",470,2026275386908,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6525,3.98,0.7934243679046631
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",471,2026275386909,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5575,3.92,0.8028712272644043
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",472,2026275386910,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5325,3.98,0.7770712375640869
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",149,2026273686536,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,29.675,65.65,3.2252068519592285
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",473,2026275386911,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.54,4.02,0.7675838470458984
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",14,2026277086503,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.005,15.895,2.9951088428497314
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",474,2026275386912,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5725,4.09,0.8264355659484863
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",475,2026275386913,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.55,4.015,0.7637572288513184
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",0,2026278786540,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.55,35.28,2.9272055625915527
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",476,2026275386914,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.555,4.225,0.7525479793548584
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",150,2026273686537,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.125,62.0,3.2310755252838135
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",477,2026275386915,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5525,3.99,0.7655308246612549
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",15,2026277086504,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.97,15.975,2.91271710395813
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",478,2026275386916,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5575,3.87,0.7987408638000488
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",479,2026275386917,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5875,3.94,0.7369365692138672
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",1,2026278786541,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.52,35.16,2.8046562671661377
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",480,2026275386918,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.625,4.125,0.7667145729064941
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",151,2026273686538,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,29.825,63.2,2.94907283782959
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",481,2026275386919,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.56,3.9,0.7914376258850098
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",16,2026277086505,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.1125,16.1,2.8262720108032227
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",482,2026275386920,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6175,3.91,0.7446227073669434
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",2,2026278786542,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.31,35.56,2.7079954147338867
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",483,2026275386921,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.605,3.815,0.7307608127593994
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",484,2026275386922,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5425,3.985,0.7241721153259277
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",152,2026273686539,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.425,65.1,3.0546324253082275
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",485,2026275386923,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5675,4.04,0.7874541282653809
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",17,2026277086506,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.0275,16.33,2.8809995651245117
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",486,2026275386924,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.535,4.05,0.728212833404541
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",3,2026278786543,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.4,35.88,2.6010098457336426
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",487,2026275386925,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.625,4.08,0.7593584060668945
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",488,2026275386926,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5175,4.04,0.8778715133666992
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",18,2026277086507,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.8575,15.865,2.8229830265045166
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",153,2026273686540,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.675,63.4,3.4420394897460938
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",489,2026275386927,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6075,4.055,0.7887539863586426
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",4,2026278786544,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.13,35.92,2.609354257583618
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",490,2026275386928,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.635,4.175,0.7496683597564697
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",491,2026275386929,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5675,4.14,0.7063357830047607
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",492,2026275386930,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.56,4.135,0.7170617580413818
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",19,2026277086508,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.8175,16.145,2.7844398021698
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",154,2026273686541,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.675,61.2,3.0276145935058594
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",5,2026278786545,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.19,35.5,2.7096588611602783
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",493,2026275386931,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.555,3.94,0.776648998260498
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",494,2026275386932,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.55,4.095,0.7343065738677979
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",495,2026275386933,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5675,4.215,0.7259528636932373
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",496,2026275386934,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5475,3.86,0.7471418380737305
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",20,2026277086509,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.93,16.075,2.8105924129486084
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",6,2026278786546,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.27,36.68,2.8026890754699707
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",155,2026273686542,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.125,64.45,3.005516290664673
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",497,2026275386935,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6,3.98,0.7297594547271729
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",498,2026275386936,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5925,3.92,0.7412865161895752
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",499,2026275386937,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6125,3.885,0.7341599464416504
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",21,2026277086510,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.8725,16.245,2.849559783935547
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",500,2026275386938,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.57,4.11,0.8119697570800781
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",7,2026278786547,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.05,36.54,2.6976683139801025
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",501,2026275386939,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.555,4.04,0.8169848918914795
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",156,2026273686543,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.5,62.3,3.2704989910125732
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",502,2026275386940,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.57,3.845,0.7610385417938232
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",503,2026275386941,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6275,3.985,0.7901406288146973
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",22,2026277086511,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.885,16.185,2.799757242202759
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",504,2026275386942,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.655,3.945,0.7825536727905273
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",8,2026278786548,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.86,35.42,2.96003794670105
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",505,2026275386943,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.615,3.995,0.7701520919799805
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",157,2026273686544,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.725,65.2,3.1860241889953613
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",506,2026275386944,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.62,4.18,0.7817084789276123
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",507,2026275386945,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5775,4.005,0.7767744064331055
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",23,2026277086512,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.8575,15.9,2.870626211166382
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",9,2026278786549,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.25,34.52,2.572437286376953
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",508,2026275386946,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6125,4.045,0.7858960628509521
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",509,2026275386947,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.63,3.95,0.7657656669616699
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",158,2026273686545,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.675,64.05,3.2648231983184814
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",510,2026275386948,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.575,4.085,0.7517104148864746
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",24,2026277086513,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.955,15.86,2.779468536376953
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",10,2026278786550,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.12,35.86,2.7347278594970703
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",511,2026275386949,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.625,4.025,0.7645668983459473
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",512,2026275386950,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.565,3.975,0.8103206157684326
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",513,2026275386951,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.595,4.08,0.7489821910858154
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",159,2026273686546,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.075,63.5,3.180692195892334
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",514,2026275386952,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.575,3.825,0.8388488292694092
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",11,2026278786551,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.55,35.74,2.7801225185394287
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",25,2026277086514,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.1025,16.07,2.9043424129486084
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",515,2026275386953,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5925,3.875,0.7268459796905518
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",516,2026275386954,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.575,3.925,0.7286984920501709
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",517,2026275386955,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6075,3.95,0.7158851623535156
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",160,2026273686547,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.1,62.85,3.276515483856201
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",518,2026275386956,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5375,3.935,0.7120661735534668
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",12,2026278786552,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.93,35.44,2.982118844985962
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",26,2026277086515,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.1025,15.945,2.9817769527435303
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",519,2026275386957,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5225,3.93,0.7221536636352539
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",520,2026275386958,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6325,4.105,0.7910466194152832
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",521,2026275386959,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6,3.99,0.7205965518951416
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",522,2026275386960,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.55,4.025,0.7173163890838623
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",13,2026278786553,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.56,35.82,2.727972984313965
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",161,2026273686548,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.075,64.25,3.2290170192718506
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",27,2026277086516,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.945,15.96,2.7857940196990967
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",523,2026275386961,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.565,3.955,0.7199504375457764
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",524,2026275386962,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.54,4.13,0.7089080810546875
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",525,2026275386963,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.57,4.1,0.7095277309417725
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",526,2026275386964,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5725,3.915,0.7177033424377441
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",28,2026277086517,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.81,16.23,2.764503240585327
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",14,2026278786554,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.15,36.98,2.873469591140747
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",162,2026273686549,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.25,63.7,3.2416954040527344
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",527,2026275386965,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.62,3.895,0.7879550457000732
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",528,2026275386966,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.575,3.965,0.8073413372039795
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",529,2026275386967,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5725,3.925,0.8093435764312744
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",15,2026278786555,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.27,33.9,2.739840507507324
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",29,2026277086518,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.8725,16.175,2.9450018405914307
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",530,2026275386968,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.57,4.055,0.7820775508880615
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",163,2026273686550,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,32.475,63.4,3.3056528568267822
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",531,2026275386969,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.555,3.91,0.7917063236236572
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",532,2026275386970,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.64,4.04,0.7665178775787354
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",533,2026275386971,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6125,4.07,0.7501704692840576
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",16,2026278786556,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.84,35.26,2.683276414871216
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",30,2026277086519,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.945,15.92,2.8477823734283447
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",534,2026275386972,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5775,3.87,0.7125344276428223
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",535,2026275386973,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6225,4.14,0.7193648815155029
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",164,2026273686551,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.375,65.15,3.178757429122925
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",536,2026275386974,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5775,4.055,0.7605404853820801
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",17,2026278786557,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.89,36.12,2.7446141242980957
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",537,2026275386975,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.535,4.14,0.7812063694000244
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",31,2026277086520,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.9225,15.83,2.7809314727783203
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",538,2026275386976,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.59,3.99,0.822401762008667
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",539,2026275386977,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.51,4.145,0.7627072334289551
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",165,2026273686552,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,29.425,62.3,3.183443784713745
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",540,2026275386978,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.57,4.125,0.8026623725891113
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",18,2026278786558,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.59,36.08,2.717461347579956
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",32,2026277086521,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.9025,15.945,2.813288927078247
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",541,2026275386979,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.51,3.905,0.792370080947876
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",542,2026275386980,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.485,3.775,0.7816998958587646
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",543,2026275386981,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.54,3.955,0.8214888572692871
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",166,2026273686553,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,29.5,61.55,3.3517942428588867
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",19,2026278786559,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.03,36.56,2.6830458641052246
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",544,2026275386982,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.57,3.895,0.8640420436859131
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",33,2026277086522,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.81,16.09,2.882453680038452
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",545,2026275386983,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.515,4.115,0.790947437286377
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",546,2026275386984,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5375,4.245,0.8230462074279785
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",20,2026278786560,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.38,35.7,2.4397177696228027
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",547,2026275386985,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5875,3.905,0.7566962242126465
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",167,2026273686554,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.425,63.25,2.9140987396240234
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",548,2026275386986,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.575,4.195,0.7615468502044678
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",34,2026277086523,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,2.9925,16.025,2.9334380626678467
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",549,2026275386987,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.55,4.04,0.8231699466705322
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",21,2026278786561,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.3,36.34,2.531130313873291
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",550,2026275386988,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5975,3.97,0.7686030864715576
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",168,2026273686555,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,29.075,64.8,3.0005440711975098
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",551,2026275386989,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.54,4.105,0.7627933025360107
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",35,2026277086524,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.0375,16.305,2.877516031265259
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",552,2026275386990,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.51,4.075,0.8062896728515625
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",553,2026275386991,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.51,4.03,0.8112249374389648
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",22,2026278786562,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.63,35.64,2.6585099697113037
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",554,2026275386992,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5525,4.0,0.7997312545776367
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",169,2026273686556,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.575,65.9,3.1680870056152344
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",555,2026275386993,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5775,4.045,0.7183704376220703
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",36,2026277086525,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.935,15.785,2.7617483139038086
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",556,2026275386994,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6375,4.0,0.7287204265594482
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",23,2026278786563,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.66,36.38,2.6066205501556396
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",557,2026275386995,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6225,3.89,0.7128353118896484
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",558,2026275386996,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.615,4.02,0.7183411121368408
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",559,2026275386997,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.645,4.04,0.7608022689819336
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",37,2026277086526,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.0125,15.535,2.8474624156951904
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",170,2026273686557,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,28.8,62.55,3.238851547241211
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",560,2026275386998,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6425,3.97,0.7675800323486328
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",24,2026278786564,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.15,36.26,2.6303200721740723
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",561,2026275386999,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.6125,4.105,0.7806413173675537
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",562,2026275387000,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.665,4.01,0.8388092517852783
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",38,2026277086527,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.91,16.195,2.93125319480896
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",563,2026275387001,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6075,3.945,0.8351192474365234
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",171,2026273686558,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.025,63.3,3.1224613189697266
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",25,2026278786565,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.21,35.46,2.6041157245635986
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",564,2026275387002,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.605,4.0,0.7697494029998779
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",565,2026275387003,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5525,3.905,0.7907860279083252
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",566,2026275387004,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6,4.035,0.7589724063873291
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",39,2026277086528,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.925,16.325,2.8506343364715576
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",26,2026278786566,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.71,35.68,2.7042415142059326
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",567,2026275387005,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.54,4.01,0.7488391399383545
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",172,2026273686559,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.5,1,1,31.65,65.2,3.301300048828125
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",568,2026275387006,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5575,4.145,0.7567093372344971
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",569,2026275387007,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.605,4.03,0.7884137630462646
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",570,2026275387008,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.56,4.08,0.7798373699188232
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",40,2026277086529,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.165,16.025,2.699087142944336
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",27,2026278786567,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.62,36.24,2.6843559741973877
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",571,2026275387009,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.535,4.17,0.8077526092529297
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",173,2026273686560,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,29.875,65.75,3.3484864234924316
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",572,2026275387010,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5525,3.975,0.733863353729248
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",573,2026275387011,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.595,3.945,0.7174310684204102
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",41,2026277086530,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.9925,16.1,2.6436362266540527
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",28,2026278786568,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.26,34.92,2.4886231422424316
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",574,2026275387012,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5575,3.91,0.700143575668335
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",575,2026275387013,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.535,4.09,0.7890329360961914
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",576,2026275387014,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.53,3.995,0.7010805606842041
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",174,2026273686561,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.15,64.45,3.097275972366333
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",577,2026275387015,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5825,3.995,0.7595977783203125
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",42,2026277086531,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.9025,15.955,2.80161714553833
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",29,2026278786569,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.64,35.98,2.668962240219116
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",578,2026275387016,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.595,3.945,0.7686989307403564
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",579,2026275387017,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.585,4.025,0.7554416656494141
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",580,2026275387018,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6275,4.05,0.736750602722168
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",175,2026273686562,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.45,64.3,3.127699136734009
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",43,2026277086532,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.8775,15.89,2.7055764198303223
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",581,2026275387019,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.64,3.945,0.7140049934387207
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",30,2026278786570,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.05,35.66,2.7278709411621094
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",582,2026275387020,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5275,3.875,0.7095041275024414
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",583,2026275387021,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.545,4.015,0.727109432220459
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",584,2026275387022,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.62,4.07,0.7358555793762207
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",31,2026278786571,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.07,35.68,2.523643970489502
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",176,2026273686563,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.55,62.15,3.17741060256958
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",44,2026277086533,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.95,16.06,2.80255389213562
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",585,2026275387023,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.54,4.015,0.713111162185669
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",586,2026275387024,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.495,4.13,0.7051355838775635
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",587,2026275387025,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.545,4.14,0.7128705978393555
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",588,2026275387026,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.575,4.05,0.7126092910766602
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",32,2026278786572,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.75,37.1,2.972034215927124
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",45,2026277086534,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.1725,16.21,2.8926830291748047
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",589,2026275387027,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5325,4.195,0.7234091758728027
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",177,2026273686564,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.4,65.05,3.184244394302368
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",590,2026275387028,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.53,4.11,0.7199404239654541
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",591,2026275387029,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6375,4.045,0.7243635654449463
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",592,2026275387030,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5725,4.055,0.7802009582519531
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",33,2026278786573,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.9,36.94,2.5722951889038086
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",46,2026277086535,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.9175,15.825,2.8096323013305664
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",593,2026275387031,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.585,4.0,0.7559025287628174
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",178,2026273686565,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.4,63.65,3.250669240951538
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",594,2026275387032,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.4925,3.865,0.7945764064788818
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",595,2026275387033,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.56,3.97,0.8100237846374512
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",34,2026278786574,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.03,34.72,2.8849241733551025
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",596,2026275387034,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5875,3.945,0.7765665054321289
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",47,2026277086536,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.05,16.0,2.816220283508301
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",597,2026275387035,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5575,3.905,0.784466028213501
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",179,2026273686566,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.125,62.2,3.0648062229156494
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",598,2026275387036,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5825,3.935,0.7755637168884277
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",599,2026275387037,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5825,3.995,0.7588129043579102
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",35,2026278786575,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.42,34.82,2.6738712787628174
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",48,2026277086537,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.9,15.985,2.9227120876312256
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",600,2026275387038,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5875,3.945,0.7788810729980469
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",601,2026275387039,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6025,3.92,0.7925233840942383
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",180,2026273686567,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.125,61.85,3.089116334915161
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",602,2026275387040,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.595,3.98,0.8013918399810791
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",36,2026278786576,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.3,35.94,2.6735165119171143
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",603,2026275387041,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.58,3.97,0.8276786804199219
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",49,2026277086538,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.98,15.89,2.7504169940948486
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",604,2026275387042,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.615,3.985,0.7052903175354004
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",605,2026275387043,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6275,4.04,0.7384071350097656
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",181,2026273686568,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.325,63.9,3.114656448364258
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",37,2026278786577,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.46,36.58,2.5407156944274902
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",606,2026275387044,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.57,4.115,0.7812275886535645
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",607,2026275387045,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5725,3.96,0.766373872756958
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",50,2026277086539,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.975,16.06,2.9088823795318604
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",608,2026275387046,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.53,4.085,0.7775180339813232
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",609,2026275387047,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.52,3.995,0.7569897174835205
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",182,2026273686569,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.1,63.15,3.1596248149871826
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",38,2026278786578,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.29,35.38,2.6916637420654297
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",610,2026275387048,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5925,3.9,0.77543044090271
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",51,2026277086540,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.9375,15.915,2.650526523590088
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",611,2026275387049,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5675,4.205,0.7620372772216797
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",612,2026275387050,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5525,3.995,0.7542738914489746
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",39,2026278786579,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.07,35.72,2.5701684951782227
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",613,2026275387051,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5625,3.825,0.7571516036987305
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",183,2026273686570,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.95,62.45,3.1102113723754883
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",614,2026275387052,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5975,4.06,0.7561023235321045
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",52,2026277086541,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.975,15.99,2.6181833744049072
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",615,2026275387053,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6125,3.915,0.8347980976104736
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",616,2026275387054,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.59,3.915,0.7271363735198975
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",40,2026278786580,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.28,35.66,2.735750913619995
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",617,2026275387055,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.58,3.915,0.7354950904846191
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",184,2026273686571,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.75,63.35,3.174488067626953
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",618,2026275387056,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.57,3.89,0.7419455051422119
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",53,2026277086542,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.98,16.085,2.958836317062378
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",619,2026275387057,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.575,3.795,0.8325440883636475
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",41,2026278786581,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.56,36.66,2.7018651962280273
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",620,2026275387058,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.4925,4.105,0.7976498603820801
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",621,2026275387059,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5925,4.005,0.8075551986694336
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",54,2026277086543,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.9975,16.365,2.6923153400421143
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",185,2026273686572,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,31.225,63.4,3.188652753829956
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",622,2026275387060,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.595,3.97,0.8009157180786133
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",623,2026275387061,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.545,4.105,0.8104162216186523
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",42,2026278786582,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.9,35.78,2.6545562744140625
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",624,2026275387062,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.6175,3.845,0.7827596664428711
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",55,2026277086544,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.0625,15.835,2.654571533203125
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",625,2026275387063,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6075,4.045,0.8465733528137207
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",186,2026273686573,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,29.575,64.15,2.9696781635284424
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",626,2026275387064,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5475,3.845,0.826613187789917
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",43,2026278786583,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.5,36.16,2.7480568885803223
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",627,2026275387065,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5625,4.005,0.7674736976623535
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",628,2026275387066,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6325,3.92,0.8242254257202148
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",56,2026277086545,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.97,16.205,2.7855443954467773
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",629,2026275387067,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.515,3.945,0.7583920955657959
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",187,2026273686574,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.35,62.95,3.155717611312866
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",630,2026275387068,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.565,4.035,0.738771915435791
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",44,2026278786584,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.82,35.14,2.9403631687164307
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",631,2026275387069,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.55,4.015,0.7414507865905762
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",57,2026277086546,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.935,15.99,2.8572022914886475
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",632,2026275387070,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6025,4.07,0.7438604831695557
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",633,2026275387071,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5975,4.18,0.8278160095214844
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",45,2026278786585,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.37,35.22,2.540255546569824
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0005|pi0|k1,1,1",188,2026273686575,B,quiet,9,0.0005,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,30.95,63.15,3.3405323028564453
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",634,2026275387072,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6,3.895,0.7674055099487305
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",635,2026275387073,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.58,4.035,0.7727177143096924
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",58,2026277086547,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.075,16.045,2.9004430770874023
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",636,2026275387074,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.565,4.085,0.7268333435058594
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",637,2026275387075,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.595,4.085,0.7737331390380859
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",46,2026278786586,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.23,35.14,2.7448582649230957
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",638,2026275387076,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6025,4.2,0.7630329132080078
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",639,2026275387077,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.4975,3.885,0.7611005306243896
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",59,2026277086548,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.99,16.235,2.9164679050445557
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",640,2026275387078,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5625,4.135,0.759713888168335
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",47,2026278786587,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.2,35.6,2.5985255241394043
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",641,2026275387079,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.635,3.88,0.7698442935943604
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",642,2026275387080,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5825,3.96,0.7649195194244385
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",0,2026280486591,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.2,65.25,3.2498085498809814
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",60,2026277086549,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.94,16.13,2.654003143310547
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",643,2026275387081,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6025,4.135,0.7893967628479004
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",48,2026278786588,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.26,36.04,2.5175507068634033
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",644,2026275387082,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.6125,4.19,0.7641675472259521
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",645,2026275387083,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5775,4.125,0.7842521667480469
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",646,2026275387084,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.595,3.875,0.798926591873169
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",61,2026277086550,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.995,16.225,2.8725523948669434
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",1,2026280486592,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.575,63.5,3.068143606185913
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",647,2026275387085,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5475,3.855,0.81058669090271
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",49,2026278786589,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.84,37.06,2.9159305095672607
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",648,2026275387086,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6475,3.965,0.8028073310852051
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",649,2026275387087,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6375,4.035,0.8004341125488281
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",650,2026275387088,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5625,3.925,0.8127806186676025
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",62,2026277086551,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.99,15.87,2.8287343978881836
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",2,2026280486593,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.925,62.9,3.0434436798095703
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",651,2026275387089,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.53,3.915,0.7420535087585449
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",50,2026278786590,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.92,36.54,2.910717725753784
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",652,2026275387090,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.62,3.995,0.7085025310516357
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",653,2026275387091,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5525,3.965,0.7222068309783936
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",654,2026275387092,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.6225,4.04,0.7293605804443359
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",63,2026277086552,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.105,16.09,2.8768563270568848
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",3,2026280486594,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.3,61.15,2.6870622634887695
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",655,2026275387093,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5725,4.125,0.7062337398529053
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",51,2026278786591,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.2,35.56,2.7749762535095215
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",656,2026275387094,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.585,3.915,0.7842469215393066
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",657,2026275387095,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.565,3.91,0.7781181335449219
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",64,2026277086553,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.8775,15.935,2.749375343322754
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",658,2026275387096,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.585,3.965,0.7790451049804688
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",4,2026280486595,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.925,63.25,3.055419683456421
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",52,2026278786592,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.04,35.2,2.6273322105407715
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",659,2026275387097,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5225,4.03,0.7619278430938721
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",660,2026275387098,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.62,3.96,0.7619788646697998
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",661,2026275387099,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.4725,4.19,0.7157189846038818
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",65,2026277086554,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.8875,15.875,2.7344369888305664
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",53,2026278786593,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.59,36.46,2.4733166694641113
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",662,2026275387100,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5425,4.095,0.7680275440216064
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",5,2026280486596,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.0,66.95,3.17903208732605
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",663,2026275387101,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5475,3.8,0.745244026184082
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",664,2026275387102,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.565,3.97,0.8335092067718506
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",66,2026277086555,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.03,15.895,2.6814465522766113
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",665,2026275387103,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5475,4.02,0.7568330764770508
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",54,2026278786594,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.48,36.06,2.754152536392212
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",666,2026275387104,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5375,3.93,0.7505273818969727
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",6,2026280486597,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.475,65.25,3.1425201892852783
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",667,2026275387105,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.57,3.955,0.804598331451416
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",668,2026275387106,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5575,3.945,0.8262948989868164
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",67,2026277086556,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.0975,16.0,2.799234390258789
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",55,2026278786595,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.11,35.14,2.7421159744262695
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",669,2026275387107,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5875,3.95,0.7914800643920898
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",670,2026275387108,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6125,3.955,0.8176705837249756
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",7,2026280486598,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.65,64.8,3.026442289352417
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",671,2026275387109,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6,3.995,0.8133378028869629
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",68,2026277086557,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.9825,15.615,2.714174747467041
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",56,2026278786596,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.08,36.2,2.4520885944366455
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",672,2026275387110,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6125,3.79,0.8596875667572021
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",673,2026275387111,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.565,4.135,0.7907290458679199
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",8,2026280486599,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.075,64.2,2.9306254386901855
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",674,2026275387112,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.54,3.925,0.7760739326477051
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",675,2026275387113,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6025,4.06,0.745962381362915
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",69,2026277086558,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.0575,16.305,2.8048813343048096
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",57,2026278786597,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.21,36.42,2.770750045776367
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",676,2026275387114,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.565,4.03,0.7142603397369385
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",677,2026275387115,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.61,4.085,0.7184464931488037
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",9,2026280486600,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.4,63.25,2.9488518238067627
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",678,2026275387116,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.585,3.99,0.7455446720123291
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",679,2026275387117,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5975,3.915,0.7345883846282959
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",70,2026277086559,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.0025,15.92,2.805666208267212
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",58,2026278786598,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.29,37.42,2.780374526977539
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",680,2026275387118,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5825,4.085,0.726710319519043
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",681,2026275387119,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5325,3.985,0.7180509567260742
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",682,2026275387120,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.585,4.09,0.7368435859680176
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",10,2026280486601,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.65,63.0,2.9915618896484375
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",71,2026277086560,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.8675,15.925,2.6882026195526123
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",683,2026275387121,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.575,4.07,0.7225677967071533
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",59,2026278786599,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.72,35.84,2.8318750858306885
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",684,2026275387122,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6875,3.82,0.7332487106323242
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",685,2026275387123,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5525,4.02,0.7179348468780518
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",686,2026275387124,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.62,3.985,0.7472219467163086
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",72,2026277086561,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.05,16.03,2.71089768409729
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",11,2026280486602,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.05,66.2,3.2448463439941406
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",60,2026278786600,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.42,35.62,2.836320161819458
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",687,2026275387125,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.5,1,1,0.56,4.14,0.7323212623596191
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",688,2026275387126,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6075,4.14,0.7926690578460693
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",689,2026275387127,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5575,4.035,0.7058200836181641
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",690,2026275387128,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5425,4.01,0.7182283401489258
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",73,2026277086562,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.85,15.765,2.9639554023742676
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",12,2026280486603,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.875,63.2,3.095785140991211
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",61,2026278786601,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.55,36.82,2.8276915550231934
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",691,2026275387129,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.555,4.055,0.7134096622467041
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",692,2026275387130,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.61,4.11,0.7217855453491211
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",693,2026275387131,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5425,4.005,0.6960093975067139
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",694,2026275387132,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5875,3.935,0.7268483638763428
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",74,2026277086563,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.9025,15.94,2.88442063331604
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",62,2026278786602,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.27,36.7,2.7531344890594482
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",13,2026280486604,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.425,65.4,2.8998191356658936
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",695,2026275387133,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5425,4.02,0.7661397457122803
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",696,2026275387134,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5875,3.905,0.7643969058990479
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",697,2026275387135,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5675,4.23,0.7293109893798828
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",698,2026275387136,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5575,3.96,0.7408168315887451
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",63,2026278786603,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.56,35.96,2.621633768081665
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",75,2026277086564,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.0025,16.1,2.848177909851074
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",14,2026280486605,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.125,63.85,3.1117324829101562
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",699,2026275387137,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.555,4.18,0.7323322296142578
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",700,2026275387138,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.63,4.22,0.698631763458252
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",701,2026275387139,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5525,4.09,0.701007604598999
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",64,2026278786604,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.31,35.84,2.622049570083618
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",702,2026275387140,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6525,3.98,0.7118675708770752
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",76,2026277086565,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.935,15.715,2.7102484703063965
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",15,2026280486606,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.525,65.15,2.8265480995178223
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",703,2026275387141,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.54,4.0,0.7079710960388184
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",704,2026275387142,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.605,3.915,0.7692198753356934
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",705,2026275387143,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.59,4.19,0.7700538635253906
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",65,2026278786605,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,7.84,36.26,2.766033172607422
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",77,2026277086566,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.0,15.775,2.7572097778320312
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",706,2026275387144,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5875,4.015,0.7542955875396729
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",16,2026280486607,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.5,64.1,3.0592310428619385
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",707,2026275387145,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.57,3.93,0.7384941577911377
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",708,2026275387146,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.55,4.095,0.7627441883087158
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",66,2026278786606,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.83,35.3,2.703739881515503
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",709,2026275387147,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.585,4.105,0.7693490982055664
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",78,2026277086567,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.06,16.325,2.8455796241760254
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",710,2026275387148,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.58,3.935,0.7600257396697998
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",17,2026280486608,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.95,64.35,3.207520008087158
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",711,2026275387149,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.57,3.925,0.8763720989227295
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",712,2026275387150,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.58,4.075,0.7184972763061523
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",67,2026278786607,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.33,34.26,2.6938581466674805
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",713,2026275387151,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.58,3.94,0.7354316711425781
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",79,2026277086568,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.99,15.89,2.786463499069214
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",714,2026275387152,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.625,3.955,0.7151436805725098
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",715,2026275387153,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.53,3.97,0.6999301910400391
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",18,2026280486609,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.1,64.9,2.9293370246887207
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",716,2026275387154,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6025,4.185,0.7374382019042969
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",68,2026278786608,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.21,35.96,2.611783266067505
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",80,2026277086569,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.8575,15.945,2.632319211959839
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",717,2026275387155,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.61,4.065,0.7239646911621094
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",718,2026275387156,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5425,3.945,0.7087109088897705
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",719,2026275387157,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.4975,4.09,0.7260706424713135
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",19,2026280486610,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.65,63.4,3.058302640914917
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",69,2026278786609,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.24,35.32,2.567077875137329
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",720,2026275387158,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.545,4.07,0.7097861766815186
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",81,2026277086570,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.81,16.76,2.6740143299102783
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",721,2026275387159,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.615,3.78,0.8043324947357178
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",722,2026275387160,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5675,4.125,0.832909345626831
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",70,2026278786610,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.45,35.68,2.529413938522339
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",20,2026280486611,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.65,64.3,2.803431272506714
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",723,2026275387161,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6,3.995,0.7908108234405518
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",82,2026277086571,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.7975,15.76,2.6470530033111572
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",724,2026275387162,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.625,3.975,0.8048920631408691
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",725,2026275387163,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.52,3.855,0.7759215831756592
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",726,2026275387164,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5925,4.03,0.7817015647888184
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",71,2026278786611,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.47,35.72,2.709879159927368
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",21,2026280486612,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.225,63.1,3.1196248531341553
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",727,2026275387165,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.54,3.975,0.7740027904510498
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",83,2026277086572,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.035,15.88,2.8722481727600098
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",728,2026275387166,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5225,4.06,0.7617921829223633
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",729,2026275387167,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6125,3.835,0.783750057220459
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",72,2026278786612,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.1,36.32,2.6886610984802246
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",730,2026275387168,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5775,4.1,0.7726707458496094
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",22,2026280486613,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.5,64.85,2.7370316982269287
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",731,2026275387169,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.515,4.145,0.7972598075866699
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",84,2026277086573,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.04,15.325,2.8710317611694336
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",732,2026275387170,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6,4.01,0.7199008464813232
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",733,2026275387171,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.63,4.085,0.7259395122528076
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",73,2026278786613,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.83,34.34,2.833665609359741
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",734,2026275387172,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.535,4.01,0.7136704921722412
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",23,2026280486614,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.1,65.2,2.9457075595855713
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",85,2026277086574,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.8725,15.88,2.61918306350708
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",735,2026275387173,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.58,3.9,0.727567195892334
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",736,2026275387174,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5775,4.13,0.83597731590271
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",737,2026275387175,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5625,4.135,0.8506259918212891
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",74,2026278786614,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.33,36.66,2.7942800521850586
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",738,2026275387176,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.57,3.965,0.7416341304779053
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",24,2026280486615,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.2,63.85,3.0393483638763428
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",86,2026277086575,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.5,1,1,2.88,16.08,2.9970531463623047
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",739,2026275387177,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5525,3.945,0.7262933254241943
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",740,2026275387178,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.605,3.835,0.7641468048095703
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",75,2026278786615,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.6,35.56,2.8311986923217773
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",741,2026275387179,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5675,4.065,0.8400886058807373
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",742,2026275387180,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.4975,4.05,0.7690577507019043
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",25,2026280486616,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.025,63.25,2.9841415882110596
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",87,2026277086576,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.8225,15.89,3.0099635124206543
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",743,2026275387181,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6,4.26,0.7536182403564453
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",744,2026275387182,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5525,3.99,0.7619662284851074
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",76,2026278786616,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.4,35.66,2.7645392417907715
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",745,2026275387183,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.59,4.21,0.7084853649139404
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",26,2026280486617,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.925,63.2,2.9240262508392334
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",746,2026275387184,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6225,4.135,0.7305190563201904
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",88,2026277086577,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.88,16.165,2.7462029457092285
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",747,2026275387185,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.61,3.92,0.7111098766326904
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",748,2026275387186,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.58,4.19,0.7195127010345459
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",77,2026278786617,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.83,36.2,2.7417101860046387
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",749,2026275387187,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6225,4.07,0.7291886806488037
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",89,2026277086578,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.8625,16.085,2.705078125
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",27,2026280486618,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.825,62.45,2.844909906387329
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",750,2026275387188,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.605,3.845,0.7328414916992188
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",751,2026275387189,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6525,3.96,0.7199714183807373
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",752,2026275387190,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5775,3.94,0.7444982528686523
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",78,2026278786618,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.72,36.2,2.8872532844543457
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",753,2026275387191,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5925,4.065,0.7564055919647217
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",90,2026277086579,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.025,15.835,2.9903371334075928
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",28,2026280486619,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.175,66.25,2.906482219696045
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",754,2026275387192,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.6275,4.02,0.7401342391967773
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",755,2026275387193,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6225,4.0,0.7137200832366943
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",79,2026278786619,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.43,37.2,2.6955041885375977
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",756,2026275387194,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.56,3.91,0.8693101406097412
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",757,2026275387195,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5725,4.065,0.7610008716583252
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",91,2026277086580,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.0525,15.835,2.845310688018799
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",29,2026280486620,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.15,64.85,3.1005964279174805
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",758,2026275387196,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5875,4.015,0.7798810005187988
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",759,2026275387197,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5975,3.84,0.72878098487854
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",80,2026278786620,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.45,36.26,2.8881888389587402
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",760,2026275387198,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.595,4.03,0.7155570983886719
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",761,2026275387199,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.61,4.12,0.7470083236694336
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",92,2026277086581,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.015,16.06,2.8360040187835693
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",30,2026280486621,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.925,63.9,3.187481164932251
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",762,2026275387200,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5625,4.11,0.8732059001922607
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",81,2026278786621,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.73,34.94,2.712881088256836
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",763,2026275387201,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.58,3.945,0.7441270351409912
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",764,2026275387202,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5775,3.985,0.8029606342315674
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",93,2026277086582,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.8275,16.08,2.8142476081848145
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",765,2026275387203,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.665,4.005,0.7932443618774414
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",31,2026280486622,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.825,63.8,3.0138700008392334
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",766,2026275387204,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.525,4.065,0.7903614044189453
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",82,2026278786622,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.58,36.66,2.6603524684906006
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",767,2026275387205,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.55,3.995,0.8020293712615967
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",768,2026275387206,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.56,4.065,0.7990283966064453
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",94,2026277086583,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.925,15.99,2.8660151958465576
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",769,2026275387207,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6075,3.95,0.7613644599914551
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",83,2026278786623,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.18,35.74,2.584317922592163
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",32,2026280486623,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.375,65.2,3.0188536643981934
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",770,2026275387208,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6325,4.18,0.8627238273620605
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",771,2026275387209,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.63,3.955,0.7414617538452148
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",772,2026275387210,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.6475,3.945,0.7504134178161621
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",95,2026277086584,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.9775,15.92,2.9120922088623047
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",773,2026275387211,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5975,4.095,0.7319638729095459
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",84,2026278786624,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.15,34.74,2.666195869445801
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",33,2026280486624,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.65,63.65,2.992645025253296
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",774,2026275387212,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5925,4.08,0.7151141166687012
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",775,2026275387213,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.57,3.86,0.7026867866516113
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",776,2026275387214,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.49,3.985,0.7824873924255371
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",96,2026277086585,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.0275,15.63,2.913092613220215
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",85,2026278786625,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.52,35.44,2.8398473262786865
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",777,2026275387215,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.6075,4.04,0.7707114219665527
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",34,2026280486625,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.825,61.95,3.035665273666382
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",778,2026275387216,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5525,3.94,0.7311079502105713
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",779,2026275387217,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5475,4.17,0.7142140865325928
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",97,2026277086586,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.0275,15.85,2.8831796646118164
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",780,2026275387218,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.595,3.825,0.7315030097961426
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",86,2026278786626,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.26,36.28,2.7589311599731445
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",781,2026275387219,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,1.0,1,0,0.5425,4.015,0.7275228500366211
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",35,2026280486626,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.125,64.9,3.0078341960906982
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",782,2026275387220,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.555,4.135,0.7686047554016113
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",783,2026275387221,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5775,3.915,0.7386937141418457
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",98,2026277086587,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.8475,15.965,2.924818992614746
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",784,2026275387222,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.55,4.025,0.7375659942626953
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",87,2026278786627,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.57,35.52,2.6234467029571533
production-20261029T053925Z-29f79c,"B|quiet|d3|p0.0003|pi0|k1,1,1",785,2026275387223,B,quiet,3,0.0003,0.0,1.0,1.0,1.0,3,3,3,400,0.0,0,0,0.5275,4.215,0.7381072044372559
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",36,2026280486627,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.275,64.95,3.147550344467163
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",99,2026277086588,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.9825,15.95,2.7066848278045654
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",88,2026278786628,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.78,36.56,2.940364360809326
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",37,2026280486628,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.375,63.7,3.1009130477905273
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",100,2026277086589,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.0975,16.015,2.7525975704193115
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",89,2026278786629,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.33,36.12,2.7140538692474365
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",38,2026280486629,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.375,69.05,3.1216962337493896
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",101,2026277086590,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.235,16.17,2.8993053436279297
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",90,2026278786630,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,7.96,36.12,2.7822012901306152
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",39,2026280486630,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.275,61.65,2.8055076599121094
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",91,2026278786631,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.44,35.58,2.665708065032959
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",102,2026277086591,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.8625,16.07,2.9348700046539307
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",40,2026280486631,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.225,64.8,2.9063289165496826
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",92,2026278786632,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.72,36.12,2.6139729022979736
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",103,2026277086592,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.8075,15.705,2.9060659408569336
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",93,2026278786633,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.08,36.4,2.547018527984619
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",41,2026280486632,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.6,63.8,3.0803890228271484
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",104,2026277086593,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.0025,15.97,2.7625417709350586
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",94,2026278786634,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.36,35.34,2.608944892883301
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",42,2026280486633,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.75,62.55,2.9356625080108643
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",105,2026277086594,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.855,15.71,2.592848539352417
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",95,2026278786635,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.29,36.66,2.4808273315429688
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",106,2026277086595,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.9475,15.845,2.6771271228790283
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",43,2026280486634,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.075,64.65,2.828803062438965
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",96,2026278786636,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.89,34.88,2.6138782501220703
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",107,2026277086596,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.1625,16.135,2.914029359817505
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",44,2026280486635,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.525,62.95,3.007124185562134
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",97,2026278786637,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.44,34.64,2.705704689025879
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",108,2026277086597,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.2025,16.035,2.7637617588043213
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",45,2026280486636,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.425,64.3,3.227924108505249
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",98,2026278786638,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.5,36.24,2.7035305500030518
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",109,2026277086598,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.9475,16.11,2.7532832622528076
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",46,2026280486637,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,16.85,63.0,3.1484622955322266
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",99,2026278786639,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.68,36.92,2.664015054702759
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",110,2026277086599,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.9175,16.195,2.7182939052581787
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",47,2026280486638,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.35,64.85,3.110952615737915
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",100,2026278786640,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.53,35.82,2.7932541370391846
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",111,2026277086600,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.07,15.84,3.113837957382202
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",101,2026278786641,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.36,35.48,2.6970748901367188
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",48,2026280486639,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.45,63.3,2.9872689247131348
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",112,2026277086601,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.06,15.875,2.6675214767456055
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",102,2026278786642,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.72,36.7,2.5703747272491455
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",49,2026280486640,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.075,62.65,3.0096781253814697
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",113,2026277086602,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.93,16.355,2.741088628768921
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",103,2026278786643,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.38,36.52,2.73258900642395
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",50,2026280486641,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.65,64.85,3.0363101959228516
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",114,2026277086603,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.0575,16.295,3.002652168273926
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",104,2026278786644,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.13,36.92,2.468214750289917
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",51,2026280486642,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.275,64.3,2.9940121173858643
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",115,2026277086604,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.01,15.795,2.8872814178466797
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",105,2026278786645,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.38,35.72,2.5274970531463623
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",52,2026280486643,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.15,65.35,2.916853189468384
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",116,2026277086605,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.88,15.82,2.929670810699463
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",106,2026278786646,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.47,36.82,2.840442419052124
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",53,2026280486644,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.125,60.15,2.90657114982605
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",117,2026277086606,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.9775,16.035,2.8928921222686768
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",107,2026278786647,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.81,35.04,2.941789150238037
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",54,2026280486645,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.275,61.7,2.9659345149993896
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",108,2026278786648,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.09,35.92,2.4188570976257324
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",118,2026277086607,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.015,15.88,2.694808006286621
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",55,2026280486646,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.75,64.5,2.9787051677703857
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",109,2026278786649,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.22,36.42,2.522434711456299
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",119,2026277086608,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.81,16.155,2.8322253227233887
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",56,2026280486647,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.525,65.2,2.778028726577759
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",110,2026278786650,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.76,36.36,2.9175195693969727
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",120,2026277086609,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.0075,15.79,2.70318341255188
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",111,2026278786651,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.26,35.6,2.768160104751587
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",57,2026280486648,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.875,64.4,3.030479907989502
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",121,2026277086610,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.8925,16.14,2.656538963317871
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",112,2026278786652,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.36,36.32,2.566941738128662
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",122,2026277086611,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.085,15.865,2.8795554637908936
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",58,2026280486649,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.8,64.95,3.0504610538482666
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",113,2026278786653,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.26,35.74,2.6270275115966797
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",123,2026277086612,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.115,16.2,2.9201242923736572
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",59,2026280486650,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.475,63.2,3.1430561542510986
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",114,2026278786654,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.35,35.72,2.6627845764160156
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",124,2026277086613,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.9475,16.04,2.888170003890991
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",60,2026280486651,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.8,63.65,3.0551323890686035
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",115,2026278786655,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.41,36.8,2.6723294258117676
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",125,2026277086614,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.9625,15.985,2.848740816116333
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",61,2026280486652,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.475,66.15,3.143754243850708
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",116,2026278786656,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.31,36.12,2.7146546840667725
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",126,2026277086615,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.0275,16.23,2.8559415340423584
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",62,2026280486653,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.125,62.4,3.104447841644287
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",117,2026278786657,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,7.88,36.78,2.743903160095215
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",127,2026277086616,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.105,16.195,2.8607146739959717
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",63,2026280486654,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.85,63.05,2.872148036956787
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",118,2026278786658,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.54,34.92,2.832261800765991
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",128,2026277086617,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.97,16.14,2.8419835567474365
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",64,2026280486655,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.025,65.2,2.994192600250244
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",119,2026278786659,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.45,35.06,2.7038962841033936
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",129,2026277086618,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.0125,16.12,2.8577795028686523
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",120,2026278786660,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.48,36.26,2.5952460765838623
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",65,2026280486656,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.025,64.3,2.8275179862976074
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",130,2026277086619,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.84,16.09,2.8545351028442383
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",121,2026278786661,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.26,36.0,2.8562231063842773
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",66,2026280486657,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.125,62.55,3.0581533908843994
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",131,2026277086620,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.0175,16.245,2.8900365829467773
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",122,2026278786662,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.3,35.46,2.8496456146240234
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",67,2026280486658,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.05,62.9,3.031097650527954
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",132,2026277086621,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,3.0325,16.125,2.9030706882476807
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",123,2026278786663,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.19,35.12,2.7367515563964844
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",68,2026280486659,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.75,63.25,3.098111629486084
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",133,2026277086622,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.1025,15.945,2.735520839691162
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",124,2026278786664,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.32,35.58,2.7850592136383057
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",69,2026280486660,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.45,62.25,2.9284064769744873
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",134,2026277086623,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.7825,16.015,2.778196334838867
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",125,2026278786665,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.29,36.28,2.993368625640869
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",70,2026280486661,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.775,64.15,3.131312608718872
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",135,2026277086624,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.935,16.23,2.8708441257476807
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",126,2026278786666,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.35,36.02,2.704601287841797
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",71,2026280486662,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.275,63.8,3.0415236949920654
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",136,2026277086625,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.9625,16.02,2.849346876144409
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",127,2026278786667,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.41,35.4,2.6961874961853027
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",72,2026280486663,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.85,64.9,3.096707820892334
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",137,2026277086626,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.8925,16.015,2.920508623123169
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",128,2026278786668,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.39,35.34,2.532254457473755
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",73,2026280486664,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.15,63.6,3.0456247329711914
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",138,2026277086627,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.9175,16.065,2.976170063018799
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",129,2026278786669,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,7.78,35.6,2.638915538787842
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",74,2026280486665,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.875,63.65,2.885530948638916
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",139,2026277086628,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.0325,16.185,2.8050343990325928
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",130,2026278786670,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.97,36.14,2.852203130722046
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",140,2026277086629,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.9625,16.13,2.6595449447631836
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",131,2026278786671,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.79,36.9,2.749619483947754
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",75,2026280486666,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.675,65.4,2.994062662124634
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",141,2026277086630,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.975,16.06,2.7988858222961426
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",132,2026278786672,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.31,34.68,2.811491012573242
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",76,2026280486667,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.575,62.95,3.0607199668884277
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",142,2026277086631,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.0325,15.97,2.998178482055664
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",133,2026278786673,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.38,35.62,2.8132128715515137
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",77,2026280486668,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.675,63.45,3.015392780303955
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",134,2026278786674,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.71,35.88,2.699375867843628
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",143,2026277086632,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.005,15.63,2.869141101837158
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",78,2026280486669,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.95,61.25,2.9128165245056152
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",135,2026278786675,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.32,36.46,2.6653542518615723
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",144,2026277086633,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.85,15.83,2.592510223388672
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",79,2026280486670,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.8,64.95,2.873903751373291
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",136,2026278786676,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.8,35.38,2.813156843185425
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",145,2026277086634,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.965,15.66,2.8082518577575684
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",80,2026280486671,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.4,65.05,2.884887218475342
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",137,2026278786677,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,7.62,35.36,2.6791272163391113
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",146,2026277086635,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.98,15.73,2.9963173866271973
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",81,2026280486672,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.025,63.7,2.997581958770752
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",138,2026278786678,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.57,34.96,2.7472288608551025
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",147,2026277086636,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.8525,15.865,2.8117105960845947
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",82,2026280486673,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.575,63.4,2.94211745262146
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",139,2026278786679,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.41,35.0,2.7259256839752197
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",148,2026277086637,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.875,16.02,2.7145328521728516
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",83,2026280486674,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.275,63.15,2.941800117492676
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",140,2026278786680,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.38,36.48,2.800022840499878
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",149,2026277086638,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.1875,15.8,2.791640281677246
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",84,2026280486675,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.35,63.0,2.9586021900177
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",141,2026278786681,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.82,36.26,2.6204075813293457
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",150,2026277086639,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.005,15.91,2.8978166580200195
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",85,2026280486676,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.8,63.9,2.974846124649048
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",142,2026278786682,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,7.73,36.92,2.6994407176971436
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",151,2026277086640,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.085,15.83,2.9182636737823486
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",86,2026280486677,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.925,65.0,2.967912435531616
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",143,2026278786683,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,7.75,36.02,2.6771960258483887
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",152,2026277086641,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.06,15.74,2.806758403778076
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",87,2026280486678,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.45,64.45,2.872098445892334
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",144,2026278786684,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.65,37.02,2.7573492527008057
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",153,2026277086642,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.81,15.745,2.76257586479187
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",145,2026278786685,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.45,37.62,2.770435094833374
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",88,2026280486679,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.025,65.05,2.985507011413574
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",154,2026277086643,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.8475,16.01,2.9002599716186523
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",146,2026278786686,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.31,35.8,2.757840633392334
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",89,2026280486680,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.575,64.05,3.074781894683838
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",155,2026277086644,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.8875,16.005,2.8667092323303223
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",147,2026278786687,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.04,35.94,2.45220685005188
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",90,2026280486681,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.725,62.9,2.8924949169158936
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",156,2026277086645,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.0625,15.93,2.9655539989471436
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",148,2026278786688,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.09,35.42,2.747321367263794
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",91,2026280486682,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.175,62.3,3.005140542984009
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",157,2026277086646,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.0825,16.045,2.8771514892578125
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",149,2026278786689,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.36,36.48,2.646341562271118
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",92,2026280486683,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.175,63.05,3.0135295391082764
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",158,2026277086647,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.995,15.825,2.8451366424560547
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",150,2026278786690,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.56,35.28,2.8234992027282715
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",93,2026280486684,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.475,64.15,3.1232528686523438
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",159,2026277086648,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.895,16.19,2.961195468902588
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",151,2026278786691,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.16,35.96,2.7493484020233154
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",94,2026280486685,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.45,64.5,3.166034698486328
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",160,2026277086649,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.0425,16.255,2.810091972351074
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",152,2026278786692,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.37,36.08,2.635802745819092
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",161,2026277086650,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.945,16.25,2.9792137145996094
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",153,2026278786693,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.56,36.08,2.6598284244537354
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",95,2026280486686,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.425,62.5,3.1333343982696533
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",154,2026278786694,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.57,35.34,2.64268159866333
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",162,2026277086651,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.94,15.99,2.7374441623687744
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",96,2026280486687,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.85,61.7,2.9553070068359375
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",155,2026278786695,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.5,36.82,2.6854588985443115
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",163,2026277086652,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.9525,16.205,2.7504308223724365
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",97,2026280486688,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.65,64.1,3.0754640102386475
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",156,2026278786696,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.72,35.98,2.7255990505218506
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",164,2026277086653,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.9425,16.315,2.836216688156128
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",98,2026280486689,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.05,65.9,2.9592843055725098
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",157,2026278786697,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.26,36.4,2.543523073196411
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",165,2026277086654,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.9875,15.625,2.831818103790283
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",99,2026280486690,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.725,63.4,3.126715660095215
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",158,2026278786698,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.74,36.06,2.585108518600464
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",166,2026277086655,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.1225,15.795,2.8505470752716064
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",100,2026280486691,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.45,64.8,2.991137742996216
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",159,2026278786699,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.05,36.24,2.45129132270813
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",167,2026277086656,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.97,16.24,2.8322060108184814
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",160,2026278786700,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.64,36.28,2.625509738922119
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",101,2026280486692,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.275,63.35,3.1782567501068115
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",168,2026277086657,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.93,15.79,2.6386470794677734
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",161,2026278786701,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.46,37.6,2.5446054935455322
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",102,2026280486693,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.55,65.25,3.3219363689422607
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",169,2026277086658,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,2.8525,15.91,2.771634817123413
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",162,2026278786702,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.35,34.92,2.6814279556274414
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",103,2026280486694,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.7,65.1,2.963253974914551
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",170,2026277086659,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.9225,15.78,3.066227436065674
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",163,2026278786703,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.66,35.84,2.7071115970611572
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",104,2026280486695,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.575,63.05,2.9563918113708496
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",171,2026277086660,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.8325,15.605,2.7216732501983643
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",164,2026278786704,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.55,35.42,2.7156214714050293
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",105,2026280486696,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.6,64.4,3.0025250911712646
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",172,2026277086661,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.6775,15.61,2.7325475215911865
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",165,2026278786705,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,7.84,36.64,2.6395254135131836
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",173,2026277086662,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.995,15.75,2.675119400024414
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",106,2026280486697,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.9,62.8,2.9674243927001953
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",166,2026278786706,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.6,36.86,2.675743341445923
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",174,2026277086663,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.0325,15.84,2.9115705490112305
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",107,2026280486698,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.125,64.65,3.158724546432495
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",167,2026278786707,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.38,35.72,2.9566829204559326
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",175,2026277086664,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.1225,16.09,2.830749273300171
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",108,2026280486699,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.725,65.1,3.0336430072784424
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",168,2026278786708,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.69,35.84,2.7230098247528076
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",176,2026277086665,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.0525,16.09,2.8732798099517822
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",169,2026278786709,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.07,35.86,2.852494716644287
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",109,2026280486700,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.925,65.6,3.0447912216186523
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",177,2026277086666,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.0125,16.045,2.672624111175537
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",170,2026278786710,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,7.97,36.2,2.683030366897583
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",110,2026280486701,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.825,62.85,3.0413758754730225
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",178,2026277086667,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.895,15.955,2.6992034912109375
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",171,2026278786711,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.79,35.96,2.9204771518707275
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",111,2026280486702,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.15,64.2,2.7393410205841064
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",179,2026277086668,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.055,15.905,2.8917770385742188
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",172,2026278786712,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.27,36.78,2.7764570713043213
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",112,2026280486703,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.875,64.35,2.919471025466919
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",180,2026277086669,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.8175,16.205,2.6800365447998047
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",173,2026278786713,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.18,35.5,2.8622424602508545
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",113,2026280486704,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.275,64.55,2.9847395420074463
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",181,2026277086670,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.965,15.695,2.690671920776367
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",174,2026278786714,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.41,35.28,2.8377726078033447
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",114,2026280486705,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.0,62.65,2.932971477508545
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",182,2026277086671,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.9675,16.21,2.813450813293457
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",175,2026278786715,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.54,35.4,2.835528612136841
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",115,2026280486706,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.475,64.2,2.8659136295318604
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",183,2026277086672,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.075,15.905,2.6323249340057373
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",176,2026278786716,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.21,37.34,2.6953861713409424
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",116,2026280486707,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.975,63.45,2.8360157012939453
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",184,2026277086673,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.99,15.93,2.7683145999908447
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",177,2026278786717,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.12,35.04,2.6769514083862305
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",117,2026280486708,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.675,65.6,2.9686522483825684
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",185,2026277086674,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.9325,16.085,2.913133144378662
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",178,2026278786718,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,7.94,35.2,2.599670886993408
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",118,2026280486709,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.325,63.35,2.849153995513916
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",186,2026277086675,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.9125,15.655,2.8598833084106445
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",179,2026278786719,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.16,36.12,2.8071415424346924
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",119,2026280486710,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.475,64.25,2.98077392578125
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",187,2026277086676,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.0425,15.54,3.005392074584961
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",180,2026278786720,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.41,35.26,2.6762990951538086
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",120,2026280486711,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.5,65.45,3.0293264389038086
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",188,2026277086677,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.9725,16.065,2.9955496788024902
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",181,2026278786721,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.37,35.54,2.7684247493743896
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",121,2026280486712,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.425,64.45,3.1425986289978027
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",189,2026277086678,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.135,16.045,2.9655673503875732
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",182,2026278786722,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.61,35.74,2.754337787628174
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",122,2026280486713,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.0,64.4,2.9692137241363525
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",183,2026278786723,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,9.2,35.3,2.7238857746124268
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",190,2026277086679,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.0275,15.875,2.8398141860961914
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",123,2026280486714,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.525,63.5,2.8378994464874268
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",191,2026277086680,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.01,16.145,2.8369638919830322
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",184,2026278786724,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.75,36.28,3.0540456771850586
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",124,2026280486715,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.15,64.65,3.100026845932007
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",192,2026277086681,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.915,16.12,2.8728346824645996
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",185,2026278786725,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.79,36.36,3.0363411903381348
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",125,2026280486716,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.2,62.8,2.833836555480957
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",193,2026277086682,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.0325,15.84,2.8751964569091797
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",186,2026278786726,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.55,36.74,2.9425466060638428
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",194,2026277086683,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.9575,15.73,2.8027937412261963
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",126,2026280486717,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.85,62.8,3.0781054496765137
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",187,2026278786727,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.47,35.84,2.8561835289001465
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",195,2026277086684,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.9875,15.905,2.9936740398406982
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",188,2026278786728,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.48,35.98,2.7343616485595703
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",127,2026280486718,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.15,63.15,3.200813055038452
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",189,2026278786729,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.31,36.26,2.690866470336914
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",196,2026277086685,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.88,15.685,3.033832311630249
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",128,2026280486719,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.5,62.55,3.0550150871276855
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",190,2026278786730,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.38,35.9,2.975127935409546
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",197,2026277086686,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.91,16.235,2.786984920501709
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",129,2026280486720,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.325,62.75,2.9351441860198975
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",198,2026277086687,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.0375,16.085,2.6904351711273193
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",191,2026278786731,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.77,36.9,2.85748291015625
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",130,2026280486721,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.325,64.3,3.107640027999878
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",199,2026277086688,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.02,15.895,2.820645332336426
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",192,2026278786732,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.53,36.6,2.9541120529174805
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",131,2026280486722,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.85,66.55,2.887346029281616
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",200,2026277086689,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.91,16.2,2.7530527114868164
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",193,2026278786733,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,7.8,36.8,2.593921661376953
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",132,2026280486723,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.325,64.35,2.792642831802368
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",201,2026277086690,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.825,15.805,2.6595263481140137
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",194,2026278786734,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.1,36.32,2.8731889724731445
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",133,2026280486724,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.925,65.0,2.7529234886169434
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",202,2026277086691,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.9125,16.295,2.780810832977295
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",195,2026278786735,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.2,36.78,2.7375028133392334
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",134,2026280486725,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,16.525,63.05,2.899111270904541
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",203,2026277086692,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.0,15.915,2.897024393081665
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",196,2026278786736,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.74,36.14,2.6936373710632324
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",135,2026280486726,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.2,65.7,2.9141643047332764
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",197,2026278786737,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.42,35.8,2.5815377235412598
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",204,2026277086693,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,1.0,1,0,2.9825,16.065,2.953667640686035
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",136,2026280486727,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.875,63.0,3.100116491317749
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",198,2026278786738,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.78,36.64,2.834347724914551
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",205,2026277086694,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.0225,16.25,2.7519314289093018
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",137,2026280486728,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.125,64.1,3.2684154510498047
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",199,2026278786739,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.17,36.82,2.807943105697632
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",206,2026277086695,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.8575,15.88,2.8385205268859863
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",138,2026280486729,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.05,63.0,3.0181970596313477
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",200,2026278786740,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.31,35.64,2.745450496673584
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",207,2026277086696,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,3.015,16.215,2.788454294204712
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",139,2026280486730,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.675,65.8,3.1434881687164307
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",208,2026277086697,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.955,15.955,2.6866090297698975
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",201,2026278786741,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.67,35.16,2.9717369079589844
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",202,2026278786742,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.43,35.86,2.7200732231140137
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",140,2026280486731,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.55,61.95,3.012713670730591
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",209,2026277086698,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.8975,16.18,2.7834999561309814
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",203,2026278786743,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.72,36.92,2.7789289951324463
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",141,2026280486732,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.35,62.35,3.030255079269409
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",210,2026277086699,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.99,16.145,3.122279405593872
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",204,2026278786744,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.98,35.72,2.958031177520752
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",211,2026277086700,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.99,16.13,2.8674392700195312
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",142,2026280486733,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.125,64.2,3.0059499740600586
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",205,2026278786745,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.75,36.64,2.6496410369873047
production-20261029T053925Z-29f79c,"B|quiet|d5|p0.0003|pi0|k1,1,1",212,2026277086701,B,quiet,5,0.0003,0.0,1.0,1.0,1.0,5,5,5,400,0.0,0,0,2.99,15.96,2.862957239151001
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",143,2026280486734,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.35,63.8,2.901655673980713
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",206,2026278786746,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.88,35.44,2.8551087379455566
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",144,2026280486735,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.8,62.3,2.9789693355560303
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",207,2026278786747,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.43,36.58,2.796658754348755
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",145,2026280486736,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.3,64.3,3.0044591426849365
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",208,2026278786748,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.0,35.84,2.6808359622955322
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",146,2026280486737,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.025,64.35,2.982064723968506
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",209,2026278786749,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.28,36.02,2.7509491443634033
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",147,2026280486738,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.5,64.3,3.000159978866577
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",210,2026278786750,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.72,36.06,2.75015926361084
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",148,2026280486739,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.175,63.4,2.8054933547973633
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",211,2026278786751,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.07,35.98,2.622230052947998
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",149,2026280486740,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.8,63.6,3.0620458126068115
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",212,2026278786752,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.63,36.38,2.631336212158203
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",150,2026280486741,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.825,62.3,2.8835630416870117
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",213,2026278786753,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.34,36.74,2.805298328399658
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",151,2026280486742,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.225,63.1,2.9363696575164795
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",214,2026278786754,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.7,36.14,2.7253501415252686
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",215,2026278786755,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.29,36.52,2.4629878997802734
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",152,2026280486743,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.15,63.85,2.878230094909668
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",216,2026278786756,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.64,36.24,2.6998956203460693
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",153,2026280486744,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.75,67.1,2.8290164470672607
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",217,2026278786757,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.22,36.3,2.84790301322937
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",154,2026280486745,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.475,66.1,3.0925838947296143
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",218,2026278786758,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.11,35.16,2.924752712249756
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",155,2026280486746,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.025,63.25,2.998842239379883
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",219,2026278786759,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.04,35.44,2.5774965286254883
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",156,2026280486747,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.175,64.7,3.062760591506958
production-20261029T053925Z-29f79c,"B|quiet|d7|p0.0003|pi0|k1,1,1",220,2026278786760,B,quiet,7,0.0003,0.0,1.0,1.0,1.0,7,7,7,100,0.0,0,0,8.75,35.6,2.644167900085449
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",157,2026280486748,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.65,66.0,3.078646659851074
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",158,2026280486749,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.55,65.7,3.18007493019104
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",159,2026280486750,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.35,62.6,3.0587947368621826
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",160,2026280486751,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.975,65.35,2.91214656829834
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",161,2026280486752,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.875,65.05,3.0790066719055176
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",162,2026280486753,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.825,61.85,2.960968017578125
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",163,2026280486754,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.15,64.65,2.9691126346588135
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",164,2026280486755,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.225,65.85,2.9537415504455566
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",165,2026280486756,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.275,62.65,3.0279319286346436
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",166,2026280486757,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.35,65.8,2.834843635559082
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",167,2026280486758,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.95,63.75,3.2158546447753906
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",168,2026280486759,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.3,65.4,2.971820592880249
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",169,2026280486760,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.175,63.75,3.117384910583496
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",170,2026280486761,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.05,66.05,3.1025357246398926
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",171,2026280486762,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.85,63.1,3.010237693786621
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",172,2026280486763,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.575,65.1,2.96356201171875
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",173,2026280486764,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.925,61.6,3.0492148399353027
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",174,2026280486765,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.2,64.5,2.9219844341278076
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",175,2026280486766,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.825,64.0,2.867051839828491
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",176,2026280486767,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.325,64.6,2.9623122215270996
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",177,2026280486768,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.525,63.8,2.950587034225464
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",178,2026280486769,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.85,64.6,3.1137490272521973
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",179,2026280486770,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.325,63.85,2.9386324882507324
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",180,2026280486771,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.425,64.35,2.9435975551605225
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",181,2026280486772,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.55,63.25,3.0154457092285156
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",182,2026280486773,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.75,63.25,2.992647171020508
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",183,2026280486774,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.85,63.8,2.902073383331299
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",184,2026280486775,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.5,63.3,2.9811904430389404
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",185,2026280486776,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.725,64.55,2.941758871078491
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",186,2026280486777,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.475,63.55,2.9917943477630615
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",187,2026280486778,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.475,62.25,2.958706855773926
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",188,2026280486779,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.55,65.55,2.895413398742676
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",189,2026280486780,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.525,63.2,2.906625509262085
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",190,2026280486781,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.325,64.3,2.953875780105591
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",191,2026280486782,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.6,63.95,3.031283140182495
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",192,2026280486783,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.05,64.5,3.1043031215667725
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",193,2026280486784,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,17.15,63.55,3.035045862197876
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",194,2026280486785,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.8,61.1,3.0904741287231445
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",195,2026280486786,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.325,63.05,3.0342185497283936
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",196,2026280486787,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.825,63.35,3.138401985168457
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",197,2026280486788,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.225,64.2,3.08254075050354
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",198,2026280486789,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,19.2,63.5,3.068356990814209
production-20261029T053925Z-29f79c,"B|quiet|d9|p0.0003|pi0|k1,1,1",199,2026280486790,B,quiet,9,0.0003,0.0,1.0,1.0,1.0,9,9,9,40,0.0,0,0,18.15,61.1,2.8954083919525146
"""
EMBED_CAL = """run_id,scenario,d,p,p_idle,shots,rounds,PL_x,PL_z,pL_per_round
production-20261029T053925Z-29f79c,A,3,0.002,0.002,400000,9,0.0209475,0.0205375,0.004696729982226866
production-20261029T053925Z-29f79c,A,3,0.003,0.003,400000,9,0.044095,0.0431,0.010085852179974819
production-20261029T053925Z-29f79c,A,3,0.004,0.004,400000,9,0.0732575,0.0718525,0.01726955015777304
production-20261029T053925Z-29f79c,A,3,0.006,0.006,400000,9,0.142255,0.13831,0.03592786887737964
production-20261029T053925Z-29f79c,A,5,0.002,0.002,400000,15,0.0082475,0.0082075,0.0011055143359350317
production-20261029T053925Z-29f79c,A,5,0.003,0.003,400000,15,0.0260225,0.0257625,0.0035386604166322
production-20261029T053925Z-29f79c,A,5,0.004,0.004,400000,15,0.0565275,0.05554,0.007892696443234926
production-20261029T053925Z-29f79c,A,5,0.006,0.006,400000,15,0.145945,0.1462225,0.022773209560969565
production-20261029T053925Z-29f79c,A,7,0.002,0.002,200000,21,0.00253,0.00248,0.0002391425117175272
production-20261029T053925Z-29f79c,A,7,0.003,0.003,200000,21,0.012855,0.012705,0.0012322089325351726
production-20261029T053925Z-29f79c,A,7,0.004,0.004,200000,21,0.037035,0.03661,0.003636127918316079
production-20261029T053925Z-29f79c,A,7,0.006,0.006,200000,21,0.1371,0.138405,0.01522959876110136
production-20261029T053925Z-29f79c,A,9,0.002,0.002,100000,27,0.00074,0.00097,6.338648271725367e-05
production-20261029T053925Z-29f79c,A,9,0.003,0.003,100000,27,0.00602,0.00615,0.00045340344658495457
production-20261029T053925Z-29f79c,A,9,0.004,0.004,100000,27,0.02354,0.02377,0.0017934174893302068
production-20261029T053925Z-29f79c,A,9,0.006,0.006,100000,27,0.12733,0.12847,0.010882815392591005
production-20261029T053925Z-29f79c,A,11,0.002,0.002,100000,33,0.00024,0.00024,1.454884073659457e-05
production-20261029T053925Z-29f79c,A,11,0.003,0.003,100000,33,0.00273,0.00243,0.00015675749602317435
production-20261029T053925Z-29f79c,A,11,0.004,0.004,100000,33,0.01364,0.01393,0.0008468299834580861
production-20261029T053925Z-29f79c,A,11,0.006,0.006,100000,33,0.11682,0.11764,0.008063507072532394
production-20261029T053925Z-29f79c,B,3,0.002,0.0,400000,9,0.0047475,0.0047925,0.0010645216855857398
production-20261029T053925Z-29f79c,B,3,0.003,0.0,400000,9,0.0106825,0.010475,0.002373240269971544
production-20261029T053925Z-29f79c,B,3,0.004,0.0,400000,9,0.01903,0.0182725,0.004215126227569699
production-20261029T053925Z-29f79c,B,3,0.006,0.0,400000,9,0.0395425,0.038555,0.008994452307498624
production-20261029T053925Z-29f79c,B,5,0.002,0.0,400000,15,0.0015925,0.0015825,0.00021198093243945948
production-20261029T053925Z-29f79c,B,5,0.003,0.0,400000,15,0.0047775,0.0049575,0.0006519680646405179
production-20261029T053925Z-29f79c,B,5,0.004,0.0,400000,15,0.0115025,0.011345,0.0015396509309996143
production-20261029T053925Z-29f79c,B,5,0.006,0.0,400000,15,0.0358675,0.0353675,0.004914528498189186
production-20261029T053925Z-29f79c,B,7,0.002,0.0,200000,21,0.00037,0.00031,3.239152402534362e-05
production-20261029T053925Z-29f79c,B,7,0.003,0.0,200000,21,0.00192,0.001825,0.00017865234451752698
production-20261029T053925Z-29f79c,B,7,0.004,0.0,200000,21,0.005655,0.005745,0.0005458263185821877
production-20261029T053925Z-29f79c,B,7,0.006,0.0,200000,21,0.02715,0.02709,0.0026520221035039038
production-20261029T053925Z-29f79c,B,9,0.002,0.0,100000,27,6e-05,8e-05,5.185541872121391e-06
production-20261029T053925Z-29f79c,B,9,0.003,0.0,100000,27,0.00066,0.00054,4.447040103361388e-05
production-20261029T053925Z-29f79c,B,9,0.004,0.0,100000,27,0.00242,0.00261,0.00018674961898990095
production-20261029T053925Z-29f79c,B,9,0.006,0.0,100000,27,0.01829,0.01872,0.001395778603096809
production-20261029T053925Z-29f79c,B,11,0.002,0.0,100000,33,1e-05,0.0,3.030332415554504e-07
production-20261029T053925Z-29f79c,B,11,0.003,0.0,100000,33,0.00015,0.00012,8.182902676290293e-06
production-20261029T053925Z-29f79c,B,11,0.004,0.0,100000,33,0.00098,0.00102,6.066493100692627e-05
production-20261029T053925Z-29f79c,B,11,0.006,0.0,100000,33,0.01167,0.01194,0.0007237748968457747
"""
REF_SHA = {'mc': 'cf223095e465b040', 'cal': '5176e76e3667c816'}
sha = lambda t: hashlib.sha256(t.encode()).hexdigest()[:16]
if DATA_MODE == 'reference':
    mc_text, cal_text, src = EMBED_MC, EMBED_CAL, 'embedded reference data'
elif DATA_MODE in ('production', 'quick'):
    suf = '' if DATA_MODE == 'production' else '_quick'
    paths = [f'{RESULTS}/d4_noisy_mc{suf}.csv', f'{RESULTS}/surface_code_plog{suf}.csv']
    missing = [x for x in paths if not os.path.exists(x)]
    if missing:
        raise FileNotFoundError(f'DATA_MODE={DATA_MODE!r}: {missing} not found. Run D4_noisy_simulation.ipynb with '
                                f'QUICK={DATA_MODE == "quick"} (and upload its results/ here), or set DATA_MODE = "reference".')
    mc_text, cal_text, src = open(paths[0]).read(), open(paths[1]).read(), ' + '.join(paths)
else:
    raise ValueError(DATA_MODE)
mc, cal = pd.read_csv(io.StringIO(mc_text)), pd.read_csv(io.StringIO(cal_text))
OUT_SUFFIX = '_quick' if DATA_MODE == 'quick' else ''
PROV = dict(data_mode=DATA_MODE, mc_sha256=sha(mc_text), cal_sha256=sha(cal_text),
            mc_run_ids=';'.join(sorted(map(str, mc.run_id.unique()))) if 'run_id' in mc else 'unknown',
            cal_run_ids=';'.join(sorted(map(str, cal.run_id.unique()))) if 'run_id' in cal else 'unknown',
            mc_shots=int(mc.shots.sum()))
print('source:', src)
for k, v in PROV.items():
    print(f'  {k} = {v}')
print('  identical to the embedded reference:', PROV['mc_sha256'] == REF_SHA['mc'] and PROV['cal_sha256'] == REF_SHA['cal'])
print('  python', platform.python_version(), '| numpy', np.__version__, '| stim', stim.__version__, '| pymatching', pymatching.__version__)
# the same noise convention on both sides: p_idle / p per scenario must agree
for s, f in (('A', 1.0), ('B', 0.0)):
    for name, df in (('MC', mc), ('calibration', cal)):
        r_ = df[df.scenario == s]
        assert np.allclose(r_.p_idle, f * r_.p), f'{name}: scenario {s} has p_idle != {f} p'
print(len(mc), 'Monte Carlo chunks,', len(cal), 'calibration points')

## 第I部 表面符号の論理誤り率の較正（Litinski 方式の入力）
1 ラウンドあたりの論理誤り（X と Z の和）$p_L$ を $p_L = A\,(p/p_{\rm th})^{(d+1)/2}$ に当てはめる（GoSC 式 (10) と同じ形。GoSC は $A=0.1$, $p_{\rm th}=0.01$）。閾値より十分下の $p\le0.004$ で、失敗が 20 回以上ある点を使う。$p_{\rm th}$ は仮定した関数形の当てはめの係数で、有限サイズスケーリングで求めた閾値ではない。測定は $p=0.002$〜$0.006$ なので、$p\le10^{-3}$ の $p_L$ は**この関数形による外挿**である。不確かさは各点の失敗数の二項分布による再標本化（`N_BOOT` 回）の 16〜84% 範囲。

In [ ]:
PLOG = {}
rows = []
rng_b = np.random.default_rng(SEED)
for s in SCENARIOS:
    c_ = cal[(cal.scenario == s) & (cal.pL_per_round > 0)]
    w_ok = c_[((c_.PL_x + c_.PL_z) * c_.shots >= 20) & (c_.p <= 0.004)]
    A, pth = fit_plog(list(zip(w_ok.d, w_ok.p, w_ok.pL_per_round)))
    PLOG[s] = make_plog(A, pth) if PLOG_SOURCE == 'calibrated' else plog_gosc
    bs = []
    for b in range(N_BOOT):
        px = rng_b.binomial(w_ok.shots, w_ok.PL_x) / w_ok.shots
        pz = rng_b.binomial(w_ok.shots, w_ok.PL_z) / w_ok.shots
        r_ = w_ok.rounds
        pl = (1 - (1 - 2 * px) ** (1 / r_)) / 2 + (1 - (1 - 2 * pz) ** (1 / r_)) / 2
        ok = pl > 0
        bs.append(fit_plog(list(zip(w_ok.d[ok], w_ok.p[ok], pl[ok]))))
    bs = np.array(bs)
    pl11 = [float(make_plog(a_, t_)(1e-3, 11)) for a_, t_ in bs]
    rows.append(dict(scenario=s, A=A, A_16=np.quantile(bs[:, 0], 0.16), A_84=np.quantile(bs[:, 0], 0.84),
                     p_th=pth, p_th_16=np.quantile(bs[:, 1], 0.16), p_th_84=np.quantile(bs[:, 1], 0.84), points=len(w_ok),
                     pL_d11_p1e3=float(make_plog(A, pth)(1e-3, 11)), pL_d11_p1e3_16=np.quantile(pl11, 0.16),
                     pL_d11_p1e3_84=np.quantile(pl11, 0.84)))
pd.DataFrame(rows)

## 第II部 Kobayashi 方式
### II-1. モンテカルロの結果
ショットごとの不忠実度（0, 1/2, 1）の平均 $\hat\varepsilon_K$ と標準誤差（SE、95% 信頼区間ではない）。集計は全ての条件（シナリオ、$d$、$p$、$p_{\rm idle}$、$\kappa_0,\kappa_{\rm in},\kappa_{\rm out}$、`window`）ごとで、異なる条件は混ぜない。
* `window = noisy`：本来のシミュレーション（全ての位置に雑音）。
* `window = quiet`：**診断用の別の実験**。切り替えの時間窓（D4 相最後のラウンド、$U$ 直前の $Z$ ラウンド、$U$、switch-out、FSC に戻った最初のラウンド）だけ雑音を 0 にし、**同じ復号器**で評価したもの。本研究の復号器が 1 個の故障を訂正できない場所の大部分がこの窓にある（`validation.ipynb` N5）ので、「窓の外の誤りがどれだけ抑えられるか」を見る。**最適な復号器の性能の限界（上界・下界）ではない**：最適な復号器は窓の外の誤りも改善しうるし、窓の誤りの影響が 0 になるとも限らない。
* $\kappa_{\rm in}\neq1$ の行は switch-in のラウンド数の感度。

In [ ]:
KEYS = ['window', 'scenario', 'd', 'p', 'p_idle', 'kappa0', 'kappa_in', 'kappa_out']
g = mc.groupby(KEYS).agg(shots=('shots', 'sum'), s=('sum_infidelity', 'sum'), n_fail=('n_fail', 'sum'),
                         n_half=('n_half', 'sum'), errors_per_shot=('mean_errors', 'mean')).reset_index()
g['eps'] = g.s / g.shots
n_one = g.n_fail - g.n_half
g['stderr'] = np.sqrt(np.maximum((n_one + 0.25 * g.n_half) / g.shots - g.eps ** 2, 0) / g.shots)
g['eps_over_p'] = g.eps / g.p
k1 = (g.kappa0 == 1) & (g.kappa_in == 1) & (g.kappa_out == 1)
main = g[k1 & (g.window == 'noisy')].copy()
quiet = g[k1 & (g.window == 'quiet')].copy()
g.round({'eps': 6, 'stderr': 6, 'errors_per_shot': 1, 'eps_over_p': 2})

### II-2. $d$ と $p$ への依存のモデル（外挿の根拠）
データの構造（II-1）：noisy では $d\ge5$ で $\varepsilon_K$ が $d$ でほとんど下がらない。quiet（窓の雑音なし）では $d$ とともに下がる。そこで次の**モデル（仮定）**を使う。
* **bulk**（窓の外の寄与）：quiet のデータに $\varepsilon_{\rm bulk}=\exp[a + c\,d + (b + e\,d)\log p]$。
* **窓の項**：noisy のデータ（$d\ge5$）に $\varepsilon_K=\varepsilon_{\rm bulk}+W$、$W(p)=w\,p^{\gamma}$（$d$ によらないと**仮定**）。感度を見るため $W=w\,p^{\gamma}e^{\delta(d-7)}$（$d$ に依存しうる）も当てはめる。
* 当てはめは最尤法：各点の不忠実度の和 $s$（1/2 単位で数えた $2s$）を、平均 $2\,\mathrm{shots}\,\varepsilon$ の Poisson 分布として扱う。**失敗 0 の点も含める**（失敗の多い点だけの対数回帰にしない）。
* **当てはめの可否の判定**：bulk の点について、設計行列の階数 4、異なる $d$ が 3 個以上、異なる $p$ が 2 個以上、失敗のある点が 5 個以上、失敗の合計 20 回以上。満たさなければ**ここで停止し、最小費用は計算しない**（`quick` のデータはこれを満たさない）。
* **不確かさ**：当てはめたモデルから各点の $2s$ を Poisson で再標本化して当てはめ直すパラメトリック・ブートストラップ（`N_BOOT` 回；bulk、窓の項、$d$ 依存の窓の項はそれぞれ自分の当てはめから再標本化）。打ち切り規則（時間または失敗数）の影響は考慮していない。
* **外挿**：$d>9$ は $d$ 方向の外挿、quiet の最小の $p$ 未満は bulk の $p$ 方向の外挿。表では別々の列で示す。

In [ ]:
from scipy.optimize import minimize
def _design_bulk(df):
    return np.stack([np.ones(len(df)), df.d, np.log(df.p), df.d * np.log(df.p)], axis=1)
def fit_bulk(df, y2):
    X_ = _design_bulk(df); ex2 = 2 * df.shots.values
    pos = y2 > 0
    th0, *_ = np.linalg.lstsq(X_[pos], np.log(y2[pos] / ex2[pos]), rcond=None)
    f = lambda th: np.sum(ex2 * np.exp(X_ @ th) - y2 * (X_ @ th))
    return minimize(f, th0, method='BFGS').x
def bulk_eps(th, d, p):
    return np.exp(th[0] + th[1] * d + (th[2] + th[3] * d) * np.log(p))
def win_eps(t, d, p):
    return np.exp(t[0] + t[1] * np.log(p) + (t[2] * (d - 7) if len(t) == 3 else 0))
def fit_window(df, y2, th_b, alt=False):
    ex2 = 2 * df.shots.values; b_ = bulk_eps(th_b, df.d.values, df.p.values)
    def f(t):
        mu = ex2 * (b_ + win_eps(t, df.d.values, df.p.values))
        return np.sum(mu - y2 * np.log(mu))
    t0 = np.array([np.log(5.0), 1.0] + ([0.0] if alt else []))
    return minimize(f, t0, method='Nelder-Mead', options=dict(maxiter=4000, xatol=1e-6, fatol=1e-9)).x

def identifiable(df):
    X_ = _design_bulk(df)
    return dict(points=len(df), rank=int(np.linalg.matrix_rank(X_)) if len(df) else 0, distinct_d=df.d.nunique(),
                distinct_p=df.p.nunique(), points_with_failures=int((df.n_fail > 0).sum()), failures=int(df.n_fail.sum()))
CHECK = {s: identifiable(quiet[quiet.scenario == s]) for s in SCENARIOS}
print(pd.DataFrame(CHECK).T)
bad = [s for s, c_ in CHECK.items() if not (c_['rank'] == 4 and c_['distinct_d'] >= 3 and c_['distinct_p'] >= 2
                                           and c_['points_with_failures'] >= 5 and c_['failures'] >= 20)]
if bad:
    raise RuntimeError(f'not enough data to identify the bulk model for scenario(s) {bad} (DATA_MODE={DATA_MODE!r}): '
                       'the analysis stops here (aggregated results above); minimum costs are not computed.')

FITS = {}
rows = []
rng_b = np.random.default_rng(SEED + 1)
q16 = lambda v: np.quantile(v, 0.16)
q84 = lambda v: np.quantile(v, 0.84)
supp = lambda th, p: np.exp(-2 * (th[1] + th[3] * np.log(p)))
for s in SCENARIOS:
    q_ = quiet[quiet.scenario == s]; n_ = main[(main.scenario == s) & (main.d >= 5)]
    yq, yn = 2 * q_.s.values, 2 * n_.s.values
    th_b = fit_bulk(q_, yq); t_w = fit_window(n_, yn, th_b); t_a = fit_window(n_, yn, th_b, alt=True)
    boot = []
    muq = 2 * q_.shots.values * bulk_eps(th_b, q_.d.values, q_.p.values)
    mun = 2 * n_.shots.values * (bulk_eps(th_b, n_.d.values, n_.p.values) + win_eps(t_w, n_.d.values, n_.p.values))
    mua = 2 * n_.shots.values * (bulk_eps(th_b, n_.d.values, n_.p.values) + win_eps(t_a, n_.d.values, n_.p.values))
    for b in range(N_BOOT):                  # each model is resampled from its own fit
        yq_b = rng_b.poisson(muq).astype(float)
        yn_b, ya_b = rng_b.poisson(mun).astype(float), rng_b.poisson(mua).astype(float)
        tb = fit_bulk(q_, yq_b)
        boot.append((tb, fit_window(n_, yn_b, tb), fit_window(n_, ya_b, tb, alt=True)))
    p_ref = float(q_.p.min())
    FITS[s] = dict(central=(th_b, t_w, t_a), boot=boot, p_min_bulk=p_ref, d_max_data=int(main.d.max()))
    sup_b = [supp(bt[0], p_ref) for bt in boot]
    Wp = [win_eps(bt[1], 7, p_ref) / p_ref for bt in boot]
    dl = [bt[2][2] for bt in boot]
    rows.append(dict(scenario=s, bulk_points=len(q_), window_points=len(n_), p_ref=p_ref,
                     bulk_suppression_per_2d=supp(th_b, p_ref), bulk_supp_16=q16(sup_b), bulk_supp_84=q84(sup_b),
                     W_over_p=win_eps(t_w, 7, p_ref) / p_ref, W_over_p_16=q16(Wp), W_over_p_84=q84(Wp),
                     W_gamma=t_w[1], alt_delta=t_a[2], alt_delta_16=q16(dl), alt_delta_84=q84(dl)))

def eps_K_model(s, d, p, w='noisy', params=None, alt=False):
    th_b, t_w, t_a = params if params is not None else FITS[s]['central']
    v = bulk_eps(th_b, d, p)
    if w == 'noisy':
        v = v + win_eps(t_a if alt else t_w, d, p)
    return float(min(v, 1.0))
pd.DataFrame(rows).round(4)

`bulk_suppression_per_2d`：当てはめた bulk のモデルで距離を 2 増やしたときの減少率（$p$ = `p_ref`＝quiet の最小の $p$、測定範囲の端）。`W_over_p`：窓の項 $W/p$（$d=7$、同じ $p$）。`alt_delta`：$W$ の $d$ 依存を許したときの $\delta$（$\delta<0$ なら $d$ とともに減る）。いずれも**当てはめたモデルの値**であり、実測の抑制率ではない。

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8), sharex=True, sharey=True)
mk = {3: 'o', 5: 's', 7: '^', 9: 'D'}
cols = {3: '#8fb8ea', 5: '#5a95dc', 7: '#2a78d6', 9: '#174a8a'}
pp = np.logspace(-4.2, -2.9, 40)
for i, (w_, data) in enumerate((('noisy', main), ('quiet', quiet))):
    for j, s in enumerate(SCENARIOS):
        ax = axes[i, j]
        for d in sorted(data.d.unique()):
            m_ = data[(data.scenario == s) & (data.d == d)].sort_values('p')
            pos = m_[m_.n_fail > 0]; zero = m_[m_.n_fail == 0]
            ax.errorbar(pos.p, pos.eps, yerr=pos.stderr, fmt=mk.get(d, 'o'), color=cols.get(d, 'k'), capsize=2,
                        label=f'd = {d} (data ± SE)')
            if len(zero):
                ax.plot(zero.p, 3 / zero.shots, 'v', color=cols.get(d, 'k'), mfc='none')
            ax.plot(pp, [eps_K_model(s, d, x, w_) for x in pp], '-', color=cols.get(d, 'k'), lw=1, alpha=0.7)
        ax.plot(pp, pp, ':', color='#52514e', lw=1, label='error = p')
        ax.set_xscale('log'); ax.set_yscale('log')
        ax.set_title(f'[{DATA_MODE}] scenario {s}, ' + ('full simulation' if w_ == 'noisy' else
                     'diagnostic: switching window noise-free'), fontsize=9)
        if i == 1: ax.set_xlabel('physical error rate p')
        if j == 0: ax.set_ylabel('output infidelity  ε_K(d, p)')
axes[0, 1].legend(fontsize=7, title='lines: fitted model (extrapolated outside the data)\nopen triangles: no failure, 3/shots',
                  title_fontsize=7)
plt.tight_layout(); plt.savefig(f'{FIGURES}/d4_logical_error{OUT_SUFFIX}.png', dpi=150); plt.show()

### II-3. 資源（同じ回路から数える）
量子ビット数（ピーク）と時間ステップ数を、シミュレーションに使ったのと同じ回路（$\kappa_0=\kappa_{\rm in}=\kappa_{\rm out}=1$）から $d=3,\dots,11$ で数え、$d$ の多項式に当てはめて大きな $d$ に延長する（$d=3..11$ で直接数えた値と一致することを確認）。体積は「各ブロックで使う量子ビット数 × そのブロックの時間ステップ数」の和（雑音なしの最終ラウンドは含めない）。switch-out は $g$ の読み出し 1 ステップ＋補正 CZ の固定スケジュールの層数（測定結果によらない最悪値。シミュレーションの仮想的な遅延測定のモーメントは時間に数えない：`block_steps`）。
**感度**：FSC のラウンドを、検証済みの並列スケジュール（全検査を同時に測る 8 ステップ：`validation.ipynb` N10）で数えた場合の費用も `*_fsc8` として示す。**時間・体積だけの感度**であり、その誤り率はシミュレーションしていない（本研究の flux 復号器では並列スケジュールの方が $\varepsilon_K$ が大きくなったため、本番は 14 ステップのスケジュールでシミュレーションしている：`docs/methods.md` §4.1）。
**含めていない費用**：FSC から通常の回転表面符号パッチへの変換（未構成、[KZH] にも記述なし）、古典処理の待ち時間、フィードフォワードの $X$ 補正（雑音なし・時間 0 の理想化）。一方、測定スケジュールは最適化していない（FSC 14 ステップ/ラウンド、表面符号は 6）。前者は費用の過小評価、後者は過大評価の要因で、全体として一方向に「保守的」とは言えない。

In [ ]:
def kobayashi_counts(d, fsc_steps=None):
    """fsc_steps: count every FSC round as this many time steps (sensitivity: 8 = the verified parallel
    schedule, validation N10) instead of the simulated 14-step schedule."""
    circ = D4SwitchingCircuit(d, d, d, d)
    vol, steps = 0, 0
    for blk in circ.blocks:
        if blk['name'][0] == 'FIN':
            continue
        code_ = blk['code']
        if code_ == 'FSC':
            active = set(circ.data_rb) | set(circ.anc['FSC'])
        elif code_ == 'D4':
            active = set(circ.data_all) | set(circ.anc['D4'])
        else:
            active = set(circ.data_all)                   # U and the switch-out
        n_steps = block_steps(circ, blk) if (fsc_steps is None or code_ != 'FSC' or blk['name'][0] == 'FIN') else fsc_steps
        vol += len(active) * n_steps
        steps += n_steps
    return dict(d=d, qubits=circ.nq, time_steps=steps, volume=vol, switch_out_steps=circ.so_steps)
KC = pd.DataFrame([kobayashi_counts(d) for d in (3, 5, 7, 9, 11)])
POLY = {k: np.polyfit(KC.d, KC[k], 3 if k == 'volume' else 2) for k in ('qubits', 'time_steps', 'volume')}
def kobayashi_cost(d):
    return {k: float(np.polyval(POLY[k], d)) for k in POLY}
KC['volume_poly'] = [kobayashi_cost(d)['volume'] for d in KC.d]
assert np.allclose(KC.volume_poly, KC.volume) and np.allclose([kobayashi_cost(d)['time_steps'] for d in KC.d], KC.time_steps)
KC8 = pd.DataFrame([kobayashi_counts(d, fsc_steps=8) for d in (3, 5, 7, 9, 11)])
POLY8 = {k: np.polyfit(KC8.d, KC8[k], 3 if k == 'volume' else 2) for k in ('qubits', 'time_steps', 'volume')}
def kobayashi_cost_fsc8(d):
    return {k: float(np.polyval(POLY8[k], d)) for k in POLY8}
KC['time_steps_fsc8'], KC['volume_fsc8'] = KC8.time_steps, KC8.volume
print({k: np.round(v, 2) for k, v in POLY.items()})
print('with 8-step FSC rounds:', {k: np.round(v, 2) for k, v in POLY8.items()})
KC

## 第III部 Litinski 方式（誤り予算モデル）
* **GoSC 15-to-1**（Fig. 17c：11 tiles × 11τ、1τ = $d$ cycles）：単一の距離 $d$。
* **GoSC 225-to-1**（Fig. 19：176 tiles × 15τ）：1 段目 $d_1$、2 段目 $d_2$。
* **Litinski 2019 15-to-1** $(d_X,d_Z,d_m)$ と **(15-to-1)×(15-to-1)**：Litinski（2019）の配置と費用式（公開コードと同じ）。
* 入力の魔法状態：確率 $p$ の Pauli 誤り（GoSC Sec. 4 の仮定）。出力誤りは事後選択後の不忠実度、時間は成功確率で割る（期待値）。
* 量子ビット数はデータ＋測定量子ビット（1 tile = $2d^2$）、時間は code cycle × `STEPS_PER_CYCLE`。
* 以下の「最小」は、この格子探索の**候補集合の中の最小**である。

In [ ]:
import itertools as _it
def litinski_candidates(p, s):
    pl = PLOG[s]
    out = []
    for d in range(3, 52, 2):
        out.append(gosc_15to1(p, d, pl))
    lvl1 = {d1: run_steps(gosc_15to1_steps(p, d1, pl), out_qubit=4) for d1 in range(3, 26, 2)}
    for d1 in range(3, 26, 2):
        for d2 in range(d1, 52, 2):
            out.append(gosc_225to1(p, d1, d2, pl, level1=lvl1[d1]))
    for dx in range(5, 40, 2):
        for dz in range(3, dx + 1, 2):
            out.append(litinski2019_one_level(p, dx, dz, dz, pl))
    for dx, dz, dx2, dz2, nl1 in _it.product((7, 9, 11, 13), (3, 5), range(13, 42, 2), (5, 7, 9, 11, 13), (4, 6)):
        if dz < dx and dz2 < dx2:
            out.append(litinski2019_two_level(p, dx, dz, dz, dx2, dz2, dz2, nl1, pl))
    df = pd.DataFrame(out)
    df['family'] = df.protocol.str.extract(r'^(GoSC 15-to-1|GoSC 225-to-1|Litinski2019 \(15-to-1\)x|Litinski2019 \(15-to-1\))')[0]
    df['time_steps'] = df.cycles * STEPS_PER_CYCLE
    df['volume_steps'] = df.qubits * df.time_steps
    return df
import time as _time
t0 = _time.time()
LIT = {(s, p): litinski_candidates(p, s) for s in SCENARIOS for p in P_EVAL}
print('evaluated', sum(len(v) for v in LIT.values()), 'factory configurations in %.0f s' % (_time.time() - t0))

## 第IV部 比較
### IV-1. 出力誤り vs 時空間体積
Litinski 方式は候補の中のパレート前線。Kobayashi 方式は $d=3,\dots,9$ がシミュレーション（マーカー）、線は II-2 のモデル（$d>9$ は外挿）：実線＝本来のシミュレーションのモデル（bulk＋窓の項）、点線＝診断（窓の雑音なし、bulk のみ）。点線は**限界ではなく**、「窓の誤りが無かったら」という診断である。

In [ ]:
def pareto(df):
    df = df.sort_values('volume_steps')
    best, rows_ = np.inf, []
    for _, r in df.iterrows():
        if r.eps < best:
            best = r.eps; rows_.append(r)
    return pd.DataFrame(rows_)
show_p = [1e-4, 3e-4, 1e-3]
fig, axes = plt.subplots(2, len(show_p), figsize=(4.2 * len(show_p), 7.6), sharex=True, sharey=True)
ds = np.arange(3, 61, 2)
vk = np.array([kobayashi_cost(d)['volume'] for d in ds])
for i, s in enumerate(SCENARIOS):
    for j, p in enumerate(show_p):
        ax = axes[i, j]
        fr = pareto(LIT[(s, p)])
        ax.step(fr.volume_steps, fr.eps, where='post', color=C_LIT, label='Litinski (best in the candidate set)')
        ax.plot(vk, [eps_K_model(s, d, p) for d in ds], '-', color=C_KOB, lw=1.2, label='Kobayashi, model (d > 9 extrapolated)')
        ax.plot(vk, [eps_K_model(s, d, p, 'quiet') for d in ds], ':', color=C_KOB, lw=1.5,
                label='Kobayashi, diagnostic: window noise-free')
        sim = main[(main.scenario == s) & (main.p == p)]
        ax.errorbar([kobayashi_cost(d)['volume'] for d in sim.d], sim.eps, yerr=sim.stderr, fmt='o', color=C_KOB,
                    label='Kobayashi, simulated (± SE)')
        ax.axhline(p, color='#52514e', ls=':', lw=1)
        ax.set_xscale('log'); ax.set_yscale('log'); ax.set_ylim(1e-14, 1)
        ax.set_title(f'[{DATA_MODE}] scenario {s},  p = {p:g}', fontsize=10)
        if j == 0: ax.set_ylabel('output error of one T state')
        if i == 1: ax.set_xlabel('space-time volume [physical qubits x time steps]')
axes[0, 0].legend(fontsize=7)
plt.tight_layout(); plt.savefig(f'{FIGURES}/cost_comparison{OUT_SUFFIX}.png', dpi=150); plt.show()

### IV-2. 要求誤りごとの最小費用
* Litinski 方式：要求を満たす候補のうち体積最小のもの（候補集合内の最小）。
* Kobayashi 方式：要求を満たす最小の $d$（$d\le61$ の範囲で探索）。`kobayashi_*` は本来のシミュレーションのモデル（bulk＋窓の項）、`kobayashi_diag_*` は診断（窓の雑音なし）、`kobayashi_altW_*` は窓の項の $d$ 依存を許したモデル（感度）。
* `status`：ブートストラップの複製のうち、$d\le61$ で要求に届く割合が 84% 以上なら `reached`、16% 以下なら `not_reached_model`（このモデル・探索範囲では届かない。方式として不可能という意味ではない）、その間は `unresolved`。体積の `_16` / `_84` は届いた複製の体積の 16〜84% 範囲。
* `d_extrapolated`：選ばれた $d$ がシミュレーションした最大の $d$ を超える（$d$ 方向の外挿）。`p_extrapolated`：$p$ が bulk の当てはめに使った最小の $p$ より小さい（$p$ 方向の外挿）。外挿でなくても値はモデルによる推定であり、その要求誤りを直接観測したものではない。
* `kobayashi_diag_volume_fsc8`：同じ $d$ で FSC のラウンドを 8 ステップで数えた体積（時間だけの感度、II-3）。
* 各行に入力データの来歴（`data_mode`, `mc_sha256`, `cal_sha256`, `mc_run_ids`, ...）を記録する。

In [ ]:
def dmin(s, p, t, w_='noisy', params=None, alt=False):
    return next((d for d in range(3, 62, 2) if eps_K_model(s, d, p, w_, params, alt) <= t), None)
def kob_entry(s, p, t, w_, alt=False):
    dc = dmin(s, p, t, w_, alt=alt)
    db = [dmin(s, p, t, w_, bt, alt) for bt in FITS[s]['boot']]
    reach = [d for d in db if d is not None]
    frac = len(reach) / len(db)
    status = 'reached' if frac >= 0.84 else 'not_reached_model' if frac <= 0.16 else 'unresolved'
    vols = [kobayashi_cost(d)['volume'] for d in reach]
    kc = kobayashi_cost(dc) if dc else None
    return dict(status=status, frac_reached=frac, d=dc if dc else '—',
                d_extrapolated=bool(dc is not None and dc > FITS[s]['d_max_data']),
                qubits=kc['qubits'] if kc else np.nan, time_steps=kc['time_steps'] if kc else np.nan,
                volume=kc['volume'] if kc else np.nan,
                volume_16=np.quantile(vols, 0.16) if vols else np.nan, volume_84=np.quantile(vols, 0.84) if vols else np.nan)
rows = []
for s in SCENARIOS:
    for p in P_EVAL:
        df = LIT[(s, p)]
        for t in TARGETS:
            ok = df[df.eps <= t]
            lit = ok.loc[ok.volume_steps.idxmin()] if len(ok) else None
            r = dict(scenario=s, p=p, target=t, p_extrapolated=bool(p < FITS[s]['p_min_bulk']),
                     litinski_protocol=lit.protocol if lit is not None else 'none in candidate set',
                     litinski_eps=lit.eps if lit is not None else np.nan,
                     litinski_qubits=lit.qubits if lit is not None else np.nan,
                     litinski_time_steps=lit.time_steps if lit is not None else np.nan,
                     litinski_volume=lit.volume_steps if lit is not None else np.nan)
            for pre, w_, alt in (('kobayashi', 'noisy', False), ('kobayashi_diag', 'quiet', False),
                                 ('kobayashi_altW', 'noisy', True)):
                r.update({f'{pre}_{k}': v for k, v in kob_entry(s, p, t, w_, alt).items()})
            rows.append(r)
CMP = pd.DataFrame(rows)
for pre in ('kobayashi', 'kobayashi_diag', 'kobayashi_altW'):
    CMP[f'ratio_{pre}_over_litinski'] = CMP[f'{pre}_volume'] / CMP.litinski_volume
CMP['kobayashi_diag_volume_fsc8'] = [kobayashi_cost_fsc8(d)['volume'] if d != '—' else np.nan for d in CMP.kobayashi_diag_d]
CMP['ratio_kobayashi_diag_fsc8_over_litinski'] = CMP.kobayashi_diag_volume_fsc8 / CMP.litinski_volume
for k, v in PROV.items():
    CMP[k] = v
pd.set_option('display.max_colwidth', 40)
CMP[['scenario', 'p', 'target', 'litinski_protocol', 'litinski_volume', 'kobayashi_status', 'kobayashi_volume',
     'kobayashi_altW_status', 'kobayashi_diag_status', 'kobayashi_diag_d', 'kobayashi_diag_d_extrapolated', 'p_extrapolated',
     'ratio_kobayashi_diag_over_litinski', 'kobayashi_diag_volume_16', 'kobayashi_diag_volume_84',
     'ratio_kobayashi_diag_fsc8_over_litinski']]

### IV-3. Kobayashi 方式が同じ体積に収まるために必要な改善（診断値）
要求誤りごとに、Litinski 方式の最小体積以下に収まる Kobayashi 方式の最大の $d$（$d_{\max}$）を求め、そこでの $\varepsilon_K(d_{\max},p)$（本来のシミュレーションのモデル）が要求誤りの何倍か（`required_improvement`）を示す。**現在の費用モデルを固定した場合の目安**であり、復号器やスケジュールを変えれば量子ビット数・時間も変わるので、この倍率の改善で必ず同じ費用に並ぶという保証ではない。

In [ ]:
rows = []
for _, r in CMP.iterrows():
    if not np.isfinite(r.litinski_volume):
        continue
    ds_ = [d for d in range(3, 62, 2) if kobayashi_cost(d)['volume'] <= r.litinski_volume]
    dmax = max(ds_) if ds_ else None
    ek = eps_K_model(r.scenario, dmax, r.p) if dmax else np.nan
    rows.append(dict(scenario=r.scenario, p=r.p, target=r.target, litinski_volume=r.litinski_volume,
                     kobayashi_d_max=dmax, eps_K_at_d_max=ek, required_improvement=ek / r.target if dmax else np.nan))
IMP = pd.DataFrame(rows)
CMP = CMP.merge(IMP[['scenario', 'p', 'target', 'kobayashi_d_max', 'required_improvement']], how='left',
                on=['scenario', 'p', 'target'])
CMP.to_csv(f'{RESULTS}/cost_comparison{OUT_SUFFIX}.csv', index=False)
print('wrote', f'{RESULTS}/cost_comparison{OUT_SUFFIX}.csv', '| data_mode =', DATA_MODE)
IMP.round({'eps_K_at_d_max': 8, 'required_improvement': 1})

## 結論
本研究で構成した回路・スケジュール・復号器と、§II-2 のモデル（外挿を含む）に基づく結論。データ：`production-20261029T053925Z-29f79c`（329 万ショット）。数値は `results/cost_comparison.csv`。

1. **本来のシミュレーション**：Kobayashi 方式の出力誤りは $d\ge5$ で $d$ をほとんど下げず、シナリオ B で $\varepsilon_K\approx(3\text{–}8)\,p$、A ではさらに大きい（$p=10^{-3}$ で約 20%）。主因は切り替えの窓での単一故障で、`validation.ipynb` N5 で窓を全列挙した一次の係数は $c_1=3.1$（B, $d=5$）、$3.9$（B, $d=7$）、$3.0$（A, $d=5$）と $d\ge5$ でほぼ一定、測定値と整合する。このモデル（窓の項の $d$ 依存を許した感度モデルでも）では、評価した全ての $(p,$ 要求誤り$)$ で要求に届かない（`not_reached_model`）。これは**本研究の回路・復号器での結果**であり、方式として不可能という意味ではない。
2. **窓の雑音を除いた診断**（同じ復号器）では $\varepsilon_K$ は $d$ とともに下がる（測定範囲の端 $p=3\times10^{-4}$ で、当てはめたモデルの減少率は距離 2 ごとに A 約 4.2 倍、B 約 3.5 倍）。この診断のモデルを費用比較に入れると、要求 $10^{-6}$ 以下では Kobayashi 方式の体積は Litinski 方式（候補集合内の最小）の 2.7〜39 倍（A：4.2〜39 倍、$p=10^{-3}$ では届かない；B：2.7〜33 倍）。並ぶか下回るのは要求 $10^{-4}$ の一部（B・$p=10^{-4}$ で 0.52 倍）だけ。多くの行は $d>9$ や低い $p$ への外挿で、ブートストラップの範囲は表の `_16`/`_84` 列。
3. **解釈**：本研究の実装では、Kobayashi 方式の性能は切り替えの窓の扱いで決まっている。窓の雑音を除いた診断でも蒸留の方が体積が小さい傾向が残るが、診断は最適な復号器の性能の限界（上界・下界）ではなく、また外挿に依存するので、**最適化された D4 方式に対する一般的な優劣を示すものではない**。
4. 費用の差の内訳：D4 相の 1 ラウンドが長く（CCZ を含む X 型安定化子の測定 28 ステップ＋$Z$ 検査 6 ステップ）雑音の位置が多い、出力 1 個のために $d\times d$ 格子全体を $O(d)$ ラウンド使う（体積 $\approx632d^3$）。switch-in のラウンド数は $\kappa_{\rm in}\ge0.6$ で $\varepsilon_K$ がほぼ変わらない（窓の項が支配するため）。
5. **結論を変えうる要因**（いずれも未解決）：切り替えの窓を扱える復号器（[Dav] の JIT decoder）や回路の改良、測定スケジュールの最適化（FSC 14 → 6 ステップ）、CCZ の実際の誤り率、FSC → 通常パッチの変換（未計上）、Litinski 側の状態注入（未計上）、Litinski 側を回路レベルでシミュレーションした場合との差。